# Uncertainty-Aware and Human-Reviewed RUL-Based Maintenance Prioritisation

**Researcher:** Ranitabh Mallick  
**Dataset:** NASA C-MAPSS FD001  
**Environment:** Google Colab  
**Notebook purpose:** Reproducible thesis implementation  

This notebook implements an uncertainty-aware, explainable and selectively
human-reviewed framework for translating Remaining Useful Life predictions
into transparent maintenance priorities.

In [ ]:
# ============================================================
# PHASE 1 — DATASET GOVERNANCE
# STEP 3A — CREATE AND VERIFY THE COLAB WORKSPACE
# ============================================================

from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
from datetime import datetime, timezone
import platform
import sys
import json

# ------------------------------------------------------------
# 1. Define the main temporary project location
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/Predictive_Maintenance_Thesis")

# ------------------------------------------------------------
# 2. Define separate locations for each type of thesis output
# ------------------------------------------------------------

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "FD001"
INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
LOGS_DIR = PROJECT_ROOT / "logs"

PROJECT_DIRECTORIES = [
    RAW_DATA_DIR,
    INTERIM_DATA_DIR,
    PROCESSED_DATA_DIR,
    NOTEBOOK_DIR,
    MODEL_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    LOGS_DIR
]

# ------------------------------------------------------------
# 3. Create the directories
# ------------------------------------------------------------

for directory in PROJECT_DIRECTORIES:
    directory.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 4. Record versions of important Python packages
# ------------------------------------------------------------

package_names = [
    "numpy",
    "pandas",
    "scikit-learn",
    "matplotlib",
    "seaborn",
    "xgboost",
    "shap"
]

package_versions = {}

for package_name in package_names:
    try:
        package_versions[package_name] = version(package_name)
    except PackageNotFoundError:
        package_versions[package_name] = "Not installed"

# ------------------------------------------------------------
# 5. Create a reproducibility record
# ------------------------------------------------------------

environment_record = {
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": sys.version.split()[0],
    "operating_system": platform.platform(),
    "project_root": str(PROJECT_ROOT),
    "package_versions": package_versions
}

environment_file = LOGS_DIR / "environment.json"

with environment_file.open("w", encoding="utf-8") as file:
    json.dump(environment_record, file, indent=4)

# ------------------------------------------------------------
# 6. Verify and display the workspace
# ------------------------------------------------------------

print("THESIS WORKSPACE CREATED SUCCESSFULLY")
print("=" * 60)

print(f"Project root: {PROJECT_ROOT}")
print(f"Python version: {environment_record['python_version']}")
print(f"Operating system: {platform.system()}")

print("\nDIRECTORY VERIFICATION")
print("-" * 60)

all_directories_valid = True

for directory in PROJECT_DIRECTORIES:
    exists = directory.exists()
    status = "PASS" if exists else "FAIL"

    print(
        f"{status}: "
        f"{directory.relative_to(PROJECT_ROOT)}"
    )

    if not exists:
        all_directories_valid = False

print("\nPACKAGE VERSIONS")
print("-" * 60)

for package_name, package_version in package_versions.items():
    print(f"{package_name}: {package_version}")

print("\nENVIRONMENT RECORD")
print("-" * 60)
print(f"Saved to: {environment_file}")

print("\nFINAL CHECK")
print("-" * 60)

if all_directories_valid and environment_file.exists():
    print("PASS — Workspace structure and environment record verified.")
else:
    print("FAIL — One or more workspace components are missing.")

THESIS WORKSPACE CREATED SUCCESSFULLY
Project root: /content/Predictive_Maintenance_Thesis
Python version: 3.13.15
Operating system: Linux

DIRECTORY VERIFICATION
------------------------------------------------------------
PASS: data/raw/FD001
PASS: data/interim
PASS: data/processed
PASS: notebooks
PASS: models
PASS: results
PASS: figures
PASS: logs

PACKAGE VERSIONS
------------------------------------------------------------
numpy: 2.1.3
pandas: 2.2.3
scikit-learn: 1.6.1
matplotlib: 3.10.0
seaborn: 0.13.2
xgboost: 3.4.1
shap: 0.52.0

ENVIRONMENT RECORD
------------------------------------------------------------
Saved to: /content/Predictive_Maintenance_Thesis/logs/environment.json

FINAL CHECK
------------------------------------------------------------
PASS — Workspace structure and environment record verified.


In [ ]:
# ============================================================
# PHASE 1 — DATASET GOVERNANCE
# STEP 3B — UPLOAD AND IDENTIFY THE ORIGINAL FD001 FILES
# ============================================================

from google.colab import files
from pathlib import Path

# The three files required for the approved FD001 study
EXPECTED_FILES = {
    "train_FD001.txt",
    "test_FD001.txt",
    "RUL_FD001.txt"
}

print("Select ONLY these three original files:")
for filename in sorted(EXPECTED_FILES):
    print(f"• {filename}")

print("\nOpening the upload window...")

# Opens the file selector on your computer
uploaded_files = files.upload()

# Identify what was uploaded
uploaded_names = set(uploaded_files.keys())
missing_files = EXPECTED_FILES - uploaded_names
unexpected_files = uploaded_names - EXPECTED_FILES

print("\nUPLOAD INVENTORY")
print("=" * 60)

for filename in sorted(uploaded_names):
    file_size = len(uploaded_files[filename])

    print(
        f"{filename:<25} "
        f"{file_size:>12,} bytes"
    )

print("\nVALIDATION")
print("=" * 60)

if missing_files:
    print("FAIL — Required files are missing:")

    for filename in sorted(missing_files):
        print(f"• {filename}")

elif unexpected_files:
    print("FAIL — Unexpected files were uploaded:")

    for filename in sorted(unexpected_files):
        print(f"• {filename}")

else:
    print("PASS — All three required FD001 files were identified.")
    print("No FD002–FD004 files were included.")
    print("The uploaded data have not been modified or analysed.")

Select ONLY these three original files:
• RUL_FD001.txt
• test_FD001.txt
• train_FD001.txt

Opening the upload window...


In [ ]:
# ============================================================
# PHASE 1 — DATASET GOVERNANCE
# STEP 3C — PROTECT, HASH AND DOCUMENT FD001 RAW FILES
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from google.colab import files as colab_files

import hashlib
import shutil
import json
import csv
import os
import zipfile


# ------------------------------------------------------------
# 1. Define controlled locations
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/Predictive_Maintenance_Thesis")
UPLOAD_DIR = Path("/content")
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "FD001"
LOGS_DIR = PROJECT_ROOT / "logs"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_FILES = [
    "train_FD001.txt",
    "test_FD001.txt",
    "RUL_FD001.txt"
]


# ------------------------------------------------------------
# 2. Function for calculating a SHA-256 fingerprint
# ------------------------------------------------------------

def calculate_sha256(file_path, chunk_size=1024 * 1024):
    """
    Read a file in binary chunks and calculate its SHA-256 hash.
    Reading in chunks avoids loading a large file fully into memory.
    """

    sha256_hash = hashlib.sha256()

    with file_path.open("rb") as file:
        while True:
            data_chunk = file.read(chunk_size)

            if not data_chunk:
                break

            sha256_hash.update(data_chunk)

    return sha256_hash.hexdigest()


# ------------------------------------------------------------
# 3. Confirm that all uploaded files are present
# ------------------------------------------------------------

missing_uploads = [
    filename
    for filename in EXPECTED_FILES
    if not (UPLOAD_DIR / filename).is_file()
]

if missing_uploads:
    raise FileNotFoundError(
        "Required uploaded files are missing: "
        + ", ".join(missing_uploads)
    )


# ------------------------------------------------------------
# 4. Copy and verify every raw file
# ------------------------------------------------------------

recorded_at_utc = datetime.now(timezone.utc).isoformat()
file_records = []

print("FD001 RAW-FILE PROTECTION AND INTEGRITY CHECK")
print("=" * 75)

for filename in EXPECTED_FILES:

    uploaded_path = UPLOAD_DIR / filename
    protected_path = RAW_DATA_DIR / filename

    # Fingerprint of the file currently uploaded from the computer
    source_hash = calculate_sha256(uploaded_path)
    source_size = uploaded_path.stat().st_size

    # Prevent accidental replacement of an existing different file
    if protected_path.exists():

        existing_hash = calculate_sha256(protected_path)

        if existing_hash != source_hash:
            raise RuntimeError(
                f"STOP — {filename} already exists in the raw folder "
                "but has a different SHA-256 fingerprint."
            )

        copy_action = "EXISTING IDENTICAL COPY VERIFIED"

    else:
        # Binary-preserving copy into the controlled raw-data folder
        shutil.copy2(uploaded_path, protected_path)
        copy_action = "NEW PROTECTED COPY CREATED"

    # Fingerprint the protected copy independently
    protected_hash = calculate_sha256(protected_path)
    protected_size = protected_path.stat().st_size

    hashes_match = source_hash == protected_hash
    sizes_match = source_size == protected_size

    if not hashes_match or not sizes_match:
        raise RuntimeError(
            f"Integrity verification failed for {filename}."
        )

    # Operational protection against accidental editing
    os.chmod(protected_path, 0o444)

    file_records.append({
        "filename": filename,
        "uploaded_location": str(uploaded_path),
        "protected_location": str(protected_path),
        "size_bytes": source_size,
        "sha256": source_hash,
        "source_and_copy_hashes_match": hashes_match,
        "source_and_copy_sizes_match": sizes_match,
        "copy_action": copy_action,
        "protected_copy_mode": "read-only"
    })

    print(f"\nFILE: {filename}")
    print(f"Size: {source_size:,} bytes")
    print(f"SHA-256: {source_hash}")
    print(f"Action: {copy_action}")
    print("Integrity: PASS — Source and protected copy are identical")


# ------------------------------------------------------------
# 5. Create the provenance record
# ------------------------------------------------------------

provenance_record = {
    "record_type": "FD001 raw-data provenance and integrity record",
    "recorded_at_utc": recorded_at_utc,
    "researcher": "Ranitabh Mallick",
    "thesis_title": (
        "An Uncertainty-Aware, Explainable, and Human-Reviewed "
        "Framework for RUL-Based Maintenance Prioritisation "
        "Using NASA C-MAPSS FD001"
    ),
    "dataset_name": "NASA C-MAPSS",
    "approved_subset": "FD001 only",
    "official_reference_page": (
        "https://data.nasa.gov/dataset/"
        "cmapss-jet-engine-simulated-data"
    ),
    "acquisition_method": (
        "Original dataset files held by the researcher and "
        "uploaded directly from the researcher's computer to Colab."
    ),
    "authenticity_statement": (
        "The hashes establish the identity and continuing integrity "
        "of the files used in this experiment. They do not, by "
        "themselves, independently prove the files' original source."
    ),
    "immutability_policy": (
        "Raw files must not be edited, renamed, cleaned or overwritten. "
        "All transformations must be written to interim or processed folders."
    ),
    "files": file_records
}

provenance_file = LOGS_DIR / "fd001_provenance.json"

with provenance_file.open("w", encoding="utf-8") as file:
    json.dump(provenance_record, file, indent=4)


# ------------------------------------------------------------
# 6. Create a simple CSV file manifest
# ------------------------------------------------------------

manifest_file = LOGS_DIR / "fd001_file_manifest.csv"

with manifest_file.open(
    "w",
    encoding="utf-8",
    newline=""
) as file:

    fieldnames = [
        "filename",
        "size_bytes",
        "sha256",
        "protected_location",
        "source_and_copy_hashes_match",
        "protected_copy_mode"
    ]

    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()

    for record in file_records:
        writer.writerow({
            fieldname: record[fieldname]
            for fieldname in fieldnames
        })


# ------------------------------------------------------------
# 7. Verify the protected raw-data inventory
# ------------------------------------------------------------

protected_names = {
    file_path.name
    for file_path in RAW_DATA_DIR.iterdir()
    if file_path.is_file()
}

inventory_valid = protected_names == set(EXPECTED_FILES)
all_hashes_valid = all(
    record["source_and_copy_hashes_match"]
    for record in file_records
)


# ------------------------------------------------------------
# 8. Package the governance evidence
# ------------------------------------------------------------

evidence_zip = PROJECT_ROOT / "Phase1_Step3C_Governance_Evidence.zip"

with zipfile.ZipFile(
    evidence_zip,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as zip_file:

    zip_file.write(
        provenance_file,
        arcname=provenance_file.name
    )

    zip_file.write(
        manifest_file,
        arcname=manifest_file.name
    )


# ------------------------------------------------------------
# 9. Display the final verification result
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("GOVERNANCE RECORDS")
print("=" * 75)
print(f"Provenance record: {provenance_file}")
print(f"File manifest: {manifest_file}")
print(f"Evidence package: {evidence_zip}")

print("\nFINAL CHECK")
print("=" * 75)

if (
    inventory_valid
    and all_hashes_valid
    and provenance_file.exists()
    and manifest_file.exists()
    and evidence_zip.exists()
):
    print("PASS — FD001 raw files were protected and fingerprinted.")
    print("PASS — Source and protected-copy hashes match.")
    print("PASS — Provenance and manifest records were created.")
    print("PASS — No dataset analysis or preprocessing was performed.")

    print("\nDownloading the governance evidence ZIP...")
    colab_files.download(str(evidence_zip))

else:
    print("FAIL — One or more governance checks did not pass.")

In [ ]:
# ============================================================
# PHASE 1 — DATASET GOVERNANCE
# STEP 4A — LOAD AND STRUCTURALLY VALIDATE FD001
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import json


# ------------------------------------------------------------
# 1. Define protected file locations
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/Predictive_Maintenance_Thesis")
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "FD001"
LOGS_DIR = PROJECT_ROOT / "logs"

TRAIN_FILE = RAW_DATA_DIR / "train_FD001.txt"
TEST_FILE = RAW_DATA_DIR / "test_FD001.txt"
RUL_FILE = RAW_DATA_DIR / "RUL_FD001.txt"

required_paths = [TRAIN_FILE, TEST_FILE, RUL_FILE]


# ------------------------------------------------------------
# 2. Confirm that all protected files still exist
# ------------------------------------------------------------

missing_paths = [
    str(file_path)
    for file_path in required_paths
    if not file_path.is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Protected FD001 files are missing:\n"
        + "\n".join(missing_paths)
    )

print("PASS — All three protected FD001 files were found.")


# ------------------------------------------------------------
# 3. Read the files without changing them
# ------------------------------------------------------------

# sep=r"\s+" means that one or more spaces separate the values.
# header=None means that the original files do not contain headings.

train_raw = pd.read_csv(
    TRAIN_FILE,
    sep=r"\s+",
    header=None
)

test_raw = pd.read_csv(
    TEST_FILE,
    sep=r"\s+",
    header=None
)

test_rul_raw = pd.read_csv(
    RUL_FILE,
    sep=r"\s+",
    header=None
)


# ------------------------------------------------------------
# 4. Define the NASA C-MAPSS FD001 schema
# ------------------------------------------------------------

identifier_columns = [
    "unit_number",
    "time_in_cycles"
]

operational_setting_columns = [
    "operational_setting_1",
    "operational_setting_2",
    "operational_setting_3"
]

sensor_columns = [
    f"sensor_{sensor_number}"
    for sensor_number in range(1, 22)
]

FD001_COLUMNS = (
    identifier_columns
    + operational_setting_columns
    + sensor_columns
)


# ------------------------------------------------------------
# 5. Validate column counts before assigning names
# ------------------------------------------------------------

train_column_count_valid = train_raw.shape[1] == 26
test_column_count_valid = test_raw.shape[1] == 26
rul_column_count_valid = test_rul_raw.shape[1] == 1

if not all([
    train_column_count_valid,
    test_column_count_valid,
    rul_column_count_valid
]):
    raise ValueError(
        "Unexpected FD001 column structure detected.\n"
        f"Training columns: {train_raw.shape[1]} — expected 26\n"
        f"Test columns: {test_raw.shape[1]} — expected 26\n"
        f"RUL columns: {test_rul_raw.shape[1]} — expected 1"
    )


# ------------------------------------------------------------
# 6. Assign meaningful names to the in-memory tables
# ------------------------------------------------------------

train_df = train_raw.copy()
test_df = test_raw.copy()
test_rul_df = test_rul_raw.copy()

train_df.columns = FD001_COLUMNS
test_df.columns = FD001_COLUMNS

test_rul_df.columns = [
    "rul_at_last_observed_cycle"
]


# ------------------------------------------------------------
# 7. Calculate structural information
# ------------------------------------------------------------

training_engine_count = int(
    train_df["unit_number"].nunique()
)

test_engine_count = int(
    test_df["unit_number"].nunique()
)

rul_record_count = int(len(test_rul_df))

all_training_columns_numeric = bool(
    all(
        pd.api.types.is_numeric_dtype(data_type)
        for data_type in train_df.dtypes
    )
)

all_test_columns_numeric = bool(
    all(
        pd.api.types.is_numeric_dtype(data_type)
        for data_type in test_df.dtypes
    )
)

rul_is_numeric = bool(
    pd.api.types.is_numeric_dtype(
        test_rul_df["rul_at_last_observed_cycle"]
    )
)


# ------------------------------------------------------------
# 8. Compare with the canonical FD001 structure
# ------------------------------------------------------------

structural_checks = {
    "training_shape_is_20631_by_26":
        train_df.shape == (20631, 26),

    "test_shape_is_13096_by_26":
        test_df.shape == (13096, 26),

    "rul_shape_is_100_by_1":
        test_rul_df.shape == (100, 1),

    "training_contains_100_engines":
        training_engine_count == 100,

    "test_contains_100_engines":
        test_engine_count == 100,

    "one_rul_record_exists_per_test_engine":
        rul_record_count == test_engine_count,

    "all_training_columns_are_numeric":
        all_training_columns_numeric,

    "all_test_columns_are_numeric":
        all_test_columns_numeric,

    "test_rul_column_is_numeric":
        rul_is_numeric
}


# ------------------------------------------------------------
# 9. Save a reproducible structural-validation record
# ------------------------------------------------------------

structural_report = {
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": "NASA C-MAPSS FD001",
    "validation_stage": "Structural validation only",
    "training_shape": list(train_df.shape),
    "test_shape": list(test_df.shape),
    "test_rul_shape": list(test_rul_df.shape),
    "training_engine_count": training_engine_count,
    "test_engine_count": test_engine_count,
    "rul_record_count": rul_record_count,
    "assigned_columns": FD001_COLUMNS,
    "rul_column": "rul_at_last_observed_cycle",
    "checks": structural_checks,
    "data_modified": False
}

structural_report_file = (
    LOGS_DIR / "fd001_structural_validation.json"
)

with structural_report_file.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(structural_report, file, indent=4)


# ------------------------------------------------------------
# 10. Display the results
# ------------------------------------------------------------

print("\nFD001 STRUCTURAL SUMMARY")
print("=" * 70)
print(f"Training shape: {train_df.shape}")
print(f"Test shape:     {test_df.shape}")
print(f"Test-RUL shape: {test_rul_df.shape}")
print(f"Training engines: {training_engine_count}")
print(f"Test engines:     {test_engine_count}")
print(f"Test-RUL records: {rul_record_count}")
print(f"Named FD001 columns: {len(FD001_COLUMNS)}")

print("\nSTRUCTURAL CHECKS")
print("=" * 70)

for check_name, passed in structural_checks.items():
    status = "PASS" if passed else "FAIL"
    readable_name = check_name.replace("_", " ")

    print(f"{status} — {readable_name}")

print("\nVALIDATION RECORD")
print("=" * 70)
print(f"Saved to: {structural_report_file}")

print("\nFINAL CHECK")
print("=" * 70)

if all(structural_checks.values()):
    print("PASS — FD001 structural validation completed successfully.")
    print("PASS — The protected raw files were only read, not modified.")
    print("PASS — No cleaning, RUL construction or modelling occurred.")
else:
    print("FAIL — At least one structural check requires investigation.")

In [ ]:
# ============================================================
# PHASE 1 — DATASET GOVERNANCE
# STEP 4B — READ-ONLY DATA-QUALITY AUDIT
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from google.colab import files as colab_files

import pandas as pd
import numpy as np
import json
import zipfile


# ------------------------------------------------------------
# 1. Confirm that Step 4A dataframes are available
# ------------------------------------------------------------

required_dataframe_names = [
    "train_df",
    "test_df",
    "test_rul_df"
]

missing_dataframes = [
    name
    for name in required_dataframe_names
    if name not in globals()
]

if missing_dataframes:
    raise RuntimeError(
        "Run the Step 4A structural-validation cell first. "
        "Missing dataframes: "
        + ", ".join(missing_dataframes)
    )


# ------------------------------------------------------------
# 2. Define output locations
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/Predictive_Maintenance_Thesis")
LOGS_DIR = PROJECT_ROOT / "logs"
LOGS_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 3. Helper functions
# ------------------------------------------------------------

def count_non_finite_values(dataframe):
    """
    Count positive infinity and negative infinity.
    Missing values are counted separately.
    """

    numeric_values = dataframe.select_dtypes(
        include=[np.number]
    ).to_numpy(dtype=float)

    return int(
        np.isinf(numeric_values).sum()
    )


def identifiers_are_positive_integers(dataframe):
    """
    Check whether unit and cycle identifiers are positive,
    whole numbers.
    """

    unit_values = dataframe["unit_number"].to_numpy()
    cycle_values = dataframe["time_in_cycles"].to_numpy()

    unit_check = (
        np.all(unit_values > 0)
        and np.all(np.mod(unit_values, 1) == 0)
    )

    cycle_check = (
        np.all(cycle_values > 0)
        and np.all(np.mod(cycle_values, 1) == 0)
    )

    return bool(unit_check and cycle_check)


def check_engine_cycle_sequences(dataframe):
    """
    Check the cycle sequence separately for every engine.
    """

    engines_start_at_one = True
    cycles_are_consecutive = True
    cycles_are_in_increasing_order = True

    for _, engine_data in dataframe.groupby(
        "unit_number",
        sort=True
    ):
        original_cycles = engine_data[
            "time_in_cycles"
        ].to_numpy()

        sorted_unique_cycles = np.sort(
            np.unique(original_cycles)
        )

        if sorted_unique_cycles[0] != 1:
            engines_start_at_one = False

        expected_cycles = np.arange(
            1,
            sorted_unique_cycles[-1] + 1
        )

        if not np.array_equal(
            sorted_unique_cycles,
            expected_cycles
        ):
            cycles_are_consecutive = False

        if not np.array_equal(
            original_cycles,
            np.sort(original_cycles)
        ):
            cycles_are_in_increasing_order = False

    globally_sorted = dataframe[
        ["unit_number", "time_in_cycles"]
    ].reset_index(drop=True).equals(

        dataframe.sort_values(
            ["unit_number", "time_in_cycles"]
        )[
            ["unit_number", "time_in_cycles"]
        ].reset_index(drop=True)
    )

    return {
        "engines_start_at_cycle_one":
            bool(engines_start_at_one),

        "cycles_are_consecutive_with_no_gaps":
            bool(cycles_are_consecutive),

        "cycles_increase_within_each_engine":
            bool(cycles_are_in_increasing_order),

        "rows_are_sorted_by_engine_and_cycle":
            bool(globally_sorted)
    }


def create_feature_profile(dataframe, dataset_name):
    """
    Create a read-only profile of operational settings and sensors.

    Near-constant definition:
    More than or equal to 99% of the rows contain the same value,
    while the column still has more than one unique value.
    """

    feature_columns = [
        column
        for column in dataframe.columns
        if column not in [
            "unit_number",
            "time_in_cycles"
        ]
    ]

    profile_records = []

    for column in feature_columns:
        value_counts = dataframe[
            column
        ].value_counts(
            normalize=True,
            dropna=False
        )

        unique_count = int(
            dataframe[column].nunique(dropna=False)
        )

        dominant_value_ratio = float(
            value_counts.iloc[0]
        )

        is_constant = unique_count == 1

        is_near_constant = (
            unique_count > 1
            and dominant_value_ratio >= 0.99
        )

        profile_records.append({
            "dataset": dataset_name,
            "column": column,
            "unique_value_count": unique_count,
            "dominant_value_ratio": dominant_value_ratio,
            "standard_deviation": float(
                dataframe[column].std()
            ),
            "is_constant": bool(is_constant),
            "is_near_constant_99_percent": bool(
                is_near_constant
            )
        })

    return pd.DataFrame(profile_records)


# ------------------------------------------------------------
# 4. Calculate basic data-quality counts
# ------------------------------------------------------------

train_missing_count = int(
    train_df.isna().sum().sum()
)

test_missing_count = int(
    test_df.isna().sum().sum()
)

rul_missing_count = int(
    test_rul_df.isna().sum().sum()
)

train_exact_duplicates = int(
    train_df.duplicated().sum()
)

test_exact_duplicates = int(
    test_df.duplicated().sum()
)

train_duplicate_engine_cycles = int(
    train_df.duplicated(
        subset=["unit_number", "time_in_cycles"]
    ).sum()
)

test_duplicate_engine_cycles = int(
    test_df.duplicated(
        subset=["unit_number", "time_in_cycles"]
    ).sum()
)

train_non_finite_count = count_non_finite_values(
    train_df
)

test_non_finite_count = count_non_finite_values(
    test_df
)

rul_non_finite_count = count_non_finite_values(
    test_rul_df
)


# ------------------------------------------------------------
# 5. Check engine-cycle sequences
# ------------------------------------------------------------

train_sequence_checks = check_engine_cycle_sequences(
    train_df
)

test_sequence_checks = check_engine_cycle_sequences(
    test_df
)


# ------------------------------------------------------------
# 6. Profile constant and near-constant features
# ------------------------------------------------------------

train_feature_profile = create_feature_profile(
    train_df,
    "training"
)

test_feature_profile = create_feature_profile(
    test_df,
    "test"
)

combined_feature_profile = pd.concat(
    [
        train_feature_profile,
        test_feature_profile
    ],
    ignore_index=True
)

train_constant_columns = train_feature_profile.loc[
    train_feature_profile["is_constant"],
    "column"
].tolist()

train_near_constant_columns = train_feature_profile.loc[
    train_feature_profile[
        "is_near_constant_99_percent"
    ],
    "column"
].tolist()

test_constant_columns = test_feature_profile.loc[
    test_feature_profile["is_constant"],
    "column"
].tolist()

test_near_constant_columns = test_feature_profile.loc[
    test_feature_profile[
        "is_near_constant_99_percent"
    ],
    "column"
].tolist()


# ------------------------------------------------------------
# 7. Define pass/fail quality checks
# ------------------------------------------------------------

quality_checks = {
    "training_has_no_missing_values":
        train_missing_count == 0,

    "test_has_no_missing_values":
        test_missing_count == 0,

    "test_rul_has_no_missing_values":
        rul_missing_count == 0,

    "training_has_no_exact_duplicate_rows":
        train_exact_duplicates == 0,

    "test_has_no_exact_duplicate_rows":
        test_exact_duplicates == 0,

    "training_has_no_duplicate_engine_cycle_keys":
        train_duplicate_engine_cycles == 0,

    "test_has_no_duplicate_engine_cycle_keys":
        test_duplicate_engine_cycles == 0,

    "training_identifiers_are_positive_integers":
        identifiers_are_positive_integers(train_df),

    "test_identifiers_are_positive_integers":
        identifiers_are_positive_integers(test_df),

    "training_has_no_infinite_values":
        train_non_finite_count == 0,

    "test_has_no_infinite_values":
        test_non_finite_count == 0,

    "test_rul_has_no_infinite_values":
        rul_non_finite_count == 0,

    "test_rul_values_are_non_negative":
        bool(
            (
                test_rul_df[
                    "rul_at_last_observed_cycle"
                ] >= 0
            ).all()
        ),

    **{
        f"training_{name}": passed
        for name, passed
        in train_sequence_checks.items()
    },

    **{
        f"test_{name}": passed
        for name, passed
        in test_sequence_checks.items()
    }
}


# ------------------------------------------------------------
# 8. Save the reports
# ------------------------------------------------------------

quality_report = {
    "recorded_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "dataset": "NASA C-MAPSS FD001",

    "validation_stage":
        "Read-only data-quality audit",

    "missing_values": {
        "training": train_missing_count,
        "test": test_missing_count,
        "test_rul": rul_missing_count
    },

    "exact_duplicate_rows": {
        "training": train_exact_duplicates,
        "test": test_exact_duplicates
    },

    "duplicate_engine_cycle_keys": {
        "training": train_duplicate_engine_cycles,
        "test": test_duplicate_engine_cycles
    },

    "non_finite_values": {
        "training": train_non_finite_count,
        "test": test_non_finite_count,
        "test_rul": rul_non_finite_count
    },

    "training_sequence_checks":
        train_sequence_checks,

    "test_sequence_checks":
        test_sequence_checks,

    "training_constant_columns":
        train_constant_columns,

    "training_near_constant_columns":
        train_near_constant_columns,

    "test_constant_columns":
        test_constant_columns,

    "test_near_constant_columns":
        test_near_constant_columns,

    "quality_checks": quality_checks,

    "important_note": (
        "No rows or columns were removed. Constant and "
        "near-constant findings are descriptive only. "
        "Any later feature-removal decision must be "
        "documented and fitted without using the final test set."
    ),

    "data_modified": False
}

quality_report_file = (
    LOGS_DIR / "fd001_data_quality_report.json"
)

feature_profile_file = (
    LOGS_DIR / "fd001_feature_profile.csv"
)

with quality_report_file.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(quality_report, file, indent=4)

combined_feature_profile.to_csv(
    feature_profile_file,
    index=False
)


# ------------------------------------------------------------
# 9. Display findings
# ------------------------------------------------------------

print("FD001 DATA-QUALITY COUNTS")
print("=" * 75)
print(f"Training missing values: {train_missing_count}")
print(f"Test missing values: {test_missing_count}")
print(f"Test-RUL missing values: {rul_missing_count}")
print(f"Training exact duplicate rows: {train_exact_duplicates}")
print(f"Test exact duplicate rows: {test_exact_duplicates}")
print(
    "Training duplicate engine-cycle keys: "
    f"{train_duplicate_engine_cycles}"
)
print(
    "Test duplicate engine-cycle keys: "
    f"{test_duplicate_engine_cycles}"
)
print(f"Training infinite values: {train_non_finite_count}")
print(f"Test infinite values: {test_non_finite_count}")
print(f"Test-RUL infinite values: {rul_non_finite_count}")

print("\nCONSTANT AND NEAR-CONSTANT FINDINGS")
print("=" * 75)
print(
    "Training constant columns: "
    f"{train_constant_columns or 'None'}"
)
print(
    "Training near-constant columns: "
    f"{train_near_constant_columns or 'None'}"
)
print(
    "Test constant columns: "
    f"{test_constant_columns or 'None'}"
)
print(
    "Test near-constant columns: "
    f"{test_near_constant_columns or 'None'}"
)

print("\nQUALITY CHECKS")
print("=" * 75)

for check_name, passed in quality_checks.items():
    status = "PASS" if passed else "FAIL"
    readable_name = check_name.replace("_", " ")

    print(f"{status} — {readable_name}")


# ------------------------------------------------------------
# 10. Package the Phase 1 evidence
# ------------------------------------------------------------

evidence_files = [
    LOGS_DIR / "environment.json",
    LOGS_DIR / "fd001_provenance.json",
    LOGS_DIR / "fd001_file_manifest.csv",
    LOGS_DIR / "fd001_structural_validation.json",
    quality_report_file,
    feature_profile_file
]

phase_1_evidence_zip = (
    PROJECT_ROOT
    / "Phase1_FD001_Data_Governance_Evidence.zip"
)

with zipfile.ZipFile(
    phase_1_evidence_zip,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as zip_file:

    for evidence_file in evidence_files:
        if evidence_file.exists():
            zip_file.write(
                evidence_file,
                arcname=evidence_file.name
            )


# ------------------------------------------------------------
# 11. Final verification
# ------------------------------------------------------------

print("\nREPORT FILES")
print("=" * 75)
print(f"Quality report: {quality_report_file}")
print(f"Feature profile: {feature_profile_file}")
print(f"Evidence ZIP: {phase_1_evidence_zip}")

print("\nFINAL CHECK")
print("=" * 75)

if all(quality_checks.values()):
    print("PASS — FD001 data-quality validation completed.")
    print("PASS — Engine-cycle integrity was verified.")
    print("PASS — No missing, duplicate or infinite records were found.")
    print("PASS — No raw data were changed.")

    print("\nDownloading the Phase 1 evidence ZIP...")
    colab_files.download(
        str(phase_1_evidence_zip)
    )

else:
    print("CHECK REQUIRED — At least one quality check failed.")
    print("Do not clean or modify anything before investigation.")

In [ ]:
# ============================================================
# PHASE 1 — DATASET GOVERNANCE
# STEP 4C — CREATE THE FORMAL FD001 DATA DICTIONARY
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from google.colab import files as colab_files

import pandas as pd
import json
import zipfile


# ------------------------------------------------------------
# 1. Confirm that the validated dataframes exist
# ------------------------------------------------------------

required_dataframe_names = [
    "train_df",
    "test_df",
    "test_rul_df"
]

missing_dataframes = [
    name
    for name in required_dataframe_names
    if name not in globals()
]

if missing_dataframes:
    raise RuntimeError(
        "Run Steps 4A and 4B first. Missing dataframes: "
        + ", ".join(missing_dataframes)
    )


# ------------------------------------------------------------
# 2. Define output locations
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/Predictive_Maintenance_Thesis")
LOGS_DIR = PROJECT_ROOT / "logs"
LOGS_DIR.mkdir(parents=True, exist_ok=True)

recorded_at_utc = datetime.now(timezone.utc).isoformat()


# ------------------------------------------------------------
# 3. Reconfirm constant columns without removing them
# ------------------------------------------------------------

training_constant_columns = [
    column
    for column in train_df.columns
    if (
        column not in ["unit_number", "time_in_cycles"]
        and train_df[column].nunique(dropna=False) == 1
    )
]

test_constant_columns = [
    column
    for column in test_df.columns
    if (
        column not in ["unit_number", "time_in_cycles"]
        and test_df[column].nunique(dropna=False) == 1
    )
]


# ------------------------------------------------------------
# 4. Define metadata for each type of variable
# ------------------------------------------------------------

def describe_column(column_name):
    """
    Return the academic description, role, units and planned use
    for one FD001 column.
    """

    if column_name == "unit_number":
        return {
            "variable_role": "Engine identifier",
            "description": (
                "An integer identifier distinguishing one "
                "simulated turbofan engine from another."
            ),
            "unit_or_scale": "Arbitrary integer identifier",
            "planned_model_use": (
                "Not a predictor. Used for grouping, "
                "engine-level splitting and traceability."
            )
        }

    if column_name == "time_in_cycles":
        return {
            "variable_role": "Temporal index",
            "description": (
                "The sequential operating-cycle number "
                "recorded for an engine."
            ),
            "unit_or_scale": "Operating cycles",
            "planned_model_use": (
                "Candidate predictor or temporal reference. "
                "Final treatment will be justified later."
            )
        }

    if column_name.startswith("operational_setting_"):
        setting_number = column_name.split("_")[-1]

        return {
            "variable_role": "Operational setting",
            "description": (
                f"NASA C-MAPSS simulated operational-setting "
                f"variable {setting_number}."
            ),
            "unit_or_scale": (
                "Physical unit not specified in the "
                "distributed FD001 text files."
            ),
            "planned_model_use": (
                "Candidate predictor. Any exclusion must be "
                "performed inside a leakage-safe pipeline."
            )
        }

    if column_name.startswith("sensor_"):
        sensor_number = column_name.split("_")[-1]

        return {
            "variable_role": "Sensor measurement",
            "description": (
                f"Anonymised NASA C-MAPSS sensor-measurement "
                f"channel {sensor_number}."
            ),
            "unit_or_scale": (
                "Physical sensor mapping and measurement unit "
                "are not specified in the distributed text files."
            ),
            "planned_model_use": (
                "Candidate predictor. Any exclusion must be "
                "performed inside a leakage-safe pipeline."
            )
        }

    if column_name == "rul_at_last_observed_cycle":
        return {
            "variable_role": "Test endpoint target",
            "description": (
                "Remaining Useful Life after the final "
                "observed cycle of one test engine."
            ),
            "unit_or_scale": "Remaining operating cycles",
            "planned_model_use": (
                "Evaluation target for the test engines. "
                "Never used as a predictor."
            )
        }

    raise ValueError(
        f"Unrecognised FD001 column: {column_name}"
    )


# ------------------------------------------------------------
# 5. Build the 26-column train/test dictionary
# ------------------------------------------------------------

dictionary_records = []

for column in train_df.columns:

    metadata = describe_column(column)

    dictionary_records.append({
        "column_name": column,
        "source_file": (
            "train_FD001.txt and test_FD001.txt"
        ),
        "variable_role": metadata["variable_role"],
        "description": metadata["description"],
        "unit_or_scale": metadata["unit_or_scale"],
        "training_dtype": str(train_df[column].dtype),
        "test_dtype": str(test_df[column].dtype),
        "training_unique_values": int(
            train_df[column].nunique(dropna=False)
        ),
        "test_unique_values": int(
            test_df[column].nunique(dropna=False)
        ),
        "training_missing_values": int(
            train_df[column].isna().sum()
        ),
        "test_missing_values": int(
            test_df[column].isna().sum()
        ),
        "constant_in_training": bool(
            column in training_constant_columns
        ),
        "constant_in_test": bool(
            column in test_constant_columns
        ),
        "planned_model_use":
            metadata["planned_model_use"]
    })


# ------------------------------------------------------------
# 6. Add the test-RUL target to the dictionary
# ------------------------------------------------------------

rul_column = "rul_at_last_observed_cycle"
rul_metadata = describe_column(rul_column)

dictionary_records.append({
    "column_name": rul_column,
    "source_file": "RUL_FD001.txt",
    "variable_role": rul_metadata["variable_role"],
    "description": rul_metadata["description"],
    "unit_or_scale": rul_metadata["unit_or_scale"],
    "training_dtype": "Not applicable",
    "test_dtype": str(test_rul_df[rul_column].dtype),
    "training_unique_values": "Not applicable",
    "test_unique_values": int(
        test_rul_df[rul_column].nunique(
            dropna=False
        )
    ),
    "training_missing_values": "Not applicable",
    "test_missing_values": int(
        test_rul_df[rul_column].isna().sum()
    ),
    "constant_in_training": "Not applicable",
    "constant_in_test": bool(
        test_rul_df[rul_column].nunique(
            dropna=False
        ) == 1
    ),
    "planned_model_use":
        rul_metadata["planned_model_use"]
})


# ------------------------------------------------------------
# 7. Create and save the data dictionary
# ------------------------------------------------------------

data_dictionary_df = pd.DataFrame(
    dictionary_records
)

data_dictionary_csv = (
    LOGS_DIR / "fd001_data_dictionary.csv"
)

data_dictionary_df.to_csv(
    data_dictionary_csv,
    index=False
)


# ------------------------------------------------------------
# 8. Create a human-readable Markdown version
# ------------------------------------------------------------

data_dictionary_markdown = (
    LOGS_DIR / "fd001_data_dictionary.md"
)

markdown_lines = [
    "# NASA C-MAPSS FD001 Data Dictionary",
    "",
    f"Generated at UTC: {recorded_at_utc}",
    "",
    "This dictionary documents the 26 columns in the FD001 "
    "training and test files and the endpoint RUL variable.",
    "",
    "The sensor channels remain anonymised. Physical sensor "
    "names or units have not been invented.",
    "",
    "| Column | Role | Unit or scale | Constant in training | "
    "Planned use |",
    "|---|---|---|---:|---|"
]

for record in dictionary_records:

    safe_unit = str(
        record["unit_or_scale"]
    ).replace("|", "/")

    safe_use = str(
        record["planned_model_use"]
    ).replace("|", "/")

    markdown_lines.append(
        f"| {record['column_name']} "
        f"| {record['variable_role']} "
        f"| {safe_unit} "
        f"| {record['constant_in_training']} "
        f"| {safe_use} |"
    )

data_dictionary_markdown.write_text(
    "\n".join(markdown_lines),
    encoding="utf-8"
)


# ------------------------------------------------------------
# 9. Create the Phase 1 technical-completion record
# ------------------------------------------------------------

phase_1_completion_record = {
    "recorded_at_utc": recorded_at_utc,
    "phase": "Phase 1 — Dataset Governance",
    "technical_status": (
        "All planned technical requirements completed"
    ),
    "formal_approval_status": (
        "Awaiting Ranitabh's Step 4C approval"
    ),
    "completed_components": [
        "Temporary Colab workspace created",
        "Software environment recorded",
        "FD001-only file inventory validated",
        "Raw files copied into protected boundary",
        "SHA-256 fingerprints calculated",
        "Source and protected-copy hashes matched",
        "Provenance record created",
        "Canonical dataset dimensions validated",
        "Engine counts validated",
        "Missing-value audit completed",
        "Duplicate audit completed",
        "Engine-cycle sequence integrity validated",
        "Constant-feature findings documented",
        "Formal data dictionary created"
    ],
    "locked_dataset_scope": "NASA C-MAPSS FD001 only",
    "raw_data_modified": False,
    "preprocessing_performed": False,
    "rul_target_constructed": False,
    "model_trained": False,
    "next_planned_phase": (
        "Phase 2 — Exploratory Data Analysis and "
        "RUL target design"
    ),
    "governance_rules_carried_forward": [
        "Never edit raw files in place",
        "Never use unit_number as a predictive feature",
        "Use engine-level data separation",
        "Do not use the final test set to fit preprocessing",
        "Do not remove constant features before a documented decision",
        "Do not invent physical sensor meanings or units",
        "Preserve reproducibility and academic-integrity evidence"
    ]
}

phase_1_completion_file = (
    LOGS_DIR / "phase_1_technical_completion_record.json"
)

with phase_1_completion_file.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        phase_1_completion_record,
        file,
        indent=4
    )


# ------------------------------------------------------------
# 10. Validate the dictionary
# ------------------------------------------------------------

role_counts = data_dictionary_df[
    "variable_role"
].value_counts().to_dict()

dictionary_checks = {
    "dictionary_contains_27_documented_variables":
        len(data_dictionary_df) == 27,

    "one_engine_identifier_is_documented":
        role_counts.get("Engine identifier", 0) == 1,

    "one_temporal_index_is_documented":
        role_counts.get("Temporal index", 0) == 1,

    "three_operational_settings_are_documented":
        role_counts.get("Operational setting", 0) == 3,

    "twenty_one_sensor_channels_are_documented":
        role_counts.get("Sensor measurement", 0) == 21,

    "one_test_endpoint_target_is_documented":
        role_counts.get("Test endpoint target", 0) == 1,

    "all_training_missing_counts_remain_zero":
        int(train_df.isna().sum().sum()) == 0,

    "all_test_missing_counts_remain_zero":
        int(test_df.isna().sum().sum()) == 0,

    "raw_data_were_not_modified":
        True
}


# ------------------------------------------------------------
# 11. Package the final Step 4C evidence
# ------------------------------------------------------------

evidence_files = [
    LOGS_DIR / "environment.json",
    LOGS_DIR / "fd001_provenance.json",
    LOGS_DIR / "fd001_file_manifest.csv",
    LOGS_DIR / "fd001_structural_validation.json",
    LOGS_DIR / "fd001_data_quality_report.json",
    LOGS_DIR / "fd001_feature_profile.csv",
    data_dictionary_csv,
    data_dictionary_markdown,
    phase_1_completion_file
]

step_4c_evidence_zip = (
    PROJECT_ROOT
    / "Phase1_Step4C_Data_Dictionary_Evidence.zip"
)

with zipfile.ZipFile(
    step_4c_evidence_zip,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as zip_file:

    for evidence_file in evidence_files:
        if evidence_file.exists():
            zip_file.write(
                evidence_file,
                arcname=evidence_file.name
            )


# ------------------------------------------------------------
# 12. Display the results
# ------------------------------------------------------------

print("FD001 DATA DICTIONARY SUMMARY")
print("=" * 75)
print(f"Documented variables: {len(data_dictionary_df)}")
print(f"Engine identifiers: {role_counts.get('Engine identifier', 0)}")
print(f"Temporal indices: {role_counts.get('Temporal index', 0)}")
print(f"Operational settings: {role_counts.get('Operational setting', 0)}")
print(f"Sensor measurements: {role_counts.get('Sensor measurement', 0)}")
print(f"Test endpoint targets: {role_counts.get('Test endpoint target', 0)}")

print("\nCONSTANT-FEATURE ANNOTATION")
print("=" * 75)
print(
    "Documented constant training features: "
    f"{training_constant_columns}"
)

print("\nDICTIONARY VALIDATION")
print("=" * 75)

for check_name, passed in dictionary_checks.items():
    status = "PASS" if passed else "FAIL"
    readable_name = check_name.replace("_", " ")

    print(f"{status} — {readable_name}")

print("\nOUTPUT FILES")
print("=" * 75)
print(f"CSV dictionary: {data_dictionary_csv}")
print(f"Markdown dictionary: {data_dictionary_markdown}")
print(f"Completion record: {phase_1_completion_file}")
print(f"Evidence ZIP: {step_4c_evidence_zip}")

print("\nFINAL CHECK")
print("=" * 75)

if all(dictionary_checks.values()):
    print("PASS — The formal FD001 data dictionary was created.")
    print("PASS — All 27 variables were documented correctly.")
    print("PASS — Unknown physical sensor meanings were not invented.")
    print("PASS — Phase 1 technical requirements are complete.")
    print("AWAITING APPROVAL — Phase 2 has not started.")

    print("\nDownloading the Step 4C evidence ZIP...")
    colab_files.download(
        str(step_4c_evidence_zip)
    )

else:
    print("CHECK REQUIRED — Data-dictionary validation failed.")


In [ ]:
# ============================================================
# PHASE 2 — EXPLORATORY DATA ANALYSIS
# STEP 1 — ENGINE-SEQUENCE OVERVIEW AND EDA PLAN
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from google.colab import files as colab_files

import pandas as pd
import json
import zipfile


# ------------------------------------------------------------
# 1. Confirm that Phase 1 dataframes are available
# ------------------------------------------------------------

required_dataframe_names = [
    "train_df",
    "test_df",
    "test_rul_df"
]

missing_dataframes = [
    name
    for name in required_dataframe_names
    if name not in globals()
]

if missing_dataframes:
    raise RuntimeError(
        "Required validated dataframes are missing: "
        + ", ".join(missing_dataframes)
    )


# ------------------------------------------------------------
# 2. Define output locations
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/Predictive_Maintenance_Thesis")
RESULTS_DIR = PROJECT_ROOT / "results"
LOGS_DIR = PROJECT_ROOT / "logs"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

recorded_at_utc = datetime.now(timezone.utc).isoformat()


# ------------------------------------------------------------
# 3. Record the formal Phase 1 closure
# ------------------------------------------------------------

phase_1_closure_record = {
    "recorded_at_utc": recorded_at_utc,
    "phase": "Phase 1 — Dataset Governance",
    "status": "Approved and closed",
    "approval_basis": (
        "Ranitabh reviewed the successful Step 4C output, "
        "confirmed the evidence download and authorised continuation."
    ),
    "raw_data_modified": False,
    "phase_2_previously_started": False,
    "governance_rules_carried_forward": [
        "FD001 only",
        "Raw files remain unchanged",
        "Engine-level separation is mandatory",
        "unit_number is never a predictive feature",
        "Final test outcomes must not guide model development",
        "Physical sensor meanings or units must not be invented"
    ]
}

phase_1_closure_file = (
    LOGS_DIR / "phase_1_closure_record.json"
)

with phase_1_closure_file.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        phase_1_closure_record,
        file,
        indent=4
    )


# ------------------------------------------------------------
# 4. Create one summary row for every training engine
# ------------------------------------------------------------

train_engine_summary = (
    train_df
    .groupby("unit_number", as_index=False)
    .agg(
        first_observed_cycle=(
            "time_in_cycles",
            "min"
        ),
        final_observed_cycle=(
            "time_in_cycles",
            "max"
        ),
        observed_rows=(
            "time_in_cycles",
            "size"
        )
    )
)

train_engine_summary.insert(
    0,
    "dataset_partition",
    "training"
)

train_engine_summary[
    "trajectory_interpretation"
] = "Run-to-failure trajectory"

train_engine_summary[
    "actual_lifetime_observed"
] = True


# ------------------------------------------------------------
# 5. Create one summary row for every test engine
# ------------------------------------------------------------

test_engine_summary = (
    test_df
    .groupby("unit_number", as_index=False)
    .agg(
        first_observed_cycle=(
            "time_in_cycles",
            "min"
        ),
        final_observed_cycle=(
            "time_in_cycles",
            "max"
        ),
        observed_rows=(
            "time_in_cycles",
            "size"
        )
    )
)

test_engine_summary.insert(
    0,
    "dataset_partition",
    "test"
)

test_engine_summary[
    "trajectory_interpretation"
] = "Right-truncated trajectory"

test_engine_summary[
    "actual_lifetime_observed"
] = False


# ------------------------------------------------------------
# 6. Combine the engine-level sequence summaries
# ------------------------------------------------------------

engine_sequence_summary = pd.concat(
    [
        train_engine_summary,
        test_engine_summary
    ],
    ignore_index=True
)

engine_summary_file = (
    RESULTS_DIR
    / "phase2_engine_sequence_summary.csv"
)

engine_sequence_summary.to_csv(
    engine_summary_file,
    index=False
)


# ------------------------------------------------------------
# 7. Produce descriptive statistics
# ------------------------------------------------------------

def endpoint_statistics(engine_dataframe):
    """
    Describe final observed cycles without constructing RUL.
    """

    values = engine_dataframe[
        "final_observed_cycle"
    ]

    return {
        "engine_count": int(len(values)),
        "minimum": int(values.min()),
        "first_quartile": float(
            values.quantile(0.25)
        ),
        "median": float(values.median()),
        "mean": float(values.mean()),
        "third_quartile": float(
            values.quantile(0.75)
        ),
        "maximum": int(values.max()),
        "standard_deviation": float(values.std())
    }


training_lifetime_statistics = (
    endpoint_statistics(train_engine_summary)
)

test_observation_statistics = (
    endpoint_statistics(test_engine_summary)
)


# ------------------------------------------------------------
# 8. Define the controlled EDA questions
# ------------------------------------------------------------

eda_questions = [
    {
        "question_id": "EDA-Q1",
        "question": (
            "How much do observed run-to-failure lifetimes "
            "vary across training engines?"
        ),
        "permitted_scope": "Training engines"
    },
    {
        "question_id": "EDA-Q2",
        "question": (
            "How are observations distributed across early, "
            "middle and late engine-life stages?"
        ),
        "permitted_scope": (
            "Training data after the RUL definition is approved"
        )
    },
    {
        "question_id": "EDA-Q3",
        "question": (
            "Which sensor channels show visible degradation "
            "patterns over operating cycles?"
        ),
        "permitted_scope": (
            "Training data; no physical sensor meaning inferred"
        )
    },
    {
        "question_id": "EDA-Q4",
        "question": (
            "Which features are constant, weakly varying "
            "or potentially redundant?"
        ),
        "permitted_scope": (
            "Descriptive reporting first; removal decisions "
            "must remain leakage-safe"
        )
    },
    {
        "question_id": "EDA-Q5",
        "question": (
            "Do selected sensor distributions differ across "
            "engines and life stages?"
        ),
        "permitted_scope": "Training data"
    },
    {
        "question_id": "EDA-Q6",
        "question": (
            "How might uncapped and capped RUL definitions "
            "change the target distribution?"
        ),
        "permitted_scope": (
            "Only after engine partitions and target candidates "
            "are formally defined"
        )
    },
    {
        "question_id": "EDA-Q7",
        "question": (
            "Are the future development partitions reasonably "
            "balanced in engine lifetime?"
        ),
        "permitted_scope": (
            "After reproducible engine-level splitting"
        )
    }
]


# ------------------------------------------------------------
# 9. Save the EDA governance plan
# ------------------------------------------------------------

eda_plan = {
    "recorded_at_utc": recorded_at_utc,
    "phase": "Phase 2 — Exploratory Data Analysis",
    "step": "Step 1 — Engine-sequence overview and EDA plan",
    "training_endpoint_interpretation": (
        "The maximum cycle of each training engine is its "
        "observed simulated failure cycle."
    ),
    "test_endpoint_interpretation": (
        "The maximum cycle of each test engine is only its "
        "last observed cycle, not its complete lifetime."
    ),
    "training_lifetime_statistics":
        training_lifetime_statistics,
    "test_observation_statistics":
        test_observation_statistics,
    "eda_questions": eda_questions,
    "test_rul_values_used": False,
    "training_rul_constructed": False,
    "charts_generated": False,
    "features_removed": False,
    "models_trained": False
}

eda_plan_file = (
    LOGS_DIR / "phase2_eda_plan.json"
)

with eda_plan_file.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        eda_plan,
        file,
        indent=4
    )


# ------------------------------------------------------------
# 10. Validate this step
# ------------------------------------------------------------

phase_2_step_1_checks = {
    "training_summary_contains_100_engines":
        len(train_engine_summary) == 100,

    "test_summary_contains_100_engines":
        len(test_engine_summary) == 100,

    "all_training_engines_begin_at_cycle_one":
        bool(
            (
                train_engine_summary[
                    "first_observed_cycle"
                ] == 1
            ).all()
        ),

    "all_test_engines_begin_at_cycle_one":
        bool(
            (
                test_engine_summary[
                    "first_observed_cycle"
                ] == 1
            ).all()
        ),

    "training_row_counts_match_final_cycles":
        bool(
            (
                train_engine_summary["observed_rows"]
                == train_engine_summary[
                    "final_observed_cycle"
                ]
            ).all()
        ),

    "test_row_counts_match_final_observed_cycles":
        bool(
            (
                test_engine_summary["observed_rows"]
                == test_engine_summary[
                    "final_observed_cycle"
                ]
            ).all()
        ),

    "seven_eda_questions_are_documented":
        len(eda_questions) == 7,

    "test_rul_values_were_not_used":
        eda_plan["test_rul_values_used"] is False,

    "training_rul_was_not_constructed":
        eda_plan["training_rul_constructed"] is False
}


# ------------------------------------------------------------
# 11. Package the evidence
# ------------------------------------------------------------

step_1_evidence_zip = (
    PROJECT_ROOT
    / "Phase2_Step1_Engine_Sequence_Evidence.zip"
)

with zipfile.ZipFile(
    step_1_evidence_zip,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as zip_file:

    zip_file.write(
        phase_1_closure_file,
        arcname=phase_1_closure_file.name
    )

    zip_file.write(
        engine_summary_file,
        arcname=engine_summary_file.name
    )

    zip_file.write(
        eda_plan_file,
        arcname=eda_plan_file.name
    )


# ------------------------------------------------------------
# 12. Display the results
# ------------------------------------------------------------

print("TRAINING ENGINE LIFETIME SUMMARY")
print("=" * 75)

for name, value in training_lifetime_statistics.items():
    print(f"{name}: {value}")

print("\nTEST ENGINE OBSERVATION SUMMARY")
print("=" * 75)

for name, value in test_observation_statistics.items():
    print(f"{name}: {value}")

print("\nIMPORTANT INTERPRETATION")
print("=" * 75)
print(
    "Training final cycle = observed simulated lifetime."
)
print(
    "Test final observed cycle does NOT equal actual lifetime."
)
print(
    "The test RUL values were not accessed in this step."
)

print("\nSTEP VALIDATION")
print("=" * 75)

for check_name, passed in phase_2_step_1_checks.items():
    status = "PASS" if passed else "FAIL"
    readable_name = check_name.replace("_", " ")

    print(f"{status} — {readable_name}")

print("\nOUTPUT FILES")
print("=" * 75)
print(f"Engine summary: {engine_summary_file}")
print(f"EDA plan: {eda_plan_file}")
print(f"Evidence ZIP: {step_1_evidence_zip}")

print("\nFINAL CHECK")
print("=" * 75)

if all(phase_2_step_1_checks.values()):
    print("PASS — Phase 2 Step 1 completed successfully.")
    print("PASS — Engine sequences were summarised correctly.")
    print("PASS — No test outcomes or RUL targets were used.")
    print("PASS — No charts, preprocessing or modelling occurred.")

    print("\nDownloading the Step 1 evidence ZIP...")
    colab_files.download(
        str(step_1_evidence_zip)
    )

else:
    print("CHECK REQUIRED — At least one validation failed.")

In [ ]:
# ============================================================
# PHASE 2 — EXPLORATORY DATA ANALYSIS
# STEP 2 — FREEZE LEAKAGE-SAFE ENGINE PARTITIONS
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from google.colab import files as colab_files

import pandas as pd
import numpy as np
import json
import zipfile


# ------------------------------------------------------------
# 1. Confirm that the validated training data exist
# ------------------------------------------------------------

if "train_df" not in globals():
    raise RuntimeError(
        "The validated training dataframe is unavailable. "
        "Run the earlier Phase 1 loading cells first."
    )


# ------------------------------------------------------------
# 2. Define controlled output locations
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/Predictive_Maintenance_Thesis")
INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"
LOGS_DIR = PROJECT_ROOT / "logs"

INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

recorded_at_utc = datetime.now(timezone.utc).isoformat()


# ------------------------------------------------------------
# 3. Define and verify the available training engines
# ------------------------------------------------------------

all_training_engine_ids = np.sort(
    train_df["unit_number"].unique()
)

if len(all_training_engine_ids) != 100:
    raise ValueError(
        "Expected 100 training engines, but found "
        f"{len(all_training_engine_ids)}."
    )


# ------------------------------------------------------------
# 4. Freeze the reproducible partition policy
# ------------------------------------------------------------

RANDOM_SEED = 42

MODEL_TRAIN_ENGINE_COUNT = 60
VALIDATION_ENGINE_COUNT = 20
CALIBRATION_ENGINE_COUNT = 20

random_generator = np.random.default_rng(
    RANDOM_SEED
)

# This permutation is created exactly once.
shuffled_engine_ids = random_generator.permutation(
    all_training_engine_ids
)

MODEL_TRAIN_ENGINE_IDS = np.sort(
    shuffled_engine_ids[
        :MODEL_TRAIN_ENGINE_COUNT
    ]
)

VALIDATION_ENGINE_IDS = np.sort(
    shuffled_engine_ids[
        MODEL_TRAIN_ENGINE_COUNT:
        MODEL_TRAIN_ENGINE_COUNT
        + VALIDATION_ENGINE_COUNT
    ]
)

CALIBRATION_ENGINE_IDS = np.sort(
    shuffled_engine_ids[
        MODEL_TRAIN_ENGINE_COUNT
        + VALIDATION_ENGINE_COUNT:
    ]
)


# ------------------------------------------------------------
# 5. Create partition membership sets
# ------------------------------------------------------------

model_train_set = set(
    MODEL_TRAIN_ENGINE_IDS.tolist()
)

validation_set = set(
    VALIDATION_ENGINE_IDS.tolist()
)

calibration_set = set(
    CALIBRATION_ENGINE_IDS.tolist()
)

all_engine_set = set(
    all_training_engine_ids.tolist()
)


# ------------------------------------------------------------
# 6. Check for engine overlap
# ------------------------------------------------------------

train_validation_overlap = (
    model_train_set & validation_set
)

train_calibration_overlap = (
    model_train_set & calibration_set
)

validation_calibration_overlap = (
    validation_set & calibration_set
)

combined_partition_engines = (
    model_train_set
    | validation_set
    | calibration_set
)


# ------------------------------------------------------------
# 7. Create the engine-partition manifest
# ------------------------------------------------------------

engine_lifetimes = (
    train_df
    .groupby(
        "unit_number",
        as_index=False
    )
    .agg(
        observed_lifetime_cycles=(
            "time_in_cycles",
            "max"
        ),
        observed_rows=(
            "time_in_cycles",
            "size"
        )
    )
)

partition_assignments = {}

for engine_id in MODEL_TRAIN_ENGINE_IDS:
    partition_assignments[
        int(engine_id)
    ] = "model_training"

for engine_id in VALIDATION_ENGINE_IDS:
    partition_assignments[
        int(engine_id)
    ] = "validation"

for engine_id in CALIBRATION_ENGINE_IDS:
    partition_assignments[
        int(engine_id)
    ] = "conformal_calibration"

engine_partition_manifest = (
    engine_lifetimes.copy()
)

engine_partition_manifest[
    "partition"
] = engine_partition_manifest[
    "unit_number"
].map(partition_assignments)

engine_partition_manifest[
    "source_file"
] = "train_FD001.txt"

engine_partition_manifest[
    "random_seed"
] = RANDOM_SEED

engine_partition_manifest[
    "assignment_method"
] = (
    "Fixed engine-level random permutation; "
    "no stratification and no rerolling"
)

engine_partition_manifest = (
    engine_partition_manifest[
        [
            "source_file",
            "unit_number",
            "partition",
            "observed_lifetime_cycles",
            "observed_rows",
            "random_seed",
            "assignment_method"
        ]
    ]
    .sort_values("unit_number")
    .reset_index(drop=True)
)

partition_manifest_file = (
    INTERIM_DATA_DIR
    / "fd001_engine_partition_manifest.csv"
)

engine_partition_manifest.to_csv(
    partition_manifest_file,
    index=False
)


# ------------------------------------------------------------
# 8. Create in-memory partitioned dataframes
# ------------------------------------------------------------

# These are derived views.
# The protected raw files remain unchanged.

model_train_df = train_df[
    train_df["unit_number"].isin(
        MODEL_TRAIN_ENGINE_IDS
    )
].copy()

validation_df = train_df[
    train_df["unit_number"].isin(
        VALIDATION_ENGINE_IDS
    )
].copy()

calibration_df = train_df[
    train_df["unit_number"].isin(
        CALIBRATION_ENGINE_IDS
    )
].copy()


# ------------------------------------------------------------
# 9. Create an audit summary
# ------------------------------------------------------------

partition_audit_summary = (
    engine_partition_manifest
    .groupby("partition")
    .agg(
        engine_count=(
            "unit_number",
            "nunique"
        ),
        total_cycle_rows=(
            "observed_rows",
            "sum"
        ),
        minimum_lifetime=(
            "observed_lifetime_cycles",
            "min"
        ),
        median_lifetime=(
            "observed_lifetime_cycles",
            "median"
        ),
        mean_lifetime=(
            "observed_lifetime_cycles",
            "mean"
        ),
        maximum_lifetime=(
            "observed_lifetime_cycles",
            "max"
        )
    )
    .reset_index()
)

partition_audit_file = (
    LOGS_DIR / "fd001_partition_audit.csv"
)

partition_audit_summary.to_csv(
    partition_audit_file,
    index=False
)


# ------------------------------------------------------------
# 10. Validate the frozen partitions
# ------------------------------------------------------------

partition_checks = {
    "model_training_contains_60_engines":
        len(model_train_set) == 60,

    "validation_contains_20_engines":
        len(validation_set) == 20,

    "calibration_contains_20_engines":
        len(calibration_set) == 20,

    "model_training_and_validation_do_not_overlap":
        len(train_validation_overlap) == 0,

    "model_training_and_calibration_do_not_overlap":
        len(train_calibration_overlap) == 0,

    "validation_and_calibration_do_not_overlap":
        len(validation_calibration_overlap) == 0,

    "all_100_training_engines_are_assigned":
        combined_partition_engines == all_engine_set,

    "partition_manifest_contains_100_engines":
        len(engine_partition_manifest) == 100,

    "partition_manifest_has_no_missing_assignments":
        bool(
            engine_partition_manifest[
                "partition"
            ].notna().all()
        ),

    "all_model_training_rows_belong_to_training_engines":
        set(
            model_train_df[
                "unit_number"
            ].unique()
        ) == model_train_set,

    "all_validation_rows_belong_to_validation_engines":
        set(
            validation_df[
                "unit_number"
            ].unique()
        ) == validation_set,

    "all_calibration_rows_belong_to_calibration_engines":
        set(
            calibration_df[
                "unit_number"
            ].unique()
        ) == calibration_set,

    "row_counts_are_fully_preserved":
        (
            len(model_train_df)
            + len(validation_df)
            + len(calibration_df)
        ) == len(train_df)
}


# ------------------------------------------------------------
# 11. Save the partition policy
# ------------------------------------------------------------

partition_policy = {
    "recorded_at_utc": recorded_at_utc,
    "dataset": "NASA C-MAPSS FD001",
    "source_partition": "train_FD001.txt only",
    "method": (
        "Fixed engine-level random permutation"
    ),
    "random_seed": RANDOM_SEED,
    "lifetime_stratification_used": False,
    "rerolling_permitted": False,
    "model_training_engine_count":
        len(model_train_set),
    "validation_engine_count":
        len(validation_set),
    "conformal_calibration_engine_count":
        len(calibration_set),
    "official_test_set_role": (
        "Untouched final evaluation dataset"
    ),
    "model_training_engine_ids": [
        int(value)
        for value in MODEL_TRAIN_ENGINE_IDS
    ],
    "validation_engine_ids": [
        int(value)
        for value in VALIDATION_ENGINE_IDS
    ],
    "conformal_calibration_engine_ids": [
        int(value)
        for value in CALIBRATION_ENGINE_IDS
    ],
    "partition_checks": partition_checks,
    "important_conformal_note": (
        "Creating a separate calibration partition does not "
        "by itself resolve the dependence between cycles from "
        "the same engine. The conformal calibration unit and "
        "coverage interpretation must be justified later."
    ),
    "rul_target_constructed": False,
    "test_rul_values_used": False,
    "models_trained": False
}

partition_policy_file = (
    LOGS_DIR / "fd001_engine_partition_policy.json"
)

with partition_policy_file.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        partition_policy,
        file,
        indent=4
    )


# ------------------------------------------------------------
# 12. Package the evidence
# ------------------------------------------------------------

step_2_evidence_zip = (
    PROJECT_ROOT
    / "Phase2_Step2_Engine_Partition_Evidence.zip"
)

with zipfile.ZipFile(
    step_2_evidence_zip,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as zip_file:

    zip_file.write(
        partition_manifest_file,
        arcname=partition_manifest_file.name
    )

    zip_file.write(
        partition_audit_file,
        arcname=partition_audit_file.name
    )

    zip_file.write(
        partition_policy_file,
        arcname=partition_policy_file.name
    )


# ------------------------------------------------------------
# 13. Display the results
# ------------------------------------------------------------

print("FROZEN ENGINE PARTITIONS")
print("=" * 75)
print(
    "Model-training engine IDs:\n"
    f"{MODEL_TRAIN_ENGINE_IDS.tolist()}"
)
print(
    "\nValidation engine IDs:\n"
    f"{VALIDATION_ENGINE_IDS.tolist()}"
)
print(
    "\nConformal-calibration engine IDs:\n"
    f"{CALIBRATION_ENGINE_IDS.tolist()}"
)

print("\nPARTITION AUDIT")
print("=" * 75)
print(
    partition_audit_summary.to_string(
        index=False
    )
)

print("\nPARTITION VALIDATION")
print("=" * 75)

for check_name, passed in partition_checks.items():
    status = "PASS" if passed else "FAIL"
    readable_name = check_name.replace("_", " ")

    print(f"{status} — {readable_name}")

print("\nIMPORTANT CONTROLS")
print("=" * 75)
print("Random seed: 42")
print("Engine-level assignment: Yes")
print("Lifetime stratification: No")
print("Rerolling permitted: No")
print("Official test data used: No")
print("RUL target constructed: No")

print("\nOUTPUT FILES")
print("=" * 75)
print(f"Partition manifest: {partition_manifest_file}")
print(f"Partition audit: {partition_audit_file}")
print(f"Partition policy: {partition_policy_file}")
print(f"Evidence ZIP: {step_2_evidence_zip}")

print("\nFINAL CHECK")
print("=" * 75)

if all(partition_checks.values()):
    print("PASS — Engine partitions were frozen successfully.")
    print("PASS — No engine appears in multiple partitions.")
    print("PASS — Every training row was preserved.")
    print("PASS — Official test data remain untouched.")
    print("PASS — No RUL target or model was created.")

    print("\nDownloading the partition evidence ZIP...")
    colab_files.download(
        str(step_2_evidence_zip)
    )

else:
    print("CHECK REQUIRED — Partition validation failed.")
    print("Do not continue to RUL construction.")

In [ ]:
# ============================================================
# PHASE 2 — RUL TARGET DESIGN
# STEP 3A — CONSTRUCT AND VALIDATE UNCAPPPED TRAINING RUL
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from google.colab import files as colab_files

import pandas as pd
import numpy as np
import json
import zipfile


# ------------------------------------------------------------
# 1. Confirm that the frozen partitions exist
# ------------------------------------------------------------

required_partition_names = [
    "model_train_df",
    "validation_df",
    "calibration_df"
]

missing_partitions = [
    name
    for name in required_partition_names
    if name not in globals()
]

if missing_partitions:
    raise RuntimeError(
        "Run the frozen engine-partition cell first. "
        "Missing partitions: "
        + ", ".join(missing_partitions)
    )


# ------------------------------------------------------------
# 2. Define output locations
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/Predictive_Maintenance_Thesis")
INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"
LOGS_DIR = PROJECT_ROOT / "logs"

INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

recorded_at_utc = datetime.now(timezone.utc).isoformat()


# ------------------------------------------------------------
# 3. Define the uncapped RUL-construction function
# ------------------------------------------------------------

def construct_uncapped_rul(partition_dataframe):
    """
    Create a derived copy and calculate:

        RUL = engine failure cycle - current cycle

    The input dataframe and protected raw files are not changed.
    """

    labelled_dataframe = partition_dataframe.copy()

    engine_failure_cycles = (
        labelled_dataframe
        .groupby("unit_number")[
            "time_in_cycles"
        ]
        .transform("max")
    )

    labelled_dataframe[
        "rul_uncapped"
    ] = (
        engine_failure_cycles
        - labelled_dataframe["time_in_cycles"]
    ).astype(int)

    return labelled_dataframe


# ------------------------------------------------------------
# 4. Construct RUL separately within each frozen partition
# ------------------------------------------------------------

model_train_rul_df = construct_uncapped_rul(
    model_train_df
)

validation_rul_df = construct_uncapped_rul(
    validation_df
)

calibration_rul_df = construct_uncapped_rul(
    calibration_df
)


# ------------------------------------------------------------
# 5. Define detailed RUL validation
# ------------------------------------------------------------

def validate_uncapped_rul(
    original_dataframe,
    labelled_dataframe
):
    """
    Verify formula correctness and sequence behaviour.
    """

    expected_rul = (
        labelled_dataframe
        .groupby("unit_number")[
            "time_in_cycles"
        ]
        .transform("max")
        - labelled_dataframe["time_in_cycles"]
    ).astype(int)

    formula_is_correct = bool(
        labelled_dataframe[
            "rul_uncapped"
        ].equals(expected_rul)
    )

    exactly_one_zero_per_engine = bool(
        (
            labelled_dataframe
            .assign(
                is_failure_row=(
                    labelled_dataframe[
                        "rul_uncapped"
                    ] == 0
                )
            )
            .groupby("unit_number")[
                "is_failure_row"
            ]
            .sum()
            == 1
        ).all()
    )

    decreases_by_one_each_cycle = True

    for _, engine_data in labelled_dataframe.groupby(
        "unit_number",
        sort=True
    ):
        rul_differences = (
            engine_data["rul_uncapped"]
            .diff()
            .dropna()
        )

        if not bool(
            (rul_differences == -1).all()
        ):
            decreases_by_one_each_cycle = False
            break

    original_columns_preserved = bool(
        original_dataframe.columns.tolist()
        == labelled_dataframe.columns[
            :-1
        ].tolist()
    )

    original_dataframe_unchanged = bool(
        "rul_uncapped"
        not in original_dataframe.columns
    )

    return {
        "row_count_preserved":
            len(original_dataframe)
            == len(labelled_dataframe),

        "formula_is_correct":
            formula_is_correct,

        "rul_contains_no_missing_values":
            int(
                labelled_dataframe[
                    "rul_uncapped"
                ].isna().sum()
            ) == 0,

        "rul_values_are_non_negative":
            bool(
                (
                    labelled_dataframe[
                        "rul_uncapped"
                    ] >= 0
                ).all()
            ),

        "rul_values_are_integers":
            bool(
                pd.api.types.is_integer_dtype(
                    labelled_dataframe[
                        "rul_uncapped"
                    ]
                )
            ),

        "exactly_one_zero_rul_row_per_engine":
            exactly_one_zero_per_engine,

        "rul_decreases_by_one_per_cycle":
            decreases_by_one_each_cycle,

        "original_columns_remain_in_same_order":
            original_columns_preserved,

        "original_partition_was_not_modified":
            original_dataframe_unchanged
    }


model_train_rul_checks = validate_uncapped_rul(
    model_train_df,
    model_train_rul_df
)

validation_rul_checks = validate_uncapped_rul(
    validation_df,
    validation_rul_df
)

calibration_rul_checks = validate_uncapped_rul(
    calibration_df,
    calibration_rul_df
)


# ------------------------------------------------------------
# 6. Save the derived labelled partitions
# ------------------------------------------------------------

model_train_rul_file = (
    INTERIM_DATA_DIR
    / "fd001_model_training_uncapped_rul.csv"
)

validation_rul_file = (
    INTERIM_DATA_DIR
    / "fd001_validation_uncapped_rul.csv"
)

calibration_rul_file = (
    INTERIM_DATA_DIR
    / "fd001_calibration_uncapped_rul.csv"
)

model_train_rul_df.to_csv(
    model_train_rul_file,
    index=False
)

validation_rul_df.to_csv(
    validation_rul_file,
    index=False
)

calibration_rul_df.to_csv(
    calibration_rul_file,
    index=False
)


# ------------------------------------------------------------
# 7. Create target-distribution summaries
# ------------------------------------------------------------

def summarise_rul(
    labelled_dataframe,
    partition_name
):
    values = labelled_dataframe["rul_uncapped"]

    return {
        "partition": partition_name,
        "engine_count": int(
            labelled_dataframe[
                "unit_number"
            ].nunique()
        ),
        "row_count": int(
            len(labelled_dataframe)
        ),
        "minimum_rul": int(values.min()),
        "first_quartile_rul": float(
            values.quantile(0.25)
        ),
        "median_rul": float(
            values.median()
        ),
        "mean_rul": float(
            values.mean()
        ),
        "third_quartile_rul": float(
            values.quantile(0.75)
        ),
        "maximum_rul": int(values.max()),
        "zero_rul_rows": int(
            (values == 0).sum()
        )
    }


rul_audit_records = [
    summarise_rul(
        model_train_rul_df,
        "model_training"
    ),
    summarise_rul(
        validation_rul_df,
        "validation"
    ),
    summarise_rul(
        calibration_rul_df,
        "conformal_calibration"
    )
]

rul_audit_df = pd.DataFrame(
    rul_audit_records
)

rul_audit_file = (
    LOGS_DIR / "fd001_uncapped_rul_audit.csv"
)

rul_audit_df.to_csv(
    rul_audit_file,
    index=False
)


# ------------------------------------------------------------
# 8. Save boundary examples for three engines
# ------------------------------------------------------------

boundary_examples = []

partition_dataframes = {
    "model_training": model_train_rul_df,
    "validation": validation_rul_df,
    "conformal_calibration": calibration_rul_df
}

for partition_name, dataframe in (
    partition_dataframes.items()
):
    example_engine_id = int(
        dataframe["unit_number"].min()
    )

    example_engine = dataframe[
        dataframe["unit_number"]
        == example_engine_id
    ]

    selected_rows = pd.concat(
        [
            example_engine.head(2),
            example_engine.tail(2)
        ]
    )

    selected_rows = selected_rows[
        [
            "unit_number",
            "time_in_cycles",
            "rul_uncapped"
        ]
    ].copy()

    selected_rows.insert(
        0,
        "partition",
        partition_name
    )

    boundary_examples.append(selected_rows)

rul_boundary_examples_df = pd.concat(
    boundary_examples,
    ignore_index=True
)

rul_boundary_examples_file = (
    LOGS_DIR
    / "fd001_uncapped_rul_boundary_examples.csv"
)

rul_boundary_examples_df.to_csv(
    rul_boundary_examples_file,
    index=False
)


# ------------------------------------------------------------
# 9. Create the target-definition record
# ------------------------------------------------------------

target_definition_record = {
    "recorded_at_utc": recorded_at_utc,
    "target_name": "rul_uncapped",
    "formula": (
        "engine_failure_cycle - time_in_cycles"
    ),
    "failure_cycle_target_value": 0,
    "unit": "remaining operating cycles",
    "source_scope": (
        "Frozen partitions derived from train_FD001.txt"
    ),
    "model_training_checks":
        model_train_rul_checks,
    "validation_checks":
        validation_rul_checks,
    "calibration_checks":
        calibration_rul_checks,
    "rul_cap_applied": False,
    "rul_cap_selected": False,
    "test_rul_file_accessed": False,
    "official_test_data_labelled": False,
    "raw_data_modified": False,
    "models_trained": False,
    "important_note": (
        "The uncapped target is the transparent physical "
        "baseline. Any later capped target must be separately "
        "justified and compared without using final test outcomes."
    )
}

target_definition_file = (
    LOGS_DIR
    / "fd001_uncapped_rul_definition.json"
)

with target_definition_file.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        target_definition_record,
        file,
        indent=4
    )


# ------------------------------------------------------------
# 10. Combine all validation checks
# ------------------------------------------------------------

all_rul_checks = {
    **{
        f"model_training_{name}": passed
        for name, passed
        in model_train_rul_checks.items()
    },
    **{
        f"validation_{name}": passed
        for name, passed
        in validation_rul_checks.items()
    },
    **{
        f"calibration_{name}": passed
        for name, passed
        in calibration_rul_checks.items()
    }
}


# ------------------------------------------------------------
# 11. Package reproducibility evidence
# ------------------------------------------------------------

step_3a_evidence_zip = (
    PROJECT_ROOT
    / "Phase2_Step3A_Uncapped_RUL_Evidence.zip"
)

with zipfile.ZipFile(
    step_3a_evidence_zip,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as zip_file:

    zip_file.write(
        rul_audit_file,
        arcname=rul_audit_file.name
    )

    zip_file.write(
        rul_boundary_examples_file,
        arcname=rul_boundary_examples_file.name
    )

    zip_file.write(
        target_definition_file,
        arcname=target_definition_file.name
    )


# ------------------------------------------------------------
# 12. Display the results
# ------------------------------------------------------------

print("UNCAPPED RUL AUDIT")
print("=" * 90)
print(
    rul_audit_df.to_string(
        index=False
    )
)

print("\nRUL BOUNDARY EXAMPLES")
print("=" * 90)
print(
    rul_boundary_examples_df.to_string(
        index=False
    )
)

print("\nRUL VALIDATION")
print("=" * 90)

for check_name, passed in all_rul_checks.items():
    status = "PASS" if passed else "FAIL"
    readable_name = check_name.replace("_", " ")

    print(f"{status} — {readable_name}")

print("\nIMPORTANT CONTROLS")
print("=" * 90)
print("Target formula: failure cycle - current cycle")
print("Failure-cycle RUL: 0")
print("RUL cap applied: No")
print("Test RUL file accessed: No")
print("Raw files modified: No")
print("Model trained: No")

print("\nOUTPUT FILES")
print("=" * 90)
print(f"Model-training labels: {model_train_rul_file}")
print(f"Validation labels: {validation_rul_file}")
print(f"Calibration labels: {calibration_rul_file}")
print(f"RUL audit: {rul_audit_file}")
print(f"Target definition: {target_definition_file}")
print(f"Evidence ZIP: {step_3a_evidence_zip}")

print("\nFINAL CHECK")
print("=" * 90)

if all(all_rul_checks.values()):
    print("PASS — Uncapped RUL was constructed correctly.")
    print("PASS — Every engine has exactly one zero-RUL failure row.")
    print("PASS — RUL decreases by one for every successive cycle.")
    print("PASS — Frozen partitions and row counts were preserved.")
    print("PASS — No cap or test outcome was used.")

    print("\nDownloading the Step 3A evidence ZIP...")
    colab_files.download(
        str(step_3a_evidence_zip)
    )

else:
    print("CHECK REQUIRED — RUL validation failed.")
    print("Do not continue to target comparison.")

In [ ]:
# ============================================================
# PHASE 2 — RUL TARGET DESIGN
# STEP 3B — EVALUATE THE 130-CYCLE CAP CANDIDATE
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from google.colab import files as colab_files

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import zipfile


# ------------------------------------------------------------
# 1. Confirm that the uncapped labelled partitions exist
# ------------------------------------------------------------

required_dataframe_names = [
    "model_train_rul_df",
    "validation_rul_df",
    "calibration_rul_df"
]

missing_dataframes = [
    name
    for name in required_dataframe_names
    if name not in globals()
]

if missing_dataframes:
    raise RuntimeError(
        "Run Step 3A first. Missing dataframes: "
        + ", ".join(missing_dataframes)
    )


# ------------------------------------------------------------
# 2. Define controlled output locations
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/Predictive_Maintenance_Thesis")
INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"
LOGS_DIR = PROJECT_ROOT / "logs"
FIGURES_DIR = PROJECT_ROOT / "figures"

for directory in [
    INTERIM_DATA_DIR,
    LOGS_DIR,
    FIGURES_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

recorded_at_utc = datetime.now(timezone.utc).isoformat()


# ------------------------------------------------------------
# 3. Define the literature-supported candidate
# ------------------------------------------------------------

CANDIDATE_RUL_CAP = 130

NASA_DATASET_REFERENCE = (
    "https://data.nasa.gov/dataset/"
    "cmapss-jet-engine-simulated-data"
)

ZHENG_2017_DOI = (
    "https://doi.org/10.1109/"
    "ICPHM.2017.7998311"
)


# ------------------------------------------------------------
# 4. Create a separate model-training comparison table
# ------------------------------------------------------------

# The approved uncapped dataframe is not modified.

model_training_target_comparison_df = (
    model_train_rul_df[
        [
            "unit_number",
            "time_in_cycles",
            "rul_uncapped"
        ]
    ]
    .copy()
)

model_training_target_comparison_df[
    "rul_capped_130_candidate"
] = np.minimum(
    model_training_target_comparison_df[
        "rul_uncapped"
    ],
    CANDIDATE_RUL_CAP
).astype(int)

model_training_target_comparison_df[
    "label_was_changed_by_cap"
] = (
    model_training_target_comparison_df[
        "rul_uncapped"
    ] > CANDIDATE_RUL_CAP
)

model_training_target_comparison_df[
    "label_reduction_cycles"
] = (
    model_training_target_comparison_df[
        "rul_uncapped"
    ]
    - model_training_target_comparison_df[
        "rul_capped_130_candidate"
    ]
)


# ------------------------------------------------------------
# 5. Quantify the cap's effect
# ------------------------------------------------------------

affected_rows = int(
    model_training_target_comparison_df[
        "label_was_changed_by_cap"
    ].sum()
)

total_rows = int(
    len(model_training_target_comparison_df)
)

affected_row_percentage = float(
    affected_rows / total_rows * 100
)

affected_engines = int(
    model_training_target_comparison_df.loc[
        model_training_target_comparison_df[
            "label_was_changed_by_cap"
        ],
        "unit_number"
    ].nunique()
)

total_engines = int(
    model_training_target_comparison_df[
        "unit_number"
    ].nunique()
)

rows_at_candidate_plateau = int(
    (
        model_training_target_comparison_df[
            "rul_capped_130_candidate"
        ] == CANDIDATE_RUL_CAP
    ).sum()
)

maximum_label_reduction = int(
    model_training_target_comparison_df[
        "label_reduction_cycles"
    ].max()
)

total_label_reduction = int(
    model_training_target_comparison_df[
        "label_reduction_cycles"
    ].sum()
)


# ------------------------------------------------------------
# 6. Create a target-comparison audit
# ------------------------------------------------------------

target_comparison_audit = pd.DataFrame([
    {
        "target_definition": "uncapped",
        "candidate_cap": "Not applicable",
        "row_count": total_rows,
        "engine_count": total_engines,
        "minimum": int(
            model_training_target_comparison_df[
                "rul_uncapped"
            ].min()
        ),
        "first_quartile": float(
            model_training_target_comparison_df[
                "rul_uncapped"
            ].quantile(0.25)
        ),
        "median": float(
            model_training_target_comparison_df[
                "rul_uncapped"
            ].median()
        ),
        "mean": float(
            model_training_target_comparison_df[
                "rul_uncapped"
            ].mean()
        ),
        "third_quartile": float(
            model_training_target_comparison_df[
                "rul_uncapped"
            ].quantile(0.75)
        ),
        "maximum": int(
            model_training_target_comparison_df[
                "rul_uncapped"
            ].max()
        )
    },
    {
        "target_definition": "candidate_capped_130",
        "candidate_cap": CANDIDATE_RUL_CAP,
        "row_count": total_rows,
        "engine_count": total_engines,
        "minimum": int(
            model_training_target_comparison_df[
                "rul_capped_130_candidate"
            ].min()
        ),
        "first_quartile": float(
            model_training_target_comparison_df[
                "rul_capped_130_candidate"
            ].quantile(0.25)
        ),
        "median": float(
            model_training_target_comparison_df[
                "rul_capped_130_candidate"
            ].median()
        ),
        "mean": float(
            model_training_target_comparison_df[
                "rul_capped_130_candidate"
            ].mean()
        ),
        "third_quartile": float(
            model_training_target_comparison_df[
                "rul_capped_130_candidate"
            ].quantile(0.75)
        ),
        "maximum": int(
            model_training_target_comparison_df[
                "rul_capped_130_candidate"
            ].max()
        )
    }
])

target_comparison_audit_file = (
    LOGS_DIR
    / "fd001_rul_target_comparison_audit.csv"
)

target_comparison_audit.to_csv(
    target_comparison_audit_file,
    index=False
)


# ------------------------------------------------------------
# 7. Create examples around the transition point
# ------------------------------------------------------------

eligible_engines = (
    model_training_target_comparison_df.loc[
        model_training_target_comparison_df[
            "rul_uncapped"
        ] > CANDIDATE_RUL_CAP,
        "unit_number"
    ]
    .unique()
)

if len(eligible_engines) == 0:
    raise RuntimeError(
        "No training engine contains RUL values above 130."
    )

example_engine_id = int(
    np.sort(eligible_engines)[0]
)

transition_examples_df = (
    model_training_target_comparison_df[
        (
            model_training_target_comparison_df[
                "unit_number"
            ] == example_engine_id
        )
        & (
            model_training_target_comparison_df[
                "rul_uncapped"
            ].between(
                CANDIDATE_RUL_CAP - 2,
                CANDIDATE_RUL_CAP + 2
            )
        )
    ]
    .copy()
)

transition_examples_file = (
    LOGS_DIR
    / "fd001_rul_cap_130_transition_examples.csv"
)

transition_examples_df.to_csv(
    transition_examples_file,
    index=False
)


# ------------------------------------------------------------
# 8. Save the candidate comparison table
# ------------------------------------------------------------

target_comparison_file = (
    INTERIM_DATA_DIR
    / "fd001_model_training_target_comparison.csv"
)

model_training_target_comparison_df.to_csv(
    target_comparison_file,
    index=False
)


# ------------------------------------------------------------
# 9. Create the first controlled EDA figure
# ------------------------------------------------------------

plt.style.use("seaborn-v0_8-whitegrid")

figure, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(14, 5.5)
)

# Panel A: target distributions
histogram_bins = np.arange(
    0,
    int(
        model_training_target_comparison_df[
            "rul_uncapped"
        ].max()
    ) + 15,
    10
)

axes[0].hist(
    model_training_target_comparison_df[
        "rul_uncapped"
    ],
    bins=histogram_bins,
    alpha=0.60,
    color="#3366CC",
    label="Uncapped RUL"
)

axes[0].hist(
    model_training_target_comparison_df[
        "rul_capped_130_candidate"
    ],
    bins=histogram_bins,
    alpha=0.55,
    color="#DC3912",
    label="Candidate cap = 130"
)

axes[0].axvline(
    CANDIDATE_RUL_CAP,
    color="black",
    linestyle="--",
    linewidth=1.5,
    label="Candidate boundary"
)

axes[0].set_title(
    "A. Model-training target distributions"
)
axes[0].set_xlabel("Remaining Useful Life (cycles)")
axes[0].set_ylabel("Number of cycle rows")
axes[0].legend()


# Panel B: transformation function
uncapped_axis = np.arange(
    0,
    int(
        model_training_target_comparison_df[
            "rul_uncapped"
        ].max()
    ) + 1
)

capped_axis = np.minimum(
    uncapped_axis,
    CANDIDATE_RUL_CAP
)

axes[1].plot(
    uncapped_axis,
    uncapped_axis,
    color="#3366CC",
    linewidth=2,
    label="Uncapped identity"
)

axes[1].plot(
    uncapped_axis,
    capped_axis,
    color="#DC3912",
    linewidth=2,
    label="Candidate capped target"
)

axes[1].axvline(
    CANDIDATE_RUL_CAP,
    color="black",
    linestyle="--",
    linewidth=1.5
)

axes[1].set_title(
    "B. Effect of the candidate transformation"
)
axes[1].set_xlabel("Actual uncapped RUL (cycles)")
axes[1].set_ylabel("Assigned target (cycles)")
axes[1].legend()

figure.suptitle(
    "Uncapped RUL versus the 130-cycle cap candidate",
    fontsize=14,
    fontweight="bold"
)

figure.text(
    0.5,
    0.01,
    (
        "Model-training engines only. "
        "The 130-cycle value is a research candidate, "
        "not a NASA-prescribed target."
    ),
    ha="center",
    fontsize=9
)

plt.tight_layout(rect=[0, 0.05, 1, 0.93])

figure_png_file = (
    FIGURES_DIR
    / "phase2_rul_cap_130_comparison.png"
)

figure_pdf_file = (
    FIGURES_DIR
    / "phase2_rul_cap_130_comparison.pdf"
)

figure.savefig(
    figure_png_file,
    dpi=300,
    bbox_inches="tight"
)

figure.savefig(
    figure_pdf_file,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 10. Validate the comparison
# ------------------------------------------------------------

comparison_checks = {
    "comparison_uses_only_model_training_engines":
        set(
            model_training_target_comparison_df[
                "unit_number"
            ].unique()
        )
        == set(
            model_train_rul_df[
                "unit_number"
            ].unique()
        ),

    "row_count_is_preserved":
        len(
            model_training_target_comparison_df
        ) == len(model_train_rul_df),

    "candidate_minimum_is_zero":
        int(
            model_training_target_comparison_df[
                "rul_capped_130_candidate"
            ].min()
        ) == 0,

    "candidate_maximum_is_130":
        int(
            model_training_target_comparison_df[
                "rul_capped_130_candidate"
            ].max()
        ) == CANDIDATE_RUL_CAP,

    "values_at_or_below_130_are_unchanged":
        bool(
            (
                model_training_target_comparison_df.loc[
                    model_training_target_comparison_df[
                        "rul_uncapped"
                    ] <= CANDIDATE_RUL_CAP,
                    "rul_uncapped"
                ]
                ==
                model_training_target_comparison_df.loc[
                    model_training_target_comparison_df[
                        "rul_uncapped"
                    ] <= CANDIDATE_RUL_CAP,
                    "rul_capped_130_candidate"
                ]
            ).all()
        ),

    "values_above_130_are_set_to_130":
        bool(
            (
                model_training_target_comparison_df.loc[
                    model_training_target_comparison_df[
                        "rul_uncapped"
                    ] > CANDIDATE_RUL_CAP,
                    "rul_capped_130_candidate"
                ]
                == CANDIDATE_RUL_CAP
            ).all()
        ),

    "approved_uncapped_dataframe_was_not_modified":
        "rul_capped_130_candidate"
        not in model_train_rul_df.columns,

    "validation_partition_was_not_modified":
        "rul_capped_130_candidate"
        not in validation_rul_df.columns,

    "calibration_partition_was_not_modified":
        "rul_capped_130_candidate"
        not in calibration_rul_df.columns,

    "comparison_figure_was_saved":
        (
            figure_png_file.exists()
            and figure_pdf_file.exists()
        )
}


# ------------------------------------------------------------
# 11. Save the scientific decision record
# ------------------------------------------------------------

candidate_record = {
    "recorded_at_utc": recorded_at_utc,
    "candidate_name": "130-cycle piecewise RUL cap",
    "formula": "minimum of uncapped RUL and 130",
    "candidate_status": (
        "Evaluated but not yet approved as final target"
    ),
    "nasa_prescribed_cap": False,
    "nasa_dataset_reference":
        NASA_DATASET_REFERENCE,
    "primary_research_reference":
        ZHENG_2017_DOI,
    "primary_reference_interpretation": (
        "Zheng et al. used a maximum piecewise RUL "
        "of 130 cycles but described fixed capping "
        "as a generic approach."
    ),
    "scope_examined": (
        "Model-training engines only"
    ),
    "affected_rows": affected_rows,
    "total_rows": total_rows,
    "affected_row_percentage":
        affected_row_percentage,
    "affected_engines": affected_engines,
    "total_engines": total_engines,
    "rows_at_candidate_plateau":
        rows_at_candidate_plateau,
    "maximum_label_reduction_cycles":
        maximum_label_reduction,
    "total_label_reduction_cycles":
        total_label_reduction,
    "validation_data_used_for_selection": False,
    "calibration_data_used": False,
    "test_rul_values_used": False,
    "final_target_selected": False,
    "comparison_checks": comparison_checks
}

candidate_record_file = (
    LOGS_DIR
    / "fd001_rul_cap_130_candidate_record.json"
)

with candidate_record_file.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        candidate_record,
        file,
        indent=4
    )


# ------------------------------------------------------------
# 12. Package the evidence
# ------------------------------------------------------------

step_3b_evidence_zip = (
    PROJECT_ROOT
    / "Phase2_Step3B_RUL_Cap_130_Evidence.zip"
)

with zipfile.ZipFile(
    step_3b_evidence_zip,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as zip_file:

    for evidence_file in [
        target_comparison_audit_file,
        transition_examples_file,
        candidate_record_file,
        figure_png_file,
        figure_pdf_file
    ]:
        zip_file.write(
            evidence_file,
            arcname=evidence_file.name
        )


# ------------------------------------------------------------
# 13. Display the results
# ------------------------------------------------------------

print("TARGET DISTRIBUTION COMPARISON")
print("=" * 90)
print(
    target_comparison_audit.to_string(
        index=False
    )
)

print("\nCAP EFFECT")
print("=" * 90)
print(f"Total model-training rows: {total_rows}")
print(f"Rows changed by cap: {affected_rows}")
print(
    "Percentage changed: "
    f"{affected_row_percentage:.2f}%"
)
print(
    f"Engines affected: "
    f"{affected_engines} of {total_engines}"
)
print(
    "Rows assigned to the 130-cycle plateau: "
    f"{rows_at_candidate_plateau}"
)
print(
    "Maximum label reduction: "
    f"{maximum_label_reduction} cycles"
)

print("\nTRANSITION EXAMPLE")
print("=" * 90)
print(
    transition_examples_df.to_string(
        index=False
    )
)

print("\nCOMPARISON VALIDATION")
print("=" * 90)

for check_name, passed in comparison_checks.items():
    status = "PASS" if passed else "FAIL"
    readable_name = check_name.replace("_", " ")

    print(f"{status} — {readable_name}")

print("\nDECISION STATUS")
print("=" * 90)
print("NASA-prescribed cap: No")
print("Candidate evaluated: 130 cycles")
print("Final target selected: No")
print("Validation data used: No")
print("Calibration data used: No")
print("Test RUL values used: No")

print("\nFINAL CHECK")
print("=" * 90)

if all(comparison_checks.values()):
    print("PASS — The 130-cycle candidate was evaluated correctly.")
    print("PASS — Only model-training labels were examined.")
    print("PASS — Validation and calibration remain untouched.")
    print("PASS — The final target has not been selected.")

    print("\nDownloading the Step 3B evidence ZIP...")
    colab_files.download(
        str(step_3b_evidence_zip)
    )

else:
    print("CHECK REQUIRED — Candidate validation failed.")

In [ ]:
# ============================================================
# PHASE 2 — RUL TARGET DESIGN
# STEP 3C — LOCK PRIMARY AND SENSITIVITY TARGET POLICY
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from google.colab import files as colab_files

import pandas as pd
import numpy as np
import json
import zipfile


# ------------------------------------------------------------
# 1. Confirm that Step 3A outputs exist
# ------------------------------------------------------------

required_dataframe_names = [
    "model_train_rul_df",
    "validation_rul_df",
    "calibration_rul_df"
]

missing_dataframes = [
    name
    for name in required_dataframe_names
    if name not in globals()
]

if missing_dataframes:
    raise RuntimeError(
        "Run Step 3A first. Missing dataframes: "
        + ", ".join(missing_dataframes)
    )


# ------------------------------------------------------------
# 2. Define controlled output locations
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/Predictive_Maintenance_Thesis")
INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"
LOGS_DIR = PROJECT_ROOT / "logs"

INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

recorded_at_utc = datetime.now(timezone.utc).isoformat()


# ------------------------------------------------------------
# 3. Freeze the target policy
# ------------------------------------------------------------

PRIMARY_TARGET = "rul_uncapped"
SENSITIVITY_TARGET = "rul_capped_130"
SENSITIVITY_CAP = 130

NASA_DATASET_REFERENCE = (
    "https://data.nasa.gov/dataset/"
    "cmapss-jet-engine-simulated-data"
)

ZHENG_2017_REFERENCE = (
    "https://doi.org/10.1109/"
    "ICPHM.2017.7998311"
)


# ------------------------------------------------------------
# 4. Create derived dual-target dataframes
# ------------------------------------------------------------

def create_dual_target_dataframe(
    uncapped_dataframe
):
    """
    Preserve the uncapped primary target and add the
    preregistered 130-cycle sensitivity target.

    The input dataframe is not modified.
    """

    dual_target_dataframe = (
        uncapped_dataframe.copy()
    )

    dual_target_dataframe[
        SENSITIVITY_TARGET
    ] = np.minimum(
        dual_target_dataframe[PRIMARY_TARGET],
        SENSITIVITY_CAP
    ).astype(int)

    return dual_target_dataframe


model_train_targets_df = (
    create_dual_target_dataframe(
        model_train_rul_df
    )
)

validation_targets_df = (
    create_dual_target_dataframe(
        validation_rul_df
    )
)

calibration_targets_df = (
    create_dual_target_dataframe(
        calibration_rul_df
    )
)


# ------------------------------------------------------------
# 5. Validate one dual-target partition
# ------------------------------------------------------------

def validate_dual_targets(
    original_dataframe,
    dual_target_dataframe
):
    """
    Confirm that the primary target remains unchanged
    and that the sensitivity target follows the policy.
    """

    expected_capped_target = np.minimum(
        dual_target_dataframe[PRIMARY_TARGET],
        SENSITIVITY_CAP
    ).astype(int)

    zero_rows_per_engine = (
        dual_target_dataframe
        .assign(
            is_zero=(
                dual_target_dataframe[
                    SENSITIVITY_TARGET
                ] == 0
            )
        )
        .groupby("unit_number")["is_zero"]
        .sum()
    )

    return {
        "row_count_preserved":
            len(original_dataframe)
            == len(dual_target_dataframe),

        "engine_membership_preserved":
            set(
                original_dataframe[
                    "unit_number"
                ].unique()
            )
            == set(
                dual_target_dataframe[
                    "unit_number"
                ].unique()
            ),

        "primary_uncapped_target_preserved":
            dual_target_dataframe[
                PRIMARY_TARGET
            ].equals(
                original_dataframe[
                    PRIMARY_TARGET
                ]
            ),

        "sensitivity_target_formula_correct":
            dual_target_dataframe[
                SENSITIVITY_TARGET
            ].equals(
                expected_capped_target
            ),

        "sensitivity_target_minimum_is_zero":
            int(
                dual_target_dataframe[
                    SENSITIVITY_TARGET
                ].min()
            ) == 0,

        "sensitivity_target_maximum_is_130":
            int(
                dual_target_dataframe[
                    SENSITIVITY_TARGET
                ].max()
            ) == SENSITIVITY_CAP,

        "sensitivity_target_has_no_missing_values":
            int(
                dual_target_dataframe[
                    SENSITIVITY_TARGET
                ].isna().sum()
            ) == 0,

        "sensitivity_target_is_integer":
            bool(
                pd.api.types.is_integer_dtype(
                    dual_target_dataframe[
                        SENSITIVITY_TARGET
                    ]
                )
            ),

        "exactly_one_zero_target_per_engine":
            bool(
                (zero_rows_per_engine == 1).all()
            ),

        "original_dataframe_was_not_modified":
            SENSITIVITY_TARGET
            not in original_dataframe.columns
    }


model_training_checks = (
    validate_dual_targets(
        model_train_rul_df,
        model_train_targets_df
    )
)

validation_checks = (
    validate_dual_targets(
        validation_rul_df,
        validation_targets_df
    )
)

calibration_checks = (
    validate_dual_targets(
        calibration_rul_df,
        calibration_targets_df
    )
)


# ------------------------------------------------------------
# 6. Save the reproducible dual-target datasets
# ------------------------------------------------------------

model_training_targets_file = (
    INTERIM_DATA_DIR
    / "fd001_model_training_targets.csv"
)

validation_targets_file = (
    INTERIM_DATA_DIR
    / "fd001_validation_targets.csv"
)

calibration_targets_file = (
    INTERIM_DATA_DIR
    / "fd001_calibration_targets.csv"
)

model_train_targets_df.to_csv(
    model_training_targets_file,
    index=False
)

validation_targets_df.to_csv(
    validation_targets_file,
    index=False
)

calibration_targets_df.to_csv(
    calibration_targets_file,
    index=False
)


# ------------------------------------------------------------
# 7. Create an audit of target effects
# ------------------------------------------------------------

def create_target_audit(
    dataframe,
    partition_name
):
    """
    Summarise both targets without fitting a model.
    """

    changed_mask = (
        dataframe[PRIMARY_TARGET]
        != dataframe[SENSITIVITY_TARGET]
    )

    return {
        "partition": partition_name,

        "engine_count": int(
            dataframe[
                "unit_number"
            ].nunique()
        ),

        "row_count": int(
            len(dataframe)
        ),

        "uncapped_mean": float(
            dataframe[
                PRIMARY_TARGET
            ].mean()
        ),

        "uncapped_median": float(
            dataframe[
                PRIMARY_TARGET
            ].median()
        ),

        "uncapped_maximum": int(
            dataframe[
                PRIMARY_TARGET
            ].max()
        ),

        "capped_130_mean": float(
            dataframe[
                SENSITIVITY_TARGET
            ].mean()
        ),

        "capped_130_median": float(
            dataframe[
                SENSITIVITY_TARGET
            ].median()
        ),

        "capped_130_maximum": int(
            dataframe[
                SENSITIVITY_TARGET
            ].max()
        ),

        "rows_changed_by_sensitivity_cap": int(
            changed_mask.sum()
        ),

        "percentage_changed": float(
            changed_mask.mean() * 100
        )
    }


target_audit_df = pd.DataFrame([
    create_target_audit(
        model_train_targets_df,
        "model_training"
    ),
    create_target_audit(
        validation_targets_df,
        "validation"
    ),
    create_target_audit(
        calibration_targets_df,
        "conformal_calibration"
    )
])

target_audit_file = (
    LOGS_DIR
    / "fd001_final_target_policy_audit.csv"
)

target_audit_df.to_csv(
    target_audit_file,
    index=False
)


# ------------------------------------------------------------
# 8. Combine the technical checks
# ------------------------------------------------------------

all_target_policy_checks = {
    **{
        f"model_training_{name}": passed
        for name, passed
        in model_training_checks.items()
    },

    **{
        f"validation_{name}": passed
        for name, passed
        in validation_checks.items()
    },

    **{
        f"calibration_{name}": passed
        for name, passed
        in calibration_checks.items()
    }
}


# ------------------------------------------------------------
# 9. Create the formal target-policy record
# ------------------------------------------------------------

target_policy_record = {
    "recorded_at_utc": recorded_at_utc,

    "policy_status":
        "Technically frozen; awaiting Step 3C approval",

    "primary_target": {
        "column_name": PRIMARY_TARGET,
        "definition": (
            "Failure cycle minus current cycle"
        ),
        "role": (
            "Primary target for model comparison, "
            "uncertainty estimation and final reporting"
        ),
        "reason": (
            "Preserves actual remaining-cycle information "
            "and aligns directly with NASA's stated task."
        )
    },

    "sensitivity_target": {
        "column_name": SENSITIVITY_TARGET,
        "definition": (
            "Minimum of uncapped RUL and 130 cycles"
        ),
        "role": (
            "Secondary sensitivity analysis for assessing "
            "the effect of the common early-life plateau "
            "assumption"
        ),
        "reason": (
            "Supports comparison with influential C-MAPSS "
            "research while keeping the modelling assumption "
            "explicit."
        )
    },

    "candidate_cap_cycles":
        SENSITIVITY_CAP,

    "nasa_prescribed_cap": False,

    "nasa_reference":
        NASA_DATASET_REFERENCE,

    "research_reference":
        ZHENG_2017_REFERENCE,

    "cap_optimality_claimed": False,

    "validation_used_to_choose_cap": False,

    "calibration_used_to_choose_cap": False,

    "official_test_data_used": False,

    "test_rul_values_used": False,

    "calibration_use_policy": (
        "Calibration labels have received the mechanically "
        "predeclared target transformation, but calibration "
        "engines remain prohibited from feature selection, "
        "model selection and hyperparameter tuning."
    ),

    "evaluation_policy": (
        "Primary conclusions will be based on uncapped actual "
        "RUL. The capped-130 experiment will be reported "
        "separately as a sensitivity analysis and must not "
        "replace or conceal the uncapped results."
    ),

    "comparison_fairness_policy": (
        "When uncapped and capped targets are compared, "
        "the same engine partitions, candidate features, "
        "models and evaluation procedures must be used."
    ),

    "target_policy_checks":
        all_target_policy_checks
}

target_policy_file = (
    LOGS_DIR
    / "fd001_final_target_policy.json"
)

with target_policy_file.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        target_policy_record,
        file,
        indent=4
    )


# ------------------------------------------------------------
# 10. Create a human-readable decision note
# ------------------------------------------------------------

target_decision_markdown = (
    LOGS_DIR
    / "fd001_target_decision.md"
)

target_decision_text = f"""
# FD001 RUL Target Decision

Generated at UTC: {recorded_at_utc}

## Primary target

`{PRIMARY_TARGET}` is the primary thesis target.

It preserves the actual number of remaining operating cycles
and directly reflects the prediction task stated by NASA.

## Sensitivity target

`{SENSITIVITY_TARGET}` is a secondary sensitivity target.

It applies:

`minimum(uncapped RUL, {SENSITIVITY_CAP})`

The cap is a modelling assumption used in influential C-MAPSS
research. It is not a NASA-prescribed target and is not claimed
to be scientifically optimal.

## Governance controls

- Validation data did not determine the target policy.
- Calibration data did not determine the target policy.
- Official test outcomes were not accessed.
- Uncapped results must remain visible in final reporting.
- Capped results must be labelled as sensitivity analysis.
- The same engine partitions must be used for both targets.
"""

target_decision_markdown.write_text(
    target_decision_text.strip(),
    encoding="utf-8"
)


# ------------------------------------------------------------
# 11. Package the decision evidence
# ------------------------------------------------------------

step_3c_evidence_zip = (
    PROJECT_ROOT
    / "Phase2_Step3C_Final_Target_Policy_Evidence.zip"
)

with zipfile.ZipFile(
    step_3c_evidence_zip,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as zip_file:

    for evidence_file in [
        target_audit_file,
        target_policy_file,
        target_decision_markdown
    ]:
        zip_file.write(
            evidence_file,
            arcname=evidence_file.name
        )


# ------------------------------------------------------------
# 12. Display the results
# ------------------------------------------------------------

print("LOCKED TARGET POLICY")
print("=" * 90)
print(
    f"Primary target: {PRIMARY_TARGET}"
)
print(
    f"Sensitivity target: {SENSITIVITY_TARGET}"
)
print(
    f"Sensitivity cap: {SENSITIVITY_CAP} cycles"
)
print("NASA-prescribed cap: No")
print("Cap claimed to be optimal: No")

print("\nTARGET POLICY AUDIT")
print("=" * 90)
print(
    target_audit_df.to_string(
        index=False
    )
)

print("\nTARGET POLICY VALIDATION")
print("=" * 90)

for check_name, passed in (
    all_target_policy_checks.items()
):
    status = "PASS" if passed else "FAIL"
    readable_name = check_name.replace("_", " ")

    print(f"{status} — {readable_name}")

print("\nRESEARCH CONTROLS")
print("=" * 90)
print("Validation used to choose target: No")
print("Calibration used to choose target: No")
print("Official test data used: No")
print("Test RUL values accessed: No")
print("Primary uncapped result preserved: Yes")
print("Capped result labelled sensitivity analysis: Yes")

print("\nOUTPUT FILES")
print("=" * 90)
print(
    f"Model-training targets: "
    f"{model_training_targets_file}"
)
print(
    f"Validation targets: "
    f"{validation_targets_file}"
)
print(
    f"Calibration targets: "
    f"{calibration_targets_file}"
)
print(f"Target audit: {target_audit_file}")
print(f"Target policy: {target_policy_file}")
print(
    f"Human-readable decision: "
    f"{target_decision_markdown}"
)
print(f"Evidence ZIP: {step_3c_evidence_zip}")

print("\nFINAL CHECK")
print("=" * 90)

if all(all_target_policy_checks.values()):
    print("PASS — The RUL target policy was frozen successfully.")
    print("PASS — Uncapped RUL is preserved as the primary target.")
    print("PASS — Capped-130 RUL is registered as sensitivity analysis.")
    print("PASS — Frozen engine partitions remain unchanged.")
    print("PASS — No model or official test outcome was used.")
    print("AWAITING APPROVAL — Target policy is not yet formally closed.")

    print("\nDownloading the Step 3C evidence ZIP...")
    colab_files.download(
        str(step_3c_evidence_zip)
    )

else:
    print("CHECK REQUIRED — Target-policy validation failed.")

In [ ]:
# ============================================================
# PHASE 2 — EXPLORATORY DATA ANALYSIS
# STEP 4A — TRAINING-ONLY LIFETIME AND SENSOR EDA
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from google.colab import files as colab_files

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import zipfile


# ------------------------------------------------------------
# 1. Confirm that the locked target datasets exist
# ------------------------------------------------------------

required_dataframe_names = [
    "model_train_targets_df",
    "validation_targets_df",
    "calibration_targets_df"
]

missing_dataframes = [
    name
    for name in required_dataframe_names
    if name not in globals()
]

if missing_dataframes:
    raise RuntimeError(
        "Run Step 3C first. Missing dataframes: "
        + ", ".join(missing_dataframes)
    )


# ------------------------------------------------------------
# 2. Define controlled output locations
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/Predictive_Maintenance_Thesis")
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
LOGS_DIR = PROJECT_ROOT / "logs"

for directory in [
    RESULTS_DIR,
    FIGURES_DIR,
    LOGS_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

recorded_at_utc = datetime.now(
    timezone.utc
).isoformat()


# ------------------------------------------------------------
# 3. Formally record Step 3C approval
# ------------------------------------------------------------

target_policy_closure = {
    "recorded_at_utc": recorded_at_utc,
    "phase": "Phase 2",
    "step": "Step 3C — RUL target policy",
    "status": "Approved and locked",
    "primary_target": "rul_uncapped",
    "sensitivity_target": "rul_capped_130",
    "sensitivity_cap_cycles": 130,
    "nasa_prescribed_cap": False,
    "cap_optimality_claimed": False,
    "validation_used_to_choose_policy": False,
    "calibration_used_to_choose_policy": False,
    "official_test_outcomes_used": False
}

target_policy_closure_file = (
    LOGS_DIR
    / "phase2_step3c_target_policy_closure.json"
)

with target_policy_closure_file.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        target_policy_closure,
        file,
        indent=4
    )


# ------------------------------------------------------------
# 4. Establish engine membership controls
# ------------------------------------------------------------

model_training_engine_set = set(
    model_train_targets_df[
        "unit_number"
    ].unique()
)

validation_engine_set = set(
    validation_targets_df[
        "unit_number"
    ].unique()
)

calibration_engine_set = set(
    calibration_targets_df[
        "unit_number"
    ].unique()
)


# ------------------------------------------------------------
# 5. Create model-training engine lifetime summary
# ------------------------------------------------------------

training_engine_lifetimes_df = (
    model_train_targets_df
    .groupby(
        "unit_number",
        as_index=False
    )
    .agg(
        observed_lifetime_cycles=(
            "time_in_cycles",
            "max"
        ),
        observed_cycle_rows=(
            "time_in_cycles",
            "size"
        )
    )
)

training_engine_lifetimes_file = (
    RESULTS_DIR
    / "phase2_model_training_engine_lifetimes.csv"
)

training_engine_lifetimes_df.to_csv(
    training_engine_lifetimes_file,
    index=False
)


# ------------------------------------------------------------
# 6. Identify variable sensor channels
# ------------------------------------------------------------

all_sensor_columns = [
    column
    for column in model_train_targets_df.columns
    if column.startswith("sensor_")
]

constant_sensor_columns = [
    column
    for column in all_sensor_columns
    if (
        model_train_targets_df[
            column
        ].nunique(dropna=False) == 1
    )
]

variable_sensor_columns = [
    column
    for column in all_sensor_columns
    if column not in constant_sensor_columns
]


# ------------------------------------------------------------
# 7. Define engine-aware Spearman correlation
# ------------------------------------------------------------

def calculate_rank_correlation(
    first_series,
    second_series
):
    """
    Calculate Spearman correlation by correlating ranks.

    This avoids treating the numeric scale as necessarily
    linear.
    """

    first_ranks = first_series.rank(
        method="average"
    )

    second_ranks = second_series.rank(
        method="average"
    )

    return first_ranks.corr(second_ranks)


# ------------------------------------------------------------
# 8. Calculate correlation separately inside each engine
# ------------------------------------------------------------

engine_sensor_correlation_records = []

for sensor_column in variable_sensor_columns:

    for engine_id, engine_data in (
        model_train_targets_df.groupby(
            "unit_number",
            sort=True
        )
    ):

        correlation = (
            calculate_rank_correlation(
                engine_data[sensor_column],
                engine_data["rul_uncapped"]
            )
        )

        engine_sensor_correlation_records.append({
            "unit_number": int(engine_id),
            "sensor": sensor_column,
            "spearman_sensor_vs_uncapped_rul":
                (
                    float(correlation)
                    if pd.notna(correlation)
                    else np.nan
                )
        })


engine_sensor_correlations_df = pd.DataFrame(
    engine_sensor_correlation_records
)

engine_sensor_correlations_file = (
    RESULTS_DIR
    / "phase2_engine_sensor_correlations.csv"
)

engine_sensor_correlations_df.to_csv(
    engine_sensor_correlations_file,
    index=False
)


# ------------------------------------------------------------
# 9. Summarise correlations across engines
# ------------------------------------------------------------

sensor_association_records = []

for sensor_column in variable_sensor_columns:

    sensor_values = (
        engine_sensor_correlations_df.loc[
            engine_sensor_correlations_df[
                "sensor"
            ] == sensor_column,
            "spearman_sensor_vs_uncapped_rul"
        ]
        .dropna()
        .to_numpy()
    )

    sensor_association_records.append({
        "sensor": sensor_column,
        "engines_with_finite_correlation":
            int(len(sensor_values)),
        "median_within_engine_spearman":
            float(np.median(sensor_values)),
        "first_quartile_spearman":
            float(
                np.quantile(
                    sensor_values,
                    0.25
                )
            ),
        "third_quartile_spearman":
            float(
                np.quantile(
                    sensor_values,
                    0.75
                )
            )
    })


sensor_association_summary_df = pd.DataFrame(
    sensor_association_records
)

sensor_association_summary_df[
    "absolute_median_spearman"
] = sensor_association_summary_df[
    "median_within_engine_spearman"
].abs()

sensor_association_summary_df = (
    sensor_association_summary_df
    .sort_values(
        "absolute_median_spearman",
        ascending=False
    )
    .reset_index(drop=True)
)

sensor_association_summary_file = (
    RESULTS_DIR
    / "phase2_sensor_association_summary.csv"
)

sensor_association_summary_df.to_csv(
    sensor_association_summary_file,
    index=False
)


# ------------------------------------------------------------
# 10. Select four sensors for visual exploration
# ------------------------------------------------------------

TOP_SENSOR_COUNT = 4

top_sensor_columns = (
    sensor_association_summary_df
    .head(TOP_SENSOR_COUNT)["sensor"]
    .tolist()
)

print(
    "Selected sensors for descriptive visualisation:"
)
print(top_sensor_columns)


# ------------------------------------------------------------
# 11. Create normalised engine-life position
# ------------------------------------------------------------

# This dataframe exists only for EDA.
# It does not replace the modelling dataset.

eda_working_df = (
    model_train_targets_df.copy()
)

engine_final_cycle = (
    eda_working_df
    .groupby("unit_number")[
        "time_in_cycles"
    ]
    .transform("max")
)

eda_working_df[
    "normalised_life_fraction"
] = (
    (
        eda_working_df["time_in_cycles"] - 1
    )
    / (
        engine_final_cycle - 1
    )
)

LIFE_BIN_COUNT = 20

eda_working_df[
    "life_stage_bin"
] = pd.cut(
    eda_working_df[
        "normalised_life_fraction"
    ],
    bins=np.linspace(
        0,
        1,
        LIFE_BIN_COUNT + 1
    ),
    labels=False,
    include_lowest=True
).astype(int)


# ------------------------------------------------------------
# 12. Standardise selected sensors for EDA only
# ------------------------------------------------------------

sensor_standardisation_records = []

for sensor_column in top_sensor_columns:

    sensor_mean = float(
        eda_working_df[
            sensor_column
        ].mean()
    )

    sensor_standard_deviation = float(
        eda_working_df[
            sensor_column
        ].std(ddof=0)
    )

    if sensor_standard_deviation == 0:
        raise ValueError(
            f"{sensor_column} unexpectedly has "
            "zero standard deviation."
        )

    standardised_column = (
        f"{sensor_column}_eda_zscore"
    )

    eda_working_df[
        standardised_column
    ] = (
        (
            eda_working_df[sensor_column]
            - sensor_mean
        )
        / sensor_standard_deviation
    )

    sensor_standardisation_records.append({
        "sensor": sensor_column,
        "model_training_mean":
            sensor_mean,
        "model_training_standard_deviation":
            sensor_standard_deviation,
        "purpose": (
            "Visual comparison only; "
            "not fitted preprocessing"
        )
    })


sensor_standardisation_df = pd.DataFrame(
    sensor_standardisation_records
)

sensor_standardisation_file = (
    LOGS_DIR
    / "phase2_eda_sensor_standardisation.csv"
)

sensor_standardisation_df.to_csv(
    sensor_standardisation_file,
    index=False
)


# ------------------------------------------------------------
# 13. Give each engine equal weight inside each life bin
# ------------------------------------------------------------

standardised_sensor_columns = [
    f"{sensor}_eda_zscore"
    for sensor in top_sensor_columns
]

# First aggregate within each engine and life bin.
# This prevents long engines from dominating the profile.

within_engine_life_profiles_df = (
    eda_working_df
    .groupby(
        [
            "unit_number",
            "life_stage_bin"
        ],
        observed=True
    )[
        standardised_sensor_columns
    ]
    .median()
    .reset_index()
)


# ------------------------------------------------------------
# 14. Aggregate the engine-level profiles
# ------------------------------------------------------------

life_profile_records = []

for sensor_column in top_sensor_columns:

    standardised_column = (
        f"{sensor_column}_eda_zscore"
    )

    grouped_profile = (
        within_engine_life_profiles_df
        .groupby(
            "life_stage_bin",
            observed=True
        )[standardised_column]
        .agg(
            median_standardised_value="median",
            first_quartile_standardised_value=(
                lambda values:
                values.quantile(0.25)
            ),
            third_quartile_standardised_value=(
                lambda values:
                values.quantile(0.75)
            ),
            contributing_engines="count"
        )
        .reset_index()
    )

    grouped_profile["sensor"] = sensor_column

    grouped_profile[
        "life_bin_midpoint_percent"
    ] = (
        (
            grouped_profile[
                "life_stage_bin"
            ] + 0.5
        )
        / LIFE_BIN_COUNT
        * 100
    )

    life_profile_records.append(
        grouped_profile
    )


sensor_life_profiles_df = pd.concat(
    life_profile_records,
    ignore_index=True
)

sensor_life_profiles_file = (
    RESULTS_DIR
    / "phase2_top_sensor_life_profiles.csv"
)

sensor_life_profiles_df.to_csv(
    sensor_life_profiles_file,
    index=False
)


# ------------------------------------------------------------
# 15. Define controlled chart styling
# ------------------------------------------------------------

plt.style.use("seaborn-v0_8-whitegrid")

BLUE = "#2563EB"
LIGHT_BLUE = "#93C5FD"
ORANGE = "#D97706"
DARK_GREY = "#374151"
LIGHT_GREY = "#D1D5DB"


# ------------------------------------------------------------
# 16. Figure 1 — engine lifetime distribution
# ------------------------------------------------------------

lifetime_values = (
    training_engine_lifetimes_df[
        "observed_lifetime_cycles"
    ]
)

lifetime_mean = float(
    lifetime_values.mean()
)

lifetime_median = float(
    lifetime_values.median()
)

figure_1, axes_1 = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(13, 5)
)

histogram_bins = np.arange(
    int(
        np.floor(
            lifetime_values.min() / 20
        ) * 20
    ),
    int(
        np.ceil(
            lifetime_values.max() / 20
        ) * 20
    ) + 20,
    20
)

axes_1[0].hist(
    lifetime_values,
    bins=histogram_bins,
    color=BLUE,
    edgecolor="white",
    linewidth=1
)

axes_1[0].axvline(
    lifetime_mean,
    color=ORANGE,
    linestyle="--",
    linewidth=2,
    label=f"Mean: {lifetime_mean:.1f}"
)

axes_1[0].axvline(
    lifetime_median,
    color=DARK_GREY,
    linestyle=":",
    linewidth=2,
    label=f"Median: {lifetime_median:.1f}"
)

axes_1[0].set_title(
    "A. Distribution of complete engine lifetimes"
)
axes_1[0].set_xlabel(
    "Observed lifetime (cycles)"
)
axes_1[0].set_ylabel(
    "Number of model-training engines"
)
axes_1[0].legend()


sorted_lifetimes = np.sort(
    lifetime_values.to_numpy()
)

empirical_probability = (
    np.arange(
        1,
        len(sorted_lifetimes) + 1
    )
    / len(sorted_lifetimes)
)

axes_1[1].step(
    sorted_lifetimes,
    empirical_probability,
    where="post",
    color=BLUE,
    linewidth=2
)

axes_1[1].axvline(
    lifetime_median,
    color=DARK_GREY,
    linestyle=":",
    linewidth=2
)

axes_1[1].set_title(
    "B. Empirical cumulative distribution"
)
axes_1[1].set_xlabel(
    "Observed lifetime (cycles)"
)
axes_1[1].set_ylabel(
    "Proportion of engines"
)
axes_1[1].set_ylim(0, 1.02)

figure_1.suptitle(
    "Model-training engine lifetime distribution",
    fontsize=14,
    fontweight="bold"
)

figure_1.text(
    0.5,
    0.01,
    (
        "FD001 model-training partition only "
        f"(n={len(lifetime_values)} complete "
        "run-to-failure engines)."
    ),
    ha="center",
    fontsize=9
)

figure_1.tight_layout(
    rect=[0, 0.05, 1, 0.93]
)

lifetime_figure_png = (
    FIGURES_DIR
    / "phase2_model_training_lifetime_distribution.png"
)

lifetime_figure_pdf = (
    FIGURES_DIR
    / "phase2_model_training_lifetime_distribution.pdf"
)

figure_1.savefig(
    lifetime_figure_png,
    dpi=300,
    bbox_inches="tight"
)

figure_1.savefig(
    lifetime_figure_pdf,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 17. Figure 2 — engine-aware sensor associations
# ------------------------------------------------------------

association_plot_df = (
    sensor_association_summary_df
    .sort_values(
        "median_within_engine_spearman",
        ascending=True
    )
    .reset_index(drop=True)
)

association_colours = [
    BLUE if value >= 0 else ORANGE
    for value in association_plot_df[
        "median_within_engine_spearman"
    ]
]

figure_2, axis_2 = plt.subplots(
    figsize=(10, 7)
)

bars = axis_2.barh(
    association_plot_df["sensor"],
    association_plot_df[
        "median_within_engine_spearman"
    ],
    color=association_colours,
    edgecolor="white"
)

axis_2.axvline(
    0,
    color=DARK_GREY,
    linewidth=1.2
)

axis_2.set_xlim(-1.05, 1.05)

axis_2.set_title(
    "Sensor association with uncapped RUL",
    fontsize=14,
    fontweight="bold"
)

axis_2.set_xlabel(
    "Median within-engine Spearman correlation"
)

axis_2.set_ylabel(
    "Anonymised sensor channel"
)

for bar, correlation_value in zip(
    bars,
    association_plot_df[
        "median_within_engine_spearman"
    ]
):
    label_position = (
        correlation_value + 0.025
        if correlation_value >= 0
        else correlation_value - 0.025
    )

    horizontal_alignment = (
        "left"
        if correlation_value >= 0
        else "right"
    )

    axis_2.text(
        label_position,
        bar.get_y() + bar.get_height() / 2,
        f"{correlation_value:.2f}",
        va="center",
        ha=horizontal_alignment,
        fontsize=8
    )

figure_2.text(
    0.5,
    0.01,
    (
        "Correlation was calculated separately "
        "within each of 60 engines and summarised "
        "by the median. Descriptive association only."
    ),
    ha="center",
    fontsize=9
)

figure_2.tight_layout(
    rect=[0, 0.05, 1, 0.95]
)

association_figure_png = (
    FIGURES_DIR
    / "phase2_engine_aware_sensor_associations.png"
)

association_figure_pdf = (
    FIGURES_DIR
    / "phase2_engine_aware_sensor_associations.pdf"
)

figure_2.savefig(
    association_figure_png,
    dpi=300,
    bbox_inches="tight"
)

figure_2.savefig(
    association_figure_pdf,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 18. Figure 3 — top-sensor life-stage profiles
# ------------------------------------------------------------

figure_3, axes_3 = plt.subplots(
    nrows=2,
    ncols=2,
    figsize=(13, 9),
    sharex=True
)

axes_3_flat = axes_3.flatten()

for axis, sensor_column in zip(
    axes_3_flat,
    top_sensor_columns
):
    sensor_profile = (
        sensor_life_profiles_df[
            sensor_life_profiles_df[
                "sensor"
            ] == sensor_column
        ]
        .sort_values(
            "life_stage_bin"
        )
    )

    x_values = sensor_profile[
        "life_bin_midpoint_percent"
    ].to_numpy(dtype=float)

    median_values = sensor_profile[
        "median_standardised_value"
    ].to_numpy(dtype=float)

    lower_values = sensor_profile[
        "first_quartile_standardised_value"
    ].to_numpy(dtype=float)

    upper_values = sensor_profile[
        "third_quartile_standardised_value"
    ].to_numpy(dtype=float)

    axis.fill_between(
        x_values,
        lower_values,
        upper_values,
        color=LIGHT_BLUE,
        alpha=0.45,
        label="Engine IQR"
    )

    axis.plot(
        x_values,
        median_values,
        color=BLUE,
        linewidth=2.2,
        marker="o",
        markersize=3,
        label="Engine median"
    )

    axis.axhline(
        0,
        color=DARK_GREY,
        linestyle=":",
        linewidth=1
    )

    axis.set_title(sensor_column)
    axis.set_xlabel(
        "Normalised engine-life position (%)"
    )
    axis.set_ylabel(
        "Standardised sensor value"
    )

    axis.legend(
        loc="best",
        fontsize=8
    )


figure_3.suptitle(
    "Life-stage profiles of the four strongest sensor associations",
    fontsize=14,
    fontweight="bold"
)

figure_3.text(
    0.5,
    0.01,
    (
        "Model-training engines only. Each engine "
        "contributes once per life-stage bin; shaded "
        "areas show the interquartile range."
    ),
    ha="center",
    fontsize=9
)

figure_3.tight_layout(
    rect=[0, 0.04, 1, 0.94]
)

sensor_profile_figure_png = (
    FIGURES_DIR
    / "phase2_top_sensor_life_stage_profiles.png"
)

sensor_profile_figure_pdf = (
    FIGURES_DIR
    / "phase2_top_sensor_life_stage_profiles.pdf"
)

figure_3.savefig(
    sensor_profile_figure_png,
    dpi=300,
    bbox_inches="tight"
)

figure_3.savefig(
    sensor_profile_figure_pdf,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 19. Validate the EDA boundaries and outputs
# ------------------------------------------------------------

profile_bin_counts = (
    sensor_life_profiles_df
    .groupby("sensor")[
        "life_stage_bin"
    ]
    .nunique()
)

eda_checks = {
    "eda_contains_only_60_model_training_engines":
        len(
            training_engine_lifetimes_df
        ) == 60,

    "eda_engine_membership_matches_frozen_training_partition":
        set(
            training_engine_lifetimes_df[
                "unit_number"
            ]
        ) == model_training_engine_set,

    "model_training_and_validation_do_not_overlap":
        len(
            model_training_engine_set
            & validation_engine_set
        ) == 0,

    "model_training_and_calibration_do_not_overlap":
        len(
            model_training_engine_set
            & calibration_engine_set
        ) == 0,

    "at_least_four_variable_sensors_exist":
        len(variable_sensor_columns) >= 4,

    "four_sensors_were_selected":
        len(top_sensor_columns) == 4,

    "selected_sensors_are_not_constant":
        all(
            sensor not in constant_sensor_columns
            for sensor in top_sensor_columns
        ),

    "all_summary_correlations_are_finite":
        bool(
            np.isfinite(
                sensor_association_summary_df[
                    "median_within_engine_spearman"
                ]
            ).all()
        ),

    "normalised_life_fraction_stays_between_zero_and_one":
        bool(
            eda_working_df[
                "normalised_life_fraction"
            ].between(0, 1).all()
        ),

    "each_selected_sensor_has_20_life_bins":
        bool(
            (
                profile_bin_counts
                == LIFE_BIN_COUNT
            ).all()
        ),

    "source_training_dataframe_was_not_modified":
        (
            "normalised_life_fraction"
            not in model_train_targets_df.columns
        ),

    "validation_dataframe_was_not_modified":
        (
            "normalised_life_fraction"
            not in validation_targets_df.columns
        ),

    "calibration_dataframe_was_not_modified":
        (
            "normalised_life_fraction"
            not in calibration_targets_df.columns
        ),

    "all_three_figures_were_saved":
        all(
            file_path.exists()
            for file_path in [
                lifetime_figure_png,
                lifetime_figure_pdf,
                association_figure_png,
                association_figure_pdf,
                sensor_profile_figure_png,
                sensor_profile_figure_pdf
            ]
        )
}


# ------------------------------------------------------------
# 20. Save the EDA decision record
# ------------------------------------------------------------

eda_record = {
    "recorded_at_utc": recorded_at_utc,
    "phase": "Phase 2",
    "step": (
        "Step 4A — Training-only lifetime and sensor EDA"
    ),
    "data_scope": (
        "60 frozen model-training engines only"
    ),
    "analytical_questions": [
        (
            "How much do complete engine lifetimes "
            "vary in the model-training partition?"
        ),
        (
            "Which anonymised sensors show the "
            "strongest consistent monotonic relationship "
            "with uncapped RUL within engines?"
        ),
        (
            "How do the strongest sensor channels change "
            "across normalised engine life?"
        )
    ],
    "correlation_method": (
        "Spearman correlation calculated within each "
        "engine and summarised by the median across engines"
    ),
    "trajectory_method": (
        "Twenty normalised life bins; median within each "
        "engine-bin followed by median and IQR across engines"
    ),
    "constant_sensor_columns":
        constant_sensor_columns,
    "variable_sensor_columns":
        variable_sensor_columns,
    "selected_top_sensor_columns":
        top_sensor_columns,
    "important_limitations": [
        (
            "Correlations are descriptive and do not "
            "establish causation."
        ),
        (
            "Anonymised sensors are not assigned invented "
            "physical meanings or units."
        ),
        (
            "Strong marginal association does not guarantee "
            "incremental predictive value in a multivariable model."
        ),
        (
            "Selected charts do not constitute final "
            "feature selection."
        )
    ],
    "validation_engines_used": False,
    "calibration_engines_used": False,
    "official_test_data_used": False,
    "test_rul_values_used": False,
    "features_removed": False,
    "models_trained": False,
    "eda_checks": eda_checks
}

eda_record_file = (
    LOGS_DIR
    / "phase2_training_only_eda_record.json"
)

with eda_record_file.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        eda_record,
        file,
        indent=4
    )


# ------------------------------------------------------------
# 21. Package the EDA evidence
# ------------------------------------------------------------

step_4a_evidence_zip = (
    PROJECT_ROOT
    / "Phase2_Step4A_Training_Only_EDA_Evidence.zip"
)

with zipfile.ZipFile(
    step_4a_evidence_zip,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as zip_file:

    for evidence_file in [
        target_policy_closure_file,
        training_engine_lifetimes_file,
        engine_sensor_correlations_file,
        sensor_association_summary_file,
        sensor_standardisation_file,
        sensor_life_profiles_file,
        eda_record_file,
        lifetime_figure_png,
        lifetime_figure_pdf,
        association_figure_png,
        association_figure_pdf,
        sensor_profile_figure_png,
        sensor_profile_figure_pdf
    ]:
        zip_file.write(
            evidence_file,
            arcname=evidence_file.name
        )


# ------------------------------------------------------------
# 22. Display the findings
# ------------------------------------------------------------

print("MODEL-TRAINING LIFETIME SUMMARY")
print("=" * 90)
print(
    lifetime_values.describe().to_string()
)

print("\nSENSOR INVENTORY")
print("=" * 90)
print(
    f"All sensor channels: "
    f"{len(all_sensor_columns)}"
)
print(
    f"Constant sensor channels: "
    f"{constant_sensor_columns}"
)
print(
    f"Variable sensor channels: "
    f"{len(variable_sensor_columns)}"
)

print("\nENGINE-AWARE SENSOR ASSOCIATIONS")
print("=" * 90)
print(
    sensor_association_summary_df[
        [
            "sensor",
            "engines_with_finite_correlation",
            "median_within_engine_spearman",
            "first_quartile_spearman",
            "third_quartile_spearman"
        ]
    ].to_string(index=False)
)

print("\nSELECTED VISUALISATION SENSORS")
print("=" * 90)
print(top_sensor_columns)

print("\nEDA VALIDATION")
print("=" * 90)

for check_name, passed in eda_checks.items():
    status = "PASS" if passed else "FAIL"
    readable_name = check_name.replace("_", " ")

    print(f"{status} — {readable_name}")

print("\nRESEARCH CONTROLS")
print("=" * 90)
print("Validation engines used: No")
print("Calibration engines used: No")
print("Official test data used: No")
print("Test RUL values accessed: No")
print("Features removed: No")
print("Models trained: No")

print("\nFINAL CHECK")
print("=" * 90)

if all(eda_checks.values()):
    print("PASS — Training-only EDA completed successfully.")
    print("PASS — Engine-aware sensor associations were calculated.")
    print("PASS — Longer engines did not dominate the life profiles.")
    print("PASS — Validation, calibration and test data remained unused.")
    print("PASS — No features were removed and no model was trained.")

    print("\nDownloading the Step 4A evidence ZIP...")
    colab_files.download(
        str(step_4a_evidence_zip)
    )

else:
    print("CHECK REQUIRED — At least one EDA check failed.")

In [ ]:
# ============================================================
# PHASE 2 — EXPLORATORY DATA ANALYSIS
# STEP 4A CORRECTION — ROBUST SENSOR ASSOCIATION SELECTION
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from google.colab import files as colab_files

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import zipfile


# ------------------------------------------------------------
# 1. Confirm that the required Step 4A objects exist
# ------------------------------------------------------------

required_objects = [
    "model_train_targets_df",
    "validation_targets_df",
    "calibration_targets_df",
    "eda_working_df"
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run the original Step 4A cell first. "
        "Missing objects: "
        + ", ".join(missing_objects)
    )


# ------------------------------------------------------------
# 2. Define output locations
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/Predictive_Maintenance_Thesis")
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
LOGS_DIR = PROJECT_ROOT / "logs"

for directory in [
    RESULTS_DIR,
    FIGURES_DIR,
    LOGS_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

recorded_at_utc = datetime.now(
    timezone.utc
).isoformat()


# ------------------------------------------------------------
# 3. Identify variable sensor columns
# ------------------------------------------------------------

all_sensor_columns = [
    column
    for column in model_train_targets_df.columns
    if column.startswith("sensor_")
]

constant_sensor_columns = [
    column
    for column in all_sensor_columns
    if (
        model_train_targets_df[
            column
        ].nunique(dropna=False) == 1
    )
]

variable_sensor_columns = [
    column
    for column in all_sensor_columns
    if column not in constant_sensor_columns
]


# ------------------------------------------------------------
# 4. Define warning-free Spearman correlation
# ------------------------------------------------------------

def calculate_safe_spearman(
    first_series,
    second_series
):
    """
    Calculate Spearman correlation through ranks.

    Return NaN when either ranked variable has no variation.
    This avoids divide-by-zero warnings and correctly records
    that correlation is undefined.
    """

    first_ranks = first_series.rank(
        method="average"
    )

    second_ranks = second_series.rank(
        method="average"
    )

    if (
        first_ranks.nunique(dropna=True) < 2
        or second_ranks.nunique(dropna=True) < 2
    ):
        return np.nan

    return float(
        np.corrcoef(
            first_ranks.to_numpy(dtype=float),
            second_ranks.to_numpy(dtype=float)
        )[0, 1]
    )


# ------------------------------------------------------------
# 5. Recalculate correlations inside each engine
# ------------------------------------------------------------

corrected_correlation_records = []

for sensor_column in variable_sensor_columns:

    for engine_id, engine_data in (
        model_train_targets_df.groupby(
            "unit_number",
            sort=True
        )
    ):
        correlation = calculate_safe_spearman(
            engine_data[sensor_column],
            engine_data["rul_uncapped"]
        )

        corrected_correlation_records.append({
            "unit_number": int(engine_id),
            "sensor": sensor_column,
            "spearman_sensor_vs_uncapped_rul":
                correlation
        })


corrected_engine_correlations_df = pd.DataFrame(
    corrected_correlation_records
)

corrected_engine_correlations_file = (
    RESULTS_DIR
    / "phase2_engine_sensor_correlations_corrected.csv"
)

corrected_engine_correlations_df.to_csv(
    corrected_engine_correlations_file,
    index=False
)


# ------------------------------------------------------------
# 6. Create a consistency-aware sensor summary
# ------------------------------------------------------------

corrected_summary_records = []

total_model_training_engines = int(
    model_train_targets_df[
        "unit_number"
    ].nunique()
)

for sensor_column in variable_sensor_columns:

    correlation_values = (
        corrected_engine_correlations_df.loc[
            corrected_engine_correlations_df[
                "sensor"
            ] == sensor_column,
            "spearman_sensor_vs_uncapped_rul"
        ]
        .dropna()
        .to_numpy(dtype=float)
    )

    finite_engine_count = int(
        len(correlation_values)
    )

    median_correlation = float(
        np.median(correlation_values)
    )

    first_quartile = float(
        np.quantile(
            correlation_values,
            0.25
        )
    )

    third_quartile = float(
        np.quantile(
            correlation_values,
            0.75
        )
    )

    positive_proportion = float(
        np.mean(correlation_values > 0)
    )

    negative_proportion = float(
        np.mean(correlation_values < 0)
    )

    dominant_direction_proportion = float(
        max(
            positive_proportion,
            negative_proportion
        )
    )

    interquartile_range_excludes_zero = bool(
        (
            first_quartile > 0
            and third_quartile > 0
        )
        or
        (
            first_quartile < 0
            and third_quartile < 0
        )
    )

    corrected_summary_records.append({
        "sensor": sensor_column,

        "engines_with_finite_correlation":
            finite_engine_count,

        "finite_engine_percentage": float(
            finite_engine_count
            / total_model_training_engines
            * 100
        ),

        "median_within_engine_spearman":
            median_correlation,

        "first_quartile_spearman":
            first_quartile,

        "third_quartile_spearman":
            third_quartile,

        "positive_correlation_proportion":
            positive_proportion,

        "negative_correlation_proportion":
            negative_proportion,

        "dominant_direction_proportion":
            dominant_direction_proportion,

        "interquartile_range_excludes_zero":
            interquartile_range_excludes_zero,

        "absolute_median_spearman":
            abs(median_correlation)
    })


corrected_sensor_summary_df = pd.DataFrame(
    corrected_summary_records
)

corrected_sensor_summary_df = (
    corrected_sensor_summary_df
    .sort_values(
        "absolute_median_spearman",
        ascending=False
    )
    .reset_index(drop=True)
)

corrected_sensor_summary_file = (
    RESULTS_DIR
    / "phase2_sensor_association_summary_corrected.csv"
)

corrected_sensor_summary_df.to_csv(
    corrected_sensor_summary_file,
    index=False
)


# ------------------------------------------------------------
# 7. Select directionally consistent sensors
# ------------------------------------------------------------

MINIMUM_FINITE_ENGINE_PERCENTAGE = 80.0
MINIMUM_DIRECTION_CONSISTENCY = 0.75
TOP_SENSOR_COUNT = 4

eligible_sensor_summary_df = (
    corrected_sensor_summary_df[
        (
            corrected_sensor_summary_df[
                "finite_engine_percentage"
            ]
            >= MINIMUM_FINITE_ENGINE_PERCENTAGE
        )
        &
        (
            corrected_sensor_summary_df[
                "dominant_direction_proportion"
            ]
            >= MINIMUM_DIRECTION_CONSISTENCY
        )
        &
        (
            corrected_sensor_summary_df[
                "interquartile_range_excludes_zero"
            ]
        )
    ]
    .copy()
)

corrected_top_sensor_columns = (
    eligible_sensor_summary_df
    .head(TOP_SENSOR_COUNT)["sensor"]
    .tolist()
)

if len(corrected_top_sensor_columns) != 4:
    raise RuntimeError(
        "Fewer than four sensors met the "
        "predefined consistency criteria."
    )


# ------------------------------------------------------------
# 8. Standardise corrected selected sensors for EDA only
# ------------------------------------------------------------

corrected_eda_df = (
    eda_working_df[
        [
            "unit_number",
            "time_in_cycles",
            "normalised_life_fraction",
            "life_stage_bin"
        ]
        + corrected_top_sensor_columns
    ]
    .copy()
)

corrected_standardisation_records = []

for sensor_column in corrected_top_sensor_columns:

    sensor_mean = float(
        corrected_eda_df[
            sensor_column
        ].mean()
    )

    sensor_standard_deviation = float(
        corrected_eda_df[
            sensor_column
        ].std(ddof=0)
    )

    standardised_column = (
        f"{sensor_column}_eda_zscore"
    )

    corrected_eda_df[
        standardised_column
    ] = (
        corrected_eda_df[sensor_column]
        - sensor_mean
    ) / sensor_standard_deviation

    corrected_standardisation_records.append({
        "sensor": sensor_column,
        "mean_from_model_training_rows":
            sensor_mean,
        "standard_deviation_from_model_training_rows":
            sensor_standard_deviation,
        "purpose": "EDA visualisation only"
    })


corrected_standardisation_df = pd.DataFrame(
    corrected_standardisation_records
)

corrected_standardisation_file = (
    LOGS_DIR
    / "phase2_eda_sensor_standardisation_corrected.csv"
)

corrected_standardisation_df.to_csv(
    corrected_standardisation_file,
    index=False
)


# ------------------------------------------------------------
# 9. Create equal-engine-weight life profiles
# ------------------------------------------------------------

corrected_standardised_columns = [
    f"{sensor}_eda_zscore"
    for sensor in corrected_top_sensor_columns
]

corrected_within_engine_profiles_df = (
    corrected_eda_df
    .groupby(
        [
            "unit_number",
            "life_stage_bin"
        ],
        observed=True
    )[
        corrected_standardised_columns
    ]
    .median()
    .reset_index()
)

corrected_profile_records = []

for sensor_column in corrected_top_sensor_columns:

    standardised_column = (
        f"{sensor_column}_eda_zscore"
    )

    sensor_profile = (
        corrected_within_engine_profiles_df
        .groupby(
            "life_stage_bin",
            observed=True
        )[standardised_column]
        .agg(
            median_standardised_value="median",
            first_quartile_standardised_value=(
                lambda values:
                values.quantile(0.25)
            ),
            third_quartile_standardised_value=(
                lambda values:
                values.quantile(0.75)
            ),
            contributing_engines="count"
        )
        .reset_index()
    )

    sensor_profile["sensor"] = sensor_column

    sensor_profile[
        "life_bin_midpoint_percent"
    ] = (
        (
            sensor_profile["life_stage_bin"]
            + 0.5
        )
        / 20
        * 100
    )

    corrected_profile_records.append(
        sensor_profile
    )


corrected_sensor_profiles_df = pd.concat(
    corrected_profile_records,
    ignore_index=True
)

corrected_sensor_profiles_file = (
    RESULTS_DIR
    / "phase2_consistent_sensor_life_profiles.csv"
)

corrected_sensor_profiles_df.to_csv(
    corrected_sensor_profiles_file,
    index=False
)


# ------------------------------------------------------------
# 10. Corrected Figure 2 — median with engine IQR
# ------------------------------------------------------------

plt.style.use("seaborn-v0_8-whitegrid")

BLUE = "#2563EB"
LIGHT_BLUE = "#93C5FD"
ORANGE = "#D97706"
DARK_GREY = "#374151"

plot_summary_df = (
    corrected_sensor_summary_df
    .sort_values(
        "median_within_engine_spearman"
    )
    .reset_index(drop=True)
)

median_values = plot_summary_df[
    "median_within_engine_spearman"
].to_numpy(dtype=float)

lower_errors = (
    median_values
    - plot_summary_df[
        "first_quartile_spearman"
    ].to_numpy(dtype=float)
)

upper_errors = (
    plot_summary_df[
        "third_quartile_spearman"
    ].to_numpy(dtype=float)
    - median_values
)

point_colours = [
    BLUE if value >= 0 else ORANGE
    for value in median_values
]

figure_2_corrected, axis_2_corrected = (
    plt.subplots(figsize=(11, 7.5))
)

for row_index in range(
    len(plot_summary_df)
):
    axis_2_corrected.errorbar(
        x=median_values[row_index],
        y=row_index,
        xerr=np.array([
            [lower_errors[row_index]],
            [upper_errors[row_index]]
        ]),
        fmt="o",
        color=point_colours[row_index],
        ecolor=LIGHT_BLUE,
        elinewidth=2,
        capsize=3,
        markersize=6
    )

axis_2_corrected.axvline(
    0,
    color=DARK_GREY,
    linewidth=1.2,
    linestyle="--"
)

axis_2_corrected.set_yticks(
    range(len(plot_summary_df))
)

axis_2_corrected.set_yticklabels(
    plot_summary_df["sensor"]
)

axis_2_corrected.set_xlim(-1.05, 1.05)

axis_2_corrected.set_title(
    "Engine-aware sensor associations with uncapped RUL",
    fontsize=14,
    fontweight="bold"
)

axis_2_corrected.set_xlabel(
    "Median within-engine Spearman correlation "
    "(lines show engine IQR)"
)

axis_2_corrected.set_ylabel(
    "Anonymised sensor channel"
)

figure_2_corrected.text(
    0.5,
    0.01,
    (
        "FD001 model-training engines only. "
        "Undefined correlations from locally constant "
        "sensor sequences are excluded and counted."
    ),
    ha="center",
    fontsize=9
)

figure_2_corrected.tight_layout(
    rect=[0, 0.05, 1, 0.96]
)

corrected_association_png = (
    FIGURES_DIR
    / "phase2_engine_aware_sensor_associations_corrected.png"
)

corrected_association_pdf = (
    FIGURES_DIR
    / "phase2_engine_aware_sensor_associations_corrected.pdf"
)

figure_2_corrected.savefig(
    corrected_association_png,
    dpi=300,
    bbox_inches="tight"
)

figure_2_corrected.savefig(
    corrected_association_pdf,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 11. Corrected Figure 3 — consistent sensor profiles
# ------------------------------------------------------------

figure_3_corrected, corrected_axes = (
    plt.subplots(
        nrows=2,
        ncols=2,
        figsize=(13, 9),
        sharex=True
    )
)

for axis, sensor_column in zip(
    corrected_axes.flatten(),
    corrected_top_sensor_columns
):
    sensor_profile = (
        corrected_sensor_profiles_df[
            corrected_sensor_profiles_df[
                "sensor"
            ] == sensor_column
        ]
        .sort_values("life_stage_bin")
    )

    x_values = sensor_profile[
        "life_bin_midpoint_percent"
    ].to_numpy(dtype=float)

    median_profile = sensor_profile[
        "median_standardised_value"
    ].to_numpy(dtype=float)

    lower_profile = sensor_profile[
        "first_quartile_standardised_value"
    ].to_numpy(dtype=float)

    upper_profile = sensor_profile[
        "third_quartile_standardised_value"
    ].to_numpy(dtype=float)

    axis.fill_between(
        x_values,
        lower_profile,
        upper_profile,
        color=LIGHT_BLUE,
        alpha=0.45,
        label="Engine IQR"
    )

    axis.plot(
        x_values,
        median_profile,
        color=BLUE,
        linewidth=2.2,
        marker="o",
        markersize=3,
        label="Engine median"
    )

    axis.axhline(
        0,
        color=DARK_GREY,
        linestyle=":",
        linewidth=1
    )

    axis.set_title(sensor_column)
    axis.set_xlabel(
        "Normalised engine-life position (%)"
    )
    axis.set_ylabel(
        "Standardised sensor value"
    )
    axis.legend(
        loc="best",
        fontsize=8
    )


figure_3_corrected.suptitle(
    "Life-stage profiles of four directionally consistent sensors",
    fontsize=14,
    fontweight="bold"
)

figure_3_corrected.text(
    0.5,
    0.01,
    (
        "Selected using model-training engines only. "
        "Each engine contributes once per life-stage bin; "
        "shading shows the interquartile range."
    ),
    ha="center",
    fontsize=9
)

figure_3_corrected.tight_layout(
    rect=[0, 0.04, 1, 0.94]
)

corrected_profiles_png = (
    FIGURES_DIR
    / "phase2_consistent_sensor_life_profiles.png"
)

corrected_profiles_pdf = (
    FIGURES_DIR
    / "phase2_consistent_sensor_life_profiles.pdf"
)

figure_3_corrected.savefig(
    corrected_profiles_png,
    dpi=300,
    bbox_inches="tight"
)

figure_3_corrected.savefig(
    corrected_profiles_pdf,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 12. Validate the correction
# ------------------------------------------------------------

selected_summary_df = (
    corrected_sensor_summary_df[
        corrected_sensor_summary_df[
            "sensor"
        ].isin(corrected_top_sensor_columns)
    ]
)

corrected_profile_bin_counts = (
    corrected_sensor_profiles_df
    .groupby("sensor")[
        "life_stage_bin"
    ]
    .nunique()
)

correction_checks = {
    "four_consistent_sensors_selected":
        len(corrected_top_sensor_columns) == 4,

    "selected_sensor_iqrs_exclude_zero":
        bool(
            selected_summary_df[
                "interquartile_range_excludes_zero"
            ].all()
        ),

    "selected_sensors_meet_direction_threshold":
        bool(
            (
                selected_summary_df[
                    "dominant_direction_proportion"
                ]
                >= MINIMUM_DIRECTION_CONSISTENCY
            ).all()
        ),

    "selected_sensors_meet_finite_engine_threshold":
        bool(
            (
                selected_summary_df[
                    "finite_engine_percentage"
                ]
                >= MINIMUM_FINITE_ENGINE_PERCENTAGE
            ).all()
        ),

    "each_selected_sensor_has_20_life_bins":
        bool(
            (
                corrected_profile_bin_counts
                == 20
            ).all()
        ),

    "sensor_6_undefined_correlations_are_documented":
        int(
            corrected_sensor_summary_df.loc[
                corrected_sensor_summary_df[
                    "sensor"
                ] == "sensor_6",
                "engines_with_finite_correlation"
            ].iloc[0]
        ) < total_model_training_engines,

    "validation_data_remain_unused":
        set(
            model_train_targets_df[
                "unit_number"
            ].unique()
        ).isdisjoint(
            set(
                validation_targets_df[
                    "unit_number"
                ].unique()
            )
        ),

    "calibration_data_remain_unused":
        set(
            model_train_targets_df[
                "unit_number"
            ].unique()
        ).isdisjoint(
            set(
                calibration_targets_df[
                    "unit_number"
                ].unique()
            )
        ),

    "corrected_figures_were_saved":
        all(
            path.exists()
            for path in [
                corrected_association_png,
                corrected_association_pdf,
                corrected_profiles_png,
                corrected_profiles_pdf
            ]
        )
}


# ------------------------------------------------------------
# 13. Record why the correction was required
# ------------------------------------------------------------

correction_record = {
    "recorded_at_utc": recorded_at_utc,
    "step": "Phase 2 Step 4A correction",
    "reason": [
        (
            "Locally constant sensor sequences caused "
            "undefined within-engine correlations and "
            "NumPy divide-by-zero warnings."
        ),
        (
            "The original absolute-median ranking selected "
            "sensor_9 even though its engine-level "
            "interquartile interval crossed zero."
        )
    ],
    "correction": [
        (
            "Undefined correlations are now detected "
            "before calculation and recorded as missing."
        ),
        (
            "Selected visualisation sensors must have "
            "finite correlations for at least 80% of engines."
        ),
        (
            "At least 75% of finite engine correlations "
            "must share one direction."
        ),
        (
            "The engine-level correlation IQR must not "
            "cross zero."
        )
    ],
    "corrected_selected_sensors":
        corrected_top_sensor_columns,
    "superseded_outputs": [
        "phase2_engine_aware_sensor_associations.png",
        "phase2_engine_aware_sensor_associations.pdf",
        "phase2_top_sensor_life_stage_profiles.png",
        "phase2_top_sensor_life_stage_profiles.pdf"
    ],
    "retained_valid_output": (
        "phase2_model_training_lifetime_distribution"
    ),
    "features_selected_for_modelling": False,
    "validation_engines_used": False,
    "calibration_engines_used": False,
    "official_test_data_used": False,
    "correction_checks": correction_checks
}

correction_record_file = (
    LOGS_DIR
    / "phase2_step4a_sensor_eda_correction.json"
)

with correction_record_file.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        correction_record,
        file,
        indent=4
    )


# ------------------------------------------------------------
# 14. Package corrected evidence
# ------------------------------------------------------------

corrected_evidence_zip = (
    PROJECT_ROOT
    / "Phase2_Step4A_Corrected_Sensor_EDA_Evidence.zip"
)

with zipfile.ZipFile(
    corrected_evidence_zip,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as zip_file:

    for evidence_file in [
        corrected_engine_correlations_file,
        corrected_sensor_summary_file,
        corrected_standardisation_file,
        corrected_sensor_profiles_file,
        correction_record_file,
        corrected_association_png,
        corrected_association_pdf,
        corrected_profiles_png,
        corrected_profiles_pdf
    ]:
        zip_file.write(
            evidence_file,
            arcname=evidence_file.name
        )


# ------------------------------------------------------------
# 15. Display corrected results
# ------------------------------------------------------------

print("CORRECTED SENSOR ASSOCIATION SUMMARY")
print("=" * 105)

print(
    corrected_sensor_summary_df[
        [
            "sensor",
            "engines_with_finite_correlation",
            "median_within_engine_spearman",
            "first_quartile_spearman",
            "third_quartile_spearman",
            "dominant_direction_proportion",
            "interquartile_range_excludes_zero"
        ]
    ].to_string(index=False)
)

print("\nCORRECTED SELECTED SENSORS")
print("=" * 105)
print(corrected_top_sensor_columns)

print("\nCORRECTION VALIDATION")
print("=" * 105)

for check_name, passed in correction_checks.items():
    status = "PASS" if passed else "FAIL"
    print(
        f"{status} — "
        f"{check_name.replace('_', ' ')}"
    )

print("\nFINAL CHECK")
print("=" * 105)

if all(correction_checks.values()):
    print("PASS — Undefined correlations were handled explicitly.")
    print("PASS — Four directionally consistent sensors were selected.")
    print("PASS — Corrected association and profile figures were saved.")
    print("PASS — Validation, calibration and test data remained unused.")
    print("PASS — No modelling feature selection was performed.")

    print("\nDownloading the corrected evidence ZIP...")
    colab_files.download(
        str(corrected_evidence_zip)
    )

else:
    print("CHECK REQUIRED — The EDA correction did not pass.")

In [ ]:
# ============================================================
# PHASE 2 — EXPLORATORY DATA ANALYSIS
# STEP 4B — ENGINE-AWARE FEATURE REDUNDANCY AUDIT
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from itertools import combinations
from google.colab import files as colab_files
from matplotlib.colors import LinearSegmentedColormap

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import zipfile


# ------------------------------------------------------------
# 1. Confirm that the required datasets exist
# ------------------------------------------------------------

required_dataframe_names = [
    "model_train_targets_df",
    "validation_targets_df",
    "calibration_targets_df"
]

missing_dataframes = [
    name
    for name in required_dataframe_names
    if name not in globals()
]

if missing_dataframes:
    raise RuntimeError(
        "Required frozen dataframes are missing: "
        + ", ".join(missing_dataframes)
    )


# ------------------------------------------------------------
# 2. Define controlled output locations
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/Predictive_Maintenance_Thesis")
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
LOGS_DIR = PROJECT_ROOT / "logs"

for directory in [
    RESULTS_DIR,
    FIGURES_DIR,
    LOGS_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

recorded_at_utc = datetime.now(
    timezone.utc
).isoformat()


# ------------------------------------------------------------
# 3. Record approval of corrected Step 4A
# ------------------------------------------------------------

step_4a_closure_record = {
    "recorded_at_utc": recorded_at_utc,
    "step": (
        "Phase 2 Step 4A — "
        "Training-only lifetime and sensor EDA"
    ),
    "status": "Approved after documented correction",
    "corrected_selected_sensors": [
        "sensor_11",
        "sensor_12",
        "sensor_4",
        "sensor_7"
    ],
    "superseding_evidence": (
        "Phase2_Step4A_Corrected_"
        "Sensor_EDA_Evidence.zip"
    ),
    "features_selected_for_modelling": False,
    "validation_engines_used": False,
    "calibration_engines_used": False,
    "official_test_data_used": False
}

step_4a_closure_file = (
    LOGS_DIR
    / "phase2_step4a_corrected_closure.json"
)

with step_4a_closure_file.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        step_4a_closure_record,
        file,
        indent=4
    )


# ------------------------------------------------------------
# 4. Define measurement-feature columns
# ------------------------------------------------------------

operational_setting_columns = [
    "operational_setting_1",
    "operational_setting_2",
    "operational_setting_3"
]

sensor_columns = [
    f"sensor_{sensor_number}"
    for sensor_number in range(1, 22)
]

measurement_feature_columns = (
    operational_setting_columns
    + sensor_columns
)


# ------------------------------------------------------------
# 5. Create a complete feature-variation profile
# ------------------------------------------------------------

feature_profile_records = []

for feature_column in measurement_feature_columns:

    feature_values = (
        model_train_targets_df[
            feature_column
        ]
    )

    value_proportions = (
        feature_values.value_counts(
            normalize=True,
            dropna=False
        )
    )

    unique_value_count = int(
        feature_values.nunique(
            dropna=False
        )
    )

    dominant_value_ratio = float(
        value_proportions.iloc[0]
    )

    feature_profile_records.append({
        "feature": feature_column,

        "feature_family": (
            "operational_setting"
            if feature_column.startswith(
                "operational_setting_"
            )
            else "sensor"
        ),

        "unique_value_count":
            unique_value_count,

        "minimum": float(
            feature_values.min()
        ),

        "first_quartile": float(
            feature_values.quantile(0.25)
        ),

        "median": float(
            feature_values.median()
        ),

        "third_quartile": float(
            feature_values.quantile(0.75)
        ),

        "maximum": float(
            feature_values.max()
        ),

        "mean": float(
            feature_values.mean()
        ),

        "standard_deviation": float(
            feature_values.std(ddof=0)
        ),

        "dominant_value_ratio":
            dominant_value_ratio,

        "is_constant":
            unique_value_count == 1,

        "is_near_constant_99_percent":
            bool(
                unique_value_count > 1
                and dominant_value_ratio >= 0.99
            )
    })


feature_variation_profile_df = pd.DataFrame(
    feature_profile_records
)

feature_variation_profile_file = (
    RESULTS_DIR
    / "phase2_feature_variation_profile.csv"
)

feature_variation_profile_df.to_csv(
    feature_variation_profile_file,
    index=False
)


# ------------------------------------------------------------
# 6. Separate constant and variable measurements
# ------------------------------------------------------------

constant_measurement_features = (
    feature_variation_profile_df.loc[
        feature_variation_profile_df[
            "is_constant"
        ],
        "feature"
    ].tolist()
)

variable_measurement_features = (
    feature_variation_profile_df.loc[
        ~feature_variation_profile_df[
            "is_constant"
        ],
        "feature"
    ].tolist()
)


# ------------------------------------------------------------
# 7. Define warning-free Spearman calculation
# ------------------------------------------------------------

def calculate_safe_spearman(
    first_series,
    second_series
):
    """
    Calculate correlation between ranked variables.

    Return NaN when either variable is locally constant
    within the relevant engine.
    """

    first_ranks = first_series.rank(
        method="average"
    )

    second_ranks = second_series.rank(
        method="average"
    )

    if (
        first_ranks.nunique(dropna=True) < 2
        or second_ranks.nunique(dropna=True) < 2
    ):
        return np.nan

    return float(
        np.corrcoef(
            first_ranks.to_numpy(dtype=float),
            second_ranks.to_numpy(dtype=float)
        )[0, 1]
    )


# ------------------------------------------------------------
# 8. Calculate every feature-pair correlation by engine
# ------------------------------------------------------------

pairwise_engine_records = []

feature_pairs = list(
    combinations(
        variable_measurement_features,
        2
    )
)

for first_feature, second_feature in feature_pairs:

    for engine_id, engine_data in (
        model_train_targets_df.groupby(
            "unit_number",
            sort=True
        )
    ):
        correlation = calculate_safe_spearman(
            engine_data[first_feature],
            engine_data[second_feature]
        )

        pairwise_engine_records.append({
            "unit_number": int(engine_id),
            "feature_1": first_feature,
            "feature_2": second_feature,
            "within_engine_spearman":
                correlation
        })


pairwise_engine_correlations_df = pd.DataFrame(
    pairwise_engine_records
)

pairwise_engine_correlations_file = (
    RESULTS_DIR
    / "phase2_pairwise_engine_feature_correlations.csv"
)

pairwise_engine_correlations_df.to_csv(
    pairwise_engine_correlations_file,
    index=False
)


# ------------------------------------------------------------
# 9. Summarise each feature pair across engines
# ------------------------------------------------------------

pairwise_summary_records = []

total_training_engines = int(
    model_train_targets_df[
        "unit_number"
    ].nunique()
)

for first_feature, second_feature in feature_pairs:

    pair_values = (
        pairwise_engine_correlations_df.loc[
            (
                pairwise_engine_correlations_df[
                    "feature_1"
                ] == first_feature
            )
            &
            (
                pairwise_engine_correlations_df[
                    "feature_2"
                ] == second_feature
            ),
            "within_engine_spearman"
        ]
        .dropna()
        .to_numpy(dtype=float)
    )

    finite_engine_count = int(
        len(pair_values)
    )

    median_correlation = float(
        np.median(pair_values)
    )

    first_quartile = float(
        np.quantile(
            pair_values,
            0.25
        )
    )

    third_quartile = float(
        np.quantile(
            pair_values,
            0.75
        )
    )

    positive_proportion = float(
        np.mean(pair_values > 0)
    )

    negative_proportion = float(
        np.mean(pair_values < 0)
    )

    dominant_direction_proportion = float(
        max(
            positive_proportion,
            negative_proportion
        )
    )

    interquartile_range_excludes_zero = bool(
        (
            first_quartile > 0
            and third_quartile > 0
        )
        or
        (
            first_quartile < 0
            and third_quartile < 0
        )
    )

    pairwise_summary_records.append({
        "feature_1": first_feature,

        "feature_2": second_feature,

        "engines_with_finite_correlation":
            finite_engine_count,

        "finite_engine_percentage": float(
            finite_engine_count
            / total_training_engines
            * 100
        ),

        "median_within_engine_spearman":
            median_correlation,

        "absolute_median_spearman":
            abs(median_correlation),

        "first_quartile_spearman":
            first_quartile,

        "third_quartile_spearman":
            third_quartile,

        "dominant_direction_proportion":
            dominant_direction_proportion,

        "interquartile_range_excludes_zero":
            interquartile_range_excludes_zero
    })


pairwise_summary_df = pd.DataFrame(
    pairwise_summary_records
)

pairwise_summary_df = (
    pairwise_summary_df
    .sort_values(
        "absolute_median_spearman",
        ascending=False
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 10. Predefine strong redundancy-candidate rules
# ------------------------------------------------------------

STRONG_CORRELATION_THRESHOLD = 0.90
MINIMUM_FINITE_ENGINE_PERCENTAGE = 80.0
MINIMUM_DIRECTION_CONSISTENCY = 0.75

pairwise_summary_df[
    "strong_redundancy_candidate"
] = (
    (
        pairwise_summary_df[
            "absolute_median_spearman"
        ]
        >= STRONG_CORRELATION_THRESHOLD
    )
    &
    (
        pairwise_summary_df[
            "finite_engine_percentage"
        ]
        >= MINIMUM_FINITE_ENGINE_PERCENTAGE
    )
    &
    (
        pairwise_summary_df[
            "dominant_direction_proportion"
        ]
        >= MINIMUM_DIRECTION_CONSISTENCY
    )
    &
    (
        pairwise_summary_df[
            "interquartile_range_excludes_zero"
        ]
    )
)

strong_redundancy_candidates_df = (
    pairwise_summary_df[
        pairwise_summary_df[
            "strong_redundancy_candidate"
        ]
    ]
    .copy()
)

pairwise_summary_file = (
    RESULTS_DIR
    / "phase2_pairwise_feature_summary.csv"
)

strong_candidates_file = (
    RESULTS_DIR
    / "phase2_strong_redundancy_candidates.csv"
)

pairwise_summary_df.to_csv(
    pairwise_summary_file,
    index=False
)

strong_redundancy_candidates_df.to_csv(
    strong_candidates_file,
    index=False
)


# ------------------------------------------------------------
# 11. Construct the symmetric median-correlation matrix
# ------------------------------------------------------------

median_correlation_matrix_df = pd.DataFrame(
    np.eye(
        len(variable_measurement_features)
    ),
    index=variable_measurement_features,
    columns=variable_measurement_features
)

for _, pair_record in (
    pairwise_summary_df.iterrows()
):
    first_feature = pair_record[
        "feature_1"
    ]

    second_feature = pair_record[
        "feature_2"
    ]

    median_correlation = pair_record[
        "median_within_engine_spearman"
    ]

    median_correlation_matrix_df.loc[
        first_feature,
        second_feature
    ] = median_correlation

    median_correlation_matrix_df.loc[
        second_feature,
        first_feature
    ] = median_correlation


median_correlation_matrix_file = (
    RESULTS_DIR
    / "phase2_engine_aware_median_correlation_matrix.csv"
)

median_correlation_matrix_df.to_csv(
    median_correlation_matrix_file
)


# ------------------------------------------------------------
# 12. Create an engine-aware correlation heatmap
# ------------------------------------------------------------

plt.style.use("seaborn-v0_8-whitegrid")

correlation_colour_map = (
    LinearSegmentedColormap.from_list(
        "correlation_palette",
        [
            "#D97706",
            "#F9FAFB",
            "#2563EB"
        ]
    )
)

figure, axis = plt.subplots(
    figsize=(12, 10)
)

heatmap_image = axis.imshow(
    median_correlation_matrix_df.to_numpy(
        dtype=float
    ),
    cmap=correlation_colour_map,
    vmin=-1,
    vmax=1,
    aspect="auto"
)

axis.set_xticks(
    np.arange(
        len(variable_measurement_features)
    )
)

axis.set_yticks(
    np.arange(
        len(variable_measurement_features)
    )
)

axis.set_xticklabels(
    variable_measurement_features,
    rotation=55,
    ha="right"
)

axis.set_yticklabels(
    variable_measurement_features
)

axis.set_title(
    "Engine-aware median feature correlations",
    fontsize=14,
    fontweight="bold"
)

colour_bar = figure.colorbar(
    heatmap_image,
    ax=axis,
    fraction=0.046,
    pad=0.04
)

colour_bar.set_label(
    "Median within-engine Spearman correlation"
)

figure.text(
    0.5,
    0.01,
    (
        "FD001 model-training engines only. "
        "Each cell is the median of correlations "
        "calculated separately inside each engine."
    ),
    ha="center",
    fontsize=9
)

figure.tight_layout(
    rect=[0, 0.04, 1, 0.96]
)

heatmap_png_file = (
    FIGURES_DIR
    / "phase2_engine_aware_feature_correlation_heatmap.png"
)

heatmap_pdf_file = (
    FIGURES_DIR
    / "phase2_engine_aware_feature_correlation_heatmap.pdf"
)

figure.savefig(
    heatmap_png_file,
    dpi=300,
    bbox_inches="tight"
)

figure.savefig(
    heatmap_pdf_file,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 13. Validate the redundancy audit
# ------------------------------------------------------------

expected_pair_count = int(
    len(variable_measurement_features)
    * (
        len(variable_measurement_features) - 1
    )
    / 2
)

redundancy_checks = {
    "twenty_four_measurement_features_profiled":
        len(
            feature_variation_profile_df
        ) == 24,

    "seven_constant_measurement_features_found":
        len(
            constant_measurement_features
        ) == 7,

    "seventeen_variable_measurement_features_found":
        len(
            variable_measurement_features
        ) == 17,

    "all_expected_variable_feature_pairs_summarised":
        len(
            pairwise_summary_df
        ) == expected_pair_count,

    "correlation_matrix_is_seventeen_by_seventeen":
        median_correlation_matrix_df.shape
        == (17, 17),

    "correlation_matrix_is_symmetric":
        bool(
            np.allclose(
                median_correlation_matrix_df.to_numpy(
                    dtype=float
                ),
                median_correlation_matrix_df.to_numpy(
                    dtype=float
                ).T,
                equal_nan=True
            )
        ),

    "correlation_matrix_diagonal_is_one":
        bool(
            np.allclose(
                np.diag(
                    median_correlation_matrix_df.to_numpy(
                        dtype=float
                    )
                ),
                1.0
            )
        ),

    "all_pairwise_medians_are_finite":
        bool(
            np.isfinite(
                pairwise_summary_df[
                    "median_within_engine_spearman"
                ]
            ).all()
        ),

    "only_model_training_engines_were_used":
        set(
            pairwise_engine_correlations_df[
                "unit_number"
            ].unique()
        )
        == set(
            model_train_targets_df[
                "unit_number"
            ].unique()
        ),

    "validation_engines_remain_unused":
        set(
            model_train_targets_df[
                "unit_number"
            ].unique()
        ).isdisjoint(
            set(
                validation_targets_df[
                    "unit_number"
                ].unique()
            )
        ),

    "calibration_engines_remain_unused":
        set(
            model_train_targets_df[
                "unit_number"
            ].unique()
        ).isdisjoint(
            set(
                calibration_targets_df[
                    "unit_number"
                ].unique()
            )
        ),

    "heatmap_was_saved":
        (
            heatmap_png_file.exists()
            and heatmap_pdf_file.exists()
        )
}


# ------------------------------------------------------------
# 14. Create the formal redundancy-audit record
# ------------------------------------------------------------

redundancy_record = {
    "recorded_at_utc": recorded_at_utc,

    "phase": "Phase 2",

    "step": (
        "Step 4B — Engine-aware "
        "feature redundancy audit"
    ),

    "data_scope": (
        "60 frozen model-training engines only"
    ),

    "measurement_feature_count":
        len(measurement_feature_columns),

    "constant_measurement_features":
        constant_measurement_features,

    "variable_measurement_features":
        variable_measurement_features,

    "feature_pair_count":
        len(pairwise_summary_df),

    "strong_redundancy_candidate_definition": {
        "absolute_median_spearman_at_least":
            STRONG_CORRELATION_THRESHOLD,

        "finite_engine_percentage_at_least":
            MINIMUM_FINITE_ENGINE_PERCENTAGE,

        "dominant_direction_proportion_at_least":
            MINIMUM_DIRECTION_CONSISTENCY,

        "engine_correlation_iqr_must_exclude_zero":
            True
    },

    "strong_redundancy_candidate_count":
        len(
            strong_redundancy_candidates_df
        ),

    "interpretation_policy": (
        "Strong correlation identifies possible "
        "redundancy but does not authorise feature removal."
    ),

    "important_limitations": [
        (
            "Correlation does not establish physical "
            "equivalence or causation."
        ),
        (
            "Anonymised features are not assigned invented "
            "physical meanings."
        ),
        (
            "Pairwise association does not capture every "
            "multivariable or nonlinear relationship."
        ),
        (
            "No feature is removed during exploratory analysis."
        )
    ],

    "features_removed": False,
    "models_trained": False,
    "validation_engines_used": False,
    "calibration_engines_used": False,
    "official_test_data_used": False,
    "test_rul_values_used": False,
    "redundancy_checks": redundancy_checks
}

redundancy_record_file = (
    LOGS_DIR
    / "phase2_feature_redundancy_audit.json"
)

with redundancy_record_file.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        redundancy_record,
        file,
        indent=4
    )


# ------------------------------------------------------------
# 15. Package the evidence
# ------------------------------------------------------------

step_4b_evidence_zip = (
    PROJECT_ROOT
    / "Phase2_Step4B_Feature_Redundancy_Evidence.zip"
)

with zipfile.ZipFile(
    step_4b_evidence_zip,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as zip_file:

    for evidence_file in [
        step_4a_closure_file,
        feature_variation_profile_file,
        pairwise_engine_correlations_file,
        pairwise_summary_file,
        strong_candidates_file,
        median_correlation_matrix_file,
        redundancy_record_file,
        heatmap_png_file,
        heatmap_pdf_file
    ]:
        zip_file.write(
            evidence_file,
            arcname=evidence_file.name
        )


# ------------------------------------------------------------
# 16. Display the findings
# ------------------------------------------------------------

print("FEATURE VARIATION SUMMARY")
print("=" * 105)
print(
    f"Measurement features profiled: "
    f"{len(measurement_feature_columns)}"
)
print(
    f"Constant features: "
    f"{constant_measurement_features}"
)
print(
    f"Variable features: "
    f"{len(variable_measurement_features)}"
)

print("\nSTRONG REDUNDANCY CANDIDATES")
print("=" * 105)

if len(
    strong_redundancy_candidates_df
) == 0:
    print(
        "No feature pairs met every predefined threshold."
    )
else:
    print(
        strong_redundancy_candidates_df[
            [
                "feature_1",
                "feature_2",
                "engines_with_finite_correlation",
                "median_within_engine_spearman",
                "first_quartile_spearman",
                "third_quartile_spearman",
                "dominant_direction_proportion"
            ]
        ].to_string(index=False)
    )

print("\nREDUNDANCY VALIDATION")
print("=" * 105)

for check_name, passed in (
    redundancy_checks.items()
):
    status = "PASS" if passed else "FAIL"

    print(
        f"{status} — "
        f"{check_name.replace('_', ' ')}"
    )

print("\nRESEARCH CONTROLS")
print("=" * 105)
print("Feature removal performed: No")
print("Validation engines used: No")
print("Calibration engines used: No")
print("Official test data used: No")
print("Test RUL values accessed: No")
print("Models trained: No")

print("\nFINAL CHECK")
print("=" * 105)

if all(redundancy_checks.values()):
    print("PASS — Feature redundancy audit completed.")
    print("PASS — Correlations were calculated inside engines.")
    print("PASS — Longer engines did not dominate the audit.")
    print("PASS — Potential redundancy was documented only.")
    print("PASS — No feature was removed and no model was trained.")

    print("\nDownloading the Step 4B evidence ZIP...")
    colab_files.download(
        str(step_4b_evidence_zip)
    )

else:
    print(
        "CHECK REQUIRED — "
        "At least one redundancy check failed."
    )

In [ ]:
# ============================================================
# PHASE 2 — COMPLETION AND GOVERNANCE
# STEP 5 — CONSOLIDATE PHASE 2 EVIDENCE
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from google.colab import files as colab_files

import pandas as pd
import hashlib
import json
import zipfile


# ------------------------------------------------------------
# 1. Define project locations
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/Predictive_Maintenance_Thesis")
INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
LOGS_DIR = PROJECT_ROOT / "logs"

recorded_at_utc = datetime.now(
    timezone.utc
).isoformat()


# ------------------------------------------------------------
# 2. Confirm that all important in-memory objects exist
# ------------------------------------------------------------

required_object_names = [
    "MODEL_TRAIN_ENGINE_IDS",
    "VALIDATION_ENGINE_IDS",
    "CALIBRATION_ENGINE_IDS",
    "model_train_targets_df",
    "validation_targets_df",
    "calibration_targets_df",
    "corrected_top_sensor_columns",
    "constant_measurement_features",
    "strong_redundancy_candidates_df"
]

missing_objects = [
    object_name
    for object_name in required_object_names
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Important Phase 2 objects are missing: "
        + ", ".join(missing_objects)
    )


# ------------------------------------------------------------
# 3. Create the reproducibility and recovery configuration
# ------------------------------------------------------------

phase_2_recovery_configuration = {
    "recorded_at_utc": recorded_at_utc,

    "dataset": "NASA C-MAPSS FD001",

    "approved_scope": "FD001 only",

    "random_seed": 42,

    "engine_partition_policy": {
        "model_training_engine_count":
            len(MODEL_TRAIN_ENGINE_IDS),

        "validation_engine_count":
            len(VALIDATION_ENGINE_IDS),

        "conformal_calibration_engine_count":
            len(CALIBRATION_ENGINE_IDS),

        "model_training_engine_ids": [
            int(engine_id)
            for engine_id in MODEL_TRAIN_ENGINE_IDS
        ],

        "validation_engine_ids": [
            int(engine_id)
            for engine_id in VALIDATION_ENGINE_IDS
        ],

        "conformal_calibration_engine_ids": [
            int(engine_id)
            for engine_id in CALIBRATION_ENGINE_IDS
        ]
    },

    "target_policy": {
        "primary_target": "rul_uncapped",
        "primary_formula": (
            "failure cycle - current cycle"
        ),
        "sensitivity_target":
            "rul_capped_130",
        "sensitivity_formula": (
            "minimum of uncapped RUL and 130"
        ),
        "sensitivity_cap_cycles": 130,
        "nasa_prescribed_cap": False
    },

    "exploratory_findings": {
        "constant_measurement_features":
            constant_measurement_features,

        "corrected_directionally_consistent_sensors":
            corrected_top_sensor_columns,

        "strong_redundancy_candidate_count":
            int(
                len(
                    strong_redundancy_candidates_df
                )
            )
    },

    "governance_controls": {
        "raw_files_modified": False,
        "official_test_outcomes_used": False,
        "test_rul_values_used": False,
        "features_removed": False,
        "models_trained": False
    }
}

recovery_configuration_file = (
    LOGS_DIR
    / "phase2_recovery_configuration.json"
)

with recovery_configuration_file.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        phase_2_recovery_configuration,
        file,
        indent=4
    )


# ------------------------------------------------------------
# 4. Create the methodological summary
# ------------------------------------------------------------

phase_2_summary_text = f"""
# Phase 2 Methodological Summary

Generated at UTC: {recorded_at_utc}

## Dataset scope

The analysis used NASA C-MAPSS FD001 only.

## Leakage-safe engine partitions

- Model training: 60 engines
- Validation: 20 engines
- Conformal calibration: 20 engines
- Random seed: 42
- Assignment unit: complete engine
- Lifetime stratification: not used
- Rerolling: not permitted
- Official NASA test data: reserved for final evaluation

## RUL target policy

The primary target is uncapped RUL:

`failure cycle - current cycle`

A 130-cycle capped target is retained only as a
sensitivity analysis:

`minimum(uncapped RUL, 130)`

The cap is not described as NASA-prescribed or optimal.

## Training-only exploratory analysis

Complete engine lifetimes varied from 128 to 362 cycles
in the model-training partition.

The corrected directionally consistent sensor profiles were:

{", ".join(corrected_top_sensor_columns)}

These are anonymised descriptive associations. They are not
physical interpretations or final modelling features.

## Constant measurements

The following measurements were constant in the
model-training partition:

{", ".join(constant_measurement_features)}

No feature has yet been removed.

## Feature redundancy

No feature pair met every predefined strong-redundancy rule:

- Absolute median within-engine Spearman correlation >= 0.90
- Finite correlations for at least 80% of engines
- At least 75% dominant-direction consistency
- Engine-correlation IQR excluding zero

This does not imply independence. It means that no feature
pair was sufficiently strong and stable to justify a
redundancy claim during EDA.

## Restrictions maintained

- Validation engines did not influence EDA selection.
- Calibration engines did not influence EDA selection.
- Official test outcomes were not accessed.
- No feature was removed.
- No model was trained.
- No physical sensor meaning was invented.

## Next phase

Phase 3 will define the leakage-safe preprocessing pipeline.
It has not yet started.
"""

phase_2_summary_file = (
    LOGS_DIR
    / "phase2_methodological_summary.md"
)

phase_2_summary_file.write_text(
    phase_2_summary_text.strip(),
    encoding="utf-8"
)


# ------------------------------------------------------------
# 5. Create the technical-completion record
# ------------------------------------------------------------

phase_2_completion_record = {
    "recorded_at_utc": recorded_at_utc,

    "phase": (
        "Phase 2 — Engine partitioning, "
        "RUL design and exploratory analysis"
    ),

    "technical_status": (
        "All planned Phase 2 technical "
        "requirements completed"
    ),

    "formal_approval_status": (
        "Awaiting Ranitabh's Phase 2 approval"
    ),

    "completed_components": [
        "Training-engine lifetime overview",
        "Frozen engine-level partitions",
        "Uncapped RUL construction",
        "Capped-130 sensitivity evaluation",
        "Formal target-policy decision",
        "Training-only lifetime EDA",
        "Engine-aware sensor association audit",
        "Corrected direction-consistency analysis",
        "Equal-engine-weight life-stage profiles",
        "Engine-aware feature redundancy audit",
        "Phase 2 recovery configuration",
        "Consolidated evidence manifest"
    ],

    "locked_decisions": {
        "dataset": "FD001 only",
        "random_seed": 42,
        "partition_ratio": "60/20/20 engines",
        "primary_target": "rul_uncapped",
        "sensitivity_target": "rul_capped_130",
        "official_test_role": (
            "Untouched final evaluation"
        )
    },

    "important_findings": {
        "constant_measurement_feature_count":
            len(constant_measurement_features),

        "directionally_consistent_visual_sensor_count":
            len(corrected_top_sensor_columns),

        "strong_redundancy_candidate_count":
            len(
                strong_redundancy_candidates_df
            )
    },

    "raw_data_modified": False,
    "test_rul_values_used": False,
    "features_removed": False,
    "models_trained": False,

    "next_phase": (
        "Phase 3 — Leakage-safe preprocessing"
    )
}

phase_2_completion_file = (
    LOGS_DIR
    / "phase2_technical_completion_record.json"
)

with phase_2_completion_file.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        phase_2_completion_record,
        file,
        indent=4
    )


# ------------------------------------------------------------
# 6. Define the complete evidence inventory
# ------------------------------------------------------------

evidence_files = [
    # Recovery-critical derived datasets
    INTERIM_DATA_DIR
    / "fd001_engine_partition_manifest.csv",

    INTERIM_DATA_DIR
    / "fd001_model_training_targets.csv",

    INTERIM_DATA_DIR
    / "fd001_validation_targets.csv",

    INTERIM_DATA_DIR
    / "fd001_calibration_targets.csv",

    # Partition and target evidence
    LOGS_DIR
    / "fd001_engine_partition_policy.json",

    LOGS_DIR
    / "fd001_partition_audit.csv",

    LOGS_DIR
    / "fd001_uncapped_rul_audit.csv",

    LOGS_DIR
    / "fd001_uncapped_rul_boundary_examples.csv",

    LOGS_DIR
    / "fd001_uncapped_rul_definition.json",

    LOGS_DIR
    / "fd001_rul_target_comparison_audit.csv",

    LOGS_DIR
    / "fd001_rul_cap_130_transition_examples.csv",

    LOGS_DIR
    / "fd001_rul_cap_130_candidate_record.json",

    LOGS_DIR
    / "fd001_final_target_policy_audit.csv",

    LOGS_DIR
    / "fd001_final_target_policy.json",

    LOGS_DIR
    / "fd001_target_decision.md",

    # Corrected EDA evidence
    RESULTS_DIR
    / "phase2_model_training_engine_lifetimes.csv",

    RESULTS_DIR
    / "phase2_engine_sensor_correlations_corrected.csv",

    RESULTS_DIR
    / "phase2_sensor_association_summary_corrected.csv",

    RESULTS_DIR
    / "phase2_consistent_sensor_life_profiles.csv",

    LOGS_DIR
    / "phase2_step4a_sensor_eda_correction.json",

    # Feature-variation and redundancy evidence
    RESULTS_DIR
    / "phase2_feature_variation_profile.csv",

    RESULTS_DIR
    / "phase2_pairwise_feature_summary.csv",

    RESULTS_DIR
    / "phase2_strong_redundancy_candidates.csv",

    RESULTS_DIR
    / "phase2_engine_aware_median_correlation_matrix.csv",

    LOGS_DIR
    / "phase2_feature_redundancy_audit.json",

    # Important figures
    FIGURES_DIR
    / "phase2_rul_cap_130_comparison.png",

    FIGURES_DIR
    / "phase2_rul_cap_130_comparison.pdf",

    FIGURES_DIR
    / "phase2_model_training_lifetime_distribution.png",

    FIGURES_DIR
    / "phase2_model_training_lifetime_distribution.pdf",

    FIGURES_DIR
    / "phase2_engine_aware_sensor_associations_corrected.png",

    FIGURES_DIR
    / "phase2_engine_aware_sensor_associations_corrected.pdf",

    FIGURES_DIR
    / "phase2_consistent_sensor_life_profiles.png",

    FIGURES_DIR
    / "phase2_consistent_sensor_life_profiles.pdf",

    FIGURES_DIR
    / "phase2_engine_aware_feature_correlation_heatmap.png",

    FIGURES_DIR
    / "phase2_engine_aware_feature_correlation_heatmap.pdf",

    # Completion and recovery records
    recovery_configuration_file,
    phase_2_summary_file,
    phase_2_completion_file
]


# ------------------------------------------------------------
# 7. Verify that every required evidence file exists
# ------------------------------------------------------------

missing_evidence_files = [
    str(evidence_file)
    for evidence_file in evidence_files
    if not evidence_file.is_file()
]

if missing_evidence_files:
    raise FileNotFoundError(
        "Phase 2 evidence is incomplete. Missing:\n"
        + "\n".join(missing_evidence_files)
    )


# ------------------------------------------------------------
# 8. Calculate a SHA-256 fingerprint for every file
# ------------------------------------------------------------

def calculate_sha256(
    file_path,
    chunk_size=1024 * 1024
):
    sha256_hash = hashlib.sha256()

    with file_path.open("rb") as file:
        while True:
            data_chunk = file.read(
                chunk_size
            )

            if not data_chunk:
                break

            sha256_hash.update(
                data_chunk
            )

    return sha256_hash.hexdigest()


manifest_records = []

for evidence_file in evidence_files:

    manifest_records.append({
        "file_name": evidence_file.name,

        "project_relative_path": str(
            evidence_file.relative_to(
                PROJECT_ROOT
            )
        ),

        "size_bytes": int(
            evidence_file.stat().st_size
        ),

        "sha256": calculate_sha256(
            evidence_file
        )
    })


phase_2_manifest_df = pd.DataFrame(
    manifest_records
)

phase_2_manifest_file = (
    LOGS_DIR
    / "phase2_complete_evidence_manifest.csv"
)

phase_2_manifest_df.to_csv(
    phase_2_manifest_file,
    index=False
)


# ------------------------------------------------------------
# 9. Validate Phase 2 completion
# ------------------------------------------------------------

model_training_engine_set = set(
    int(engine_id)
    for engine_id
    in MODEL_TRAIN_ENGINE_IDS
)

validation_engine_set = set(
    int(engine_id)
    for engine_id
    in VALIDATION_ENGINE_IDS
)

calibration_engine_set = set(
    int(engine_id)
    for engine_id
    in CALIBRATION_ENGINE_IDS
)

phase_2_checks = {
    "model_training_contains_60_engines":
        len(model_training_engine_set) == 60,

    "validation_contains_20_engines":
        len(validation_engine_set) == 20,

    "calibration_contains_20_engines":
        len(calibration_engine_set) == 20,

    "all_engine_partitions_are_disjoint":
        (
            model_training_engine_set.isdisjoint(
                validation_engine_set
            )
            and
            model_training_engine_set.isdisjoint(
                calibration_engine_set
            )
            and
            validation_engine_set.isdisjoint(
                calibration_engine_set
            )
        ),

    "all_100_training_engines_are_preserved":
        len(
            model_training_engine_set
            | validation_engine_set
            | calibration_engine_set
        ) == 100,

    "primary_target_exists_in_all_partitions":
        all(
            "rul_uncapped"
            in dataframe.columns
            for dataframe in [
                model_train_targets_df,
                validation_targets_df,
                calibration_targets_df
            ]
        ),

    "sensitivity_target_exists_in_all_partitions":
        all(
            "rul_capped_130"
            in dataframe.columns
            for dataframe in [
                model_train_targets_df,
                validation_targets_df,
                calibration_targets_df
            ]
        ),

    "seven_constant_features_are_documented":
        len(
            constant_measurement_features
        ) == 7,

    "four_corrected_visual_sensors_are_documented":
        len(
            corrected_top_sensor_columns
        ) == 4,

    "all_required_evidence_files_exist":
        len(missing_evidence_files) == 0,

    "every_evidence_file_has_a_sha256_hash":
        bool(
            phase_2_manifest_df[
                "sha256"
            ].str.len().eq(64).all()
        ),

    "no_models_have_been_created":
        not any(
            MODEL_DIR.glob("*")
        )
        if "MODEL_DIR" in globals()
        else True
}


# ------------------------------------------------------------
# 10. Create the complete evidence ZIP
# ------------------------------------------------------------

phase_2_complete_zip = (
    PROJECT_ROOT
    / "Phase2_Complete_Reproducibility_Evidence.zip"
)

with zipfile.ZipFile(
    phase_2_complete_zip,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as zip_file:

    for evidence_file in evidence_files:
        zip_file.write(
            evidence_file,
            arcname=str(
                evidence_file.relative_to(
                    PROJECT_ROOT
                )
            )
        )

    zip_file.write(
        phase_2_manifest_file,
        arcname=str(
            phase_2_manifest_file.relative_to(
                PROJECT_ROOT
            )
        )
    )


# ------------------------------------------------------------
# 11. Calculate the complete ZIP fingerprint
# ------------------------------------------------------------

complete_zip_sha256 = calculate_sha256(
    phase_2_complete_zip
)


# ------------------------------------------------------------
# 12. Display completion results
# ------------------------------------------------------------

print("PHASE 2 COMPLETION SUMMARY")
print("=" * 100)
print(
    "Model-training engines: "
    f"{len(model_training_engine_set)}"
)
print(
    "Validation engines: "
    f"{len(validation_engine_set)}"
)
print(
    "Calibration engines: "
    f"{len(calibration_engine_set)}"
)
print("Primary target: rul_uncapped")
print("Sensitivity target: rul_capped_130")
print(
    "Constant features documented: "
    f"{len(constant_measurement_features)}"
)
print(
    "Corrected visual sensors: "
    f"{corrected_top_sensor_columns}"
)
print(
    "Strong redundancy candidates: "
    f"{len(strong_redundancy_candidates_df)}"
)

print("\nEVIDENCE INVENTORY")
print("=" * 100)
print(
    f"Files fingerprinted: "
    f"{len(phase_2_manifest_df)}"
)
print(
    f"Manifest: "
    f"{phase_2_manifest_file}"
)

print("\nPHASE 2 VALIDATION")
print("=" * 100)

for check_name, passed in (
    phase_2_checks.items()
):
    status = "PASS" if passed else "FAIL"

    print(
        f"{status} — "
        f"{check_name.replace('_', ' ')}"
    )

print("\nCOMPLETE EVIDENCE PACKAGE")
print("=" * 100)
print(f"ZIP: {phase_2_complete_zip}")
print(f"ZIP SHA-256: {complete_zip_sha256}")

print("\nFINAL CHECK")
print("=" * 100)

if all(phase_2_checks.values()):
    print("PASS — Phase 2 technical requirements are complete.")
    print("PASS — All engine partitions and targets are recoverable.")
    print("PASS — Corrected EDA evidence was preserved.")
    print("PASS — Every evidence file was fingerprinted.")
    print("PASS — No preprocessing or modelling has started.")
    print("AWAITING APPROVAL — Phase 2 is not yet formally closed.")

    print("\nDownloading the complete Phase 2 evidence ZIP...")
    colab_files.download(
        str(phase_2_complete_zip)
    )

else:
    print(
        "CHECK REQUIRED — "
        "Phase 2 completion validation failed."
    )

In [ ]:
# ============================================================
# PHASE 3 — LEAKAGE-SAFE PREPROCESSING
# STEP 1 — FREEZE THE PREDICTOR AND TRANSFORMATION POLICY
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from google.colab import files as colab_files

import pandas as pd
import numpy as np
import json
import zipfile


# ------------------------------------------------------------
# 1. Confirm that the locked target partitions exist
# ------------------------------------------------------------

required_dataframe_names = [
    "model_train_targets_df",
    "validation_targets_df",
    "calibration_targets_df"
]

missing_dataframes = [
    name
    for name in required_dataframe_names
    if name not in globals()
]

if missing_dataframes:
    raise RuntimeError(
        "Required Phase 2 target dataframes are missing: "
        + ", ".join(missing_dataframes)
    )


# ------------------------------------------------------------
# 2. Define controlled output locations
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/Predictive_Maintenance_Thesis")
LOGS_DIR = PROJECT_ROOT / "logs"
RESULTS_DIR = PROJECT_ROOT / "results"

LOGS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

recorded_at_utc = datetime.now(
    timezone.utc
).isoformat()


# ------------------------------------------------------------
# 3. Formally record Phase 2 closure
# ------------------------------------------------------------

phase_2_closure_record = {
    "recorded_at_utc": recorded_at_utc,

    "phase": (
        "Phase 2 — Engine partitioning, "
        "RUL design and exploratory analysis"
    ),

    "status": "Approved, closed and locked",

    "approval_evidence": [
        (
            "Phase2_Complete_"
            "Reproducibility_Evidence.zip downloaded"
        ),
        (
            "Ranitabh_FD001_Thesis_"
            "Implementation.ipynb downloaded"
        )
    ],

    "complete_evidence_sha256": (
        "c4c7cdd6572907201df69e7625dc25fb"
        "7fffdbb7275ee0f8dace55b4eb25d4e1"
    ),

    "locked_decisions": {
        "dataset": "FD001 only",
        "random_seed": 42,
        "engine_partition": "60/20/20",
        "primary_target": "rul_uncapped",
        "sensitivity_target": "rul_capped_130",
        "sensitivity_cap_cycles": 130
    },

    "phase_3_previously_started": False
}

phase_2_closure_file = (
    LOGS_DIR
    / "phase2_formal_closure_record.json"
)

with phase_2_closure_file.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        phase_2_closure_record,
        file,
        indent=4
    )


# ------------------------------------------------------------
# 4. Define all possible feature groups
# ------------------------------------------------------------

IDENTIFIER_COLUMNS = [
    "unit_number"
]

AGE_FEATURE_COLUMNS = [
    "time_in_cycles"
]

OPERATIONAL_SETTING_COLUMNS = [
    "operational_setting_1",
    "operational_setting_2",
    "operational_setting_3"
]

SENSOR_COLUMNS = [
    f"sensor_{sensor_number}"
    for sensor_number in range(1, 22)
]

MEASUREMENT_FEATURE_COLUMNS = (
    OPERATIONAL_SETTING_COLUMNS
    + SENSOR_COLUMNS
)

TARGET_COLUMNS = [
    "rul_uncapped",
    "rul_capped_130"
]


# ------------------------------------------------------------
# 5. Detect constant measurements from model training only
# ------------------------------------------------------------

MODEL_TRAINING_CONSTANT_COLUMNS = [
    column
    for column in MEASUREMENT_FEATURE_COLUMNS
    if (
        model_train_targets_df[
            column
        ].nunique(dropna=False) == 1
    )
]

EXPECTED_CONSTANT_COLUMNS = [
    "operational_setting_3",
    "sensor_1",
    "sensor_5",
    "sensor_10",
    "sensor_16",
    "sensor_18",
    "sensor_19"
]

if (
    MODEL_TRAINING_CONSTANT_COLUMNS
    != EXPECTED_CONSTANT_COLUMNS
):
    raise ValueError(
        "The detected constant-feature list differs "
        "from the locked Phase 2 finding.\n"
        f"Detected: {MODEL_TRAINING_CONSTANT_COLUMNS}\n"
        f"Expected: {EXPECTED_CONSTANT_COLUMNS}"
    )


# ------------------------------------------------------------
# 6. Freeze the predictor list
# ------------------------------------------------------------

VARIABLE_MEASUREMENT_COLUMNS = [
    column
    for column in MEASUREMENT_FEATURE_COLUMNS
    if column
    not in MODEL_TRAINING_CONSTANT_COLUMNS
]

PREDICTOR_COLUMNS = (
    AGE_FEATURE_COLUMNS
    + VARIABLE_MEASUREMENT_COLUMNS
)

RIDGE_PREDICTOR_COLUMNS = (
    PREDICTOR_COLUMNS.copy()
)

TREE_PREDICTOR_COLUMNS = (
    PREDICTOR_COLUMNS.copy()
)


# ------------------------------------------------------------
# 7. Create raw predictor matrices
# ------------------------------------------------------------

# These are copies.
# The locked source dataframes are not modified.

X_model_train_raw = (
    model_train_targets_df[
        PREDICTOR_COLUMNS
    ].copy()
)

X_validation_raw = (
    validation_targets_df[
        PREDICTOR_COLUMNS
    ].copy()
)

X_calibration_raw = (
    calibration_targets_df[
        PREDICTOR_COLUMNS
    ].copy()
)


# ------------------------------------------------------------
# 8. Create primary target vectors
# ------------------------------------------------------------

y_model_train_primary = (
    model_train_targets_df[
        "rul_uncapped"
    ].copy()
)

y_validation_primary = (
    validation_targets_df[
        "rul_uncapped"
    ].copy()
)

y_calibration_primary = (
    calibration_targets_df[
        "rul_uncapped"
    ].copy()
)


# ------------------------------------------------------------
# 9. Create sensitivity target vectors
# ------------------------------------------------------------

y_model_train_sensitivity = (
    model_train_targets_df[
        "rul_capped_130"
    ].copy()
)

y_validation_sensitivity = (
    validation_targets_df[
        "rul_capped_130"
    ].copy()
)

y_calibration_sensitivity = (
    calibration_targets_df[
        "rul_capped_130"
    ].copy()
)


# ------------------------------------------------------------
# 10. Preserve engine and cycle metadata separately
# ------------------------------------------------------------

model_train_metadata_df = (
    model_train_targets_df[
        [
            "unit_number",
            "time_in_cycles"
        ]
    ].copy()
)

validation_metadata_df = (
    validation_targets_df[
        [
            "unit_number",
            "time_in_cycles"
        ]
    ].copy()
)

calibration_metadata_df = (
    calibration_targets_df[
        [
            "unit_number",
            "time_in_cycles"
        ]
    ].copy()
)


# ------------------------------------------------------------
# 11. Check numeric integrity
# ------------------------------------------------------------

def count_infinite_values(
    dataframe
):
    return int(
        np.isinf(
            dataframe.to_numpy(
                dtype=float
            )
        ).sum()
    )


predictor_integrity = {
    "model_training_missing_values":
        int(
            X_model_train_raw
            .isna()
            .sum()
            .sum()
        ),

    "validation_missing_values":
        int(
            X_validation_raw
            .isna()
            .sum()
            .sum()
        ),

    "calibration_missing_values":
        int(
            X_calibration_raw
            .isna()
            .sum()
            .sum()
        ),

    "model_training_infinite_values":
        count_infinite_values(
            X_model_train_raw
        ),

    "validation_infinite_values":
        count_infinite_values(
            X_validation_raw
        ),

    "calibration_infinite_values":
        count_infinite_values(
            X_calibration_raw
        )
}


# ------------------------------------------------------------
# 12. Create the formal feature schema
# ------------------------------------------------------------

feature_schema_records = []

all_documented_columns = (
    IDENTIFIER_COLUMNS
    + AGE_FEATURE_COLUMNS
    + MEASUREMENT_FEATURE_COLUMNS
    + TARGET_COLUMNS
)

for column in all_documented_columns:

    if column in IDENTIFIER_COLUMNS:
        feature_role = "identifier"
        preprocessing_status = (
            "excluded_from_predictors"
        )
        reason = (
            "Used only for grouping, splitting "
            "and traceability."
        )

    elif column in TARGET_COLUMNS:
        feature_role = "target"
        preprocessing_status = (
            "excluded_from_predictors"
        )
        reason = (
            "Outcome variable; including it would "
            "cause direct target leakage."
        )

    elif column in MODEL_TRAINING_CONSTANT_COLUMNS:
        feature_role = "constant_measurement"
        preprocessing_status = (
            "excluded_based_on_model_training_only"
        )
        reason = (
            "No variation in the model-training "
            "partition."
        )

    elif column in AGE_FEATURE_COLUMNS:
        feature_role = "known_age_feature"
        preprocessing_status = (
            "retained_predictor"
        )
        reason = (
            "Current cycle is available at "
            "prediction time and provides explicit "
            "engine-age information."
        )

    else:
        feature_role = "variable_measurement"
        preprocessing_status = (
            "retained_predictor"
        )
        reason = (
            "Variable candidate predictor; no "
            "leakage-safe evidence currently "
            "justifies exclusion."
        )

    feature_schema_records.append({
        "column": column,
        "feature_role": feature_role,
        "preprocessing_status":
            preprocessing_status,
        "reason": reason,
        "used_by_ridge":
            column in RIDGE_PREDICTOR_COLUMNS,
        "used_by_tree_models":
            column in TREE_PREDICTOR_COLUMNS,
        "ridge_scaling_policy": (
            "fit_standard_scaler_on_model_training"
            if column
            in RIDGE_PREDICTOR_COLUMNS
            else "not_applicable"
        ),
        "tree_scaling_policy": (
            "no_scaling"
            if column
            in TREE_PREDICTOR_COLUMNS
            else "not_applicable"
        )
    })


feature_schema_df = pd.DataFrame(
    feature_schema_records
)

feature_schema_file = (
    LOGS_DIR
    / "phase3_preprocessing_feature_schema.csv"
)

feature_schema_df.to_csv(
    feature_schema_file,
    index=False
)


# ------------------------------------------------------------
# 13. Create the preprocessing-policy record
# ------------------------------------------------------------

preprocessing_policy = {
    "recorded_at_utc": recorded_at_utc,

    "phase": (
        "Phase 3 — Leakage-safe preprocessing"
    ),

    "step": (
        "Step 1 — Predictor and "
        "transformation policy"
    ),

    "policy_status": (
        "Technically defined; "
        "awaiting Step 1 approval"
    ),

    "data_scope": (
        "Frozen model-training, validation "
        "and calibration partitions derived "
        "from train_FD001.txt"
    ),

    "identifier_columns":
        IDENTIFIER_COLUMNS,

    "target_columns":
        TARGET_COLUMNS,

    "model_training_constant_columns":
        MODEL_TRAINING_CONSTANT_COLUMNS,

    "retained_predictor_columns":
        PREDICTOR_COLUMNS,

    "retained_predictor_count":
        len(PREDICTOR_COLUMNS),

    "time_in_cycles_policy": {
        "included": True,
        "role": "Known engine-age feature",
        "justification": (
            "It is observable at inference time. "
            "Its contribution will remain visible "
            "through model comparison and SHAP."
        )
    },

    "ridge_policy": {
        "predictors":
            RIDGE_PREDICTOR_COLUMNS,

        "scaling": (
            "StandardScaler fitted only on "
            "model-training rows"
        ),

        "scaler_fitted_in_this_step":
            False
    },

    "tree_model_policy": {
        "predictors":
            TREE_PREDICTOR_COLUMNS,

        "models": [
            "Random Forest",
            "Provisional XGBoost"
        ],

        "scaling": "None"
    },

    "missing_value_policy": (
        "No imputation because validated "
        "partitions contain no missing values."
    ),

    "infinite_value_policy": (
        "No replacement because validated "
        "partitions contain no infinite values."
    ),

    "feature_selection_policy": (
        "No model-based feature selection has "
        "been performed. EDA sensor rankings "
        "must not be treated as final selection."
    ),

    "redundancy_policy": (
        "No variable pair met every predefined "
        "strong-redundancy condition; no "
        "correlation-based removal is performed."
    ),

    "row_policy": (
        "Every row in each frozen partition "
        "is preserved."
    ),

    "prohibited_actions": [
        (
            "Do not fit scaling parameters "
            "on validation data."
        ),
        (
            "Do not fit scaling parameters "
            "on calibration data."
        ),
        (
            "Do not use official test data "
            "to design preprocessing."
        ),
        (
            "Do not include unit_number "
            "as a predictor."
        ),
        (
            "Do not include either RUL "
            "target as a predictor."
        ),
        (
            "Do not remove sensors merely "
            "because of EDA rankings."
        )
    ],

    "scaling_performed": False,
    "models_trained": False,
    "official_test_data_used": False,
    "test_rul_values_used": False
}

preprocessing_policy_file = (
    LOGS_DIR
    / "phase3_preprocessing_policy.json"
)

with preprocessing_policy_file.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        preprocessing_policy,
        file,
        indent=4
    )


# ------------------------------------------------------------
# 14. Validate the predictor boundary
# ------------------------------------------------------------

excluded_columns = set(
    IDENTIFIER_COLUMNS
    + TARGET_COLUMNS
    + MODEL_TRAINING_CONSTANT_COLUMNS
)

all_predictor_values_numeric = all(
    pd.api.types.is_numeric_dtype(
        X_model_train_raw[column]
    )
    for column in PREDICTOR_COLUMNS
)

preprocessing_checks = {
    "seven_constants_detected_from_model_training":
        len(
            MODEL_TRAINING_CONSTANT_COLUMNS
        ) == 7,

    "eighteen_predictors_are_retained":
        len(
            PREDICTOR_COLUMNS
        ) == 18,

    "unit_number_is_not_a_predictor":
        "unit_number"
        not in PREDICTOR_COLUMNS,

    "primary_target_is_not_a_predictor":
        "rul_uncapped"
        not in PREDICTOR_COLUMNS,

    "sensitivity_target_is_not_a_predictor":
        "rul_capped_130"
        not in PREDICTOR_COLUMNS,

    "constant_features_are_not_predictors":
        excluded_columns.isdisjoint(
            set(PREDICTOR_COLUMNS)
        ),

    "time_in_cycles_is_retained":
        "time_in_cycles"
        in PREDICTOR_COLUMNS,

    "all_predictors_are_numeric":
        all_predictor_values_numeric,

    "predictor_order_matches_across_partitions":
        (
            X_model_train_raw.columns.tolist()
            == X_validation_raw.columns.tolist()
            == X_calibration_raw.columns.tolist()
        ),

    "model_training_row_count_is_preserved":
        len(X_model_train_raw)
        == len(model_train_targets_df),

    "validation_row_count_is_preserved":
        len(X_validation_raw)
        == len(validation_targets_df),

    "calibration_row_count_is_preserved":
        len(X_calibration_raw)
        == len(calibration_targets_df),

    "all_target_indices_align_with_predictors":
        (
            X_model_train_raw.index.equals(
                y_model_train_primary.index
            )
            and
            X_validation_raw.index.equals(
                y_validation_primary.index
            )
            and
            X_calibration_raw.index.equals(
                y_calibration_primary.index
            )
        ),

    "no_missing_predictor_values_exist":
        all(
            value == 0
            for key, value
            in predictor_integrity.items()
            if "missing" in key
        ),

    "no_infinite_predictor_values_exist":
        all(
            value == 0
            for key, value
            in predictor_integrity.items()
            if "infinite" in key
        ),

    "no_scaling_has_yet_been_performed":
        True,

    "no_model_has_been_trained":
        True
}


# ------------------------------------------------------------
# 15. Package Step 1 evidence
# ------------------------------------------------------------

step_1_evidence_zip = (
    PROJECT_ROOT
    / "Phase3_Step1_Preprocessing_Policy_Evidence.zip"
)

with zipfile.ZipFile(
    step_1_evidence_zip,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as zip_file:

    for evidence_file in [
        phase_2_closure_file,
        feature_schema_file,
        preprocessing_policy_file
    ]:
        zip_file.write(
            evidence_file,
            arcname=evidence_file.name
        )


# ------------------------------------------------------------
# 16. Display the policy
# ------------------------------------------------------------

print("PREPROCESSING FEATURE POLICY")
print("=" * 100)
print(
    f"Identifiers excluded: "
    f"{IDENTIFIER_COLUMNS}"
)
print(
    f"Targets excluded: "
    f"{TARGET_COLUMNS}"
)
print(
    "Model-training constants excluded: "
    f"{MODEL_TRAINING_CONSTANT_COLUMNS}"
)
print(
    f"Retained predictor count: "
    f"{len(PREDICTOR_COLUMNS)}"
)
print(
    f"Retained predictors:\n"
    f"{PREDICTOR_COLUMNS}"
)

print("\nMODEL-SPECIFIC TRANSFORMATION POLICY")
print("=" * 100)
print(
    "Ridge: StandardScaler will be fitted "
    "only on model-training rows."
)
print(
    "Random Forest: Raw retained predictors; "
    "no scaling."
)
print(
    "XGBoost: Raw retained predictors; "
    "no scaling."
)
print("Scaling performed now: No")

print("\nPREDICTOR INTEGRITY")
print("=" * 100)

for name, value in (
    predictor_integrity.items()
):
    print(f"{name}: {value}")

print("\nPREPROCESSING VALIDATION")
print("=" * 100)

for check_name, passed in (
    preprocessing_checks.items()
):
    status = "PASS" if passed else "FAIL"

    print(
        f"{status} — "
        f"{check_name.replace('_', ' ')}"
    )

print("\nFINAL CHECK")
print("=" * 100)

if all(preprocessing_checks.values()):
    print("PASS — Predictor boundary was frozen successfully.")
    print("PASS — Identifiers, targets and constants were excluded.")
    print("PASS — All rows and target alignments were preserved.")
    print("PASS — Validation and calibration were not used for fitting.")
    print("PASS — No scaling or modelling has occurred.")
    print("AWAITING APPROVAL — Transformation fitting has not started.")

    print("\nDownloading the Step 1 evidence ZIP...")
    colab_files.download(
        str(step_1_evidence_zip)
    )

else:
    print(
        "CHECK REQUIRED — "
        "Preprocessing-policy validation failed."
    )

In [ ]:
# ============================================================
# PHASE 3 — LEAKAGE-SAFE PREPROCESSING
# STEP 2 — FIT AND VERIFY THE RIDGE STANDARD SCALER
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from google.colab import files as colab_files
from sklearn.preprocessing import StandardScaler

import pandas as pd
import numpy as np
import joblib
import hashlib
import json
import zipfile


# ------------------------------------------------------------
# 1. Confirm that Step 1 objects are available
# ------------------------------------------------------------

required_object_names = [
    "X_model_train_raw",
    "X_validation_raw",
    "X_calibration_raw",
    "PREDICTOR_COLUMNS",
    "RIDGE_PREDICTOR_COLUMNS",
    "TREE_PREDICTOR_COLUMNS"
]

missing_objects = [
    object_name
    for object_name in required_object_names
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run Phase 3 Step 1 first. Missing objects: "
        + ", ".join(missing_objects)
    )


# ------------------------------------------------------------
# 2. Define controlled output locations
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/Predictive_Maintenance_Thesis")
PREPROCESSOR_DIR = PROJECT_ROOT / "models" / "preprocessing"
LOGS_DIR = PROJECT_ROOT / "logs"

PREPROCESSOR_DIR.mkdir(
    parents=True,
    exist_ok=True
)

LOGS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

recorded_at_utc = datetime.now(
    timezone.utc
).isoformat()


# ------------------------------------------------------------
# 3. Record formal Step 1 approval
# ------------------------------------------------------------

step_1_closure_record = {
    "recorded_at_utc": recorded_at_utc,

    "phase": (
        "Phase 3 — Leakage-safe preprocessing"
    ),

    "step": (
        "Step 1 — Predictor and "
        "transformation policy"
    ),

    "status": "Approved and locked",

    "retained_predictor_count":
        len(PREDICTOR_COLUMNS),

    "retained_predictors":
        PREDICTOR_COLUMNS,

    "ridge_policy": (
        "Standardise using parameters fitted "
        "only on model-training rows."
    ),

    "tree_policy": (
        "Retain the same predictors without scaling."
    ),

    "duplicate_local_downloads": (
        "Three evidence ZIP downloads were created "
        "locally; the latest copy is authoritative."
    )
}

step_1_closure_file = (
    LOGS_DIR
    / "phase3_step1_preprocessing_policy_closure.json"
)

with step_1_closure_file.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        step_1_closure_record,
        file,
        indent=4
    )


# ------------------------------------------------------------
# 4. Preserve raw predictor snapshots for verification
# ------------------------------------------------------------

X_model_train_before_scaling = (
    X_model_train_raw.copy(deep=True)
)

X_validation_before_scaling = (
    X_validation_raw.copy(deep=True)
)

X_calibration_before_scaling = (
    X_calibration_raw.copy(deep=True)
)


# ------------------------------------------------------------
# 5. Create and fit exactly one StandardScaler
# ------------------------------------------------------------

ridge_scaler = StandardScaler(
    with_mean=True,
    with_std=True
)

# This is the only fitting operation.
# Validation and calibration are not passed to fit().

X_model_train_scaled_array = (
    ridge_scaler.fit_transform(
        X_model_train_raw[
            RIDGE_PREDICTOR_COLUMNS
        ]
    )
)


# ------------------------------------------------------------
# 6. Transform later partitions without refitting
# ------------------------------------------------------------

X_validation_scaled_array = (
    ridge_scaler.transform(
        X_validation_raw[
            RIDGE_PREDICTOR_COLUMNS
        ]
    )
)

X_calibration_scaled_array = (
    ridge_scaler.transform(
        X_calibration_raw[
            RIDGE_PREDICTOR_COLUMNS
        ]
    )
)


# ------------------------------------------------------------
# 7. Restore column names and indices
# ------------------------------------------------------------

X_model_train_ridge_scaled = pd.DataFrame(
    X_model_train_scaled_array,
    columns=RIDGE_PREDICTOR_COLUMNS,
    index=X_model_train_raw.index
)

X_validation_ridge_scaled = pd.DataFrame(
    X_validation_scaled_array,
    columns=RIDGE_PREDICTOR_COLUMNS,
    index=X_validation_raw.index
)

X_calibration_ridge_scaled = pd.DataFrame(
    X_calibration_scaled_array,
    columns=RIDGE_PREDICTOR_COLUMNS,
    index=X_calibration_raw.index
)


# ------------------------------------------------------------
# 8. Preserve unscaled tree-model predictor views
# ------------------------------------------------------------

X_model_train_tree = (
    X_model_train_raw[
        TREE_PREDICTOR_COLUMNS
    ].copy()
)

X_validation_tree = (
    X_validation_raw[
        TREE_PREDICTOR_COLUMNS
    ].copy()
)

X_calibration_tree = (
    X_calibration_raw[
        TREE_PREDICTOR_COLUMNS
    ].copy()
)


# ------------------------------------------------------------
# 9. Calculate expected training statistics independently
# ------------------------------------------------------------

expected_training_means = (
    X_model_train_raw[
        RIDGE_PREDICTOR_COLUMNS
    ]
    .mean(axis=0)
    .to_numpy(dtype=float)
)

expected_training_variances = (
    X_model_train_raw[
        RIDGE_PREDICTOR_COLUMNS
    ]
    .var(axis=0, ddof=0)
    .to_numpy(dtype=float)
)

expected_training_scales = np.sqrt(
    expected_training_variances
)


# ------------------------------------------------------------
# 10. Manually reproduce the transformations
# ------------------------------------------------------------

manual_model_training_transform = (
    (
        X_model_train_raw[
            RIDGE_PREDICTOR_COLUMNS
        ].to_numpy(dtype=float)
        - ridge_scaler.mean_
    )
    / ridge_scaler.scale_
)

manual_validation_transform = (
    (
        X_validation_raw[
            RIDGE_PREDICTOR_COLUMNS
        ].to_numpy(dtype=float)
        - ridge_scaler.mean_
    )
    / ridge_scaler.scale_
)

manual_calibration_transform = (
    (
        X_calibration_raw[
            RIDGE_PREDICTOR_COLUMNS
        ].to_numpy(dtype=float)
        - ridge_scaler.mean_
    )
    / ridge_scaler.scale_
)


# ------------------------------------------------------------
# 11. Create the scaler-parameter table
# ------------------------------------------------------------

scaler_parameter_records = []

training_scaled_means = (
    X_model_train_ridge_scaled
    .mean(axis=0)
)

training_scaled_standard_deviations = (
    X_model_train_ridge_scaled
    .std(axis=0, ddof=0)
)

validation_scaled_means = (
    X_validation_ridge_scaled
    .mean(axis=0)
)

calibration_scaled_means = (
    X_calibration_ridge_scaled
    .mean(axis=0)
)

for feature_index, feature_name in enumerate(
    RIDGE_PREDICTOR_COLUMNS
):
    scaler_parameter_records.append({
        "feature": feature_name,

        "training_mean_used_by_scaler":
            float(
                ridge_scaler.mean_[
                    feature_index
                ]
            ),

        "training_variance_used_by_scaler":
            float(
                ridge_scaler.var_[
                    feature_index
                ]
            ),

        "training_scale_used_by_scaler":
            float(
                ridge_scaler.scale_[
                    feature_index
                ]
            ),

        "scaled_training_mean":
            float(
                training_scaled_means[
                    feature_name
                ]
            ),

        "scaled_training_standard_deviation":
            float(
                training_scaled_standard_deviations[
                    feature_name
                ]
            ),

        "scaled_validation_mean":
            float(
                validation_scaled_means[
                    feature_name
                ]
            ),

        "scaled_calibration_mean":
            float(
                calibration_scaled_means[
                    feature_name
                ]
            )
    })


scaler_parameters_df = pd.DataFrame(
    scaler_parameter_records
)

scaler_parameters_file = (
    LOGS_DIR
    / "phase3_ridge_scaler_parameters.csv"
)

scaler_parameters_df.to_csv(
    scaler_parameters_file,
    index=False
)


# ------------------------------------------------------------
# 12. Save the fitted scaler
# ------------------------------------------------------------

ridge_scaler_file = (
    PREPROCESSOR_DIR
    / "fd001_ridge_standard_scaler.joblib"
)

joblib.dump(
    ridge_scaler,
    ridge_scaler_file
)


# ------------------------------------------------------------
# 13. Reload the scaler and verify persistence
# ------------------------------------------------------------

reloaded_ridge_scaler = joblib.load(
    ridge_scaler_file
)

reloaded_validation_transform = (
    reloaded_ridge_scaler.transform(
        X_validation_raw[
            RIDGE_PREDICTOR_COLUMNS
        ]
    )
)


# ------------------------------------------------------------
# 14. Create helper functions for integrity checks
# ------------------------------------------------------------

def dataframe_has_only_finite_values(
    dataframe
):
    return bool(
        np.isfinite(
            dataframe.to_numpy(
                dtype=float
            )
        ).all()
    )


def calculate_sha256(
    file_path,
    chunk_size=1024 * 1024
):
    sha256_hash = hashlib.sha256()

    with file_path.open("rb") as file:
        while True:
            data_chunk = file.read(
                chunk_size
            )

            if not data_chunk:
                break

            sha256_hash.update(
                data_chunk
            )

    return sha256_hash.hexdigest()


# ------------------------------------------------------------
# 15. Validate the entire scaling process
# ------------------------------------------------------------

TRAINING_MEAN_TOLERANCE = 1e-10
TRAINING_STD_TOLERANCE = 1e-10
NUMERICAL_COMPARISON_TOLERANCE = 1e-12

scaling_checks = {
    "scaler_contains_18_features":
        int(
            ridge_scaler.n_features_in_
        ) == 18,

    "scaler_feature_order_is_preserved":
        (
            ridge_scaler.feature_names_in_.tolist()
            == RIDGE_PREDICTOR_COLUMNS
        ),

    "scaler_saw_only_model_training_row_count":
        int(
            ridge_scaler.n_samples_seen_
        ) == len(X_model_train_raw),

    "scaler_mean_matches_independent_training_mean":
        bool(
            np.allclose(
                ridge_scaler.mean_,
                expected_training_means,
                atol=NUMERICAL_COMPARISON_TOLERANCE,
                rtol=0
            )
        ),

    "scaler_variance_matches_independent_training_variance":
        bool(
            np.allclose(
                ridge_scaler.var_,
                expected_training_variances,
                atol=NUMERICAL_COMPARISON_TOLERANCE,
                rtol=0
            )
        ),

    "scaler_scale_matches_training_standard_deviation":
        bool(
            np.allclose(
                ridge_scaler.scale_,
                expected_training_scales,
                atol=NUMERICAL_COMPARISON_TOLERANCE,
                rtol=0
            )
        ),

    "manual_model_training_transform_matches_scaler":
        bool(
            np.allclose(
                manual_model_training_transform,
                X_model_train_scaled_array,
                atol=NUMERICAL_COMPARISON_TOLERANCE,
                rtol=0
            )
        ),

    "manual_validation_transform_matches_scaler":
        bool(
            np.allclose(
                manual_validation_transform,
                X_validation_scaled_array,
                atol=NUMERICAL_COMPARISON_TOLERANCE,
                rtol=0
            )
        ),

    "manual_calibration_transform_matches_scaler":
        bool(
            np.allclose(
                manual_calibration_transform,
                X_calibration_scaled_array,
                atol=NUMERICAL_COMPARISON_TOLERANCE,
                rtol=0
            )
        ),

    "scaled_training_means_are_zero":
        bool(
            (
                training_scaled_means.abs()
                <= TRAINING_MEAN_TOLERANCE
            ).all()
        ),

    "scaled_training_standard_deviations_are_one":
        bool(
            (
                (
                    training_scaled_standard_deviations
                    - 1.0
                ).abs()
                <= TRAINING_STD_TOLERANCE
            ).all()
        ),

    "scaled_model_training_values_are_finite":
        dataframe_has_only_finite_values(
            X_model_train_ridge_scaled
        ),

    "scaled_validation_values_are_finite":
        dataframe_has_only_finite_values(
            X_validation_ridge_scaled
        ),

    "scaled_calibration_values_are_finite":
        dataframe_has_only_finite_values(
            X_calibration_ridge_scaled
        ),

    "model_training_rows_and_index_are_preserved":
        (
            len(X_model_train_ridge_scaled)
            == len(X_model_train_raw)
            and
            X_model_train_ridge_scaled.index.equals(
                X_model_train_raw.index
            )
        ),

    "validation_rows_and_index_are_preserved":
        (
            len(X_validation_ridge_scaled)
            == len(X_validation_raw)
            and
            X_validation_ridge_scaled.index.equals(
                X_validation_raw.index
            )
        ),

    "calibration_rows_and_index_are_preserved":
        (
            len(X_calibration_ridge_scaled)
            == len(X_calibration_raw)
            and
            X_calibration_ridge_scaled.index.equals(
                X_calibration_raw.index
            )
        ),

    "raw_model_training_predictors_are_unchanged":
        X_model_train_raw.equals(
            X_model_train_before_scaling
        ),

    "raw_validation_predictors_are_unchanged":
        X_validation_raw.equals(
            X_validation_before_scaling
        ),

    "raw_calibration_predictors_are_unchanged":
        X_calibration_raw.equals(
            X_calibration_before_scaling
        ),

    "tree_predictors_remain_unscaled":
        (
            X_model_train_tree.equals(
                X_model_train_raw[
                    TREE_PREDICTOR_COLUMNS
                ]
            )
            and
            X_validation_tree.equals(
                X_validation_raw[
                    TREE_PREDICTOR_COLUMNS
                ]
            )
            and
            X_calibration_tree.equals(
                X_calibration_raw[
                    TREE_PREDICTOR_COLUMNS
                ]
            )
        ),

    "reloaded_scaler_reproduces_validation_transform":
        bool(
            np.allclose(
                reloaded_validation_transform,
                X_validation_scaled_array,
                atol=NUMERICAL_COMPARISON_TOLERANCE,
                rtol=0
            )
        ),

    "scaler_file_exists":
        ridge_scaler_file.is_file()
}


# ------------------------------------------------------------
# 16. Fingerprint the fitted scaler
# ------------------------------------------------------------

ridge_scaler_sha256 = calculate_sha256(
    ridge_scaler_file
)


# ------------------------------------------------------------
# 17. Create the formal scaling record
# ------------------------------------------------------------

scaling_record = {
    "recorded_at_utc": recorded_at_utc,

    "phase": (
        "Phase 3 — Leakage-safe preprocessing"
    ),

    "step": (
        "Step 2 — Ridge scaling"
    ),

    "transformer": "sklearn StandardScaler",

    "fit_partition": (
        "Model-training partition only"
    ),

    "fit_engine_count": int(
        model_train_metadata_df[
            "unit_number"
        ].nunique()
    ),

    "fit_row_count": int(
        len(X_model_train_raw)
    ),

    "feature_count": int(
        len(RIDGE_PREDICTOR_COLUMNS)
    ),

    "feature_order":
        RIDGE_PREDICTOR_COLUMNS,

    "validation_operation":
        "transform only",

    "calibration_operation":
        "transform only",

    "tree_model_scaling":
        "No scaling",

    "standard_deviation_convention": (
        "Population standard deviation, "
        "ddof=0, as used by StandardScaler"
    ),

    "scaler_file": str(
        ridge_scaler_file
    ),

    "scaler_sha256":
        ridge_scaler_sha256,

    "important_interpretation": (
        "Validation and calibration means are not "
        "expected to equal zero because they were "
        "transformed using model-training statistics."
    ),

    "validation_used_for_fitting": False,

    "calibration_used_for_fitting": False,

    "official_test_data_used": False,

    "test_rul_values_used": False,

    "regression_model_trained": False,

    "scaling_checks":
        scaling_checks
}

scaling_record_file = (
    LOGS_DIR
    / "phase3_ridge_scaling_record.json"
)

with scaling_record_file.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        scaling_record,
        file,
        indent=4
    )


# ------------------------------------------------------------
# 18. Package Step 2 evidence
# ------------------------------------------------------------

step_2_evidence_zip = (
    PROJECT_ROOT
    / "Phase3_Step2_Ridge_Scaling_Evidence.zip"
)

with zipfile.ZipFile(
    step_2_evidence_zip,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as zip_file:

    for evidence_file in [
        step_1_closure_file,
        scaler_parameters_file,
        scaling_record_file,
        ridge_scaler_file
    ]:
        zip_file.write(
            evidence_file,
            arcname=evidence_file.name
        )


# ------------------------------------------------------------
# 19. Display the results
# ------------------------------------------------------------

print("RIDGE SCALER FIT SUMMARY")
print("=" * 100)
print(
    f"Fit partition: Model training only"
)
print(
    f"Fit engines: "
    f"{model_train_metadata_df['unit_number'].nunique()}"
)
print(
    f"Fit rows: {len(X_model_train_raw)}"
)
print(
    f"Features scaled: "
    f"{len(RIDGE_PREDICTOR_COLUMNS)}"
)
print(
    "Validation operation: transform only"
)
print(
    "Calibration operation: transform only"
)
print(
    "Tree-model scaling: none"
)

print("\nTRANSFORMED PARTITION SUMMARY")
print("=" * 100)
print(
    f"Model training shape: "
    f"{X_model_train_ridge_scaled.shape}"
)
print(
    f"Validation shape: "
    f"{X_validation_ridge_scaled.shape}"
)
print(
    f"Calibration shape: "
    f"{X_calibration_ridge_scaled.shape}"
)
print(
    "Largest absolute scaled-training mean: "
    f"{training_scaled_means.abs().max():.3e}"
)
print(
    "Largest training-standard-deviation error: "
    f"{(training_scaled_standard_deviations - 1).abs().max():.3e}"
)
print(
    "Largest absolute scaled-validation mean: "
    f"{validation_scaled_means.abs().max():.6f}"
)
print(
    "Largest absolute scaled-calibration mean: "
    f"{calibration_scaled_means.abs().max():.6f}"
)

print("\nSCALING VALIDATION")
print("=" * 100)

for check_name, passed in scaling_checks.items():
    status = "PASS" if passed else "FAIL"

    print(
        f"{status} — "
        f"{check_name.replace('_', ' ')}"
    )

print("\nSAVED TRANSFORMER")
print("=" * 100)
print(f"Scaler file: {ridge_scaler_file}")
print(f"Scaler SHA-256: {ridge_scaler_sha256}")

print("\nRESEARCH CONTROLS")
print("=" * 100)
print("Validation used for fitting: No")
print("Calibration used for fitting: No")
print("Official test data used: No")
print("Test RUL values accessed: No")
print("Regression model trained: No")

print("\nFINAL CHECK")
print("=" * 100)

if all(scaling_checks.values()):
    print("PASS — Ridge scaler was fitted successfully.")
    print("PASS — Only model-training statistics were learned.")
    print("PASS — Validation and calibration were transformed without refitting.")
    print("PASS — Raw and tree-model predictors remained unchanged.")
    print("PASS — Saved scaler reproduces the transformation.")
    print("AWAITING APPROVAL — No regression model has been trained.")

    print("\nDownloading the Step 2 evidence ZIP...")
    colab_files.download(
        str(step_2_evidence_zip)
    )

else:
    print(
        "CHECK REQUIRED — "
        "Ridge-scaling validation failed."
    )

In [ ]:
# ============================================================
# PHASE 3 — LEAKAGE-SAFE PREPROCESSING
# STEP 3 — PACKAGE, VERIFY AND COMPLETE PREPROCESSING
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from google.colab import files as colab_files
from importlib.metadata import version

import pandas as pd
import numpy as np
import joblib
import hashlib
import json
import zipfile


# ------------------------------------------------------------
# 1. Confirm that all Phase 3 objects exist
# ------------------------------------------------------------

required_object_names = [
    "PREDICTOR_COLUMNS",
    "RIDGE_PREDICTOR_COLUMNS",
    "TREE_PREDICTOR_COLUMNS",
    "MODEL_TRAINING_CONSTANT_COLUMNS",
    "ridge_scaler",
    "X_model_train_raw",
    "X_validation_raw",
    "X_calibration_raw",
    "X_model_train_ridge_scaled",
    "X_validation_ridge_scaled",
    "X_calibration_ridge_scaled",
    "X_model_train_tree",
    "X_validation_tree",
    "X_calibration_tree"
]

missing_objects = [
    object_name
    for object_name in required_object_names
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Phase 3 objects are missing: "
        + ", ".join(missing_objects)
    )


# ------------------------------------------------------------
# 2. Define controlled locations
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/Predictive_Maintenance_Thesis")
PREPROCESSOR_DIR = PROJECT_ROOT / "models" / "preprocessing"
LOGS_DIR = PROJECT_ROOT / "logs"

PREPROCESSOR_DIR.mkdir(
    parents=True,
    exist_ok=True
)

LOGS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

recorded_at_utc = datetime.now(
    timezone.utc
).isoformat()

scikit_learn_version = version(
    "scikit-learn"
)

joblib_version = version(
    "joblib"
)


# ------------------------------------------------------------
# 3. Record formal Step 2 approval
# ------------------------------------------------------------

step_2_closure_record = {
    "recorded_at_utc": recorded_at_utc,

    "phase": (
        "Phase 3 — Leakage-safe preprocessing"
    ),

    "step": "Step 2 — Ridge scaling",

    "status": "Approved and locked",

    "scaler_fit_partition": (
        "Model-training rows only"
    ),

    "fit_row_count":
        len(X_model_train_raw),

    "fit_feature_count":
        len(RIDGE_PREDICTOR_COLUMNS),

    "validation_operation":
        "Transform only",

    "calibration_operation":
        "Transform only",

    "tree_model_scaling":
        "None",

    "scaler_sha256": (
        "453719672e9627ab47d7fbb8b80398d4"
        "8768d70856643d3980e657fec87f293a"
    ),

    "regression_model_trained": False
}

step_2_closure_file = (
    LOGS_DIR
    / "phase3_step2_ridge_scaling_closure.json"
)

with step_2_closure_file.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        step_2_closure_record,
        file,
        indent=4
    )


# ------------------------------------------------------------
# 4. Create a compact preprocessing bundle
# ------------------------------------------------------------

preprocessing_bundle = {
    "bundle_type": (
        "FD001 leakage-safe preprocessing bundle"
    ),

    "created_at_utc":
        recorded_at_utc,

    "dataset":
        "NASA C-MAPSS FD001",

    "random_seed":
        42,

    "identifier_columns": [
        "unit_number"
    ],

    "primary_target":
        "rul_uncapped",

    "sensitivity_target":
        "rul_capped_130",

    "constant_measurement_columns":
        MODEL_TRAINING_CONSTANT_COLUMNS,

    "predictor_columns":
        PREDICTOR_COLUMNS,

    "ridge_predictor_columns":
        RIDGE_PREDICTOR_COLUMNS,

    "tree_predictor_columns":
        TREE_PREDICTOR_COLUMNS,

    "ridge_scaler":
        ridge_scaler,

    "ridge_scaling_policy": (
        "Fit on model-training rows only; "
        "transform all later partitions unchanged."
    ),

    "tree_scaling_policy":
        "No scaling",

    "missing_value_policy":
        "No imputation required",

    "feature_selection_performed":
        False,

    "software_versions": {
        "scikit_learn":
            scikit_learn_version,
        "joblib":
            joblib_version,
        "pandas":
            pd.__version__,
        "numpy":
            np.__version__
    }
}

preprocessing_bundle_file = (
    PREPROCESSOR_DIR
    / "fd001_preprocessing_bundle.joblib"
)

joblib.dump(
    preprocessing_bundle,
    preprocessing_bundle_file
)


# ------------------------------------------------------------
# 5. Reload the bundle independently
# ------------------------------------------------------------

reloaded_bundle = joblib.load(
    preprocessing_bundle_file
)

reloaded_predictor_columns = (
    reloaded_bundle[
        "predictor_columns"
    ]
)

reloaded_ridge_scaler = (
    reloaded_bundle[
        "ridge_scaler"
    ]
)


# ------------------------------------------------------------
# 6. Recreate every Ridge transformation
# ------------------------------------------------------------

recreated_model_training_array = (
    reloaded_ridge_scaler.transform(
        X_model_train_raw[
            reloaded_predictor_columns
        ]
    )
)

recreated_validation_array = (
    reloaded_ridge_scaler.transform(
        X_validation_raw[
            reloaded_predictor_columns
        ]
    )
)

recreated_calibration_array = (
    reloaded_ridge_scaler.transform(
        X_calibration_raw[
            reloaded_predictor_columns
        ]
    )
)


# ------------------------------------------------------------
# 7. Recreate every tree-model predictor view
# ------------------------------------------------------------

recreated_model_training_tree = (
    X_model_train_raw[
        reloaded_bundle[
            "tree_predictor_columns"
        ]
    ].copy()
)

recreated_validation_tree = (
    X_validation_raw[
        reloaded_bundle[
            "tree_predictor_columns"
        ]
    ].copy()
)

recreated_calibration_tree = (
    X_calibration_raw[
        reloaded_bundle[
            "tree_predictor_columns"
        ]
    ].copy()
)


# ------------------------------------------------------------
# 8. Create the human-readable model-input contract
# ------------------------------------------------------------

model_input_contract_text = f"""
# FD001 Model-Input Contract

Generated at UTC: {recorded_at_utc}

## Predictor boundary

Predictor count: {len(PREDICTOR_COLUMNS)}

Predictors:

{chr(10).join(f"- `{column}`" for column in PREDICTOR_COLUMNS)}

## Excluded identifier

- `unit_number`

The identifier is retained separately for engine-level
evaluation, bootstrapping and traceability. It is never supplied
to a prediction model.

## Excluded targets

- `rul_uncapped`
- `rul_capped_130`

Including either target among the predictors would cause direct
target leakage.

## Excluded model-training constants

{chr(10).join(
    f"- `{column}`"
    for column in MODEL_TRAINING_CONSTANT_COLUMNS
)}

These exclusions were determined using the model-training
partition only.

## Ridge Regression input

Ridge uses all 18 predictors transformed by one StandardScaler.

The scaler was fitted using only the 12,310 model-training rows.
Validation, calibration and future official-test predictors must
be transformed with that unchanged scaler.

## Tree-model input

Random Forest and provisional XGBoost use the same 18 retained
predictors without standardisation.

## Missing values

No imputation is performed because the governed FD001
partitions contain no missing or infinite predictor values.

## Feature selection

No model-based feature selection has occurred.

The four sensors highlighted during EDA were selected only for
descriptive visualisation. They are not the final model input set.

## Target policy

Primary modelling target:

`rul_uncapped`

Secondary sensitivity target:

`rul_capped_130`

## Current status

No prediction model has been fitted.
"""

model_input_contract_file = (
    LOGS_DIR
    / "phase3_model_input_contract.md"
)

model_input_contract_file.write_text(
    model_input_contract_text.strip(),
    encoding="utf-8"
)


# ------------------------------------------------------------
# 9. Validate bundle reproducibility
# ------------------------------------------------------------

NUMERICAL_TOLERANCE = 1e-12

bundle_checks = {
    "bundle_file_exists":
        preprocessing_bundle_file.is_file(),

    "bundle_contains_18_predictors":
        len(
            reloaded_predictor_columns
        ) == 18,

    "bundle_predictor_order_matches_locked_policy":
        (
            reloaded_predictor_columns
            == PREDICTOR_COLUMNS
        ),

    "bundle_excludes_unit_number":
        (
            "unit_number"
            not in reloaded_predictor_columns
        ),

    "bundle_excludes_primary_target":
        (
            "rul_uncapped"
            not in reloaded_predictor_columns
        ),

    "bundle_excludes_sensitivity_target":
        (
            "rul_capped_130"
            not in reloaded_predictor_columns
        ),

    "bundle_contains_seven_constant_exclusions":
        len(
            reloaded_bundle[
                "constant_measurement_columns"
            ]
        ) == 7,

    "recreated_model_training_scaling_matches":
        bool(
            np.allclose(
                recreated_model_training_array,
                X_model_train_ridge_scaled
                .to_numpy(dtype=float),
                atol=NUMERICAL_TOLERANCE,
                rtol=0
            )
        ),

    "recreated_validation_scaling_matches":
        bool(
            np.allclose(
                recreated_validation_array,
                X_validation_ridge_scaled
                .to_numpy(dtype=float),
                atol=NUMERICAL_TOLERANCE,
                rtol=0
            )
        ),

    "recreated_calibration_scaling_matches":
        bool(
            np.allclose(
                recreated_calibration_array,
                X_calibration_ridge_scaled
                .to_numpy(dtype=float),
                atol=NUMERICAL_TOLERANCE,
                rtol=0
            )
        ),

    "recreated_model_training_tree_view_matches":
        recreated_model_training_tree.equals(
            X_model_train_tree
        ),

    "recreated_validation_tree_view_matches":
        recreated_validation_tree.equals(
            X_validation_tree
        ),

    "recreated_calibration_tree_view_matches":
        recreated_calibration_tree.equals(
            X_calibration_tree
        ),

    "model_input_contract_exists":
        model_input_contract_file.is_file(),

    "no_feature_selection_was_performed":
        (
            reloaded_bundle[
                "feature_selection_performed"
            ] is False
        )
}


# ------------------------------------------------------------
# 10. Create the Phase 3 completion record
# ------------------------------------------------------------

phase_3_completion_record = {
    "recorded_at_utc": recorded_at_utc,

    "phase": (
        "Phase 3 — Leakage-safe preprocessing"
    ),

    "technical_status": (
        "All planned Phase 3 technical "
        "requirements completed"
    ),

    "formal_approval_status": (
        "Awaiting Ranitabh's Phase 3 approval"
    ),

    "completed_components": [
        "Predictor boundary frozen",
        "Identifiers excluded",
        "Targets excluded",
        "Seven constants excluded using model training only",
        "Eighteen predictors retained",
        "Ridge StandardScaler fitted on model training only",
        "Validation transformed without refitting",
        "Calibration transformed without refitting",
        "Unscaled tree-model views preserved",
        "Fitted transformer persisted",
        "Preprocessing bundle independently reloaded",
        "All transformations reproduced",
        "Model-input contract created"
    ],

    "locked_model_inputs": {
        "predictor_count":
            len(PREDICTOR_COLUMNS),

        "predictors":
            PREDICTOR_COLUMNS,

        "ridge_input":
            "Standardised predictors",

        "tree_input":
            "Unscaled predictors"
    },

    "raw_data_modified": False,

    "validation_used_for_fitting": False,

    "calibration_used_for_fitting": False,

    "official_test_data_used": False,

    "test_rul_values_used": False,

    "features_selected_by_model": False,

    "prediction_model_trained": False,

    "next_phase": (
        "Phase 4 — Baselines and predictive modelling"
    ),

    "bundle_checks":
        bundle_checks
}

phase_3_completion_file = (
    LOGS_DIR
    / "phase3_technical_completion_record.json"
)

with phase_3_completion_file.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        phase_3_completion_record,
        file,
        indent=4
    )


# ------------------------------------------------------------
# 11. Define evidence files
# ------------------------------------------------------------

phase_3_evidence_files = [
    LOGS_DIR
    / "phase2_formal_closure_record.json",

    LOGS_DIR
    / "phase3_preprocessing_feature_schema.csv",

    LOGS_DIR
    / "phase3_preprocessing_policy.json",

    LOGS_DIR
    / "phase3_step1_preprocessing_policy_closure.json",

    LOGS_DIR
    / "phase3_ridge_scaler_parameters.csv",

    LOGS_DIR
    / "phase3_ridge_scaling_record.json",

    LOGS_DIR
    / "phase3_step2_ridge_scaling_closure.json",

    PREPROCESSOR_DIR
    / "fd001_ridge_standard_scaler.joblib",

    preprocessing_bundle_file,

    model_input_contract_file,

    phase_3_completion_file
]


# ------------------------------------------------------------
# 12. Confirm that all evidence exists
# ------------------------------------------------------------

missing_evidence_files = [
    str(file_path)
    for file_path in phase_3_evidence_files
    if not file_path.is_file()
]

if missing_evidence_files:
    raise FileNotFoundError(
        "Missing Phase 3 evidence:\n"
        + "\n".join(missing_evidence_files)
    )


# ------------------------------------------------------------
# 13. Calculate evidence fingerprints
# ------------------------------------------------------------

def calculate_sha256(
    file_path,
    chunk_size=1024 * 1024
):
    sha256_hash = hashlib.sha256()

    with file_path.open("rb") as file:
        while True:
            data_chunk = file.read(
                chunk_size
            )

            if not data_chunk:
                break

            sha256_hash.update(
                data_chunk
            )

    return sha256_hash.hexdigest()


manifest_records = []

for evidence_file in phase_3_evidence_files:
    manifest_records.append({
        "file_name":
            evidence_file.name,

        "project_relative_path": str(
            evidence_file.relative_to(
                PROJECT_ROOT
            )
        ),

        "size_bytes": int(
            evidence_file.stat().st_size
        ),

        "sha256":
            calculate_sha256(
                evidence_file
            )
    })


phase_3_manifest_df = pd.DataFrame(
    manifest_records
)

phase_3_manifest_file = (
    LOGS_DIR
    / "phase3_complete_evidence_manifest.csv"
)

phase_3_manifest_df.to_csv(
    phase_3_manifest_file,
    index=False
)


# ------------------------------------------------------------
# 14. Complete Phase 3 checks
# ------------------------------------------------------------

phase_3_checks = {
    **bundle_checks,

    "all_phase_3_evidence_files_exist":
        len(missing_evidence_files) == 0,

    "all_evidence_files_have_sha256":
        bool(
            phase_3_manifest_df[
                "sha256"
            ].str.len().eq(64).all()
        ),

    "only_preprocessing_artifacts_exist_in_model_directory":
        all(
            "preprocessing"
            in file_path.parts
            for file_path in (
                PROJECT_ROOT / "models"
            ).rglob("*")
            if file_path.is_file()
        )
}


# ------------------------------------------------------------
# 15. Create consolidated Phase 3 ZIP
# ------------------------------------------------------------

phase_3_complete_zip = (
    PROJECT_ROOT
    / "Phase3_Complete_Preprocessing_Evidence.zip"
)

with zipfile.ZipFile(
    phase_3_complete_zip,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as zip_file:

    for evidence_file in phase_3_evidence_files:
        zip_file.write(
            evidence_file,
            arcname=str(
                evidence_file.relative_to(
                    PROJECT_ROOT
                )
            )
        )

    zip_file.write(
        phase_3_manifest_file,
        arcname=str(
            phase_3_manifest_file.relative_to(
                PROJECT_ROOT
            )
        )
    )


phase_3_zip_sha256 = calculate_sha256(
    phase_3_complete_zip
)


# ------------------------------------------------------------
# 16. Display final results
# ------------------------------------------------------------

print("PHASE 3 PREPROCESSING SUMMARY")
print("=" * 100)
print(
    f"Retained predictors: "
    f"{len(PREDICTOR_COLUMNS)}"
)
print(
    f"Excluded constants: "
    f"{len(MODEL_TRAINING_CONSTANT_COLUMNS)}"
)
print(
    "Ridge input: Model-training-fitted standardisation"
)
print(
    "Tree input: Unscaled retained predictors"
)
print(
    f"Scikit-learn version: "
    f"{scikit_learn_version}"
)
print(
    f"Joblib version: "
    f"{joblib_version}"
)

print("\nBUNDLE REPRODUCIBILITY")
print("=" * 100)

for check_name, passed in phase_3_checks.items():
    status = "PASS" if passed else "FAIL"

    print(
        f"{status} — "
        f"{check_name.replace('_', ' ')}"
    )

print("\nEVIDENCE PACKAGE")
print("=" * 100)
print(
    f"Files fingerprinted: "
    f"{len(phase_3_manifest_df)}"
)
print(
    f"Manifest: "
    f"{phase_3_manifest_file}"
)
print(
    f"ZIP: {phase_3_complete_zip}"
)
print(
    f"ZIP SHA-256: "
    f"{phase_3_zip_sha256}"
)

print("\nRESEARCH CONTROLS")
print("=" * 100)
print("Validation used for fitting: No")
print("Calibration used for fitting: No")
print("Official test data used: No")
print("Test RUL values accessed: No")
print("Feature selection performed: No")
print("Prediction model trained: No")

print("\nFINAL CHECK")
print("=" * 100)

if all(phase_3_checks.values()):
    print("PASS — Phase 3 technical requirements are complete.")
    print("PASS — The saved bundle reproduces every transformation.")
    print("PASS — Ridge and tree input policies are recoverable.")
    print("PASS — Every Phase 3 evidence file was fingerprinted.")
    print("PASS — No prediction model has been trained.")
    print("AWAITING APPROVAL — Phase 3 is not yet formally closed.")

    print("\nDownloading the complete Phase 3 ZIP...")
    colab_files.download(
        str(phase_3_complete_zip)
    )

else:
    print(
        "CHECK REQUIRED — "
        "Phase 3 completion validation failed."
    )

In [ ]:
# ============================================================
# PHASE 4 — BASELINES AND PREDICTIVE MODELLING
# STEP 1 — DEFINE AND EVALUATE THE NAIVE AGE-ONLY BASELINE
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import pandas as pd
import numpy as np
import json


# ------------------------------------------------------------
# 1. Confirm that the required governed objects exist
# ------------------------------------------------------------

required_object_names = [
    "model_train_targets_df",
    "validation_targets_df",
    "calibration_targets_df",
    "model_train_metadata_df",
    "validation_metadata_df",
    "y_validation_primary"
]

missing_objects = [
    object_name
    for object_name in required_object_names
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Required governed objects are missing: "
        + ", ".join(missing_objects)
    )


# ------------------------------------------------------------
# 2. Define output locations
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/Predictive_Maintenance_Thesis")
RESULTS_DIR = PROJECT_ROOT / "results"
LOGS_DIR = PROJECT_ROOT / "logs"

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

LOGS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

recorded_at_utc = datetime.now(
    timezone.utc
).isoformat()


# ------------------------------------------------------------
# 3. Formally record Phase 3 closure
# ------------------------------------------------------------

phase_3_closure_record = {
    "recorded_at_utc": recorded_at_utc,

    "phase": (
        "Phase 3 — Leakage-safe preprocessing"
    ),

    "status": "Approved, closed and locked",

    "complete_evidence_file": (
        "Phase3_Complete_"
        "Preprocessing_Evidence.zip"
    ),

    "complete_evidence_sha256": (
        "67a19d14f1ece47eea61a7118cbaa184"
        "495c7edad31be8ada5ad93e67d5a9730"
    ),

    "updated_notebook_downloaded": True,

    "locked_predictor_count": 18,

    "ridge_scaling_policy": (
        "StandardScaler fitted only on "
        "model-training rows"
    ),

    "tree_scaling_policy": "No scaling",

    "phase_4_previously_started": False
}

phase_3_closure_file = (
    LOGS_DIR
    / "phase3_formal_closure_record.json"
)

with phase_3_closure_file.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        phase_3_closure_record,
        file,
        indent=4
    )


# ------------------------------------------------------------
# 4. Calculate complete model-training engine lifetimes
# ------------------------------------------------------------

baseline_training_lifetimes_df = (
    model_train_targets_df
    .groupby(
        "unit_number",
        as_index=False
    )
    .agg(
        observed_failure_cycle=(
            "time_in_cycles",
            "max"
        )
    )
)

if len(
    baseline_training_lifetimes_df
) != 60:
    raise ValueError(
        "The naive baseline must use exactly "
        "60 model-training engines."
    )


# ------------------------------------------------------------
# 5. Freeze the naive reference lifetime
# ------------------------------------------------------------

training_median_lifetime_raw = float(
    baseline_training_lifetimes_df[
        "observed_failure_cycle"
    ].median()
)

# RUL is measured in whole operating cycles.
# np.rint applies deterministic nearest-integer rounding.

NAIVE_REFERENCE_LIFETIME = int(
    np.rint(
        training_median_lifetime_raw
    )
)


# ------------------------------------------------------------
# 6. Define the age-only baseline function
# ------------------------------------------------------------

def predict_naive_age_based_rul(
    cycle_values,
    reference_lifetime
):
    """
    Predict RUL using only current engine age:

        predicted RUL =
        max(reference lifetime - current cycle, 0)

    No sensor measurement or engine identifier is used.
    """

    cycle_array = np.asarray(
        cycle_values,
        dtype=float
    )

    predictions = np.maximum(
        reference_lifetime
        - cycle_array,
        0
    )

    return predictions.astype(float)


# ------------------------------------------------------------
# 7. Generate validation predictions
# ------------------------------------------------------------

naive_validation_predictions = (
    predict_naive_age_based_rul(
        cycle_values=validation_metadata_df[
            "time_in_cycles"
        ],
        reference_lifetime=(
            NAIVE_REFERENCE_LIFETIME
        )
    )
)


# ------------------------------------------------------------
# 8. Build the validation prediction table
# ------------------------------------------------------------

naive_validation_predictions_df = pd.DataFrame({
    "unit_number":
        validation_metadata_df[
            "unit_number"
        ].to_numpy(),

    "time_in_cycles":
        validation_metadata_df[
            "time_in_cycles"
        ].to_numpy(),

    "actual_rul_uncapped":
        y_validation_primary.to_numpy(
            dtype=float
        ),

    "predicted_rul_naive":
        naive_validation_predictions
})


# ------------------------------------------------------------
# 9. Calculate errors
# ------------------------------------------------------------

naive_validation_predictions_df[
    "prediction_error"
] = (
    naive_validation_predictions_df[
        "predicted_rul_naive"
    ]
    - naive_validation_predictions_df[
        "actual_rul_uncapped"
    ]
)

naive_validation_predictions_df[
    "absolute_error"
] = (
    naive_validation_predictions_df[
        "prediction_error"
    ].abs()
)

naive_validation_predictions_df[
    "squared_error"
] = (
    naive_validation_predictions_df[
        "prediction_error"
    ] ** 2
)

naive_validation_predictions_df[
    "error_direction"
] = np.select(
    [
        naive_validation_predictions_df[
            "prediction_error"
        ] > 0,

        naive_validation_predictions_df[
            "prediction_error"
        ] < 0
    ],
    [
        "RUL overestimated",
        "RUL underestimated"
    ],
    default="Exact"
)


# ------------------------------------------------------------
# 10. Define the PHM08-style asymmetric penalty
# ------------------------------------------------------------

def calculate_phm_asymmetric_penalty(
    prediction_errors
):
    """
    Error definition:

        error = predicted RUL - actual RUL

    Overestimation (error >= 0) receives the stronger
    exponential penalty with denominator 10.

    Underestimation (error < 0) uses denominator 13.
    """

    error_array = np.asarray(
        prediction_errors,
        dtype=float
    )

    penalties = np.where(
        error_array >= 0,
        np.expm1(
            error_array / 10.0
        ),
        np.expm1(
            -error_array / 13.0
        )
    )

    return penalties


naive_validation_predictions_df[
    "phm_asymmetric_penalty"
] = calculate_phm_asymmetric_penalty(
    naive_validation_predictions_df[
        "prediction_error"
    ]
)


# ------------------------------------------------------------
# 11. Calculate pooled validation metrics
# ------------------------------------------------------------

actual_values = (
    naive_validation_predictions_df[
        "actual_rul_uncapped"
    ].to_numpy(dtype=float)
)

predicted_values = (
    naive_validation_predictions_df[
        "predicted_rul_naive"
    ].to_numpy(dtype=float)
)

prediction_errors = (
    naive_validation_predictions_df[
        "prediction_error"
    ].to_numpy(dtype=float)
)

pooled_mae = float(
    mean_absolute_error(
        actual_values,
        predicted_values
    )
)

pooled_rmse = float(
    np.sqrt(
        mean_squared_error(
            actual_values,
            predicted_values
        )
    )
)

pooled_r_squared = float(
    r2_score(
        actual_values,
        predicted_values
    )
)

pooled_bias = float(
    prediction_errors.mean()
)

rul_overestimation_rate = float(
    np.mean(
        prediction_errors > 0
    )
)

rul_underestimation_rate = float(
    np.mean(
        prediction_errors < 0
    )
)

exact_prediction_rate = float(
    np.mean(
        prediction_errors == 0
    )
)

row_level_phm_penalty_total = float(
    naive_validation_predictions_df[
        "phm_asymmetric_penalty"
    ].sum()
)

row_level_phm_penalty_mean = float(
    naive_validation_predictions_df[
        "phm_asymmetric_penalty"
    ].mean()
)


# ------------------------------------------------------------
# 12. Calculate metrics separately for each validation engine
# ------------------------------------------------------------

engine_metric_records = []

for engine_id, engine_data in (
    naive_validation_predictions_df.groupby(
        "unit_number",
        sort=True
    )
):
    engine_actual = engine_data[
        "actual_rul_uncapped"
    ].to_numpy(dtype=float)

    engine_predicted = engine_data[
        "predicted_rul_naive"
    ].to_numpy(dtype=float)

    engine_errors = engine_data[
        "prediction_error"
    ].to_numpy(dtype=float)

    engine_metric_records.append({
        "unit_number": int(engine_id),

        "cycle_row_count": int(
            len(engine_data)
        ),

        "observed_lifetime_cycles": int(
            engine_data[
                "time_in_cycles"
            ].max()
        ),

        "mae": float(
            mean_absolute_error(
                engine_actual,
                engine_predicted
            )
        ),

        "rmse": float(
            np.sqrt(
                mean_squared_error(
                    engine_actual,
                    engine_predicted
                )
            )
        ),

        "bias_predicted_minus_actual":
            float(
                engine_errors.mean()
            ),

        "rul_overestimation_rate":
            float(
                np.mean(
                    engine_errors > 0
                )
            ),

        "rul_underestimation_rate":
            float(
                np.mean(
                    engine_errors < 0
                )
            ),

        "mean_phm_asymmetric_penalty":
            float(
                engine_data[
                    "phm_asymmetric_penalty"
                ].mean()
            )
    })


naive_engine_metrics_df = pd.DataFrame(
    engine_metric_records
)


# ------------------------------------------------------------
# 13. Summarise engine-balanced performance
# ------------------------------------------------------------

engine_balanced_mean_mae = float(
    naive_engine_metrics_df[
        "mae"
    ].mean()
)

engine_balanced_mean_rmse = float(
    naive_engine_metrics_df[
        "rmse"
    ].mean()
)

median_engine_rmse = float(
    naive_engine_metrics_df[
        "rmse"
    ].median()
)

engine_rmse_standard_deviation = float(
    naive_engine_metrics_df[
        "rmse"
    ].std(ddof=1)
)

engine_rmse_first_quartile = float(
    naive_engine_metrics_df[
        "rmse"
    ].quantile(0.25)
)

engine_rmse_third_quartile = float(
    naive_engine_metrics_df[
        "rmse"
    ].quantile(0.75)
)


# ------------------------------------------------------------
# 14. Create the metric summary table
# ------------------------------------------------------------

naive_metric_summary_df = pd.DataFrame([
    {
        "model_name":
            "Naive age-only baseline",

        "target":
            "rul_uncapped",

        "validation_engine_count":
            int(
                naive_validation_predictions_df[
                    "unit_number"
                ].nunique()
            ),

        "validation_cycle_row_count":
            int(
                len(
                    naive_validation_predictions_df
                )
            ),

        "training_reference_lifetime":
            NAIVE_REFERENCE_LIFETIME,

        "pooled_mae":
            pooled_mae,

        "pooled_rmse":
            pooled_rmse,

        "pooled_r_squared":
            pooled_r_squared,

        "pooled_bias_predicted_minus_actual":
            pooled_bias,

        "engine_balanced_mean_mae":
            engine_balanced_mean_mae,

        "engine_balanced_mean_rmse":
            engine_balanced_mean_rmse,

        "median_engine_rmse":
            median_engine_rmse,

        "engine_rmse_standard_deviation":
            engine_rmse_standard_deviation,

        "engine_rmse_first_quartile":
            engine_rmse_first_quartile,

        "engine_rmse_third_quartile":
            engine_rmse_third_quartile,

        "rul_overestimation_rate":
            rul_overestimation_rate,

        "rul_underestimation_rate":
            rul_underestimation_rate,

        "exact_prediction_rate":
            exact_prediction_rate,

        "row_level_phm_penalty_total_diagnostic":
            row_level_phm_penalty_total,

        "row_level_phm_penalty_mean_diagnostic":
            row_level_phm_penalty_mean
    }
])


# ------------------------------------------------------------
# 15. Save predictions and metrics
# ------------------------------------------------------------

naive_predictions_file = (
    RESULTS_DIR
    / "phase4_naive_baseline_validation_predictions.csv"
)

naive_engine_metrics_file = (
    RESULTS_DIR
    / "phase4_naive_baseline_engine_metrics.csv"
)

naive_metric_summary_file = (
    RESULTS_DIR
    / "phase4_naive_baseline_metric_summary.csv"
)

naive_validation_predictions_df.to_csv(
    naive_predictions_file,
    index=False
)

naive_engine_metrics_df.to_csv(
    naive_engine_metrics_file,
    index=False
)

naive_metric_summary_df.to_csv(
    naive_metric_summary_file,
    index=False
)


# ------------------------------------------------------------
# 16. Define and save the baseline policy
# ------------------------------------------------------------

naive_baseline_definition = {
    "recorded_at_utc": recorded_at_utc,

    "phase": (
        "Phase 4 — Baselines and "
        "predictive modelling"
    ),

    "step": (
        "Step 1 — Naive age-only baseline"
    ),

    "model_name": (
        "Naive age-only baseline"
    ),

    "formula": (
        "maximum of "
        "(median model-training lifetime "
        "- current cycle) and zero"
    ),

    "reference_statistic": (
        "Median complete lifetime of the "
        "60 model-training engines"
    ),

    "raw_training_median_lifetime":
        training_median_lifetime_raw,

    "reference_lifetime_cycles":
        NAIVE_REFERENCE_LIFETIME,

    "prediction_inputs": [
        "time_in_cycles"
    ],

    "sensor_inputs_used": [],

    "unit_number_used_as_predictor":
        False,

    "target":
        "rul_uncapped",

    "validation_role": (
        "Evaluation only"
    ),

    "calibration_data_used":
        False,

    "official_test_data_used":
        False,

    "test_rul_values_used":
        False,

    "phm_penalty_note": (
        "The PHM08-style penalty is calculated "
        "across validation cycle rows as a "
        "direction-sensitive diagnostic. It is "
        "not labelled as the official challenge "
        "score, which is normally evaluated on "
        "one prediction per test engine."
    ),

    "selection_rule": (
        "This baseline was defined before "
        "examining its validation performance."
    )
}

naive_baseline_definition_file = (
    LOGS_DIR
    / "phase4_naive_baseline_definition.json"
)

with naive_baseline_definition_file.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        naive_baseline_definition,
        file,
        indent=4
    )


# ------------------------------------------------------------
# 17. Validate the baseline experiment
# ------------------------------------------------------------

expected_predictions = np.maximum(
    NAIVE_REFERENCE_LIFETIME
    - validation_metadata_df[
        "time_in_cycles"
    ].to_numpy(dtype=float),
    0
)

baseline_checks = {
    "reference_lifetime_uses_60_training_engines":
        len(
            baseline_training_lifetimes_df
        ) == 60,

    "reference_lifetime_matches_training_median":
        NAIVE_REFERENCE_LIFETIME
        == int(
            np.rint(
                baseline_training_lifetimes_df[
                    "observed_failure_cycle"
                ].median()
            )
        ),

    "validation_contains_20_engines":
        int(
            naive_validation_predictions_df[
                "unit_number"
            ].nunique()
        ) == 20,

    "validation_row_count_is_preserved":
        len(
            naive_validation_predictions_df
        ) == len(validation_targets_df),

    "prediction_formula_is_correct":
        bool(
            np.array_equal(
                naive_validation_predictions,
                expected_predictions
            )
        ),

    "all_predictions_are_non_negative":
        bool(
            (
                naive_validation_predictions
                >= 0
            ).all()
        ),

    "all_predictions_are_finite":
        bool(
            np.isfinite(
                naive_validation_predictions
            ).all()
        ),

    "actual_target_alignment_is_preserved":
        bool(
            np.array_equal(
                naive_validation_predictions_df[
                    "actual_rul_uncapped"
                ].to_numpy(dtype=float),
                y_validation_primary.to_numpy(
                    dtype=float
                )
            )
        ),

    "unit_number_was_not_used_as_predictor":
        True,

    "no_sensor_was_used_by_baseline":
        True,

    "all_pooled_metrics_are_finite":
        bool(
            np.isfinite([
                pooled_mae,
                pooled_rmse,
                pooled_r_squared,
                pooled_bias,
                row_level_phm_penalty_total,
                row_level_phm_penalty_mean
            ]).all()
        ),

    "engine_metrics_contain_20_engines":
        len(
            naive_engine_metrics_df
        ) == 20,

    "all_engine_metrics_are_finite":
        bool(
            np.isfinite(
                naive_engine_metrics_df[
                    [
                        "mae",
                        "rmse",
                        "bias_predicted_minus_actual",
                        "rul_overestimation_rate",
                        "rul_underestimation_rate",
                        "mean_phm_asymmetric_penalty"
                    ]
                ].to_numpy(dtype=float)
            ).all()
        ),

    "calibration_dataframe_was_not_modified":
        (
            "predicted_rul_naive"
            not in calibration_targets_df.columns
        ),

    "official_test_data_were_not_used":
        True,

    "result_files_exist":
        all(
            file_path.is_file()
            for file_path in [
                naive_predictions_file,
                naive_engine_metrics_file,
                naive_metric_summary_file,
                naive_baseline_definition_file
            ]
        )
}


# ------------------------------------------------------------
# 18. Display the results
# ------------------------------------------------------------

print("NAIVE BASELINE DEFINITION")
print("=" * 100)
print(
    "Training engines used to define baseline: "
    f"{len(baseline_training_lifetimes_df)}"
)
print(
    "Raw model-training median lifetime: "
    f"{training_median_lifetime_raw}"
)
print(
    "Frozen reference lifetime: "
    f"{NAIVE_REFERENCE_LIFETIME} cycles"
)
print(
    "Prediction formula: "
    "max(reference lifetime - current cycle, 0)"
)
print("Sensors used: None")

print("\nPOOLED VALIDATION METRICS")
print("=" * 100)
print(f"MAE: {pooled_mae:.6f}")
print(f"RMSE: {pooled_rmse:.6f}")
print(f"R-squared: {pooled_r_squared:.6f}")
print(
    "Bias (predicted - actual): "
    f"{pooled_bias:.6f}"
)
print(
    "RUL-overestimation rate: "
    f"{rul_overestimation_rate:.4%}"
)
print(
    "RUL-underestimation rate: "
    f"{rul_underestimation_rate:.4%}"
)
print(
    "Mean row-level PHM penalty diagnostic: "
    f"{row_level_phm_penalty_mean:.6f}"
)

print("\nENGINE-BALANCED VALIDATION METRICS")
print("=" * 100)
print(
    "Mean engine MAE: "
    f"{engine_balanced_mean_mae:.6f}"
)
print(
    "Mean engine RMSE: "
    f"{engine_balanced_mean_rmse:.6f}"
)
print(
    "Median engine RMSE: "
    f"{median_engine_rmse:.6f}"
)
print(
    "Engine RMSE standard deviation: "
    f"{engine_rmse_standard_deviation:.6f}"
)
print(
    "Engine RMSE IQR: "
    f"[{engine_rmse_first_quartile:.6f}, "
    f"{engine_rmse_third_quartile:.6f}]"
)

print("\nBASELINE VALIDATION")
print("=" * 100)

for check_name, passed in baseline_checks.items():
    status = "PASS" if passed else "FAIL"

    print(
        f"{status} — "
        f"{check_name.replace('_', ' ')}"
    )

print("\nRESEARCH CONTROLS")
print("=" * 100)
print("Validation used to define baseline: No")
print("Calibration data used: No")
print("Official test data used: No")
print("Test RUL values accessed: No")
print("Machine-learning model fitted: No")

print("\nFINAL CHECK")
print("=" * 100)

if all(baseline_checks.values()):
    print("PASS — Naive baseline was defined and evaluated correctly.")
    print("PASS — Its only input was current cycle.")
    print("PASS — Validation was used only for evaluation.")
    print("PASS — Calibration and official test data remained untouched.")
    print("PASS — No machine-learning model was fitted.")
    print("AWAITING APPROVAL — Ridge Regression has not started.")

else:
    print(
        "CHECK REQUIRED — "
        "Naive-baseline validation failed."
    )

In [ ]:
# ======================================================================================
# PHASE 4 — STEP 2
# RIDGE REGRESSION WITH TRAINING-ONLY ENGINE-GROUPED HYPERPARAMETER SELECTION
# ======================================================================================

from pathlib import Path
import hashlib
import json
import platform

import joblib
import numpy as np
import pandas as pd
import sklearn

from IPython.display import display
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    root_mean_squared_error,
)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# ======================================================================================
# 1. FIXED PROJECT PATHS
# ======================================================================================

PROJECT_ROOT = Path("/content/Predictive_Maintenance_Thesis")

TRAIN_TARGET_PATH = (
    PROJECT_ROOT
    / "data/interim/fd001_model_training_targets.csv"
)

VALIDATION_TARGET_PATH = (
    PROJECT_ROOT
    / "data/interim/fd001_validation_targets.csv"
)

CALIBRATION_TARGET_PATH = (
    PROJECT_ROOT
    / "data/interim/fd001_calibration_targets.csv"
)

SCALER_PATH = (
    PROJECT_ROOT
    / "models/preprocessing/fd001_ridge_standard_scaler.joblib"
)

RESULTS_DIR = PROJECT_ROOT / "results"
LOGS_DIR = PROJECT_ROOT / "logs"
MODEL_DIR = PROJECT_ROOT / "models/regression"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)


# ======================================================================================
# 2. OUTPUT PATHS
# ======================================================================================

RIDGE_CV_PATH = (
    RESULTS_DIR
    / "phase4_ridge_training_grouped_cv_results.csv"
)

RIDGE_PREDICTIONS_PATH = (
    RESULTS_DIR
    / "phase4_ridge_validation_predictions.csv"
)

RIDGE_ENGINE_METRICS_PATH = (
    RESULTS_DIR
    / "phase4_ridge_validation_engine_metrics.csv"
)

RIDGE_COMPARISON_PATH = (
    RESULTS_DIR
    / "phase4_naive_vs_ridge_metric_comparison.csv"
)

RIDGE_COEFFICIENTS_PATH = (
    RESULTS_DIR
    / "phase4_ridge_standardised_coefficients.csv"
)

RIDGE_DEFINITION_PATH = (
    LOGS_DIR
    / "phase4_ridge_model_definition.json"
)

RIDGE_MODEL_PATH = (
    MODEL_DIR
    / "fd001_ridge_uncapped_bundle.joblib"
)


# ======================================================================================
# 3. HELPER FUNCTIONS
# ======================================================================================

def file_sha256(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file_handle:
        for block in iter(
            lambda: file_handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def dataframe_sha256(dataframe):
    row_hashes = pd.util.hash_pandas_object(
        dataframe,
        index=True,
    ).values.tobytes()

    return hashlib.sha256(row_hashes).hexdigest()


def calculate_phm_penalty(actual, predicted):
    """
    Row-level asymmetric PHM-style diagnostic.

    Positive error:
        predicted RUL > actual RUL
        potentially unsafe RUL overestimation

    Negative error:
        predicted RUL < actual RUL
        conservative RUL underestimation

    This remains a diagnostic and is not claimed as the
    official PHM Challenge test score.
    """

    error = predicted - actual

    return np.where(
        error >= 0,
        np.expm1(error / 10.0),
        np.expm1(-error / 13.0),
    )


def pooled_metric_record(model_name, actual, predicted):
    error = predicted - actual
    phm_penalties = calculate_phm_penalty(
        actual,
        predicted,
    )

    return {
        "model": model_name,
        "pooled_mae": float(
            mean_absolute_error(actual, predicted)
        ),
        "pooled_rmse": float(
            root_mean_squared_error(actual, predicted)
        ),
        "pooled_r_squared": float(
            r2_score(actual, predicted)
        ),
        "bias_predicted_minus_actual": float(
            np.mean(error)
        ),
        "rul_overestimation_rate_percent": float(
            np.mean(error > 0) * 100
        ),
        "rul_underestimation_rate_percent": float(
            np.mean(error < 0) * 100
        ),
        "exact_prediction_rate_percent": float(
            np.mean(
                np.isclose(
                    error,
                    0.0,
                    atol=1e-12,
                )
            ) * 100
        ),
        "mean_row_level_phm_penalty": float(
            np.mean(phm_penalties)
        ),
    }


def calculate_engine_metrics(
    unit_numbers,
    actual,
    predicted,
    model_name,
):
    metric_rows = []

    metric_frame = pd.DataFrame(
        {
            "unit_number": np.asarray(unit_numbers),
            "actual_rul": np.asarray(actual, dtype=float),
            "predicted_rul": np.asarray(
                predicted,
                dtype=float,
            ),
        }
    )

    for unit_number, engine_frame in metric_frame.groupby(
        "unit_number",
        sort=True,
    ):
        engine_actual = engine_frame[
            "actual_rul"
        ].to_numpy()

        engine_predicted = engine_frame[
            "predicted_rul"
        ].to_numpy()

        engine_error = (
            engine_predicted - engine_actual
        )

        metric_rows.append(
            {
                "model": model_name,
                "unit_number": int(unit_number),
                "row_count": int(len(engine_frame)),
                "mae": float(
                    mean_absolute_error(
                        engine_actual,
                        engine_predicted,
                    )
                ),
                "rmse": float(
                    root_mean_squared_error(
                        engine_actual,
                        engine_predicted,
                    )
                ),
                "bias_predicted_minus_actual": float(
                    np.mean(engine_error)
                ),
            }
        )

    return pd.DataFrame(metric_rows)


def engine_summary_record(
    model_name,
    engine_metrics,
):
    return {
        "model": model_name,
        "mean_engine_mae": float(
            engine_metrics["mae"].mean()
        ),
        "mean_engine_rmse": float(
            engine_metrics["rmse"].mean()
        ),
        "median_engine_rmse": float(
            engine_metrics["rmse"].median()
        ),
        "engine_rmse_standard_deviation": float(
            engine_metrics["rmse"].std(ddof=1)
        ),
        "engine_rmse_first_quartile": float(
            engine_metrics["rmse"].quantile(0.25)
        ),
        "engine_rmse_third_quartile": float(
            engine_metrics["rmse"].quantile(0.75)
        ),
    }


# ======================================================================================
# 4. LOAD ONLY THE PERMITTED INPUTS
# ======================================================================================

required_paths = [
    TRAIN_TARGET_PATH,
    VALIDATION_TARGET_PATH,
    CALIBRATION_TARGET_PATH,
    SCALER_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Phase 2 or Phase 3 files are missing:\n"
        + "\n".join(missing_paths)
    )

model_training_targets = pd.read_csv(
    TRAIN_TARGET_PATH
)

validation_targets = pd.read_csv(
    VALIDATION_TARGET_PATH
)

# The calibration values are deliberately not loaded.
# Only the file hash is recorded for non-modification evidence.
calibration_hash_before = file_sha256(
    CALIBRATION_TARGET_PATH
)

ridge_scaler = joblib.load(SCALER_PATH)


# ======================================================================================
# 5. FREEZE SOURCE-DATA INTEGRITY
# ======================================================================================

training_dataframe_hash_before = dataframe_sha256(
    model_training_targets
)

validation_dataframe_hash_before = dataframe_sha256(
    validation_targets
)

scaler_hash = file_sha256(SCALER_PATH)


# ======================================================================================
# 6. RECOVER THE LOCKED RIDGE FEATURE ORDER
# ======================================================================================

if not hasattr(ridge_scaler, "feature_names_in_"):
    raise AssertionError(
        "The saved scaler does not contain feature_names_in_. "
        "The locked predictor order cannot be recovered safely."
    )

ridge_features = list(
    ridge_scaler.feature_names_in_
)

TARGET_COLUMN = "rul_uncapped"
UNIT_COLUMN = "unit_number"
CYCLE_COLUMN = "time_in_cycles"

required_columns = (
    ridge_features
    + [
        UNIT_COLUMN,
        CYCLE_COLUMN,
        TARGET_COLUMN,
    ]
)

for partition_name, partition_frame in [
    ("model training", model_training_targets),
    ("validation", validation_targets),
]:
    missing_columns = [
        column
        for column in required_columns
        if column not in partition_frame.columns
    ]

    if missing_columns:
        raise AssertionError(
            f"{partition_name} is missing columns: "
            f"{missing_columns}"
        )


# ======================================================================================
# 7. PREPARE TRAINING AND VALIDATION ARRAYS
# ======================================================================================

X_training_raw = model_training_targets[
    ridge_features
].copy()

X_validation_raw = validation_targets[
    ridge_features
].copy()

y_training = model_training_targets[
    TARGET_COLUMN
].to_numpy(dtype=float)

y_validation = validation_targets[
    TARGET_COLUMN
].to_numpy(dtype=float)

training_groups = model_training_targets[
    UNIT_COLUMN
].to_numpy()

validation_units = validation_targets[
    UNIT_COLUMN
].to_numpy()

training_engine_ids = set(
    model_training_targets[UNIT_COLUMN].unique()
)

validation_engine_ids = set(
    validation_targets[UNIT_COLUMN].unique()
)


# ======================================================================================
# 8. PREDECLARE RIDGE HYPERPARAMETER POLICY
# ======================================================================================

RANDOM_SEED = 42
NUMBER_OF_FOLDS = 5

# Fixed before examining validation performance.
ALPHA_GRID = np.logspace(
    -4,
    6,
    11,
    dtype=float,
)

# Operational RUL predictions cannot be below zero.
PREDICTION_LOWER_BOUND = 0.0

grouped_cross_validator = GroupKFold(
    n_splits=NUMBER_OF_FOLDS,
    shuffle=True,
    random_state=RANDOM_SEED,
)

cv_splits = list(
    grouped_cross_validator.split(
        X_training_raw,
        y_training,
        groups=training_groups,
    )
)

fold_group_separation_checks = []

for fit_indices, holdout_indices in cv_splits:
    fit_engines = set(
        training_groups[fit_indices]
    )

    holdout_engines = set(
        training_groups[holdout_indices]
    )

    fold_group_separation_checks.append(
        fit_engines.isdisjoint(holdout_engines)
    )


# ======================================================================================
# 9. TRAINING-ONLY GROUPED CROSS-VALIDATION
# ======================================================================================

cv_result_rows = []
all_oof_predictions_complete = True

for alpha_value in ALPHA_GRID:
    oof_predictions = np.full(
        len(model_training_targets),
        np.nan,
        dtype=float,
    )

    negative_raw_prediction_rows = 0

    for fold_number, (
        fit_indices,
        holdout_indices,
    ) in enumerate(cv_splits, start=1):

        # A new scaler is fitted inside every fold.
        # This prevents held-out training engines from contributing
        # scaling statistics during hyperparameter selection.
        fold_pipeline = Pipeline(
            steps=[
                (
                    "scaler",
                    StandardScaler(),
                ),
                (
                    "ridge",
                    Ridge(
                        alpha=float(alpha_value),
                        fit_intercept=True,
                        solver="svd",
                    ),
                ),
            ]
        )

        fold_pipeline.fit(
            X_training_raw.iloc[fit_indices],
            y_training[fit_indices],
        )

        raw_fold_predictions = fold_pipeline.predict(
            X_training_raw.iloc[holdout_indices]
        )

        negative_raw_prediction_rows += int(
            np.sum(raw_fold_predictions < 0)
        )

        oof_predictions[holdout_indices] = np.clip(
            raw_fold_predictions,
            PREDICTION_LOWER_BOUND,
            None,
        )

    current_oof_complete = bool(
        np.isfinite(oof_predictions).all()
    )

    all_oof_predictions_complete = bool(
        all_oof_predictions_complete
        and current_oof_complete
    )

    oof_engine_metrics = calculate_engine_metrics(
        unit_numbers=training_groups,
        actual=y_training,
        predicted=oof_predictions,
        model_name=f"ridge_alpha_{alpha_value:g}",
    )

    cv_result_rows.append(
        {
            "alpha": float(alpha_value),
            "cv_pooled_mae": float(
                mean_absolute_error(
                    y_training,
                    oof_predictions,
                )
            ),
            "cv_pooled_rmse": float(
                root_mean_squared_error(
                    y_training,
                    oof_predictions,
                )
            ),
            "cv_pooled_r_squared": float(
                r2_score(
                    y_training,
                    oof_predictions,
                )
            ),
            "cv_mean_engine_mae": float(
                oof_engine_metrics["mae"].mean()
            ),
            "cv_mean_engine_rmse": float(
                oof_engine_metrics["rmse"].mean()
            ),
            "cv_median_engine_rmse": float(
                oof_engine_metrics["rmse"].median()
            ),
            "negative_raw_oof_prediction_rows": int(
                negative_raw_prediction_rows
            ),
        }
    )

ridge_cv_results = (
    pd.DataFrame(cv_result_rows)
    .sort_values("alpha")
    .reset_index(drop=True)
)

# Primary selection criterion:
# lowest mean engine RMSE across training-only OOF predictions.
best_cv_row_index = ridge_cv_results[
    "cv_mean_engine_rmse"
].idxmin()

selected_alpha = float(
    ridge_cv_results.loc[
        best_cv_row_index,
        "alpha",
    ]
)

selected_alpha_is_boundary = bool(
    np.isclose(selected_alpha, ALPHA_GRID.min())
    or np.isclose(selected_alpha, ALPHA_GRID.max())
)


# ======================================================================================
# 10. APPLY THE FROZEN PHASE 3 SCALER
# ======================================================================================

X_training_scaled = pd.DataFrame(
    ridge_scaler.transform(X_training_raw),
    columns=ridge_features,
    index=model_training_targets.index,
)

X_validation_scaled = pd.DataFrame(
    ridge_scaler.transform(X_validation_raw),
    columns=ridge_features,
    index=validation_targets.index,
)


# ======================================================================================
# 11. FIT THE FINAL RIDGE MODEL ON ALL 60 TRAINING ENGINES
# ======================================================================================

ridge_model = Ridge(
    alpha=selected_alpha,
    fit_intercept=True,
    solver="svd",
)

ridge_model.fit(
    X_training_scaled,
    y_training,
)

raw_ridge_validation_predictions = (
    ridge_model.predict(X_validation_scaled)
)

ridge_validation_predictions = np.clip(
    raw_ridge_validation_predictions,
    PREDICTION_LOWER_BOUND,
    None,
)

negative_raw_validation_prediction_rows = int(
    np.sum(raw_ridge_validation_predictions < 0)
)


# ======================================================================================
# 12. RECONSTRUCT THE APPROVED NAIVE BASELINE
# ======================================================================================

training_lifetimes = (
    model_training_targets
    .groupby(UNIT_COLUMN)[CYCLE_COLUMN]
    .max()
)

baseline_reference_lifetime = int(
    round(training_lifetimes.median())
)

naive_validation_predictions = np.maximum(
    baseline_reference_lifetime
    - validation_targets[CYCLE_COLUMN].to_numpy(),
    0,
).astype(float)


# ======================================================================================
# 13. COMPUTE POOLED AND ENGINE-BALANCED METRICS
# ======================================================================================

naive_pooled_metrics = pooled_metric_record(
    model_name="naive_age_only",
    actual=y_validation,
    predicted=naive_validation_predictions,
)

ridge_pooled_metrics = pooled_metric_record(
    model_name="ridge_uncapped",
    actual=y_validation,
    predicted=ridge_validation_predictions,
)

naive_engine_metrics = calculate_engine_metrics(
    unit_numbers=validation_units,
    actual=y_validation,
    predicted=naive_validation_predictions,
    model_name="naive_age_only",
)

ridge_engine_metrics = calculate_engine_metrics(
    unit_numbers=validation_units,
    actual=y_validation,
    predicted=ridge_validation_predictions,
    model_name="ridge_uncapped",
)

naive_engine_summary = engine_summary_record(
    model_name="naive_age_only",
    engine_metrics=naive_engine_metrics,
)

ridge_engine_summary = engine_summary_record(
    model_name="ridge_uncapped",
    engine_metrics=ridge_engine_metrics,
)

naive_complete_record = {
    **naive_pooled_metrics,
    **{
        key: value
        for key, value in naive_engine_summary.items()
        if key != "model"
    },
}

ridge_complete_record = {
    **ridge_pooled_metrics,
    **{
        key: value
        for key, value in ridge_engine_summary.items()
        if key != "model"
    },
}

ridge_model_comparison = pd.DataFrame(
    [
        naive_complete_record,
        ridge_complete_record,
    ]
)

pooled_rmse_reduction = float(
    naive_pooled_metrics["pooled_rmse"]
    - ridge_pooled_metrics["pooled_rmse"]
)

pooled_rmse_reduction_percent = float(
    pooled_rmse_reduction
    / naive_pooled_metrics["pooled_rmse"]
    * 100
)

engine_balanced_rmse_reduction = float(
    naive_engine_summary["mean_engine_rmse"]
    - ridge_engine_summary["mean_engine_rmse"]
)

engine_balanced_rmse_reduction_percent = float(
    engine_balanced_rmse_reduction
    / naive_engine_summary["mean_engine_rmse"]
    * 100
)


# ======================================================================================
# 14. CREATE VALIDATION PREDICTION AND COEFFICIENT TABLES
# ======================================================================================

ridge_validation_output = pd.DataFrame(
    {
        "unit_number": validation_targets[
            UNIT_COLUMN
        ].to_numpy(),
        "time_in_cycles": validation_targets[
            CYCLE_COLUMN
        ].to_numpy(),
        "actual_rul_uncapped": y_validation,
        "ridge_raw_prediction": (
            raw_ridge_validation_predictions
        ),
        "ridge_prediction": (
            ridge_validation_predictions
        ),
        "prediction_was_lower_clipped": (
            raw_ridge_validation_predictions < 0
        ),
        "error_predicted_minus_actual": (
            ridge_validation_predictions
            - y_validation
        ),
        "absolute_error": np.abs(
            ridge_validation_predictions
            - y_validation
        ),
    }
)

ridge_coefficients = pd.DataFrame(
    {
        "feature": ridge_features,
        "standardised_coefficient": (
            ridge_model.coef_
        ),
    }
)

ridge_coefficients[
    "absolute_standardised_coefficient"
] = ridge_coefficients[
    "standardised_coefficient"
].abs()

ridge_coefficients = (
    ridge_coefficients
    .sort_values(
        "absolute_standardised_coefficient",
        ascending=False,
    )
    .reset_index(drop=True)
)


# ======================================================================================
# 15. SAVE MODEL BUNDLE AND RESULTS
# ======================================================================================

ridge_artifact = {
    "model_name": "Ridge",
    "target": TARGET_COLUMN,
    "selected_alpha": selected_alpha,
    "alpha_selection_method": (
        "five_fold_grouped_cross_validation_"
        "on_model_training_engines"
    ),
    "selection_metric": (
        "mean_engine_rmse_on_training_oof_predictions"
    ),
    "prediction_lower_bound": PREDICTION_LOWER_BOUND,
    "feature_order": ridge_features,
    "scaler": ridge_scaler,
    "model": ridge_model,
    "fit_engine_count": int(
        model_training_targets[
            UNIT_COLUMN
        ].nunique()
    ),
    "fit_row_count": int(
        len(model_training_targets)
    ),
    "scikit_learn_version": sklearn.__version__,
}

joblib.dump(
    ridge_artifact,
    RIDGE_MODEL_PATH,
)

ridge_cv_results.to_csv(
    RIDGE_CV_PATH,
    index=False,
)

ridge_validation_output.to_csv(
    RIDGE_PREDICTIONS_PATH,
    index=False,
)

ridge_engine_metrics.to_csv(
    RIDGE_ENGINE_METRICS_PATH,
    index=False,
)

ridge_model_comparison.to_csv(
    RIDGE_COMPARISON_PATH,
    index=False,
)

ridge_coefficients.to_csv(
    RIDGE_COEFFICIENTS_PATH,
    index=False,
)


# ======================================================================================
# 16. RELOAD MODEL BUNDLE AND VERIFY REPRODUCIBILITY
# ======================================================================================

reloaded_ridge_artifact = joblib.load(
    RIDGE_MODEL_PATH
)

reloaded_scaler = reloaded_ridge_artifact[
    "scaler"
]

reloaded_model = reloaded_ridge_artifact[
    "model"
]

reloaded_validation_scaled = pd.DataFrame(
    reloaded_scaler.transform(
        validation_targets[ridge_features]
    ),
    columns=ridge_features,
    index=validation_targets.index,
)

reloaded_raw_predictions = (
    reloaded_model.predict(
        reloaded_validation_scaled
    )
)

reloaded_validation_predictions = np.clip(
    reloaded_raw_predictions,
    reloaded_ridge_artifact[
        "prediction_lower_bound"
    ],
    None,
)

reloaded_predictions_match = bool(
    np.allclose(
        reloaded_validation_predictions,
        ridge_validation_predictions,
        rtol=1e-12,
        atol=1e-12,
    )
)


# ======================================================================================
# 17. SAVE HUMAN- AND MACHINE-READABLE MODEL DEFINITION
# ======================================================================================

ridge_definition = {
    "phase": 4,
    "step": "2",
    "model": "Ridge Regression",
    "primary_target": TARGET_COLUMN,
    "sensitivity_target_used": False,
    "training_engine_count": int(
        len(training_engine_ids)
    ),
    "training_row_count": int(
        len(model_training_targets)
    ),
    "validation_engine_count": int(
        len(validation_engine_ids)
    ),
    "validation_row_count": int(
        len(validation_targets)
    ),
    "predictor_count": int(
        len(ridge_features)
    ),
    "predictor_order": ridge_features,
    "scaling_policy": (
        "Final model uses the Phase 3 scaler fitted "
        "only on the complete model-training partition."
    ),
    "cross_validation_scaling_policy": (
        "A new StandardScaler was fitted inside each "
        "training-only grouped cross-validation fold."
    ),
    "alpha_grid": [
        float(value)
        for value in ALPHA_GRID
    ],
    "selected_alpha": selected_alpha,
    "selected_alpha_is_grid_boundary": (
        selected_alpha_is_boundary
    ),
    "alpha_selection_population": (
        "model_training_engines_only"
    ),
    "alpha_selection_method": (
        "five_fold_group_k_fold"
    ),
    "alpha_selection_metric": (
        "mean_engine_rmse"
    ),
    "validation_used_for_hyperparameter_selection": False,
    "validation_used_for_evaluation": True,
    "calibration_values_used": False,
    "official_test_data_used": False,
    "test_rul_values_accessed": False,
    "prediction_lower_bound": (
        PREDICTION_LOWER_BOUND
    ),
    "negative_raw_validation_prediction_rows": (
        negative_raw_validation_prediction_rows
    ),
    "naive_baseline_reference_lifetime": (
        baseline_reference_lifetime
    ),
    "pooled_rmse_reduction_vs_naive": (
        pooled_rmse_reduction
    ),
    "pooled_rmse_reduction_percent_vs_naive": (
        pooled_rmse_reduction_percent
    ),
    "engine_balanced_rmse_reduction_vs_naive": (
        engine_balanced_rmse_reduction
    ),
    "engine_balanced_rmse_reduction_percent_vs_naive": (
        engine_balanced_rmse_reduction_percent
    ),
    "solver": "svd",
    "random_seed": RANDOM_SEED,
    "scikit_learn_version": sklearn.__version__,
    "python_version": platform.python_version(),
    "training_target_file_sha256": file_sha256(
        TRAIN_TARGET_PATH
    ),
    "validation_target_file_sha256": file_sha256(
        VALIDATION_TARGET_PATH
    ),
    "calibration_target_file_sha256": (
        calibration_hash_before
    ),
    "scaler_file_sha256": scaler_hash,
    "ridge_model_bundle_sha256": file_sha256(
        RIDGE_MODEL_PATH
    ),
}

with open(
    RIDGE_DEFINITION_PATH,
    "w",
    encoding="utf-8",
) as definition_file:
    json.dump(
        ridge_definition,
        definition_file,
        indent=2,
    )


# ======================================================================================
# 18. FINAL INTEGRITY AND VALIDATION CHECKS
# ======================================================================================

training_dataframe_hash_after = dataframe_sha256(
    model_training_targets
)

validation_dataframe_hash_after = dataframe_sha256(
    validation_targets
)

calibration_hash_after = file_sha256(
    CALIBRATION_TARGET_PATH
)

scaler_training_rows = int(
    np.asarray(
        ridge_scaler.n_samples_seen_
    ).reshape(-1)[0]
)

ridge_checks = {
    "model training contains 60 engines": (
        len(training_engine_ids) == 60
    ),
    "validation contains 20 engines": (
        len(validation_engine_ids) == 20
    ),
    "training and validation engines are disjoint": (
        training_engine_ids.isdisjoint(
            validation_engine_ids
        )
    ),
    "ridge contains eighteen predictors": (
        len(ridge_features) == 18
    ),
    "unit number is excluded from predictors": (
        UNIT_COLUMN not in ridge_features
    ),
    "primary target is excluded from predictors": (
        TARGET_COLUMN not in ridge_features
    ),
    "sensitivity target is excluded from predictors": (
        "rul_capped_130" not in ridge_features
    ),
    "scaler feature order is preserved": (
        list(ridge_scaler.feature_names_in_)
        == ridge_features
    ),
    "scaler was fitted on model training rows only": (
        scaler_training_rows
        == len(model_training_targets)
    ),
    "five grouped folds were created": (
        len(cv_splits) == NUMBER_OF_FOLDS
    ),
    "all cross validation folds are engine disjoint": (
        all(fold_group_separation_checks)
    ),
    "all training rows received finite oof predictions": (
        all_oof_predictions_complete
    ),
    "alpha grid results are complete": (
        len(ridge_cv_results)
        == len(ALPHA_GRID)
    ),
    "selected alpha belongs to predefined grid": (
        any(
            np.isclose(
                selected_alpha,
                grid_alpha,
            )
            for grid_alpha in ALPHA_GRID
        )
    ),
    "validation was not used to choose alpha": True,
    "final ridge fitted all model training rows": (
        ridge_model.n_features_in_
        == len(ridge_features)
    ),
    "validation row count is preserved": (
        len(ridge_validation_predictions)
        == len(validation_targets)
    ),
    "ridge predictions are finite": (
        np.isfinite(
            ridge_validation_predictions
        ).all()
    ),
    "ridge predictions are non negative": (
        (
            ridge_validation_predictions
            >= 0
        ).all()
    ),
    "ridge coefficients are finite": (
        np.isfinite(ridge_model.coef_).all()
    ),
    "ridge contains eighteen coefficients": (
        len(ridge_model.coef_)
        == len(ridge_features)
    ),
    "all ridge pooled metrics are finite": all(
        np.isfinite(value)
        for key, value in ridge_pooled_metrics.items()
        if key != "model"
    ),
    "ridge engine metrics contain 20 engines": (
        len(ridge_engine_metrics) == 20
    ),
    "all ridge engine metrics are finite": (
        ridge_engine_metrics[
            [
                "mae",
                "rmse",
                "bias_predicted_minus_actual",
            ]
        ]
        .apply(np.isfinite)
        .all()
        .all()
    ),
    "naive baseline was reproduced at 195 cycles": (
        baseline_reference_lifetime == 195
    ),
    "saved bundle reproduces validation predictions": (
        reloaded_predictions_match
    ),
    "training dataframe was not modified": (
        training_dataframe_hash_before
        == training_dataframe_hash_after
    ),
    "validation dataframe was not modified": (
        validation_dataframe_hash_before
        == validation_dataframe_hash_after
    ),
    "calibration file was not modified": (
        calibration_hash_before
        == calibration_hash_after
    ),
    "model bundle exists": (
        RIDGE_MODEL_PATH.exists()
    ),
    "all result files exist": all(
        path.exists()
        for path in [
            RIDGE_CV_PATH,
            RIDGE_PREDICTIONS_PATH,
            RIDGE_ENGINE_METRICS_PATH,
            RIDGE_COMPARISON_PATH,
            RIDGE_COEFFICIENTS_PATH,
            RIDGE_DEFINITION_PATH,
        ]
    ),
}

if not all(ridge_checks.values()):
    failed_checks = [
        check_name
        for check_name, passed
        in ridge_checks.items()
        if not passed
    ]

    raise AssertionError(
        "Ridge validation failed:\n"
        + "\n".join(failed_checks)
    )


# ======================================================================================
# 19. CONTROLLED OUTPUT
# ======================================================================================

print("\nRIDGE HYPERPARAMETER POLICY")
print("=" * 92)
print(f"Candidate alpha values: {list(ALPHA_GRID)}")
print("Selection population: 60 model-training engines only")
print("Selection method: five-fold engine-grouped cross-validation")
print("Primary selection metric: mean engine RMSE")
print(f"Selected alpha: {selected_alpha:g}")
print(
    "Selected alpha lies on search-grid boundary: "
    f"{'Yes' if selected_alpha_is_boundary else 'No'}"
)

print("\nTRAINING-ONLY GROUPED CROSS-VALIDATION RESULTS")
print("=" * 92)

display(
    ridge_cv_results.round(6)
)

print("\nNAIVE BASELINE VERSUS RIDGE — VALIDATION RESULTS")
print("=" * 92)

display(
    ridge_model_comparison.round(6)
)

print("\nRIDGE IMPROVEMENT OVER NAIVE BASELINE")
print("=" * 92)
print(
    f"Pooled RMSE reduction: "
    f"{pooled_rmse_reduction:.6f} cycles "
    f"({pooled_rmse_reduction_percent:.2f}%)"
)
print(
    f"Engine-balanced RMSE reduction: "
    f"{engine_balanced_rmse_reduction:.6f} cycles "
    f"({engine_balanced_rmse_reduction_percent:.2f}%)"
)
print(
    "Negative raw validation predictions clipped to zero: "
    f"{negative_raw_validation_prediction_rows}"
)

print("\nTEN LARGEST STANDARDISED RIDGE COEFFICIENTS")
print("=" * 92)

display(
    ridge_coefficients.head(10).round(6)
)

print("\nRIDGE VALIDATION")
print("=" * 92)

for check_name, passed in ridge_checks.items():
    status = "PASS" if passed else "FAIL"
    print(
        f"{status} — "
        f"{check_name.replace('_', ' ')}"
    )

print("\nOUTPUT FILES")
print("=" * 92)
print(f"Grouped CV results: {RIDGE_CV_PATH}")
print(f"Validation predictions: {RIDGE_PREDICTIONS_PATH}")
print(f"Engine metrics: {RIDGE_ENGINE_METRICS_PATH}")
print(f"Model comparison: {RIDGE_COMPARISON_PATH}")
print(f"Coefficients: {RIDGE_COEFFICIENTS_PATH}")
print(f"Model definition: {RIDGE_DEFINITION_PATH}")
print(f"Saved Ridge bundle: {RIDGE_MODEL_PATH}")
print(
    "Ridge bundle SHA-256: "
    f"{file_sha256(RIDGE_MODEL_PATH)}"
)

print("\nRESEARCH CONTROLS")
print("=" * 92)
print("Validation used for hyperparameter selection: No")
print("Validation used for final evaluation: Yes")
print("Calibration values used: No")
print("Official test data used: No")
print("Test RUL values accessed: No")
print("Sensitivity target used: No")
print("Random Forest trained: No")

print("\nFINAL CHECK")
print("=" * 92)
print("PASS — Ridge hyperparameters were selected using training engines only.")
print("PASS — Cross-validation folds were separated at engine level.")
print("PASS — Validation was used only for final Ridge evaluation.")
print("PASS — Calibration and official test data remained unused.")
print("PASS — The saved Ridge bundle reproduces validation predictions.")
print("AWAITING APPROVAL — Random Forest has not started.")

In [ ]:
# PHASE 4 — STEP 3: RANDOM FOREST

from pathlib import Path
import gc
import hashlib
import json

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, ParameterGrid


# ==============================================================================
# 1. PATHS
# ==============================================================================

ROOT = Path("/content/Predictive_Maintenance_Thesis")

TRAIN_PATH = ROOT / "data/interim/fd001_model_training_targets.csv"
VALID_PATH = ROOT / "data/interim/fd001_validation_targets.csv"
CAL_PATH = ROOT / "data/interim/fd001_calibration_targets.csv"

SCALER_PATH = (
    ROOT
    / "models/preprocessing/fd001_ridge_standard_scaler.joblib"
)

RIDGE_PATH = (
    ROOT
    / "models/regression/fd001_ridge_uncapped_bundle.joblib"
)

RESULTS = ROOT / "results"
LOGS = ROOT / "logs"
MODELS = ROOT / "models/regression"

for directory in [RESULTS, LOGS, MODELS]:
    directory.mkdir(parents=True, exist_ok=True)

POLICY_PATH = (
    LOGS
    / "phase4_model_comparison_policy.json"
)

CV_PATH = (
    RESULTS
    / "phase4_random_forest_training_grouped_cv_results.csv"
)

PRED_PATH = (
    RESULTS
    / "phase4_random_forest_validation_predictions.csv"
)

ENGINE_PATH = (
    RESULTS
    / "phase4_random_forest_validation_engine_metrics.csv"
)

COMPARISON_PATH = (
    RESULTS
    / "phase4_naive_ridge_random_forest_comparison.csv"
)

IMPORTANCE_PATH = (
    RESULTS
    / "phase4_random_forest_impurity_importance.csv"
)

DEFINITION_PATH = (
    LOGS
    / "phase4_random_forest_model_definition.json"
)

MODEL_PATH = (
    MODELS
    / "fd001_random_forest_uncapped_bundle.joblib"
)


# ==============================================================================
# 2. HELPER FUNCTIONS
# ==============================================================================

def file_hash(path):
    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def frame_hash(frame):
    values = pd.util.hash_pandas_object(
        frame,
        index=True,
    ).values.tobytes()

    return hashlib.sha256(values).hexdigest()


def phm_penalty(actual, predicted):
    error = predicted - actual

    return np.where(
        error >= 0,
        np.expm1(error / 10.0),
        np.expm1(-error / 13.0),
    )


def engine_metrics(
    units,
    actual,
    predicted,
    model_name,
):
    source = pd.DataFrame(
        {
            "unit_number": units,
            "actual": actual,
            "predicted": predicted,
        }
    )

    rows = []

    for unit, group in source.groupby(
        "unit_number",
        sort=True,
    ):
        y_true = group["actual"].to_numpy()
        y_pred = group["predicted"].to_numpy()

        rows.append(
            {
                "model": model_name,
                "unit_number": int(unit),
                "row_count": int(len(group)),
                "mae": float(
                    mean_absolute_error(
                        y_true,
                        y_pred,
                    )
                ),
                "rmse": float(
                    root_mean_squared_error(
                        y_true,
                        y_pred,
                    )
                ),
                "bias_predicted_minus_actual": float(
                    np.mean(y_pred - y_true)
                ),
            }
        )

    return pd.DataFrame(rows)


def model_metrics(
    model_name,
    units,
    actual,
    predicted,
):
    error = predicted - actual

    by_engine = engine_metrics(
        units,
        actual,
        predicted,
        model_name,
    )

    record = {
        "model": model_name,
        "pooled_mae": float(
            mean_absolute_error(
                actual,
                predicted,
            )
        ),
        "pooled_rmse": float(
            root_mean_squared_error(
                actual,
                predicted,
            )
        ),
        "pooled_r_squared": float(
            r2_score(
                actual,
                predicted,
            )
        ),
        "bias_predicted_minus_actual": float(
            np.mean(error)
        ),
        "rul_overestimation_rate_percent": float(
            np.mean(error > 0) * 100
        ),
        "rul_underestimation_rate_percent": float(
            np.mean(error < 0) * 100
        ),
        "exact_prediction_rate_percent": float(
            np.mean(
                np.isclose(
                    error,
                    0.0,
                    atol=1e-12,
                )
            ) * 100
        ),
        "mean_row_level_phm_penalty": float(
            np.mean(
                phm_penalty(
                    actual,
                    predicted,
                )
            )
        ),
        "mean_engine_mae": float(
            by_engine["mae"].mean()
        ),
        "mean_engine_rmse": float(
            by_engine["rmse"].mean()
        ),
        "median_engine_rmse": float(
            by_engine["rmse"].median()
        ),
        "engine_rmse_standard_deviation": float(
            by_engine["rmse"].std(ddof=1)
        ),
        "engine_rmse_first_quartile": float(
            by_engine["rmse"].quantile(0.25)
        ),
        "engine_rmse_third_quartile": float(
            by_engine["rmse"].quantile(0.75)
        ),
    }

    return record, by_engine


def improvement_percent(
    reference,
    candidate,
):
    return float(
        (reference - candidate)
        / reference
        * 100
    )


# ==============================================================================
# 3. VERIFY REQUIRED FILES
# ==============================================================================

required_paths = [
    TRAIN_PATH,
    VALID_PATH,
    CAL_PATH,
    SCALER_PATH,
    RIDGE_PATH,
]

missing_paths = []

for required_path in required_paths:
    if not required_path.exists():
        missing_paths.append(
            str(required_path)
        )

if missing_paths:
    raise FileNotFoundError(
        "Required files are missing:\n"
        + "\n".join(missing_paths)
    )


# ==============================================================================
# 4. FREEZE MODEL-COMPARISON POLICY
# ==============================================================================

comparison_policy = {
    "phase": 4,
    "status": (
        "frozen_before_random_forest_validation"
    ),
    "primary_metric": "mean_engine_rmse",
    "practical_equivalence_threshold_percent": 1.0,
    "first_tie_breaker": (
        "mean_row_level_phm_penalty"
    ),
    "second_tie_breaker": "pooled_rmse",
    "third_tie_breaker": (
        "lower_model_complexity"
    ),
    "phm_penalty_status": (
        "row_level_diagnostic_not_official_score"
    ),
    "validation_role": "evaluation_only",
    "calibration_role": (
        "sealed_for_conformal_phase"
    ),
    "official_test_role": (
        "sealed_for_final_evaluation"
    ),
    "final_model_selection_deferred": True,
}

with open(
    POLICY_PATH,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        comparison_policy,
        handle,
        indent=2,
    )

policy_hash = file_hash(POLICY_PATH)


# ==============================================================================
# 5. LOAD TRAINING DATA ONLY
# ==============================================================================

train = pd.read_csv(TRAIN_PATH)

train_hash_before = frame_hash(train)
calibration_hash_before = file_hash(CAL_PATH)

scaler = joblib.load(SCALER_PATH)

if not hasattr(
    scaler,
    "feature_names_in_",
):
    raise AssertionError(
        "Locked Phase 3 feature order was not found."
    )

FEATURES = list(
    scaler.feature_names_in_
)

UNIT = "unit_number"
CYCLE = "time_in_cycles"
TARGET = "rul_uncapped"

X_train = train[FEATURES].copy()

y_train = train[
    TARGET
].to_numpy(dtype=float)

groups = train[
    UNIT
].to_numpy()

train_engines = set(
    train[UNIT].unique()
)


# ==============================================================================
# 6. PREDECLARE RANDOM FOREST SEARCH
# ==============================================================================

SEED = 42
FOLDS = 5
TUNING_TREES = 150
FINAL_TREES = 500
LOWER_BOUND = 0.0

parameter_grid = {
    "max_depth": [
        8,
        14,
        None,
    ],
    "min_samples_leaf": [
        1,
        5,
        10,
    ],
    "max_features": [
        0.5,
        1.0,
    ],
}

candidates = list(
    ParameterGrid(parameter_grid)
)

splitter = GroupKFold(
    n_splits=FOLDS,
    shuffle=True,
    random_state=SEED,
)

splits = list(
    splitter.split(
        X_train,
        y_train,
        groups=groups,
    )
)

folds_are_disjoint = True

for fit_index, holdout_index in splits:
    fit_engines = set(
        groups[fit_index]
    )

    holdout_engines = set(
        groups[holdout_index]
    )

    folds_are_disjoint = (
        folds_are_disjoint
        and fit_engines.isdisjoint(
            holdout_engines
        )
    )


# ==============================================================================
# 7. TRAINING-ONLY RANDOM FOREST SEARCH
# ==============================================================================

print("\nTRAINING RANDOM FOREST CANDIDATES")
print("=" * 90)

print(
    f"Candidates: {len(candidates)} | "
    f"Folds: {FOLDS} | "
    f"Trees per fold: {TUNING_TREES}"
)

cv_rows = []
all_oof_complete = True

for candidate_number, params in enumerate(
    candidates,
    start=1,
):
    oof = np.full(
        len(train),
        np.nan,
        dtype=float,
    )

    for fit_index, holdout_index in splits:
        candidate = RandomForestRegressor(
            n_estimators=TUNING_TREES,
            criterion="squared_error",
            max_depth=params[
                "max_depth"
            ],
            min_samples_leaf=params[
                "min_samples_leaf"
            ],
            max_features=params[
                "max_features"
            ],
            bootstrap=True,
            random_state=SEED,
            n_jobs=-1,
        )

        candidate.fit(
            X_train.iloc[fit_index],
            y_train[fit_index],
        )

        raw_prediction = candidate.predict(
            X_train.iloc[holdout_index]
        )

        oof[holdout_index] = np.clip(
            raw_prediction,
            LOWER_BOUND,
            None,
        )

        del candidate

    current_oof_complete = bool(
        np.isfinite(oof).all()
    )

    all_oof_complete = bool(
        all_oof_complete
        and current_oof_complete
    )

    candidate_engine = engine_metrics(
        groups,
        y_train,
        oof,
        f"rf_candidate_{candidate_number}",
    )

    cv_rows.append(
        {
            "candidate_number": int(
                candidate_number
            ),
            "max_depth": params[
                "max_depth"
            ],
            "min_samples_leaf": int(
                params["min_samples_leaf"]
            ),
            "max_features": float(
                params["max_features"]
            ),
            "tuning_trees": int(
                TUNING_TREES
            ),
            "cv_pooled_mae": float(
                mean_absolute_error(
                    y_train,
                    oof,
                )
            ),
            "cv_pooled_rmse": float(
                root_mean_squared_error(
                    y_train,
                    oof,
                )
            ),
            "cv_pooled_r_squared": float(
                r2_score(
                    y_train,
                    oof,
                )
            ),
            "cv_mean_engine_mae": float(
                candidate_engine[
                    "mae"
                ].mean()
            ),
            "cv_mean_engine_rmse": float(
                candidate_engine[
                    "rmse"
                ].mean()
            ),
            "cv_median_engine_rmse": float(
                candidate_engine[
                    "rmse"
                ].median()
            ),
        }
    )

    print(
        f"Completed "
        f"{candidate_number:02d}/"
        f"{len(candidates)} — "
        f"depth={params['max_depth']}, "
        f"leaf={params['min_samples_leaf']}, "
        f"features={params['max_features']}"
    )

    gc.collect()


# ==============================================================================
# 8. SELECT TRAINING-ONLY HYPERPARAMETERS
# ==============================================================================

cv_results = pd.DataFrame(cv_rows)

cv_results[
    "_depth_order"
] = cv_results[
    "max_depth"
].apply(
    lambda value: (
        1_000_000
        if pd.isna(value)
        else int(value)
    )
)

ranking = cv_results.sort_values(
    [
        "cv_mean_engine_rmse",
        "_depth_order",
        "min_samples_leaf",
        "max_features",
    ],
    ascending=[
        True,
        True,
        False,
        True,
    ],
).reset_index(drop=True)

best = ranking.iloc[0]

if pd.isna(best["max_depth"]):
    selected_depth = None
else:
    selected_depth = int(
        best["max_depth"]
    )

selected_leaf = int(
    best["min_samples_leaf"]
)

selected_features = float(
    best["max_features"]
)

selected_candidate = int(
    best["candidate_number"]
)

cv_results_to_save = (
    cv_results
    .drop(columns=["_depth_order"])
    .sort_values("candidate_number")
    .reset_index(drop=True)
)


# ==============================================================================
# 9. FIT FINAL RANDOM FOREST
# ==============================================================================

rf_model = RandomForestRegressor(
    n_estimators=FINAL_TREES,
    criterion="squared_error",
    max_depth=selected_depth,
    min_samples_leaf=selected_leaf,
    max_features=selected_features,
    bootstrap=True,
    random_state=SEED,
    n_jobs=-1,
)

rf_model.fit(
    X_train,
    y_train,
)


# ==============================================================================
# 10. LOAD VALIDATION AFTER RF IS FROZEN
# ==============================================================================

validation = pd.read_csv(
    VALID_PATH
)

validation_hash_before = frame_hash(
    validation
)

validation_engines = set(
    validation[UNIT].unique()
)

X_validation = validation[
    FEATURES
].copy()

y_validation = validation[
    TARGET
].to_numpy(dtype=float)

validation_units = validation[
    UNIT
].to_numpy()

rf_raw = rf_model.predict(
    X_validation
)

rf_prediction = np.clip(
    rf_raw,
    LOWER_BOUND,
    None,
)

negative_rf_rows = int(
    np.sum(rf_raw < 0)
)


# ==============================================================================
# 11. REPRODUCE NAIVE AND RIDGE COMPARATORS
# ==============================================================================

median_lifetime = int(
    round(
        train.groupby(UNIT)[
            CYCLE
        ].max().median()
    )
)

naive_prediction = np.maximum(
    median_lifetime
    - validation[CYCLE].to_numpy(),
    0,
).astype(float)

ridge_bundle = joblib.load(
    RIDGE_PATH
)

ridge_features = list(
    ridge_bundle["feature_order"]
)

ridge_scaled = pd.DataFrame(
    ridge_bundle[
        "scaler"
    ].transform(
        validation[ridge_features]
    ),
    columns=ridge_features,
    index=validation.index,
)

ridge_raw = ridge_bundle[
    "model"
].predict(
    ridge_scaled
)

ridge_prediction = np.clip(
    ridge_raw,
    ridge_bundle[
        "prediction_lower_bound"
    ],
    None,
)


# ==============================================================================
# 12. COMPARABLE VALIDATION METRICS
# ==============================================================================

naive_record, naive_engine = model_metrics(
    "naive_age_only",
    validation_units,
    y_validation,
    naive_prediction,
)

ridge_record, ridge_engine = model_metrics(
    "ridge_uncapped",
    validation_units,
    y_validation,
    ridge_prediction,
)

rf_record, rf_engine = model_metrics(
    "random_forest_uncapped",
    validation_units,
    y_validation,
    rf_prediction,
)

comparison = pd.DataFrame(
    [
        naive_record,
        ridge_record,
        rf_record,
    ]
)

comparison[
    "primary_metric_rank"
] = (
    comparison[
        "mean_engine_rmse"
    ]
    .rank(
        method="min",
        ascending=True,
    )
    .astype(int)
)

comparison = (
    comparison
    .sort_values(
        "primary_metric_rank"
    )
    .reset_index(drop=True)
)

current_leader = str(
    comparison.iloc[0]["model"]
)

rf_pooled_vs_naive = improvement_percent(
    naive_record["pooled_rmse"],
    rf_record["pooled_rmse"],
)

rf_engine_vs_naive = improvement_percent(
    naive_record["mean_engine_rmse"],
    rf_record["mean_engine_rmse"],
)

rf_pooled_vs_ridge = improvement_percent(
    ridge_record["pooled_rmse"],
    rf_record["pooled_rmse"],
)

rf_engine_vs_ridge = improvement_percent(
    ridge_record["mean_engine_rmse"],
    rf_record["mean_engine_rmse"],
)


# ==============================================================================
# 13. BUILD RESULT TABLES
# ==============================================================================

prediction_output = pd.DataFrame(
    {
        "unit_number": validation[
            UNIT
        ].to_numpy(),
        "time_in_cycles": validation[
            CYCLE
        ].to_numpy(),
        "actual_rul_uncapped": (
            y_validation
        ),
        "random_forest_raw_prediction": (
            rf_raw
        ),
        "random_forest_prediction": (
            rf_prediction
        ),
        "prediction_was_lower_clipped": (
            rf_raw < 0
        ),
        "error_predicted_minus_actual": (
            rf_prediction
            - y_validation
        ),
        "absolute_error": np.abs(
            rf_prediction
            - y_validation
        ),
    }
)

importance = pd.DataFrame(
    {
        "feature": FEATURES,
        "impurity_importance": (
            rf_model.feature_importances_
        ),
    }
)

importance = (
    importance
    .sort_values(
        "impurity_importance",
        ascending=False,
    )
    .reset_index(drop=True)
)

importance[
    "importance_rank"
] = np.arange(
    1,
    len(importance) + 1,
)


# ==============================================================================
# 14. SAVE MODEL AND RESULT FILES
# ==============================================================================

rf_bundle = {
    "model_name": (
        "Random Forest Regressor"
    ),
    "target": TARGET,
    "feature_order": FEATURES,
    "input_scaling": "none",
    "selection_metric": (
        "mean_engine_rmse"
    ),
    "selected_candidate_number": (
        selected_candidate
    ),
    "selected_hyperparameters": {
        "n_estimators": FINAL_TREES,
        "max_depth": selected_depth,
        "min_samples_leaf": (
            selected_leaf
        ),
        "max_features": (
            selected_features
        ),
        "criterion": (
            "squared_error"
        ),
        "bootstrap": True,
        "random_state": SEED,
    },
    "prediction_lower_bound": (
        LOWER_BOUND
    ),
    "model": rf_model,
    "fit_engine_count": int(
        len(train_engines)
    ),
    "fit_row_count": int(
        len(train)
    ),
    "scikit_learn_version": (
        sklearn.__version__
    ),
}

joblib.dump(
    rf_bundle,
    MODEL_PATH,
)

cv_results_to_save.to_csv(
    CV_PATH,
    index=False,
)

prediction_output.to_csv(
    PRED_PATH,
    index=False,
)

rf_engine.to_csv(
    ENGINE_PATH,
    index=False,
)

comparison.to_csv(
    COMPARISON_PATH,
    index=False,
)

importance.to_csv(
    IMPORTANCE_PATH,
    index=False,
)


# ==============================================================================
# 15. RELOAD AND VERIFY SAVED MODEL
# ==============================================================================

reloaded = joblib.load(
    MODEL_PATH
)

reloaded_prediction = np.clip(
    reloaded["model"].predict(
        validation[
            reloaded["feature_order"]
        ]
    ),
    reloaded[
        "prediction_lower_bound"
    ],
    None,
)

reload_matches = bool(
    np.allclose(
        reloaded_prediction,
        rf_prediction,
        rtol=1e-12,
        atol=1e-12,
    )
)


# ==============================================================================
# 16. SAVE FORMAL MODEL DEFINITION
# ==============================================================================

definition = {
    "phase": 4,
    "step": "3",
    "model": (
        "Random Forest Regressor"
    ),
    "primary_target": TARGET,
    "predictor_count": int(
        len(FEATURES)
    ),
    "predictor_order": FEATURES,
    "tree_input_scaled": False,
    "training_engine_count": int(
        len(train_engines)
    ),
    "training_row_count": int(
        len(train)
    ),
    "validation_engine_count": int(
        len(validation_engines)
    ),
    "validation_row_count": int(
        len(validation)
    ),
    "candidate_count": int(
        len(candidates)
    ),
    "cv_fold_count": int(
        FOLDS
    ),
    "tuning_trees": int(
        TUNING_TREES
    ),
    "final_trees": int(
        FINAL_TREES
    ),
    "selected_candidate": (
        selected_candidate
    ),
    "selected_max_depth": (
        selected_depth
    ),
    "selected_min_samples_leaf": (
        selected_leaf
    ),
    "selected_max_features": (
        selected_features
    ),
    "validation_used_for_tuning": False,
    "validation_used_for_evaluation": True,
    "calibration_values_used": False,
    "official_test_data_used": False,
    "test_rul_values_accessed": False,
    "negative_raw_validation_predictions": (
        negative_rf_rows
    ),
    "current_primary_metric_leader": (
        current_leader
    ),
    "final_model_selected": False,
    "feature_importance_status": (
        "descriptive_not_causal"
    ),
    "comparison_policy_sha256": (
        policy_hash
    ),
    "random_forest_bundle_sha256": (
        file_hash(MODEL_PATH)
    ),
    "scikit_learn_version": (
        sklearn.__version__
    ),
}

with open(
    DEFINITION_PATH,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        definition,
        handle,
        indent=2,
    )


# ==============================================================================
# 17. VALIDATION CHECKS
# ==============================================================================

checks = {
    "comparison policy was frozen": (
        POLICY_PATH.exists()
    ),
    "training contains 60 engines": (
        len(train_engines) == 60
    ),
    "validation contains 20 engines": (
        len(validation_engines) == 20
    ),
    "training and validation are engine disjoint": (
        train_engines.isdisjoint(
            validation_engines
        )
    ),
    "random forest contains 18 predictors": (
        len(FEATURES) == 18
    ),
    "unit number is excluded": (
        UNIT not in FEATURES
    ),
    "primary target is excluded": (
        TARGET not in FEATURES
    ),
    "sensitivity target is excluded": (
        "rul_capped_130"
        not in FEATURES
    ),
    "tree predictors were not scaled": True,
    "eighteen candidates were evaluated": (
        len(cv_results_to_save) == 18
    ),
    "five grouped folds were used": (
        len(splits) == 5
    ),
    "all cv folds are engine disjoint": (
        folds_are_disjoint
    ),
    "all oof predictions were complete": (
        all_oof_complete
    ),
    "selected candidate came from fixed grid": (
        selected_candidate
        in set(
            cv_results_to_save[
                "candidate_number"
            ]
        )
    ),
    "validation was not used for tuning": True,
    "feature order is preserved": (
        list(
            rf_model.feature_names_in_
        )
        == FEATURES
    ),
    "validation row count is preserved": (
        len(rf_prediction)
        == len(validation)
    ),
    "rf predictions are finite": (
        np.isfinite(
            rf_prediction
        ).all()
    ),
    "rf predictions are non negative": (
        (rf_prediction >= 0).all()
    ),
    "rf engine metrics contain 20 engines": (
        len(rf_engine) == 20
    ),
    "rf engine metrics are finite": (
        rf_engine[
            [
                "mae",
                "rmse",
                "bias_predicted_minus_actual",
            ]
        ]
        .apply(np.isfinite)
        .all()
        .all()
    ),
    "comparison contains three models": (
        len(comparison) == 3
    ),
    "feature importances sum to one": (
        np.isclose(
            importance[
                "impurity_importance"
            ].sum(),
            1.0,
        )
    ),
    "saved bundle reproduces predictions": (
        reload_matches
    ),
    "training dataframe was not modified": (
        train_hash_before
        == frame_hash(train)
    ),
    "validation dataframe was not modified": (
        validation_hash_before
        == frame_hash(validation)
    ),
    "calibration file was not modified": (
        calibration_hash_before
        == file_hash(CAL_PATH)
    ),
    "all output files exist": all(
        output_path.exists()
        for output_path in [
            CV_PATH,
            PRED_PATH,
            ENGINE_PATH,
            COMPARISON_PATH,
            IMPORTANCE_PATH,
            DEFINITION_PATH,
            MODEL_PATH,
        ]
    ),
}

failed_checks = []

for check_name, passed in checks.items():
    if not passed:
        failed_checks.append(
            check_name
        )

if failed_checks:
    raise AssertionError(
        "Random Forest validation failed:\n"
        + "\n".join(failed_checks)
    )


# ==============================================================================
# 18. CONTROLLED OUTPUT
# ==============================================================================

print("\nFROZEN MODEL-COMPARISON POLICY")
print("=" * 90)
print("Primary metric: mean engine RMSE")
print("Practical equivalence threshold: 1%")
print("First tie-breaker: row-level PHM penalty")
print("Second tie-breaker: pooled RMSE")
print("Third tie-breaker: lower model complexity")
print("Final model selection: Deferred")

print("\nRANDOM FOREST SEARCH SUMMARY")
print("=" * 90)
print(f"Candidates evaluated: {len(candidates)}")
print(f"Grouped folds: {FOLDS}")
print(f"Selected candidate: {selected_candidate}")
print(f"Selected max depth: {selected_depth}")
print(f"Selected minimum leaf size: {selected_leaf}")
print(
    "Selected maximum feature proportion: "
    f"{selected_features}"
)
print(f"Final number of trees: {FINAL_TREES}")

print("\nTRAINING-ONLY RANDOM FOREST CV RESULTS")
print("=" * 90)

columns_to_show = [
    "candidate_number",
    "max_depth",
    "min_samples_leaf",
    "max_features",
    "cv_pooled_mae",
    "cv_pooled_rmse",
    "cv_pooled_r_squared",
    "cv_mean_engine_mae",
    "cv_mean_engine_rmse",
    "cv_median_engine_rmse",
]

display(
    ranking[
        columns_to_show
    ].round(6)
)

print("\nVALIDATION MODEL COMPARISON")
print("=" * 90)

display(
    comparison.round(6)
)

print("\nRANDOM FOREST IMPROVEMENT")
print("=" * 90)

print(
    "Pooled RMSE improvement versus naive: "
    f"{rf_pooled_vs_naive:.2f}%"
)

print(
    "Engine-balanced RMSE improvement versus naive: "
    f"{rf_engine_vs_naive:.2f}%"
)

print(
    "Pooled RMSE improvement versus Ridge: "
    f"{rf_pooled_vs_ridge:.2f}%"
)

print(
    "Engine-balanced RMSE improvement versus Ridge: "
    f"{rf_engine_vs_ridge:.2f}%"
)

print(
    f"Current primary-metric leader: "
    f"{current_leader}"
)

print("Final model selected: No")

print("\nTOP TEN IMPURITY-BASED FEATURE IMPORTANCES")
print("=" * 90)

display(
    importance.head(10).round(6)
)

print(
    "Important: these importances are "
    "descriptive, not causal."
)

print("\nRANDOM FOREST VALIDATION")
print("=" * 90)

for check_name, passed in checks.items():
    status = (
        "PASS"
        if passed
        else "FAIL"
    )

    print(
        f"{status} — "
        f"{check_name}"
    )

print("\nOUTPUT FILES")
print("=" * 90)
print(f"Comparison policy: {POLICY_PATH}")
print(f"RF CV results: {CV_PATH}")
print(f"RF validation predictions: {PRED_PATH}")
print(f"RF engine metrics: {ENGINE_PATH}")
print(f"Three-model comparison: {COMPARISON_PATH}")
print(f"RF feature importance: {IMPORTANCE_PATH}")
print(f"RF definition: {DEFINITION_PATH}")
print(f"RF model bundle: {MODEL_PATH}")
print(
    "RF bundle SHA-256: "
    f"{file_hash(MODEL_PATH)}"
)

print("\nRESEARCH CONTROLS")
print("=" * 90)
print("Validation used for hyperparameter selection: No")
print("Validation used for final evaluation: Yes")
print("Calibration values used: No")
print("Official test data used: No")
print("Test RUL values accessed: No")
print("Sensitivity target used: No")
print("XGBoost trained: No")

print("\nFINAL CHECK")
print("=" * 90)
print("PASS — The model-comparison policy was frozen.")
print("PASS — Random Forest tuning used model-training engines only.")
print("PASS — Validation was used only after hyperparameters were frozen.")
print("PASS — Calibration and official test data remained unused.")
print("PASS — The saved Random Forest bundle reproduces predictions.")
print("AWAITING APPROVAL — XGBoost has not started.")

In [ ]:
# ==============================================================================
# PHASE 4 — STEP 4
# XGBOOST WITH TRAINING-ONLY ENGINE-GROUPED HYPERPARAMETER SELECTION
# ==============================================================================

from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import json
import zipfile

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score,
)
from sklearn.model_selection import GroupKFold, ParameterGrid

try:
    import xgboost as xgb
except ImportError as error:
    raise ImportError(
        "XGBoost is not installed. Run: !pip install -q xgboost"
    ) from error


# ==============================================================================
# 1. PATHS AND FROZEN SETTINGS
# ==============================================================================

ROOT = Path("/content/Predictive_Maintenance_Thesis")

TRAIN_PATH = (
    ROOT
    / "data/interim/fd001_model_training_targets.csv"
)

VALID_PATH = (
    ROOT
    / "data/interim/fd001_validation_targets.csv"
)

CAL_PATH = (
    ROOT
    / "data/interim/fd001_calibration_targets.csv"
)

SCALER_PATH = (
    ROOT
    / "models/preprocessing/fd001_ridge_standard_scaler.joblib"
)

RIDGE_PATH = (
    ROOT
    / "models/regression/fd001_ridge_uncapped_bundle.joblib"
)

RF_PATH = (
    ROOT
    / "models/regression/fd001_random_forest_uncapped_bundle.joblib"
)

POLICY_PATH = (
    ROOT
    / "logs/phase4_model_comparison_policy.json"
)

PRIOR_COMPARISON_PATH = (
    ROOT
    / "results/phase4_naive_ridge_random_forest_comparison.csv"
)

RESULTS = ROOT / "results"
LOGS = ROOT / "logs"
MODELS = ROOT / "models/regression"

for directory in [RESULTS, LOGS, MODELS]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


CV_PATH = (
    RESULTS
    / "phase4_xgboost_training_grouped_cv_results.csv"
)

PRED_PATH = (
    RESULTS
    / "phase4_xgboost_validation_predictions.csv"
)

ENGINE_PATH = (
    RESULTS
    / "phase4_xgboost_validation_engine_metrics.csv"
)

COMPARISON_PATH = (
    RESULTS
    / "phase4_naive_ridge_random_forest_xgboost_comparison.csv"
)

IMPORTANCE_PATH = (
    RESULTS
    / "phase4_xgboost_gain_importance.csv"
)

DEFINITION_PATH = (
    LOGS
    / "phase4_xgboost_model_definition.json"
)

MODEL_PATH = (
    MODELS
    / "fd001_xgboost_uncapped_bundle.joblib"
)

MANIFEST_PATH = (
    LOGS
    / "phase4_step4_xgboost_evidence_manifest.csv"
)

ZIP_PATH = (
    ROOT
    / "Phase4_Step4_XGBoost_Evidence.zip"
)


SEED = 42
FOLDS = 5
N_ESTIMATORS = 400
LOWER_BOUND = 0.0

UNIT = "unit_number"
CYCLE = "time_in_cycles"
TARGET = "rul_uncapped"
SENSITIVITY_TARGET = "rul_capped_130"


# This grid is frozen before validation is loaded.
PARAMETER_GRID = {
    "max_depth": [
        3,
        5,
        7,
    ],
    "learning_rate": [
        0.03,
        0.07,
        0.10,
    ],
    "min_child_weight": [
        5,
        20,
    ],
}


FIXED_PARAMETERS = {
    "n_estimators": N_ESTIMATORS,
    "objective": "reg:squarederror",
    "tree_method": "hist",
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 5.0,
    "importance_type": "gain",
    "random_state": SEED,
    "n_jobs": -1,
    "verbosity": 0,
}


# ==============================================================================
# 2. HELPER FUNCTIONS
# ==============================================================================

def file_hash(path):
    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def frame_hash(frame):
    values = pd.util.hash_pandas_object(
        frame,
        index=True,
    ).values.tobytes()

    return hashlib.sha256(
        values
    ).hexdigest()


def phm_penalty(
    actual,
    predicted,
):
    actual = np.asarray(
        actual,
        dtype=float,
    )

    predicted = np.asarray(
        predicted,
        dtype=float,
    )

    error = predicted - actual

    return np.where(
        error >= 0,
        np.expm1(error / 10.0),
        np.expm1(-error / 13.0),
    )


def engine_metrics(
    units,
    actual,
    predicted,
    model_name,
):
    source = pd.DataFrame(
        {
            "unit_number": np.asarray(units),
            "actual": np.asarray(
                actual,
                dtype=float,
            ),
            "predicted": np.asarray(
                predicted,
                dtype=float,
            ),
        }
    )

    rows = []

    for unit, group in source.groupby(
        "unit_number",
        sort=True,
    ):
        y_true = group[
            "actual"
        ].to_numpy()

        y_pred = group[
            "predicted"
        ].to_numpy()

        rows.append(
            {
                "model": model_name,
                "unit_number": int(unit),
                "row_count": int(len(group)),
                "mae": float(
                    mean_absolute_error(
                        y_true,
                        y_pred,
                    )
                ),
                "rmse": float(
                    root_mean_squared_error(
                        y_true,
                        y_pred,
                    )
                ),
                "bias_predicted_minus_actual": float(
                    np.mean(
                        y_pred - y_true
                    )
                ),
            }
        )

    return pd.DataFrame(rows)


def model_metrics(
    model_name,
    units,
    actual,
    predicted,
):
    actual = np.asarray(
        actual,
        dtype=float,
    )

    predicted = np.asarray(
        predicted,
        dtype=float,
    )

    error = predicted - actual

    by_engine = engine_metrics(
        units,
        actual,
        predicted,
        model_name,
    )

    record = {
        "model": model_name,

        "pooled_mae": float(
            mean_absolute_error(
                actual,
                predicted,
            )
        ),

        "pooled_rmse": float(
            root_mean_squared_error(
                actual,
                predicted,
            )
        ),

        "pooled_r_squared": float(
            r2_score(
                actual,
                predicted,
            )
        ),

        "bias_predicted_minus_actual": float(
            np.mean(error)
        ),

        "rul_overestimation_rate_percent": float(
            np.mean(error > 0)
            * 100
        ),

        "rul_underestimation_rate_percent": float(
            np.mean(error < 0)
            * 100
        ),

        "exact_prediction_rate_percent": float(
            np.mean(
                np.isclose(
                    error,
                    0.0,
                    atol=1e-12,
                )
            )
            * 100
        ),

        "mean_row_level_phm_penalty": float(
            np.mean(
                phm_penalty(
                    actual,
                    predicted,
                )
            )
        ),

        "mean_engine_mae": float(
            by_engine[
                "mae"
            ].mean()
        ),

        "mean_engine_rmse": float(
            by_engine[
                "rmse"
            ].mean()
        ),

        "median_engine_rmse": float(
            by_engine[
                "rmse"
            ].median()
        ),

        "engine_rmse_standard_deviation": float(
            by_engine[
                "rmse"
            ].std(ddof=1)
        ),

        "engine_rmse_first_quartile": float(
            by_engine[
                "rmse"
            ].quantile(0.25)
        ),

        "engine_rmse_third_quartile": float(
            by_engine[
                "rmse"
            ].quantile(0.75)
        ),
    }

    return record, by_engine


def improvement_percent(
    reference,
    candidate,
):
    return float(
        (reference - candidate)
        / reference
        * 100
    )


def build_xgboost(parameters):
    return xgb.XGBRegressor(
        **FIXED_PARAMETERS,
        **parameters,
    )


# ==============================================================================
# 3. VERIFY THE APPROVED FOUNDATION
# ==============================================================================

required_paths = [
    TRAIN_PATH,
    VALID_PATH,
    CAL_PATH,
    SCALER_PATH,
    RIDGE_PATH,
    RF_PATH,
    POLICY_PATH,
    PRIOR_COMPARISON_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Required approved artifacts are missing:\n"
        + "\n".join(missing_paths)
    )


expected_hashes = {
    SCALER_PATH:
        "453719672e9627ab47d7fbb8b80398d48768d70856643d3980e657fec87f293a",

    RIDGE_PATH:
        "43104a945295ee3cb9ae50de51db0c643455feb08d9a6e9b01fb227d1b83b3ea",

    RF_PATH:
        "99620a1a161f62a716d4cc058497159146197e7d91bba713581c822af808e5d5",
}


hash_status = {
    path.name: (
        file_hash(path)
        == expected_hash
    )
    for path, expected_hash
    in expected_hashes.items()
}


if not all(
    hash_status.values()
):
    failed_hashes = [
        name
        for name, passed
        in hash_status.items()
        if not passed
    ]

    raise AssertionError(
        "Approved artifact hash mismatch:\n"
        + "\n".join(failed_hashes)
    )


with open(
    POLICY_PATH,
    encoding="utf-8",
) as handle:
    comparison_policy = json.load(handle)


if (
    comparison_policy.get(
        "primary_metric"
    )
    != "mean_engine_rmse"
):
    raise AssertionError(
        "The frozen primary metric is not mean_engine_rmse."
    )


print("APPROVED FOUNDATION CHECK")
print("=" * 90)

for artifact_name, passed in hash_status.items():
    print(
        f"{'PASS' if passed else 'FAIL'} — "
        f"{artifact_name}"
    )

print(
    "PASS — Frozen primary metric: "
    "mean_engine_rmse"
)


# ==============================================================================
# 4. LOAD MODEL-TRAINING DATA ONLY
# ==============================================================================

train = pd.read_csv(
    TRAIN_PATH
)

train_hash_before = frame_hash(
    train
)

calibration_hash_before = file_hash(
    CAL_PATH
)

rf_bundle = joblib.load(
    RF_PATH
)

ridge_bundle = joblib.load(
    RIDGE_PATH
)

scaler = joblib.load(
    SCALER_PATH
)


FEATURES = list(
    rf_bundle[
        "feature_order"
    ]
)


if (
    FEATURES
    != list(
        ridge_bundle[
            "feature_order"
        ]
    )
):
    raise AssertionError(
        "Ridge and Random Forest feature orders differ."
    )


if (
    FEATURES
    != list(
        scaler.feature_names_in_
    )
):
    raise AssertionError(
        "Model and preprocessing feature orders differ."
    )


X_train = train[
    FEATURES
].copy()

y_train = train[
    TARGET
].to_numpy(dtype=float)

groups = train[
    UNIT
].to_numpy()

training_engines = set(
    train[
        UNIT
    ].unique()
)


# ==============================================================================
# 5. FREEZE ENGINE-GROUPED CROSS-VALIDATION
# ==============================================================================

candidates = list(
    ParameterGrid(
        PARAMETER_GRID
    )
)

splitter = GroupKFold(
    n_splits=FOLDS,
    shuffle=True,
    random_state=SEED,
)

splits = list(
    splitter.split(
        X_train,
        y_train,
        groups=groups,
    )
)


folds_are_disjoint = all(
    set(
        groups[
            fit_index
        ]
    ).isdisjoint(
        set(
            groups[
                holdout_index
            ]
        )
    )
    for fit_index, holdout_index
    in splits
)


if not folds_are_disjoint:
    raise AssertionError(
        "Cross-validation folds are not engine-disjoint."
    )


search_record = {
    "frozen_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "selection_population":
        "60 model-training engines only",

    "selection_metric":
        "mean_engine_rmse",

    "candidate_grid":
        PARAMETER_GRID,

    "fixed_parameters":
        FIXED_PARAMETERS,

    "candidate_count":
        len(candidates),

    "fold_count":
        FOLDS,

    "validation_loaded_during_search":
        False,

    "calibration_values_used":
        False,

    "official_test_data_used":
        False,
}


# ==============================================================================
# 6. TRAINING-ONLY ENGINE-GROUPED XGBOOST SEARCH
# ==============================================================================

print("\nTRAINING XGBOOST CANDIDATES")
print("=" * 90)

print(
    f"Candidates: {len(candidates)} | "
    f"Folds: {FOLDS} | "
    f"Boosting rounds per fit: {N_ESTIMATORS}"
)

cv_rows = []
all_oof_complete = True


for candidate_number, parameters in enumerate(
    candidates,
    start=1,
):
    out_of_fold = np.full(
        len(train),
        np.nan,
        dtype=float,
    )

    negative_raw_oof_rows = 0

    for fit_index, holdout_index in splits:
        fold_model = build_xgboost(
            parameters
        )

        fold_model.fit(
            X_train.iloc[
                fit_index
            ],
            y_train[
                fit_index
            ],
        )

        raw_fold_prediction = fold_model.predict(
            X_train.iloc[
                holdout_index
            ]
        )

        negative_raw_oof_rows += int(
            np.sum(
                raw_fold_prediction
                < 0
            )
        )

        out_of_fold[
            holdout_index
        ] = np.clip(
            raw_fold_prediction,
            LOWER_BOUND,
            None,
        )

        del fold_model
        gc.collect()

    oof_complete = bool(
        np.isfinite(
            out_of_fold
        ).all()
    )

    all_oof_complete = (
        all_oof_complete
        and oof_complete
    )

    if not oof_complete:
        raise AssertionError(
            f"Candidate {candidate_number} "
            "produced incomplete OOF predictions."
        )

    cv_record, _ = model_metrics(
        f"xgboost_candidate_{candidate_number}",
        groups,
        y_train,
        out_of_fold,
    )

    cv_rows.append(
        {
            "candidate_number":
                candidate_number,

            **parameters,

            "n_estimators":
                N_ESTIMATORS,

            "subsample":
                FIXED_PARAMETERS[
                    "subsample"
                ],

            "colsample_bytree":
                FIXED_PARAMETERS[
                    "colsample_bytree"
                ],

            "reg_lambda":
                FIXED_PARAMETERS[
                    "reg_lambda"
                ],

            "cv_pooled_mae":
                cv_record[
                    "pooled_mae"
                ],

            "cv_pooled_rmse":
                cv_record[
                    "pooled_rmse"
                ],

            "cv_pooled_r_squared":
                cv_record[
                    "pooled_r_squared"
                ],

            "cv_mean_engine_mae":
                cv_record[
                    "mean_engine_mae"
                ],

            "cv_mean_engine_rmse":
                cv_record[
                    "mean_engine_rmse"
                ],

            "cv_median_engine_rmse":
                cv_record[
                    "median_engine_rmse"
                ],

            "negative_raw_oof_prediction_rows":
                negative_raw_oof_rows,
        }
    )

    print(
        f"Completed "
        f"{candidate_number:02d}/"
        f"{len(candidates)} — "
        f"depth="
        f"{parameters['max_depth']}, "
        f"learning_rate="
        f"{parameters['learning_rate']}, "
        f"min_child_weight="
        f"{parameters['min_child_weight']}"
    )


cv_results = pd.DataFrame(
    cv_rows
)


cv_results = (
    cv_results
    .sort_values(
        [
            "cv_mean_engine_rmse",
            "max_depth",
            "min_child_weight",
            "learning_rate",
        ],
        ascending=[
            True,
            True,
            False,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


best = cv_results.iloc[0]

selected_candidate = int(
    best[
        "candidate_number"
    ]
)

selected_parameters = {
    "max_depth": int(
        best[
            "max_depth"
        ]
    ),

    "learning_rate": float(
        best[
            "learning_rate"
        ]
    ),

    "min_child_weight": int(
        best[
            "min_child_weight"
        ]
    ),
}


# Hyperparameters are now frozen.
# Validation has not yet been loaded.

xgb_model = build_xgboost(
    selected_parameters
)

xgb_model.fit(
    X_train,
    y_train,
)


# ==============================================================================
# 7. LOAD VALIDATION ONLY AFTER XGBOOST IS FROZEN
# ==============================================================================

validation = pd.read_csv(
    VALID_PATH
)

validation_hash_before = frame_hash(
    validation
)

validation_engines = set(
    validation[
        UNIT
    ].unique()
)

validation_units = validation[
    UNIT
].to_numpy()

y_validation = validation[
    TARGET
].to_numpy(dtype=float)

X_validation = validation[
    FEATURES
].copy()


xgb_raw = xgb_model.predict(
    X_validation
)

xgb_prediction = np.clip(
    xgb_raw,
    LOWER_BOUND,
    None,
)

negative_xgb_rows = int(
    np.sum(
        xgb_raw < 0
    )
)


# ==============================================================================
# 8. REPRODUCE THE APPROVED COMPARATORS
# ==============================================================================

median_lifetime = int(
    round(
        train.groupby(
            UNIT
        )[
            CYCLE
        ].max().median()
    )
)


naive_prediction = np.maximum(
    median_lifetime
    - validation[
        CYCLE
    ].to_numpy(),
    0,
).astype(float)


ridge_features = list(
    ridge_bundle[
        "feature_order"
    ]
)


ridge_scaled = pd.DataFrame(
    ridge_bundle[
        "scaler"
    ].transform(
        validation[
            ridge_features
        ]
    ),
    columns=ridge_features,
    index=validation.index,
)


ridge_prediction = np.clip(
    ridge_bundle[
        "model"
    ].predict(
        ridge_scaled
    ),
    ridge_bundle[
        "prediction_lower_bound"
    ],
    None,
)


rf_prediction = np.clip(
    rf_bundle[
        "model"
    ].predict(
        validation[
            rf_bundle[
                "feature_order"
            ]
        ]
    ),
    rf_bundle[
        "prediction_lower_bound"
    ],
    None,
)


# ==============================================================================
# 9. COMPARABLE VALIDATION METRICS
# ==============================================================================

naive_record, _ = model_metrics(
    "naive_age_only",
    validation_units,
    y_validation,
    naive_prediction,
)

ridge_record, _ = model_metrics(
    "ridge_uncapped",
    validation_units,
    y_validation,
    ridge_prediction,
)

rf_record, _ = model_metrics(
    "random_forest_uncapped",
    validation_units,
    y_validation,
    rf_prediction,
)

xgb_record, xgb_engine = model_metrics(
    "xgboost_uncapped",
    validation_units,
    y_validation,
    xgb_prediction,
)


comparison = pd.DataFrame(
    [
        naive_record,
        ridge_record,
        rf_record,
        xgb_record,
    ]
)


comparison[
    "primary_metric_rank"
] = (
    comparison[
        "mean_engine_rmse"
    ]
    .rank(
        method="min",
        ascending=True,
    )
    .astype(int)
)


comparison = (
    comparison
    .sort_values(
        "primary_metric_rank"
    )
    .reset_index(
        drop=True
    )
)


current_leader = str(
    comparison.iloc[0][
        "model"
    ]
)


# ==============================================================================
# 10. VERIFY THE PREVIOUS THREE MODEL RESULTS DID NOT CHANGE
# ==============================================================================

prior_comparison = pd.read_csv(
    PRIOR_COMPARISON_PATH
)


reproduced_prior = comparison[
    comparison[
        "model"
    ].isin(
        [
            "naive_age_only",
            "ridge_uncapped",
            "random_forest_uncapped",
        ]
    )
]


metric_columns = [
    "pooled_mae",
    "pooled_rmse",
    "pooled_r_squared",
    "mean_row_level_phm_penalty",
    "mean_engine_mae",
    "mean_engine_rmse",
    "median_engine_rmse",
]


prior_merged = prior_comparison.merge(
    reproduced_prior,
    on="model",
    suffixes=(
        "_saved",
        "_reproduced",
    ),
)


prior_metrics_match = all(
    np.allclose(
        prior_merged[
            f"{metric}_saved"
        ],
        prior_merged[
            f"{metric}_reproduced"
        ],
        rtol=1e-9,
        atol=1e-9,
    )
    for metric in metric_columns
)


# ==============================================================================
# 11. DESCRIPTIVE XGBOOST IMPORTANCE
# ==============================================================================

importance = pd.DataFrame(
    {
        "feature":
            FEATURES,

        "gain_importance":
            xgb_model.feature_importances_,
    }
)


importance = (
    importance
    .sort_values(
        "gain_importance",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


importance[
    "importance_rank"
] = np.arange(
    1,
    len(importance) + 1,
)


# ==============================================================================
# 12. PREPARE OUTPUT TABLES
# ==============================================================================

prediction_output = pd.DataFrame(
    {
        "unit_number":
            validation[
                UNIT
            ].to_numpy(),

        "time_in_cycles":
            validation[
                CYCLE
            ].to_numpy(),

        "actual_rul_uncapped":
            y_validation,

        "xgboost_raw_prediction":
            xgb_raw,

        "xgboost_prediction":
            xgb_prediction,

        "prediction_was_lower_clipped":
            xgb_raw < 0,

        "error_predicted_minus_actual":
            xgb_prediction
            - y_validation,

        "absolute_error":
            np.abs(
                xgb_prediction
                - y_validation
            ),
    }
)


# ==============================================================================
# 13. SAVE XGBOOST MODEL BUNDLE
# ==============================================================================

xgb_bundle = {
    "model_name":
        "XGBoost Regressor",

    "target":
        TARGET,

    "feature_order":
        FEATURES,

    "input_scaling":
        "none",

    "selection_metric":
        "mean_engine_rmse",

    "selected_candidate_number":
        selected_candidate,

    "search_grid":
        PARAMETER_GRID,

    "fixed_parameters":
        FIXED_PARAMETERS,

    "selected_hyperparameters": {
        **FIXED_PARAMETERS,
        **selected_parameters,
    },

    "prediction_lower_bound":
        LOWER_BOUND,

    "model":
        xgb_model,

    "fit_engine_count":
        int(
            len(
                training_engines
            )
        ),

    "fit_row_count":
        int(
            len(train)
        ),

    "xgboost_version":
        xgb.__version__,

    "scikit_learn_version":
        sklearn.__version__,
}


joblib.dump(
    xgb_bundle,
    MODEL_PATH,
)


cv_results.to_csv(
    CV_PATH,
    index=False,
)

prediction_output.to_csv(
    PRED_PATH,
    index=False,
)

xgb_engine.to_csv(
    ENGINE_PATH,
    index=False,
)

comparison.to_csv(
    COMPARISON_PATH,
    index=False,
)

importance.to_csv(
    IMPORTANCE_PATH,
    index=False,
)


# ==============================================================================
# 14. RELOAD AND VERIFY THE SAVED MODEL
# ==============================================================================

reloaded = joblib.load(
    MODEL_PATH
)


reloaded_prediction = np.clip(
    reloaded[
        "model"
    ].predict(
        validation[
            reloaded[
                "feature_order"
            ]
        ]
    ),
    reloaded[
        "prediction_lower_bound"
    ],
    None,
)


reload_matches = bool(
    np.allclose(
        reloaded_prediction,
        xgb_prediction,
        rtol=1e-10,
        atol=1e-10,
    )
)


# ==============================================================================
# 15. SAVE FORMAL XGBOOST DEFINITION
# ==============================================================================

definition = {
    "recorded_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "phase":
        4,

    "step":
        "4",

    "model":
        "XGBoost Regressor",

    "primary_target":
        TARGET,

    "predictor_count":
        len(FEATURES),

    "predictor_order":
        FEATURES,

    "tree_input_scaled":
        False,

    "training_engine_count":
        len(
            training_engines
        ),

    "training_row_count":
        len(train),

    "validation_engine_count":
        len(
            validation_engines
        ),

    "validation_row_count":
        len(validation),

    "candidate_count":
        len(candidates),

    "cv_fold_count":
        FOLDS,

    "selected_candidate":
        selected_candidate,

    "selected_hyperparameters":
        selected_parameters,

    "fixed_parameters":
        FIXED_PARAMETERS,

    "validation_used_for_tuning":
        False,

    "validation_used_for_evaluation":
        True,

    "calibration_values_used":
        False,

    "official_test_data_used":
        False,

    "test_rul_values_accessed":
        False,

    "sensitivity_target_used":
        False,

    "negative_raw_validation_predictions":
        negative_xgb_rows,

    "current_primary_metric_leader":
        current_leader,

    "final_model_selected":
        False,

    "importance_status":
        "gain_based_descriptive_not_causal",

    "comparison_policy_sha256":
        file_hash(
            POLICY_PATH
        ),

    "xgboost_bundle_sha256":
        file_hash(
            MODEL_PATH
        ),

    "xgboost_version":
        xgb.__version__,

    "scikit_learn_version":
        sklearn.__version__,

    "search_record":
        search_record,
}


with open(
    DEFINITION_PATH,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        definition,
        handle,
        indent=2,
    )


# ==============================================================================
# 16. VALIDATION CHECKS
# ==============================================================================

checks = {
    "approved artifact hashes match":
        all(
            hash_status.values()
        ),

    "training contains 60 engines":
        len(
            training_engines
        ) == 60,

    "validation contains 20 engines":
        len(
            validation_engines
        ) == 20,

    "training and validation engines are disjoint":
        training_engines.isdisjoint(
            validation_engines
        ),

    "xgboost contains eighteen predictors":
        len(FEATURES) == 18,

    "unit number is excluded":
        UNIT not in FEATURES,

    "primary target is excluded":
        TARGET not in FEATURES,

    "sensitivity target is excluded":
        SENSITIVITY_TARGET
        not in FEATURES,

    "tree predictors were not scaled":
        True,

    "eighteen candidates were evaluated":
        len(cv_results) == 18,

    "five grouped folds were used":
        len(splits) == 5,

    "all cv folds are engine disjoint":
        folds_are_disjoint,

    "all oof predictions were complete":
        all_oof_complete,

    "selected candidate came from fixed grid":
        selected_candidate
        in set(
            cv_results[
                "candidate_number"
            ]
        ),

    "validation was not used for tuning":
        True,

    "feature order is preserved":
        list(
            xgb_model.feature_names_in_
        ) == FEATURES,

    "validation row count is preserved":
        len(
            xgb_prediction
        ) == len(
            validation
        ),

    "xgboost predictions are finite":
        bool(
            np.isfinite(
                xgb_prediction
            ).all()
        ),

    "xgboost predictions are non negative":
        bool(
            (
                xgb_prediction >= 0
            ).all()
        ),

    "xgboost engine metrics contain 20 engines":
        len(
            xgb_engine
        ) == 20,

    "xgboost engine metrics are finite":
        bool(
            xgb_engine[
                [
                    "mae",
                    "rmse",
                    "bias_predicted_minus_actual",
                ]
            ]
            .apply(
                np.isfinite
            )
            .all()
            .all()
        ),

    "comparison contains four models":
        len(
            comparison
        ) == 4,

    "approved comparator metrics were reproduced":
        prior_metrics_match,

    "gain importances sum to one":
        bool(
            np.isclose(
                importance[
                    "gain_importance"
                ].sum(),
                1.0,
                atol=1e-6,
            )
        ),

    "saved bundle reproduces validation predictions":
        reload_matches,

    "training dataframe was not modified":
        train_hash_before
        == frame_hash(
            train
        ),

    "validation dataframe was not modified":
        validation_hash_before
        == frame_hash(
            validation
        ),

    "calibration file was not modified":
        calibration_hash_before
        == file_hash(
            CAL_PATH
        ),

    "all result files exist":
        all(
            path.exists()
            for path in [
                CV_PATH,
                PRED_PATH,
                ENGINE_PATH,
                COMPARISON_PATH,
                IMPORTANCE_PATH,
                DEFINITION_PATH,
                MODEL_PATH,
            ]
        ),
}


failed_checks = []

print("\nXGBOOST VALIDATION")
print("=" * 90)


for check_name, passed in checks.items():
    print(
        f"{'PASS' if passed else 'FAIL'} — "
        f"{check_name}"
    )

    if not passed:
        failed_checks.append(
            check_name
        )


if failed_checks:
    raise AssertionError(
        "XGBoost validation failed:\n"
        + "\n".join(
            failed_checks
        )
    )


# ==============================================================================
# 17. CREATE STEP 4 EVIDENCE PACKAGE
# ==============================================================================

evidence_paths = [
    POLICY_PATH,
    CV_PATH,
    PRED_PATH,
    ENGINE_PATH,
    COMPARISON_PATH,
    IMPORTANCE_PATH,
    DEFINITION_PATH,
    MODEL_PATH,
]


manifest = pd.DataFrame(
    [
        {
            "file_name":
                path.name,

            "project_relative_path":
                str(
                    path.relative_to(
                        ROOT
                    )
                ),

            "size_bytes":
                path.stat().st_size,

            "sha256":
                file_hash(
                    path
                ),
        }
        for path in evidence_paths
    ]
)


manifest.to_csv(
    MANIFEST_PATH,
    index=False,
)


with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    zipfile.ZIP_DEFLATED,
) as archive:
    for path in (
        evidence_paths
        + [MANIFEST_PATH]
    ):
        archive.write(
            path,
            str(
                path.relative_to(
                    ROOT
                )
            ),
        )


zip_sha256 = file_hash(
    ZIP_PATH
)


# ==============================================================================
# 18. DISPLAY RESULTS
# ==============================================================================

print("\nXGBOOST SEARCH SUMMARY")
print("=" * 90)

print(
    f"Candidates evaluated: "
    f"{len(candidates)}"
)

print(
    f"Grouped folds: "
    f"{FOLDS}"
)

print(
    f"Selected candidate: "
    f"{selected_candidate}"
)

print(
    f"Selected maximum depth: "
    f"{selected_parameters['max_depth']}"
)

print(
    f"Selected learning rate: "
    f"{selected_parameters['learning_rate']}"
)

print(
    "Selected minimum child weight: "
    f"{selected_parameters['min_child_weight']}"
)

print(
    f"Boosting rounds: "
    f"{N_ESTIMATORS}"
)

print(
    f"XGBoost version: "
    f"{xgb.__version__}"
)


print(
    "\nTOP TEN TRAINING-ONLY "
    "XGBOOST CV RESULTS"
)

print("=" * 90)

display(
    cv_results
    .head(10)
    .round(6)
)


print(
    "\nVALIDATION MODEL COMPARISON"
)

print("=" * 90)

display(
    comparison.round(6)
)


print("\nXGBOOST IMPROVEMENT")
print("=" * 90)

print(
    "Pooled RMSE improvement "
    "versus Random Forest: "
    f"{improvement_percent(
        rf_record['pooled_rmse'],
        xgb_record['pooled_rmse'],
    ):.2f}%"
)

print(
    "Engine-balanced RMSE improvement "
    "versus Random Forest: "
    f"{improvement_percent(
        rf_record['mean_engine_rmse'],
        xgb_record['mean_engine_rmse'],
    ):.2f}%"
)

print(
    f"Current primary-metric leader: "
    f"{current_leader}"
)

print(
    "Final model selected: No"
)


print(
    "\nTOP TEN GAIN-BASED "
    "FEATURE IMPORTANCES"
)

print("=" * 90)

display(
    importance
    .head(10)
    .round(6)
)

print(
    "Important: these importances are descriptive, "
    "not causal, and are not SHAP values."
)


print("\nRESEARCH CONTROLS")
print("=" * 90)

print(
    "Validation used for "
    "hyperparameter selection: No"
)

print(
    "Validation used for final "
    "XGBoost evaluation: Yes"
)

print(
    "Calibration values used: No"
)

print(
    "Official test data used: No"
)

print(
    "Test RUL values accessed: No"
)

print(
    "Sensitivity target used: No"
)

print(
    "Final model selected: No"
)


print("\nOUTPUT FILES")
print("=" * 90)

for path in evidence_paths:
    print(path)

print(
    f"Evidence manifest: "
    f"{MANIFEST_PATH}"
)

print(
    f"Evidence ZIP: "
    f"{ZIP_PATH}"
)

print(
    "XGBoost bundle SHA-256: "
    f"{file_hash(MODEL_PATH)}"
)

print(
    "Evidence ZIP SHA-256: "
    f"{zip_sha256}"
)


print("\nFINAL CHECK")
print("=" * 90)

print(
    "PASS — XGBoost tuning used "
    "model-training engines only."
)

print(
    "PASS — Cross-validation folds "
    "were separated at engine level."
)

print(
    "PASS — Validation was used only "
    "after XGBoost was frozen."
)

print(
    "PASS — Calibration and official "
    "test data remained unused."
)

print(
    "PASS — The saved XGBoost bundle "
    "reproduces validation predictions."
)

print(
    "AWAITING APPROVAL — Final model "
    "selection has not started."
)


# ==============================================================================
# 19. DOWNLOAD EVIDENCE ZIP
# ==============================================================================

try:
    from google.colab import files

    print(
        "\nDownloading the Step 4 "
        "XGBoost evidence ZIP..."
    )

    files.download(
        str(
            ZIP_PATH
        )
    )

except Exception as error:
    print(
        f"Automatic download unavailable: "
        f"{error}"
    )

    print(
        f"Download manually from: "
        f"{ZIP_PATH}"
    )

In [ ]:
# ==============================================================================
# PHASE 4 — STEP 5
# APPLY THE FROZEN MODEL-SELECTION POLICY AND FREEZE THE FINAL POINT MODEL
# ==============================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import zipfile

import numpy as np
import pandas as pd
from IPython.display import display


# ==============================================================================
# 1. PATHS
# ==============================================================================

ROOT = Path(
    "/content/Predictive_Maintenance_Thesis"
)

RESULTS = ROOT / "results"
LOGS = ROOT / "logs"
MODELS = ROOT / "models/regression"

for directory in [
    RESULTS,
    LOGS,
    MODELS,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


POLICY_PATH = (
    LOGS
    / "phase4_model_comparison_policy.json"
)

SOURCE_COMPARISON_PATH = (
    RESULTS
    / "phase4_naive_ridge_random_forest_xgboost_comparison.csv"
)

RIDGE_BUNDLE_PATH = (
    MODELS
    / "fd001_ridge_uncapped_bundle.joblib"
)

RF_BUNDLE_PATH = (
    MODELS
    / "fd001_random_forest_uncapped_bundle.joblib"
)

XGB_BUNDLE_PATH = (
    MODELS
    / "fd001_xgboost_uncapped_bundle.joblib"
)

CALIBRATION_PATH = (
    ROOT
    / "data/interim/fd001_calibration_targets.csv"
)

RIDGE_DEFINITION_PATH = (
    LOGS
    / "phase4_ridge_model_definition.json"
)

RF_DEFINITION_PATH = (
    LOGS
    / "phase4_random_forest_model_definition.json"
)

XGB_DEFINITION_PATH = (
    LOGS
    / "phase4_xgboost_model_definition.json"
)


FINAL_COMPARISON_PATH = (
    RESULTS
    / "phase4_final_model_comparison.csv"
)

SELECTION_RECORD_PATH = (
    LOGS
    / "phase4_final_model_selection_record.json"
)

FINAL_REGISTRY_PATH = (
    LOGS
    / "phase4_final_model_registry.json"
)

MANIFEST_PATH = (
    LOGS
    / "phase4_step5_final_model_selection_manifest.csv"
)

ZIP_PATH = (
    ROOT
    / "Phase4_Step5_Final_Model_Selection_Evidence.zip"
)


# ==============================================================================
# 2. HELPERS
# ==============================================================================

def file_hash(path):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def native_value(value):
    if pd.isna(value):
        return None

    if isinstance(
        value,
        np.integer,
    ):
        return int(value)

    if isinstance(
        value,
        np.floating,
    ):
        return float(value)

    if isinstance(
        value,
        np.bool_,
    ):
        return bool(value)

    return value


# ==============================================================================
# 3. VERIFY REQUIRED ARTIFACTS
# ==============================================================================

required_paths = [
    POLICY_PATH,
    SOURCE_COMPARISON_PATH,
    RIDGE_BUNDLE_PATH,
    RF_BUNDLE_PATH,
    XGB_BUNDLE_PATH,
    CALIBRATION_PATH,
    RIDGE_DEFINITION_PATH,
    RF_DEFINITION_PATH,
    XGB_DEFINITION_PATH,
]


missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]


if missing_paths:
    raise FileNotFoundError(
        "Required Phase 4 files are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


EXPECTED_BUNDLE_HASHES = {
    "ridge_uncapped": (
        "43104a945295ee3cb9ae50de51db0c643"
        "455feb08d9a6e9b01fb227d1b83b3ea"
    ),

    "random_forest_uncapped": (
        "99620a1a161f62a716d4cc058497159146"
        "197e7d91bba713581c822af808e5d5"
    ),

    "xgboost_uncapped": (
    "720f9b8c4efcf9511c74579f969b49a"
    "df3374ee83fc06b2849d1cee1fae3e1aa"
    ),
}


BUNDLE_PATHS = {
    "ridge_uncapped":
        RIDGE_BUNDLE_PATH,

    "random_forest_uncapped":
        RF_BUNDLE_PATH,

    "xgboost_uncapped":
        XGB_BUNDLE_PATH,
}


actual_bundle_hashes = {
    model_name:
        file_hash(path)

    for model_name, path
    in BUNDLE_PATHS.items()
}


bundle_hash_checks = {
    model_name: (
        actual_bundle_hashes[
            model_name
        ]
        == EXPECTED_BUNDLE_HASHES[
            model_name
        ]
    )

    for model_name
    in EXPECTED_BUNDLE_HASHES
}


print("MODEL ARTIFACT INTEGRITY")
print("=" * 90)

for model_name, passed in bundle_hash_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL'} — "
        f"{model_name}"
    )


if not all(
    bundle_hash_checks.values()
):
    failed_models = [
        model_name
        for model_name, passed
        in bundle_hash_checks.items()
        if not passed
    ]

    raise AssertionError(
        "Model bundle hash mismatch:\n"
        + "\n".join(
            failed_models
        )
    )


calibration_hash_before = file_hash(
    CALIBRATION_PATH
)


# ==============================================================================
# 4. LOAD THE FROZEN POLICY
# ==============================================================================

with POLICY_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    policy = json.load(handle)


required_policy = {
    "primary_metric":
        "mean_engine_rmse",

    "practical_equivalence_threshold_percent":
        1.0,

    "first_tie_breaker":
        "mean_row_level_phm_penalty",

    "second_tie_breaker":
        "pooled_rmse",

    "third_tie_breaker":
        "lower_model_complexity",
}


policy_checks = {
    field: (
        policy.get(field)
        == expected_value
    )

    for field, expected_value
    in required_policy.items()
}


print("\nFROZEN POLICY VERIFICATION")
print("=" * 90)

for field, passed in policy_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL'} — "
        f"{field}"
    )


if not all(
    policy_checks.values()
):
    failed_policy_fields = [
        field
        for field, passed
        in policy_checks.items()
        if not passed
    ]

    raise AssertionError(
        "Frozen comparison policy mismatch:\n"
        + "\n".join(
            failed_policy_fields
        )
    )


policy_sha256 = file_hash(
    POLICY_PATH
)


# ==============================================================================
# 5. LOAD AND VALIDATE THE FOUR-MODEL COMPARISON
# ==============================================================================

comparison = pd.read_csv(
    SOURCE_COMPARISON_PATH
)


EXPECTED_MODELS = {
    "naive_age_only",
    "ridge_uncapped",
    "random_forest_uncapped",
    "xgboost_uncapped",
}


required_metric_columns = [
    "model",
    "pooled_rmse",
    "mean_row_level_phm_penalty",
    "mean_engine_rmse",
]


missing_columns = [
    column
    for column in required_metric_columns
    if column not in comparison.columns
]


if missing_columns:
    raise KeyError(
        "Required comparison columns are missing:\n"
        + "\n".join(
            missing_columns
        )
    )


if set(
    comparison["model"]
) != EXPECTED_MODELS:
    raise AssertionError(
        "The comparison does not contain exactly "
        "the four approved models."
    )


if comparison[
    "model"
].duplicated().any():
    raise AssertionError(
        "The model comparison contains duplicate model rows."
    )


numeric_metric_columns = [
    column
    for column in comparison.columns
    if column != "model"
]


if not (
    comparison[
        numeric_metric_columns
    ]
    .apply(np.isfinite)
    .all()
    .all()
):
    raise AssertionError(
        "The comparison contains non-finite metric values."
    )


# ==============================================================================
# 6. APPLY THE FROZEN SELECTION HIERARCHY
# ==============================================================================

PRIMARY_METRIC = (
    policy[
        "primary_metric"
    ]
)

EQUIVALENCE_PERCENT = float(
    policy[
        "practical_equivalence_threshold_percent"
    ]
)


COMPLEXITY_ORDER = {
    "naive_age_only": 1,
    "ridge_uncapped": 2,
    "random_forest_uncapped": 3,
    "xgboost_uncapped": 4,
}


comparison[
    "complexity_rank"
] = comparison[
    "model"
].map(
    COMPLEXITY_ORDER
)


if comparison[
    "complexity_rank"
].isna().any():
    raise AssertionError(
        "A model is missing from the frozen complexity order."
    )


primary_leader_value = float(
    comparison[
        PRIMARY_METRIC
    ].min()
)


practical_equivalence_limit = (
    primary_leader_value
    * (
        1
        + EQUIVALENCE_PERCENT
        / 100
    )
)


comparison[
    "within_practical_equivalence_band"
] = (
    comparison[
        PRIMARY_METRIC
    ]
    <= practical_equivalence_limit
)


equivalent_models = comparison[
    comparison[
        "within_practical_equivalence_band"
    ]
].copy()


if equivalent_models.empty:
    raise AssertionError(
        "No model falls inside the practical-equivalence band."
    )


# Apply the frozen tie-breakers in their declared order.
selection_ranking = (
    equivalent_models
    .sort_values(
        [
            "mean_row_level_phm_penalty",
            "pooled_rmse",
            "complexity_rank",
        ],
        ascending=[
            True,
            True,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


selected_row = selection_ranking.iloc[0]

selected_model = str(
    selected_row[
        "model"
    ]
)


comparison[
    "selected_final_model"
] = (
    comparison[
        "model"
    ]
    == selected_model
)


# ==============================================================================
# 7. CONSTRUCT THE DECISION TRACE
# ==============================================================================

models_in_equivalence_band = list(
    equivalent_models[
        "model"
    ]
)


decision_trace = [
    (
        f"Primary-metric leader value: "
        f"{primary_leader_value:.6f}"
    ),

    (
        f"Practical-equivalence threshold: "
        f"{EQUIVALENCE_PERCENT:.2f}%"
    ),

    (
        f"Practical-equivalence upper limit: "
        f"{practical_equivalence_limit:.6f}"
    ),

    (
        "Models inside the equivalence band: "
        + ", ".join(
            models_in_equivalence_band
        )
    ),
]


if len(
    equivalent_models
) == 1:
    decision_trace.append(
        "Only one model was inside the practical-equivalence "
        "band; no tie-breaker was required."
    )

else:
    decision_trace.append(
        "Multiple models were practically equivalent."
    )

    decision_trace.append(
        "Tie-breaker 1 was applied: "
        "lower mean row-level PHM penalty."
    )

    decision_trace.append(
        f"Tie-breaker 1 selected {selected_model} "
        f"with a PHM diagnostic of "
        f"{selected_row['mean_row_level_phm_penalty']:.6f}."
    )


# ==============================================================================
# 8. FREEZE THE SELECTED MODEL
# ==============================================================================

selected_bundle_path = (
    BUNDLE_PATHS.get(
        selected_model
    )
)


if selected_bundle_path is None:
    raise AssertionError(
        "The frozen policy selected the naive baseline. "
        "A separate deterministic baseline registry would be required."
    )


selected_bundle_sha256 = file_hash(
    selected_bundle_path
)


selected_metrics = {
    key:
        native_value(value)

    for key, value
    in selected_row.to_dict().items()
}


registry = {
    "recorded_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "phase":
        4,

    "step":
        "5",

    "status":
        "final_point_model_frozen",

    "selection_policy": {
        "primary_metric":
            PRIMARY_METRIC,

        "practical_equivalence_threshold_percent":
            EQUIVALENCE_PERCENT,

        "first_tie_breaker":
            policy[
                "first_tie_breaker"
            ],

        "second_tie_breaker":
            policy[
                "second_tie_breaker"
            ],

        "third_tie_breaker":
            policy[
                "third_tie_breaker"
            ],

        "policy_sha256":
            policy_sha256,
    },

    "models_compared":
        list(
            comparison[
                "model"
            ]
        ),

    "models_within_practical_equivalence":
        models_in_equivalence_band,

    "primary_metric_leader_value":
        primary_leader_value,

    "practical_equivalence_upper_limit":
        practical_equivalence_limit,

    "decision_trace":
        decision_trace,

    "selected_model":
        selected_model,

    "selected_model_path":
        str(
            selected_bundle_path
        ),

    "selected_model_project_relative_path":
        str(
            selected_bundle_path.relative_to(
                ROOT
            )
        ),

    "selected_model_sha256":
        selected_bundle_sha256,

    "selected_model_metrics":
        selected_metrics,

    "all_model_bundle_hashes":
        actual_bundle_hashes,

    "point_model_frozen":
        True,

    "validation_reloaded_for_selection":
        False,

    "calibration_values_used":
        False,

    "official_test_data_used":
        False,

    "test_rul_values_accessed":
        False,

    "sensitivity_target_used":
        False,

    "next_permitted_step":
        (
            "Phase 4 Step 6 — capped-130 "
            "sensitivity analysis"
        ),
}


selection_record = {
    "recorded_at_utc":
        registry[
            "recorded_at_utc"
        ],

    "selected_model":
        selected_model,

    "selected_model_sha256":
        selected_bundle_sha256,

    "primary_metric":
        PRIMARY_METRIC,

    "primary_metric_value":
        float(
            selected_row[
                PRIMARY_METRIC
            ]
        ),

    "practical_equivalence_percent":
        EQUIVALENCE_PERCENT,

    "equivalent_models":
        models_in_equivalence_band,

    "tie_breaker_used":
        (
            policy[
                "first_tie_breaker"
            ]
            if len(
                equivalent_models
            ) > 1
            else None
        ),

    "tie_breaker_value":
        (
            float(
                selected_row[
                    "mean_row_level_phm_penalty"
                ]
            )
            if len(
                equivalent_models
            ) > 1
            else None
        ),

    "decision_trace":
        decision_trace,

    "final_model_selected":
        True,

    "model_retrained_during_selection":
        False,

    "validation_predictions_recomputed":
        False,

    "calibration_values_used":
        False,

    "official_test_data_used":
        False,
}


comparison.to_csv(
    FINAL_COMPARISON_PATH,
    index=False,
)


with FINAL_REGISTRY_PATH.open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        registry,
        handle,
        indent=2,
    )


with SELECTION_RECORD_PATH.open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        selection_record,
        handle,
        indent=2,
    )


# ==============================================================================
# 9. FINAL VALIDATION
# ==============================================================================

checks = {
    "frozen policy was preserved":
        file_hash(
            POLICY_PATH
        )
        == policy_sha256,

    "four approved models were compared":
        set(
            comparison[
                "model"
            ]
        )
        == EXPECTED_MODELS,

    "all comparison metrics are finite":
        bool(
            comparison[
                numeric_metric_columns
            ]
            .apply(np.isfinite)
            .all()
            .all()
        ),

    "practical-equivalence band is non-empty":
        len(
            equivalent_models
        ) >= 1,

    "selected model came from the comparison":
        selected_model
        in EXPECTED_MODELS,

    "selected model has a saved bundle":
        selected_bundle_path.exists(),

    "selected bundle hash is preserved":
        file_hash(
            selected_bundle_path
        )
        == selected_bundle_sha256,

    "selected model is Random Forest":
        selected_model
        == "random_forest_uncapped",

    "Random Forest and naive are practically equivalent":
        {
            "random_forest_uncapped",
            "naive_age_only",
        }.issubset(
            set(
                models_in_equivalence_band
            )
        ),

    "XGBoost is outside the equivalence band":
        "xgboost_uncapped"
        not in models_in_equivalence_band,

    "Ridge is outside the equivalence band":
        "ridge_uncapped"
        not in models_in_equivalence_band,

    "final registry exists":
        FINAL_REGISTRY_PATH.exists(),

    "selection record exists":
        SELECTION_RECORD_PATH.exists(),

    "final comparison exists":
        FINAL_COMPARISON_PATH.exists(),

    "calibration file was not modified":
        calibration_hash_before
        == file_hash(
            CALIBRATION_PATH
        ),
}


print("\nFINAL MODEL-SELECTION VALIDATION")
print("=" * 90)

failed_checks = []

for check_name, passed in checks.items():
    print(
        f"{'PASS' if passed else 'FAIL'} — "
        f"{check_name}"
    )

    if not passed:
        failed_checks.append(
            check_name
        )


if failed_checks:
    raise AssertionError(
        "Final model-selection checks failed:\n"
        + "\n".join(
            failed_checks
        )
    )


# ==============================================================================
# 10. EVIDENCE PACKAGE
# ==============================================================================

evidence_paths = [
    POLICY_PATH,
    SOURCE_COMPARISON_PATH,
    FINAL_COMPARISON_PATH,
    SELECTION_RECORD_PATH,
    FINAL_REGISTRY_PATH,
    RIDGE_DEFINITION_PATH,
    RF_DEFINITION_PATH,
    XGB_DEFINITION_PATH,
    selected_bundle_path,
]


manifest = pd.DataFrame(
    [
        {
            "file_name":
                path.name,

            "project_relative_path":
                str(
                    path.relative_to(
                        ROOT
                    )
                ),

            "size_bytes":
                path.stat().st_size,

            "sha256":
                file_hash(
                    path
                ),
        }

        for path in evidence_paths
    ]
)


manifest.to_csv(
    MANIFEST_PATH,
    index=False,
)


with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    zipfile.ZIP_DEFLATED,
) as archive:
    for path in (
        evidence_paths
        + [MANIFEST_PATH]
    ):
        archive.write(
            path,
            str(
                path.relative_to(
                    ROOT
                )
            ),
        )


zip_sha256 = file_hash(
    ZIP_PATH
)


# ==============================================================================
# 11. DISPLAY THE FINAL DECISION
# ==============================================================================

print("\nFROZEN MODEL-SELECTION POLICY")
print("=" * 90)

print(
    f"Primary metric: "
    f"{PRIMARY_METRIC}"
)

print(
    f"Practical-equivalence threshold: "
    f"{EQUIVALENCE_PERCENT:.2f}%"
)

print(
    "First tie-breaker: "
    f"{policy['first_tie_breaker']}"
)

print(
    "Second tie-breaker: "
    f"{policy['second_tie_breaker']}"
)

print(
    "Third tie-breaker: "
    f"{policy['third_tie_breaker']}"
)


print("\nFINAL MODEL COMPARISON")
print("=" * 90)

display(
    comparison[
        [
            "model",
            "mean_engine_rmse",
            "mean_row_level_phm_penalty",
            "pooled_rmse",
            "complexity_rank",
            "within_practical_equivalence_band",
            "selected_final_model",
        ]
    ].round(6)
)


print("\nDECISION TRACE")
print("=" * 90)

for decision_line in decision_trace:
    print(
        f"• {decision_line}"
    )


print("\nFINAL MODEL SELECTION")
print("=" * 90)

print(
    f"Selected model: "
    f"{selected_model}"
)

print(
    f"Selected mean engine RMSE: "
    f"{selected_row['mean_engine_rmse']:.6f}"
)

print(
    f"Selected PHM diagnostic: "
    f"{selected_row['mean_row_level_phm_penalty']:.6f}"
)

print(
    f"Selected pooled RMSE: "
    f"{selected_row['pooled_rmse']:.6f}"
)

print(
    f"Selected model SHA-256: "
    f"{selected_bundle_sha256}"
)


print("\nRESEARCH CONTROLS")
print("=" * 90)

print(
    "Model retrained during selection: No"
)

print(
    "Validation predictions recomputed: No"
)

print(
    "Calibration values used: No"
)

print(
    "Official test data used: No"
)

print(
    "Test RUL values accessed: No"
)

print(
    "Sensitivity target used: No"
)


print("\nOUTPUT FILES")
print("=" * 90)

print(
    f"Final comparison: "
    f"{FINAL_COMPARISON_PATH}"
)

print(
    f"Selection record: "
    f"{SELECTION_RECORD_PATH}"
)

print(
    f"Final model registry: "
    f"{FINAL_REGISTRY_PATH}"
)

print(
    f"Evidence manifest: "
    f"{MANIFEST_PATH}"
)

print(
    f"Evidence ZIP: "
    f"{ZIP_PATH}"
)

print(
    f"Evidence ZIP SHA-256: "
    f"{zip_sha256}"
)


print("\nFINAL CHECK")
print("=" * 90)

print(
    "PASS — The frozen model-selection policy "
    "was applied mechanically."
)

print(
    "PASS — Random Forest was selected through "
    "the predefined practical-equivalence and "
    "tie-breaker hierarchy."
)

print(
    "PASS — The selected Random Forest bundle "
    "was not retrained or modified."
)

print(
    "PASS — Calibration and official test data "
    "remain sealed."
)

print(
    "PASS — The final point model is frozen."
)

print(
    "AWAITING APPROVAL — Capped-130 sensitivity "
    "analysis has not started."
)


# ==============================================================================
# 12. DOWNLOAD STEP 5 EVIDENCE
# ==============================================================================

try:
    from google.colab import files

    print(
        "\nDownloading the Step 5 "
        "final-model-selection evidence ZIP..."
    )

    files.download(
        str(
            ZIP_PATH
        )
    )

except Exception as error:
    print(
        f"Automatic download unavailable: "
        f"{error}"
    )

    print(
        f"Download manually from: "
        f"{ZIP_PATH}"
    )

In [ ]:
# ==============================================================================
# PHASE 4 — STEP 6
# CAPPED-130 TARGET SENSITIVITY ANALYSIS
# ==============================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import zipfile

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score,
)


# ==============================================================================
# 1. PATHS AND LOCKED SETTINGS
# ==============================================================================

ROOT = Path(
    "/content/Predictive_Maintenance_Thesis"
)

TRAIN_PATH = (
    ROOT
    / "data/interim/fd001_model_training_targets.csv"
)

VALID_PATH = (
    ROOT
    / "data/interim/fd001_validation_targets.csv"
)

CALIBRATION_PATH = (
    ROOT
    / "data/interim/fd001_calibration_targets.csv"
)

FINAL_REGISTRY_PATH = (
    ROOT
    / "logs/phase4_final_model_registry.json"
)

PRIMARY_MODEL_PATH = (
    ROOT
    / "models/regression/fd001_random_forest_uncapped_bundle.joblib"
)

RESULTS = ROOT / "results"
LOGS = ROOT / "logs"
MODELS = ROOT / "models/regression"

for directory in [
    RESULTS,
    LOGS,
    MODELS,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


CAPPED_MODEL_PATH = (
    MODELS
    / "fd001_random_forest_capped130_sensitivity_bundle.joblib"
)

CAPPED_SCALE_PATH = (
    RESULTS
    / "phase4_capped130_comparison_capped_scale.csv"
)

ACTUAL_SCALE_PATH = (
    RESULTS
    / "phase4_capped130_comparison_actual_scale.csv"
)

STAGE_PATH = (
    RESULTS
    / "phase4_capped130_lifecycle_stage_errors.csv"
)

PREDICTION_PATH = (
    RESULTS
    / "phase4_capped130_validation_predictions.csv"
)

SENSITIVITY_RECORD_PATH = (
    LOGS
    / "phase4_capped130_sensitivity_record.json"
)

MANIFEST_PATH = (
    LOGS
    / "phase4_step6_capped130_evidence_manifest.csv"
)

ZIP_PATH = (
    ROOT
    / "Phase4_Step6_Capped130_Sensitivity_Evidence.zip"
)


PRIMARY_TARGET = "rul_uncapped"
SENSITIVITY_TARGET = "rul_capped_130"
UNIT = "unit_number"
CYCLE = "time_in_cycles"

CAP_CYCLES = 130
LOWER_BOUND = 0.0

EXPECTED_PRIMARY_MODEL_SHA256 = (
    "99620a1a161f62a716d4cc058497159146"
    "197e7d91bba713581c822af808e5d5"
)


# ==============================================================================
# 2. HELPERS
# ==============================================================================

def file_hash(path):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def frame_hash(frame):
    values = pd.util.hash_pandas_object(
        frame,
        index=True,
    ).values.tobytes()

    return hashlib.sha256(
        values
    ).hexdigest()


def phm_penalty(
    actual,
    predicted,
):
    actual = np.asarray(
        actual,
        dtype=float,
    )

    predicted = np.asarray(
        predicted,
        dtype=float,
    )

    error = predicted - actual

    return np.where(
        error >= 0,
        np.expm1(error / 10.0),
        np.expm1(-error / 13.0),
    )


def engine_metrics(
    units,
    actual,
    predicted,
    model_name,
):
    frame = pd.DataFrame(
        {
            "unit_number":
                np.asarray(units),

            "actual":
                np.asarray(
                    actual,
                    dtype=float,
                ),

            "predicted":
                np.asarray(
                    predicted,
                    dtype=float,
                ),
        }
    )

    rows = []

    for unit, group in frame.groupby(
        "unit_number",
        sort=True,
    ):
        y_true = group[
            "actual"
        ].to_numpy()

        y_pred = group[
            "predicted"
        ].to_numpy()

        rows.append(
            {
                "model":
                    model_name,

                "unit_number":
                    int(unit),

                "row_count":
                    int(len(group)),

                "mae":
                    float(
                        mean_absolute_error(
                            y_true,
                            y_pred,
                        )
                    ),

                "rmse":
                    float(
                        root_mean_squared_error(
                            y_true,
                            y_pred,
                        )
                    ),

                "bias_predicted_minus_actual":
                    float(
                        np.mean(
                            y_pred - y_true
                        )
                    ),
            }
        )

    return pd.DataFrame(
        rows
    )


def model_metrics(
    model_name,
    units,
    actual,
    predicted,
):
    actual = np.asarray(
        actual,
        dtype=float,
    )

    predicted = np.asarray(
        predicted,
        dtype=float,
    )

    error = predicted - actual

    by_engine = engine_metrics(
        units,
        actual,
        predicted,
        model_name,
    )

    record = {
        "model":
            model_name,

        "pooled_mae":
            float(
                mean_absolute_error(
                    actual,
                    predicted,
                )
            ),

        "pooled_rmse":
            float(
                root_mean_squared_error(
                    actual,
                    predicted,
                )
            ),

        "pooled_r_squared":
            float(
                r2_score(
                    actual,
                    predicted,
                )
            ),

        "bias_predicted_minus_actual":
            float(
                np.mean(error)
            ),

        "rul_overestimation_rate_percent":
            float(
                np.mean(
                    error > 0
                ) * 100
            ),

        "rul_underestimation_rate_percent":
            float(
                np.mean(
                    error < 0
                ) * 100
            ),

        "mean_row_level_phm_penalty":
            float(
                np.mean(
                    phm_penalty(
                        actual,
                        predicted,
                    )
                )
            ),

        "mean_engine_mae":
            float(
                by_engine[
                    "mae"
                ].mean()
            ),

        "mean_engine_rmse":
            float(
                by_engine[
                    "rmse"
                ].mean()
            ),

        "median_engine_rmse":
            float(
                by_engine[
                    "rmse"
                ].median()
            ),

        "engine_rmse_standard_deviation":
            float(
                by_engine[
                    "rmse"
                ].std(ddof=1)
            ),
    }

    return record, by_engine


# ==============================================================================
# 3. VERIFY THE FROZEN PRIMARY MODEL
# ==============================================================================

required_paths = [
    TRAIN_PATH,
    VALID_PATH,
    CALIBRATION_PATH,
    FINAL_REGISTRY_PATH,
    PRIMARY_MODEL_PATH,
]


missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]


if missing_paths:
    raise FileNotFoundError(
        "Required files are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


primary_hash_before = file_hash(
    PRIMARY_MODEL_PATH
)


if (
    primary_hash_before
    != EXPECTED_PRIMARY_MODEL_SHA256
):
    raise AssertionError(
        "The frozen Random Forest bundle hash has changed."
    )


with FINAL_REGISTRY_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    final_registry = json.load(
        handle
    )


if (
    final_registry.get(
        "selected_model"
    )
    != "random_forest_uncapped"
):
    raise AssertionError(
        "The final registry does not select "
        "random_forest_uncapped."
    )


if not final_registry.get(
    "point_model_frozen",
    False,
):
    raise AssertionError(
        "The final point model is not marked as frozen."
    )


if (
    final_registry.get(
        "selected_model_sha256"
    )
    != EXPECTED_PRIMARY_MODEL_SHA256
):
    raise AssertionError(
        "The final registry contains an unexpected model hash."
    )


print("FROZEN PRIMARY MODEL CHECK")
print("=" * 90)

print(
    "PASS — Selected model: "
    "random_forest_uncapped"
)

print(
    "PASS — Primary model SHA-256: "
    f"{primary_hash_before}"
)

print(
    "PASS — Final point model is frozen."
)


# ==============================================================================
# 4. LOAD TRAINING DATA AND VALIDATE THE SENSITIVITY TARGET
# ==============================================================================

training = pd.read_csv(
    TRAIN_PATH
)

training_hash_before = frame_hash(
    training
)

calibration_hash_before = file_hash(
    CALIBRATION_PATH
)

primary_bundle = joblib.load(
    PRIMARY_MODEL_PATH
)

FEATURES = list(
    primary_bundle[
        "feature_order"
    ]
)


required_columns = (
    FEATURES
    + [
        UNIT,
        CYCLE,
        PRIMARY_TARGET,
        SENSITIVITY_TARGET,
    ]
)


missing_training_columns = [
    column
    for column in required_columns
    if column not in training.columns
]


if missing_training_columns:
    raise KeyError(
        "Training columns are missing:\n"
        + "\n".join(
            missing_training_columns
        )
    )


X_train = training[
    FEATURES
].copy()

y_train_uncapped = training[
    PRIMARY_TARGET
].to_numpy(dtype=float)

y_train_capped = training[
    SENSITIVITY_TARGET
].to_numpy(dtype=float)


training_cap_formula_correct = bool(
    np.array_equal(
        y_train_capped,
        np.minimum(
            y_train_uncapped,
            CAP_CYCLES,
        ),
    )
)


if not training_cap_formula_correct:
    raise AssertionError(
        "Training capped target does not equal "
        "min(rul_uncapped, 130)."
    )


training_engines = set(
    training[
        UNIT
    ].unique()
)


# ==============================================================================
# 5. REFIT THE SAME RANDOM FOREST ARCHITECTURE ON CAPPED LABELS
# ==============================================================================

# No hyperparameter search occurs here.
# The frozen primary Random Forest architecture is cloned exactly.

capped_model = clone(
    primary_bundle[
        "model"
    ]
)


if (
    capped_model.get_params()
    != primary_bundle[
        "model"
    ].get_params()
):
    raise AssertionError(
        "The cloned sensitivity model parameters changed."
    )


capped_model.fit(
    X_train,
    y_train_capped,
)


# ==============================================================================
# 6. LOAD VALIDATION ONLY FOR SENSITIVITY EVALUATION
# ==============================================================================

validation = pd.read_csv(
    VALID_PATH
)

validation_hash_before = frame_hash(
    validation
)


missing_validation_columns = [
    column
    for column in required_columns
    if column not in validation.columns
]


if missing_validation_columns:
    raise KeyError(
        "Validation columns are missing:\n"
        + "\n".join(
            missing_validation_columns
        )
    )


validation_engines = set(
    validation[
        UNIT
    ].unique()
)

validation_units = validation[
    UNIT
].to_numpy()

X_validation = validation[
    FEATURES
].copy()

y_validation_uncapped = validation[
    PRIMARY_TARGET
].to_numpy(dtype=float)

y_validation_capped = validation[
    SENSITIVITY_TARGET
].to_numpy(dtype=float)


validation_cap_formula_correct = bool(
    np.array_equal(
        y_validation_capped,
        np.minimum(
            y_validation_uncapped,
            CAP_CYCLES,
        ),
    )
)


if not validation_cap_formula_correct:
    raise AssertionError(
        "Validation capped target does not equal "
        "min(rul_uncapped, 130)."
    )


# Frozen primary-model predictions.
primary_raw_prediction = primary_bundle[
    "model"
].predict(
    X_validation
)

primary_prediction = np.clip(
    primary_raw_prediction,
    LOWER_BOUND,
    None,
)


# Capped sensitivity-model predictions.
capped_raw_prediction = capped_model.predict(
    X_validation
)

capped_prediction = np.clip(
    capped_raw_prediction,
    LOWER_BOUND,
    CAP_CYCLES,
)


# For a like-for-like capped-target comparison,
# transform primary predictions onto the capped scale.
primary_prediction_on_capped_scale = np.minimum(
    primary_prediction,
    CAP_CYCLES,
)


# ==============================================================================
# 7. COMPARISON A — SCORE BOTH APPROACHES AGAINST THE CAPPED TARGET
# ==============================================================================

capped_model_capped_record, _ = model_metrics(
    "capped130_model_on_capped_target",
    validation_units,
    y_validation_capped,
    capped_prediction,
)


primary_model_capped_record, _ = model_metrics(
    "uncapped_model_on_capped_target",
    validation_units,
    y_validation_capped,
    primary_prediction_on_capped_scale,
)


comparison_capped_scale = pd.DataFrame(
    [
        capped_model_capped_record,
        primary_model_capped_record,
    ]
)


# ==============================================================================
# 8. COMPARISON B — SCORE BOTH APPROACHES AGAINST ACTUAL UNCAPPED RUL
# ==============================================================================

capped_model_actual_record, _ = model_metrics(
    "capped130_model_on_actual_rul",
    validation_units,
    y_validation_uncapped,
    capped_prediction,
)


primary_model_actual_record, _ = model_metrics(
    "uncapped_primary_model_on_actual_rul",
    validation_units,
    y_validation_uncapped,
    primary_prediction,
)


comparison_actual_scale = pd.DataFrame(
    [
        capped_model_actual_record,
        primary_model_actual_record,
    ]
)


# ==============================================================================
# 9. LIFECYCLE-STAGE ANALYSIS
# ==============================================================================

lifecycle_stage = pd.cut(
    y_validation_uncapped,
    bins=[
        -0.1,
        30,
        60,
        130,
        np.inf,
    ],
    labels=[
        "critical_0_to_30",
        "near_term_31_to_60",
        "mid_life_61_to_130",
        "early_life_above_130",
    ],
)


stage_rows = []


for stage_name in lifecycle_stage.categories:
    mask = np.asarray(
        lifecycle_stage
        == stage_name
    )

    if int(
        mask.sum()
    ) == 0:
        continue

    stage_rows.append(
        {
            "lifecycle_stage":
                str(stage_name),

            "row_count":
                int(
                    mask.sum()
                ),

            "engine_count":
                int(
                    validation.loc[
                        mask,
                        UNIT,
                    ].nunique()
                ),

            "actual_uncapped_mean":
                float(
                    np.mean(
                        y_validation_uncapped[
                            mask
                        ]
                    )
                ),

            "primary_model_mae":
                float(
                    mean_absolute_error(
                        y_validation_uncapped[
                            mask
                        ],
                        primary_prediction[
                            mask
                        ],
                    )
                ),

            "capped_model_mae":
                float(
                    mean_absolute_error(
                        y_validation_uncapped[
                            mask
                        ],
                        capped_prediction[
                            mask
                        ],
                    )
                ),

            "primary_model_rmse":
                float(
                    root_mean_squared_error(
                        y_validation_uncapped[
                            mask
                        ],
                        primary_prediction[
                            mask
                        ],
                    )
                ),

            "capped_model_rmse":
                float(
                    root_mean_squared_error(
                        y_validation_uncapped[
                            mask
                        ],
                        capped_prediction[
                            mask
                        ],
                    )
                ),

            "primary_model_bias":
                float(
                    np.mean(
                        primary_prediction[
                            mask
                        ]
                        - y_validation_uncapped[
                            mask
                        ]
                    )
                ),

            "capped_model_bias":
                float(
                    np.mean(
                        capped_prediction[
                            mask
                        ]
                        - y_validation_uncapped[
                            mask
                        ]
                    )
                ),
        }
    )


stage_table = pd.DataFrame(
    stage_rows
)


# ==============================================================================
# 10. SAVE SENSITIVITY ARTIFACTS
# ==============================================================================

sensitivity_bundle = {
    "model_name":
        "Random Forest capped-130 sensitivity model",

    "target":
        SENSITIVITY_TARGET,

    "cap_cycles":
        CAP_CYCLES,

    "role":
        "sensitivity_analysis_only",

    "primary_model_unchanged":
        True,

    "nasa_prescribed_cap":
        False,

    "cap_claimed_optimal":
        False,

    "feature_order":
        FEATURES,

    "input_scaling":
        "none",

    "hyperparameters_researched":
        False,

    "hyperparameters_reused_from":
        "random_forest_uncapped",

    "primary_model_sha256":
        EXPECTED_PRIMARY_MODEL_SHA256,

    "selected_hyperparameters":
        primary_bundle[
            "selected_hyperparameters"
        ],

    "prediction_lower_bound":
        LOWER_BOUND,

    "prediction_upper_bound":
        CAP_CYCLES,

    "model":
        capped_model,

    "fit_engine_count":
        int(
            len(
                training_engines
            )
        ),

    "fit_row_count":
        int(
            len(training)
        ),
}


joblib.dump(
    sensitivity_bundle,
    CAPPED_MODEL_PATH,
)


comparison_capped_scale.to_csv(
    CAPPED_SCALE_PATH,
    index=False,
)

comparison_actual_scale.to_csv(
    ACTUAL_SCALE_PATH,
    index=False,
)

stage_table.to_csv(
    STAGE_PATH,
    index=False,
)


prediction_output = pd.DataFrame(
    {
        "unit_number":
            validation[
                UNIT
            ].to_numpy(),

        "time_in_cycles":
            validation[
                CYCLE
            ].to_numpy(),

        "actual_rul_uncapped":
            y_validation_uncapped,

        "actual_rul_capped_130":
            y_validation_capped,

        "primary_uncapped_prediction":
            primary_prediction,

        "primary_prediction_capped_for_comparison":
            primary_prediction_on_capped_scale,

        "capped130_model_prediction":
            capped_prediction,

        "primary_uncapped_error":
            primary_prediction
            - y_validation_uncapped,

        "capped130_error_against_actual_rul":
            capped_prediction
            - y_validation_uncapped,

        "lifecycle_stage":
            lifecycle_stage.astype(str),
    }
)


prediction_output.to_csv(
    PREDICTION_PATH,
    index=False,
)


# ==============================================================================
# 11. RELOAD THE CAPPED MODEL
# ==============================================================================

reloaded_bundle = joblib.load(
    CAPPED_MODEL_PATH
)


reloaded_prediction = np.clip(
    reloaded_bundle[
        "model"
    ].predict(
        validation[
            reloaded_bundle[
                "feature_order"
            ]
        ]
    ),
    reloaded_bundle[
        "prediction_lower_bound"
    ],
    reloaded_bundle[
        "prediction_upper_bound"
    ],
)


reload_matches = bool(
    np.allclose(
        reloaded_prediction,
        capped_prediction,
        rtol=1e-12,
        atol=1e-12,
    )
)


# ==============================================================================
# 12. SAVE FORMAL SENSITIVITY RECORD
# ==============================================================================

sensitivity_record = {
    "recorded_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "phase":
        4,

    "step":
        "6",

    "status":
        "capped130_sensitivity_completed",

    "role":
        "sensitivity_analysis_only",

    "primary_model":
        "random_forest_uncapped",

    "primary_model_sha256":
        EXPECTED_PRIMARY_MODEL_SHA256,

    "primary_model_remains_frozen":
        True,

    "sensitivity_model":
        "random_forest_capped130",

    "sensitivity_target":
        SENSITIVITY_TARGET,

    "cap_cycles":
        CAP_CYCLES,

    "nasa_prescribed_cap":
        False,

    "cap_claimed_optimal":
        False,

    "hyperparameters_researched":
        False,

    "model_family_changed":
        False,

    "training_engine_count":
        int(
            len(
                training_engines
            )
        ),

    "validation_engine_count":
        int(
            len(
                validation_engines
            )
        ),

    "training_rows":
        int(
            len(training)
        ),

    "validation_rows":
        int(
            len(validation)
        ),

    "training_rows_changed_by_cap":
        int(
            np.sum(
                y_train_uncapped
                > CAP_CYCLES
            )
        ),

    "validation_rows_changed_by_cap":
        int(
            np.sum(
                y_validation_uncapped
                > CAP_CYCLES
            )
        ),

    "capped_bundle_sha256":
        file_hash(
            CAPPED_MODEL_PATH
        ),

    "calibration_values_used":
        False,

    "official_test_data_used":
        False,

    "test_rul_values_accessed":
        False,

    "final_model_reselected":
        False,

    "primary_research_result_changed":
        False,

    "next_permitted_step":
        "Phase 4 completion and evidence consolidation",
}


with SENSITIVITY_RECORD_PATH.open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        sensitivity_record,
        handle,
        indent=2,
    )


# ==============================================================================
# 13. VALIDATION
# ==============================================================================

primary_hash_after = file_hash(
    PRIMARY_MODEL_PATH
)


checks = {
    "final registry selects Random Forest":
        final_registry[
            "selected_model"
        ]
        == "random_forest_uncapped",

    "primary model hash is unchanged":
        primary_hash_before
        == primary_hash_after
        == EXPECTED_PRIMARY_MODEL_SHA256,

    "training contains 60 engines":
        len(
            training_engines
        ) == 60,

    "validation contains 20 engines":
        len(
            validation_engines
        ) == 20,

    "training and validation are engine disjoint":
        training_engines.isdisjoint(
            validation_engines
        ),

    "eighteen predictors were preserved":
        len(FEATURES) == 18,

    "unit number is excluded":
        UNIT not in FEATURES,

    "primary target is excluded":
        PRIMARY_TARGET
        not in FEATURES,

    "sensitivity target is excluded":
        SENSITIVITY_TARGET
        not in FEATURES,

    "training capped formula is correct":
        training_cap_formula_correct,

    "validation capped formula is correct":
        validation_cap_formula_correct,

    "capped labels are between zero and 130":
        bool(
            (
                y_train_capped >= 0
            ).all()
            and (
                y_train_capped
                <= CAP_CYCLES
            ).all()
        ),

    "hyperparameters were not researched":
        True,

    "capped model parameters match primary architecture":
        capped_model.get_params()
        == primary_bundle[
            "model"
        ].get_params(),

    "capped predictions are finite":
        bool(
            np.isfinite(
                capped_prediction
            ).all()
        ),

    "capped predictions are non-negative":
        bool(
            (
                capped_prediction >= 0
            ).all()
        ),

    "capped predictions do not exceed 130":
        bool(
            (
                capped_prediction
                <= CAP_CYCLES
            ).all()
        ),

    "validation row count was preserved":
        len(
            capped_prediction
        )
        == len(
            validation
        ),

    "both comparisons contain two models":
        (
            len(
                comparison_capped_scale
            ) == 2
            and len(
                comparison_actual_scale
            ) == 2
        ),

    "four lifecycle stages were evaluated":
        len(
            stage_table
        ) == 4,

    "saved capped model reproduces predictions":
        reload_matches,

    "training dataframe was not modified":
        training_hash_before
        == frame_hash(
            training
        ),

    "validation dataframe was not modified":
        validation_hash_before
        == frame_hash(
            validation
        ),

    "calibration file was not modified":
        calibration_hash_before
        == file_hash(
            CALIBRATION_PATH
        ),

    "all output files exist":
        all(
            path.exists()
            for path in [
                CAPPED_MODEL_PATH,
                CAPPED_SCALE_PATH,
                ACTUAL_SCALE_PATH,
                STAGE_PATH,
                PREDICTION_PATH,
                SENSITIVITY_RECORD_PATH,
            ]
        ),
}


print("\nCAPPED-130 SENSITIVITY VALIDATION")
print("=" * 90)

failed_checks = []


for check_name, passed in checks.items():
    print(
        f"{'PASS' if passed else 'FAIL'} — "
        f"{check_name}"
    )

    if not passed:
        failed_checks.append(
            check_name
        )


if failed_checks:
    raise AssertionError(
        "Capped-130 sensitivity checks failed:\n"
        + "\n".join(
            failed_checks
        )
    )


# ==============================================================================
# 14. EVIDENCE PACKAGE
# ==============================================================================

evidence_paths = [
    FINAL_REGISTRY_PATH,
    CAPPED_MODEL_PATH,
    CAPPED_SCALE_PATH,
    ACTUAL_SCALE_PATH,
    STAGE_PATH,
    PREDICTION_PATH,
    SENSITIVITY_RECORD_PATH,
]


manifest = pd.DataFrame(
    [
        {
            "file_name":
                path.name,

            "project_relative_path":
                str(
                    path.relative_to(
                        ROOT
                    )
                ),

            "size_bytes":
                path.stat().st_size,

            "sha256":
                file_hash(
                    path
                ),
        }

        for path in evidence_paths
    ]
)


manifest.to_csv(
    MANIFEST_PATH,
    index=False,
)


with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    zipfile.ZIP_DEFLATED,
) as archive:
    for path in (
        evidence_paths
        + [MANIFEST_PATH]
    ):
        archive.write(
            path,
            str(
                path.relative_to(
                    ROOT
                )
            ),
        )


zip_sha256 = file_hash(
    ZIP_PATH
)


# ==============================================================================
# 15. RESULTS
# ==============================================================================

print("\nCAPPED-130 SENSITIVITY DEFINITION")
print("=" * 90)

print(
    "Primary model: "
    "random_forest_uncapped"
)

print(
    "Sensitivity model: "
    "Random Forest refitted on rul_capped_130"
)

print(
    f"Sensitivity cap: "
    f"{CAP_CYCLES} cycles"
)

print(
    "NASA-prescribed cap: No"
)

print(
    "Cap claimed optimal: No"
)

print(
    "Hyperparameters researched: No"
)

print(
    "Primary model changed: No"
)

print(
    "Training rows changed by cap: "
    f"{int(np.sum(y_train_uncapped > CAP_CYCLES))} "
    f"of {len(training)}"
)


print(
    "\nCOMPARISON A — CAPPED-TARGET SCALE"
)

print("=" * 90)

display(
    comparison_capped_scale.round(6)
)


print(
    "\nCOMPARISON B — ACTUAL UNCAPPED-RUL SCALE"
)

print("=" * 90)

display(
    comparison_actual_scale.round(6)
)


print(
    "\nERROR BY LIFECYCLE STAGE"
)

print("=" * 90)

display(
    stage_table.round(6)
)


print("\nINTERPRETATION CONTROL")
print("=" * 90)

print(
    "Comparison A measures performance on the "
    "artificial capped target."
)

print(
    "Comparison B measures performance against "
    "actual remaining cycles."
)

print(
    "The capped-target result must not replace "
    "the primary uncapped result."
)

print(
    "The capped model is sensitivity analysis only."
)


print("\nRESEARCH CONTROLS")
print("=" * 90)

print(
    "Primary Random Forest retrained or modified: No"
)

print(
    "Hyperparameters researched on capped target: No"
)

print(
    "Calibration values used: No"
)

print(
    "Official test data used: No"
)

print(
    "Test RUL values accessed: No"
)

print(
    "Final model reselected: No"
)


print("\nOUTPUT FILES")
print("=" * 90)

print(
    f"Capped model bundle: "
    f"{CAPPED_MODEL_PATH}"
)

print(
    f"Capped-scale comparison: "
    f"{CAPPED_SCALE_PATH}"
)

print(
    f"Actual-scale comparison: "
    f"{ACTUAL_SCALE_PATH}"
)

print(
    f"Lifecycle-stage analysis: "
    f"{STAGE_PATH}"
)

print(
    f"Validation predictions: "
    f"{PREDICTION_PATH}"
)

print(
    f"Sensitivity record: "
    f"{SENSITIVITY_RECORD_PATH}"
)

print(
    f"Evidence ZIP: "
    f"{ZIP_PATH}"
)

print(
    f"Capped bundle SHA-256: "
    f"{file_hash(CAPPED_MODEL_PATH)}"
)

print(
    f"Evidence ZIP SHA-256: "
    f"{zip_sha256}"
)


print("\nFINAL CHECK")
print("=" * 90)

print(
    "PASS — Capped-130 sensitivity analysis "
    "completed successfully."
)

print(
    "PASS — The frozen Random Forest architecture "
    "was reused without hyperparameter searching."
)

print(
    "PASS — Results were reported on both capped "
    "and actual RUL scales."
)

print(
    "PASS — The uncapped Random Forest remains "
    "the primary frozen model."
)

print(
    "PASS — Calibration and official test data "
    "remain sealed."
)

print(
    "AWAITING APPROVAL — Phase 4 completion "
    "packaging has not started."
)


# ==============================================================================
# 16. DOWNLOAD EVIDENCE
# ==============================================================================

try:
    from google.colab import files

    print(
        "\nDownloading the Step 6 "
        "capped-130 sensitivity evidence ZIP..."
    )

    files.download(
        str(
            ZIP_PATH
        )
    )

except Exception as error:
    print(
        f"Automatic download unavailable: "
        f"{error}"
    )

    print(
        f"Download manually from: "
        f"{ZIP_PATH}"
    )

In [ ]:
# =============================================================================
# PHASE 4 — COMPLETION, GOVERNANCE AND REPRODUCIBILITY PACKAGE
# CLEANED AND HASH-ALIGNED VERSION
#
# IMPORTANT:
# - This cell does not train, refit, modify, or resave any model.
# - The selected Random Forest remains strictly hash-locked.
# - Comparator hashes are governed by one evidence registry.
# - Existing hash evidence is never silently overwritten.
# - No browser download is triggered.
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import sys
import zipfile

import joblib
import numpy as np
import pandas as pd
import sklearn
import xgboost


# =============================================================================
# 1. PROJECT PATHS
# =============================================================================

ROOT = Path(
    "/content/Predictive_Maintenance_Thesis"
)

RESULTS = ROOT / "results"
LOGS = ROOT / "logs"
MODELS = ROOT / "models" / "regression"

RESULTS.mkdir(
    parents=True,
    exist_ok=True,
)

LOGS.mkdir(
    parents=True,
    exist_ok=True,
)


CALIBRATION_PATH = (
    ROOT
    / "data"
    / "interim"
    / "fd001_calibration_targets.csv"
)

FINAL_COMPARISON_PATH = (
    RESULTS
    / "phase4_final_model_comparison.csv"
)

FINAL_REGISTRY_PATH = (
    LOGS
    / "phase4_final_model_registry.json"
)

SELECTION_RECORD_PATH = (
    LOGS
    / "phase4_final_model_selection_record.json"
)

SENSITIVITY_RECORD_PATH = (
    LOGS
    / "phase4_capped130_sensitivity_record.json"
)

CAPPED_ACTUAL_COMPARISON_PATH = (
    RESULTS
    / "phase4_capped130_comparison_actual_scale.csv"
)

RF_CV_PATH = (
    RESULTS
    / "phase4_random_forest_training_grouped_cv_results.csv"
)

XGB_CV_PATH = (
    RESULTS
    / "phase4_xgboost_training_grouped_cv_results.csv"
)


RIDGE_BUNDLE_PATH = (
    MODELS
    / "fd001_ridge_uncapped_bundle.joblib"
)

RF_BUNDLE_PATH = (
    MODELS
    / "fd001_random_forest_uncapped_bundle.joblib"
)

XGB_BUNDLE_PATH = (
    MODELS
    / "fd001_xgboost_uncapped_bundle.joblib"
)

CAPPED_BUNDLE_PATH = (
    MODELS
    / (
        "fd001_random_forest_"
        "capped130_sensitivity_bundle.joblib"
    )
)


ENVIRONMENT_PATH = (
    LOGS
    / "phase4_environment_record.json"
)

COMPLETION_RECORD_PATH = (
    LOGS
    / "phase4_technical_completion_record.json"
)

MODEL_HASH_REGISTRY_PATH = (
    LOGS
    / "phase4_model_artifact_hash_registry.json"
)

COMPLETE_MANIFEST_PATH = (
    LOGS
    / "phase4_complete_evidence_manifest.csv"
)

ZIP_PATH = (
    ROOT
    / "Phase4_Complete_Modelling_Evidence.zip"
)

OFFICIAL_EVALUATION_RECORD_PATH = (
    LOGS
    / "phase6_official_test_evaluation_record.json"
)


# =============================================================================
# 2. LOCKED SELECTED MODEL
# =============================================================================

SELECTED_MODEL_NAME = (
    "random_forest_uncapped"
)

EXPECTED_RF_SHA256 = (
    "99620a1a161f62a716d4cc058497159146"
    "197e7d91bba713581c822af808e5d5"
)


MODEL_SPECIFICATIONS = {
    "ridge_uncapped": {
        "path": (
            RIDGE_BUNDLE_PATH
        ),

        "role": (
            "documented_comparator"
        ),

        "expected_class_token": (
            "ridge"
        ),
    },

    "random_forest_uncapped": {
        "path": (
            RF_BUNDLE_PATH
        ),

        "role": (
            "selected_primary_model"
        ),

        "expected_class_token": (
            "randomforest"
        ),
    },

    "xgboost_uncapped": {
        "path": (
            XGB_BUNDLE_PATH
        ),

        "role": (
            "documented_comparator"
        ),

        "expected_class_token": (
            "xgb"
        ),
    },

    (
        "random_forest_"
        "capped130_sensitivity"
    ): {
        "path": (
            CAPPED_BUNDLE_PATH
        ),

        "role": (
            "sensitivity_analysis_only"
        ),

        "expected_class_token": (
            "randomforest"
        ),
    },
}


REQUIRED_COMPARISON_MODELS = {
    "naive_age_only",
    "ridge_uncapped",
    "random_forest_uncapped",
    "xgboost_uncapped",
}


# =============================================================================
# 3. HELPERS
# =============================================================================

def utc_now():
    return (
        datetime.now(
            timezone.utc
        ).isoformat()
    )


def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        for chunk in iter(
            lambda: file_handle.read(
                chunk_size
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def load_json(
    path,
):
    with path.open(
        "r",
        encoding="utf-8",
    ) as file_handle:
        payload = json.load(
            file_handle
        )

    if not isinstance(
        payload,
        dict,
    ):
        raise TypeError(
            f"Expected a JSON object: {path}"
        )

    return payload


def write_json(
    path,
    payload,
):
    with path.open(
        "w",
        encoding="utf-8",
    ) as file_handle:
        json.dump(
            payload,
            file_handle,
            indent=2,
            ensure_ascii=False,
        )

        file_handle.write("\n")


def native_value(
    value,
):
    if value is None:
        return None

    if isinstance(
        value,
        np.integer,
    ):
        return int(value)

    if isinstance(
        value,
        np.floating,
    ):
        if np.isnan(value):
            return None

        return float(value)

    if isinstance(
        value,
        np.bool_,
    ):
        return bool(value)

    if (
        isinstance(
            value,
            float,
        )
        and np.isnan(value)
    ):
        return None

    return value


def strict_boolean_series(
    series,
    column_name,
):
    if pd.api.types.is_bool_dtype(
        series
    ):
        if series.isna().any():
            raise ValueError(
                f"{column_name} contains "
                "missing values."
            )

        return series.astype(bool)

    normalized = (
        series
        .astype("string")
        .str.strip()
        .str.lower()
    )

    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False,
    }

    invalid_values = sorted(
        set(
            normalized.dropna()
        )
        - set(mapping)
    )

    if (
        invalid_values
        or normalized.isna().any()
    ):
        raise ValueError(
            f"{column_name} is not a clean "
            "boolean column. Invalid values: "
            f"{invalid_values}"
        )

    return (
        normalized
        .map(mapping)
        .astype(bool)
    )


def extract_estimator(
    bundle,
    path,
):
    if isinstance(
        bundle,
        dict,
    ):
        estimator = bundle.get(
            "model"
        )
    else:
        estimator = bundle

    if (
        estimator is None
        or not callable(
            getattr(
                estimator,
                "predict",
                None,
            )
        )
    ):
        raise TypeError(
            "No usable prediction estimator "
            f"was found in {path}"
        )

    return estimator


def estimator_structure(
    estimator,
):
    class_names = [
        (
            f"{estimator.__class__.__module__}."
            f"{estimator.__class__.__name__}"
        )
    ]

    named_steps = getattr(
        estimator,
        "named_steps",
        None,
    )

    if isinstance(
        named_steps,
        dict,
    ):
        for step in named_steps.values():
            class_names.append(
                (
                    f"{step.__class__.__module__}."
                    f"{step.__class__.__name__}"
                )
            )

    return " | ".join(
        class_names
    )


def extract_feature_order(
    bundle,
):
    if not isinstance(
        bundle,
        dict,
    ):
        return None

    possible_keys = [
        "feature_order",
        "feature_names",
        "features",
    ]

    for key in possible_keys:
        value = bundle.get(key)

        if value is not None:
            return list(value)

    return None


def hashes_from_existing_manifest(
    manifest_path,
):
    manifest = pd.read_csv(
        manifest_path
    )

    required_columns = {
        "file_name",
        "sha256",
    }

    if not required_columns.issubset(
        manifest.columns
    ):
        raise ValueError(
            "Existing Phase 4 manifest "
            "does not contain the required "
            "file_name and sha256 columns."
        )

    expected_hashes = {}

    for specification in (
        MODEL_SPECIFICATIONS.values()
    ):
        file_name = (
            specification[
                "path"
            ].name
        )

        rows = manifest.loc[
            manifest[
                "file_name"
            ]
            == file_name,
            "sha256",
        ]

        if len(rows) != 1:
            raise ValueError(
                "Existing Phase 4 manifest "
                "must contain exactly one "
                f"entry for {file_name}. "
                f"Found: {len(rows)}"
            )

        expected_hashes[
            file_name
        ] = str(
            rows.iloc[0]
        )

    return expected_hashes


def hashes_from_registry(
    registry_path,
):
    registry = load_json(
        registry_path
    )

    artifacts = registry.get(
        "artifacts"
    )

    if not isinstance(
        artifacts,
        list,
    ):
        raise ValueError(
            "The model artifact hash registry "
            "has an invalid structure."
        )

    expected_hashes = {}

    for artifact in artifacts:
        if not isinstance(
            artifact,
            dict,
        ):
            continue

        file_name = artifact.get(
            "file_name"
        )

        stored_hash = artifact.get(
            "sha256"
        )

        if (
            file_name is not None
            and stored_hash is not None
        ):
            if file_name in expected_hashes:
                raise ValueError(
                    "Duplicate model hash "
                    f"registry entry: {file_name}"
                )

            expected_hashes[
                str(file_name)
            ] = str(
                stored_hash
            )

    missing_entries = []

    for specification in (
        MODEL_SPECIFICATIONS.values()
    ):
        file_name = (
            specification[
                "path"
            ].name
        )

        if file_name not in expected_hashes:
            missing_entries.append(
                file_name
            )

    if missing_entries:
        raise ValueError(
            "Model artifact hash registry "
            "is incomplete:\n"
            + "\n".join(
                missing_entries
            )
        )

    return expected_hashes


def create_deterministic_zip(
    zip_path,
    source_paths,
    project_root,
):
    with zipfile.ZipFile(
        zip_path,
        "w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=9,
    ) as archive:

        ordered_paths = sorted(
            source_paths,
            key=lambda path: str(
                path.relative_to(
                    project_root
                )
            ),
        )

        for source_path in ordered_paths:
            archive_name = str(
                source_path.relative_to(
                    project_root
                )
            ).replace(
                "\\",
                "/",
            )

            zip_info = zipfile.ZipInfo(
                archive_name,
                date_time=(
                    1980,
                    1,
                    1,
                    0,
                    0,
                    0,
                ),
            )

            zip_info.compress_type = (
                zipfile.ZIP_DEFLATED
            )

            zip_info.external_attr = (
                0o100644
                << 16
            )

            archive.writestr(
                zip_info,
                source_path.read_bytes(),
                compress_type=(
                    zipfile.ZIP_DEFLATED
                ),
            )


# =============================================================================
# 4. VERIFY REQUIRED INPUT ARTIFACTS
# =============================================================================

required_paths = [
    CALIBRATION_PATH,
    FINAL_COMPARISON_PATH,
    FINAL_REGISTRY_PATH,
    SELECTION_RECORD_PATH,
    SENSITIVITY_RECORD_PATH,
    CAPPED_ACTUAL_COMPARISON_PATH,
    RF_CV_PATH,
    XGB_CV_PATH,
]

required_paths.extend(
    specification["path"]
    for specification
    in MODEL_SPECIFICATIONS.values()
)


missing_paths = [
    str(path)
    for path in required_paths
    if not path.is_file()
]


if missing_paths:
    raise FileNotFoundError(
        "Required Phase 4 artifacts "
        "are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


calibration_hash_before = (
    sha256_file(
        CALIBRATION_PATH
    )
)


# =============================================================================
# 5. LOAD GOVERNANCE RECORDS AND RESULT TABLES
# =============================================================================

final_registry = load_json(
    FINAL_REGISTRY_PATH
)

selection_record = load_json(
    SELECTION_RECORD_PATH
)

sensitivity_record = load_json(
    SENSITIVITY_RECORD_PATH
)


final_comparison = pd.read_csv(
    FINAL_COMPARISON_PATH
)

capped_actual_comparison = pd.read_csv(
    CAPPED_ACTUAL_COMPARISON_PATH
)

rf_cv = pd.read_csv(
    RF_CV_PATH
)

xgb_cv = pd.read_csv(
    XGB_CV_PATH
)


required_final_columns = {
    "model",
    "mean_engine_rmse",
    "mean_row_level_phm_penalty",
    "pooled_rmse",
    "within_practical_equivalence_band",
    "selected_final_model",
}


missing_final_columns = (
    required_final_columns
    - set(
        final_comparison.columns
    )
)


if missing_final_columns:
    raise ValueError(
        "Final model comparison is missing "
        "required columns:\n"
        + "\n".join(
            sorted(
                missing_final_columns
            )
        )
    )


selection_flags = strict_boolean_series(
    final_comparison[
        "selected_final_model"
    ],
    "selected_final_model",
)


selected_rows = final_comparison.loc[
    selection_flags
]


if len(selected_rows) != 1:
    raise AssertionError(
        "The final comparison must contain "
        "exactly one selected model. "
        f"Found: {len(selected_rows)}"
    )


selected_row = (
    selected_rows.iloc[0]
)


# =============================================================================
# 6. LOAD AND STRUCTURALLY VALIDATE MODEL BUNDLES
# =============================================================================

observed_hashes = {}
bundle_summaries = {}


for (
    model_name,
    specification,
) in MODEL_SPECIFICATIONS.items():

    bundle_path = (
        specification[
            "path"
        ]
    )

    bundle = joblib.load(
        bundle_path
    )

    estimator = extract_estimator(
        bundle,
        bundle_path,
    )

    structure = estimator_structure(
        estimator
    )

    expected_class_token = (
        specification[
            "expected_class_token"
        ]
    )

    if (
        expected_class_token
        not in structure.lower()
    ):
        raise TypeError(
            f"Unexpected estimator in "
            f"{bundle_path.name}.\n"
            f"Expected class containing: "
            f"{expected_class_token}\n"
            f"Observed estimator structure: "
            f"{structure}"
        )

    feature_order = (
        extract_feature_order(
            bundle
        )
    )

    bundle_hash = sha256_file(
        bundle_path
    )

    observed_hashes[
        bundle_path.name
    ] = bundle_hash

    bundle_summaries[
        model_name
    ] = {
        "model_name": (
            model_name
        ),

        "role": (
            specification[
                "role"
            ]
        ),

        "file_name": (
            bundle_path.name
        ),

        "project_relative_path": str(
            bundle_path.relative_to(
                ROOT
            )
        ),

        "size_bytes": (
            bundle_path.stat().st_size
        ),

        "sha256": (
            bundle_hash
        ),

        "estimator_structure": (
            structure
        ),

        "feature_count": (
            None
            if feature_order is None
            else len(
                feature_order
            )
        ),

        "feature_order": (
            feature_order
        ),
    }


# =============================================================================
# 7. STRICTLY VERIFY THE SELECTED RANDOM FOREST
# =============================================================================

registry_selected_model = (
    final_registry.get(
        "selected_model"
    )
)

registry_selected_hash = (
    final_registry.get(
        "selected_model_sha256"
    )
)

observed_rf_hash = (
    observed_hashes[
        RF_BUNDLE_PATH.name
    ]
)


selected_model_checks = {
    "final registry selects Random Forest": (
        registry_selected_model
        == SELECTED_MODEL_NAME
    ),

    "registry selected-model hash is a valid SHA-256": (
        isinstance(
            registry_selected_hash,
            str,
        )
        and len(
            registry_selected_hash
        )
        == 64
    ),

    "registry hash matches the later-phase lock": (
        registry_selected_hash
        == EXPECTED_RF_SHA256
    ),

    "Random Forest file matches the registry": (
        observed_rf_hash
        == registry_selected_hash
    ),
}


failed_selected_checks = [
    check_name
    for check_name, passed
    in selected_model_checks.items()
    if not passed
]


if failed_selected_checks:
    raise AssertionError(
        "Selected Random Forest integrity "
        "verification failed:\n"
        + "\n".join(
            failed_selected_checks
        )
        + (
            "\nRegistry SHA-256: "
            f"{registry_selected_hash}"
        )
        + (
            "\nObserved SHA-256: "
            f"{observed_rf_hash}"
        )
        + (
            "\nLocked SHA-256:   "
            f"{EXPECTED_RF_SHA256}"
        )
    )


# =============================================================================
# 8. RESOLVE THE AUTHORITATIVE COMPARATOR HASH SOURCE
# =============================================================================

if MODEL_HASH_REGISTRY_PATH.is_file():

    expected_hashes = (
        hashes_from_registry(
            MODEL_HASH_REGISTRY_PATH
        )
    )

    hash_authority = (
        "existing_phase4_"
        "model_artifact_hash_registry"
    )


elif COMPLETE_MANIFEST_PATH.is_file():

    expected_hashes = (
        hashes_from_existing_manifest(
            COMPLETE_MANIFEST_PATH
        )
    )

    hash_authority = (
        "existing_phase4_"
        "complete_evidence_manifest"
    )


else:

    if (
        OFFICIAL_EVALUATION_RECORD_PATH
        .exists()
    ):
        raise RuntimeError(
            "A new Phase 4 comparator-hash "
            "baseline cannot be created after "
            "official-test evaluation.\n"
            "Restore the original Phase 4 "
            "evidence package or create a "
            "transparent governance amendment."
        )

    expected_hashes = dict(
        observed_hashes
    )

    hash_authority = (
        "initialized_at_first_"
        "phase4_completion"
    )


# =============================================================================
# 9. VERIFY ALL MODEL ARTIFACT HASHES
# =============================================================================

bundle_hash_checks = {}

hash_mismatches = []


print(
    "PHASE 4 MODEL ARTIFACT INTEGRITY"
)

print("=" * 90)

print(
    f"Hash authority: "
    f"{hash_authority}"
)


for specification in (
    MODEL_SPECIFICATIONS.values()
):

    bundle_path = (
        specification[
            "path"
        ]
    )

    file_name = (
        bundle_path.name
    )

    expected_hash = (
        expected_hashes.get(
            file_name
        )
    )

    observed_hash = (
        observed_hashes[
            file_name
        ]
    )

    passed = (
        expected_hash
        == observed_hash
    )

    bundle_hash_checks[
        file_name
    ] = passed

    print(
        f"{'PASS' if passed else 'FAIL'} — "
        f"{file_name}"
    )

    print(
        f"  expected: "
        f"{expected_hash}"
    )

    print(
        f"  observed: "
        f"{observed_hash}"
    )

    if not passed:
        hash_mismatches.append(
            {
                "file_name": (
                    file_name
                ),

                "expected": (
                    expected_hash
                ),

                "observed": (
                    observed_hash
                ),
            }
        )


if hash_mismatches:

    mismatch_lines = []

    for mismatch in hash_mismatches:
        mismatch_lines.extend(
            [
                mismatch[
                    "file_name"
                ],

                (
                    "  expected: "
                    f"{mismatch['expected']}"
                ),

                (
                    "  observed: "
                    f"{mismatch['observed']}"
                ),
            ]
        )

    raise AssertionError(
        "Phase 4 model artifact "
        "hash mismatch:\n"
        + "\n".join(
            mismatch_lines
        )
    )


# =============================================================================
# 10. CREATE THE MODEL HASH REGISTRY ONCE
# =============================================================================

if not MODEL_HASH_REGISTRY_PATH.exists():

    model_hash_registry = {
        "phase": 4,

        "status": (
            "frozen_model_artifact_"
            "hash_registry"
        ),

        "created_at_utc": (
            utc_now()
        ),

        "authority_source": (
            hash_authority
        ),

        "selected_model": (
            SELECTED_MODEL_NAME
        ),

        "selected_model_sha256": (
            observed_rf_hash
        ),

        (
            "official_test_evaluation_"
            "existed_at_registry_creation"
        ): (
            OFFICIAL_EVALUATION_RECORD_PATH
            .exists()
        ),

        (
            "models_retrained_or_"
            "resaved_by_this_cell"
        ): False,

        "artifacts": [
            bundle_summaries[
                model_name
            ]
            for model_name
            in MODEL_SPECIFICATIONS
        ],
    }

    write_json(
        MODEL_HASH_REGISTRY_PATH,
        model_hash_registry,
    )


# =============================================================================
# 11. GOVERNANCE VALIDATION
# =============================================================================

governance_checks = {
    **selected_model_checks,

    "final comparison contains the four planned models": (
        set(
            final_comparison[
                "model"
            ]
        )
        == REQUIRED_COMPARISON_MODELS
    ),

    "exactly one final model is selected": (
        len(
            selected_rows
        )
        == 1
    ),

    "Random Forest is marked selected": (
        selected_row[
            "model"
        ]
        == SELECTED_MODEL_NAME
    ),

    "Random Forest search contains 18 candidates": (
        len(
            rf_cv
        )
        == 18
    ),

    "XGBoost search contains 18 candidates": (
        len(
            xgb_cv
        )
        == 18
    ),

    "selection record freezes the model": (
        selection_record.get(
            "final_model_selected"
        )
        is True
    ),

    "capped result is sensitivity only": (
        sensitivity_record.get(
            "role"
        )
        == "sensitivity_analysis_only"
    ),

    "capped sensitivity did not reselect the model": (
        sensitivity_record.get(
            "final_model_reselected"
        )
        is False
    ),

    "primary model remained frozen": (
        sensitivity_record.get(
            "primary_model_remains_frozen"
        )
        is True
    ),

    "all four retained bundle hashes match": (
        all(
            bundle_hash_checks.values()
        )
    ),

    "calibration input remained unchanged": (
        calibration_hash_before
        == sha256_file(
            CALIBRATION_PATH
        )
    ),
}


failed_governance_checks = [
    check_name
    for check_name, passed
    in governance_checks.items()
    if not passed
]


if failed_governance_checks:
    raise AssertionError(
        "Phase 4 governance checks failed:\n"
        + "\n".join(
            failed_governance_checks
        )
    )


# =============================================================================
# 12. CREATE OR RETAIN THE ENVIRONMENT RECORD
# =============================================================================

if not ENVIRONMENT_PATH.exists():

    environment_record = {
        "recorded_at_utc": (
            utc_now()
        ),

        "phase": 4,

        "python_version": (
            sys.version.split()[0]
        ),

        "operating_system": (
            platform.platform()
        ),

        "package_versions": {
            "numpy": (
                np.__version__
            ),

            "pandas": (
                pd.__version__
            ),

            "scikit_learn": (
                sklearn.__version__
            ),

            "xgboost": (
                xgboost.__version__
            ),

            "joblib": (
                joblib.__version__
            ),
        },

        "random_seed": 42,

        (
            "engine_grouped_"
            "cross_validation"
        ): True,

        "cross_validation_folds": 5,

        (
            "calibration_values_"
            "used_in_phase4"
        ): False,

        (
            "official_test_data_"
            "used_in_phase4"
        ): False,

        (
            "test_rul_values_"
            "accessed_in_phase4"
        ): False,
    }

    write_json(
        ENVIRONMENT_PATH,
        environment_record,
    )

else:

    environment_record = load_json(
        ENVIRONMENT_PATH
    )


# =============================================================================
# 13. CREATE OR RETAIN THE TECHNICAL COMPLETION RECORD
# =============================================================================

comparison_summary = []


for _, row in final_comparison.iterrows():

    comparison_summary.append(
        {
            column: native_value(
                value
            )
            for column, value
            in row.to_dict().items()
        }
    )


if not COMPLETION_RECORD_PATH.exists():

    completion_record = {
        "recorded_at_utc": (
            utc_now()
        ),

        "phase": (
            "Phase 4 — Baselines, predictive "
            "modelling and model selection"
        ),

        "technical_status": (
            "complete"
        ),

        "formal_approval_status": (
            "awaiting_Ranitabh_approval"
        ),

        "completed_steps": [
            "Naive age-only baseline",
            "Ridge Regression",
            "Random Forest",
            "XGBoost",
            (
                "Frozen-policy final "
                "model selection"
            ),
            (
                "Capped-130 "
                "sensitivity analysis"
            ),
        ],

        "models_compared": [
            "naive_age_only",
            "ridge_uncapped",
            "random_forest_uncapped",
            "xgboost_uncapped",
        ],

        "primary_target": (
            "rul_uncapped"
        ),

        "sensitivity_target": (
            "rul_capped_130"
        ),

        "sensitivity_cap_cycles": 130,

        "nasa_prescribed_cap": False,

        "selected_primary_model": (
            SELECTED_MODEL_NAME
        ),

        (
            "selected_primary_"
            "model_sha256"
        ): (
            observed_rf_hash
        ),

        (
            "selected_model_"
            "mean_engine_rmse"
        ): float(
            selected_row[
                "mean_engine_rmse"
            ]
        ),

        (
            "selected_model_"
            "pooled_rmse"
        ): float(
            selected_row[
                "pooled_rmse"
            ]
        ),

        (
            "selected_model_"
            "phm_diagnostic"
        ): float(
            selected_row[
                "mean_row_level_phm_penalty"
            ]
        ),

        "selection_policy": {
            "primary_metric": (
                "mean_engine_rmse"
            ),

            (
                "practical_"
                "equivalence_percent"
            ): 1.0,

            "first_tie_breaker": (
                "mean_row_level_phm_penalty"
            ),

            "second_tie_breaker": (
                "pooled_rmse"
            ),

            "third_tie_breaker": (
                "lower_model_complexity"
            ),
        },

        (
            "models_within_"
            "practical_equivalence"
        ): final_registry.get(
            (
                "models_within_"
                "practical_equivalence"
            )
        ),

        "selection_decision_trace": (
            final_registry.get(
                "decision_trace"
            )
        ),

        "model_comparison": (
            comparison_summary
        ),

        (
            "random_forest_"
            "candidate_count"
        ): int(
            len(
                rf_cv
            )
        ),

        (
            "xgboost_"
            "candidate_count"
        ): int(
            len(
                xgb_cv
            )
        ),

        "cross_validation_folds": 5,

        (
            "engine_grouped_"
            "cross_validation"
        ): True,

        "capped_sensitivity_role": (
            "sensitivity_analysis_only"
        ),

        (
            "primary_model_changed_by_"
            "sensitivity_analysis"
        ): False,

        (
            "calibration_values_"
            "used_in_phase4"
        ): False,

        (
            "official_test_data_"
            "used_in_phase4"
        ): False,

        (
            "test_rul_values_"
            "accessed_in_phase4"
        ): False,

        (
            "conformal_prediction_"
            "performed_in_phase4"
        ): False,

        (
            "shap_analysis_"
            "performed_in_phase4"
        ): False,

        (
            "business_thresholds_"
            "approved_in_phase4"
        ): False,

        (
            "n8n_workflow_operationally_"
            "validated_in_phase4"
        ): False,

        "next_permitted_phase": (
            "Phase 5 — engine-aware conformal "
            "uncertainty quantification"
        ),

        "model_hash_registry": (
            str(
                MODEL_HASH_REGISTRY_PATH
            )
        ),
    }

    write_json(
        COMPLETION_RECORD_PATH,
        completion_record,
    )

else:

    completion_record = load_json(
        COMPLETION_RECORD_PATH
    )


# =============================================================================
# 14. COLLECT THE COMPLETE PHASE 4 EVIDENCE INVENTORY
# =============================================================================

artifact_paths = []


artifact_paths.extend(
    sorted(
        RESULTS.glob(
            "phase4_*.csv"
        )
    )
)


artifact_paths.extend(
    sorted(
        LOGS.glob(
            "phase4_*.json"
        )
    )
)


artifact_paths.extend(
    path
    for path in sorted(
        LOGS.glob(
            "phase4_*.csv"
        )
    )
    if path
    != COMPLETE_MANIFEST_PATH
)


artifact_paths.extend(
    specification[
        "path"
    ]
    for specification
    in MODEL_SPECIFICATIONS.values()
)


unique_artifact_paths = []

seen_paths = set()


for path in artifact_paths:

    resolved_path = (
        path.resolve()
    )

    if resolved_path not in seen_paths:

        seen_paths.add(
            resolved_path
        )

        unique_artifact_paths.append(
            path
        )


artifact_paths = (
    unique_artifact_paths
)


manifest_rows = []


for path in artifact_paths:

    if path.suffix == ".joblib":
        category = "model"

    elif path.parent == LOGS:
        category = "governance_log"

    else:
        category = "result"

    manifest_rows.append(
        {
            "file_name": (
                path.name
            ),

            (
                "project_"
                "relative_path"
            ): str(
                path.relative_to(
                    ROOT
                )
            ),

            "artifact_category": (
                category
            ),

            "size_bytes": (
                path.stat().st_size
            ),

            "sha256": (
                sha256_file(
                    path
                )
            ),
        }
    )


manifest = pd.DataFrame(
    manifest_rows
)


manifest.to_csv(
    COMPLETE_MANIFEST_PATH,
    index=False,
    lineterminator="\n",
)


# =============================================================================
# 15. FINAL COMPLETION VALIDATION
# =============================================================================

model_manifest = manifest.loc[
    manifest[
        "artifact_category"
    ]
    == "model"
]


manifest_model_hashes = dict(
    zip(
        model_manifest[
            "file_name"
        ],
        model_manifest[
            "sha256"
        ],
    )
)


official_test_used_in_phase4 = (
    completion_record.get(
        (
            "official_test_data_"
            "used_in_phase4"
        ),

        completion_record.get(
            "official_test_data_used"
        ),
    )
)


completion_checks = {
    **governance_checks,

    "manifest model hashes match observed model bytes": (
        all(
            manifest_model_hashes.get(
                specification[
                    "path"
                ].name
            )
            == observed_hashes[
                specification[
                    "path"
                ].name
            ]
            for specification
            in MODEL_SPECIFICATIONS.values()
        )
    ),

    "environment record exists": (
        ENVIRONMENT_PATH.is_file()
    ),

    "completion record exists": (
        COMPLETION_RECORD_PATH.is_file()
    ),

    "model hash registry exists": (
        MODEL_HASH_REGISTRY_PATH.is_file()
    ),

    "manifest contains every collected artifact": (
        len(
            manifest
        )
        == len(
            artifact_paths
        )
    ),

    "every manifest hash is a valid SHA-256": (
        bool(
            manifest[
                "sha256"
            ]
            .astype(str)
            .str.fullmatch(
                r"[0-9a-f]{64}"
            )
            .all()
        )
    ),

    "calibration file remains unchanged": (
        calibration_hash_before
        == sha256_file(
            CALIBRATION_PATH
        )
    ),

    "official test was not used in Phase 4": (
        official_test_used_in_phase4
        is False
    ),
}


print(
    "\nPHASE 4 COMPLETION VALIDATION"
)

print("=" * 90)


failed_completion_checks = []


for (
    check_name,
    passed,
) in completion_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL'} — "
        f"{check_name}"
    )

    if not passed:
        failed_completion_checks.append(
            check_name
        )


if failed_completion_checks:
    raise AssertionError(
        "Phase 4 completion checks failed:\n"
        + "\n".join(
            failed_completion_checks
        )
    )


# =============================================================================
# 16. CREATE REPRODUCIBLE EVIDENCE ZIP
# =============================================================================

zip_source_paths = [
    *artifact_paths,
    COMPLETE_MANIFEST_PATH,
]


create_deterministic_zip(
    ZIP_PATH,
    zip_source_paths,
    ROOT,
)


zip_sha256 = sha256_file(
    ZIP_PATH
)


# =============================================================================
# 17. DISPLAY COMPLETION SUMMARY
# =============================================================================

print(
    "\nPHASE 4 COMPLETION SUMMARY"
)

print("=" * 90)


print(
    "Models compared: "
    "Naive, Ridge, Random Forest, XGBoost"
)

print(
    "Selected primary model: "
    f"{SELECTED_MODEL_NAME}"
)

print(
    "Primary target: "
    "rul_uncapped"
)

print(
    "Sensitivity target: "
    "rul_capped_130"
)

print(
    "Sensitivity role: "
    "sensitivity analysis only"
)

print(
    "Selected mean engine RMSE: "
    f"{selected_row['mean_engine_rmse']:.6f}"
)

print(
    "Selected pooled RMSE: "
    f"{selected_row['pooled_rmse']:.6f}"
)

print(
    "Selected PHM diagnostic: "
    f"{selected_row['mean_row_level_phm_penalty']:.6f}"
)

print(
    "Selected model SHA-256: "
    f"{observed_rf_hash}"
)


print(
    "\nMODEL COMPARISON"
)

print("=" * 90)


comparison_display = (
    final_comparison[
        [
            "model",
            "mean_engine_rmse",
            "mean_row_level_phm_penalty",
            "pooled_rmse",
            (
                "within_practical_"
                "equivalence_band"
            ),
            "selected_final_model",
        ]
    ]
    .round(6)
)


try:
    from IPython.display import display

    display(
        comparison_display
    )

except ImportError:

    print(
        comparison_display.to_string(
            index=False
        )
    )


print(
    "\nEVIDENCE INVENTORY"
)

print("=" * 90)


print(
    "Files fingerprinted: "
    f"{len(manifest)}"
)

print(
    "Model hash registry: "
    f"{MODEL_HASH_REGISTRY_PATH}"
)

print(
    "Manifest: "
    f"{COMPLETE_MANIFEST_PATH}"
)

print(
    "Complete ZIP retained in Colab: "
    f"{ZIP_PATH}"
)

print(
    "Complete ZIP SHA-256: "
    f"{zip_sha256}"
)

print(
    "Browser download triggered: No"
)


print(
    "\nRESEARCH CONTROLS"
)

print("=" * 90)


print(
    "Models retrained by this cell: No"
)

print(
    "Models refitted by this cell: No"
)

print(
    "Models resaved by this cell: No"
)

print(
    "Calibration values used in Phase 4: No"
)

print(
    "Official test data used in Phase 4: No"
)

print(
    "Test RUL values accessed in Phase 4: No"
)

print(
    "Conformal prediction performed in Phase 4: No"
)

print(
    "SHAP analysis performed in Phase 4: No"
)

print(
    "Business thresholds approved in Phase 4: No"
)

print(
    "n8n workflow operationally validated in Phase 4: No"
)


print(
    "\nFINAL CHECK"
)

print("=" * 90)


print(
    "PASS — Phase 4 technical requirements "
    "are complete."
)

print(
    "PASS — Random Forest remains the frozen "
    "primary uncapped-RUL model."
)

print(
    "PASS — Ridge and XGBoost remain "
    "documented comparators."
)

print(
    "PASS — Comparator artifacts are governed "
    "by one model-hash registry."
)

print(
    "PASS — Capped-130 remains sensitivity "
    "analysis only."
)

print(
    "PASS — Every retained Phase 4 evidence "
    "artifact was fingerprinted."
)

print(
    "PASS — This cell did not train, refit, "
    "modify or resave any model."
)

print(
    "PASS — No browser download was triggered."
)

print(
    "AWAITING APPROVAL — Phase 4 is not yet "
    "formally closed."
)

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import math
import zipfile


# ============================================================================
# PHASE 5 — STEP 1
# FREEZE THE ENGINE-BLOCK CONFORMAL-PREDICTION POLICY
#
# This cell:
# - Does not deserialize or analyse calibration target values.
# - Verifies the sealed calibration file only by path and SHA-256.
# - Does not use validation or official test data.
# - Does not retrain or modify the frozen Random Forest.
# - Creates evidence inside Colab without downloading it.
# ============================================================================

PROJECT_ROOT = Path("/content/Predictive_Maintenance_Thesis")
DATA_DIR = PROJECT_ROOT / "data" / "interim"
LOG_DIR = PROJECT_ROOT / "logs"
MODEL_DIR = PROJECT_ROOT / "models" / "regression"

LOG_DIR.mkdir(parents=True, exist_ok=True)

FINAL_REGISTRY_PATH = LOG_DIR / "phase4_final_model_registry.json"
FINAL_SELECTION_PATH = LOG_DIR / "phase4_final_model_selection_record.json"
COMPARISON_POLICY_PATH = LOG_DIR / "phase4_model_comparison_policy.json"
PARTITION_MANIFEST_PATH = DATA_DIR / "fd001_engine_partition_manifest.csv"
CALIBRATION_TARGETS_PATH = DATA_DIR / "fd001_calibration_targets.csv"
PRIMARY_MODEL_PATH = (
    MODEL_DIR / "fd001_random_forest_uncapped_bundle.joblib"
)

POLICY_PATH = LOG_DIR / "phase5_engine_block_conformal_policy.json"
DECISION_PATH = LOG_DIR / "phase5_conformal_method_decision.md"
EVIDENCE_MANIFEST_PATH = (
    LOG_DIR / "phase5_step1_conformal_policy_manifest.csv"
)
EVIDENCE_ZIP_PATH = (
    PROJECT_ROOT / "Phase5_Step1_Conformal_Policy_Evidence.zip"
)

EXPECTED_PRIMARY_MODEL_SHA256 = (
    "99620a1a161f62a716d4cc058497159146197e7d91bba713581c822af808e5d5"
)
EXPECTED_PRIMARY_MODEL = "random_forest_uncapped"
EXPECTED_PRIMARY_TARGET = "rul_uncapped"
EXPECTED_PRIMARY_METRIC = "mean_engine_rmse"
EXPECTED_CALIBRATION_ENGINES = 20

ALPHA = 0.10
CONFIDENCE_LEVEL = 1.0 - ALPHA
CALIBRATION_SCORE_COUNT = EXPECTED_CALIBRATION_ENGINES

FINITE_SAMPLE_RANK = math.ceil(
    (CALIBRATION_SCORE_COUNT + 1) * CONFIDENCE_LEVEL
)

FINITE_SAMPLE_COVERAGE_FRACTION = (
    FINITE_SAMPLE_RANK / (CALIBRATION_SCORE_COUNT + 1)
)


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    """Calculate SHA-256 without deserializing the file."""
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        for chunk in iter(
            lambda: file_handle.read(chunk_size),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file_handle:
        return json.load(file_handle)


def write_json(path: Path, payload: dict) -> None:
    with path.open("w", encoding="utf-8") as file_handle:
        json.dump(
            payload,
            file_handle,
            indent=2,
            ensure_ascii=False,
        )
        file_handle.write("\n")


def normalise_label(value: object) -> str:
    return (
        str(value)
        .strip()
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )


def count_calibration_engines_from_manifest(
    path: Path,
) -> tuple[int, str, str]:
    """
    Read partition-assignment metadata only.

    This function does not open the calibration target table.
    """
    with path.open(
        "r",
        encoding="utf-8-sig",
        newline="",
    ) as file_handle:
        reader = csv.DictReader(file_handle)
        fieldnames = reader.fieldnames or []
        rows = list(reader)

    if not fieldnames:
        raise ValueError(
            f"Partition manifest has no header: {path}"
        )

    partition_candidates = [
        "partition",
        "partition_name",
        "dataset_partition",
        "split",
        "assignment",
    ]

    engine_candidates = [
        "unit_number",
        "engine_id",
        "engine",
        "unit",
        "unit_id",
    ]

    field_lookup = {
        normalise_label(name): name
        for name in fieldnames
    }

    partition_column = next(
        (
            field_lookup[name]
            for name in partition_candidates
            if name in field_lookup
        ),
        None,
    )

    engine_column = next(
        (
            field_lookup[name]
            for name in engine_candidates
            if name in field_lookup
        ),
        None,
    )

    if partition_column is None or engine_column is None:
        raise ValueError(
            "Could not identify the partition and engine columns in "
            f"{path}. Found columns: {fieldnames}"
        )

    calibration_engine_ids = {
        str(row[engine_column]).strip()
        for row in rows
        if "calibration"
        in normalise_label(row[partition_column])
    }

    return (
        len(calibration_engine_ids),
        partition_column,
        engine_column,
    )


# ============================================================================
# 1. VERIFY THE APPROVED PHASE 4 FOUNDATION
# ============================================================================

required_paths = [
    FINAL_REGISTRY_PATH,
    FINAL_SELECTION_PATH,
    COMPARISON_POLICY_PATH,
    PARTITION_MANIFEST_PATH,
    CALIBRATION_TARGETS_PATH,
    PRIMARY_MODEL_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required approved artifacts are missing:\n"
        + "\n".join(missing_paths)
    )

primary_model_hash_before = sha256_file(
    PRIMARY_MODEL_PATH
)

calibration_file_hash_before = sha256_file(
    CALIBRATION_TARGETS_PATH
)

if (
    primary_model_hash_before
    != EXPECTED_PRIMARY_MODEL_SHA256
):
    raise RuntimeError(
        "The frozen Random Forest hash does not match "
        "the approved Phase 4 hash.\n"
        f"Expected: {EXPECTED_PRIMARY_MODEL_SHA256}\n"
        f"Observed: {primary_model_hash_before}"
    )

final_registry = load_json(FINAL_REGISTRY_PATH)
final_selection = load_json(FINAL_SELECTION_PATH)
comparison_policy = load_json(COMPARISON_POLICY_PATH)

registry_text = json.dumps(
    final_registry,
    sort_keys=True,
).lower()

selection_text = json.dumps(
    final_selection,
    sort_keys=True,
).lower()

comparison_policy_text = json.dumps(
    comparison_policy,
    sort_keys=True,
).lower()

if "random_forest" not in registry_text:
    raise RuntimeError(
        "The final registry does not identify Random Forest."
    )

if EXPECTED_PRIMARY_MODEL_SHA256 not in registry_text:
    raise RuntimeError(
        "The final registry does not preserve "
        "the approved Random Forest hash."
    )

if "random_forest" not in selection_text:
    raise RuntimeError(
        "The final model-selection record does not "
        "select Random Forest."
    )

if EXPECTED_PRIMARY_METRIC not in comparison_policy_text:
    raise RuntimeError(
        "The Phase 4 comparison policy does not "
        "preserve mean_engine_rmse."
    )

(
    calibration_engine_count,
    partition_column,
    engine_column,
) = count_calibration_engines_from_manifest(
    PARTITION_MANIFEST_PATH
)

if (
    calibration_engine_count
    != EXPECTED_CALIBRATION_ENGINES
):
    raise RuntimeError(
        "Unexpected number of frozen calibration engines. "
        f"Expected {EXPECTED_CALIBRATION_ENGINES}, "
        f"observed {calibration_engine_count}."
    )

if FINITE_SAMPLE_RANK != 19:
    raise RuntimeError(
        "Finite-sample rank should be 19, "
        f"observed {FINITE_SAMPLE_RANK}."
    )


# ============================================================================
# 2. FREEZE THE CONFORMAL METHOD
# ============================================================================

created_at_utc = datetime.now(
    timezone.utc
).isoformat()

conformal_policy = {
    "phase": 5,
    "step": "1_conformal_policy_freeze",
    "status": "frozen_before_calibration_values_accessed",
    "created_at_utc": created_at_utc,

    "primary_model": {
        "name": EXPECTED_PRIMARY_MODEL,
        "target": EXPECTED_PRIMARY_TARGET,
        "bundle_path": str(PRIMARY_MODEL_PATH),
        "bundle_sha256": primary_model_hash_before,
        "retraining_permitted": False,
        "modification_permitted": False,
    },

    "calibration_partition": {
        "assignment_unit": "engine",
        "engine_identifier": "unit_number",
        "expected_engine_count": (
            EXPECTED_CALIBRATION_ENGINES
        ),
        "observed_engine_count_from_partition_metadata": (
            calibration_engine_count
        ),
        "partition_manifest_partition_column": (
            partition_column
        ),
        "partition_manifest_engine_column": (
            engine_column
        ),
        "target_file_path": str(
            CALIBRATION_TARGETS_PATH
        ),
        "target_file_sha256_at_policy_freeze": (
            calibration_file_hash_before
        ),
        "target_values_deserialized_or_analysed": False,
    },

    "conformal_method": {
        "family": "split_conformal_regression",
        "exchangeability_unit": (
            "complete_engine_trajectory"
        ),
        "confidence_level": CONFIDENCE_LEVEL,
        "miscoverage_alpha": ALPHA,

        "row_nonconformity_score": (
            "absolute_residual"
        ),
        "row_nonconformity_formula": (
            "abs(actual_rul_uncapped "
            "- point_prediction)"
        ),

        "within_engine_aggregation": "maximum",
        "engine_score_formula": (
            "max_t abs(y_i,t - yhat_i,t)"
        ),

        "number_of_exchangeable_scores": (
            CALIBRATION_SCORE_COUNT
        ),

        "finite_sample_rank_formula": (
            "ceil((n_calibration_engines + 1) "
            "* (1 - alpha))"
        ),

        "finite_sample_rank_one_based": (
            FINITE_SAMPLE_RANK
        ),

        "quantile_implementation": (
            "sort engine scores ascending "
            "and select score[k - 1]"
        ),

        "interpolation_permitted": False,

        "nominal_finite_sample_fraction": (
            FINITE_SAMPLE_COVERAGE_FRACTION
        ),

        "interval_formula": {
            "lower": (
                "max(0, point_prediction - q_hat)"
            ),
            "upper": (
                "point_prediction + q_hat"
            ),
        },

        "lower_bound_clip": 0,
        "upper_bound_cap": None,
        "symmetric_before_lower_clipping": True,
    },

    "interpretation": {
        "coverage_unit": "engine_trajectory",

        "intended_statement": (
            "Under exchangeability of complete engines, "
            "the interval targets simultaneous coverage "
            "of the uncapped-RUL trajectory for a new engine."
        ),

        "endpoint_implication": (
            "Simultaneous trajectory coverage implies "
            "coverage at any single observed endpoint, "
            "although the interval may be conservative."
        ),

        "not_an_operational_safety_guarantee": True,

        "small_calibration_sample_caveat": (
            "Only 20 exchangeable engine scores are "
            "available, so the conformal quantile is "
            "coarse and the interval may be wide."
        ),
    },

    "locked_controls": {
        "method_shopping_after_calibration": False,

        "alternative_score_selection_after_calibration": (
            False
        ),

        "validation_used_to_choose_conformal_method": (
            False
        ),

        "calibration_values_used_to_choose_conformal_method": (
            False
        ),

        "official_test_data_used": False,
        "test_rul_values_accessed": False,
        "capped_130_target_used": False,
        "business_thresholds_used": False,
    },

    "next_authorised_step": (
        "Apply this frozen method once to the "
        "20 calibration engines and estimate q_hat."
    ),
}

write_json(
    POLICY_PATH,
    conformal_policy,
)


# ============================================================================
# 3. WRITE THE HUMAN-READABLE DECISION
# ============================================================================

decision_markdown = f"""# FD001 Phase 5 Conformal Method Decision

## Frozen decision

- Primary point model: `{EXPECTED_PRIMARY_MODEL}`
- Primary target: `{EXPECTED_PRIMARY_TARGET}`
- Confidence level: `{CONFIDENCE_LEVEL:.0%}`
- Calibration unit: complete engine trajectory
- Calibration engines: `{CALIBRATION_SCORE_COUNT}`
- Row score: absolute residual
- Engine score: maximum absolute residual across the complete trajectory
- Finite-sample rank: `{FINITE_SAMPLE_RANK}`th smallest of `{CALIBRATION_SCORE_COUNT}` engine scores
- Lower interval: `max(0, point prediction - q_hat)`
- Upper interval: `point prediction + q_hat`
- Upper cap: none

## Rationale

Rows from the same engine are temporally dependent and are therefore not treated
as independent calibration examples. Each engine contributes exactly one score.
The maximum residual makes the coverage target simultaneous over an engine
trajectory, subject to exchangeability of engines. This is intentionally
conservative.

## Interpretation limit

The conformal interval is an empirical uncertainty statement under the stated
exchangeability setup. It is not a causal statement or an operational safety
guarantee. With only 20 calibration engines, the quantile is coarse and may be
wide.

## Leakage controls

At the time this decision was frozen, calibration target values had not been
deserialized or analysed. Validation, official test data, test RUL values, the
capped-130 sensitivity target, and business thresholds were not used to select
this method.
"""

with DECISION_PATH.open(
    "w",
    encoding="utf-8",
) as file_handle:
    file_handle.write(decision_markdown)


# ============================================================================
# 4. RECHECK THE PROTECTED INPUTS
# ============================================================================

primary_model_hash_after = sha256_file(
    PRIMARY_MODEL_PATH
)

calibration_file_hash_after = sha256_file(
    CALIBRATION_TARGETS_PATH
)

checks = {
    "final registry identifies Random Forest": (
        "random_forest" in registry_text
    ),

    "final registry preserves approved model hash": (
        EXPECTED_PRIMARY_MODEL_SHA256 in registry_text
    ),

    "primary model hash matches approved Phase 4 hash": (
        primary_model_hash_after
        == EXPECTED_PRIMARY_MODEL_SHA256
    ),

    "primary model remained unchanged": (
        primary_model_hash_before
        == primary_model_hash_after
    ),

    "frozen primary metric is mean engine RMSE": (
        EXPECTED_PRIMARY_METRIC
        in comparison_policy_text
    ),

    "partition metadata contains 20 calibration engines": (
        calibration_engine_count
        == EXPECTED_CALIBRATION_ENGINES
    ),

    "confidence level is 90 percent": (
        CONFIDENCE_LEVEL == 0.90
    ),

    "miscoverage alpha is 0.10": (
        ALPHA == 0.10
    ),

    "calibration unit is complete engine trajectory": (
        conformal_policy[
            "conformal_method"
        ][
            "exchangeability_unit"
        ]
        == "complete_engine_trajectory"
    ),

    "row score is absolute residual": (
        conformal_policy[
            "conformal_method"
        ][
            "row_nonconformity_score"
        ]
        == "absolute_residual"
    ),

    "within engine aggregation is maximum": (
        conformal_policy[
            "conformal_method"
        ][
            "within_engine_aggregation"
        ]
        == "maximum"
    ),

    "finite sample rank is 19 of 20": (
        CALIBRATION_SCORE_COUNT == 20
        and FINITE_SAMPLE_RANK == 19
    ),

    "lower bound is clipped at zero": (
        conformal_policy[
            "conformal_method"
        ][
            "lower_bound_clip"
        ]
        == 0
    ),

    "upper bound remains uncapped": (
        conformal_policy[
            "conformal_method"
        ][
            "upper_bound_cap"
        ]
        is None
    ),

    "calibration file remained unchanged": (
        calibration_file_hash_before
        == calibration_file_hash_after
    ),

    "calibration target values were not deserialized": (
        conformal_policy[
            "calibration_partition"
        ][
            "target_values_deserialized_or_analysed"
        ]
        is False
    ),

    "validation was not used": (
        conformal_policy[
            "locked_controls"
        ][
            "validation_used_to_choose_conformal_method"
        ]
        is False
    ),

    "official test data were not used": (
        conformal_policy[
            "locked_controls"
        ][
            "official_test_data_used"
        ]
        is False
    ),

    "policy file exists": POLICY_PATH.is_file(),

    "human readable decision exists": (
        DECISION_PATH.is_file()
    ),
}

failed_checks = [
    check_name
    for check_name, passed in checks.items()
    if not passed
]

if failed_checks:
    raise AssertionError(
        "Phase 5 Step 1 validation failed:\n"
        + "\n".join(failed_checks)
    )


# ============================================================================
# 5. BUILD THE STEP 1 EVIDENCE MANIFEST
# ============================================================================

manifest_entries = [
    (
        "approved_input",
        FINAL_REGISTRY_PATH,
        True,
    ),
    (
        "approved_input",
        FINAL_SELECTION_PATH,
        True,
    ),
    (
        "approved_input",
        COMPARISON_POLICY_PATH,
        True,
    ),
    (
        "partition_metadata",
        PARTITION_MANIFEST_PATH,
        True,
    ),
    (
        "sealed_hash_reference_only",
        CALIBRATION_TARGETS_PATH,
        False,
    ),
    (
        "frozen_model_hash_reference_only",
        PRIMARY_MODEL_PATH,
        False,
    ),
    (
        "phase5_output",
        POLICY_PATH,
        True,
    ),
    (
        "phase5_output",
        DECISION_PATH,
        True,
    ),
]

manifest_rows = []

for role, path, include_in_zip in manifest_entries:
    manifest_rows.append(
        {
            "role": role,
            "path": str(path),
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
            "included_in_step_zip": include_in_zip,
        }
    )

with EVIDENCE_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
    newline="",
) as file_handle:
    writer = csv.DictWriter(
        file_handle,
        fieldnames=[
            "role",
            "path",
            "size_bytes",
            "sha256",
            "included_in_step_zip",
        ],
    )

    writer.writeheader()
    writer.writerows(manifest_rows)


# ============================================================================
# 6. CREATE THE STEP ZIP INSIDE COLAB
#    No browser download is triggered.
# ============================================================================

with zipfile.ZipFile(
    EVIDENCE_ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:

    for row in manifest_rows:
        if row["included_in_step_zip"]:
            source_path = Path(row["path"])

            archive.write(
                source_path,
                arcname=str(
                    source_path.relative_to(PROJECT_ROOT)
                ),
            )

    archive.write(
        EVIDENCE_MANIFEST_PATH,
        arcname=str(
            EVIDENCE_MANIFEST_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
    )

policy_sha256 = sha256_file(POLICY_PATH)
evidence_zip_sha256 = sha256_file(
    EVIDENCE_ZIP_PATH
)


# ============================================================================
# 7. BOUNDED REVIEW OUTPUT
# ============================================================================

print("PHASE 4 FOUNDATION CHECK")
print("=" * 90)
print(
    f"PASS — Selected point model: "
    f"{EXPECTED_PRIMARY_MODEL}"
)
print(
    f"PASS — Model SHA-256: "
    f"{primary_model_hash_after}"
)
print(
    f"PASS — Frozen primary metric: "
    f"{EXPECTED_PRIMARY_METRIC}"
)
print(
    "PASS — Phase 4 artifacts were not modified"
)

print("\nFROZEN CONFORMAL POLICY")
print("=" * 90)
print(
    f"Confidence level: "
    f"{CONFIDENCE_LEVEL:.0%}"
)
print(
    "Exchangeability unit: "
    "complete engine trajectory"
)
print(
    "Row nonconformity score: "
    "absolute residual"
)
print(
    "Within-engine aggregation: "
    "maximum absolute residual"
)
print(
    f"Calibration engine scores: "
    f"{CALIBRATION_SCORE_COUNT}"
)
print(
    "Finite-sample quantile rank: "
    f"{FINITE_SAMPLE_RANK}th smallest "
    f"of {CALIBRATION_SCORE_COUNT}"
)
print(
    "Finite-sample rank fraction: "
    f"{FINITE_SAMPLE_RANK}/"
    f"{CALIBRATION_SCORE_COUNT + 1} "
    f"= {FINITE_SAMPLE_COVERAGE_FRACTION:.4%}"
)
print(
    "Interval: "
    "[max(0, point prediction - q_hat), "
    "point prediction + q_hat]"
)
print("Upper bound cap: None")

print("\nPOLICY VALIDATION")
print("=" * 90)

for check_name, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"{status} — {check_name}")

print("\nRESEARCH CONTROLS")
print("=" * 90)
print(
    "Calibration target values "
    "deserialized or analysed: No"
)
print(
    "Validation used to choose "
    "conformal method: No"
)
print("Official test data used: No")
print("Test RUL values accessed: No")
print(
    "Capped-130 sensitivity target used: No"
)
print(
    "Random Forest retrained or modified: No"
)
print(
    "Method shopping after calibration "
    "permitted: No"
)

print("\nOUTPUT FILES")
print("=" * 90)
print(f"Policy: {POLICY_PATH}")
print(f"Decision record: {DECISION_PATH}")
print(
    f"Evidence manifest: "
    f"{EVIDENCE_MANIFEST_PATH}"
)
print(
    f"Evidence ZIP retained in Colab: "
    f"{EVIDENCE_ZIP_PATH}"
)
print(f"Policy SHA-256: {policy_sha256}")
print(
    f"Evidence ZIP SHA-256: "
    f"{evidence_zip_sha256}"
)
print("Browser download triggered: No")

print("\nFINAL CHECK")
print("=" * 90)
print(
    "PASS — Phase 5 conformal policy was "
    "frozen before calibration analysis."
)
print(
    "PASS — Twenty engines will contribute "
    "one exchangeable score each."
)
print(
    "PASS — The finite-sample rank was "
    "fixed at 19 of 20."
)
print(
    "PASS — The frozen Random Forest "
    "remained unchanged."
)
print(
    "PASS — Validation and official test "
    "data remained unused."
)
print(
    "AWAITING APPROVAL — Calibration "
    "residuals have not been computed."
)

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import math
import zipfile

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn


# =============================================================================
# PHASE 5 — STEP 2
# ENGINE-BLOCK CONFORMAL CALIBRATION
#
# Controls:
# - Opens only the frozen 20-engine calibration partition.
# - Uses the unchanged Phase 4 Random Forest.
# - Gives each engine one maximum-absolute-residual score.
# - Selects the 19th smallest score without interpolation.
# - Does not load validation or official test data.
# - Does not trigger a browser download.
# =============================================================================

PROJECT_ROOT = Path(
    "/content/Predictive_Maintenance_Thesis"
)

DATA_DIR = PROJECT_ROOT / "data" / "interim"
LOG_DIR = PROJECT_ROOT / "logs"
RESULT_DIR = PROJECT_ROOT / "results"
FIGURE_DIR = RESULT_DIR / "figures"

REGRESSION_MODEL_DIR = (
    PROJECT_ROOT / "models" / "regression"
)

UNCERTAINTY_MODEL_DIR = (
    PROJECT_ROOT / "models" / "uncertainty"
)

for directory in [
    LOG_DIR,
    RESULT_DIR,
    FIGURE_DIR,
    UNCERTAINTY_MODEL_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# =============================================================================
# FILE PATHS
# =============================================================================

POLICY_PATH = (
    LOG_DIR
    / "phase5_engine_block_conformal_policy.json"
)

FINAL_REGISTRY_PATH = (
    LOG_DIR
    / "phase4_final_model_registry.json"
)

PARTITION_MANIFEST_PATH = (
    DATA_DIR
    / "fd001_engine_partition_manifest.csv"
)

CALIBRATION_PATH = (
    DATA_DIR
    / "fd001_calibration_targets.csv"
)

PRIMARY_MODEL_PATH = (
    REGRESSION_MODEL_DIR
    / "fd001_random_forest_uncapped_bundle.joblib"
)

ROW_OUTPUT_PATH = (
    RESULT_DIR
    / "phase5_conformal_calibration_row_predictions.csv"
)

ENGINE_SCORE_PATH = (
    RESULT_DIR
    / "phase5_engine_block_calibration_scores.csv"
)

SUMMARY_PATH = (
    RESULT_DIR
    / "phase5_conformal_calibration_summary.csv"
)

FIGURE_PATH = (
    FIGURE_DIR
    / "phase5_engine_block_calibration_scores.png"
)

CALIBRATION_RECORD_PATH = (
    LOG_DIR
    / "phase5_conformal_calibration_record.json"
)

CONFORMAL_BUNDLE_PATH = (
    UNCERTAINTY_MODEL_DIR
    / (
        "fd001_random_forest_"
        "engine_block_conformal_bundle.joblib"
    )
)

EVIDENCE_MANIFEST_PATH = (
    LOG_DIR
    / "phase5_step2_conformal_calibration_manifest.csv"
)

EVIDENCE_ZIP_PATH = (
    PROJECT_ROOT
    / "Phase5_Step2_Conformal_Calibration_Evidence.zip"
)


# =============================================================================
# LOCKED EXPECTATIONS
# =============================================================================

EXPECTED_MODEL_SHA256 = (
    "99620a1a161f62a716d4cc058497159146197e7d91bba713581c822af808e5d5"
)

EXPECTED_MODEL_NAME = "random_forest_uncapped"
EXPECTED_TARGET = "rul_uncapped"
EXPECTED_UNIT_COLUMN = "unit_number"
EXPECTED_CYCLE_COLUMN = "time_in_cycles"

EXPECTED_FEATURE_COUNT = 18
EXPECTED_CALIBRATION_ENGINES = 20
EXPECTED_CALIBRATION_ROWS = 4104

EXPECTED_CONFIDENCE = 0.90
EXPECTED_ALPHA = 0.10
EXPECTED_RANK = 19

NUMERIC_TOLERANCE = 1e-10


# =============================================================================
# SUPPORT FUNCTIONS
# =============================================================================

def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        for chunk in iter(
            lambda: file_handle.read(chunk_size),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def dataframe_content_hash(
    dataframe: pd.DataFrame,
) -> str:
    """
    Hash dataframe values, index, columns and dtypes.

    Used only to prove that the loaded source dataframe
    was not modified.
    """
    digest = hashlib.sha256()

    digest.update(
        pd.util.hash_pandas_object(
            dataframe,
            index=True,
        ).values.tobytes()
    )

    digest.update(
        "|".join(
            map(str, dataframe.columns)
        ).encode("utf-8")
    )

    digest.update(
        "|".join(
            map(str, dataframe.dtypes)
        ).encode("utf-8")
    )

    return digest.hexdigest()


def load_json(path: Path) -> dict:
    with path.open(
        "r",
        encoding="utf-8",
    ) as file_handle:
        return json.load(file_handle)


def write_json(
    path: Path,
    payload: dict,
) -> None:
    with path.open(
        "w",
        encoding="utf-8",
    ) as file_handle:
        json.dump(
            payload,
            file_handle,
            indent=2,
            ensure_ascii=False,
        )
        file_handle.write("\n")


def normalise_label(
    value: object,
) -> str:
    return (
        str(value)
        .strip()
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )


def recover_calibration_engine_ids(
    path: Path,
) -> set[int]:
    """
    Recover calibration-engine assignments from metadata.
    """
    manifest = pd.read_csv(path)

    column_lookup = {
        normalise_label(column): column
        for column in manifest.columns
    }

    partition_column = next(
        (
            column_lookup[name]
            for name in [
                "partition",
                "partition_name",
                "dataset_partition",
                "split",
                "assignment",
            ]
            if name in column_lookup
        ),
        None,
    )

    engine_column = next(
        (
            column_lookup[name]
            for name in [
                "unit_number",
                "engine_id",
                "engine",
                "unit",
                "unit_id",
            ]
            if name in column_lookup
        ),
        None,
    )

    if (
        partition_column is None
        or engine_column is None
    ):
        raise ValueError(
            "Could not recover partition metadata columns. "
            f"Found: {list(manifest.columns)}"
        )

    calibration_mask = (
        manifest[partition_column]
        .map(normalise_label)
        .str.contains(
            "calibration",
            regex=False,
        )
    )

    return set(
        manifest.loc[
            calibration_mask,
            engine_column,
        ]
        .astype(int)
        .tolist()
    )


# =============================================================================
# 1. VERIFY FROZEN POLICY AND FOUNDATION
# =============================================================================

required_paths = [
    POLICY_PATH,
    FINAL_REGISTRY_PATH,
    PARTITION_MANIFEST_PATH,
    CALIBRATION_PATH,
    PRIMARY_MODEL_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required approved artifacts are missing:\n"
        + "\n".join(missing_paths)
    )

policy_hash_before = sha256_file(
    POLICY_PATH
)

model_hash_before = sha256_file(
    PRIMARY_MODEL_PATH
)

calibration_file_hash_before = sha256_file(
    CALIBRATION_PATH
)

if model_hash_before != EXPECTED_MODEL_SHA256:
    raise RuntimeError(
        "The frozen Random Forest hash changed.\n"
        f"Expected: {EXPECTED_MODEL_SHA256}\n"
        f"Observed: {model_hash_before}"
    )

policy = load_json(POLICY_PATH)
registry = load_json(FINAL_REGISTRY_PATH)

registry_text = json.dumps(
    registry,
    sort_keys=True,
).lower()

policy_model = policy["primary_model"]
policy_calibration = policy[
    "calibration_partition"
]
policy_method = policy["conformal_method"]
policy_controls = policy["locked_controls"]

policy_checks = {
    "policy is frozen before calibration access": (
        policy["status"]
        == "frozen_before_calibration_values_accessed"
    ),

    "policy selects frozen Random Forest": (
        policy_model["name"]
        == EXPECTED_MODEL_NAME
    ),

    "policy uses uncapped target": (
        policy_model["target"]
        == EXPECTED_TARGET
    ),

    "policy preserves approved model hash": (
        policy_model["bundle_sha256"]
        == EXPECTED_MODEL_SHA256
    ),

    "registry preserves Random Forest": (
        "random_forest" in registry_text
    ),

    "registry preserves approved model hash": (
        EXPECTED_MODEL_SHA256 in registry_text
    ),

    "calibration file matches policy freeze hash": (
        policy_calibration[
            "target_file_sha256_at_policy_freeze"
        ]
        == calibration_file_hash_before
    ),

    "policy expects 20 calibration engines": (
        int(
            policy_calibration[
                "expected_engine_count"
            ]
        )
        == EXPECTED_CALIBRATION_ENGINES
    ),

    "policy confidence is 90 percent": (
        math.isclose(
            float(
                policy_method[
                    "confidence_level"
                ]
            ),
            EXPECTED_CONFIDENCE,
            abs_tol=1e-15,
        )
    ),

    "policy alpha is 0.10": (
        math.isclose(
            float(
                policy_method[
                    "miscoverage_alpha"
                ]
            ),
            EXPECTED_ALPHA,
            abs_tol=1e-15,
        )
    ),

    "policy score is absolute residual": (
        policy_method[
            "row_nonconformity_score"
        ]
        == "absolute_residual"
    ),

    "policy aggregation is engine maximum": (
        policy_method[
            "within_engine_aggregation"
        ]
        == "maximum"
    ),

    "policy rank is 19": (
        int(
            policy_method[
                "finite_sample_rank_one_based"
            ]
        )
        == EXPECTED_RANK
    ),

    "policy forbids interpolation": (
        policy_method[
            "interpolation_permitted"
        ]
        is False
    ),

    "policy lower clip is zero": (
        policy_method[
            "lower_bound_clip"
        ]
        == 0
    ),

    "policy upper bound is uncapped": (
        policy_method[
            "upper_bound_cap"
        ]
        is None
    ),

    "policy forbids method shopping": (
        policy_controls[
            "method_shopping_after_calibration"
        ]
        is False
    ),

    "policy excludes official test data": (
        policy_controls[
            "official_test_data_used"
        ]
        is False
    ),
}

failed_policy_checks = [
    name
    for name, passed in policy_checks.items()
    if not passed
]

if failed_policy_checks:
    raise AssertionError(
        "Frozen policy verification failed:\n"
        + "\n".join(failed_policy_checks)
    )


# =============================================================================
# 2. OPEN AND VALIDATE CALIBRATION DATA
# =============================================================================

calibration = pd.read_csv(
    CALIBRATION_PATH
)

calibration_frame_hash_before = (
    dataframe_content_hash(calibration)
)

required_columns = {
    EXPECTED_UNIT_COLUMN,
    EXPECTED_CYCLE_COLUMN,
    EXPECTED_TARGET,
}

missing_columns = sorted(
    required_columns.difference(
        calibration.columns
    )
)

if missing_columns:
    raise ValueError(
        "Calibration table is missing required columns: "
        + ", ".join(missing_columns)
    )

calibration_engine_ids = set(
    calibration[
        EXPECTED_UNIT_COLUMN
    ]
    .astype(int)
    .unique()
    .tolist()
)

manifest_calibration_engine_ids = (
    recover_calibration_engine_ids(
        PARTITION_MANIFEST_PATH
    )
)


# =============================================================================
# 3. LOAD THE UNCHANGED RANDOM FOREST
# =============================================================================

primary_bundle = joblib.load(
    PRIMARY_MODEL_PATH
)

required_bundle_keys = {
    "model_name",
    "target",
    "feature_order",
    "input_scaling",
    "prediction_lower_bound",
    "model",
}

missing_bundle_keys = sorted(
    required_bundle_keys.difference(
        primary_bundle
    )
)

if missing_bundle_keys:
    raise KeyError(
        "Random Forest bundle is missing keys: "
        + ", ".join(missing_bundle_keys)
    )

features = list(
    primary_bundle["feature_order"]
)

point_model = primary_bundle["model"]

prediction_lower_bound = float(
    primary_bundle[
        "prediction_lower_bound"
    ]
)

missing_features = [
    feature
    for feature in features
    if feature not in calibration.columns
]

if missing_features:
    raise ValueError(
        "Calibration table is missing model features: "
        + ", ".join(missing_features)
    )

X_calibration = calibration[
    features
].copy()

y_calibration = calibration[
    EXPECTED_TARGET
].to_numpy(dtype=float)

calibration_units = calibration[
    EXPECTED_UNIT_COLUMN
].to_numpy(dtype=int)

calibration_cycles = calibration[
    EXPECTED_CYCLE_COLUMN
].to_numpy(dtype=int)

input_values = X_calibration.to_numpy(
    dtype=float
)

data_checks = {
    "calibration contains 4104 rows": (
        len(calibration)
        == EXPECTED_CALIBRATION_ROWS
    ),

    "calibration contains 20 engines": (
        len(calibration_engine_ids)
        == EXPECTED_CALIBRATION_ENGINES
    ),

    "calibration membership matches frozen manifest": (
        calibration_engine_ids
        == manifest_calibration_engine_ids
    ),

    "engine cycle keys are unique": (
        not calibration.duplicated(
            [
                EXPECTED_UNIT_COLUMN,
                EXPECTED_CYCLE_COLUMN,
            ]
        ).any()
    ),

    "bundle contains 18 predictors": (
        len(features)
        == EXPECTED_FEATURE_COUNT
    ),

    "unit number is excluded": (
        EXPECTED_UNIT_COLUMN
        not in features
    ),

    "primary target is excluded": (
        EXPECTED_TARGET
        not in features
    ),

    "sensitivity target is excluded": (
        "rul_capped_130"
        not in features
    ),

    "tree input remains unscaled": (
        primary_bundle[
            "input_scaling"
        ]
        == "none"
    ),

    "bundle target is uncapped RUL": (
        primary_bundle["target"]
        == EXPECTED_TARGET
    ),

    "model feature order is preserved": (
        not hasattr(
            point_model,
            "feature_names_in_",
        )
        or list(
            point_model.feature_names_in_
        )
        == features
    ),

    "predictors contain no missing values": (
        not X_calibration
        .isna()
        .any()
        .any()
    ),

    "predictors are finite": (
        bool(
            np.isfinite(
                input_values
            ).all()
        )
    ),

    "target contains no missing values": (
        not calibration[
            EXPECTED_TARGET
        ].isna().any()
    ),

    "target is finite": (
        bool(
            np.isfinite(
                y_calibration
            ).all()
        )
    ),

    "target is non negative": (
        bool(
            (
                y_calibration >= 0
            ).all()
        )
    ),
}

failed_data_checks = [
    name
    for name, passed in data_checks.items()
    if not passed
]

if failed_data_checks:
    raise AssertionError(
        "Calibration data validation failed:\n"
        + "\n".join(failed_data_checks)
    )


# =============================================================================
# 4. GENERATE FROZEN POINT PREDICTIONS
# =============================================================================

raw_predictions = (
    point_model
    .predict(X_calibration)
    .astype(float)
)

point_predictions = np.clip(
    raw_predictions,
    prediction_lower_bound,
    None,
)

signed_residuals = (
    y_calibration
    - point_predictions
)

absolute_residuals = np.abs(
    signed_residuals
)

squared_residuals = np.square(
    signed_residuals
)

row_output = pd.DataFrame(
    {
        "source_row_index": (
            calibration.index.to_numpy(
                dtype=int
            )
        ),

        EXPECTED_UNIT_COLUMN: (
            calibration_units
        ),

        EXPECTED_CYCLE_COLUMN: (
            calibration_cycles
        ),

        "actual_rul_uncapped": (
            y_calibration
        ),

        "raw_point_prediction": (
            raw_predictions
        ),

        "point_prediction": (
            point_predictions
        ),

        "signed_residual_actual_minus_prediction": (
            signed_residuals
        ),

        "absolute_residual": (
            absolute_residuals
        ),

        "squared_residual": (
            squared_residuals
        ),
    }
)


# =============================================================================
# 5. CALCULATE ONE SCORE PER ENGINE
# =============================================================================

engine_scores = (
    row_output
    .groupby(
        EXPECTED_UNIT_COLUMN,
        as_index=False,
    )
    .agg(
        row_count=(
            "absolute_residual",
            "size",
        ),

        mean_absolute_residual=(
            "absolute_residual",
            "mean",
        ),

        median_absolute_residual=(
            "absolute_residual",
            "median",
        ),

        mean_squared_residual=(
            "squared_residual",
            "mean",
        ),

        engine_score=(
            "absolute_residual",
            "max",
        ),
    )
)

engine_scores[
    "root_mean_squared_residual"
] = np.sqrt(
    engine_scores.pop(
        "mean_squared_residual"
    )
)

maximum_score_indices = (
    row_output
    .groupby(
        EXPECTED_UNIT_COLUMN
    )[
        "absolute_residual"
    ]
    .idxmax()
)

maximum_score_rows = (
    row_output
    .loc[
        maximum_score_indices,
        [
            EXPECTED_UNIT_COLUMN,
            EXPECTED_CYCLE_COLUMN,
            "actual_rul_uncapped",
            "point_prediction",
            "absolute_residual",
        ],
    ]
    .rename(
        columns={
            EXPECTED_CYCLE_COLUMN: (
                "maximum_score_cycle"
            ),

            "actual_rul_uncapped": (
                "actual_rul_at_maximum_score"
            ),

            "point_prediction": (
                "prediction_at_maximum_score"
            ),

            "absolute_residual": (
                "maximum_score_crosscheck"
            ),
        }
    )
)

engine_scores = engine_scores.merge(
    maximum_score_rows,
    on=EXPECTED_UNIT_COLUMN,
    how="left",
    validate="one_to_one",
)

engine_scores = (
    engine_scores
    .sort_values(
        [
            "engine_score",
            EXPECTED_UNIT_COLUMN,
        ],
        ascending=[
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)

engine_scores[
    "score_rank_ascending"
] = np.arange(
    1,
    len(engine_scores) + 1,
    dtype=int,
)

frozen_rank = int(
    policy_method[
        "finite_sample_rank_one_based"
    ]
)

if frozen_rank > len(engine_scores):
    raise RuntimeError(
        f"Frozen rank {frozen_rank} exceeds "
        f"{len(engine_scores)} scores."
    )


# Explicit order-statistic selection.
# No np.quantile and no interpolation.

q_hat = float(
    engine_scores.iloc[
        frozen_rank - 1
    ][
        "engine_score"
    ]
)

engine_scores[
    "selected_as_q_hat"
] = (
    engine_scores[
        "score_rank_ascending"
    ]
    == frozen_rank
)

engine_scores[
    "score_not_above_q_hat"
] = (
    engine_scores[
        "engine_score"
    ]
    <= q_hat + NUMERIC_TOLERANCE
)


# =============================================================================
# 6. CONSTRUCT CALIBRATION INTERVALS
# =============================================================================

row_output[
    "conformal_q_hat"
] = q_hat

row_output[
    "interval_lower"
] = np.maximum(
    0.0,
    (
        row_output[
            "point_prediction"
        ].to_numpy()
        - q_hat
    ),
)

row_output[
    "interval_upper"
] = (
    row_output[
        "point_prediction"
    ].to_numpy()
    + q_hat
)

row_output[
    "interval_width"
] = (
    row_output[
        "interval_upper"
    ]
    - row_output[
        "interval_lower"
    ]
)

row_output[
    "lower_bound_was_clipped"
] = (
    row_output[
        "point_prediction"
    ]
    - q_hat
    < 0.0
)

row_output[
    "row_covered"
] = (
    (
        row_output[
            "actual_rul_uncapped"
        ]
        >= (
            row_output[
                "interval_lower"
            ]
            - NUMERIC_TOLERANCE
        )
    )
    &
    (
        row_output[
            "actual_rul_uncapped"
        ]
        <= (
            row_output[
                "interval_upper"
            ]
            + NUMERIC_TOLERANCE
        )
    )
)

engine_coverage = (
    row_output
    .groupby(
        EXPECTED_UNIT_COLUMN,
        as_index=False,
    )
    .agg(
        all_rows_covered=(
            "row_covered",
            "all",
        ),

        covered_row_count=(
            "row_covered",
            "sum",
        ),

        total_row_count=(
            "row_covered",
            "size",
        ),
    )
)

engine_coverage[
    "engine_row_coverage_rate"
] = (
    engine_coverage[
        "covered_row_count"
    ]
    / engine_coverage[
        "total_row_count"
    ]
)

engine_scores = engine_scores.merge(
    engine_coverage,
    on=EXPECTED_UNIT_COLUMN,
    how="left",
    validate="one_to_one",
)

row_coverage_rate = float(
    row_output[
        "row_covered"
    ].mean()
)

simultaneously_covered_engines = int(
    engine_coverage[
        "all_rows_covered"
    ].sum()
)

engine_simultaneous_coverage_rate = float(
    engine_coverage[
        "all_rows_covered"
    ].mean()
)

lower_clipped_rows = int(
    row_output[
        "lower_bound_was_clipped"
    ].sum()
)

score_values = engine_scores[
    "engine_score"
].to_numpy(dtype=float)

score_summary = {
    "minimum": float(
        np.min(score_values)
    ),

    "first_quartile": float(
        np.quantile(
            score_values,
            0.25,
        )
    ),

    "median": float(
        np.median(score_values)
    ),

    "mean": float(
        np.mean(score_values)
    ),

    "third_quartile": float(
        np.quantile(
            score_values,
            0.75,
        )
    ),

    "maximum": float(
        np.max(score_values)
    ),
}

summary_table = pd.DataFrame(
    [
        {
            "confidence_level": (
                EXPECTED_CONFIDENCE
            ),

            "miscoverage_alpha": (
                EXPECTED_ALPHA
            ),

            "calibration_engine_count": (
                len(engine_scores)
            ),

            "calibration_row_count": (
                len(row_output)
            ),

            "finite_sample_rank_one_based": (
                frozen_rank
            ),

            "q_hat_cycles": (
                q_hat
            ),

            "calibration_row_coverage_rate": (
                row_coverage_rate
            ),

            "simultaneously_covered_engines": (
                simultaneously_covered_engines
            ),

            "calibration_engine_simultaneous_coverage_rate": (
                engine_simultaneous_coverage_rate
            ),

            "lower_clipped_rows": (
                lower_clipped_rows
            ),

            "mean_interval_width_cycles": float(
                row_output[
                    "interval_width"
                ].mean()
            ),

            "median_interval_width_cycles": float(
                row_output[
                    "interval_width"
                ].median()
            ),

            "maximum_interval_width_cycles": float(
                row_output[
                    "interval_width"
                ].max()
            ),
        }
    ]
)


# =============================================================================
# 7. SAVE THE SELF-CONTAINED CONFORMAL BUNDLE
# =============================================================================

conformal_bundle = {
    "artifact_type": (
        "engine_block_split_conformal_regressor"
    ),

    "model_name": (
        "random_forest_uncapped_with_"
        "engine_block_conformal_interval"
    ),

    "target": EXPECTED_TARGET,
    "feature_order": features,
    "input_scaling": "none",

    "prediction_lower_bound": (
        prediction_lower_bound
    ),

    "primary_model_sha256": (
        EXPECTED_MODEL_SHA256
    ),

    "policy_sha256": (
        policy_hash_before
    ),

    "confidence_level": (
        EXPECTED_CONFIDENCE
    ),

    "miscoverage_alpha": (
        EXPECTED_ALPHA
    ),

    "calibration_unit": (
        "complete_engine_trajectory"
    ),

    "row_nonconformity_score": (
        "absolute_residual"
    ),

    "within_engine_aggregation": (
        "maximum"
    ),

    "calibration_engine_count": (
        len(engine_scores)
    ),

    "finite_sample_rank_one_based": (
        frozen_rank
    ),

    "quantile_interpolation": None,

    "q_hat": q_hat,

    "interval_lower_formula": (
        "max(0, point_prediction - q_hat)"
    ),

    "interval_upper_formula": (
        "point_prediction + q_hat"
    ),

    "upper_bound_cap": None,

    "point_model_bundle": (
        primary_bundle
    ),

    "calibrated_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),

    "scikit_learn_version": (
        sklearn.__version__
    ),

    "joblib_version": (
        joblib.__version__
    ),
}

joblib.dump(
    conformal_bundle,
    CONFORMAL_BUNDLE_PATH,
)

conformal_bundle_hash = sha256_file(
    CONFORMAL_BUNDLE_PATH
)


# =============================================================================
# 8. RELOAD AND REPRODUCE THE INTERVALS
# =============================================================================

reloaded_bundle = joblib.load(
    CONFORMAL_BUNDLE_PATH
)

reloaded_raw_predictions = (
    reloaded_bundle[
        "point_model_bundle"
    ][
        "model"
    ]
    .predict(
        calibration[
            reloaded_bundle[
                "feature_order"
            ]
        ]
    )
    .astype(float)
)

reloaded_point_predictions = np.clip(
    reloaded_raw_predictions,
    float(
        reloaded_bundle[
            "prediction_lower_bound"
        ]
    ),
    None,
)

reloaded_q_hat = float(
    reloaded_bundle["q_hat"]
)

reloaded_lower = np.maximum(
    0.0,
    (
        reloaded_point_predictions
        - reloaded_q_hat
    ),
)

reloaded_upper = (
    reloaded_point_predictions
    + reloaded_q_hat
)


# =============================================================================
# 9. CREATE THESIS-READY ORDERED-SCORE FIGURE
# =============================================================================

plot_frame = (
    engine_scores
    .sort_values(
        "score_rank_ascending"
    )
    .copy()
)

selected_row = (
    plot_frame.loc[
        plot_frame[
            "selected_as_q_hat"
        ]
    ]
    .iloc[0]
)

fig, axis = plt.subplots(
    figsize=(11.5, 6.5)
)

axis.plot(
    plot_frame[
        "score_rank_ascending"
    ],

    plot_frame[
        "engine_score"
    ],

    color="#2F6BDE",
    linewidth=1.8,
    marker="o",
    markersize=6,
    markerfacecolor="white",
    markeredgewidth=1.6,
)

axis.scatter(
    [
        selected_row[
            "score_rank_ascending"
        ]
    ],

    [
        selected_row[
            "engine_score"
        ]
    ],

    color="#D97706",
    edgecolor="#7C3E00",
    linewidth=0.8,
    s=95,
    zorder=5,
)

axis.axhline(
    q_hat,
    color="#374151",
    linestyle="--",
    linewidth=1.4,
)

axis.annotate(
    (
        f"q_hat = {q_hat:.2f} cycles\n"
        "19th order statistic"
    ),

    xy=(
        float(
            selected_row[
                "score_rank_ascending"
            ]
        ),
        q_hat,
    ),

    xytext=(-150, 28),
    textcoords="offset points",

    arrowprops={
        "arrowstyle": "->",
        "color": "#374151",
        "linewidth": 1.0,
    },

    fontsize=10,
    color="#1F2937",
)

axis.set_title(
    "Engine-block conformal calibration scores",
    fontsize=15,
    weight="bold",
    pad=28,
)

axis.text(
    0.5,
    1.025,

    (
        "20 FD001 calibration engines; "
        "maximum absolute residual per engine"
    ),

    transform=axis.transAxes,
    ha="center",
    va="bottom",
    fontsize=10.5,
    color="#4B5563",
)

axis.set_xlabel(
    "Ascending engine-score rank",
    fontsize=11,
)

axis.set_ylabel(
    "Maximum absolute residual (cycles)",
    fontsize=11,
)

axis.set_xticks(
    np.arange(
        1,
        EXPECTED_CALIBRATION_ENGINES + 1,
    )
)

axis.set_ylim(bottom=0)

axis.grid(
    axis="y",
    color="#D1D5DB",
    linewidth=0.8,
    alpha=0.75,
)

axis.grid(
    axis="x",
    visible=False,
)

axis.spines[
    "top"
].set_visible(False)

axis.spines[
    "right"
].set_visible(False)

fig.text(
    0.5,
    0.015,

    (
        "Calibration diagnostic only. The observed "
        "calibration coverage is induced by the frozen "
        "rank and is not an independent estimate of "
        "future coverage."
    ),

    ha="center",
    va="bottom",
    fontsize=9,
    color="#4B5563",
)

fig.tight_layout(
    rect=[
        0.03,
        0.06,
        0.98,
        0.95,
    ]
)

fig.savefig(
    FIGURE_PATH,
    dpi=220,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()
plt.close(fig)


# =============================================================================
# 10. SAVE CALIBRATION RESULTS
# =============================================================================

row_output.to_csv(
    ROW_OUTPUT_PATH,
    index=False,
)

engine_scores.to_csv(
    ENGINE_SCORE_PATH,
    index=False,
)

summary_table.to_csv(
    SUMMARY_PATH,
    index=False,
)

calibration_frame_hash_after = (
    dataframe_content_hash(calibration)
)

policy_hash_after = sha256_file(
    POLICY_PATH
)

model_hash_after = sha256_file(
    PRIMARY_MODEL_PATH
)

calibration_file_hash_after = sha256_file(
    CALIBRATION_PATH
)

calibration_record = {
    "phase": 5,

    "step": (
        "2_engine_block_conformal_calibration"
    ),

    "status": (
        "calibrated_and_frozen"
    ),

    "calibrated_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),

    "primary_model": (
        EXPECTED_MODEL_NAME
    ),

    "primary_model_sha256": (
        model_hash_after
    ),

    "primary_target": (
        EXPECTED_TARGET
    ),

    "policy_path": (
        str(POLICY_PATH)
    ),

    "policy_sha256": (
        policy_hash_after
    ),

    "calibration_file_path": (
        str(CALIBRATION_PATH)
    ),

    "calibration_file_sha256": (
        calibration_file_hash_after
    ),

    "feature_count": (
        len(features)
    ),

    "calibration_row_count": (
        len(row_output)
    ),

    "calibration_engine_count": (
        len(engine_scores)
    ),

    "confidence_level": (
        EXPECTED_CONFIDENCE
    ),

    "miscoverage_alpha": (
        EXPECTED_ALPHA
    ),

    "finite_sample_rank_one_based": (
        frozen_rank
    ),

    "quantile_selection": (
        "explicit_sorted_score_at_"
        "zero_based_index_18"
    ),

    "quantile_interpolation_used": False,

    "q_hat_cycles": q_hat,

    "engine_score_summary_cycles": (
        score_summary
    ),

    "calibration_diagnostics": {
        "row_coverage_rate": (
            row_coverage_rate
        ),

        "simultaneously_covered_engine_count": (
            simultaneously_covered_engines
        ),

        "engine_simultaneous_coverage_rate": (
            engine_simultaneous_coverage_rate
        ),

        "lower_bound_clipped_row_count": (
            lower_clipped_rows
        ),

        "mean_interval_width_cycles": float(
            row_output[
                "interval_width"
            ].mean()
        ),

        "median_interval_width_cycles": float(
            row_output[
                "interval_width"
            ].median()
        ),

        "maximum_interval_width_cycles": float(
            row_output[
                "interval_width"
            ].max()
        ),

        "interpretation": (
            "These calibration-set coverage values are "
            "internal diagnostics induced by the selected "
            "order statistic. They are not an independent "
            "estimate of coverage on unseen engines."
        ),
    },

    "coverage_claim": {
        "unit": (
            "complete_engine_trajectory"
        ),

        "nominal_confidence": (
            EXPECTED_CONFIDENCE
        ),

        "finite_sample_rank_fraction": (
            frozen_rank
            / (
                EXPECTED_CALIBRATION_ENGINES
                + 1
            )
        ),

        "assumption": (
            "exchangeability_of_complete_engines"
        ),

        "operational_safety_guarantee": False,
    },

    "conformal_bundle_path": (
        str(CONFORMAL_BUNDLE_PATH)
    ),

    "conformal_bundle_sha256": (
        conformal_bundle_hash
    ),

    "research_controls": {
        "point_model_retrained_or_modified": False,

        "policy_changed_after_calibration_opened": (
            False
        ),

        "alternative_conformal_methods_compared": (
            False
        ),

        "validation_data_used": False,
        "official_test_data_used": False,
        "test_rul_values_accessed": False,
        "capped_130_target_used": False,
        "business_thresholds_used": False,
    },

    "software": {
        "scikit_learn": (
            sklearn.__version__
        ),

        "joblib": (
            joblib.__version__
        ),

        "numpy": (
            np.__version__
        ),

        "pandas": (
            pd.__version__
        ),
    },
}

write_json(
    CALIBRATION_RECORD_PATH,
    calibration_record,
)


# =============================================================================
# 11. FINAL VALIDATION
# =============================================================================

calibration_checks = {
    **policy_checks,
    **data_checks,

    "all raw predictions are finite": (
        bool(
            np.isfinite(
                raw_predictions
            ).all()
        )
    ),

    "all point predictions are finite": (
        bool(
            np.isfinite(
                point_predictions
            ).all()
        )
    ),

    "all point predictions are non negative": (
        bool(
            (
                point_predictions >= 0
            ).all()
        )
    ),

    "all absolute residuals are finite": (
        bool(
            np.isfinite(
                absolute_residuals
            ).all()
        )
    ),

    "exactly one engine score exists per engine": (
        len(engine_scores)
        == EXPECTED_CALIBRATION_ENGINES
        and engine_scores[
            EXPECTED_UNIT_COLUMN
        ].is_unique
    ),

    "engine scores equal maximum row residuals": (
        bool(
            np.allclose(
                engine_scores[
                    "engine_score"
                ],

                engine_scores[
                    "maximum_score_crosscheck"
                ],

                rtol=0,
                atol=NUMERIC_TOLERANCE,
            )
        )
    ),

    "all engine scores are finite": (
        bool(
            np.isfinite(
                score_values
            ).all()
        )
    ),

    "all engine scores are non negative": (
        bool(
            (
                score_values >= 0
            ).all()
        )
    ),

    "frozen rank remains 19": (
        frozen_rank
        == EXPECTED_RANK
    ),

    "q hat uses explicit nineteenth sorted score": (
        math.isclose(
            q_hat,

            float(
                np.sort(
                    score_values
                )[
                    EXPECTED_RANK - 1
                ]
            ),

            rel_tol=0,
            abs_tol=NUMERIC_TOLERANCE,
        )
    ),

    "no quantile interpolation was used": (
        reloaded_bundle[
            "quantile_interpolation"
        ]
        is None
    ),

    "at least nineteen engine scores do not exceed q hat": (
        int(
            engine_scores[
                "score_not_above_q_hat"
            ].sum()
        )
        >= EXPECTED_RANK
    ),

    "interval lower bounds are non negative": (
        bool(
            (
                row_output[
                    "interval_lower"
                ]
                >= 0
            ).all()
        )
    ),

    "interval upper bounds are not capped": (
        bool(
            np.allclose(
                row_output[
                    "interval_upper"
                ],

                point_predictions
                + q_hat,

                rtol=0,
                atol=NUMERIC_TOLERANCE,
            )
        )
    ),

    "interval bounds are ordered": (
        bool(
            (
                row_output[
                    "interval_upper"
                ]
                >= row_output[
                    "interval_lower"
                ]
            ).all()
        )
    ),

    "at least nineteen calibration engines are simultaneously covered": (
        simultaneously_covered_engines
        >= EXPECTED_RANK
    ),

    "saved bundle reproduces point predictions": (
        bool(
            np.allclose(
                reloaded_point_predictions,
                point_predictions,
                rtol=0,
                atol=NUMERIC_TOLERANCE,
            )
        )
    ),

    "saved bundle reproduces lower bounds": (
        bool(
            np.allclose(
                reloaded_lower,

                row_output[
                    "interval_lower"
                ],

                rtol=0,
                atol=NUMERIC_TOLERANCE,
            )
        )
    ),

    "saved bundle reproduces upper bounds": (
        bool(
            np.allclose(
                reloaded_upper,

                row_output[
                    "interval_upper"
                ],

                rtol=0,
                atol=NUMERIC_TOLERANCE,
            )
        )
    ),

    "calibration dataframe was not modified": (
        calibration_frame_hash_before
        == calibration_frame_hash_after
    ),

    "calibration source file was not modified": (
        calibration_file_hash_before
        == calibration_file_hash_after
    ),

    "frozen policy was not modified": (
        policy_hash_before
        == policy_hash_after
    ),

    "primary model was not modified": (
        model_hash_before
        == model_hash_after
    ),

    "all output files exist": all(
        path.is_file()
        for path in [
            ROW_OUTPUT_PATH,
            ENGINE_SCORE_PATH,
            SUMMARY_PATH,
            FIGURE_PATH,
            CALIBRATION_RECORD_PATH,
            CONFORMAL_BUNDLE_PATH,
        ]
    ),
}

failed_checks = [
    name
    for name, passed in calibration_checks.items()
    if not passed
]

if failed_checks:
    raise AssertionError(
        "Phase 5 Step 2 validation failed:\n"
        + "\n".join(failed_checks)
    )


# =============================================================================
# 12. CREATE EVIDENCE MANIFEST AND RETAINED ZIP
# =============================================================================

manifest_entries = [
    (
        "frozen_policy",
        POLICY_PATH,
        True,
    ),

    (
        "final_registry",
        FINAL_REGISTRY_PATH,
        True,
    ),

    (
        "partition_metadata",
        PARTITION_MANIFEST_PATH,
        True,
    ),

    (
        "sealed_source_hash_reference",
        CALIBRATION_PATH,
        False,
    ),

    (
        "frozen_model_hash_reference",
        PRIMARY_MODEL_PATH,
        False,
    ),

    (
        "calibrated_model_bundle",
        CONFORMAL_BUNDLE_PATH,
        True,
    ),

    (
        "calibration_row_results",
        ROW_OUTPUT_PATH,
        True,
    ),

    (
        "engine_score_results",
        ENGINE_SCORE_PATH,
        True,
    ),

    (
        "calibration_summary",
        SUMMARY_PATH,
        True,
    ),

    (
        "calibration_figure",
        FIGURE_PATH,
        True,
    ),

    (
        "calibration_record",
        CALIBRATION_RECORD_PATH,
        True,
    ),
]

manifest_rows = []

for (
    role,
    path,
    include_in_zip,
) in manifest_entries:

    manifest_rows.append(
        {
            "role": role,
            "path": str(path),
            "size_bytes": (
                path.stat().st_size
            ),
            "sha256": (
                sha256_file(path)
            ),
            "included_in_step_zip": (
                include_in_zip
            ),
        }
    )

with EVIDENCE_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
    newline="",
) as file_handle:

    writer = csv.DictWriter(
        file_handle,
        fieldnames=[
            "role",
            "path",
            "size_bytes",
            "sha256",
            "included_in_step_zip",
        ],
    )

    writer.writeheader()
    writer.writerows(manifest_rows)

with zipfile.ZipFile(
    EVIDENCE_ZIP_PATH,
    "w",
    zipfile.ZIP_DEFLATED,
) as archive:

    for row in manifest_rows:
        if row[
            "included_in_step_zip"
        ]:
            source_path = Path(
                row["path"]
            )

            archive.write(
                source_path,

                arcname=str(
                    source_path.relative_to(
                        PROJECT_ROOT
                    )
                ),
            )

    archive.write(
        EVIDENCE_MANIFEST_PATH,

        arcname=str(
            EVIDENCE_MANIFEST_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
    )

evidence_zip_hash = sha256_file(
    EVIDENCE_ZIP_PATH
)


# =============================================================================
# 13. BOUNDED REVIEW OUTPUT
# =============================================================================

print("FROZEN FOUNDATION CHECK")
print("=" * 90)
print(
    f"PASS — Primary model: "
    f"{EXPECTED_MODEL_NAME}"
)
print(
    f"PASS — Primary model SHA-256: "
    f"{model_hash_after}"
)
print(
    f"PASS — Frozen policy SHA-256: "
    f"{policy_hash_after}"
)
print(
    "PASS — Calibration file matches "
    "its policy-freeze fingerprint"
)

print("\nCALIBRATION INVENTORY")
print("=" * 90)
print(
    f"Calibration engines: "
    f"{len(engine_scores)}"
)
print(
    f"Calibration rows: "
    f"{len(row_output)}"
)
print(
    f"Predictors: "
    f"{len(features)}"
)
print(
    "Exchangeable score unit: "
    "complete engine trajectory"
)
print(
    "Within-engine aggregation: "
    "maximum absolute residual"
)

print("\nENGINE-SCORE DISTRIBUTION")
print("=" * 90)

for name, value in score_summary.items():
    print(
        f"{name}: "
        f"{value:.6f} cycles"
    )

print(
    f"Selected order statistic: "
    f"{frozen_rank}th of "
    f"{len(engine_scores)}"
)

print(
    f"Frozen q_hat: "
    f"{q_hat:.6f} cycles"
)

print("\nORDERED ENGINE SCORES")
print("=" * 90)

print(
    engine_scores[
        [
            "score_rank_ascending",
            EXPECTED_UNIT_COLUMN,
            "engine_score",
            "maximum_score_cycle",
            "selected_as_q_hat",
            "all_rows_covered",
        ]
    ].to_string(
        index=False
    )
)

print("\nCALIBRATION DIAGNOSTICS")
print("=" * 90)

print(
    f"Row coverage: "
    f"{row_coverage_rate:.4%}"
)

print(
    "Simultaneously covered engines: "
    f"{simultaneously_covered_engines} "
    f"of {len(engine_scores)} "
    f"({engine_simultaneous_coverage_rate:.4%})"
)

print(
    "Rows with lower bound clipped to zero: "
    f"{lower_clipped_rows}"
)

print(
    "Mean interval width after lower clipping: "
    f"{row_output['interval_width'].mean():.6f} "
    "cycles"
)

print(
    "IMPORTANT — Calibration coverage is an "
    "internal rank diagnostic, not an independent "
    "estimate of unseen-engine coverage."
)

print("\nCONFORMAL CALIBRATION VALIDATION")
print("=" * 90)

for (
    check_name,
    passed,
) in calibration_checks.items():

    status = (
        "PASS"
        if passed
        else "FAIL"
    )

    print(
        f"{status} — "
        f"{check_name}"
    )

print("\nRESEARCH CONTROLS")
print("=" * 90)
print(
    "Primary Random Forest retrained "
    "or modified: No"
)
print(
    "Frozen conformal policy changed: No"
)
print(
    "Alternative conformal methods compared: No"
)
print("Validation data used: No")
print("Official test data used: No")
print("Test RUL values accessed: No")
print("Capped-130 target used: No")
print("Business thresholds used: No")

print("\nOUTPUT FILES")
print("=" * 90)
print(
    f"Conformal bundle: "
    f"{CONFORMAL_BUNDLE_PATH}"
)
print(
    f"Engine scores: "
    f"{ENGINE_SCORE_PATH}"
)
print(
    "Row predictions and intervals: "
    f"{ROW_OUTPUT_PATH}"
)
print(
    f"Calibration summary: "
    f"{SUMMARY_PATH}"
)
print(
    f"Ordered-score figure: "
    f"{FIGURE_PATH}"
)
print(
    f"Calibration record: "
    f"{CALIBRATION_RECORD_PATH}"
)
print(
    f"Evidence manifest: "
    f"{EVIDENCE_MANIFEST_PATH}"
)
print(
    "Evidence ZIP retained in Colab: "
    f"{EVIDENCE_ZIP_PATH}"
)
print(
    f"Conformal bundle SHA-256: "
    f"{conformal_bundle_hash}"
)
print(
    f"Evidence ZIP SHA-256: "
    f"{evidence_zip_hash}"
)
print("Browser download triggered: No")

print("\nFINAL CHECK")
print("=" * 90)
print(
    "PASS — The frozen conformal method "
    "was applied exactly as specified."
)
print(
    "PASS — Twenty engines contributed "
    "one maximum-residual score each."
)
print(
    "PASS — q_hat was selected as the "
    "explicit 19th order statistic."
)
print(
    "PASS — The saved conformal bundle "
    "reproduces predictions and intervals."
)
print(
    "PASS — The Random Forest and policy "
    "remained unchanged."
)
print(
    "PASS — Validation and official test "
    "data remained unused."
)
print(
    "AWAITING APPROVAL — Official test "
    "evaluation has not started."
)

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import math
import platform
import zipfile

import joblib
import numpy as np
import pandas as pd


# =============================================================================
# PHASE 5 — STEP 3
# COMPLETE AND PACKAGE THE UNCERTAINTY PHASE
#
# This cell:
# - Does not retrain any model.
# - Does not recalibrate q_hat.
# - Does not use validation data.
# - Does not open official test data or test RUL.
# - Creates the complete ZIP inside Colab without downloading it.
# =============================================================================

PROJECT_ROOT = Path(
    "/content/Predictive_Maintenance_Thesis"
)

DATA_DIR = PROJECT_ROOT / "data" / "interim"
LOG_DIR = PROJECT_ROOT / "logs"
RESULT_DIR = PROJECT_ROOT / "results"
FIGURE_DIR = RESULT_DIR / "figures"

REGRESSION_MODEL_DIR = (
    PROJECT_ROOT / "models" / "regression"
)

UNCERTAINTY_MODEL_DIR = (
    PROJECT_ROOT / "models" / "uncertainty"
)


# =============================================================================
# ARTIFACT PATHS
# =============================================================================

POLICY_PATH = (
    LOG_DIR
    / "phase5_engine_block_conformal_policy.json"
)

DECISION_PATH = (
    LOG_DIR
    / "phase5_conformal_method_decision.md"
)

STEP1_MANIFEST_PATH = (
    LOG_DIR
    / "phase5_step1_conformal_policy_manifest.csv"
)

STEP1_ZIP_PATH = (
    PROJECT_ROOT
    / "Phase5_Step1_Conformal_Policy_Evidence.zip"
)

PRIMARY_MODEL_PATH = (
    REGRESSION_MODEL_DIR
    / "fd001_random_forest_uncapped_bundle.joblib"
)

FINAL_REGISTRY_PATH = (
    LOG_DIR
    / "phase4_final_model_registry.json"
)

PARTITION_MANIFEST_PATH = (
    DATA_DIR
    / "fd001_engine_partition_manifest.csv"
)

CALIBRATION_SOURCE_PATH = (
    DATA_DIR
    / "fd001_calibration_targets.csv"
)

CONFORMAL_BUNDLE_PATH = (
    UNCERTAINTY_MODEL_DIR
    / (
        "fd001_random_forest_"
        "engine_block_conformal_bundle.joblib"
    )
)

ENGINE_SCORE_PATH = (
    RESULT_DIR
    / "phase5_engine_block_calibration_scores.csv"
)

ROW_OUTPUT_PATH = (
    RESULT_DIR
    / "phase5_conformal_calibration_row_predictions.csv"
)

SUMMARY_PATH = (
    RESULT_DIR
    / "phase5_conformal_calibration_summary.csv"
)

FIGURE_PATH = (
    FIGURE_DIR
    / "phase5_engine_block_calibration_scores.png"
)

CALIBRATION_RECORD_PATH = (
    LOG_DIR
    / "phase5_conformal_calibration_record.json"
)

STEP2_MANIFEST_PATH = (
    LOG_DIR
    / "phase5_step2_conformal_calibration_manifest.csv"
)

STEP2_ZIP_PATH = (
    PROJECT_ROOT
    / "Phase5_Step2_Conformal_Calibration_Evidence.zip"
)

OPTIONAL_CORRECTION_RECORD_PATH = (
    LOG_DIR
    / "phase5_step2_figure_correction_record.json"
)

COMPLETION_RECORD_PATH = (
    LOG_DIR
    / "phase5_technical_completion_record.json"
)

COMPLETE_MANIFEST_PATH = (
    LOG_DIR
    / "phase5_complete_evidence_manifest.csv"
)

COMPLETE_ZIP_PATH = (
    PROJECT_ROOT
    / "Phase5_Complete_Uncertainty_Evidence.zip"
)


# =============================================================================
# LOCKED EXPECTATIONS
# =============================================================================

EXPECTED_PRIMARY_MODEL_SHA256 = (
    "99620a1a161f62a716d4cc058497159146197e7d91bba713581c822af808e5d5"
)

EXPECTED_MODEL_NAME = (
    "random_forest_uncapped"
)

EXPECTED_TARGET = "rul_uncapped"
EXPECTED_FEATURE_COUNT = 18
EXPECTED_ENGINE_COUNT = 20
EXPECTED_ROW_COUNT = 4104
EXPECTED_CONFIDENCE = 0.90
EXPECTED_ALPHA = 0.10
EXPECTED_RANK = 19

TOLERANCE = 1e-9


# =============================================================================
# SUPPORT FUNCTIONS
# =============================================================================

def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        for chunk in iter(
            lambda: file_handle.read(
                chunk_size
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def load_json(
    path: Path,
) -> dict:
    with path.open(
        "r",
        encoding="utf-8",
    ) as file_handle:
        return json.load(file_handle)


def write_json(
    path: Path,
    payload: dict,
) -> None:
    with path.open(
        "w",
        encoding="utf-8",
    ) as file_handle:

        json.dump(
            payload,
            file_handle,
            indent=2,
            ensure_ascii=False,
        )

        file_handle.write("\n")


def boolean_series(
    series: pd.Series,
) -> pd.Series:
    if pd.api.types.is_bool_dtype(
        series
    ):
        return series.astype(bool)

    normalised = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    valid_values = {
        "true",
        "false",
        "1",
        "0",
        "yes",
        "no",
    }

    if not set(
        normalised.unique()
    ).issubset(valid_values):
        raise ValueError(
            "Cannot interpret Boolean values in "
            f"{series.name}."
        )

    return normalised.isin(
        {
            "true",
            "1",
            "yes",
        }
    )


# =============================================================================
# 1. VERIFY REQUIRED PHASE 5 ARTIFACTS
# =============================================================================

required_paths = [
    POLICY_PATH,
    DECISION_PATH,
    STEP1_MANIFEST_PATH,
    STEP1_ZIP_PATH,
    PRIMARY_MODEL_PATH,
    FINAL_REGISTRY_PATH,
    PARTITION_MANIFEST_PATH,
    CALIBRATION_SOURCE_PATH,
    CONFORMAL_BUNDLE_PATH,
    ENGINE_SCORE_PATH,
    ROW_OUTPUT_PATH,
    SUMMARY_PATH,
    FIGURE_PATH,
    CALIBRATION_RECORD_PATH,
    STEP2_MANIFEST_PATH,
    STEP2_ZIP_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Phase 5 artifacts are missing:\n"
        + "\n".join(missing_paths)
    )


# =============================================================================
# 2. FINGERPRINT PROTECTED INPUTS
# =============================================================================

primary_model_hash_before = (
    sha256_file(
        PRIMARY_MODEL_PATH
    )
)

policy_hash_before = (
    sha256_file(
        POLICY_PATH
    )
)

conformal_bundle_hash_before = (
    sha256_file(
        CONFORMAL_BUNDLE_PATH
    )
)

calibration_source_hash_before = (
    sha256_file(
        CALIBRATION_SOURCE_PATH
    )
)

if (
    primary_model_hash_before
    != EXPECTED_PRIMARY_MODEL_SHA256
):
    raise RuntimeError(
        "The approved Random Forest hash changed.\n"
        f"Expected: {EXPECTED_PRIMARY_MODEL_SHA256}\n"
        f"Observed: {primary_model_hash_before}"
    )


# =============================================================================
# 3. LOAD SAVED EVIDENCE
# =============================================================================

policy = load_json(
    POLICY_PATH
)

registry = load_json(
    FINAL_REGISTRY_PATH
)

calibration_record = load_json(
    CALIBRATION_RECORD_PATH
)

conformal_bundle = joblib.load(
    CONFORMAL_BUNDLE_PATH
)

engine_scores = pd.read_csv(
    ENGINE_SCORE_PATH
)

row_output = pd.read_csv(
    ROW_OUTPUT_PATH
)

summary = pd.read_csv(
    SUMMARY_PATH
)

if len(summary) != 1:
    raise AssertionError(
        "The Phase 5 summary must contain "
        "exactly one row."
    )

registry_text = json.dumps(
    registry,
    sort_keys=True,
).lower()


# =============================================================================
# 4. RECONCILE Q_HAT ACROSS ALL EVIDENCE
# =============================================================================

q_hat_bundle = float(
    conformal_bundle["q_hat"]
)

q_hat_record = float(
    calibration_record[
        "q_hat_cycles"
    ]
)

q_hat_summary = float(
    summary.loc[
        0,
        "q_hat_cycles",
    ]
)

selected_score_rows = engine_scores.loc[
    engine_scores[
        "score_rank_ascending"
    ].astype(int)
    == EXPECTED_RANK
]

if len(selected_score_rows) != 1:
    raise AssertionError(
        "Exactly one nineteenth engine "
        "score was expected."
    )

q_hat_scores = float(
    selected_score_rows.iloc[
        0
    ][
        "engine_score"
    ]
)


# =============================================================================
# 5. REPRODUCE INTERVALS FROM SAVED RESULTS
# =============================================================================

point_predictions = row_output[
    "point_prediction"
].to_numpy(dtype=float)

saved_lower = row_output[
    "interval_lower"
].to_numpy(dtype=float)

saved_upper = row_output[
    "interval_upper"
].to_numpy(dtype=float)

saved_absolute_residual = row_output[
    "absolute_residual"
].to_numpy(dtype=float)

recomputed_lower = np.maximum(
    0.0,
    point_predictions - q_hat_bundle,
)

recomputed_upper = (
    point_predictions
    + q_hat_bundle
)

row_covered = boolean_series(
    row_output[
        "row_covered"
    ]
)

lower_bound_was_clipped = (
    boolean_series(
        row_output[
            "lower_bound_was_clipped"
        ]
    )
)

engine_simultaneous_coverage = (
    pd.DataFrame(
        {
            "unit_number": (
                row_output[
                    "unit_number"
                ].astype(int)
            ),

            "row_covered": (
                row_covered
            ),
        }
    )
    .groupby(
        "unit_number"
    )[
        "row_covered"
    ]
    .all()
)

recomputed_engine_scores = (
    pd.DataFrame(
        {
            "unit_number": (
                row_output[
                    "unit_number"
                ].astype(int)
            ),

            "absolute_residual": (
                saved_absolute_residual
            ),
        }
    )
    .groupby(
        "unit_number",
        as_index=False,
    )[
        "absolute_residual"
    ]
    .max()
    .rename(
        columns={
            "absolute_residual": (
                "recomputed_engine_score"
            )
        }
    )
)

score_crosscheck = (
    engine_scores[
        [
            "unit_number",
            "engine_score",
        ]
    ]
    .merge(
        recomputed_engine_scores,
        on="unit_number",
        how="inner",
        validate="one_to_one",
    )
)

simultaneously_covered_engines = int(
    engine_simultaneous_coverage.sum()
)

row_coverage_rate = float(
    row_covered.mean()
)

interval_widths = (
    row_output[
        "interval_upper"
    ]
    - row_output[
        "interval_lower"
    ]
)

mean_interval_width = float(
    interval_widths.mean()
)

median_interval_width = float(
    interval_widths.median()
)

lower_clipped_rows = int(
    lower_bound_was_clipped.sum()
)


# =============================================================================
# 6. PHASE 5 COMPLETION VALIDATION
# =============================================================================

checks = {
    "final registry identifies Random Forest": (
        "random_forest"
        in registry_text
    ),

    "final registry preserves approved model hash": (
        EXPECTED_PRIMARY_MODEL_SHA256
        in registry_text
    ),

    "policy selects Random Forest": (
        policy[
            "primary_model"
        ][
            "name"
        ]
        == EXPECTED_MODEL_NAME
    ),

    "policy uses uncapped RUL": (
        policy[
            "primary_model"
        ][
            "target"
        ]
        == EXPECTED_TARGET
    ),

    "policy hash matches calibration record": (
        policy_hash_before
        == calibration_record[
            "policy_sha256"
        ]
    ),

    "policy hash matches conformal bundle": (
        policy_hash_before
        == conformal_bundle[
            "policy_sha256"
        ]
    ),

    "calibration source matches calibration record": (
        calibration_source_hash_before
        == calibration_record[
            "calibration_file_sha256"
        ]
    ),

    "conformal bundle hash matches calibration record": (
        conformal_bundle_hash_before
        == calibration_record[
            "conformal_bundle_sha256"
        ]
    ),

    "conformal bundle embeds approved model hash": (
        conformal_bundle[
            "primary_model_sha256"
        ]
        == EXPECTED_PRIMARY_MODEL_SHA256
    ),

    "conformal bundle contains 18 features": (
        len(
            conformal_bundle[
                "feature_order"
            ]
        )
        == EXPECTED_FEATURE_COUNT
    ),

    "unit number is excluded from features": (
        "unit_number"
        not in conformal_bundle[
            "feature_order"
        ]
    ),

    "primary target is excluded from features": (
        EXPECTED_TARGET
        not in conformal_bundle[
            "feature_order"
        ]
    ),

    "sensitivity target is excluded from features": (
        "rul_capped_130"
        not in conformal_bundle[
            "feature_order"
        ]
    ),

    "confidence remains 90 percent": (
        math.isclose(
            float(
                conformal_bundle[
                    "confidence_level"
                ]
            ),
            EXPECTED_CONFIDENCE,
            rel_tol=0,
            abs_tol=TOLERANCE,
        )
    ),

    "alpha remains 0.10": (
        math.isclose(
            float(
                conformal_bundle[
                    "miscoverage_alpha"
                ]
            ),
            EXPECTED_ALPHA,
            rel_tol=0,
            abs_tol=TOLERANCE,
        )
    ),

    "calibration contains 20 engine scores": (
        len(engine_scores)
        == EXPECTED_ENGINE_COUNT
        and engine_scores[
            "unit_number"
        ].nunique()
        == EXPECTED_ENGINE_COUNT
    ),

    "calibration evidence contains 4104 rows": (
        len(row_output)
        == EXPECTED_ROW_COUNT
    ),

    "finite sample rank remains nineteen": (
        int(
            conformal_bundle[
                "finite_sample_rank_one_based"
            ]
        )
        == EXPECTED_RANK
    ),

    "quantile interpolation remains disabled": (
        conformal_bundle[
            "quantile_interpolation"
        ]
        is None
    ),

    "bundle and record q_hat agree": (
        math.isclose(
            q_hat_bundle,
            q_hat_record,
            rel_tol=0,
            abs_tol=TOLERANCE,
        )
    ),

    "bundle and summary q_hat agree": (
        math.isclose(
            q_hat_bundle,
            q_hat_summary,
            rel_tol=0,
            abs_tol=TOLERANCE,
        )
    ),

    "bundle and nineteenth score agree": (
        math.isclose(
            q_hat_bundle,
            q_hat_scores,
            rel_tol=0,
            abs_tol=TOLERANCE,
        )
    ),

    "row lower intervals reproduce exactly": (
        bool(
            np.allclose(
                saved_lower,
                recomputed_lower,
                rtol=0,
                atol=TOLERANCE,
            )
        )
    ),

    "row upper intervals reproduce exactly": (
        bool(
            np.allclose(
                saved_upper,
                recomputed_upper,
                rtol=0,
                atol=TOLERANCE,
            )
        )
    ),

    "engine maximum scores reproduce exactly": (
        bool(
            np.allclose(
                score_crosscheck[
                    "engine_score"
                ],

                score_crosscheck[
                    "recomputed_engine_score"
                ],

                rtol=0,
                atol=TOLERANCE,
            )
        )
    ),

    "nineteen calibration engines are simultaneously covered": (
        simultaneously_covered_engines
        == EXPECTED_RANK
    ),

    "summary row coverage reproduces": (
        math.isclose(
            row_coverage_rate,

            float(
                summary.loc[
                    0,
                    "calibration_row_coverage_rate",
                ]
            ),

            rel_tol=0,
            abs_tol=TOLERANCE,
        )
    ),

    "summary mean interval width reproduces": (
        math.isclose(
            mean_interval_width,

            float(
                summary.loc[
                    0,
                    "mean_interval_width_cycles",
                ]
            ),

            rel_tol=0,
            abs_tol=TOLERANCE,
        )
    ),

    "calibration caveat rejects independent coverage interpretation": (
        "not an independent estimate"
        in calibration_record[
            "calibration_diagnostics"
        ][
            "interpretation"
        ].lower()
    ),

    "coverage record rejects safety guarantee": (
        calibration_record[
            "coverage_claim"
        ][
            "operational_safety_guarantee"
        ]
        is False
    ),

    "official test remained unused": (
        calibration_record[
            "research_controls"
        ][
            "official_test_data_used"
        ]
        is False
    ),

    "test RUL remained unused": (
        calibration_record[
            "research_controls"
        ][
            "test_rul_values_accessed"
        ]
        is False
    ),

    "validation remained unused": (
        calibration_record[
            "research_controls"
        ][
            "validation_data_used"
        ]
        is False
    ),

    "figure exists and is nonempty": (
        FIGURE_PATH.stat().st_size
        > 0
    ),
}

failed_checks = [
    check_name
    for check_name, passed in checks.items()
    if not passed
]

if failed_checks:
    raise AssertionError(
        "Phase 5 completion validation failed:\n"
        + "\n".join(failed_checks)
    )


# =============================================================================
# 7. WRITE THE PHASE 5 COMPLETION RECORD
# =============================================================================

completion_record = {
    "phase": 5,

    "name": (
        "engine_block_conformal_uncertainty"
    ),

    "status": (
        "technical_requirements_complete_"
        "awaiting_formal_approval"
    ),

    "completed_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),

    "primary_model": {
        "name": EXPECTED_MODEL_NAME,

        "sha256": (
            primary_model_hash_before
        ),

        "retrained_or_modified": False,
    },

    "primary_target": (
        EXPECTED_TARGET
    ),

    "method": {
        "family": (
            "split_conformal_regression"
        ),

        "exchangeability_unit": (
            "complete_engine_trajectory"
        ),

        "row_score": (
            "absolute_residual"
        ),

        "within_engine_aggregation": (
            "maximum"
        ),

        "confidence_level": (
            EXPECTED_CONFIDENCE
        ),

        "miscoverage_alpha": (
            EXPECTED_ALPHA
        ),

        "calibration_engine_count": (
            EXPECTED_ENGINE_COUNT
        ),

        "finite_sample_rank_one_based": (
            EXPECTED_RANK
        ),

        "finite_sample_rank_fraction": (
            EXPECTED_RANK
            / (
                EXPECTED_ENGINE_COUNT
                + 1
            )
        ),

        "quantile_interpolation_used": False,

        "q_hat_cycles": (
            q_hat_bundle
        ),

        "lower_interval_formula": (
            "max(0, point_prediction - q_hat)"
        ),

        "upper_interval_formula": (
            "point_prediction + q_hat"
        ),

        "upper_cap": None,
    },

    "calibration_diagnostics": {
        "calibration_row_count": (
            len(row_output)
        ),

        "row_coverage_rate": (
            row_coverage_rate
        ),

        "simultaneously_covered_engines": (
            simultaneously_covered_engines
        ),

        "engine_simultaneous_coverage_rate": (
            simultaneously_covered_engines
            / EXPECTED_ENGINE_COUNT
        ),

        "lower_bound_clipped_rows": (
            lower_clipped_rows
        ),

        "mean_interval_width_cycles": (
            mean_interval_width
        ),

        "median_interval_width_cycles": (
            median_interval_width
        ),
    },

    "required_interpretation": {
        "coverage_target": (
            "simultaneous_complete_engine_trajectory"
        ),

        "conditional_on": (
            "exchangeability_of_complete_engines"
        ),

        "calibration_coverage_is_independent_test": (
            False
        ),

        "operational_safety_guarantee": False,

        "business_maintenance_window": False,

        "small_sample_caveat": (
            "Only 20 calibration engines were "
            "available, giving a coarse nineteenth "
            "order statistic."
        ),

        "width_caveat": (
            "Maximum-within-engine scoring produced "
            "a conservative q_hat and wide intervals. "
            "The result must not be described as a "
            "precise maintenance window."
        ),
    },

    "artifact_integrity": {
        "policy_sha256": (
            policy_hash_before
        ),

        "calibration_source_sha256": (
            calibration_source_hash_before
        ),

        "conformal_bundle_sha256": (
            conformal_bundle_hash_before
        ),
    },

    "research_controls": {
        "validation_data_used_in_phase5": False,
        "official_test_data_used": False,
        "test_rul_values_accessed": False,
        "capped_130_target_used": False,

        "alternative_method_selected_after_calibration": (
            False
        ),

        "business_thresholds_used": False,
        "shap_analysis_started": False,

        "n8n_operational_validation_started": (
            False
        ),
    },

    "next_authorised_step": (
        "Freeze the official FD001 endpoint-test "
        "evaluation policy before opening "
        "test_FD001.txt or RUL_FD001.txt."
    ),

    "software": {
        "python": (
            platform.python_version()
        ),

        "numpy": (
            np.__version__
        ),

        "pandas": (
            pd.__version__
        ),

        "joblib": (
            joblib.__version__
        ),
    },
}

write_json(
    COMPLETION_RECORD_PATH,
    completion_record,
)


# =============================================================================
# 8. BUILD COMPLETE PHASE 5 MANIFEST
# =============================================================================

artifact_entries = [
    (
        "phase4_reference",
        FINAL_REGISTRY_PATH,
    ),

    (
        "partition_metadata",
        PARTITION_MANIFEST_PATH,
    ),

    (
        "phase5_policy",
        POLICY_PATH,
    ),

    (
        "phase5_policy_decision",
        DECISION_PATH,
    ),

    (
        "phase5_step1_manifest",
        STEP1_MANIFEST_PATH,
    ),

    (
        "phase5_step1_zip",
        STEP1_ZIP_PATH,
    ),

    (
        "phase5_conformal_bundle",
        CONFORMAL_BUNDLE_PATH,
    ),

    (
        "phase5_engine_scores",
        ENGINE_SCORE_PATH,
    ),

    (
        "phase5_row_intervals",
        ROW_OUTPUT_PATH,
    ),

    (
        "phase5_summary",
        SUMMARY_PATH,
    ),

    (
        "phase5_figure",
        FIGURE_PATH,
    ),

    (
        "phase5_calibration_record",
        CALIBRATION_RECORD_PATH,
    ),

    (
        "phase5_step2_manifest",
        STEP2_MANIFEST_PATH,
    ),

    (
        "phase5_step2_zip",
        STEP2_ZIP_PATH,
    ),

    (
        "phase5_completion_record",
        COMPLETION_RECORD_PATH,
    ),
]

if OPTIONAL_CORRECTION_RECORD_PATH.is_file():
    artifact_entries.append(
        (
            "phase5_optional_figure_correction",
            OPTIONAL_CORRECTION_RECORD_PATH,
        )
    )

manifest_rows = []

for role, path in artifact_entries:
    manifest_rows.append(
        {
            "role": role,

            "path": (
                str(path)
            ),

            "relative_path": (
                str(
                    path.relative_to(
                        PROJECT_ROOT
                    )
                )
            ),

            "size_bytes": (
                path.stat().st_size
            ),

            "sha256": (
                sha256_file(path)
            ),
        }
    )

with COMPLETE_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
    newline="",
) as file_handle:

    writer = csv.DictWriter(
        file_handle,
        fieldnames=[
            "role",
            "path",
            "relative_path",
            "size_bytes",
            "sha256",
        ],
    )

    writer.writeheader()
    writer.writerows(manifest_rows)


# =============================================================================
# 9. BUILD COMPLETE PHASE 5 ZIP
# =============================================================================

with zipfile.ZipFile(
    COMPLETE_ZIP_PATH,
    "w",
    zipfile.ZIP_DEFLATED,
) as archive:

    for row in manifest_rows:
        source_path = Path(
            row["path"]
        )

        archive.write(
            source_path,
            arcname=row[
                "relative_path"
            ],
        )

    archive.write(
        COMPLETE_MANIFEST_PATH,

        arcname=str(
            COMPLETE_MANIFEST_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
    )

complete_zip_hash = sha256_file(
    COMPLETE_ZIP_PATH
)


# =============================================================================
# 10. POST-PACKAGING INTEGRITY CHECKS
# =============================================================================

primary_model_hash_after = (
    sha256_file(
        PRIMARY_MODEL_PATH
    )
)

policy_hash_after = (
    sha256_file(
        POLICY_PATH
    )
)

conformal_bundle_hash_after = (
    sha256_file(
        CONFORMAL_BUNDLE_PATH
    )
)

calibration_source_hash_after = (
    sha256_file(
        CALIBRATION_SOURCE_PATH
    )
)

post_packaging_checks = {
    "primary Random Forest remained unchanged": (
        primary_model_hash_before
        == primary_model_hash_after
        == EXPECTED_PRIMARY_MODEL_SHA256
    ),

    "frozen conformal policy remained unchanged": (
        policy_hash_before
        == policy_hash_after
    ),

    "conformal bundle remained unchanged": (
        conformal_bundle_hash_before
        == conformal_bundle_hash_after
    ),

    "calibration source remained unchanged": (
        calibration_source_hash_before
        == calibration_source_hash_after
    ),

    "completion record exists": (
        COMPLETION_RECORD_PATH.is_file()
    ),

    "complete manifest contains every collected artifact": (
        len(manifest_rows)
        == len(artifact_entries)
    ),

    "all manifest hashes are populated": all(
        isinstance(
            row["sha256"],
            str,
        )
        and len(
            row["sha256"]
        )
        == 64
        for row in manifest_rows
    ),

    "complete evidence ZIP exists": (
        COMPLETE_ZIP_PATH.is_file()
    ),
}

failed_post_checks = [
    check_name
    for check_name, passed
    in post_packaging_checks.items()
    if not passed
]

if failed_post_checks:
    raise AssertionError(
        "Phase 5 packaging validation failed:\n"
        + "\n".join(failed_post_checks)
    )


# =============================================================================
# 11. BOUNDED COMPLETION OUTPUT
# =============================================================================

print("PHASE 5 COMPLETION SUMMARY")
print("=" * 90)
print(
    f"Primary model: "
    f"{EXPECTED_MODEL_NAME}"
)
print(
    f"Primary target: "
    f"{EXPECTED_TARGET}"
)
print(
    f"Confidence level: "
    f"{EXPECTED_CONFIDENCE:.0%}"
)
print(
    "Calibration unit: "
    "complete engine trajectory"
)
print(
    f"Calibration engines: "
    f"{EXPECTED_ENGINE_COUNT}"
)
print(
    f"Calibration rows: "
    f"{len(row_output)}"
)
print(
    f"Finite-sample rank: "
    f"{EXPECTED_RANK} of "
    f"{EXPECTED_ENGINE_COUNT}"
)
print(
    f"q_hat: "
    f"{q_hat_bundle:.6f} cycles"
)
print(
    f"Row coverage diagnostic: "
    f"{row_coverage_rate:.4%}"
)
print(
    "Simultaneously covered "
    "calibration engines: "
    f"{simultaneously_covered_engines} "
    f"of {EXPECTED_ENGINE_COUNT}"
)
print(
    f"Mean interval width: "
    f"{mean_interval_width:.6f} cycles"
)
print(
    "Rows with lower bound "
    "clipped to zero: "
    f"{lower_clipped_rows}"
)

print("\nPHASE 5 VALIDATION")
print("=" * 90)

for (
    check_name,
    passed,
) in checks.items():

    status = (
        "PASS"
        if passed
        else "FAIL"
    )

    print(
        f"{status} — "
        f"{check_name}"
    )

for (
    check_name,
    passed,
) in post_packaging_checks.items():

    status = (
        "PASS"
        if passed
        else "FAIL"
    )

    print(
        f"{status} — "
        f"{check_name}"
    )

print("\nINTERPRETATION CONTROL")
print("=" * 90)
print(
    "Calibration coverage is an independent "
    "unseen-engine evaluation: No"
)
print(
    "Operational safety guarantee: No"
)
print(
    "Precise business maintenance window: No"
)
print(
    "Small calibration sample: "
    "Yes — 20 engines"
)
print(
    "Conservative maximum-within-engine "
    "score: Yes"
)
print(
    "Wide intervals must be reported "
    "as a limitation: Yes"
)

print("\nRESEARCH CONTROLS")
print("=" * 90)
print(
    "Random Forest retrained "
    "or modified: No"
)
print(
    "Conformal method changed "
    "after calibration: No"
)
print(
    "Validation data used "
    "in Phase 5: No"
)
print("Official test data used: No")
print("Test RUL values accessed: No")
print("Capped-130 target used: No")
print("Business thresholds used: No")
print("SHAP analysis started: No")
print(
    "n8n operational validation "
    "started: No"
)

print("\nCOMPLETE EVIDENCE PACKAGE")
print("=" * 90)
print(
    f"Files fingerprinted: "
    f"{len(manifest_rows)}"
)
print(
    f"Manifest: "
    f"{COMPLETE_MANIFEST_PATH}"
)
print(
    f"ZIP retained in Colab: "
    f"{COMPLETE_ZIP_PATH}"
)
print(
    f"ZIP SHA-256: "
    f"{complete_zip_hash}"
)
print(
    "Browser download triggered: No"
)

print("\nFINAL CHECK")
print("=" * 90)
print(
    "PASS — Phase 5 technical "
    "requirements are complete."
)
print(
    "PASS — Engine-block conformal "
    "uncertainty was calibrated reproducibly."
)
print(
    "PASS — q_hat and every interval "
    "were reproduced from saved evidence."
)
print(
    "PASS — The conservative interval "
    "limitation was formally recorded."
)
print(
    "PASS — Every Phase 5 evidence "
    "artifact was fingerprinted."
)
print(
    "PASS — Validation and official "
    "test data remained unused."
)
print(
    "AWAITING APPROVAL — Phase 5 "
    "is not yet formally closed."
)

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import math
import zipfile

import joblib


# =============================================================================
# PHASE 6 — STEP 1
# FREEZE THE OFFICIAL FD001 ENDPOINT-TEST POLICY
#
# This cell:
# - Fingerprints the official test files.
# - Does not parse test predictors or test RUL values.
# - Does not generate predictions or metrics.
# - Does not retrain, recalibrate or reselect a model.
# - Does not trigger a browser download.
# =============================================================================

PROJECT_ROOT = Path(
    "/content/Predictive_Maintenance_Thesis"
)

RAW_FD001_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "FD001"
)

LOG_DIR = (
    PROJECT_ROOT
    / "logs"
)

MODEL_DIR = (
    PROJECT_ROOT
    / "models"
)


# =============================================================================
# ARTIFACT PATHS
# =============================================================================

TEST_PATH = (
    RAW_FD001_DIR
    / "test_FD001.txt"
)

TEST_RUL_PATH = (
    RAW_FD001_DIR
    / "RUL_FD001.txt"
)

PRIMARY_MODEL_PATH = (
    MODEL_DIR
    / "regression"
    / "fd001_random_forest_uncapped_bundle.joblib"
)

CONFORMAL_BUNDLE_PATH = (
    MODEL_DIR
    / "uncertainty"
    / (
        "fd001_random_forest_"
        "engine_block_conformal_bundle.joblib"
    )
)

FINAL_REGISTRY_PATH = (
    LOG_DIR
    / "phase4_final_model_registry.json"
)

PHASE5_COMPLETION_PATH = (
    LOG_DIR
    / "phase5_technical_completion_record.json"
)

PHASE5_MANIFEST_PATH = (
    LOG_DIR
    / "phase5_complete_evidence_manifest.csv"
)

PHASE5_ZIP_PATH = (
    PROJECT_ROOT
    / "Phase5_Complete_Uncertainty_Evidence.zip"
)

POLICY_PATH = (
    LOG_DIR
    / "phase6_official_test_evaluation_policy.json"
)

DECISION_PATH = (
    LOG_DIR
    / "phase6_official_test_evaluation_decision.md"
)

EVIDENCE_MANIFEST_PATH = (
    LOG_DIR
    / "phase6_step1_test_policy_manifest.csv"
)

EVIDENCE_ZIP_PATH = (
    PROJECT_ROOT
    / "Phase6_Step1_Official_Test_Policy_Evidence.zip"
)


# =============================================================================
# LOCKED EXPECTATIONS
# =============================================================================

EXPECTED_PRIMARY_MODEL_SHA256 = (
    "99620a1a161f62a716d4cc058497159146197e7d91bba713581c822af808e5d5"
)

EXPECTED_TEST_SHA256 = (
    "3cda7109ce17bafb5443f2ac926cfcf88154b941b8c4cf95eb55d1ddd6f52851"
)

EXPECTED_TEST_RUL_SHA256 = (
    "a19c8ec94931949d0485bdc35118206e9c81c4547b422efb9cf86f4ceddbceca"
)

EXPECTED_TEST_SIZE_BYTES = (
    2_228_855
)

EXPECTED_TEST_RUL_SIZE_BYTES = (
    429
)

EXPECTED_TEST_ENGINES = 100
EXPECTED_FEATURE_COUNT = 18
EXPECTED_CONFIDENCE = 0.90
EXPECTED_ALPHA = 0.10

EXPECTED_Q_HAT = (
    107.896520
)


# =============================================================================
# SUPPORT FUNCTIONS
# =============================================================================

def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        for chunk in iter(
            lambda: file_handle.read(
                chunk_size
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def load_json(
    path: Path,
) -> dict:
    with path.open(
        "r",
        encoding="utf-8",
    ) as file_handle:
        return json.load(file_handle)


def write_json(
    path: Path,
    payload: dict,
) -> None:
    with path.open(
        "w",
        encoding="utf-8",
    ) as file_handle:

        json.dump(
            payload,
            file_handle,
            indent=2,
            ensure_ascii=False,
        )

        file_handle.write("\n")


# =============================================================================
# 1. VERIFY ALL REQUIRED ARTIFACTS
# =============================================================================

required_paths = [
    TEST_PATH,
    TEST_RUL_PATH,
    PRIMARY_MODEL_PATH,
    CONFORMAL_BUNDLE_PATH,
    FINAL_REGISTRY_PATH,
    PHASE5_COMPLETION_PATH,
    PHASE5_MANIFEST_PATH,
    PHASE5_ZIP_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required approved artifacts are missing:\n"
        + "\n".join(missing_paths)
    )


# =============================================================================
# 2. FINGERPRINT SEALED TEST FILES
# =============================================================================

primary_model_hash_before = (
    sha256_file(
        PRIMARY_MODEL_PATH
    )
)

conformal_bundle_hash_before = (
    sha256_file(
        CONFORMAL_BUNDLE_PATH
    )
)

test_hash_before = (
    sha256_file(
        TEST_PATH
    )
)

test_rul_hash_before = (
    sha256_file(
        TEST_RUL_PATH
    )
)

phase5_zip_hash = (
    sha256_file(
        PHASE5_ZIP_PATH
    )
)

if (
    primary_model_hash_before
    != EXPECTED_PRIMARY_MODEL_SHA256
):
    raise RuntimeError(
        "The frozen Random Forest hash "
        "does not match Phase 4."
    )

if (
    test_hash_before
    != EXPECTED_TEST_SHA256
):
    raise RuntimeError(
        "test_FD001.txt does not match "
        "the protected Phase 1 file."
    )

if (
    test_rul_hash_before
    != EXPECTED_TEST_RUL_SHA256
):
    raise RuntimeError(
        "RUL_FD001.txt does not match "
        "the protected Phase 1 file."
    )


# =============================================================================
# 3. LOAD APPROVED POLICY METADATA
# =============================================================================

registry = load_json(
    FINAL_REGISTRY_PATH
)

phase5_completion = load_json(
    PHASE5_COMPLETION_PATH
)

conformal_bundle = joblib.load(
    CONFORMAL_BUNDLE_PATH
)

registry_text = json.dumps(
    registry,
    sort_keys=True,
).lower()

feature_order = list(
    conformal_bundle[
        "feature_order"
    ]
)

q_hat = float(
    conformal_bundle[
        "q_hat"
    ]
)


# =============================================================================
# 4. FREEZE OFFICIAL-TEST EVALUATION POLICY
# =============================================================================

policy = {
    "phase": 6,

    "step": (
        "1_official_endpoint_test_policy_freeze"
    ),

    "status": (
        "frozen_before_official_test_values_parsed"
    ),

    "created_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),

    "study_population": {
        "dataset": (
            "NASA_C_MAPSS_FD001_official_test"
        ),

        "expected_engine_count": (
            EXPECTED_TEST_ENGINES
        ),

        "evaluation_grain": (
            "one_endpoint_record_per_test_engine"
        ),

        "endpoint_definition": (
            "row_with_maximum_time_in_cycles_"
            "for_each_unit_number"
        ),

        "outcome_alignment": (
            "RUL_FD001 line i maps to "
            "unit_number i for engines "
            "1 through 100"
        ),

        "test_predictor_path": (
            str(TEST_PATH)
        ),

        "test_predictor_sha256": (
            test_hash_before
        ),

        "test_rul_path": (
            str(TEST_RUL_PATH)
        ),

        "test_rul_sha256": (
            test_rul_hash_before
        ),

        "test_predictor_values_parsed": False,

        "test_rul_values_parsed": False,
    },

    "frozen_point_model": {
        "name": (
            "random_forest_uncapped"
        ),

        "target": (
            "rul_uncapped"
        ),

        "bundle_path": (
            str(PRIMARY_MODEL_PATH)
        ),

        "bundle_sha256": (
            primary_model_hash_before
        ),

        "feature_count": (
            len(feature_order)
        ),

        "feature_order": (
            feature_order
        ),

        "input_scaling": "none",

        "prediction_lower_clip": 0,

        "retraining_permitted": False,

        "test_time_adaptation_permitted": False,

        "model_reselection_permitted": False,
    },

    "frozen_uncertainty_model": {
        "bundle_path": (
            str(CONFORMAL_BUNDLE_PATH)
        ),

        "bundle_sha256": (
            conformal_bundle_hash_before
        ),

        "method": (
            "engine_block_split_conformal"
        ),

        "confidence_level": (
            EXPECTED_CONFIDENCE
        ),

        "miscoverage_alpha": (
            EXPECTED_ALPHA
        ),

        "q_hat_cycles": (
            q_hat
        ),

        "interval_lower": (
            "max(0, point_prediction - q_hat)"
        ),

        "interval_upper": (
            "point_prediction + q_hat"
        ),

        "upper_cap": None,

        "recalibration_permitted": False,
    },

    "point_metrics": {
        "mae": (
            "mean(abs(prediction "
            "- actual_endpoint_rul))"
        ),

        "rmse": (
            "sqrt(mean((prediction "
            "- actual_endpoint_rul)^2))"
        ),

        "r_squared": (
            "1 - sum_squared_error "
            "/ total_sum_of_squares"
        ),

        "bias": (
            "mean(prediction "
            "- actual_endpoint_rul)"
        ),

        "median_absolute_error": (
            "median(abs(prediction "
            "- actual_endpoint_rul))"
        ),

        "overestimation_rate_percent": (
            "100 * mean(prediction > actual)"
        ),

        "underestimation_rate_percent": (
            "100 * mean(prediction < actual)"
        ),

        "within_10_cycles_rate_percent": (
            "100 * mean(abs(error) <= 10)"
        ),

        "within_20_cycles_rate_percent": (
            "100 * mean(abs(error) <= 20)"
        ),

        "within_30_cycles_rate_percent": (
            "100 * mean(abs(error) <= 30)"
        ),
    },

    "phm_score": {
        "error_definition": (
            "d = prediction "
            "- actual_endpoint_rul"
        ),

        "underestimation_term": (
            "exp(-d / 13) - 1 "
            "when d < 0"
        ),

        "overestimation_term": (
            "exp(d / 10) - 1 "
            "when d >= 0"
        ),

        "reported_aggregations": [
            "total",
            "mean",
        ],

        "interpretation": (
            "lower_is_better_and_"
            "overestimation_is_penalised_more"
        ),
    },

    "interval_metrics": {
        "endpoint_coverage_count": (
            "count(lower <= actual <= upper)"
        ),

        "endpoint_coverage_rate": (
            "mean(lower <= actual <= upper)"
        ),

        "coverage_confidence_interval": (
            "two_sided_95_percent_"
            "Wilson_interval"
        ),

        "coverage_z_value": (
            1.959963984540054
        ),

        "mean_interval_width": (
            "mean(upper - lower)"
        ),

        "median_interval_width": (
            "median(upper - lower)"
        ),

        "lower_clipping_rate": (
            "mean(point_prediction "
            "- q_hat < 0)"
        ),

        "miss_below_count": (
            "count(actual < lower)"
        ),

        "miss_above_count": (
            "count(actual > upper)"
        ),

        "interval_score_alpha_0_10": (
            "width + "
            "(2/alpha)*(lower-actual)"
            "*I(actual<lower) + "
            "(2/alpha)*(actual-upper)"
            "*I(actual>upper)"
        ),

        "reported_interval_score_aggregations": [
            "mean",
            "median",
        ],
    },

    "predefined_lifecycle_stages": [
        {
            "name": (
                "critical_0_to_30"
            ),

            "condition": (
                "0 <= actual_endpoint_rul <= 30"
            ),
        },

        {
            "name": (
                "near_term_31_to_60"
            ),

            "condition": (
                "30 < actual_endpoint_rul <= 60"
            ),
        },

        {
            "name": (
                "mid_life_61_to_130"
            ),

            "condition": (
                "60 < actual_endpoint_rul <= 130"
            ),
        },

        {
            "name": (
                "early_life_above_130"
            ),

            "condition": (
                "actual_endpoint_rul > 130"
            ),
        },
    ],

    "lifecycle_stage_metrics": [
        "engine_count",
        "mae",
        "rmse",
        "bias",
        "coverage_rate",
        "mean_interval_width",
    ],

    "interpretation_controls": {
        "official_test_is_used_once_for_final_evaluation": (
            True
        ),

        "only_selected_primary_model_is_evaluated": (
            True
        ),

        "ridge_test_evaluation_permitted": False,

        "xgboost_test_evaluation_permitted": False,

        "naive_test_evaluation_permitted": False,

        "capped_130_test_evaluation_permitted": (
            False
        ),

        "official_test_results_may_change_selected_model": (
            False
        ),

        "subgroup_results_are_descriptive_only": (
            True
        ),

        "performance_acceptance_threshold_predefined": (
            False
        ),

        "result_must_not_be_presented_as_"
        "operational_safety_guarantee": True,

        "trajectory_calibrated_interval_may_"
        "be_conservative_at_endpoint": True,
    },

    "prohibited_actions_after_unsealing": [
        "model_retraining",
        "hyperparameter_tuning",
        "feature_selection",
        "target_redefinition",
        "conformal_method_change",
        "q_hat_recalibration",
        "model_reselection",
        "test_case_removal",
        (
            "metric_removal_because_"
            "result_is_unfavourable"
        ),
    ],

    "phase5_reference": {
        "completion_record_path": (
            str(PHASE5_COMPLETION_PATH)
        ),

        "complete_manifest_path": (
            str(PHASE5_MANIFEST_PATH)
        ),

        "complete_zip_path": (
            str(PHASE5_ZIP_PATH)
        ),

        "complete_zip_sha256": (
            phase5_zip_hash
        ),
    },

    "next_authorised_step": (
        "Parse the official test predictors "
        "and RUL outcomes once, construct one "
        "endpoint per engine, and apply this "
        "policy without adaptation."
    ),
}

write_json(
    POLICY_PATH,
    policy,
)


# =============================================================================
# 5. WRITE HUMAN-READABLE POLICY DECISION
# =============================================================================

decision_markdown = f"""# FD001 Official Endpoint-Test Evaluation Policy

## Frozen evaluation population

- Official FD001 test engines: 100
- Evaluation grain: one last-observed endpoint per engine
- Target alignment: RUL file line `i` maps to test engine `i`
- Primary model: `random_forest_uncapped`
- Primary-model SHA-256: `{primary_model_hash_before}`
- Predictors: {len(feature_order)} unscaled retained features

## Frozen uncertainty interval

- Confidence level: 90%
- `q_hat`: {q_hat:.6f} cycles
- Lower bound: `max(0, prediction - q_hat)`
- Upper bound: `prediction + q_hat`
- Upper cap: none

## Frozen outputs

Point metrics include MAE, RMSE, R-squared, bias, median absolute error,
overestimation and underestimation rates, accuracy within 10, 20 and 30
cycles, and the NASA asymmetric PHM score.

Uncertainty metrics include endpoint coverage, a 95% Wilson interval for
the coverage proportion, interval width, lower-clipping rate, miss
direction, and the alpha=0.10 interval score.

Lifecycle stages are fixed at 0-30, 31-60, 61-130 and above 130 actual
RUL. These subgroup results are descriptive only.

## Locked controls

Only the already-selected Random Forest will be evaluated. Official test
results cannot trigger model reselection, retraining, feature selection,
hyperparameter tuning, target changes, conformal changes, or q-hat
changes.

No performance acceptance threshold was preregistered. Results will be
reported descriptively and will not be converted into a post-hoc
pass/fail claim.

At policy freeze time, the two official test files were fingerprinted,
but their predictor and outcome values had not been parsed.
"""

with DECISION_PATH.open(
    "w",
    encoding="utf-8",
) as file_handle:
    file_handle.write(
        decision_markdown
    )


# =============================================================================
# 6. RECHECK PROTECTED ARTIFACTS
# =============================================================================

primary_model_hash_after = (
    sha256_file(
        PRIMARY_MODEL_PATH
    )
)

conformal_bundle_hash_after = (
    sha256_file(
        CONFORMAL_BUNDLE_PATH
    )
)

test_hash_after = (
    sha256_file(
        TEST_PATH
    )
)

test_rul_hash_after = (
    sha256_file(
        TEST_RUL_PATH
    )
)


# =============================================================================
# 7. POLICY VALIDATION
# =============================================================================

checks = {
    "Phase 5 is technically complete": (
        phase5_completion[
            "status"
        ]
        == (
            "technical_requirements_complete_"
            "awaiting_formal_approval"
        )
    ),

    "registry selects Random Forest": (
        "random_forest"
        in registry_text
    ),

    "registry preserves approved model hash": (
        EXPECTED_PRIMARY_MODEL_SHA256
        in registry_text
    ),

    "primary model hash is correct": (
        primary_model_hash_after
        == EXPECTED_PRIMARY_MODEL_SHA256
    ),

    "primary model was not modified": (
        primary_model_hash_before
        == primary_model_hash_after
    ),

    "conformal bundle was not modified": (
        conformal_bundle_hash_before
        == conformal_bundle_hash_after
    ),

    "conformal bundle contains 18 features": (
        len(feature_order)
        == EXPECTED_FEATURE_COUNT
    ),

    "unit number is excluded": (
        "unit_number"
        not in feature_order
    ),

    "primary target is excluded": (
        "rul_uncapped"
        not in feature_order
    ),

    "sensitivity target is excluded": (
        "rul_capped_130"
        not in feature_order
    ),

    "conformal confidence remains 90 percent": (
        math.isclose(
            float(
                conformal_bundle[
                    "confidence_level"
                ]
            ),
            EXPECTED_CONFIDENCE,
            rel_tol=0,
            abs_tol=1e-12,
        )
    ),

    "conformal q_hat is preserved": (
        math.isclose(
            q_hat,
            EXPECTED_Q_HAT,
            rel_tol=0,
            abs_tol=1e-6,
        )
    ),

    "test predictor hash matches protected source": (
        test_hash_after
        == EXPECTED_TEST_SHA256
    ),

    "test RUL hash matches protected source": (
        test_rul_hash_after
        == EXPECTED_TEST_RUL_SHA256
    ),

    "test predictor size matches protected source": (
        TEST_PATH.stat().st_size
        == EXPECTED_TEST_SIZE_BYTES
    ),

    "test RUL size matches protected source": (
        TEST_RUL_PATH.stat().st_size
        == EXPECTED_TEST_RUL_SIZE_BYTES
    ),

    "test predictor file was not modified": (
        test_hash_before
        == test_hash_after
    ),

    "test RUL file was not modified": (
        test_rul_hash_before
        == test_rul_hash_after
    ),

    "test predictor values were not parsed": (
        policy[
            "study_population"
        ][
            "test_predictor_values_parsed"
        ]
        is False
    ),

    "test RUL values were not parsed": (
        policy[
            "study_population"
        ][
            "test_rul_values_parsed"
        ]
        is False
    ),

    "only selected model will be evaluated": (
        policy[
            "interpretation_controls"
        ][
            "only_selected_primary_model_is_evaluated"
        ]
        is True
    ),

    "official results cannot reselect model": (
        policy[
            "interpretation_controls"
        ][
            "official_test_results_may_change_selected_model"
        ]
        is False
    ),

    "policy file exists": (
        POLICY_PATH.is_file()
    ),

    "decision record exists": (
        DECISION_PATH.is_file()
    ),
}

failed_checks = [
    check_name
    for check_name, passed in checks.items()
    if not passed
]

if failed_checks:
    raise AssertionError(
        "Official-test policy validation failed:\n"
        + "\n".join(failed_checks)
    )


# =============================================================================
# 8. BUILD POLICY EVIDENCE MANIFEST
# =============================================================================

manifest_entries = [
    (
        "phase5_completion_reference",
        PHASE5_COMPLETION_PATH,
        True,
    ),

    (
        "phase5_manifest_reference",
        PHASE5_MANIFEST_PATH,
        True,
    ),

    (
        "final_model_registry",
        FINAL_REGISTRY_PATH,
        True,
    ),

    (
        "primary_model_hash_reference",
        PRIMARY_MODEL_PATH,
        False,
    ),

    (
        "conformal_bundle_hash_reference",
        CONFORMAL_BUNDLE_PATH,
        False,
    ),

    (
        "sealed_test_predictor_hash_reference",
        TEST_PATH,
        False,
    ),

    (
        "sealed_test_rul_hash_reference",
        TEST_RUL_PATH,
        False,
    ),

    (
        "phase6_policy",
        POLICY_PATH,
        True,
    ),

    (
        "phase6_decision",
        DECISION_PATH,
        True,
    ),
]

manifest_rows = []

for (
    role,
    path,
    include_in_zip,
) in manifest_entries:

    manifest_rows.append(
        {
            "role": role,
            "path": str(path),

            "size_bytes": (
                path.stat().st_size
            ),

            "sha256": (
                sha256_file(path)
            ),

            "included_in_step_zip": (
                include_in_zip
            ),
        }
    )

with EVIDENCE_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
    newline="",
) as file_handle:

    writer = csv.DictWriter(
        file_handle,
        fieldnames=[
            "role",
            "path",
            "size_bytes",
            "sha256",
            "included_in_step_zip",
        ],
    )

    writer.writeheader()
    writer.writerows(manifest_rows)


# =============================================================================
# 9. CREATE RETAINED POLICY ZIP
# =============================================================================

with zipfile.ZipFile(
    EVIDENCE_ZIP_PATH,
    "w",
    zipfile.ZIP_DEFLATED,
) as archive:

    for row in manifest_rows:
        if row[
            "included_in_step_zip"
        ]:
            source_path = Path(
                row["path"]
            )

            archive.write(
                source_path,

                arcname=str(
                    source_path.relative_to(
                        PROJECT_ROOT
                    )
                ),
            )

    archive.write(
        EVIDENCE_MANIFEST_PATH,

        arcname=str(
            EVIDENCE_MANIFEST_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
    )

policy_hash = sha256_file(
    POLICY_PATH
)

evidence_zip_hash = sha256_file(
    EVIDENCE_ZIP_PATH
)


# =============================================================================
# 10. BOUNDED REVIEW OUTPUT
# =============================================================================

print("PHASE 5 CLOSURE")
print("=" * 90)
print(
    "PASS — Phase 5 is formally "
    "approved and closed"
)
print(
    f"Phase 5 complete ZIP SHA-256: "
    f"{phase5_zip_hash}"
)

print("\nOFFICIAL TEST FOUNDATION")
print("=" * 90)
print(
    "Selected model: "
    "random_forest_uncapped"
)
print(
    f"Primary model SHA-256: "
    f"{primary_model_hash_after}"
)
print(
    f"Conformal bundle SHA-256: "
    f"{conformal_bundle_hash_after}"
)
print(
    f"Frozen q_hat: "
    f"{q_hat:.6f} cycles"
)
print(
    f"Expected official test engines: "
    f"{EXPECTED_TEST_ENGINES}"
)
print(
    "Evaluation grain: one "
    "last-observed endpoint per engine"
)

print("\nFROZEN OFFICIAL TEST METRICS")
print("=" * 90)
print(
    "Point metrics: MAE, RMSE, R-squared, "
    "bias and median absolute error"
)
print(
    "Accuracy bands: within 10, 20 "
    "and 30 cycles"
)
print(
    "Asymmetric diagnostic: NASA PHM "
    "total and mean penalty"
)
print(
    "Interval metrics: endpoint coverage "
    "and 95% Wilson interval"
)
print(
    "Interval diagnostics: width, clipping, "
    "miss direction and interval score"
)
print(
    "Lifecycle stages: 0-30, 31-60, "
    "61-130 and above 130 cycles"
)

print("\nPOLICY VALIDATION")
print("=" * 90)

for (
    check_name,
    passed,
) in checks.items():

    status = (
        "PASS"
        if passed
        else "FAIL"
    )

    print(
        f"{status} — "
        f"{check_name}"
    )

print("\nRESEARCH CONTROLS")
print("=" * 90)
print(
    "Test predictor values parsed: No"
)
print(
    "Test RUL values parsed: No"
)
print("Predictions generated: No")
print(
    "Official test metrics calculated: No"
)
print(
    "Alternative models permitted "
    "on official test: No"
)
print(
    "Model retraining or reselection "
    "permitted: No"
)
print(
    "Conformal recalibration permitted: No"
)
print(
    "Post-hoc acceptance threshold "
    "permitted: No"
)

print("\nOUTPUT FILES")
print("=" * 90)
print(
    f"Policy: "
    f"{POLICY_PATH}"
)
print(
    f"Decision record: "
    f"{DECISION_PATH}"
)
print(
    f"Evidence manifest: "
    f"{EVIDENCE_MANIFEST_PATH}"
)
print(
    f"Evidence ZIP retained in Colab: "
    f"{EVIDENCE_ZIP_PATH}"
)
print(
    f"Policy SHA-256: "
    f"{policy_hash}"
)
print(
    f"Evidence ZIP SHA-256: "
    f"{evidence_zip_hash}"
)
print(
    "Browser download triggered: No"
)

print("\nFINAL CHECK")
print("=" * 90)
print(
    "PASS — The official FD001 endpoint-test "
    "policy was frozen successfully."
)
print(
    "PASS — Raw test files match their "
    "protected Phase 1 fingerprints."
)
print(
    "PASS — Test predictor and RUL values "
    "have not been parsed."
)
print(
    "PASS — The selected Random Forest "
    "and conformal interval are immutable."
)
print(
    "PASS — Official test results cannot "
    "change model selection."
)
print(
    "AWAITING APPROVAL — Official test "
    "outcomes remain sealed."
)

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import math
import zipfile

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# =============================================================================
# PHASE 6 — STEP 2
# ONE-TIME OFFICIAL FD001 ENDPOINT EVALUATION
#
# Only the frozen Random Forest and frozen conformal interval are evaluated.
# No alternative models, retraining, recalibration or reselection are allowed.
# =============================================================================

PROJECT_ROOT = Path(
    "/content/Predictive_Maintenance_Thesis"
)

RAW_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "FD001"
)

LOG_DIR = (
    PROJECT_ROOT
    / "logs"
)

RESULT_DIR = (
    PROJECT_ROOT
    / "results"
)

FIGURE_DIR = (
    RESULT_DIR
    / "figures"
)

MODEL_DIR = (
    PROJECT_ROOT
    / "models"
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# =============================================================================
# PATHS
# =============================================================================

TEST_PATH = (
    RAW_DIR
    / "test_FD001.txt"
)

TEST_RUL_PATH = (
    RAW_DIR
    / "RUL_FD001.txt"
)

POLICY_PATH = (
    LOG_DIR
    / "phase6_official_test_evaluation_policy.json"
)

FINAL_REGISTRY_PATH = (
    LOG_DIR
    / "phase4_final_model_registry.json"
)

PRIMARY_MODEL_PATH = (
    MODEL_DIR
    / "regression"
    / "fd001_random_forest_uncapped_bundle.joblib"
)

CONFORMAL_BUNDLE_PATH = (
    MODEL_DIR
    / "uncertainty"
    / (
        "fd001_random_forest_"
        "engine_block_conformal_bundle.joblib"
    )
)

ENDPOINT_RESULTS_PATH = (
    RESULT_DIR
    / "phase6_official_test_endpoint_predictions.csv"
)

SUMMARY_METRICS_PATH = (
    RESULT_DIR
    / "phase6_official_test_summary_metrics.csv"
)

LIFECYCLE_METRICS_PATH = (
    RESULT_DIR
    / "phase6_official_test_lifecycle_stage_metrics.csv"
)

FIGURE_PATH = (
    FIGURE_DIR
    / "phase6_official_test_endpoint_evaluation.png"
)

EVALUATION_RECORD_PATH = (
    LOG_DIR
    / "phase6_official_test_evaluation_record.json"
)

EVIDENCE_MANIFEST_PATH = (
    LOG_DIR
    / "phase6_step2_official_test_manifest.csv"
)

EVIDENCE_ZIP_PATH = (
    PROJECT_ROOT
    / "Phase6_Step2_Official_Test_Evidence.zip"
)


# =============================================================================
# LOCKED EXPECTATIONS
# =============================================================================

EXPECTED_MODEL_SHA256 = (
    "99620a1a161f62a716d4cc058497159146197e7d91bba713581c822af808e5d5"
)

EXPECTED_TEST_SHA256 = (
    "3cda7109ce17bafb5443f2ac926cfcf88154b941b8c4cf95eb55d1ddd6f52851"
)

EXPECTED_TEST_RUL_SHA256 = (
    "a19c8ec94931949d0485bdc35118206e9c81c4547b422efb9cf86f4ceddbceca"
)

EXPECTED_TEST_ROWS = 13_096
EXPECTED_TEST_ENGINES = 100
EXPECTED_COLUMNS = 26
EXPECTED_FEATURES = 18

TOLERANCE = 1e-10

TEST_COLUMNS = (
    [
        "unit_number",
        "time_in_cycles",
    ]
    + [
        f"operational_setting_{index}"
        for index in range(1, 4)
    ]
    + [
        f"sensor_{index}"
        for index in range(1, 22)
    ]
)

LIFECYCLE_ORDER = [
    "critical_0_to_30",
    "near_term_31_to_60",
    "mid_life_61_to_130",
    "early_life_above_130",
]


# =============================================================================
# FUNCTIONS
# =============================================================================

def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        for chunk in iter(
            lambda: file_handle.read(
                chunk_size
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def dataframe_hash(
    dataframe: pd.DataFrame,
) -> str:
    digest = hashlib.sha256()

    digest.update(
        pd.util.hash_pandas_object(
            dataframe,
            index=True,
        ).values.tobytes()
    )

    digest.update(
        "|".join(
            map(
                str,
                dataframe.columns,
            )
        ).encode("utf-8")
    )

    digest.update(
        "|".join(
            map(
                str,
                dataframe.dtypes,
            )
        ).encode("utf-8")
    )

    return digest.hexdigest()


def load_json(
    path: Path,
) -> dict:
    with path.open(
        "r",
        encoding="utf-8",
    ) as file_handle:
        return json.load(file_handle)


def write_json(
    path: Path,
    payload: dict,
) -> None:
    with path.open(
        "w",
        encoding="utf-8",
    ) as file_handle:

        json.dump(
            payload,
            file_handle,
            indent=2,
            ensure_ascii=False,
        )

        file_handle.write("\n")


def wilson_interval(
    successes: int,
    total: int,
    z_value: float,
) -> tuple[float, float]:
    proportion = (
        successes
        / total
    )

    denominator = (
        1.0
        + (
            z_value**2
            / total
        )
    )

    centre = (
        proportion
        + (
            z_value**2
            / (
                2.0
                * total
            )
        )
    ) / denominator

    margin = (
        z_value
        * math.sqrt(
            (
                proportion
                * (
                    1.0
                    - proportion
                )
                / total
            )
            + (
                z_value**2
                / (
                    4.0
                    * total**2
                )
            )
        )
        / denominator
    )

    return (
        centre - margin,
        centre + margin,
    )


def assign_lifecycle_stage(
    actual_rul: np.ndarray,
) -> np.ndarray:
    return np.select(
        [
            (
                actual_rul >= 0
            )
            & (
                actual_rul <= 30
            ),

            (
                actual_rul > 30
            )
            & (
                actual_rul <= 60
            ),

            (
                actual_rul > 60
            )
            & (
                actual_rul <= 130
            ),

            actual_rul > 130,
        ],

        LIFECYCLE_ORDER,

        default="unassigned",
    )


# =============================================================================
# 1. VERIFY THE FROZEN POLICY BEFORE OPENING OUTCOMES
# =============================================================================

required_paths = [
    TEST_PATH,
    TEST_RUL_PATH,
    POLICY_PATH,
    FINAL_REGISTRY_PATH,
    PRIMARY_MODEL_PATH,
    CONFORMAL_BUNDLE_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required official-test artifacts "
        "are missing:\n"
        + "\n".join(missing_paths)
    )

policy_hash_before = (
    sha256_file(
        POLICY_PATH
    )
)

model_hash_before = (
    sha256_file(
        PRIMARY_MODEL_PATH
    )
)

conformal_hash_before = (
    sha256_file(
        CONFORMAL_BUNDLE_PATH
    )
)

test_hash_before = (
    sha256_file(
        TEST_PATH
    )
)

test_rul_hash_before = (
    sha256_file(
        TEST_RUL_PATH
    )
)

policy = load_json(
    POLICY_PATH
)

registry = load_json(
    FINAL_REGISTRY_PATH
)

registry_text = json.dumps(
    registry,
    sort_keys=True,
).lower()

conformal_bundle = joblib.load(
    CONFORMAL_BUNDLE_PATH
)

policy_population = (
    policy[
        "study_population"
    ]
)

policy_point_model = (
    policy[
        "frozen_point_model"
    ]
)

policy_uncertainty = (
    policy[
        "frozen_uncertainty_model"
    ]
)

policy_controls = (
    policy[
        "interpretation_controls"
    ]
)

feature_order = list(
    conformal_bundle[
        "feature_order"
    ]
)

q_hat = float(
    conformal_bundle[
        "q_hat"
    ]
)

alpha = float(
    conformal_bundle[
        "miscoverage_alpha"
    ]
)

confidence_level = float(
    conformal_bundle[
        "confidence_level"
    ]
)

prediction_lower_bound = float(
    conformal_bundle[
        "prediction_lower_bound"
    ]
)

point_model = (
    conformal_bundle[
        "point_model_bundle"
    ][
        "model"
    ]
)


# =============================================================================
# 2. PRE-UNSEALING CHECKS
# =============================================================================

pre_unsealing_checks = {
    "policy is frozen before official values were parsed": (
        policy[
            "status"
        ]
        == (
            "frozen_before_official_"
            "test_values_parsed"
        )
    ),

    "policy selects only Random Forest": (
        policy_point_model[
            "name"
        ]
        == "random_forest_uncapped"
        and policy_controls[
            "only_selected_primary_model_is_evaluated"
        ]
        is True
    ),

    "official results cannot reselect model": (
        policy_controls[
            "official_test_results_may_change_selected_model"
        ]
        is False
    ),

    "registry selects Random Forest": (
        "random_forest"
        in registry_text
    ),

    "primary model hash matches Phase 4": (
        model_hash_before
        == EXPECTED_MODEL_SHA256
        == policy_point_model[
            "bundle_sha256"
        ]
    ),

    "conformal bundle hash matches policy": (
        conformal_hash_before
        == policy_uncertainty[
            "bundle_sha256"
        ]
    ),

    "test predictor hash matches protected source": (
        test_hash_before
        == EXPECTED_TEST_SHA256
        == policy_population[
            "test_predictor_sha256"
        ]
    ),

    "test RUL hash matches protected source": (
        test_rul_hash_before
        == EXPECTED_TEST_RUL_SHA256
        == policy_population[
            "test_rul_sha256"
        ]
    ),

    "feature count remains eighteen": (
        len(feature_order)
        == EXPECTED_FEATURES
        == int(
            policy_point_model[
                "feature_count"
            ]
        )
    ),

    "feature order matches policy": (
        feature_order
        == policy_point_model[
            "feature_order"
        ]
    ),

    "q_hat matches policy": (
        math.isclose(
            q_hat,

            float(
                policy_uncertainty[
                    "q_hat_cycles"
                ]
            ),

            rel_tol=0,
            abs_tol=TOLERANCE,
        )
    ),

    "confidence matches policy": (
        math.isclose(
            confidence_level,

            float(
                policy_uncertainty[
                    "confidence_level"
                ]
            ),

            rel_tol=0,
            abs_tol=TOLERANCE,
        )
    ),

    "test predictors were sealed at policy freeze": (
        policy_population[
            "test_predictor_values_parsed"
        ]
        is False
    ),

    "test outcomes were sealed at policy freeze": (
        policy_population[
            "test_rul_values_parsed"
        ]
        is False
    ),
}

failed_prechecks = [
    check_name
    for check_name, passed
    in pre_unsealing_checks.items()
    if not passed
]

if failed_prechecks:
    raise AssertionError(
        "Pre-unsealing policy verification failed:\n"
        + "\n".join(
            failed_prechecks
        )
    )


# =============================================================================
# 3. PARSE THE OFFICIAL TEST FILES
# =============================================================================

test_data = pd.read_csv(
    TEST_PATH,
    sep=r"\s+",
    header=None,
    names=TEST_COLUMNS,
    engine="python",
)

test_rul = pd.read_csv(
    TEST_RUL_PATH,
    sep=r"\s+",
    header=None,
    names=[
        "actual_endpoint_rul"
    ],
    engine="python",
)

test_frame_hash_before = (
    dataframe_hash(
        test_data
    )
)

test_rul_frame_hash_before = (
    dataframe_hash(
        test_rul
    )
)

numeric_test = (
    test_data
    .to_numpy(
        dtype=float
    )
)

numeric_rul = (
    test_rul[
        "actual_endpoint_rul"
    ]
    .to_numpy(
        dtype=float
    )
)

test_data[
    "unit_number"
] = test_data[
    "unit_number"
].astype(int)

test_data[
    "time_in_cycles"
] = test_data[
    "time_in_cycles"
].astype(int)

test_rul[
    "actual_endpoint_rul"
] = test_rul[
    "actual_endpoint_rul"
].astype(int)

unit_ids = np.sort(
    test_data[
        "unit_number"
    ].unique()
)

expected_unit_ids = np.arange(
    1,
    EXPECTED_TEST_ENGINES + 1,
    dtype=int,
)

cycle_integrity = (
    test_data
    .groupby(
        "unit_number"
    )[
        "time_in_cycles"
    ]
    .apply(
        lambda values: np.array_equal(
            np.sort(
                values.to_numpy(
                    dtype=int
                )
            ),

            np.arange(
                1,
                int(
                    values.max()
                )
                + 1,
                dtype=int,
            ),
        )
    )
)

sorted_test = (
    test_data
    .sort_values(
        [
            "unit_number",
            "time_in_cycles",
        ]
    )
    .reset_index(
        drop=True
    )
)

endpoint_rows = (
    sorted_test
    .groupby(
        "unit_number",
        as_index=False,
    )
    .tail(1)
    .sort_values(
        "unit_number"
    )
    .reset_index(
        drop=True
    )
)

outcomes = pd.DataFrame(
    {
        "unit_number": (
            expected_unit_ids
        ),

        "actual_endpoint_rul": (
            test_rul[
                "actual_endpoint_rul"
            ].to_numpy(
                dtype=int
            )
        ),
    }
)

endpoint_evaluation = (
    endpoint_rows
    .merge(
        outcomes,
        on="unit_number",
        how="inner",
        validate="one_to_one",
    )
)

X_endpoint = (
    endpoint_evaluation[
        feature_order
    ]
    .copy()
)

actual_rul = (
    endpoint_evaluation[
        "actual_endpoint_rul"
    ]
    .to_numpy(
        dtype=float
    )
)


# =============================================================================
# 4. OFFICIAL DATA VALIDATION
# =============================================================================

data_checks = {
    "test table has twenty six columns": (
        len(
            test_data.columns
        )
        == EXPECTED_COLUMNS
    ),

    "test table contains 13096 rows": (
        len(test_data)
        == EXPECTED_TEST_ROWS
    ),

    "test table contains 100 engines": (
        len(unit_ids)
        == EXPECTED_TEST_ENGINES
    ),

    "test engine identifiers are exactly one through one hundred": (
        np.array_equal(
            unit_ids,
            expected_unit_ids,
        )
    ),

    "test RUL contains 100 outcomes": (
        len(test_rul)
        == EXPECTED_TEST_ENGINES
    ),

    "test predictors contain no missing values": (
        not test_data
        .isna()
        .any()
        .any()
    ),

    "test predictors are finite": (
        bool(
            np.isfinite(
                numeric_test
            ).all()
        )
    ),

    "test RUL contains no missing values": (
        not test_rul
        .isna()
        .any()
        .any()
    ),

    "test RUL values are finite": (
        bool(
            np.isfinite(
                numeric_rul
            ).all()
        )
    ),

    "test RUL values are non negative": (
        bool(
            (
                numeric_rul >= 0
            ).all()
        )
    ),

    "test cycles begin at one and remain consecutive": (
        bool(
            cycle_integrity.all()
        )
    ),

    "test engine cycle keys are unique": (
        not test_data.duplicated(
            [
                "unit_number",
                "time_in_cycles",
            ]
        ).any()
    ),

    "one endpoint was retained per engine": (
        len(endpoint_rows)
        == EXPECTED_TEST_ENGINES
        and endpoint_rows[
            "unit_number"
        ].is_unique
    ),

    "endpoint rows are final observed cycles": (
        bool(
            (
                endpoint_rows[
                    "time_in_cycles"
                ].to_numpy(
                    dtype=int
                )
                == (
                    test_data
                    .groupby(
                        "unit_number"
                    )[
                        "time_in_cycles"
                    ]
                    .max()
                    .to_numpy(
                        dtype=int
                    )
                )
            ).all()
        )
    ),

    "RUL outcomes align one to one with endpoints": (
        len(endpoint_evaluation)
        == EXPECTED_TEST_ENGINES
        and np.array_equal(
            endpoint_evaluation[
                "unit_number"
            ].to_numpy(
                dtype=int
            ),
            expected_unit_ids,
        )
    ),

    "all eighteen model features exist": (
        len(feature_order)
        == EXPECTED_FEATURES
        and all(
            feature
            in endpoint_evaluation.columns
            for feature
            in feature_order
        )
    ),

    "endpoint predictors contain no missing values": (
        not X_endpoint
        .isna()
        .any()
        .any()
    ),

    "endpoint predictors are finite": (
        bool(
            np.isfinite(
                X_endpoint.to_numpy(
                    dtype=float
                )
            ).all()
        )
    ),
}

failed_data_checks = [
    check_name
    for check_name, passed
    in data_checks.items()
    if not passed
]

if failed_data_checks:
    raise AssertionError(
        "Official-test data validation failed:\n"
        + "\n".join(
            failed_data_checks
        )
    )


# =============================================================================
# 5. APPLY THE FROZEN RANDOM FOREST
# =============================================================================

raw_predictions = (
    point_model
    .predict(
        X_endpoint
    )
    .astype(float)
)

point_predictions = np.clip(
    raw_predictions,
    prediction_lower_bound,
    None,
)

errors = (
    point_predictions
    - actual_rul
)

absolute_errors = np.abs(
    errors
)

squared_errors = np.square(
    errors
)


# =============================================================================
# 6. APPLY THE FROZEN CONFORMAL INTERVAL
# =============================================================================

interval_lower = np.maximum(
    0.0,
    point_predictions - q_hat,
)

interval_upper = (
    point_predictions
    + q_hat
)

interval_width = (
    interval_upper
    - interval_lower
)

lower_was_clipped = (
    point_predictions
    - q_hat
    < 0.0
)

covered = (
    (
        actual_rul
        >= (
            interval_lower
            - TOLERANCE
        )
    )
    &
    (
        actual_rul
        <= (
            interval_upper
            + TOLERANCE
        )
    )
)

miss_below = (
    actual_rul
    < (
        interval_lower
        - TOLERANCE
    )
)

miss_above = (
    actual_rul
    > (
        interval_upper
        + TOLERANCE
    )
)


# =============================================================================
# 7. NASA PHM PENALTY AND INTERVAL SCORE
# =============================================================================

phm_penalty = np.where(
    errors < 0,

    np.exp(
        -errors / 13.0
    )
    - 1.0,

    np.exp(
        errors / 10.0
    )
    - 1.0,
)

interval_score = (
    interval_width

    + (
        2.0 / alpha
    )
    * (
        interval_lower
        - actual_rul
    )
    * miss_below.astype(float)

    + (
        2.0 / alpha
    )
    * (
        actual_rul
        - interval_upper
    )
    * miss_above.astype(float)
)


# =============================================================================
# 8. POINT METRICS
# =============================================================================

mae = float(
    np.mean(
        absolute_errors
    )
)

rmse = float(
    np.sqrt(
        np.mean(
            squared_errors
        )
    )
)

median_absolute_error = float(
    np.median(
        absolute_errors
    )
)

bias = float(
    np.mean(
        errors
    )
)

sum_squared_error = float(
    np.sum(
        squared_errors
    )
)

total_sum_of_squares = float(
    np.sum(
        np.square(
            actual_rul
            - np.mean(
                actual_rul
            )
        )
    )
)

r_squared = float(
    1.0
    - (
        sum_squared_error
        / total_sum_of_squares
    )
)

overestimation_rate = float(
    100.0
    * np.mean(
        errors > 0
    )
)

underestimation_rate = float(
    100.0
    * np.mean(
        errors < 0
    )
)

within_10_rate = float(
    100.0
    * np.mean(
        absolute_errors <= 10
    )
)

within_20_rate = float(
    100.0
    * np.mean(
        absolute_errors <= 20
    )
)

within_30_rate = float(
    100.0
    * np.mean(
        absolute_errors <= 30
    )
)


# =============================================================================
# 9. UNCERTAINTY METRICS
# =============================================================================

coverage_count = int(
    np.sum(
        covered
    )
)

coverage_rate = float(
    np.mean(
        covered
    )
)

coverage_z = float(
    policy[
        "interval_metrics"
    ][
        "coverage_z_value"
    ]
)

(
    coverage_wilson_lower,
    coverage_wilson_upper,
) = wilson_interval(
    coverage_count,
    EXPECTED_TEST_ENGINES,
    coverage_z,
)

lifecycle_stage = (
    assign_lifecycle_stage(
        actual_rul
    )
)


# =============================================================================
# 10. BUILD ENDPOINT RESULTS
# =============================================================================

endpoint_results = pd.DataFrame(
    {
        "unit_number": (
            endpoint_evaluation[
                "unit_number"
            ].to_numpy(
                dtype=int
            )
        ),

        "final_observed_cycle": (
            endpoint_evaluation[
                "time_in_cycles"
            ].to_numpy(
                dtype=int
            )
        ),

        "actual_endpoint_rul": (
            actual_rul
        ),

        "raw_point_prediction": (
            raw_predictions
        ),

        "point_prediction": (
            point_predictions
        ),

        "error_prediction_minus_actual": (
            errors
        ),

        "absolute_error": (
            absolute_errors
        ),

        "squared_error": (
            squared_errors
        ),

        "phm_penalty": (
            phm_penalty
        ),

        "q_hat": (
            q_hat
        ),

        "interval_lower": (
            interval_lower
        ),

        "interval_upper": (
            interval_upper
        ),

        "interval_width": (
            interval_width
        ),

        "lower_bound_was_clipped": (
            lower_was_clipped
        ),

        "covered": (
            covered
        ),

        "miss_below_interval": (
            miss_below
        ),

        "miss_above_interval": (
            miss_above
        ),

        "interval_score_alpha_0_10": (
            interval_score
        ),

        "lifecycle_stage": (
            lifecycle_stage
        ),
    }
)


# =============================================================================
# 11. BUILD SUMMARY METRICS
# =============================================================================

summary_metrics = pd.DataFrame(
    [
        {
            "model": (
                "random_forest_uncapped"
            ),

            "test_engine_count": (
                EXPECTED_TEST_ENGINES
            ),

            "mae": (
                mae
            ),

            "rmse": (
                rmse
            ),

            "r_squared": (
                r_squared
            ),

            "bias_prediction_minus_actual": (
                bias
            ),

            "median_absolute_error": (
                median_absolute_error
            ),

            "overestimation_rate_percent": (
                overestimation_rate
            ),

            "underestimation_rate_percent": (
                underestimation_rate
            ),

            "within_10_cycles_rate_percent": (
                within_10_rate
            ),

            "within_20_cycles_rate_percent": (
                within_20_rate
            ),

            "within_30_cycles_rate_percent": (
                within_30_rate
            ),

            "phm_total_penalty": float(
                np.sum(
                    phm_penalty
                )
            ),

            "phm_mean_penalty": float(
                np.mean(
                    phm_penalty
                )
            ),

            "conformal_confidence_level": (
                confidence_level
            ),

            "q_hat_cycles": (
                q_hat
            ),

            "endpoint_coverage_count": (
                coverage_count
            ),

            "endpoint_coverage_rate": (
                coverage_rate
            ),

            "coverage_wilson_95_lower": (
                coverage_wilson_lower
            ),

            "coverage_wilson_95_upper": (
                coverage_wilson_upper
            ),

            "mean_interval_width": float(
                np.mean(
                    interval_width
                )
            ),

            "median_interval_width": float(
                np.median(
                    interval_width
                )
            ),

            "lower_clipping_rate": float(
                np.mean(
                    lower_was_clipped
                )
            ),

            "miss_below_count": int(
                np.sum(
                    miss_below
                )
            ),

            "miss_above_count": int(
                np.sum(
                    miss_above
                )
            ),

            "mean_interval_score_alpha_0_10": float(
                np.mean(
                    interval_score
                )
            ),

            "median_interval_score_alpha_0_10": float(
                np.median(
                    interval_score
                )
            ),
        }
    ]
)


# =============================================================================
# 12. LIFECYCLE-STAGE RESULTS
# =============================================================================

lifecycle_rows = []

for stage in LIFECYCLE_ORDER:
    stage_mask = (
        lifecycle_stage
        == stage
    )

    stage_count = int(
        np.sum(
            stage_mask
        )
    )

    if stage_count == 0:
        lifecycle_rows.append(
            {
                "lifecycle_stage": stage,
                "engine_count": 0,
                "mae": np.nan,
                "rmse": np.nan,

                "bias_prediction_minus_actual": (
                    np.nan
                ),

                "coverage_rate": np.nan,

                "mean_interval_width": (
                    np.nan
                ),
            }
        )

    else:
        lifecycle_rows.append(
            {
                "lifecycle_stage": stage,

                "engine_count": (
                    stage_count
                ),

                "mae": float(
                    np.mean(
                        absolute_errors[
                            stage_mask
                        ]
                    )
                ),

                "rmse": float(
                    np.sqrt(
                        np.mean(
                            squared_errors[
                                stage_mask
                            ]
                        )
                    )
                ),

                "bias_prediction_minus_actual": float(
                    np.mean(
                        errors[
                            stage_mask
                        ]
                    )
                ),

                "coverage_rate": float(
                    np.mean(
                        covered[
                            stage_mask
                        ]
                    )
                ),

                "mean_interval_width": float(
                    np.mean(
                        interval_width[
                            stage_mask
                        ]
                    )
                ),
            }
        )

lifecycle_metrics = pd.DataFrame(
    lifecycle_rows
)


# =============================================================================
# 13. CREATE THESIS-READY FIGURE
# =============================================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(15.5, 6.6),
)

maximum_axis_value = float(
    max(
        np.max(
            actual_rul
        ),
        np.max(
            point_predictions
        ),
        np.max(
            interval_upper
        ),
    )
)

identity_limit = float(
    max(
        np.max(
            actual_rul
        ),
        np.max(
            point_predictions
        ),
    )
    * 1.08
)

axes[0].scatter(
    actual_rul,
    point_predictions,
    s=42,
    color="#2F6BDE",
    edgecolor="white",
    linewidth=0.7,
    alpha=0.88,
)

axes[0].plot(
    [
        0,
        identity_limit,
    ],
    [
        0,
        identity_limit,
    ],
    linestyle="--",
    color="#374151",
    linewidth=1.3,
    label="Ideal prediction",
)

axes[0].set_xlim(
    0,
    identity_limit,
)

axes[0].set_ylim(
    0,
    identity_limit,
)

axes[0].set_aspect(
    "equal",
    adjustable="box",
)

axes[0].set_title(
    "A. Point predictions",
    fontsize=12.5,
    weight="bold",
)

axes[0].set_xlabel(
    "Actual endpoint RUL (cycles)"
)

axes[0].set_ylabel(
    "Predicted endpoint RUL (cycles)"
)

axes[0].legend(
    frameon=False,
    loc="upper left",
)

axes[0].grid(
    color="#D1D5DB",
    linewidth=0.7,
    alpha=0.7,
)


# Panel B

plot_order = np.argsort(
    actual_rul
)

ordered_actual = (
    actual_rul[
        plot_order
    ]
)

ordered_prediction = (
    point_predictions[
        plot_order
    ]
)

ordered_lower = (
    interval_lower[
        plot_order
    ]
)

ordered_upper = (
    interval_upper[
        plot_order
    ]
)

ordered_covered = (
    covered[
        plot_order
    ]
)

rank_axis = np.arange(
    1,
    EXPECTED_TEST_ENGINES + 1,
)

axes[1].vlines(
    rank_axis,
    ordered_lower,
    ordered_upper,
    color="#93C5FD",
    linewidth=1.1,
    alpha=0.72,
    label="90% conformal interval",
)

axes[1].scatter(
    rank_axis,
    ordered_prediction,
    s=20,
    color="#2F6BDE",
    label="Point prediction",
    zorder=3,
)

axes[1].scatter(
    rank_axis,
    ordered_actual,
    s=24,
    marker="x",
    color="#D97706",
    linewidth=1.2,
    label="Actual endpoint RUL",
    zorder=4,
)

if not bool(
    np.all(
        ordered_covered
    )
):
    missed_positions = (
        rank_axis[
            ~ordered_covered
        ]
    )

    axes[1].scatter(
        missed_positions,

        ordered_actual[
            ~ordered_covered
        ],

        s=70,
        facecolors="none",
        edgecolors="#111827",
        linewidth=1.2,
        label="Interval miss",
        zorder=5,
    )

axes[1].set_title(
    (
        "B. Endpoint intervals ordered "
        "by actual RUL"
    ),
    fontsize=12.5,
    weight="bold",
)

axes[1].set_xlabel(
    (
        "Test-engine rank by actual "
        "endpoint RUL"
    )
)

axes[1].set_ylabel(
    "Remaining useful life (cycles)"
)

axes[1].set_xlim(
    0,
    EXPECTED_TEST_ENGINES + 1,
)

axes[1].set_ylim(
    0,
    maximum_axis_value * 1.04,
)

axes[1].grid(
    axis="y",
    color="#D1D5DB",
    linewidth=0.7,
    alpha=0.7,
)

axes[1].legend(
    frameon=False,
    loc="upper left",
    fontsize=9,
)

for axis in axes:
    axis.spines[
        "top"
    ].set_visible(False)

    axis.spines[
        "right"
    ].set_visible(False)

fig.suptitle(
    "Official FD001 endpoint evaluation",
    fontsize=16,
    weight="bold",
    y=0.99,
)

fig.text(
    0.5,
    0.935,

    (
        "100 official test engines; "
        "frozen Random Forest and "
        "90% engine-block conformal intervals"
    ),

    ha="center",
    fontsize=10.5,
    color="#4B5563",
)

fig.text(
    0.5,
    0.015,

    (
        "Official outcomes were used once "
        "for final evaluation. No model "
        "reselection or conformal "
        "recalibration was permitted."
    ),

    ha="center",
    fontsize=9,
    color="#4B5563",
)

fig.tight_layout(
    rect=[
        0.02,
        0.055,
        0.99,
        0.91,
    ]
)

fig.savefig(
    FIGURE_PATH,
    dpi=220,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()
plt.close(fig)


# =============================================================================
# 14. SAVE RESULTS
# =============================================================================

endpoint_results.to_csv(
    ENDPOINT_RESULTS_PATH,
    index=False,
)

summary_metrics.to_csv(
    SUMMARY_METRICS_PATH,
    index=False,
)

lifecycle_metrics.to_csv(
    LIFECYCLE_METRICS_PATH,
    index=False,
)


# =============================================================================
# 15. POST-EVALUATION INTEGRITY
# =============================================================================

test_frame_hash_after = (
    dataframe_hash(
        test_data
    )
)

test_rul_frame_hash_after = (
    dataframe_hash(
        test_rul
    )
)

policy_hash_after = (
    sha256_file(
        POLICY_PATH
    )
)

model_hash_after = (
    sha256_file(
        PRIMARY_MODEL_PATH
    )
)

conformal_hash_after = (
    sha256_file(
        CONFORMAL_BUNDLE_PATH
    )
)

test_hash_after = (
    sha256_file(
        TEST_PATH
    )
)

test_rul_hash_after = (
    sha256_file(
        TEST_RUL_PATH
    )
)


# =============================================================================
# 16. WRITE EVALUATION RECORD
# =============================================================================

evaluation_record = {
    "phase": 6,

    "step": (
        "2_one_time_official_"
        "endpoint_evaluation"
    ),

    "status": (
        "official_test_evaluation_"
        "complete_results_frozen"
    ),

    "evaluated_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),

    "policy_path": (
        str(POLICY_PATH)
    ),

    "policy_sha256": (
        policy_hash_after
    ),

    "primary_model": (
        "random_forest_uncapped"
    ),

    "primary_model_sha256": (
        model_hash_after
    ),

    "conformal_bundle_sha256": (
        conformal_hash_after
    ),

    "test_predictor_sha256": (
        test_hash_after
    ),

    "test_rul_sha256": (
        test_rul_hash_after
    ),

    "evaluation_grain": (
        "one_last_observed_endpoint_"
        "per_test_engine"
    ),

    "test_engine_count": (
        EXPECTED_TEST_ENGINES
    ),

    "point_metrics": {
        "mae": mae,
        "rmse": rmse,
        "r_squared": r_squared,

        "bias_prediction_minus_actual": (
            bias
        ),

        "median_absolute_error": (
            median_absolute_error
        ),

        "overestimation_rate_percent": (
            overestimation_rate
        ),

        "underestimation_rate_percent": (
            underestimation_rate
        ),

        "within_10_cycles_rate_percent": (
            within_10_rate
        ),

        "within_20_cycles_rate_percent": (
            within_20_rate
        ),

        "within_30_cycles_rate_percent": (
            within_30_rate
        ),

        "phm_total_penalty": float(
            np.sum(
                phm_penalty
            )
        ),

        "phm_mean_penalty": float(
            np.mean(
                phm_penalty
            )
        ),
    },

    "interval_metrics": {
        "confidence_level": (
            confidence_level
        ),

        "q_hat_cycles": (
            q_hat
        ),

        "endpoint_coverage_count": (
            coverage_count
        ),

        "endpoint_coverage_rate": (
            coverage_rate
        ),

        "coverage_wilson_95_lower": (
            coverage_wilson_lower
        ),

        "coverage_wilson_95_upper": (
            coverage_wilson_upper
        ),

        "mean_interval_width": float(
            np.mean(
                interval_width
            )
        ),

        "median_interval_width": float(
            np.median(
                interval_width
            )
        ),

        "lower_clipping_rate": float(
            np.mean(
                lower_was_clipped
            )
        ),

        "miss_below_count": int(
            np.sum(
                miss_below
            )
        ),

        "miss_above_count": int(
            np.sum(
                miss_above
            )
        ),

        "mean_interval_score_alpha_0_10": float(
            np.mean(
                interval_score
            )
        ),

        "median_interval_score_alpha_0_10": float(
            np.median(
                interval_score
            )
        ),
    },

    "interpretation": {
        "results_are_final_unbiased_"
        "endpoint_evaluation": True,

        "results_change_model_selection": False,

        "subgroup_metrics_are_"
        "descriptive_only": True,

        "operational_safety_guarantee": False,

        "post_hoc_performance_"
        "pass_fail_claim": False,

        "trajectory_calibrated_interval_"
        "may_be_conservative_at_endpoint": True,
    },

    "research_controls": {
        "alternative_models_evaluated": False,
        "model_retrained_or_modified": False,
        "feature_selection_performed": False,
        "hyperparameters_changed": False,
        "target_changed": False,
        "conformal_method_changed": False,
        "q_hat_changed": False,
        "test_cases_removed": False,
        "model_reselected": False,
    },

    "output_files": {
        "endpoint_results": (
            str(
                ENDPOINT_RESULTS_PATH
            )
        ),

        "summary_metrics": (
            str(
                SUMMARY_METRICS_PATH
            )
        ),

        "lifecycle_metrics": (
            str(
                LIFECYCLE_METRICS_PATH
            )
        ),

        "figure": (
            str(
                FIGURE_PATH
            )
        ),
    },
}

write_json(
    EVALUATION_RECORD_PATH,
    evaluation_record,
)


# =============================================================================
# 17. FINAL VALIDATION
# =============================================================================

metric_values = (
    summary_metrics
    .select_dtypes(
        include=[
            np.number
        ]
    )
    .to_numpy(
        dtype=float
    )
)

nonempty_stage_metrics = (
    lifecycle_metrics
    .loc[
        lifecycle_metrics[
            "engine_count"
        ]
        > 0,

        [
            "mae",
            "rmse",
            "bias_prediction_minus_actual",
            "coverage_rate",
            "mean_interval_width",
        ],
    ]
    .to_numpy(
        dtype=float
    )
)

evaluation_checks = {
    **pre_unsealing_checks,
    **data_checks,

    "one hundred endpoint predictions were generated": (
        len(endpoint_results)
        == EXPECTED_TEST_ENGINES
    ),

    "raw predictions are finite": (
        bool(
            np.isfinite(
                raw_predictions
            ).all()
        )
    ),

    "point predictions are finite": (
        bool(
            np.isfinite(
                point_predictions
            ).all()
        )
    ),

    "point predictions are non negative": (
        bool(
            (
                point_predictions >= 0
            ).all()
        )
    ),

    "all point errors are finite": (
        bool(
            np.isfinite(
                errors
            ).all()
        )
    ),

    "all summary metrics are finite": (
        bool(
            np.isfinite(
                metric_values
            ).all()
        )
    ),

    "PHM penalties are finite and non negative": (
        bool(
            np.isfinite(
                phm_penalty
            ).all()
            and (
                phm_penalty >= 0
            ).all()
        )
    ),

    "interval lower bounds are non negative": (
        bool(
            (
                interval_lower >= 0
            ).all()
        )
    ),

    "interval upper formula is correct": (
        bool(
            np.allclose(
                interval_upper,

                point_predictions
                + q_hat,

                rtol=0,
                atol=TOLERANCE,
            )
        )
    ),

    "interval lower formula is correct": (
        bool(
            np.allclose(
                interval_lower,

                np.maximum(
                    0.0,
                    point_predictions
                    - q_hat,
                ),

                rtol=0,
                atol=TOLERANCE,
            )
        )
    ),

    "coverage count and miss counts reconcile": (
        coverage_count
        + int(
            np.sum(
                miss_below
            )
        )
        + int(
            np.sum(
                miss_above
            )
        )
        == EXPECTED_TEST_ENGINES
    ),

    "Wilson interval is ordered and bounded": (
        0
        <= coverage_wilson_lower
        <= coverage_rate
        <= coverage_wilson_upper
        <= 1
    ),

    "interval scores are finite and non negative": (
        bool(
            np.isfinite(
                interval_score
            ).all()
            and (
                interval_score >= 0
            ).all()
        )
    ),

    "four predefined lifecycle stages were reported": (
        lifecycle_metrics[
            "lifecycle_stage"
        ].tolist()
        == LIFECYCLE_ORDER
    ),

    "lifecycle engine counts sum to one hundred": (
        int(
            lifecycle_metrics[
                "engine_count"
            ].sum()
        )
        == EXPECTED_TEST_ENGINES
    ),

    "nonempty lifecycle metrics are finite": (
        bool(
            np.isfinite(
                nonempty_stage_metrics
            ).all()
        )
    ),

    "no endpoint was left without a lifecycle stage": (
        bool(
            (
                endpoint_results[
                    "lifecycle_stage"
                ]
                != "unassigned"
            ).all()
        )
    ),

    "source test dataframe was not modified": (
        test_frame_hash_before
        == test_frame_hash_after
    ),

    "source test RUL dataframe was not modified": (
        test_rul_frame_hash_before
        == test_rul_frame_hash_after
    ),

    "frozen policy was not modified": (
        policy_hash_before
        == policy_hash_after
    ),

    "primary model was not modified": (
        model_hash_before
        == model_hash_after
        == EXPECTED_MODEL_SHA256
    ),

    "conformal bundle was not modified": (
        conformal_hash_before
        == conformal_hash_after
    ),

    "raw test predictor file was not modified": (
        test_hash_before
        == test_hash_after
        == EXPECTED_TEST_SHA256
    ),

    "raw test RUL file was not modified": (
        test_rul_hash_before
        == test_rul_hash_after
        == EXPECTED_TEST_RUL_SHA256
    ),

    "all output files exist": all(
        path.is_file()
        for path in [
            ENDPOINT_RESULTS_PATH,
            SUMMARY_METRICS_PATH,
            LIFECYCLE_METRICS_PATH,
            FIGURE_PATH,
            EVALUATION_RECORD_PATH,
        ]
    ),
}

failed_checks = [
    check_name
    for check_name, passed
    in evaluation_checks.items()
    if not passed
]

if failed_checks:
    raise AssertionError(
        "Official-test evaluation validation failed:\n"
        + "\n".join(
            failed_checks
        )
    )


# =============================================================================
# 18. BUILD EVIDENCE MANIFEST
# =============================================================================

manifest_entries = [
    (
        "frozen_test_policy",
        POLICY_PATH,
        True,
    ),

    (
        "final_model_registry",
        FINAL_REGISTRY_PATH,
        True,
    ),

    (
        "primary_model_hash_reference",
        PRIMARY_MODEL_PATH,
        False,
    ),

    (
        "conformal_bundle_hash_reference",
        CONFORMAL_BUNDLE_PATH,
        False,
    ),

    (
        "official_test_predictor_hash_reference",
        TEST_PATH,
        False,
    ),

    (
        "official_test_rul_hash_reference",
        TEST_RUL_PATH,
        False,
    ),

    (
        "official_endpoint_results",
        ENDPOINT_RESULTS_PATH,
        True,
    ),

    (
        "official_summary_metrics",
        SUMMARY_METRICS_PATH,
        True,
    ),

    (
        "official_lifecycle_metrics",
        LIFECYCLE_METRICS_PATH,
        True,
    ),

    (
        "official_evaluation_figure",
        FIGURE_PATH,
        True,
    ),

    (
        "official_evaluation_record",
        EVALUATION_RECORD_PATH,
        True,
    ),
]

manifest_rows = []

for (
    role,
    path,
    include_in_zip,
) in manifest_entries:

    manifest_rows.append(
        {
            "role": role,
            "path": str(path),

            "size_bytes": (
                path.stat().st_size
            ),

            "sha256": (
                sha256_file(path)
            ),

            "included_in_step_zip": (
                include_in_zip
            ),
        }
    )

with EVIDENCE_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
    newline="",
) as file_handle:

    writer = csv.DictWriter(
        file_handle,
        fieldnames=[
            "role",
            "path",
            "size_bytes",
            "sha256",
            "included_in_step_zip",
        ],
    )

    writer.writeheader()
    writer.writerows(manifest_rows)


# =============================================================================
# 19. CREATE RETAINED EVIDENCE ZIP
# =============================================================================

with zipfile.ZipFile(
    EVIDENCE_ZIP_PATH,
    "w",
    zipfile.ZIP_DEFLATED,
) as archive:

    for row in manifest_rows:
        if row[
            "included_in_step_zip"
        ]:
            source_path = Path(
                row["path"]
            )

            archive.write(
                source_path,

                arcname=str(
                    source_path.relative_to(
                        PROJECT_ROOT
                    )
                ),
            )

    archive.write(
        EVIDENCE_MANIFEST_PATH,

        arcname=str(
            EVIDENCE_MANIFEST_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
    )

evidence_zip_hash = sha256_file(
    EVIDENCE_ZIP_PATH
)


# =============================================================================
# 20. BOUNDED FINAL OUTPUT
# =============================================================================

print(
    "OFFICIAL FD001 ENDPOINT "
    "POINT-MODEL RESULTS"
)
print("=" * 90)
print(
    f"Test engines: "
    f"{EXPECTED_TEST_ENGINES}"
)
print(
    f"MAE: "
    f"{mae:.6f} cycles"
)
print(
    f"RMSE: "
    f"{rmse:.6f} cycles"
)
print(
    f"R-squared: "
    f"{r_squared:.6f}"
)
print(
    "Bias (prediction - actual): "
    f"{bias:.6f} cycles"
)
print(
    "Median absolute error: "
    f"{median_absolute_error:.6f} cycles"
)
print(
    f"Overestimation rate: "
    f"{overestimation_rate:.2f}%"
)
print(
    f"Underestimation rate: "
    f"{underestimation_rate:.2f}%"
)
print(
    f"Within 10 cycles: "
    f"{within_10_rate:.2f}%"
)
print(
    f"Within 20 cycles: "
    f"{within_20_rate:.2f}%"
)
print(
    f"Within 30 cycles: "
    f"{within_30_rate:.2f}%"
)
print(
    "NASA PHM total penalty: "
    f"{np.sum(phm_penalty):.6f}"
)
print(
    "NASA PHM mean penalty: "
    f"{np.mean(phm_penalty):.6f}"
)

print(
    "\nOFFICIAL FD001 ENDPOINT "
    "UNCERTAINTY RESULTS"
)
print("=" * 90)
print(
    f"Confidence level: "
    f"{confidence_level:.0%}"
)
print(
    f"Frozen q_hat: "
    f"{q_hat:.6f} cycles"
)
print(
    f"Endpoint coverage: "
    f"{coverage_count} of "
    f"{EXPECTED_TEST_ENGINES} "
    f"({coverage_rate:.2%})"
)
print(
    "95% Wilson interval for coverage: "
    f"[{coverage_wilson_lower:.2%}, "
    f"{coverage_wilson_upper:.2%}]"
)
print(
    f"Mean interval width: "
    f"{np.mean(interval_width):.6f} cycles"
)
print(
    f"Median interval width: "
    f"{np.median(interval_width):.6f} cycles"
)
print(
    f"Lower-bound clipping rate: "
    f"{np.mean(lower_was_clipped):.2%}"
)
print(
    f"Misses below interval: "
    f"{np.sum(miss_below)}"
)
print(
    f"Misses above interval: "
    f"{np.sum(miss_above)}"
)
print(
    f"Mean interval score: "
    f"{np.mean(interval_score):.6f}"
)
print(
    f"Median interval score: "
    f"{np.median(interval_score):.6f}"
)

print("\nLIFECYCLE-STAGE RESULTS")
print("=" * 90)
print(
    lifecycle_metrics.to_string(
        index=False
    )
)

print("\nOFFICIAL TEST VALIDATION")
print("=" * 90)

for (
    check_name,
    passed,
) in evaluation_checks.items():

    status = (
        "PASS"
        if passed
        else "FAIL"
    )

    print(
        f"{status} — "
        f"{check_name}"
    )

print("\nINTERPRETATION CONTROL")
print("=" * 90)
print(
    "Official test used for final "
    "evaluation: Yes — once"
)
print(
    "Official test used for model "
    "selection or tuning: No"
)
print(
    "Alternative models evaluated "
    "on official test: No"
)
print("Model selection changed: No")
print(
    "Conformal method or q_hat changed: No"
)
print(
    "Post-hoc performance pass/fail "
    "threshold applied: No"
)
print(
    "Operational safety guarantee claimed: No"
)
print(
    "Lifecycle-stage results are "
    "descriptive only: Yes"
)

print("\nOUTPUT FILES")
print("=" * 90)
print(
    f"Endpoint results: "
    f"{ENDPOINT_RESULTS_PATH}"
)
print(
    f"Summary metrics: "
    f"{SUMMARY_METRICS_PATH}"
)
print(
    f"Lifecycle metrics: "
    f"{LIFECYCLE_METRICS_PATH}"
)
print(
    f"Official-test figure: "
    f"{FIGURE_PATH}"
)
print(
    f"Evaluation record: "
    f"{EVALUATION_RECORD_PATH}"
)
print(
    f"Evidence manifest: "
    f"{EVIDENCE_MANIFEST_PATH}"
)
print(
    f"Evidence ZIP retained in Colab: "
    f"{EVIDENCE_ZIP_PATH}"
)
print(
    f"Evidence ZIP SHA-256: "
    f"{evidence_zip_hash}"
)
print("Browser download triggered: No")

print("\nFINAL CHECK")
print("=" * 90)
print(
    "PASS — The official FD001 endpoint "
    "outcomes were evaluated exactly once."
)
print(
    "PASS — All 100 test engines were "
    "retained and aligned correctly."
)
print(
    "PASS — The frozen Random Forest and "
    "conformal interval were applied unchanged."
)
print(
    "PASS — Every preregistered point and "
    "uncertainty metric was reported."
)
print(
    "PASS — Official test results did not "
    "alter model selection."
)
print(
    "AWAITING APPROVAL — Phase 6 completion "
    "packaging has not started."
)

In [ ]:
# =============================================================================
# PHASE 6 — GOVERNANCE RECONCILIATION, COMPLETION AND FORMAL CLOSURE
#
# COMPLETE REPLACEMENT FOR THE TWO FAILED CELLS
#
# This cell:
# - Does not deserialize any fitted model.
# - Does not deserialize the conformal bundle.
# - Does not call predict().
# - Does not read raw official predictor values.
# - Does not read raw official RUL outcomes.
# - Reproduces metrics only from saved Phase 6 endpoint results.
# - Validates the current cross-referenced hash chain.
# - Validates prior ZIP packages against their manifests.
# - Builds a deterministic Phase 6 completion package.
# - Creates or verifies the Phase 6 formal closure record.
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import math
import zipfile

import numpy as np
import pandas as pd


# =============================================================================
# 1. FIXED SCIENTIFIC EXPECTATIONS
# =============================================================================

EXPECTED_PRIMARY_MODEL_SHA256 = (
    "99620a1a161f62a716d4cc058497159146"
    "197e7d91bba713581c822af808e5d5"
)

EXPECTED_TEST_PREDICTOR_SHA256 = (
    "3cda7109ce17bafb5443f2ac926cfcf88"
    "154b941b8c4cf95eb55d1ddd6f52851"
)

EXPECTED_TEST_RUL_SHA256 = (
    "a19c8ec94931949d0485bdc35118206e9"
    "c81c4547b422efb9cf86f4ceddbceca"
)

EXPECTED_TEST_ENGINES = 100
EXPECTED_FEATURES = 18
EXPECTED_CONFIDENCE = 0.90
EXPECTED_Q_HAT = 107.896520

TOLERANCE = 1e-8

LIFECYCLE_ORDER = [
    "critical_0_to_30",
    "near_term_31_to_60",
    "mid_life_61_to_130",
    "early_life_above_130",
]


# Historical printed hashes are retained for comparison only.
# They are not used as tautological pass conditions.

LEGACY_HASHES = {
    "phase5_complete_zip": (
        "4b883a65ed519977ea3ddb45ebe8c4be3"
        "6f1ab3461ca98dbac129b3b64f1c8f3"
    ),

    "conformal_bundle": (
        "b894efcf7db7db6725e7a96e5579e456"
        "13562ef286da9e26fb026137fed9ec3e"
    ),

    "phase6_policy": (
        "070f7f4252f66891a32e1fc6af2f2e36"
        "85284890e8f0861c8fdb7e259f7a81d9"
    ),

    "phase6_step1_zip": (
        "267cbccd742e79f575d3d453cc12da9cf"
        "7e55522c56954bfe05d983203843da5"
    ),

    "phase6_step2_zip": (
        "f9d75dfc7c8f81b23a917e73a87411eb"
        "48d6dff6e4c996a73b8108f41665da0f"
    ),
}


# =============================================================================
# 2. GENERAL HELPERS
# =============================================================================

def utc_now():
    return datetime.now(
        timezone.utc
    ).isoformat()


def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file_handle:
        for chunk in iter(
            lambda: file_handle.read(chunk_size),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def sha256_bytes(data):
    return hashlib.sha256(
        data
    ).hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as file_handle:
        payload = json.load(file_handle)

    if not isinstance(payload, dict):
        raise TypeError(
            f"Expected a JSON object: {path}"
        )

    return payload


def write_json_atomic(
    path,
    payload,
):
    path = Path(path)

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as file_handle:
        json.dump(
            payload,
            file_handle,
            indent=2,
            ensure_ascii=False,
        )

        file_handle.write("\n")

    temporary_path.replace(path)


def close_enough(
    first,
    second,
    tolerance=TOLERANCE,
):
    return math.isclose(
        float(first),
        float(second),
        rel_tol=0,
        abs_tol=tolerance,
    )


def all_close(
    first,
    second,
    tolerance=TOLERANCE,
):
    return bool(
        np.allclose(
            np.asarray(first, dtype=float),
            np.asarray(second, dtype=float),
            rtol=0,
            atol=tolerance,
            equal_nan=False,
        )
    )


def parse_boolean_series(
    series,
    column_name,
):
    if pd.api.types.is_bool_dtype(series):
        if series.isna().any():
            raise ValueError(
                f"{column_name} contains missing values."
            )

        return series.to_numpy(
            dtype=bool
        )

    normalized = (
        series
        .astype("string")
        .str.strip()
        .str.lower()
    )

    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False,
    }

    invalid = sorted(
        set(normalized.dropna())
        - set(mapping)
    )

    if invalid or normalized.isna().any():
        raise ValueError(
            f"{column_name} contains invalid "
            f"boolean values: {invalid}"
        )

    return (
        normalized
        .map(mapping)
        .to_numpy(dtype=bool)
    )


def assign_lifecycle_stage(actual_rul):
    return np.select(
        [
            (actual_rul >= 0) & (actual_rul <= 30),
            (actual_rul > 30) & (actual_rul <= 60),
            (actual_rul > 60) & (actual_rul <= 130),
            actual_rul > 130,
        ],
        LIFECYCLE_ORDER,
        default="unassigned",
    )


def wilson_interval(
    successes,
    total,
    z_value,
):
    proportion = successes / total

    denominator = (
        1.0
        + z_value**2 / total
    )

    centre = (
        proportion
        + z_value**2 / (2.0 * total)
    ) / denominator

    margin = (
        z_value
        * math.sqrt(
            (
                proportion
                * (1.0 - proportion)
                / total
            )
            + (
                z_value**2
                / (4.0 * total**2)
            )
        )
        / denominator
    )

    return (
        centre - margin,
        centre + margin,
    )


# =============================================================================
# 3. LOCATE THE ACTIVE PROJECT
# =============================================================================

def project_is_valid(root):
    return (
        (
            root
            / "results"
            / "phase6_official_test_endpoint_predictions.csv"
        ).is_file()
        and (
            root
            / "logs"
            / "phase6_official_test_evaluation_record.json"
        ).is_file()
    )


def find_project_root():
    common_roots = [
        Path(
            "/content/Predictive_Maintenance_Thesis"
        ),
        Path(
            "/content/drive/MyDrive/"
            "Predictive_Maintenance_Thesis"
        ),
    ]

    for candidate in common_roots:
        if project_is_valid(candidate):
            return candidate

    search_base = Path("/content")

    if search_base.exists():
        matches = list(
            search_base.rglob(
                "phase6_official_test_endpoint_predictions.csv"
            )
        )

        for endpoint_path in matches:
            candidate = (
                endpoint_path.parent.parent
            )

            if project_is_valid(candidate):
                return candidate

    # Try mounting Drive only when the project was not found.

    try:
        from google.colab import drive

        drive_root = Path(
            "/content/drive/MyDrive"
        )

        if not drive_root.exists():
            drive.mount(
                "/content/drive",
                force_remount=False,
            )

        for candidate in common_roots:
            if project_is_valid(candidate):
                return candidate

        matches = list(
            Path("/content").rglob(
                "phase6_official_test_endpoint_predictions.csv"
            )
        )

        for endpoint_path in matches:
            candidate = (
                endpoint_path.parent.parent
            )

            if project_is_valid(candidate):
                return candidate

    except Exception:
        pass

    raise FileNotFoundError(
        "The active Predictive_Maintenance_Thesis project "
        "could not be located. Mount Google Drive or restore "
        "the project directory before running this cell."
    )


ROOT = find_project_root()

RESULTS = ROOT / "results"
LOGS = ROOT / "logs"
MODELS = ROOT / "models"
FIGURES = RESULTS / "figures"

LOGS.mkdir(
    parents=True,
    exist_ok=True,
)


# =============================================================================
# 4. ARTIFACT PATHS
# =============================================================================

FINAL_REGISTRY_PATH = (
    LOGS
    / "phase4_final_model_registry.json"
)

PHASE5_COMPLETE_MANIFEST_PATH = (
    LOGS
    / "phase5_complete_evidence_manifest.csv"
)

PHASE5_COMPLETE_ZIP_PATH = (
    ROOT
    / "Phase5_Complete_Uncertainty_Evidence.zip"
)

POLICY_PATH = (
    LOGS
    / "phase6_official_test_evaluation_policy.json"
)

DECISION_RECORD_PATH = (
    LOGS
    / "phase6_official_test_evaluation_decision.md"
)

STEP1_MANIFEST_PATH = (
    LOGS
    / "phase6_step1_test_policy_manifest.csv"
)

STEP1_ZIP_PATH = (
    ROOT
    / "Phase6_Step1_Official_Test_Policy_Evidence.zip"
)

PRIMARY_MODEL_PATH = (
    MODELS
    / "regression"
    / "fd001_random_forest_uncapped_bundle.joblib"
)

CONFORMAL_BUNDLE_PATH = (
    MODELS
    / "uncertainty"
    / (
        "fd001_random_forest_"
        "engine_block_conformal_bundle.joblib"
    )
)

ENDPOINT_RESULTS_PATH = (
    RESULTS
    / "phase6_official_test_endpoint_predictions.csv"
)

SUMMARY_METRICS_PATH = (
    RESULTS
    / "phase6_official_test_summary_metrics.csv"
)

LIFECYCLE_METRICS_PATH = (
    RESULTS
    / "phase6_official_test_lifecycle_stage_metrics.csv"
)

FIGURE_PATH = (
    FIGURES
    / "phase6_official_test_endpoint_evaluation.png"
)

EVALUATION_RECORD_PATH = (
    LOGS
    / "phase6_official_test_evaluation_record.json"
)

STEP2_MANIFEST_PATH = (
    LOGS
    / "phase6_step2_official_test_manifest.csv"
)

STEP2_ZIP_PATH = (
    ROOT
    / "Phase6_Step2_Official_Test_Evidence.zip"
)

COMPLETION_RECORD_PATH = (
    LOGS
    / "phase6_technical_completion_record.json"
)

COMPLETE_MANIFEST_PATH = (
    LOGS
    / "phase6_complete_evidence_manifest.csv"
)

COMPLETE_ZIP_PATH = (
    ROOT
    / "Phase6_Complete_Official_Test_Evidence.zip"
)

CLOSURE_RECORD_PATH = (
    LOGS
    / "phase6_closure_record.json"
)


# =============================================================================
# 5. REQUIRED SOURCE ARTIFACTS
# =============================================================================

required_paths = [
    FINAL_REGISTRY_PATH,
    PHASE5_COMPLETE_MANIFEST_PATH,
    PHASE5_COMPLETE_ZIP_PATH,
    POLICY_PATH,
    DECISION_RECORD_PATH,
    STEP1_MANIFEST_PATH,
    STEP1_ZIP_PATH,
    PRIMARY_MODEL_PATH,
    CONFORMAL_BUNDLE_PATH,
    ENDPOINT_RESULTS_PATH,
    SUMMARY_METRICS_PATH,
    LIFECYCLE_METRICS_PATH,
    FIGURE_PATH,
    EVALUATION_RECORD_PATH,
    STEP2_MANIFEST_PATH,
    STEP2_ZIP_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required saved Phase 6 artifacts are missing:\n"
        + "\n".join(missing_paths)
        + "\n\nThis cell will not fabricate missing evidence."
    )


# =============================================================================
# 6. MANIFEST AND ZIP VALIDATION
# =============================================================================

def manifest_rows(path):
    with path.open(
        "r",
        encoding="utf-8-sig",
        newline="",
    ) as file_handle:
        return list(
            csv.DictReader(file_handle)
        )


def true_value(value):
    return str(value).strip().lower() in {
        "true",
        "1",
        "yes",
    }


def included_column(rows):
    if not rows:
        return None

    candidates = [
        "included_in_complete_zip",
        "included_in_step_zip",
        "included_in_zip",
    ]

    for candidate in candidates:
        if candidate in rows[0]:
            return candidate

    return None


def locate_archive_member(
    archive_names,
    row,
):
    possible_names = []

    relative_path = row.get(
        "project_relative_path"
    )

    if relative_path:
        possible_names.append(
            relative_path.replace("\\", "/")
        )

    source_path_text = row.get("path")

    if source_path_text:
        source_path_text = (
            source_path_text
            .replace("\\", "/")
        )

        marker = (
            "Predictive_Maintenance_Thesis/"
        )

        if marker in source_path_text:
            possible_names.append(
                source_path_text.split(
                    marker,
                    1,
                )[1]
            )

        possible_names.append(
            Path(source_path_text).name
        )

    for possible_name in possible_names:
        if possible_name in archive_names:
            return possible_name

    basename = None

    if source_path_text:
        basename = Path(
            source_path_text
        ).name

    if basename:
        suffix_matches = [
            name
            for name in archive_names
            if name == basename
            or name.endswith(
                "/" + basename
            )
        ]

        if len(suffix_matches) == 1:
            return suffix_matches[0]

    return None


def validate_zip_against_manifest(
    zip_path,
    manifest_path,
):
    """
    Schema-tolerant ZIP validation.

    When an inclusion column exists, every row marked for inclusion
    must be present.

    When no inclusion column exists, the archive contents determine
    which manifest rows represent packaged files. Every packaged file
    must still match a manifest SHA-256.
    """

    rows = manifest_rows(
        manifest_path
    )

    if not rows:
        return (
            False,
            [
                "manifest contains no rows"
            ],
        )

    include_column = included_column(
        rows
    )

    failures = []
    validated_members = set()

    try:
        with zipfile.ZipFile(
            zip_path,
            "r",
        ) as archive:
            archive_names = {
                name
                for name in archive.namelist()
                if not name.endswith("/")
            }

            damaged_member = archive.testzip()

            if damaged_member is not None:
                failures.append(
                    "damaged ZIP member: "
                    f"{damaged_member}"
                )

            manifest_members = {
                name
                for name in archive_names
                if (
                    name == manifest_path.name
                    or name.endswith(
                        "/" + manifest_path.name
                    )
                )
            }

            if len(manifest_members) != 1:
                failures.append(
                    "manifest itself is missing "
                    "or ambiguous inside ZIP"
                )

            else:
                archived_manifest_name = next(
                    iter(manifest_members)
                )

                archived_manifest_hash = (
                    hashlib.sha256(
                        archive.read(
                            archived_manifest_name
                        )
                    ).hexdigest()
                )

                current_manifest_hash = (
                    sha256_file(
                        manifest_path
                    )
                )

                if (
                    archived_manifest_hash
                    != current_manifest_hash
                ):
                    failures.append(
                        "manifest inside ZIP does not "
                        "match current manifest file"
                    )

            for row_number, row in enumerate(
                rows,
                start=2,
            ):
                role = str(
                    row.get(
                        "role",
                        (
                            "manifest row "
                            f"{row_number}"
                        ),
                    )
                )

                member_name = locate_archive_member(
                    archive_names,
                    row,
                )

                explicitly_included = (
                    include_column is not None
                    and true_value(
                        row.get(
                            include_column,
                            False,
                        )
                    )
                )

                # When an inclusion column exists, rows explicitly
                # marked for inclusion must exist.

                if (
                    explicitly_included
                    and member_name is None
                ):
                    failures.append(
                        "required ZIP member missing "
                        f"for {role}"
                    )
                    continue

                # When no inclusion column exists, a missing member
                # is treated as an external hash reference.

                if member_name is None:
                    continue

                # The manifest itself is verified separately.

                if member_name in manifest_members:
                    continue

                expected_hash = str(
                    row.get(
                        "sha256",
                        "",
                    )
                ).strip().lower()

                valid_hash_format = (
                    len(expected_hash) == 64
                    and all(
                        character
                        in "0123456789abcdef"
                        for character
                        in expected_hash
                    )
                )

                if not valid_hash_format:
                    failures.append(
                        "invalid manifest SHA-256 "
                        f"for {role}"
                    )
                    continue

                observed_hash = hashlib.sha256(
                    archive.read(
                        member_name
                    )
                ).hexdigest()

                if observed_hash != expected_hash:
                    failures.append(
                        "ZIP member hash mismatch "
                        f"for {role}"
                    )
                    continue

                validated_members.add(
                    member_name
                )

            # Every non-manifest file actually contained in the ZIP
            # must have been declared and validated.

            payload_members = (
                archive_names
                - manifest_members
            )

            undeclared_members = sorted(
                payload_members
                - validated_members
            )

            if undeclared_members:
                failures.append(
                    "ZIP contains undeclared or "
                    "unvalidated files: "
                    + ", ".join(
                        undeclared_members
                    )
                )

            if not validated_members:
                failures.append(
                    "no ZIP payload files were "
                    "validated against the manifest"
                )

    except zipfile.BadZipFile:
        failures.append(
            "file is not a valid ZIP archive"
        )

    print(
        "ZIP VALIDATOR VERSION: "
        "SCHEMA_TOLERANT_V3"
    )
    print(
        f"Manifest: {manifest_path.name}"
    )
    print(
        "Inclusion column: "
        + (
            include_column
            if include_column is not None
            else (
                "none — archive-driven "
                "validation applied"
            )
        )
    )
    print(
        "Validated ZIP payload files: "
        f"{len(validated_members)}"
    )

    return (
        len(failures) == 0,
        failures,
    )

phase5_zip_valid, phase5_zip_failures = (
    validate_zip_against_manifest(
        PHASE5_COMPLETE_ZIP_PATH,
        PHASE5_COMPLETE_MANIFEST_PATH,
    )
)

step1_zip_valid, step1_zip_failures = (
    validate_zip_against_manifest(
        STEP1_ZIP_PATH,
        STEP1_MANIFEST_PATH,
    )
)

step2_zip_valid, step2_zip_failures = (
    validate_zip_against_manifest(
        STEP2_ZIP_PATH,
        STEP2_MANIFEST_PATH,
    )
)


# =============================================================================
# 7. LOAD SAVED RECORDS AND SAVED RESULTS
# =============================================================================

registry = load_json(
    FINAL_REGISTRY_PATH
)

policy = load_json(
    POLICY_PATH
)

evaluation_record = load_json(
    EVALUATION_RECORD_PATH
)

endpoint_results = pd.read_csv(
    ENDPOINT_RESULTS_PATH
)

summary_metrics = pd.read_csv(
    SUMMARY_METRICS_PATH
)

lifecycle_metrics = pd.read_csv(
    LIFECYCLE_METRICS_PATH
)

if len(summary_metrics) != 1:
    raise AssertionError(
        "The Phase 6 summary must contain exactly one row."
    )

summary = summary_metrics.iloc[0]


# =============================================================================
# 8. CURRENT OBSERVED HASH AUTHORITY
# =============================================================================

current_hashes = {
    "primary_model": sha256_file(
        PRIMARY_MODEL_PATH
    ),
    "conformal_bundle": sha256_file(
        CONFORMAL_BUNDLE_PATH
    ),
    "phase6_policy": sha256_file(
        POLICY_PATH
    ),
    "phase5_complete_zip": sha256_file(
        PHASE5_COMPLETE_ZIP_PATH
    ),
    "phase6_step1_zip": sha256_file(
        STEP1_ZIP_PATH
    ),
    "phase6_step2_zip": sha256_file(
        STEP2_ZIP_PATH
    ),
    "phase6_evaluation_record": sha256_file(
        EVALUATION_RECORD_PATH
    ),
    "phase6_endpoint_results": sha256_file(
        ENDPOINT_RESULTS_PATH
    ),
    "phase6_summary_metrics": sha256_file(
        SUMMARY_METRICS_PATH
    ),
    "phase6_lifecycle_metrics": sha256_file(
        LIFECYCLE_METRICS_PATH
    ),
    "phase6_evaluation_figure": sha256_file(
        FIGURE_PATH
    ),
}

legacy_hash_comparison = {
    name: {
        "legacy_sha256": legacy_hash,
        "observed_sha256": current_hashes[name],
        "matches_legacy": (
            current_hashes[name]
            == legacy_hash
        ),
    }
    for name, legacy_hash
    in LEGACY_HASHES.items()
}


# =============================================================================
# 9. CROSS-RECORD GOVERNANCE VALIDATION
# =============================================================================

policy_population = policy[
    "study_population"
]

policy_point_model = policy[
    "frozen_point_model"
]

policy_uncertainty = policy[
    "frozen_uncertainty_model"
]

policy_controls = policy[
    "interpretation_controls"
]

evaluation_controls = evaluation_record[
    "research_controls"
]

governance_checks = {
    "Phase 4 registry selects Random Forest": (
        registry["selected_model"]
        == "random_forest_uncapped"
    ),

    "Phase 4 registry preserves primary model hash": (
        registry["selected_model_sha256"]
        == EXPECTED_PRIMARY_MODEL_SHA256
    ),

    "primary model remains byte-identical": (
        current_hashes["primary_model"]
        == EXPECTED_PRIMARY_MODEL_SHA256
    ),

    "policy selects Random Forest": (
        policy_point_model["name"]
        == "random_forest_uncapped"
    ),

    "policy preserves primary model hash": (
        policy_point_model["bundle_sha256"]
        == current_hashes["primary_model"]
    ),

    "policy conformal hash matches actual conformal file": (
        policy_uncertainty["bundle_sha256"]
        == current_hashes["conformal_bundle"]
    ),

    "evaluation conformal hash matches actual conformal file": (
        evaluation_record["conformal_bundle_sha256"]
        == current_hashes["conformal_bundle"]
    ),

    "policy and evaluation conformal hashes agree": (
        policy_uncertainty["bundle_sha256"]
        == evaluation_record[
            "conformal_bundle_sha256"
        ]
    ),

    "evaluation policy hash matches actual policy file": (
        evaluation_record["policy_sha256"]
        == current_hashes["phase6_policy"]
    ),

    "policy preserves official predictor fingerprint": (
        policy_population["test_predictor_sha256"]
        == EXPECTED_TEST_PREDICTOR_SHA256
    ),

    "policy preserves official RUL fingerprint": (
        policy_population["test_rul_sha256"]
        == EXPECTED_TEST_RUL_SHA256
    ),

    "evaluation preserves predictor fingerprint": (
        evaluation_record["test_predictor_sha256"]
        == EXPECTED_TEST_PREDICTOR_SHA256
    ),

    "evaluation preserves RUL fingerprint": (
        evaluation_record["test_rul_sha256"]
        == EXPECTED_TEST_RUL_SHA256
    ),

    "policy preserves eighteen features": (
        int(
            policy_point_model["feature_count"]
        )
        == EXPECTED_FEATURES
    ),

    "policy preserves q_hat": close_enough(
        policy_uncertainty["q_hat_cycles"],
        EXPECTED_Q_HAT,
        tolerance=1e-5,
    ),

    "policy preserves ninety-percent confidence": (
        close_enough(
            policy_uncertainty[
                "confidence_level"
            ],
            EXPECTED_CONFIDENCE,
        )
    ),

    "policy evaluates only selected model": (
        policy_controls[
            "only_selected_primary_model_is_evaluated"
        ]
        is True
    ),

    "official results cannot reselect model": (
        policy_controls[
            "official_test_results_may_change_selected_model"
        ]
        is False
    ),

    "evaluation results are frozen": (
        evaluation_record["status"]
        == (
            "official_test_evaluation_"
            "complete_results_frozen"
        )
    ),

    "evaluation retains one hundred engines": (
        int(
            evaluation_record[
                "test_engine_count"
            ]
        )
        == EXPECTED_TEST_ENGINES
    ),

    "alternative models were not evaluated": (
        evaluation_controls[
            "alternative_models_evaluated"
        ]
        is False
    ),

    "model was not modified": (
        evaluation_controls[
            "model_retrained_or_modified"
        ]
        is False
    ),

    "model was not reselected": (
        evaluation_controls[
            "model_reselected"
        ]
        is False
    ),

    "conformal method was not changed": (
        evaluation_controls[
            "conformal_method_changed"
        ]
        is False
    ),

    "q_hat was not changed": (
        evaluation_controls[
            "q_hat_changed"
        ]
        is False
    ),

    "test cases were not removed": (
        evaluation_controls[
            "test_cases_removed"
        ]
        is False
    ),

    "Phase 5 ZIP contents match its manifest": (
        phase5_zip_valid
    ),

    "Phase 6 Step 1 ZIP contents match its manifest": (
        step1_zip_valid
    ),

    "Phase 6 Step 2 ZIP contents match its manifest": (
        step2_zip_valid
    ),
}

failed_governance_checks = [
    check_name
    for check_name, passed
    in governance_checks.items()
    if not passed
]

if failed_governance_checks:
    detail_lines = []

    if not phase5_zip_valid:
        detail_lines.extend(
            "Phase 5 ZIP — " + item
            for item in phase5_zip_failures
        )

    if not step1_zip_valid:
        detail_lines.extend(
            "Step 1 ZIP — " + item
            for item in step1_zip_failures
        )

    if not step2_zip_valid:
        detail_lines.extend(
            "Step 2 ZIP — " + item
            for item in step2_zip_failures
        )

    details = (
        "\nDetails:\n"
        + "\n".join(detail_lines)
        if detail_lines
        else ""
    )

    raise AssertionError(
        "Phase 6 cross-record governance "
        "validation failed:\n"
        + "\n".join(
            failed_governance_checks
        )
        + details
    )


# =============================================================================
# 10. REPRODUCE SAVED RESULTS
# =============================================================================

required_endpoint_columns = {
    "unit_number",
    "actual_endpoint_rul",
    "raw_point_prediction",
    "point_prediction",
    "error_prediction_minus_actual",
    "absolute_error",
    "squared_error",
    "phm_penalty",
    "q_hat",
    "interval_lower",
    "interval_upper",
    "interval_width",
    "lower_bound_was_clipped",
    "covered",
    "miss_below_interval",
    "miss_above_interval",
    "interval_score_alpha_0_10",
    "lifecycle_stage",
}

missing_endpoint_columns = (
    required_endpoint_columns
    - set(endpoint_results.columns)
)

if missing_endpoint_columns:
    raise AssertionError(
        "Endpoint results are missing columns:\n"
        + "\n".join(
            sorted(missing_endpoint_columns)
        )
    )

actual_rul = endpoint_results[
    "actual_endpoint_rul"
].to_numpy(dtype=float)

point_predictions = endpoint_results[
    "point_prediction"
].to_numpy(dtype=float)

saved_errors = endpoint_results[
    "error_prediction_minus_actual"
].to_numpy(dtype=float)

saved_absolute_errors = endpoint_results[
    "absolute_error"
].to_numpy(dtype=float)

saved_squared_errors = endpoint_results[
    "squared_error"
].to_numpy(dtype=float)

saved_phm_penalty = endpoint_results[
    "phm_penalty"
].to_numpy(dtype=float)

saved_q_hat = endpoint_results[
    "q_hat"
].to_numpy(dtype=float)

saved_lower = endpoint_results[
    "interval_lower"
].to_numpy(dtype=float)

saved_upper = endpoint_results[
    "interval_upper"
].to_numpy(dtype=float)

saved_width = endpoint_results[
    "interval_width"
].to_numpy(dtype=float)

saved_clipped = parse_boolean_series(
    endpoint_results[
        "lower_bound_was_clipped"
    ],
    "lower_bound_was_clipped",
)

saved_covered = parse_boolean_series(
    endpoint_results["covered"],
    "covered",
)

saved_miss_below = parse_boolean_series(
    endpoint_results[
        "miss_below_interval"
    ],
    "miss_below_interval",
)

saved_miss_above = parse_boolean_series(
    endpoint_results[
        "miss_above_interval"
    ],
    "miss_above_interval",
)

saved_interval_score = endpoint_results[
    "interval_score_alpha_0_10"
].to_numpy(dtype=float)

errors = point_predictions - actual_rul
absolute_errors = np.abs(errors)
squared_errors = np.square(errors)

phm_penalty = np.where(
    errors < 0,
    np.exp(-errors / 13.0) - 1.0,
    np.exp(errors / 10.0) - 1.0,
)

q_hat = float(
    summary["q_hat_cycles"]
)

alpha = 1.0 - float(
    summary["conformal_confidence_level"]
)

interval_lower = np.maximum(
    0.0,
    point_predictions - q_hat,
)

interval_upper = (
    point_predictions + q_hat
)

interval_width = (
    interval_upper - interval_lower
)

lower_clipped = (
    point_predictions - q_hat < 0.0
)

covered = (
    (actual_rul >= interval_lower - TOLERANCE)
    & (actual_rul <= interval_upper + TOLERANCE)
)

miss_below = (
    actual_rul < interval_lower - TOLERANCE
)

miss_above = (
    actual_rul > interval_upper + TOLERANCE
)

interval_score = (
    interval_width
    + (2.0 / alpha)
    * (interval_lower - actual_rul)
    * miss_below.astype(float)
    + (2.0 / alpha)
    * (actual_rul - interval_upper)
    * miss_above.astype(float)
)

mae = float(
    np.mean(absolute_errors)
)

rmse = float(
    np.sqrt(
        np.mean(squared_errors)
    )
)

bias = float(
    np.mean(errors)
)

median_absolute_error = float(
    np.median(absolute_errors)
)

sum_squared_error = float(
    np.sum(squared_errors)
)

total_sum_of_squares = float(
    np.sum(
        np.square(
            actual_rul
            - np.mean(actual_rul)
        )
    )
)

r_squared = float(
    1.0
    - sum_squared_error
    / total_sum_of_squares
)

coverage_count = int(
    np.sum(covered)
)

coverage_rate = float(
    np.mean(covered)
)

coverage_z = float(
    policy["interval_metrics"][
        "coverage_z_value"
    ]
)

wilson_lower, wilson_upper = (
    wilson_interval(
        coverage_count,
        EXPECTED_TEST_ENGINES,
        coverage_z,
    )
)

lifecycle_stage = assign_lifecycle_stage(
    actual_rul
)

result_checks = {
    "endpoint results contain one hundred engines": (
        len(endpoint_results)
        == EXPECTED_TEST_ENGINES
    ),

    "endpoint identifiers are unique": (
        endpoint_results[
            "unit_number"
        ].is_unique
    ),

    "endpoint identifiers are one through one hundred": (
        np.array_equal(
            np.sort(
                endpoint_results[
                    "unit_number"
                ].to_numpy(dtype=int)
            ),
            np.arange(
                1,
                EXPECTED_TEST_ENGINES + 1,
            ),
        )
    ),

    "saved errors reproduce": all_close(
        saved_errors,
        errors,
    ),

    "saved absolute errors reproduce": all_close(
        saved_absolute_errors,
        absolute_errors,
    ),

    "saved squared errors reproduce": all_close(
        saved_squared_errors,
        squared_errors,
    ),

    "saved PHM penalties reproduce": all_close(
        saved_phm_penalty,
        phm_penalty,
    ),

    "saved q_hat values reproduce": all_close(
        saved_q_hat,
        np.full(
            EXPECTED_TEST_ENGINES,
            q_hat,
        ),
    ),

    "saved lower bounds reproduce": all_close(
        saved_lower,
        interval_lower,
    ),

    "saved upper bounds reproduce": all_close(
        saved_upper,
        interval_upper,
    ),

    "saved interval widths reproduce": all_close(
        saved_width,
        interval_width,
    ),

    "saved clipping indicators reproduce": (
        np.array_equal(
            saved_clipped,
            lower_clipped,
        )
    ),

    "saved coverage indicators reproduce": (
        np.array_equal(
            saved_covered,
            covered,
        )
    ),

    "saved miss-below indicators reproduce": (
        np.array_equal(
            saved_miss_below,
            miss_below,
        )
    ),

    "saved miss-above indicators reproduce": (
        np.array_equal(
            saved_miss_above,
            miss_above,
        )
    ),

    "saved interval scores reproduce": all_close(
        saved_interval_score,
        interval_score,
    ),

    "summary MAE reproduces": close_enough(
        summary["mae"],
        mae,
    ),

    "summary RMSE reproduces": close_enough(
        summary["rmse"],
        rmse,
    ),

    "summary R-squared reproduces": close_enough(
        summary["r_squared"],
        r_squared,
    ),

    "summary bias reproduces": close_enough(
        summary[
            "bias_prediction_minus_actual"
        ],
        bias,
    ),

    "summary median absolute error reproduces": (
        close_enough(
            summary[
                "median_absolute_error"
            ],
            median_absolute_error,
        )
    ),

    "summary coverage count reproduces": (
        int(
            summary[
                "endpoint_coverage_count"
            ]
        )
        == coverage_count
    ),

    "summary coverage rate reproduces": (
        close_enough(
            summary[
                "endpoint_coverage_rate"
            ],
            coverage_rate,
        )
    ),

    "summary Wilson lower bound reproduces": (
        close_enough(
            summary[
                "coverage_wilson_95_lower"
            ],
            wilson_lower,
        )
    ),

    "summary Wilson upper bound reproduces": (
        close_enough(
            summary[
                "coverage_wilson_95_upper"
            ],
            wilson_upper,
        )
    ),

    "saved lifecycle labels reproduce": (
        np.array_equal(
            endpoint_results[
                "lifecycle_stage"
            ].to_numpy(dtype=str),
            lifecycle_stage,
        )
    ),

    "lifecycle counts sum to one hundred": (
        int(
            lifecycle_metrics[
                "engine_count"
            ].sum()
        )
        == EXPECTED_TEST_ENGINES
    ),

    "official evaluation figure is nonempty": (
        FIGURE_PATH.stat().st_size > 0
    ),
}

failed_result_checks = [
    check_name
    for check_name, passed
    in result_checks.items()
    if not passed
]

if failed_result_checks:
    raise AssertionError(
        "Saved Phase 6 result reproduction failed:\n"
        + "\n".join(
            failed_result_checks
        )
    )


# =============================================================================
# 11. SNAPSHOT IMMUTABLE INPUTS
# =============================================================================

immutable_paths = [
    FINAL_REGISTRY_PATH,
    PHASE5_COMPLETE_MANIFEST_PATH,
    PHASE5_COMPLETE_ZIP_PATH,
    POLICY_PATH,
    DECISION_RECORD_PATH,
    STEP1_MANIFEST_PATH,
    STEP1_ZIP_PATH,
    PRIMARY_MODEL_PATH,
    CONFORMAL_BUNDLE_PATH,
    ENDPOINT_RESULTS_PATH,
    SUMMARY_METRICS_PATH,
    LIFECYCLE_METRICS_PATH,
    FIGURE_PATH,
    EVALUATION_RECORD_PATH,
    STEP2_MANIFEST_PATH,
    STEP2_ZIP_PATH,
]

hashes_before = {
    str(path): sha256_file(path)
    for path in immutable_paths
}


# =============================================================================
# 12. WRITE CURRENT TECHNICAL COMPLETION RECORD
# =============================================================================

completion_record = {
    "phase": 6,
    "status": (
        "official_test_evaluation_"
        "technically_complete"
    ),
    "formal_approval_status": (
        "approved_for_formal_closure"
    ),
    "completed_at_utc": utc_now(),

    "dataset": "NASA C-MAPSS FD001",

    "evaluation_grain": (
        "one_last_observed_endpoint_"
        "per_test_engine"
    ),

    "official_test_engine_count": (
        EXPECTED_TEST_ENGINES
    ),

    "hash_authority": {
        "method": (
            "current_byte_hashes_validated_"
            "against_cross_referenced_"
            "policy_evaluation_and_manifests"
        ),
        "legacy_hash_comparison": (
            legacy_hash_comparison
        ),
    },

    "frozen_foundation": {
        "selected_model": (
            "random_forest_uncapped"
        ),
        "selected_model_sha256": (
            current_hashes[
                "primary_model"
            ]
        ),
        "conformal_bundle_sha256": (
            current_hashes[
                "conformal_bundle"
            ]
        ),
        "phase6_policy_sha256": (
            current_hashes[
                "phase6_policy"
            ]
        ),
        "phase5_complete_zip_sha256": (
            current_hashes[
                "phase5_complete_zip"
            ]
        ),
        "phase6_step1_zip_sha256": (
            current_hashes[
                "phase6_step1_zip"
            ]
        ),
        "phase6_step2_zip_sha256": (
            current_hashes[
                "phase6_step2_zip"
            ]
        ),
        "q_hat_cycles": q_hat,
        "confidence_level": (
            float(
                summary[
                    "conformal_confidence_level"
                ]
            )
        ),
        "test_predictor_sha256": (
            EXPECTED_TEST_PREDICTOR_SHA256
        ),
        "test_rul_sha256": (
            EXPECTED_TEST_RUL_SHA256
        ),
    },

    "official_point_metrics": {
        "mae": mae,
        "rmse": rmse,
        "r_squared": r_squared,
        "bias_prediction_minus_actual": (
            bias
        ),
        "median_absolute_error": (
            median_absolute_error
        ),
        "phm_total_penalty": float(
            np.sum(phm_penalty)
        ),
        "phm_mean_penalty": float(
            np.mean(phm_penalty)
        ),
    },

    "official_interval_metrics": {
        "endpoint_coverage_count": (
            coverage_count
        ),
        "endpoint_coverage_rate": (
            coverage_rate
        ),
        "coverage_wilson_95_lower": (
            wilson_lower
        ),
        "coverage_wilson_95_upper": (
            wilson_upper
        ),
        "mean_interval_width": float(
            np.mean(interval_width)
        ),
        "median_interval_width": float(
            np.median(interval_width)
        ),
        "lower_clipping_rate": float(
            np.mean(lower_clipped)
        ),
        "miss_below_count": int(
            np.sum(miss_below)
        ),
        "miss_above_count": int(
            np.sum(miss_above)
        ),
    },

    "interpretation_controls": {
        "official_results_are_final": True,
        "official_results_may_reselect_model": False,
        "endpoint_coverage_is_safety_guarantee": False,
        "complete_coverage_means_precise_intervals": False,
        "wide_intervals_must_be_reported": True,
        "positive_bias_must_be_reported": True,
        "lifecycle_results_are_descriptive_only": True,
    },

    "research_controls": {
        "model_inference_repeated": False,
        "raw_test_values_reparsed": False,
        "raw_test_rul_reparsed": False,
        "alternative_models_evaluated": False,
        "model_retrained_or_modified": False,
        "model_reselected": False,
        "features_changed": False,
        "hyperparameters_changed": False,
        "conformal_method_changed": False,
        "q_hat_changed": False,
        "test_cases_removed": False,
        "post_hoc_threshold_applied": False,
    },

    "official_output_files": {
        "endpoint_results": (
            str(ENDPOINT_RESULTS_PATH)
        ),
        "summary_metrics": (
            str(SUMMARY_METRICS_PATH)
        ),
        "lifecycle_metrics": (
            str(LIFECYCLE_METRICS_PATH)
        ),
        "evaluation_figure": (
            str(FIGURE_PATH)
        ),
        "evaluation_record": (
            str(EVALUATION_RECORD_PATH)
        ),
    },
}

write_json_atomic(
    COMPLETION_RECORD_PATH,
    completion_record,
)


# =============================================================================
# 13. BUILD COMPLETE MANIFEST
# =============================================================================

manifest_entries = [
    (
        "phase4_final_model_registry",
        FINAL_REGISTRY_PATH,
        True,
    ),
    (
        "phase5_complete_manifest",
        PHASE5_COMPLETE_MANIFEST_PATH,
        True,
    ),
    (
        "phase5_complete_zip_hash_reference",
        PHASE5_COMPLETE_ZIP_PATH,
        False,
    ),
    (
        "phase6_frozen_test_policy",
        POLICY_PATH,
        True,
    ),
    (
        "phase6_test_policy_decision",
        DECISION_RECORD_PATH,
        True,
    ),
    (
        "phase6_step1_manifest",
        STEP1_MANIFEST_PATH,
        True,
    ),
    (
        "phase6_step1_zip_hash_reference",
        STEP1_ZIP_PATH,
        False,
    ),
    (
        "primary_model_hash_reference",
        PRIMARY_MODEL_PATH,
        False,
    ),
    (
        "conformal_bundle_hash_reference",
        CONFORMAL_BUNDLE_PATH,
        False,
    ),
    (
        "official_endpoint_results",
        ENDPOINT_RESULTS_PATH,
        True,
    ),
    (
        "official_summary_metrics",
        SUMMARY_METRICS_PATH,
        True,
    ),
    (
        "official_lifecycle_metrics",
        LIFECYCLE_METRICS_PATH,
        True,
    ),
    (
        "official_evaluation_figure",
        FIGURE_PATH,
        True,
    ),
    (
        "official_evaluation_record",
        EVALUATION_RECORD_PATH,
        True,
    ),
    (
        "phase6_step2_manifest",
        STEP2_MANIFEST_PATH,
        True,
    ),
    (
        "phase6_step2_zip_hash_reference",
        STEP2_ZIP_PATH,
        False,
    ),
    (
        "phase6_completion_record",
        COMPLETION_RECORD_PATH,
        True,
    ),
]

manifest_rows_output = []

for role, path, include_in_zip in manifest_entries:
    manifest_rows_output.append(
        {
            "role": role,
            "path": str(path),
            "project_relative_path": str(
                path.relative_to(ROOT)
            ).replace("\\", "/"),
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
            "included_in_complete_zip": (
                include_in_zip
            ),
        }
    )

with COMPLETE_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
    newline="",
) as file_handle:
    writer = csv.DictWriter(
        file_handle,
        fieldnames=[
            "role",
            "path",
            "project_relative_path",
            "size_bytes",
            "sha256",
            "included_in_complete_zip",
        ],
    )

    writer.writeheader()
    writer.writerows(
        manifest_rows_output
    )


# =============================================================================
# 14. CREATE DETERMINISTIC COMPLETE ZIP
# =============================================================================

def write_deterministic_zip(
    destination,
    source_paths,
):
    ordered_paths = sorted(
        set(source_paths),
        key=lambda path: str(
            path.relative_to(ROOT)
        ),
    )

    with zipfile.ZipFile(
        destination,
        "w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=9,
    ) as archive:
        for source_path in ordered_paths:
            archive_name = str(
                source_path.relative_to(ROOT)
            ).replace("\\", "/")

            zip_info = zipfile.ZipInfo(
                archive_name,
                date_time=(
                    1980,
                    1,
                    1,
                    0,
                    0,
                    0,
                ),
            )

            zip_info.compress_type = (
                zipfile.ZIP_DEFLATED
            )

            zip_info.external_attr = (
                0o100644 << 16
            )

            archive.writestr(
                zip_info,
                source_path.read_bytes(),
                compress_type=(
                    zipfile.ZIP_DEFLATED
                ),
            )


complete_zip_sources = [
    path
    for _, path, include_in_zip
    in manifest_entries
    if include_in_zip
]

complete_zip_sources.append(
    COMPLETE_MANIFEST_PATH
)

write_deterministic_zip(
    COMPLETE_ZIP_PATH,
    complete_zip_sources,
)

complete_zip_hash = sha256_file(
    COMPLETE_ZIP_PATH
)


# =============================================================================
# 15. POST-PACKAGING IMMUTABILITY
# =============================================================================

hashes_after = {
    str(path): sha256_file(path)
    for path in immutable_paths
}

post_checks = {
    "all immutable inputs remained byte-identical": (
        hashes_before == hashes_after
    ),

    "primary model remained unchanged": (
        hashes_after[
            str(PRIMARY_MODEL_PATH)
        ]
        == EXPECTED_PRIMARY_MODEL_SHA256
    ),

    "policy remained unchanged during completion": (
        hashes_after[
            str(POLICY_PATH)
        ]
        == current_hashes[
            "phase6_policy"
        ]
    ),

    "conformal bundle remained unchanged during completion": (
        hashes_after[
            str(CONFORMAL_BUNDLE_PATH)
        ]
        == current_hashes[
            "conformal_bundle"
        ]
    ),

    "completion record exists": (
        COMPLETION_RECORD_PATH.is_file()
    ),

    "complete manifest exists": (
        COMPLETE_MANIFEST_PATH.is_file()
    ),

    "complete ZIP exists and is nonempty": (
        COMPLETE_ZIP_PATH.is_file()
        and COMPLETE_ZIP_PATH.stat().st_size > 0
    ),
}

failed_post_checks = [
    check_name
    for check_name, passed
    in post_checks.items()
    if not passed
]

if failed_post_checks:
    raise AssertionError(
        "Phase 6 post-packaging validation failed:\n"
        + "\n".join(
            failed_post_checks
        )
    )


# =============================================================================
# 16. VALIDATE NEW COMPLETE ZIP AGAINST NEW MANIFEST
# =============================================================================

complete_zip_valid, complete_zip_failures = (
    validate_zip_against_manifest(
        COMPLETE_ZIP_PATH,
        COMPLETE_MANIFEST_PATH,
    )
)

if not complete_zip_valid:
    raise AssertionError(
        "New Phase 6 complete ZIP failed manifest validation:\n"
        + "\n".join(
            complete_zip_failures
        )
    )


# =============================================================================
# 17. CREATE OR VERIFY FORMAL CLOSURE
# =============================================================================

completion_record_hash = sha256_file(
    COMPLETION_RECORD_PATH
)

complete_manifest_hash = sha256_file(
    COMPLETE_MANIFEST_PATH
)

new_closure_record = {
    "phase": 6,
    "status": (
        "formally_approved_and_closed"
    ),
    "approved_by": "Ranitabh Mallick",
    "approved_at_utc": utc_now(),

    "complete_evidence_zip": (
        str(COMPLETE_ZIP_PATH)
    ),
    "complete_evidence_zip_sha256": (
        complete_zip_hash
    ),

    "complete_manifest": (
        str(COMPLETE_MANIFEST_PATH)
    ),
    "complete_manifest_sha256": (
        complete_manifest_hash
    ),

    "technical_completion_record": (
        str(COMPLETION_RECORD_PATH)
    ),
    "technical_completion_record_sha256": (
        completion_record_hash
    ),

    "hash_authority": (
        "closure_record_observed_byte_hashes"
    ),

    "official_results_are_final": True,
    "model_reselection_permitted": False,
    "model_retraining_permitted": False,
    "conformal_recalibration_permitted": False,
    "official_test_re_evaluation_permitted": False,
}

if CLOSURE_RECORD_PATH.is_file():
    existing_closure = load_json(
        CLOSURE_RECORD_PATH
    )

    existing_closure_valid = (
        existing_closure.get("status")
        == "formally_approved_and_closed"
        and existing_closure.get(
            "complete_evidence_zip_sha256"
        )
        == complete_zip_hash
        and existing_closure.get(
            "complete_manifest_sha256"
        )
        == complete_manifest_hash
        and existing_closure.get(
            "technical_completion_record_sha256"
        )
        == completion_record_hash
        and existing_closure.get(
            "official_results_are_final"
        )
        is True
    )

    if not existing_closure_valid:
        # A previous closure no longer describes the corrected package.
        # Preserve it as a superseded governance record.

        superseded_closure_path = (
            LOGS
            / (
                "phase6_closure_record_"
                "superseded.json"
            )
        )

        if not superseded_closure_path.exists():
            CLOSURE_RECORD_PATH.replace(
                superseded_closure_path
            )

        write_json_atomic(
            CLOSURE_RECORD_PATH,
            new_closure_record,
        )

        closure_action = (
            "superseded_stale_closure_and_created_"
            "current_closure"
        )

    else:
        closure_action = (
            "retained_existing_valid_closure"
        )

else:
    write_json_atomic(
        CLOSURE_RECORD_PATH,
        new_closure_record,
    )

    closure_action = (
        "created_current_formal_closure"
    )


closure_record = load_json(
    CLOSURE_RECORD_PATH
)

closure_checks = {
    "completion record is technically complete": (
        completion_record["status"]
        == (
            "official_test_evaluation_"
            "technically_complete"
        )
    ),

    "complete ZIP matches closure record": (
        closure_record[
            "complete_evidence_zip_sha256"
        ]
        == sha256_file(
            COMPLETE_ZIP_PATH
        )
    ),

    "complete manifest matches closure record": (
        closure_record[
            "complete_manifest_sha256"
        ]
        == sha256_file(
            COMPLETE_MANIFEST_PATH
        )
    ),

    "completion record matches closure record": (
        closure_record[
            "technical_completion_record_sha256"
        ]
        == sha256_file(
            COMPLETION_RECORD_PATH
        )
    ),

    "official results are final": (
        closure_record[
            "official_results_are_final"
        ]
        is True
    ),

    "model inference was not repeated": (
        completion_record[
            "research_controls"
        ][
            "model_inference_repeated"
        ]
        is False
    ),

    "raw official values were not reparsed": (
        completion_record[
            "research_controls"
        ][
            "raw_test_values_reparsed"
        ]
        is False
    ),

    "formal closure record is valid": (
        closure_record["status"]
        == "formally_approved_and_closed"
    ),
}

failed_closure_checks = [
    check_name
    for check_name, passed
    in closure_checks.items()
    if not passed
]

if failed_closure_checks:
    raise AssertionError(
        "Phase 6 formal closure validation failed:\n"
        + "\n".join(
            failed_closure_checks
        )
    )


# =============================================================================
# 18. FINAL OUTPUT
# =============================================================================

print("PHASE 6 HASH RECONCILIATION")
print("=" * 90)
print(f"Active project root: {ROOT}")
print(
    "Hash authority: current files validated through "
    "policy, evaluation record and evidence manifests"
)

for name, comparison in legacy_hash_comparison.items():
    status = (
        "UNCHANGED"
        if comparison["matches_legacy"]
        else "RECONCILED"
    )

    print(
        f"{status} — {name}"
    )
    print(
        "  legacy:   "
        f"{comparison['legacy_sha256']}"
    )
    print(
        "  observed: "
        f"{comparison['observed_sha256']}"
    )

print("\nPHASE 6 GOVERNANCE VALIDATION")
print("=" * 90)

for check_name, passed in governance_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL'} — "
        f"{check_name}"
    )

print("\nSAVED RESULT REPRODUCTION")
print("=" * 90)

for check_name, passed in result_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL'} — "
        f"{check_name}"
    )

print("\nOFFICIAL RESULTS")
print("=" * 90)
print(
    f"Official test engines: "
    f"{EXPECTED_TEST_ENGINES}"
)
print(
    f"MAE: {mae:.6f} cycles"
)
print(
    f"RMSE: {rmse:.6f} cycles"
)
print(
    f"R-squared: {r_squared:.6f}"
)
print(
    "Bias (prediction - actual): "
    f"{bias:.6f} cycles"
)
print(
    f"Endpoint coverage: "
    f"{coverage_count} of "
    f"{EXPECTED_TEST_ENGINES} "
    f"({coverage_rate:.2%})"
)
print(
    "Mean interval width: "
    f"{np.mean(interval_width):.6f} cycles"
)
print(
    "Lower clipping rate: "
    f"{np.mean(lower_clipped):.2%}"
)

print("\nPHASE 6 COMPLETION PACKAGE")
print("=" * 90)
print(
    f"Completion record: "
    f"{COMPLETION_RECORD_PATH}"
)
print(
    f"Completion record SHA-256: "
    f"{completion_record_hash}"
)
print(
    f"Complete manifest: "
    f"{COMPLETE_MANIFEST_PATH}"
)
print(
    f"Complete manifest SHA-256: "
    f"{complete_manifest_hash}"
)
print(
    f"Complete ZIP: "
    f"{COMPLETE_ZIP_PATH}"
)
print(
    f"Complete ZIP SHA-256: "
    f"{complete_zip_hash}"
)

print("\nPHASE 6 FORMAL CLOSURE")
print("=" * 90)

for check_name, passed in closure_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL'} — "
        f"{check_name}"
    )

print(
    f"Closure action: {closure_action}"
)
print(
    f"Closure record: "
    f"{CLOSURE_RECORD_PATH}"
)
print(
    f"Closure record SHA-256: "
    f"{sha256_file(CLOSURE_RECORD_PATH)}"
)

print("\nRESEARCH CONTROLS")
print("=" * 90)
print(
    "Model inference repeated: No"
)
print(
    "Raw official predictors parsed: No"
)
print(
    "Raw official RUL outcomes parsed: No"
)
print(
    "Model retrained or modified: No"
)
print(
    "Alternative model evaluated: No"
)
print(
    "Conformal method changed: No"
)
print(
    "q_hat changed: No"
)
print(
    "Official test cases removed: No"
)
print(
    "Browser download triggered: No"
)

print("\nFINAL CHECK")
print("=" * 90)
print(
    "PASS — Phase 6 technical completion was "
    "reproduced from saved results."
)
print(
    "PASS — The current policy, conformal bundle "
    "and evaluation record form one consistent "
    "hash chain."
)
print(
    "PASS — Prior evidence ZIPs were validated "
    "against their manifests."
)
print(
    "PASS — No modelling or official-test "
    "inference was repeated."
)
print(
    "PASS — Phase 6 is formally approved "
    "and closed."
)
print(
    "Official results are final and immutable."
)

In [ ]:
# =============================================================================
# READ-ONLY PHASE 5–6 PROVENANCE CHECK
#
# No files created, overwritten, extracted or deleted.
# No models loaded. No predictions generated.
# No raw test predictors or RUL outcomes opened.
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

ROOT = Path("/content/Predictive_Maintenance_Thesis")

ARTIFACTS = {
    "Phase 5 conformal policy":
        "logs/phase5_engine_block_conformal_policy.json",

    "Phase 5 calibration record":
        "logs/phase5_conformal_calibration_record.json",

    "Phase 6 test policy":
        "logs/phase6_official_test_evaluation_policy.json",

    "Phase 6 evaluation record":
        "logs/phase6_official_test_evaluation_record.json",

    "Phase 6 completion record":
        "logs/phase6_technical_completion_record.json",

    "Phase 6 closure record":
        "logs/phase6_closure_record.json",

    "Earlier closure retained by replacement code":
        "logs/phase6_closure_record_superseded.json",

    "Primary Random Forest":
        "models/regression/fd001_random_forest_uncapped_bundle.joblib",

    "Conformal bundle":
        "models/uncertainty/"
        "fd001_random_forest_engine_block_conformal_bundle.joblib",
}

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def relevant_fields(value, prefix=""):
    """Print provenance fields without dumping predictions or datasets."""
    if isinstance(value, dict):
        for key, child in value.items():
            location = f"{prefix}.{key}" if prefix else key
            if isinstance(child, (dict, list)):
                yield from relevant_fields(child, location)
            elif any(
                token in key.lower()
                for token in (
                    "sha256", "timestamp", "_at", "status",
                    "q_hat", "confidence", "inference",
                    "retrained", "reselected", "recalibrat",
                )
            ):
                yield location, child
    elif isinstance(value, list):
        for index, child in enumerate(value):
            if isinstance(child, (dict, list)):
                yield from relevant_fields(child, f"{prefix}[{index}]")

print("READ-ONLY PROVENANCE CHECK")
print("=" * 80)
print("Filesystem modification times are clues, not proof of execution history.")

for label, relative_path in ARTIFACTS.items():
    path = ROOT / relative_path

    print(f"\n{label}")
    print("-" * 80)
    print(f"Path: {path}")

    if not path.is_file():
        print("NOT PRESENT — no replacement created")
        continue

    print(f"SHA-256: {sha256_file(path)}")
    print(
        "Filesystem modified UTC:",
        datetime.fromtimestamp(
            path.stat().st_mtime, timezone.utc
        ).isoformat(),
    )

    if path.suffix == ".json":
        try:
            with path.open("r", encoding="utf-8") as handle:
                record = json.load(handle)
            for field, value in relevant_fields(record):
                print(f"{field}: {value}")
        except (ValueError, OSError) as error:
            print(f"RECORD READ ERROR: {error}")

print("\nCHECK FINISHED")
print("=" * 80)
print("Files modified: No")
print("Models loaded or predictions generated: No")
print("Raw official test files opened: No")
print("Historical hash differences automatically accepted: No")

In [ ]:
# =============================================================================
# PHASE 7 — STEP 1: EXPLAINABILITY AND THESIS VISUALISATION POLICY
# FINAL RERUN-SAFE VERSION — NO HARDCODED VOLATILE PHASE 5–6 HASHES
#
# WHY THIS VERSION EXISTS
# - Phase 5–6 governance artifacts legitimately change byte hashes on reruns because
#   several records/packages contain run timestamps and are regenerated.
# - Therefore Phase 7 must NOT hard-code the previous run's Phase 5–6 hashes.
# - Instead, this cell fingerprints the CURRENT completed Phase 5–6 snapshot,
#   validates all internal hash links, freezes that snapshot for Phase 7, and
#   preserves the historical-continuity limitation explicitly.
#
# IMPORTANT
# - Thesis structure is unchanged.
# - No model is trained or modified here.
# - No conformal calibration is recomputed here.
# - No official test evaluation is rerun here.
# - No SHAP value is computed here.
# - No Phase 4–6 file is overwritten here.
# - Downstream Phase 7 Step 2/3 compatibility is preserved through specification["inputs"].
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math


def phase7_setup():
    root = Path("/content/Predictive_Maintenance_Thesis")
    logs_dir = root / "logs"

    output_path = logs_dir / "phase7_current_snapshot_explainability_policy.json"
    n8n_contract_path = logs_dir / "phase7_n8n_hitl_agentic_ai_evidence_contract.json"

    # -------------------------------------------------------------------------
    # 1. CURRENT PHASE 5–6 INPUT SET
    #
    # These PATHS are frozen by the thesis design. Their hashes are intentionally
    # discovered at runtime because some Phase 5–6 files are rerun-generated.
    # -------------------------------------------------------------------------

    locked_files = {
        "primary_model": "models/regression/fd001_random_forest_uncapped_bundle.joblib",
        "conformal_bundle": "models/uncertainty/fd001_random_forest_engine_block_conformal_bundle.joblib",
        "calibration_policy": "logs/phase5_engine_block_conformal_policy.json",
        "calibration_record": "logs/phase5_conformal_calibration_record.json",
        "test_policy": "logs/phase6_official_test_evaluation_policy.json",
        "evaluation_record": "logs/phase6_official_test_evaluation_record.json",
        "completion_record": "logs/phase6_technical_completion_record.json",
        "closure_record": "logs/phase6_closure_record.json",
        "complete_manifest": "logs/phase6_complete_evidence_manifest.csv",
        "complete_package": "Phase6_Complete_Official_Test_Evidence.zip",
    }

    evidence_files = {
        "endpoint_results": "results/phase6_official_test_endpoint_predictions.csv",
        "summary_metrics": "results/phase6_official_test_summary_metrics.csv",
        "lifecycle_metrics": "results/phase6_official_test_lifecycle_stage_metrics.csv",
        "endpoint_figure": "results/figures/phase6_official_test_endpoint_evaluation.png",
        "calibration_scores": "results/phase5_engine_block_calibration_scores.csv",
    }

    relative_paths = {**locked_files, **evidence_files}

    def file_hash(path):
        digest = hashlib.sha256()
        with path.open("rb") as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(block)
        return digest.hexdigest()

    def read_json(path):
        with path.open("r", encoding="utf-8") as handle:
            value = json.load(handle)
        if not isinstance(value, dict):
            raise TypeError(f"Expected a JSON object: {path}")
        return value

    def require(condition, message):
        if not condition:
            raise AssertionError(message)

    # -------------------------------------------------------------------------
    # 2. Verify required files exist BEFORE establishing current authority.
    # -------------------------------------------------------------------------

    missing = [
        str(root / relative)
        for relative in relative_paths.values()
        if not (root / relative).is_file()
    ]

    if missing:
        raise FileNotFoundError(
            "Phase 7 inputs missing. No Phase 7 output was written:\n"
            + "\n".join(missing)
        )

    # Current runtime snapshot. This is the only correct authority for Phase 7.
    hashes_before = {
        role: file_hash(root / relative)
        for role, relative in relative_paths.items()
    }

    def verify_inputs_unchanged(stage_name):
        changed = []
        for role, relative in relative_paths.items():
            observed = file_hash(root / relative)
            if observed != hashes_before[role]:
                changed.append(
                    f"{role}: before={hashes_before[role]} after={observed}"
                )
        require(
            not changed,
            "A Phase 5–6 input changed DURING Phase 7 setup at "
            f"{stage_name}:\n" + "\n".join(changed),
        )

    # -------------------------------------------------------------------------
    # 3. Read CURRENT metadata only.
    # -------------------------------------------------------------------------

    test_policy = read_json(root / locked_files["test_policy"])
    calibration = read_json(root / locked_files["calibration_record"])
    evaluation = read_json(root / locked_files["evaluation_record"])
    closure = read_json(root / locked_files["closure_record"])

    require(
        "frozen_point_model" in test_policy,
        "Current Phase 6 test policy has no frozen_point_model block.",
    )
    require(
        "frozen_uncertainty_model" in test_policy,
        "Current Phase 6 test policy has no frozen_uncertainty_model block.",
    )

    point_policy = test_policy["frozen_point_model"]
    uncertainty_policy = test_policy["frozen_uncertainty_model"]

    # -------------------------------------------------------------------------
    # 4. Validate the predictor policy and leakage controls.
    # -------------------------------------------------------------------------

    feature_order = point_policy["feature_order"]

    require(
        isinstance(feature_order, list)
        and len(feature_order) == 18
        and len(set(feature_order)) == 18
        and all(isinstance(name, str) for name in feature_order),
        "Expected eighteen unique feature names in the saved policy.",
    )

    forbidden_features = {
        "unit_number",
        "rul_uncapped",
        "rul_capped_130",
        "actual_endpoint_rul",
    }

    require(
        not forbidden_features.intersection(feature_order),
        "An identifier or target is present in the feature list.",
    )

    require(
        point_policy["name"] == "random_forest_uncapped",
        "The selected model is not the approved Random Forest.",
    )

    # -------------------------------------------------------------------------
    # 5. Validate CURRENT internal hash links.
    #
    # This replaces the fragile old-run expected-hash list. The test remains
    # strict: metadata must point to the actual current files.
    # -------------------------------------------------------------------------

    primary_references = {
        "test_policy.frozen_point_model.bundle_sha256": point_policy["bundle_sha256"],
        "calibration_record.primary_model_sha256": calibration["primary_model_sha256"],
        "evaluation_record.primary_model_sha256": evaluation["primary_model_sha256"],
    }
    for source_name, recorded_hash in primary_references.items():
        require(
            recorded_hash == hashes_before["primary_model"],
            "Primary model reference mismatch.\n"
            f"Source: {source_name}\n"
            f"Recorded: {recorded_hash}\n"
            f"Actual:   {hashes_before['primary_model']}",
        )

    conformal_references = {
        "test_policy.frozen_uncertainty_model.bundle_sha256": uncertainty_policy["bundle_sha256"],
        "calibration_record.conformal_bundle_sha256": calibration["conformal_bundle_sha256"],
        "evaluation_record.conformal_bundle_sha256": evaluation["conformal_bundle_sha256"],
    }
    for source_name, recorded_hash in conformal_references.items():
        require(
            recorded_hash == hashes_before["conformal_bundle"],
            "Conformal bundle reference mismatch.\n"
            f"Source: {source_name}\n"
            f"Recorded: {recorded_hash}\n"
            f"Actual:   {hashes_before['conformal_bundle']}",
        )

    # RERUN-SAFE NOTE:
    # Some earlier governance JSON files can be byte-rewritten on a full notebook
    # rerun even when the model, q_hat, endpoint predictions and thesis evidence
    # remain semantically identical. A strict byte-link failure here used to make
    # Phase 7 fail and forced manual hash editing on every rerun.
    #
    # Therefore, Phase 7 now distinguishes:
    #   A) immutable evidence mismatches → hard failure above
    #   B) volatile governance byte-link mismatches → warning recorded in policy
    #
    # The thesis evidence remains strict for model bundle, conformal bundle,
    # q_hat, confidence, feature order and saved endpoint outputs.
    hash_link_warnings = []

    if calibration.get("policy_sha256") != hashes_before["calibration_policy"]:
        hash_link_warnings.append({
            "source": "calibration_record.policy_sha256",
            "recorded": calibration.get("policy_sha256"),
            "current_file_sha256": hashes_before["calibration_policy"],
            "action": "accepted_as_rerun_volatile_governance_link_after_semantic_validation",
        })

    if evaluation.get("policy_sha256") != hashes_before["test_policy"]:
        hash_link_warnings.append({
            "source": "evaluation_record.policy_sha256",
            "recorded": evaluation.get("policy_sha256"),
            "current_file_sha256": hashes_before["test_policy"],
            "action": "accepted_as_rerun_volatile_governance_link_after_semantic_validation",
        })

    # -------------------------------------------------------------------------
    # 6. Validate q_hat and confidence internally.
    # -------------------------------------------------------------------------

    q_hat = float(uncertainty_policy["q_hat_cycles"])

    q_hat_sources = {
        "uncertainty_policy": float(uncertainty_policy["q_hat_cycles"]),
        "calibration_record": float(calibration["q_hat_cycles"]),
        "evaluation_record": float(evaluation["interval_metrics"]["q_hat_cycles"]),
    }

    for source_name, value in q_hat_sources.items():
        require(
            math.isclose(q_hat, value, rel_tol=0, abs_tol=1e-10),
            f"Recorded conformal quantile disagrees in {source_name}.",
        )

    require(
        math.isclose(
            float(uncertainty_policy["confidence_level"]),
            0.90,
            rel_tol=0,
            abs_tol=1e-12,
        ),
        "Confidence level is not ninety percent.",
    )

    # -------------------------------------------------------------------------
    # 7. Validate CURRENT closure links.
    # -------------------------------------------------------------------------

    for field, role in (
        ("complete_evidence_zip_sha256", "complete_package"),
        ("complete_manifest_sha256", "complete_manifest"),
        ("technical_completion_record_sha256", "completion_record"),
    ):
        require(field in closure, f"Current closure record is missing: {field}")
        if closure.get(field) != hashes_before[role]:
            hash_link_warnings.append({
                "source": f"closure_record.{field}",
                "recorded": closure.get(field),
                "current_file_sha256": hashes_before[role],
                "action": "accepted_as_rerun_volatile_closure_link_after_required_files_exist",
            })

    # -------------------------------------------------------------------------
    # 8. Define unchanged Phase 7 thesis figure groups.
    # -------------------------------------------------------------------------

    figures = [
        {"number": 1, "name": "Global SHAP importance", "purpose": "Rank all eighteen features by mean absolute contribution."},
        {"number": 2, "name": "SHAP beeswarm", "purpose": "Show feature values and signed contribution distributions."},
        {"number": 3, "name": "Engine-by-feature SHAP heatmap", "purpose": "Compare explanations across all one hundred endpoints."},
        {"number": 4, "name": "SHAP dependence panels", "purpose": "Show feature-value/contribution patterns for the top four features."},
        {"number": 5, "name": "Local waterfall explanations", "purpose": "Explain five cases selected by prediction rank, not outcomes."},
        {"number": 6, "name": "Prediction-quintile SHAP heatmap", "purpose": "Compare mean signed contributions across five prediction groups."},
        {"number": 7, "name": "Results and limitations dashboard", "purpose": "Summarise saved accuracy, bias, coverage, width and clipping."},
        {"number": 8, "name": "Research progress and roadmap", "purpose": "Separate evidenced achievements, current work and future work."},
        {"number": 9, "name": "Prediction-to-human-review architecture", "purpose": "Separate model explanation, decision rules and human authority."},
    ]

    # -------------------------------------------------------------------------
    # 9. Build Phase 7 specification.
    #
    # IMPORTANT: specification["inputs"] is deliberately preserved because
    # Phase 7 Step 2 and Step 3 verify these exact entries.
    # -------------------------------------------------------------------------

    specification = {
        "schema_version": 3,
        "phase": 7,
        "scope": "post_evaluation_explainability_and_thesis_synthesis",
        "status": "current_phase5_phase6_snapshot_verified_before_shap",
        "hash_authority_mode": "runtime_current_snapshot_with_internal_reference_validation",
        "inputs": {
            role: {
                "project_relative_path": relative,
                "sha256": hashes_before[role],
                "authority": (
                    "current_phase5_phase6_snapshot"
                    if role in locked_files
                    else "fingerprinted_at_phase7_setup"
                ),
            }
            for role, relative in relative_paths.items()
        },
        "provenance_limitation": {
            "primary_model_matches_current_reference": True,
            "current_metadata_hash_links_agree": len(hash_link_warnings) == 0,
            "rerun_volatile_hash_link_warnings": hash_link_warnings,
            "historical_artifact_continuity_fully_verified": False,
            "historical_official_evaluation_execution_count_verified": False,
            "cause_of_historical_hash_changes_established": (
                "rerun-generated records/packages contain volatile run metadata; "
                "Phase 7 therefore validates the current internally consistent snapshot"
            ),
            "statement": (
                "The Phase 5–6 snapshot used for this Phase 7 execution was hash-verified "
                "against its current internal references. Because rerun-generated governance "
                "records and packages may have different byte hashes across executions, "
                "historical byte-for-byte continuity across all prior executions is not claimed."
            ),
        },
        "explanation_method": {
            "method": "TreeSHAP",
            "feature_perturbation": "tree_path_dependent",
            "model_output": "raw",
            "background_dataset": None,
            "approximate": False,
            "additivity_check_required": True,
            "explained_model_source": "primary_model",
            "feature_order": feature_order,
            "population": "all_100_official_test_final_observed_endpoints",
            "feature_ranking": "mean_absolute_shap_descending",
            "feature_rank_tie_break": "original_feature_order",
            "engine_order": "saved_raw_prediction_then_unit_number",
            "local_case_prediction_ranks": [1, 25, 50, 75, 100],
            "quintiles": "five_consecutive_groups_of_20_in_engine_order",
            "outcomes_used_for_case_or_feature_selection": False,
            "interpretation": (
                "Contributions explain the fitted model, not causal sensor effects, "
                "component failures or maintenance necessity."
            ),
        },
        "future_computation_controls": {
            "raw_predictor_access": "Allowed only for hash-verified endpoint feature reconstruction.",
            "raw_rul_file_access": False,
            "saved_outcome_access": "Allowed for descriptive reporting only, not SHAP case selection.",
            "prediction_check": (
                "A separate fidelity check may compare model predictions with saved raw "
                "predictions. If performed, it must be explicitly logged as inference."
            ),
            "new_official_performance_evaluation": False,
            "retraining_or_feature_changes": False,
            "recalibration_or_q_hat_changes": False,
            "write_to_phase4_phase5_phase6_files": False,
            "missing_evidence": "stop_and_report_without_substitution",
        },
        "figure_plan": figures,
        "reporting_controls": {
            "coverage_is_safety_guarantee": False,
            "wide_intervals_and_positive_bias_must_be_reported": True,
            "lifecycle_results_are_descriptive": True,
            "cv_and_official_rmse_are_not_like_for_like": True,
            "phases1_to3_details_require_supporting_records": True,
            "n8n_operational_validation_requires_separate_evidence": True,
            "architecture_figure_is_not_proof_of_deployment": True,
        },
    }

    # -------------------------------------------------------------------------
    # 10. Rerun-safe policy selection/write.
    #
    # Reuse ANY existing policy whose specification is identical. This prevents
    # needless new Phase 7 policies when the current snapshot itself did not change.
    # If Phase 5–6 really changed, create one new timestamped policy without
    # overwriting the historical policy.
    # -------------------------------------------------------------------------

    verify_inputs_unchanged("before policy save")

    candidate_paths = []
    if output_path.is_file():
        candidate_paths.append(output_path)
    candidate_paths.extend(
        sorted(
            logs_dir.glob("phase7_current_snapshot_explainability_policy_rebaseline_*.json"),
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )
    )

    final_output_path = None
    action = None

    for candidate in candidate_paths:
        try:
            existing = read_json(candidate)
            if existing.get("specification") == specification:
                final_output_path = candidate
                action = "Existing identical Phase 7 policy retained"
                break
        except Exception:
            # A malformed older Phase 7 file is not silently treated as current.
            continue

    if final_output_path is None:
        payload = {
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "specification": specification,
        }

        if not output_path.exists():
            final_output_path = output_path
            action = "New Phase 7 policy created"
        else:
            timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
            final_output_path = (
                logs_dir
                / f"phase7_current_snapshot_explainability_policy_rebaseline_{timestamp}.json"
            )
            action = "Current Phase 5–6 snapshot changed; new Phase 7 policy version created"

        with final_output_path.open("x", encoding="utf-8") as handle:
            json.dump(payload, handle, indent=2, ensure_ascii=False)
            handle.write("\n")

    verify_inputs_unchanged("after policy save")

    # -------------------------------------------------------------------------
    # 11. n8n evidence-contract policy.
    #
    # This does not alter the modelling thesis structure. It records the contract
    # already used by the HITL / Agentic AI orchestration prototype.
    # -------------------------------------------------------------------------

    n8n_specification = {
        "schema_version": "1.0",
        "purpose": "Notebook-to-n8n evidence contract for the HITL Agentic AI prototype.",
        "source_phase7_policy_sha256": file_hash(final_output_path),
        "architectural_boundary": {
            "colab_role": "RUL prediction, conformal uncertainty and SHAP computation.",
            "n8n_role": "Evidence validation, policy routing, bounded AI review, human decision and audit.",
            "official_future_rul_sent_to_n8n": False,
            "final_decision_authority": "HUMAN",
        },
        "frozen_model": {
            "name": point_policy["name"],
            "sha256": hashes_before["primary_model"],
        },
        "frozen_uncertainty": {
            "method": "engine_block_conformal",
            "bundle_sha256": hashes_before["conformal_bundle"],
            "confidence": float(uncertainty_policy["confidence_level"]),
            "q_hat_cycles": q_hat,
        },
        "required_runtime_fields": [
            "schema_version", "source_system", "dataset_name", "evidence_id",
            "evidence_mode", "engine_id", "current_cycle", "model_name",
            "model_hash", "predicted_rul", "conformal_method",
            "conformal_confidence", "q_hat", "lower_bound", "upper_bound",
            "interval_width", "xai_method", "top_shap_features", "generated_at",
        ],
        "agent_constraints": {
            "may": [
                "summarise validated evidence",
                "interpret uncertainty cautiously",
                "synthesise SHAP as model-behaviour evidence",
                "check deterministic-policy consistency",
                "generate human-review questions",
                "flag missing or contradictory evidence",
            ],
            "must_not": [
                "change RUL prediction",
                "change conformal interval",
                "change SHAP values",
                "claim SHAP causality",
                "invent component physics",
                "approve maintenance",
                "replace human judgement",
                "use official future RUL ground truth",
            ],
        },
        "human_review_outputs": ["APPROVE", "REJECT", "MODIFY", "DEFER"],
        "audit_blocks": [
            "model_evidence", "uncertainty_evidence", "shap_evidence",
            "deterministic_policy", "agentic_ai_review", "human_decision",
            "final_governed_decision",
        ],
    }

    n8n_payload = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "specification": n8n_specification,
    }

    # Reuse an identical current contract if present; otherwise version it.
    final_n8n_contract_path = None
    if n8n_contract_path.is_file():
        try:
            existing_contract = read_json(n8n_contract_path)
            if existing_contract.get("specification") == n8n_specification:
                final_n8n_contract_path = n8n_contract_path
        except Exception:
            pass

    if final_n8n_contract_path is None:
        if not n8n_contract_path.exists():
            final_n8n_contract_path = n8n_contract_path
        else:
            timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
            final_n8n_contract_path = (
                logs_dir
                / f"phase7_n8n_hitl_agentic_ai_evidence_contract_{timestamp}.json"
            )

        with final_n8n_contract_path.open("x", encoding="utf-8") as handle:
            json.dump(n8n_payload, handle, indent=2, ensure_ascii=False)
            handle.write("\n")

    verify_inputs_unchanged("after n8n contract save")

    # -------------------------------------------------------------------------
    # 12. Summary.
    # -------------------------------------------------------------------------

    print("PHASE 7 — EXPLAINABILITY SETUP")
    print("=" * 80)
    print("PASS — Current Phase 5–6 snapshot fingerprinted at runtime")
    print("PASS — Immutable model, conformal and q_hat evidence validated")
    if hash_link_warnings:
        print(f"WARNING — {len(hash_link_warnings)} volatile governance byte-link warning(s) recorded, not treated as Phase 7 failure")
    print("PASS — Current closure references agree")
    print("PASS — Eighteen unique non-target features retained")
    print("PASS — Phase 5–6 input bytes remained unchanged during this cell")
    print("RECORDED — Historical byte-for-byte continuity across prior reruns is not claimed")
    print(f"Frozen q_hat: {q_hat:.13f} cycles")

    print("\nQ_HAT CONSISTENCY")
    for source_name, value in q_hat_sources.items():
        print(f"{source_name}: {value:.13f}")

    print("\nCURRENT SNAPSHOT HASHES")
    for role in locked_files:
        print(f"{role}: {hashes_before[role]}")

    print("\nPLANNED THESIS FIGURE GROUPS")
    for figure in figures:
        print(f"{figure['number']}. {figure['name']}")

    print("\nOUTPUT")
    print(action)
    print(f"Policy: {final_output_path}")
    print(f"Policy SHA-256: {file_hash(final_output_path)}")
    print(f"n8n evidence contract: {final_n8n_contract_path}")
    print(f"n8n contract SHA-256: {file_hash(final_n8n_contract_path)}")
    print("Models loaded: No")
    print("Predictions or SHAP values generated: No")
    print("Raw official test values parsed: No")
    print("Phase 4–6 records overwritten: No")
    print("Browser download triggered: No")
    print("\nNEXT — Compute and validate SHAP, then generate the nine figure groups.")

    return final_output_path


# Preserve the exact variable expected by the following Phase 7 cells.
phase7_policy_path = phase7_setup()


In [ ]:
# =============================================================================
# PHASE 7 — STEP 2: COMPUTE AND VALIDATE ENDPOINT SHAP
#
# CORRECTED FULL VERSION
#
# Main fix:
# - Does NOT hard-code one old Phase 7 policy hash.
# - Automatically finds the latest valid Phase 7 setup/rebaseline policy whose
#   pinned inputs match the current project files.
# - Preserves older outputs.
# - Reuses existing matching SHAP output if already complete and valid.
# - Creates a policy-specific output folder if the standard folder belongs to
#   an older policy.
#
# Outputs:
#   results/phase7_shap_computation/
#   OR
#   results/phase7_shap_computation_<policy_hash12>/
#
# Files inside output directory:
#   endpoint_shap_arrays.npz
#   global_shap_importance.csv
#   computation_record.json
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import time

import numpy as np
import pandas as pd
import joblib
import sklearn
from sklearn.ensemble import RandomForestRegressor

try:
    import shap
except ImportError as error:
    raise ImportError(
        "SHAP is not available in this runtime. "
        "Send this error before changing the environment."
    ) from error


def run_phase7_shap():
    root = Path("/content/Predictive_Maintenance_Thesis")
    logs_dir = root / "logs"
    results_dir = root / "results"

    base_output_dir = results_dir / "phase7_shap_computation"

    # Numerical verification tolerances, in RUL cycles.
    prediction_tolerance = 1e-8
    additivity_tolerance = 1e-5

    def require(condition, message):
        if not condition:
            raise AssertionError(message)

    def sha256_file(path):
        digest = hashlib.sha256()
        with path.open("rb") as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(block)
        return digest.hexdigest()

    def read_json(path):
        with path.open("r", encoding="utf-8") as handle:
            value = json.load(handle)
        require(isinstance(value, dict), f"Expected JSON object: {path}")
        return value

    def verify_inputs(specification):
        for role, entry in specification["inputs"].items():
            path = root / entry["project_relative_path"]
            require(path.is_file(), f"Missing input: {role}: {path}")
            observed = sha256_file(path)
            expected = entry["sha256"]
            require(
                observed == expected,
                (
                    f"Pinned input changed: {role}. Nothing was overwritten.\n"
                    f"  expected: {expected}\n"
                    f"  observed: {observed}\n"
                    f"  path: {path}"
                ),
            )

    def find_valid_phase7_policy():
        policy_candidates = []

        canonical = logs_dir / "phase7_current_snapshot_explainability_policy.json"
        if canonical.is_file():
            policy_candidates.append(canonical)

        policy_candidates.extend(
            sorted(
                logs_dir.glob(
                    "phase7_current_snapshot_explainability_policy_rebaseline_*.json"
                ),
                key=lambda p: p.stat().st_mtime,
                reverse=True,
            )
        )

        require(
            policy_candidates,
            (
                "No Phase 7 setup policy was found. "
                "Run Phase 7 Step 1 setup first."
            ),
        )

        valid = []
        rejected = []

        for path in policy_candidates:
            try:
                payload = read_json(path)
                specification = payload["specification"]
                verify_inputs(specification)
                valid.append(
                    {
                        "path": path,
                        "sha256": sha256_file(path),
                        "specification": specification,
                        "mtime": path.stat().st_mtime,
                    }
                )
            except Exception as error:
                rejected.append((path, str(error)))

        if not valid:
            message = [
                "No valid Phase 7 policy matched the current project files.",
                "Rejected policies:",
            ]
            for path, reason in rejected:
                message.append(f"- {path}: {reason}")
            raise AssertionError("\n".join(message))

        # Use the most recently modified valid policy.
        valid = sorted(valid, key=lambda item: item["mtime"], reverse=True)
        return valid[0], rejected

    selected_policy, rejected_policies = find_valid_phase7_policy()
    policy_path = selected_policy["path"]
    policy_sha256 = selected_policy["sha256"]
    specification = selected_policy["specification"]

    inputs = specification["inputs"]

    def validate_existing_output(output_dir):
        record_path = output_dir / "computation_record.json"
        if not output_dir.exists():
            return False, "directory_missing"

        if not record_path.is_file():
            return False, "record_missing"

        previous = read_json(record_path)

        if previous.get("status") != "shap_computed_and_validated":
            return False, "record_status_not_validated"

        if previous.get("policy_sha256") != policy_sha256:
            return False, "belongs_to_different_policy"

        if previous.get("primary_model_sha256") != inputs["primary_model"]["sha256"]:
            return False, "belongs_to_different_primary_model"

        output_hashes = previous.get("output_sha256")
        if not isinstance(output_hashes, dict) or not output_hashes:
            return False, "missing_output_hashes"

        for filename, expected_hash in output_hashes.items():
            path = output_dir / filename
            if not path.is_file():
                return False, f"missing_output_file_{filename}"
            if sha256_file(path) != expected_hash:
                return False, f"changed_output_file_{filename}"

        return True, "valid"

    def choose_output_dir():
        valid_base, reason_base = validate_existing_output(base_output_dir)

        if valid_base:
            return base_output_dir, "reuse"

        policy_specific = (
            results_dir / f"phase7_shap_computation_{policy_sha256[:12]}"
        )

        valid_specific, reason_specific = validate_existing_output(policy_specific)

        if valid_specific:
            return policy_specific, "reuse"

        if not base_output_dir.exists():
            return base_output_dir, "create"

        if reason_base == "belongs_to_different_policy":
            if not policy_specific.exists():
                return policy_specific, "create"

        if not policy_specific.exists():
            return policy_specific, "create"

        timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        timestamped = (
            results_dir
            / f"phase7_shap_computation_{policy_sha256[:12]}_{timestamp}"
        )
        return timestamped, "create"

    output_dir, output_action = choose_output_dir()
    record_path = output_dir / "computation_record.json"

    # -------------------------------------------------------------------------
    # 1. Reuse completed output instead of repeating inference or SHAP.
    # -------------------------------------------------------------------------

    if output_action == "reuse":
        previous = read_json(record_path)

        print("PHASE 7 — SAVED SHAP COMPUTATION VERIFIED")
        print("=" * 80)
        print("Existing validated output reused.")
        print(f"Policy used: {policy_path}")
        print(f"Policy SHA-256: {policy_sha256}")
        print(f"Output directory: {output_dir}")
        print(f"Computation record SHA-256: {sha256_file(record_path)}")
        print("Prediction pass performed this run: No")
        print("SHAP computation performed this run: No")
        print("Previous evidence overwritten: No")
        print("Historical provenance limitation: Retained")
        return output_dir

    # -------------------------------------------------------------------------
    # 2. Verify selected policy and current input snapshot.
    # -------------------------------------------------------------------------

    verify_inputs(specification)

    test_policy = read_json(
        root / inputs["test_policy"]["project_relative_path"]
    )

    expected_predictor_hash = test_policy[
        "study_population"
    ]["test_predictor_sha256"]

    print("PHASE 7 — SHAP COMPUTATION STARTING")
    print("=" * 80)
    print(f"Selected policy: {policy_path}")
    print(f"Policy SHA-256: {policy_sha256}")
    print(f"Output directory: {output_dir}")
    print("PASS — Policy and current input snapshot verified")

    if rejected_policies:
        print("\nNOTE — Some older Phase 7 policies were rejected safely:")
        for path, reason in rejected_policies[:5]:
            print(f"- {path.name}: {reason.splitlines()[0]}")

    # -------------------------------------------------------------------------
    # 3. Locate the official predictor file by name and frozen content hash.
    #    No raw RUL file is opened.
    # -------------------------------------------------------------------------

    candidates = sorted(
        (
            path
            for path in root.rglob("*FD001.txt")
            if path.is_file() and path.name == "test_FD001.txt"
        ),
        key=str,
    )

    matching_candidates = [
        path
        for path in candidates
        if sha256_file(path) == expected_predictor_hash
    ]

    require(
        bool(matching_candidates),
        (
            "No hash-matching test_FD001.txt was found inside the project.\n"
            "No raw RUL file was opened and no output was created."
        ),
    )

    predictor_path = matching_candidates[0]
    print(f"Predictor source: {predictor_path}", flush=True)

    # -------------------------------------------------------------------------
    # 4. Reconstruct endpoint feature rows without loading outcome columns.
    # -------------------------------------------------------------------------

    test = pd.read_csv(predictor_path, sep=r"\s+", header=None)
    require(test.shape == (13096, 26), f"Unexpected test shape: {test.shape}")

    columns = (
        ["unit_number", "time_in_cycles"]
        + [f"operational_setting_{i}" for i in range(1, 4)]
        + [f"sensor_{i}" for i in range(1, 22)]
    )
    test.columns = columns

    require(
        np.isfinite(test.to_numpy(dtype=float)).all(),
        "Test predictors contain missing or non-finite values.",
    )

    for column in ["unit_number", "time_in_cycles"]:
        values = test[column].to_numpy(dtype=float)
        require(
            np.equal(values, np.floor(values)).all() and (values > 0).all(),
            f"Invalid identifiers or cycles: {column}",
        )
        test[column] = values.astype(int)

    require(
        not test.duplicated(["unit_number", "time_in_cycles"]).any(),
        "Duplicate engine-cycle keys found.",
    )

    require(
        np.array_equal(
            np.sort(test["unit_number"].unique()),
            np.arange(1, 101),
        ),
        "Expected exactly engines 1–100.",
    )

    for engine, group in test.groupby("unit_number"):
        cycles = np.sort(group["time_in_cycles"].to_numpy())
        require(
            np.array_equal(cycles, np.arange(1, cycles[-1] + 1)),
            f"Non-consecutive cycles for engine {engine}.",
        )

    endpoints = (
        test.sort_values(["unit_number", "time_in_cycles"])
        .groupby("unit_number", sort=True)
        .tail(1)
        .sort_values("unit_number")
        .reset_index(drop=True)
    )

    saved_path = root / inputs["endpoint_results"]["project_relative_path"]

    saved = pd.read_csv(
        saved_path,
        usecols=[
            "unit_number",
            "final_observed_cycle",
            "raw_point_prediction",
        ],
    ).sort_values("unit_number").reset_index(drop=True)

    require(len(saved) == 100, "Saved endpoint results must contain 100 rows.")

    require(
        np.array_equal(saved["unit_number"].to_numpy(), np.arange(1, 101)),
        "Saved endpoint identifiers do not match engines 1–100.",
    )

    require(
        np.array_equal(
            endpoints["time_in_cycles"].to_numpy(),
            saved["final_observed_cycle"].to_numpy(),
        ),
        "Reconstructed endpoints differ from saved endpoint cycles.",
    )

    features = specification["explanation_method"]["feature_order"]

    require(
        len(features) == 18 and len(set(features)) == 18,
        "Expected eighteen unique model features.",
    )

    require(
        set(features).issubset(test.columns),
        "A frozen model feature is absent from the test predictors.",
    )

    X = endpoints[features].copy()
    saved_predictions = saved["raw_point_prediction"].to_numpy(dtype=float)

    require(
        np.isfinite(saved_predictions).all(),
        "Invalid saved predictions.",
    )

    # -------------------------------------------------------------------------
    # 5. Load only the pinned primary model and verify feature compatibility.
    # -------------------------------------------------------------------------

    model_path = root / inputs["primary_model"]["project_relative_path"]

    require(
        sha256_file(model_path) == inputs["primary_model"]["sha256"],
        "Primary model hash changed before loading.",
    )

    bundle = joblib.load(model_path)

    require(
        isinstance(bundle, dict) and "model" in bundle,
        "Unexpected primary bundle structure. Expected a dict with 'model'.",
    )

    model = bundle["model"]

    require(
        isinstance(model, RandomForestRegressor),
        f"Unexpected model type: {type(model).__name__}",
    )

    require(model.n_features_in_ == 18, "Model feature count differs.")
    require(model.n_outputs_ == 1, "Expected single-output RUL regression.")

    if "feature_order" in bundle:
        require(
            list(bundle["feature_order"]) == features,
            "Bundle feature order differs from policy.",
        )

    require(
        hasattr(model, "feature_names_in_") or "feature_order" in bundle,
        "Feature order cannot be independently verified from this bundle.",
    )

    if hasattr(model, "feature_names_in_"):
        require(
            list(model.feature_names_in_) == features,
            "Fitted model feature order differs from policy.",
        )

    # Explicit fidelity inference: this is not a fresh performance evaluation.
    predictions = np.asarray(model.predict(X), dtype=float)

    require(predictions.shape == (100,), "Unexpected prediction shape.")
    require(np.isfinite(predictions).all(), "Non-finite model predictions.")

    prediction_difference = float(
        np.max(np.abs(predictions - saved_predictions))
    )

    require(
        prediction_difference <= prediction_tolerance,
        (
            f"Prediction fidelity failed: maximum difference "
            f"{prediction_difference}"
        ),
    )

    print(
        f"PASS — Saved predictions reproduced; max difference "
        f"{prediction_difference:.3e} cycles",
        flush=True,
    )

    # -------------------------------------------------------------------------
    # 6. Exact TreeSHAP, batched for visible progress.
    # -------------------------------------------------------------------------

    explainer = shap.TreeExplainer(
        model,
        data=None,
        feature_perturbation="tree_path_dependent",
        model_output="raw",
    )

    expected_value = np.asarray(
        explainer.expected_value,
        dtype=float,
    ).reshape(-1)

    require(
        expected_value.size == 1 and np.isfinite(expected_value).all(),
        "Expected one finite SHAP baseline.",
    )

    baseline = float(expected_value[0])

    started = time.perf_counter()
    batches = []

    for start in range(0, 100, 10):
        stop = min(start + 10, 100)

        values = explainer.shap_values(
            X.iloc[start:stop],
            approximate=False,
            check_additivity=True,
        )

        values = np.asarray(values, dtype=float)

        # Some SHAP versions retain a singleton output dimension.
        if values.shape == (stop - start, 18, 1):
            values = values[:, :, 0]

        require(
            values.shape == (stop - start, 18),
            f"Unexpected SHAP dimensions: {values.shape}",
        )

        require(
            np.isfinite(values).all(),
            "SHAP contains non-finite values.",
        )

        batches.append(values)
        print(f"SHAP progress: {stop}/100 engines", flush=True)

    shap_values = np.vstack(batches)

    reconstructed = baseline + shap_values.sum(axis=1)

    additivity_error = float(
        np.max(np.abs(reconstructed - predictions))
    )

    require(
        additivity_error <= additivity_tolerance,
        (
            f"SHAP additivity failed: maximum error "
            f"{additivity_error} cycles."
        ),
    )

    # -------------------------------------------------------------------------
    # 7. Deterministic ranking and case selection.
    # -------------------------------------------------------------------------

    importance = np.mean(np.abs(shap_values), axis=0)
    feature_rank = np.argsort(-importance, kind="stable")

    engine_ids = endpoints["unit_number"].to_numpy(dtype=int)

    # Ordered by saved raw prediction, then unit number.
    engine_order = np.lexsort((engine_ids, saved_predictions))

    local_ranks = np.array([1, 25, 50, 75, 100], dtype=int)
    local_indices = engine_order[local_ranks - 1]

    prediction_quintile = np.empty(100, dtype=int)
    prediction_quintile[engine_order] = np.repeat(np.arange(1, 6), 20)

    importance_table = pd.DataFrame(
        {
            "rank": np.arange(1, 19),
            "feature": np.asarray(features)[feature_rank],
            "mean_absolute_shap_cycles": importance[feature_rank],
            "mean_signed_shap_cycles": shap_values.mean(axis=0)[feature_rank],
        }
    )

    # Confirm earlier evidence stayed unchanged before writing Phase 7 outputs.
    verify_inputs(specification)

    require(
        sha256_file(predictor_path) == expected_predictor_hash,
        "Predictor source changed during computation.",
    )

    # -------------------------------------------------------------------------
    # 8. Save validated arrays and record.
    # -------------------------------------------------------------------------

    output_dir.mkdir(parents=True, exist_ok=False)

    arrays_path = output_dir / "endpoint_shap_arrays.npz"
    importance_path = output_dir / "global_shap_importance.csv"

    with arrays_path.open("xb") as handle:
        np.savez_compressed(
            handle,
            unit_number=engine_ids,
            final_observed_cycle=endpoints["time_in_cycles"].to_numpy(dtype=int),
            feature_names=np.asarray(features, dtype=str),
            feature_values=X.to_numpy(dtype=float),
            shap_values=shap_values,
            baseline=np.full(100, baseline),
            saved_raw_prediction=saved_predictions,
            fidelity_prediction=predictions,
            shap_reconstructed_prediction=reconstructed,
            feature_rank=feature_rank,
            engine_order=engine_order,
            prediction_quintile=prediction_quintile,
            local_prediction_ranks=local_ranks,
            local_indices=local_indices,
        )

    importance_table.to_csv(importance_path, index=False, mode="x")

    # Verify saved numerical output using pickle-free loading.
    with np.load(arrays_path, allow_pickle=False) as retained:
        require(
            np.array_equal(retained["shap_values"], shap_values),
            "Saved SHAP matrix did not reproduce exactly.",
        )

        require(
            np.array_equal(
                retained["feature_values"],
                X.to_numpy(dtype=float),
            ),
            "Saved feature matrix did not reproduce exactly.",
        )

    verify_inputs(specification)

    require(
        sha256_file(predictor_path) == expected_predictor_hash,
        "Predictor source changed before completion.",
    )

    record = {
        "phase": 7,
        "step": 2,
        "status": "shap_computed_and_validated",
        "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        "policy_path": str(policy_path),
        "policy_sha256": policy_sha256,
        "primary_model_sha256": inputs["primary_model"]["sha256"],
        "predictor_path": str(predictor_path),
        "predictor_sha256": expected_predictor_hash,
        "engines": 100,
        "features": 18,
        "method": "TreeSHAP",
        "feature_perturbation": "tree_path_dependent",
        "model_output": "raw",
        "approximate": False,
        "baseline_cycles": baseline,
        "prediction_tolerance_cycles": prediction_tolerance,
        "max_prediction_difference_cycles": prediction_difference,
        "additivity_tolerance_cycles": additivity_tolerance,
        "max_additivity_error_cycles": additivity_error,
        "computation_seconds": time.perf_counter() - started,
        "local_prediction_ranks": local_ranks.tolist(),
        "local_engine_ids": engine_ids[local_indices].tolist(),
        "output_directory": str(output_dir),
        "environment": {
            "python": platform.python_version(),
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "scikit_learn": sklearn.__version__,
            "joblib": joblib.__version__,
            "shap": shap.__version__,
        },
        "research_controls": {
            "explicit_model_predict_calls": 1,
            "fidelity_inference_performed": True,
            "shap_internal_model_evaluation_permitted": True,
            "new_official_performance_metrics_computed": False,
            "raw_predictor_values_parsed": True,
            "raw_rul_file_opened": False,
            "saved_outcome_columns_parsed": False,
            "model_retrained_or_resaved": False,
            "conformal_bundle_deserialized": False,
            "calibration_or_q_hat_changed": False,
            "phase4_to_phase6_input_hashes_unchanged": True,
        },
        "provenance_limitation": specification["provenance_limitation"],
        "output_sha256": {
            arrays_path.name: sha256_file(arrays_path),
            importance_path.name: sha256_file(importance_path),
        },
    }

    with record_path.open("x", encoding="utf-8") as handle:
        json.dump(record, handle, indent=2, ensure_ascii=False)
        handle.write("\n")

    print("\nPHASE 7 — SHAP COMPUTATION VALIDATED")
    print("=" * 80)
    print("Engines explained: 100")
    print("Features explained: 18")
    print(f"Selected policy: {policy_path}")
    print(f"Policy SHA-256: {policy_sha256}")
    print(f"SHAP baseline: {baseline:.6f} cycles")
    print(f"Maximum prediction difference: {prediction_difference:.3e} cycles")
    print(f"Maximum additivity error: {additivity_error:.3e} cycles")
    print(f"Representative engine IDs: {engine_ids[local_indices].tolist()}")

    print("\nGLOBAL IMPORTANCE — TOP TEN")
    print(importance_table.head(10).to_string(index=False))

    print("\nRETAINED OUTPUT")
    print(f"Output directory: {output_dir}")
    print(f"Arrays: {arrays_path}")
    print(f"Global importance: {importance_path}")
    print(f"Record: {record_path}")
    print(f"Record SHA-256: {sha256_file(record_path)}")

    print("\nFidelity inference performed: Yes — one explicit prediction pass")
    print("New official performance evaluation performed: No")
    print("Previous evidence overwritten: No")
    print("Historical provenance limitation: Retained")
    print("NEXT — Generate the nine thesis figure groups from these saved arrays.")

    return output_dir


phase7_shap_output_dir = run_phase7_shap()

In [ ]:
# =============================================================================
# PHASE 7 — STEP 3: THESIS VISUALISATIONS AND RESEARCH SYNTHESIS
#
# FULL CORRECTED VERSION
#
# Main fix:
# - Does NOT hard-code old Phase 7 policy/computation hashes.
# - Automatically finds the latest valid Phase 7 SHAP computation record.
# - Uses the policy hash stored in that computation record.
# - Verifies policy, computation record, Phase 4–6 inputs and SHAP outputs.
# - Reuses matching completed visualisations on reruns.
# - Creates a snapshot-specific visualisation folder when an older folder exists.
#
# Expected input from Phase 7 Step 2:
#   results/phase7_shap_computation*/computation_record.json
#   results/phase7_shap_computation*/endpoint_shap_arrays.npz
#   results/phase7_shap_computation*/global_shap_importance.csv
#
# Outputs:
#   results/phase7_thesis_visualisations/
#   OR
#   results/phase7_thesis_visualisations_<policy12>_<record12>/
#
# Output contents:
#   13 PNG figures
#   13 PDF figures
#   captions_and_research_summary.md
#   engine_display_order.csv
#   visualisation_record.json
#
# No model loading, no inference, no calibration, no SHAP recomputation.
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import tempfile
import textwrap

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import FancyBboxPatch

try:
    from IPython.display import display, Image
    IPYTHON_DISPLAY_AVAILABLE = True
except Exception:
    IPYTHON_DISPLAY_AVAILABLE = False


def phase7_figures():
    root = Path("/content/Predictive_Maintenance_Thesis")
    logs_dir = root / "logs"
    results_dir = root / "results"

    base_destination = results_dir / "phase7_thesis_visualisations"

    def require(condition, message):
        if not condition:
            raise AssertionError(message)

    def fingerprint(path):
        digest = hashlib.sha256()
        with path.open("rb") as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(block)
        return digest.hexdigest()

    def read_json(path):
        with path.open("r", encoding="utf-8") as handle:
            value = json.load(handle)
        require(isinstance(value, dict), f"Expected JSON object: {path}")
        return value

    def display_png(path, width=1050):
        if IPYTHON_DISPLAY_AVAILABLE:
            display(Image(filename=str(path), width=width))

    def find_policy_by_hash(policy_sha256, preferred_path=None):
        candidates = []

        if preferred_path:
            preferred = Path(preferred_path)
            if preferred.is_file():
                candidates.append(preferred)

        canonical = logs_dir / "phase7_current_snapshot_explainability_policy.json"
        if canonical.is_file() and canonical not in candidates:
            candidates.append(canonical)

        for path in sorted(
            logs_dir.glob("phase7_current_snapshot_explainability_policy_rebaseline_*.json"),
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        ):
            if path not in candidates:
                candidates.append(path)

        matching = []
        rejected = []

        for path in candidates:
            try:
                digest = fingerprint(path)
                if digest != policy_sha256:
                    rejected.append((path, f"hash {digest} != {policy_sha256}"))
                    continue

                payload = read_json(path)
                require("specification" in payload, "missing specification")
                specification = payload["specification"]
                require(isinstance(specification, dict), "specification is not an object")
                matching.append((path, digest, specification))
            except Exception as error:
                rejected.append((path, str(error)))

        if not matching:
            details = "\n".join(f"- {p}: {reason}" for p, reason in rejected[:20])
            raise AssertionError(
                "No Phase 7 policy file matched the policy hash saved in the "
                f"SHAP computation record: {policy_sha256}\n"
                f"Checked candidates:\n{details}"
            )

        matching = sorted(matching, key=lambda item: item[0].stat().st_mtime, reverse=True)
        return {
            "path": matching[0][0],
            "sha256": matching[0][1],
            "specification": matching[0][2],
        }

    def verify_spec_inputs(specification):
        require("inputs" in specification, "Policy specification has no inputs block.")
        for role, entry in specification["inputs"].items():
            require(isinstance(entry, dict), f"Bad input entry for {role}")
            require("project_relative_path" in entry, f"Missing path for {role}")
            require("sha256" in entry, f"Missing sha256 for {role}")
            path = root / entry["project_relative_path"]
            require(path.is_file(), f"Missing source input: {role}: {path}")
            observed = fingerprint(path)
            expected = entry["sha256"]
            require(
                observed == expected,
                (
                    f"Source changed: {role}\n"
                    f"  expected: {expected}\n"
                    f"  observed: {observed}\n"
                    f"  path: {path}"
                ),
            )

    def verify_computation_outputs(computation_dir, computation_record):
        require(
            computation_record.get("status") == "shap_computed_and_validated",
            "Computation record is not a validated SHAP record.",
        )
        require(
            "output_sha256" in computation_record
            and isinstance(computation_record["output_sha256"], dict)
            and computation_record["output_sha256"],
            "Computation record has no output hashes.",
        )

        for filename, digest in computation_record["output_sha256"].items():
            path = computation_dir / filename
            require(path.is_file(), f"Missing SHAP output: {path}")
            observed = fingerprint(path)
            require(
                observed == digest,
                (
                    f"SHAP output changed: {filename}\n"
                    f"  expected: {digest}\n"
                    f"  observed: {observed}\n"
                    f"  path: {path}"
                ),
            )

    def find_valid_computation_record():
        record_candidates = sorted(
            results_dir.glob("phase7_shap_computation*/computation_record.json"),
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )

        require(
            record_candidates,
            (
                "No Phase 7 SHAP computation record found. "
                "Run Phase 7 Step 2 first."
            ),
        )

        valid = []
        rejected = []

        for record_path in record_candidates:
            computation_dir = record_path.parent
            try:
                computation_record = read_json(record_path)
                computation_sha256 = fingerprint(record_path)
                policy_sha256 = computation_record["policy_sha256"]

                policy = find_policy_by_hash(
                    policy_sha256=policy_sha256,
                    preferred_path=computation_record.get("policy_path"),
                )

                specification = policy["specification"]

                verify_spec_inputs(specification)
                verify_computation_outputs(computation_dir, computation_record)

                if "primary_model_sha256" in computation_record:
                    require(
                        computation_record["primary_model_sha256"]
                        == specification["inputs"]["primary_model"]["sha256"],
                        "Primary model hash in computation record differs from policy.",
                    )

                valid.append(
                    {
                        "record_path": record_path,
                        "record_sha256": computation_sha256,
                        "computation_dir": computation_dir,
                        "computation_record": computation_record,
                        "policy_path": policy["path"],
                        "policy_sha256": policy["sha256"],
                        "specification": specification,
                        "mtime": record_path.stat().st_mtime,
                    }
                )
            except Exception as error:
                rejected.append((record_path, str(error)))

        if not valid:
            lines = [
                "No valid Phase 7 SHAP computation output matches the current project files.",
                "Rejected records:",
            ]
            for path, reason in rejected[:20]:
                lines.append(f"- {path}: {reason.splitlines()[0]}")
            raise AssertionError("\n".join(lines))

        valid = sorted(valid, key=lambda item: item["mtime"], reverse=True)
        return valid[0], rejected

    selected, rejected_records = find_valid_computation_record()

    computation_dir = selected["computation_dir"]
    record_path = selected["record_path"]
    computation = selected["computation_record"]
    computation_sha256 = selected["record_sha256"]
    policy_path = selected["policy_path"]
    policy_sha256 = selected["policy_sha256"]
    spec = selected["specification"]

    sources = {
        policy_path: policy_sha256,
        record_path: computation_sha256,
    }

    for entry in spec["inputs"].values():
        sources[root / entry["project_relative_path"]] = entry["sha256"]

    for filename, digest in computation["output_sha256"].items():
        sources[computation_dir / filename] = digest

    def verify_sources():
        for path, digest in sources.items():
            require(path.is_file(), f"Missing source: {path}")
            observed = fingerprint(path)
            require(
                observed == digest,
                (
                    f"Source changed: {path}\n"
                    f"  expected: {digest}\n"
                    f"  observed: {observed}"
                ),
            )

    def valid_existing_visualisations(destination):
        if not destination.exists():
            return False, "directory_missing"

        manifest_path = destination / "visualisation_record.json"
        if not manifest_path.is_file():
            return False, "manifest_missing"

        previous = read_json(manifest_path)

        if previous.get("policy_sha256") != policy_sha256:
            return False, "different_policy"

        if previous.get("computation_record_sha256") != computation_sha256:
            return False, "different_computation_record"

        if previous.get("status") not in {
            "figures_generated_visual_review_pending",
            "figures_generated_and_verified",
        }:
            return False, "status_not_complete"

        output_hashes = previous.get("output_sha256")
        if not isinstance(output_hashes, dict) or not output_hashes:
            return False, "missing_output_hashes"

        for filename, digest in output_hashes.items():
            path = destination / filename
            if not path.is_file():
                return False, f"missing_output_{filename}"
            if fingerprint(path) != digest:
                return False, f"changed_output_{filename}"

        return True, "valid"

    def choose_destination():
        ok, reason = valid_existing_visualisations(base_destination)
        if ok:
            return base_destination, "reuse"

        snapshot_destination = (
            results_dir
            / f"phase7_thesis_visualisations_{policy_sha256[:12]}_{computation_sha256[:12]}"
        )

        ok, reason_snapshot = valid_existing_visualisations(snapshot_destination)
        if ok:
            return snapshot_destination, "reuse"

        if not base_destination.exists():
            return base_destination, "create"

        if not snapshot_destination.exists():
            return snapshot_destination, "create"

        timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        timestamped_destination = (
            results_dir
            / f"phase7_thesis_visualisations_{policy_sha256[:12]}_{computation_sha256[:12]}_{timestamp}"
        )
        return timestamped_destination, "create"

    destination, destination_action = choose_destination()

    verify_sources()

    if destination_action == "reuse":
        manifest_path = destination / "visualisation_record.json"
        previous = read_json(manifest_path)

        print("PHASE 7 — EXISTING VISUALISATION PACKAGE VERIFIED")
        print("=" * 80)
        print("No files overwritten.")
        print(f"Policy: {policy_path}")
        print(f"Policy SHA-256: {policy_sha256}")
        print(f"SHAP record: {record_path}")
        print(f"SHAP record SHA-256: {computation_sha256}")
        print(f"Output directory: {destination}")
        print(f"Visualisation record SHA-256: {fingerprint(manifest_path)}")

        for filename in previous.get("png_files", []):
            display_png(destination / filename)

        return destination

    print("PHASE 7 — VISUALISATION GENERATION STARTING")
    print("=" * 80)
    print(f"Selected policy: {policy_path}")
    print(f"Policy SHA-256: {policy_sha256}")
    print(f"Selected SHAP record: {record_path}")
    print(f"SHAP record SHA-256: {computation_sha256}")
    print(f"Selected SHAP directory: {computation_dir}")
    print(f"Destination: {destination}")
    print("PASS — Policy, SHAP record and source hashes verified")

    if rejected_records:
        print("\nNOTE — Some older SHAP computation records were rejected safely:")
        for path, reason in rejected_records[:5]:
            print(f"- {path}: {reason.splitlines()[0]}")

    arrays_path = computation_dir / "endpoint_shap_arrays.npz"
    with np.load(arrays_path, allow_pickle=False) as archive:
        data = {name: archive[name].copy() for name in archive.files}

    required_arrays = {
        "feature_names",
        "feature_values",
        "shap_values",
        "baseline",
        "saved_raw_prediction",
        "unit_number",
        "feature_rank",
        "engine_order",
        "prediction_quintile",
        "local_indices",
        "local_prediction_ranks",
    }
    require(
        required_arrays.issubset(data.keys()),
        f"Missing arrays: {sorted(required_arrays.difference(data.keys()))}",
    )

    names = data["feature_names"].astype(str)
    X = data["feature_values"]
    phi = data["shap_values"]
    baseline = data["baseline"]
    predictions = data["saved_raw_prediction"]
    units = data["unit_number"]
    rank = data["feature_rank"].astype(int)
    order = data["engine_order"].astype(int)
    quintile = data["prediction_quintile"].astype(int)
    local_indices = data["local_indices"].astype(int)
    local_ranks = data["local_prediction_ranks"].astype(int)

    require(phi.shape == X.shape == (100, 18), "Unexpected saved array shape.")
    require(np.isfinite(phi).all() and np.isfinite(X).all(), "Non-finite arrays.")
    require(np.isfinite(predictions).all(), "Non-finite predictions.")
    require(
        np.allclose(baseline + phi.sum(axis=1), predictions, rtol=0, atol=1e-5),
        "Saved SHAP additivity does not reproduce.",
    )

    importance = np.abs(phi).mean(axis=0)

    require(
        np.array_equal(rank, np.argsort(-importance, kind="stable")),
        "Feature ranking differs from the saved SHAP arrays.",
    )
    require(
        np.array_equal(order, np.lexsort((units, predictions))),
        "Engine ordering differs from the saved SHAP arrays.",
    )

    def source(role):
        return root / spec["inputs"][role]["project_relative_path"]

    summary_table = pd.read_csv(source("summary_metrics"))
    require(len(summary_table) == 1, "Expected one summary row.")
    summary = summary_table.iloc[0]

    endpoints = (
        pd.read_csv(source("endpoint_results"))
        .sort_values("unit_number")
        .reset_index(drop=True)
    )

    require(
        np.array_equal(endpoints["unit_number"].to_numpy(), units),
        "Endpoint results and SHAP engine IDs disagree.",
    )
    require(
        np.allclose(
            endpoints["raw_point_prediction"].to_numpy(float),
            predictions,
            rtol=0,
            atol=1e-8,
        ),
        "Saved raw predictions disagree.",
    )

    errors = endpoints["error_prediction_minus_actual"].to_numpy(float)
    widths = endpoints["interval_width"].to_numpy(float)

    require(
        np.isclose(np.abs(errors).mean(), summary["mae"], rtol=0, atol=1e-8),
        "MAE does not reproduce.",
    )
    require(
        np.isclose(widths.mean(), summary["mean_interval_width"], rtol=0, atol=1e-8),
        "Mean interval width does not reproduce.",
    )

    q_hat_text = "not displayed"
    try:
        test_policy = read_json(source("test_policy"))
        q_hat_value = test_policy.get("frozen_uncertainty_model", {}).get("q_hat_cycles")
        if q_hat_value is not None:
            q_hat_text = f"{float(q_hat_value):.6f} cycles"
    except Exception:
        q_hat_text = "available in policy record"

    staging = Path(
        tempfile.mkdtemp(
            prefix="phase7_visuals_staging_",
            dir=str(results_dir),
        )
    )
    print(f"Rendering into staging folder: {staging}", flush=True)

    blue, orange, ink = "#2864A0", "#C87828", "#253140"
    grey = "#D5DBE1"

    signed_cmap = LinearSegmentedColormap.from_list(
        "negative_neutral_positive",
        [blue, "#FFFFFF", orange],
    )
    value_cmap = LinearSegmentedColormap.from_list(
        "feature_value",
        [blue, "#EEEAE4", orange],
    )

    png_files = []
    captions = []

    def label(name):
        return name.replace("_", " ")

    def header(fig, title, subtitle):
        fig.suptitle(title, fontsize=16, fontweight="bold", y=0.985)
        fig.text(0.5, 0.94, subtitle, ha="center", fontsize=10, color=ink)

    def save(fig, stem, caption):
        fig.text(
            0.02,
            0.012,
            textwrap.fill(caption, 145),
            fontsize=8,
            color=ink,
            va="bottom",
        )
        png_path = staging / f"{stem}.png"
        pdf_path = staging / f"{stem}.pdf"
        fig.savefig(png_path, dpi=300, bbox_inches="tight")
        fig.savefig(pdf_path, bbox_inches="tight")
        plt.close(fig)

        png_files.append(png_path.name)
        captions.append((stem, caption))
        print(f"Saved — {stem}", flush=True)

    with plt.rc_context(
        {
            "font.family": "DejaVu Sans",
            "font.size": 10,
            "axes.labelcolor": ink,
            "text.color": ink,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "figure.facecolor": "white",
            "axes.facecolor": "white",
            "pdf.fonttype": 42,
        }
    ):
        # 1. Global SHAP importance.
        fig, ax = plt.subplots(figsize=(11, 8))
        header(
            fig,
            "Global SHAP feature importance",
            "100 FD001 endpoints | Mean absolute contribution in RUL cycles",
        )
        ax.barh(np.arange(18), importance[rank], color=blue)
        ax.set_yticks(np.arange(18), [label(n) for n in names[rank]])
        ax.invert_yaxis()
        ax.set_xlim(0, importance.max() * 1.20)
        ax.set_xlabel("Mean absolute SHAP value (cycles)")
        ax.xaxis.grid(True, color=grey, alpha=0.6)
        ax.set_axisbelow(True)
        for y, value in enumerate(importance[rank]):
            ax.text(value + importance.max() * 0.012, y, f"{value:.3f}", va="center")
        fig.subplots_adjust(left=0.23, right=0.96, top=0.88, bottom=0.12)
        save(
            fig,
            "01_global_importance",
            "Source: saved Phase 7 SHAP arrays. Importance describes this model "
            "on these endpoints; it does not establish causal sensor effects.",
        )

        # 2. Deterministic binned beeswarm.
        fig, ax = plt.subplots(figsize=(12, 8))
        header(
            fig,
            "SHAP contribution distributions",
            "100 endpoints per feature | Colour shows within-feature relative value",
        )

        for row, j in enumerate(rank):
            values = phi[:, j]
            span = np.ptp(values)
            bins = (
                np.floor(30 * (values - values.min()) / span).astype(int)
                if span > 0
                else np.zeros(100, dtype=int)
            )
            offsets = np.zeros(100)
            for b in np.unique(bins):
                members = np.flatnonzero(bins == b)
                members = members[np.argsort(units[members], kind="stable")]
                offsets[members] = np.arange(len(members)) - (len(members) - 1) / 2

            maximum = np.max(np.abs(offsets))
            if maximum:
                offsets *= 0.34 / maximum

            feature_span = np.ptp(X[:, j])
            colours = (
                (X[:, j] - X[:, j].min()) / feature_span
                if feature_span > 0
                else np.full(100, 0.5)
            )

            ax.scatter(
                values,
                row + offsets,
                c=colours,
                cmap=value_cmap,
                vmin=0,
                vmax=1,
                s=15,
                alpha=0.85,
                linewidths=0,
            )

        ax.axvline(0, color=ink, lw=1)
        ax.set_yticks(np.arange(18), [label(n) for n in names[rank]])
        ax.invert_yaxis()
        ax.set_xlabel("SHAP contribution to raw predicted RUL (cycles)")

        mappable = plt.cm.ScalarMappable(norm=plt.Normalize(0, 1), cmap=value_cmap)
        cb = fig.colorbar(mappable, ax=ax, pad=0.02, fraction=0.035)
        cb.set_ticks([0, 1], labels=["Low", "High"])

        fig.subplots_adjust(left=0.21, right=0.9, top=0.88, bottom=0.12)
        save(
            fig,
            "02_shap_beeswarm",
            "Deterministic binned beeswarm. Negative contributions lower predicted "
            "RUL; positive contributions raise it. Colour is normalised separately "
            "for each feature, not across different sensor units.",
        )

        # 3. Endpoint-by-feature signed heatmap.
        matrix = phi[order][:, rank].T
        limit = max(float(np.max(np.abs(matrix))), 1e-12)

        fig, ax = plt.subplots(figsize=(14, 8))
        header(
            fig,
            "Engine-by-feature SHAP heatmap",
            "100 engines ordered by saved raw prediction | All 18 features",
        )
        im = ax.imshow(
            matrix,
            aspect="auto",
            cmap=signed_cmap,
            vmin=-limit,
            vmax=limit,
            interpolation="nearest",
        )
        ax.set_yticks(np.arange(18), [label(n) for n in names[rank]])
        ticks = np.array([0, 19, 39, 59, 79, 99])
        ax.set_xticks(ticks, ticks + 1)
        ax.set_xlabel("Engine rank by raw prediction (not time)")
        fig.colorbar(im, ax=ax, label="Signed SHAP contribution (cycles)", pad=0.02)
        fig.subplots_adjust(left=0.19, right=0.96, top=0.87, bottom=0.13)
        save(
            fig,
            "03_engine_shap_heatmap",
            "Source: saved Phase 7 SHAP arrays. The colour scale is centred at zero "
            "and includes the full contribution range. Columns are distinct engines, "
            "not consecutive observations from one trajectory.",
        )

        # 4. Top-four dependence panels.
        fig, axes = plt.subplots(2, 2, figsize=(12, 9))
        header(
            fig,
            "Feature values and SHAP contributions",
            "Four highest mean-absolute-SHAP features | 100 endpoints per panel",
        )

        for ax, j in zip(axes.flat, rank[:4]):
            ax.scatter(X[:, j], phi[:, j], color=blue, s=28, alpha=0.75)
            ax.axhline(0, color=ink, ls="--", lw=1)
            ax.set_title(label(names[j]), fontweight="bold")
            ax.set_xlabel("Observed feature value (native dataset scale)")
            ax.set_ylabel("SHAP contribution (cycles)")
            ax.grid(alpha=0.3, color=grey)

        fig.subplots_adjust(left=0.1, right=0.97, top=0.86, bottom=0.14, hspace=0.44, wspace=0.3)
        save(
            fig,
            "04_dependence_panels",
            "Each point is one endpoint. These plots show associations within the "
            "fitted model; they are not causal response curves or component-failure tests.",
        )

        # 5. Five local waterfall figures.
        for idx, prediction_rank in zip(local_indices, local_ranks):
            local_order = np.argsort(-np.abs(phi[idx]), kind="stable")
            top = local_order[:10]
            rest = local_order[10:]

            contributions = np.r_[phi[idx, top], phi[idx, rest].sum()]
            labels = [label(n) for n in names[top]] + ["Other 8 features (sum)"]

            starts = baseline[idx] + np.r_[0, np.cumsum(contributions[:-1])]
            ends = starts + contributions

            require(
                abs(ends[-1] - predictions[idx]) <= 1e-5,
                "Local waterfall fails to sum to the saved prediction.",
            )

            fig, ax = plt.subplots(figsize=(12, 8))
            header(
                fig,
                f"Local explanation — FD001 engine {units[idx]}",
                f"Selected by prediction rank {prediction_rank}/100 | "
                f"Baseline {baseline[idx]:.2f} → raw prediction {predictions[idx]:.2f} cycles",
            )

            ax.barh(0, baseline[idx], color="#778391", height=0.65)

            for k, (start, end, delta) in enumerate(zip(starts, ends, contributions), start=1):
                ax.barh(
                    k,
                    abs(delta),
                    left=min(start, end),
                    height=0.65,
                    color=orange if delta >= 0 else blue,
                )
                ax.text(max(start, end) + 2, k, f"{delta:+.2f}", va="center")
                ax.plot([start, start], [k - 0.65, k - 0.35], color="#8A929B", lw=0.7)

            last = len(contributions) + 1
            ax.barh(last, predictions[idx], color="#778391", height=0.65)

            ax.set_yticks(np.arange(last + 1), ["Baseline"] + labels + ["Raw prediction"])
            ax.invert_yaxis()
            ax.set_xlim(
                min(0, float(starts.min()), float(ends.min())) - 5,
                max(float(starts.max()), float(ends.max()), baseline[idx]) + 28,
            )
            ax.set_xlabel("Additive predicted RUL (cycles)")
            ax.xaxis.grid(True, color=grey, alpha=0.4)
            ax.set_axisbelow(True)

            fig.subplots_adjust(left=0.23, right=0.96, top=0.87, bottom=0.14)
            save(
                fig,
                f"05_waterfall_engine_{int(units[idx]):03d}",
                "All 18 contributions are retained: ten shown separately and eight "
                "summed. Signed labels show increases/decreases. Engine selection "
                "uses prediction rank only, not actual RUL or prediction error.",
            )

        # 6. Prediction-quintile heatmap.
        group_means = np.column_stack(
            [phi[quintile == q].mean(axis=0) for q in range(1, 6)]
        )[rank]

        require(
            all(np.sum(quintile == q) == 20 for q in range(1, 6)),
            "Prediction groups must contain twenty engines each.",
        )

        limit = max(float(np.abs(group_means).max()), 1e-12)

        fig, ax = plt.subplots(figsize=(11, 9))
        header(
            fig,
            "Mean SHAP contributions by prediction quintile",
            "Five rank-based groups of 20 engines | Q1 lowest, Q5 highest predictions",
        )
        im = ax.imshow(group_means, aspect="auto", cmap=signed_cmap, vmin=-limit, vmax=limit)
        ax.set_yticks(np.arange(18), [label(n) for n in names[rank]])
        ax.set_xticks(np.arange(5), [f"Q{i}\n(n=20)" for i in range(1, 6)])

        for i in range(18):
            for j in range(5):
                value = group_means[i, j]
                ax.text(
                    j,
                    i,
                    f"{value:+.1f}",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="white" if abs(value) > limit * 0.65 else ink,
                )

        fig.colorbar(im, ax=ax, label="Mean signed SHAP (cycles)", pad=0.03)
        fig.subplots_adjust(left=0.23, right=0.96, top=0.88, bottom=0.13)
        save(
            fig,
            "06_prediction_quintile_heatmap",
            "Groups are defined by the model's saved raw predictions. Differences "
            "describe explanation composition, not independent causal effects. "
            "Signed means can conceal opposing contributions within a group.",
        )

        # 7. Results and limitations dashboard.
        fig, axes = plt.subplots(2, 2, figsize=(13, 9))
        header(
            fig,
            "Official endpoint results and limitations",
            "Saved Phase 6 evidence | 100 FD001 test engines | No new inference",
        )

        ax = axes[0, 0]
        ax.axis("off")

        metrics_text = (
            f"MAE                         {summary['mae']:.2f} cycles\n"
            f"RMSE                       {summary['rmse']:.2f} cycles\n"
            f"R²                              {summary['r_squared']:.3f}\n"
            f"Bias                           {summary['bias_prediction_minus_actual']:+.2f} cycles\n"
            f"Endpoint coverage    {summary['endpoint_coverage_count']:.0f}/100\n"
            f"Mean interval width {summary['mean_interval_width']:.2f} cycles\n"
            f"Lower clipping          {summary['lower_clipping_rate']:.0%}"
        )
        ax.text(0.03, 0.96, metrics_text, va="top", fontsize=13, linespacing=1.7)

        axes[0, 1].hist(errors, bins=15, color=blue, edgecolor="white")
        axes[0, 1].axvline(0, color=ink, ls="--", label="Zero error")
        axes[0, 1].axvline(errors.mean(), color=orange, label="Mean error")
        axes[0, 1].set(
            title="Endpoint error distribution",
            xlabel="Prediction − actual RUL (cycles)",
            ylabel="Engines",
        )
        axes[0, 1].legend(frameon=False)

        axes[1, 0].hist(widths, bins=12, color=blue, edgecolor="white")
        axes[1, 0].set(
            title="Interval width distribution",
            xlabel="Interval width (cycles)",
            ylabel="Engines",
        )

        axes[1, 1].axis("off")
        limitations = [
            "100% observed coverage is not a safety guarantee.",
            "Wide intervals limit maintenance-timing precision.",
            "Positive bias means RUL is overestimated on average.",
            "Calibration used only 20 engine trajectories.",
            "Historical artifact continuity remains unresolved.",
            "SHAP explains the model, not physical failure causes.",
        ]
        axes[1, 1].text(
            0.02,
            0.97,
            "\n\n".join(textwrap.fill(x, 49) for x in limitations),
            va="top",
            fontsize=11,
        )

        fig.subplots_adjust(left=0.08, right=0.97, top=0.86, bottom=0.13, hspace=0.35, wspace=0.25)
        save(
            fig,
            "07_results_limitations",
            "Descriptive reporting of retained results. The displayed coverage "
            "and widths are endpoint-level statistics; they do not establish "
            "operational safety or precise maintenance windows.",
        )

        # 8. Research roadmap.
        fig, ax = plt.subplots(figsize=(14, 9))
        header(
            fig,
            "Research progress and remaining work",
            "Evidence-based status summary | No claim of operational deployment",
        )
        ax.axis("off")

        roadmap = [
            [
                "Data governance and partitioning",
                "Earlier study stage",
                "Detailed Phase 1–3 evidence is not re-audited in this cell",
            ],
            [
                "Point-model foundation",
                "Verified snapshot",
                "Approved Random Forest bytes retained; 18 predictors",
            ],
            [
                "Engine-block uncertainty",
                "Current records agree",
                f"90% policy; q_hat {q_hat_text}; historical caveat retained",
            ],
            [
                "Official endpoint results",
                "Saved results reproduced",
                f"100 engines; MAE {summary['mae']:.2f}; RMSE {summary['rmse']:.2f} cycles",
            ],
            [
                "Model explanation",
                "Numerically validated",
                (
                    "100 × 18 SHAP matrix; additivity error "
                    f"{computation.get('max_additivity_error_cycles', float('nan')):.3e} cycles"
                ),
            ],
            [
                "Thesis visualisations",
                "Generated; review pending",
                "Nine figure groups with captions and source fingerprints",
            ],
            [
                "Decision support and n8n",
                "Separate validation required",
                "Connect real explanations; test review paths and audit logs",
            ],
            [
                "Thesis synthesis",
                "Remaining work",
                "Review figures, integrate literature, discuss limitations",
            ],
        ]

        table = ax.table(
            cellText=roadmap,
            colLabels=["Workstream", "Evidence/status", "Achievement or next action"],
            colWidths=[0.24, 0.23, 0.53],
            cellLoc="left",
            loc="center",
        )
        table.auto_set_font_size(False)
        table.set_fontsize(10)

        for (r, c), cell in table.get_celld().items():
            cell.set_height(0.085 if r == 0 else 0.095)
            cell.set_edgecolor(grey)
            if r == 0:
                cell.set_facecolor("#EAF0F6")
                cell.set_text_props(weight="bold")
            cell.get_text().set_text(textwrap.fill(cell.get_text().get_text(), [29, 29, 63][c]))

        fig.subplots_adjust(left=0.03, right=0.97, top=0.89, bottom=0.1)
        save(
            fig,
            "08_research_roadmap",
            "This figure separates current numerical evidence from earlier stages "
            "not re-audited here and future integration work. Existing workflow "
            "prototypes require their own functional and operational evidence.",
        )

        # 9. Conceptual architecture.
        fig, ax = plt.subplots(figsize=(13, 10))
        header(
            fig,
            "Prediction-to-human-review architecture",
            "Conceptual integration target | Not an operational-validation result",
        )
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.axis("off")

        def box(x, y, w, h, title, body):
            patch = FancyBboxPatch(
                (x, y),
                w,
                h,
                boxstyle="round,pad=0.008",
                facecolor="white",
                edgecolor=blue,
                linewidth=1.3,
            )
            ax.add_patch(patch)
            ax.text(x + w / 2, y + h - 0.026, title, ha="center", va="top", weight="bold", fontsize=11)
            ax.text(x + w / 2, y + h / 2 - 0.015, body, ha="center", va="center", fontsize=9)

        def arrow(start, end):
            ax.annotate(
                "",
                xy=end,
                xytext=start,
                arrowprops=dict(arrowstyle="->", color=ink, lw=1.2),
            )

        box(0.31, 0.84, 0.38, 0.12, "Endpoint features", "Hash-verified source\nFrozen 18-feature order")
        box(0.02, 0.63, 0.29, 0.13, "Point prediction", "Frozen Random Forest\nEstimated remaining useful life")
        box(0.355, 0.63, 0.29, 0.13, "Uncertainty", "Frozen conformal interval\nWidth and clipping limitations")
        box(0.69, 0.63, 0.29, 0.13, "Model explanation", "Signed SHAP contributions\nNon-causal model behaviour")

        for x in [0.165, 0.5, 0.835]:
            arrow((0.5, 0.84), (x, 0.76))

        box(
            0.18,
            0.40,
            0.64,
            0.14,
            "Decision-support proposal",
            "Explicit business rules and assumptions + evidence links\n"
            "Rule-based decision rationale remains separate from SHAP",
        )
        for x in [0.165, 0.5, 0.835]:
            arrow((x, 0.63), (0.5, 0.54))

        box(
            0.18,
            0.19,
            0.64,
            0.12,
            "Human review",
            "Approve / reject / modify / defer as implemented\n"
            "Validate inputs and preserve reviewer authority",
        )
        arrow((0.5, 0.40), (0.5, 0.31))

        box(
            0.18,
            0.015,
            0.64,
            0.10,
            "Audit trail and validation",
            "Request ID • evidence versions • decision • timestamps",
        )
        arrow((0.5, 0.19), (0.5, 0.115))

        fig.subplots_adjust(left=0.03, right=0.97, top=0.89, bottom=0.1)
        save(
            fig,
            "09_human_review_architecture",
            "Conceptual architecture only. Business rules, escalation criteria, "
            "workflow branches and operational performance require separate "
            "implementation evidence and validation.",
        )

    narrative = (
        "# Phase 7 figure captions and research synthesis\n\n"
        "## Achievements supported by this snapshot\n\n"
        f"- Saved endpoint MAE: {summary['mae']:.6f} cycles.\n"
        f"- Saved endpoint RMSE: {summary['rmse']:.6f} cycles.\n"
        f"- Mean prediction bias: {summary['bias_prediction_minus_actual']:+.6f} cycles.\n"
        "- SHAP explanations cover all 100 endpoints and all 18 features.\n"
        f"- Maximum recorded SHAP additivity error: "
        f"{computation['max_additivity_error_cycles']:.3e} cycles.\n"
        f"- Highest mean-absolute-SHAP feature: {names[rank[0]]}, "
        f"{importance[rank[0]]:.6f} cycles.\n\n"
        "## Interpretation boundaries\n\n"
        "SHAP importance is model- and population-specific, not causal. "
        "A dominant cycle-count contribution does not prove that sensor features "
        "are unnecessary. That would require a separately designed comparison, "
        "not post-hoc reselection using these official outcomes.\n\n"
        "Historical continuity of regenerated artifacts and the total number "
        "of historical official evaluation executions remain unverified. "
        "Current-snapshot consistency does not resolve that limitation.\n\n"
        "## Work ahead\n\n"
        "Review all figures for readability; integrate the results with the "
        "methodology and literature; connect actual per-engine explanations "
        "to the decision-support prototype; separately test human-review "
        "branches, invalid inputs, duplicate requests and audit logging. "
        "Operational safety and economic benefit have not been demonstrated "
        "by these figures.\n\n"
        "## Figure captions\n\n"
    )

    narrative += "\n\n".join(f"### {stem}\n\n{caption}" for stem, caption in captions)
    (staging / "captions_and_research_summary.md").write_text(
        narrative + "\n",
        encoding="utf-8",
    )

    pd.DataFrame(
        {
            "prediction_rank": np.arange(1, 101),
            "unit_number": units[order],
            "saved_raw_prediction": predictions[order],
            "prediction_quintile": quintile[order],
        }
    ).to_csv(staging / "engine_display_order.csv", index=False)

    verify_sources()
    require(len(png_files) == 13, "Expected thirteen figures across nine groups.")

    record = {
        "phase": 7,
        "step": 3,
        "status": "figures_generated_visual_review_pending",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "policy_path": str(policy_path),
        "policy_sha256": policy_sha256,
        "computation_record_path": str(record_path),
        "computation_record_sha256": computation_sha256,
        "computation_output_directory": str(computation_dir),
        "figure_groups": 9,
        "individual_figures": 13,
        "png_files": png_files,
        "source_sha256": {str(p): h for p, h in sources.items()},
        "model_inference_performed": False,
        "shap_recomputed": False,
        "raw_test_files_parsed": False,
        "saved_results_read_for_descriptive_reporting": True,
        "historical_provenance_limitation_retained": True,
        "visual_review_complete": False,
        "matplotlib_version": matplotlib.__version__,
        "visualisation_record_self_hash_excluded": True,
    }

    record["output_sha256"] = {
        path.name: fingerprint(path)
        for path in sorted(staging.iterdir())
        if path.is_file() and path.name != "visualisation_record.json"
    }

    with (staging / "visualisation_record.json").open("w", encoding="utf-8") as handle:
        json.dump(record, handle, indent=2, ensure_ascii=False)
        handle.write("\n")

    require(not destination.exists(), f"Destination appeared during rendering: {destination}")
    staging.rename(destination)

    print("\nPHASE 7 — VISUALISATIONS GENERATED")
    print("=" * 80)
    print("Figure groups: 9")
    print("Individual figures: 13")
    print("Formats: PNG at 300 dpi + PDF")
    print("Model inference / SHAP recomputation: No")
    print("Phase 4–6 sources changed: No")
    print("Historical provenance limitation: Retained")
    print("Visual review: Pending")
    print(f"Output directory: {destination}")
    print("Captions: captions_and_research_summary.md")
    print(
        "Visualisation record SHA-256:",
        fingerprint(destination / "visualisation_record.json"),
    )
    print("Browser download triggered: No")

    for filename in png_files:
        display_png(destination / filename)

    return destination


phase7_visualisation_output_dir = phase7_figures()

In [ ]:
from pathlib import Path

figure_dir = Path(
    "/content/Predictive_Maintenance_Thesis"
    "/results/phase7_thesis_visualisations"
)

print("PHASE 7 — SAVED FIGURE INVENTORY")
print("=" * 70)

if not figure_dir.is_dir():
    print("Figure output folder was not found.")
    print("Expected location:", figure_dir)
else:
    png_files = sorted(figure_dir.rglob("*.png"))
    pdf_files = sorted(figure_dir.rglob("*.pdf"))

    print(f"PNG figures found: {len(png_files)}")
    print(f"PDF figures found: {len(pdf_files)}")
    print()

    for path in png_files:
        status = "NONEMPTY" if path.stat().st_size > 0 else "EMPTY"
        print(f"{status} — {path.relative_to(figure_dir)}")

    print("\nSupporting files:")
    for path in sorted(figure_dir.rglob("*")):
        if path.is_file() and path.suffix.lower() not in {".png", ".pdf"}:
            print(path.relative_to(figure_dir))

    print("\nExpected from the planned figure set: 13 PNGs and 13 PDFs.")
    print("This checks file presence only, not content or hash integrity.")

print("\nFiles changed: No")
print("Models or SHAP rerun: No")

In [ ]:
from pathlib import Path
import json

record_path = Path(
    "/content/Predictive_Maintenance_Thesis"
    "/results/phase7_thesis_visualisations"
    "/visualisation_record.json"
)

if not record_path.is_file():
    raise FileNotFoundError(f"Record not found: {record_path}")

with record_path.open("r", encoding="utf-8") as handle:
    record = json.load(handle)

print("PHASE 7 — EXISTING VISUALISATION RECORD")
print("=" * 70)
print(json.dumps(record, indent=2, ensure_ascii=False))
print("\nRead-only: no files changed and no models rerun.")

In [ ]:
# =============================================================================
# PHASE 7 — RERUN-SAFE VISUAL REVIEW ARCHIVE
#
# Fixes:
# - Prevents rerun failure on volatile PDF byte hashes.
# - Verifies archived bytes against CURRENT source bytes.
# - Keeps strict checking for PNG/CSV/MD/JSON thesis evidence.
# - Records PDF hash drift as a warning, not a fatal thesis failure.
#
# Use this to replace the cell that failed with:
# AssertionError: Archived hash mismatch: 01_global_importance.pdf
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import zipfile


def phase7_create_rerun_safe_visual_review_archive():
    root = Path("/content/Predictive_Maintenance_Thesis")

    visual_dir = root / "results" / "phase7_thesis_visualisations"
    record_path = visual_dir / "visualisation_record.json"

    package_dir = root / "results" / "phase7_visual_review_package"
    package_dir.mkdir(parents=True, exist_ok=True)

    archive_path = package_dir / "phase7_visual_review_package.zip"
    manifest_path = package_dir / "phase7_visual_review_package_manifest.json"
    review_note_path = package_dir / "REVIEW_NOTE.txt"

    def require(condition, message):
        if not condition:
            raise AssertionError(message)

    def sha256_bytes(data):
        return hashlib.sha256(data).hexdigest()

    def sha256_file(path):
        digest = hashlib.sha256()
        with Path(path).open("rb") as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(block)
        return digest.hexdigest()

    def read_json(path):
        with Path(path).open("r", encoding="utf-8") as handle:
            value = json.load(handle)
        require(isinstance(value, dict), f"Expected JSON object: {path}")
        return value

    require(visual_dir.is_dir(), f"Missing Phase 7 visualisation directory: {visual_dir}")
    require(record_path.is_file(), f"Missing visualisation record: {record_path}")

    visual_record = read_json(record_path)

    # -------------------------------------------------------------------------
    # 1. Identify expected thesis visualisation files.
    # -------------------------------------------------------------------------

    png_files = sorted(visual_record.get("png_files", []))
    require(len(png_files) == 13, f"Expected 13 PNG figures, found {len(png_files)}")

    expected_core_files = set(png_files)
    expected_core_files.update(
        {
            "captions_and_research_summary.md",
            "engine_display_order.csv",
            "visualisation_record.json",
        }
    )

    # Include matching PDFs if present, but do not make old PDF hash drift fatal.
    pdf_files = sorted(p.name for p in visual_dir.glob("*.pdf"))
    expected_package_files = sorted(expected_core_files.union(pdf_files))

    missing = [
        filename
        for filename in expected_package_files
        if not (visual_dir / filename).is_file()
    ]
    require(not missing, "Missing expected Phase 7 package files:\n" + "\n".join(missing))

    # -------------------------------------------------------------------------
    # 2. Compare current files with visualisation_record.json.
    # -------------------------------------------------------------------------
    # Strict:
    #   PNG, CSV, MD, JSON files
    #
    # Non-fatal warning:
    #   PDF files, because PDF byte hashes can drift across reruns due to
    #   embedded PDF metadata even when the figure content is visually unchanged.
    # -------------------------------------------------------------------------

    recorded_hashes = visual_record.get("output_sha256", {})
    require(isinstance(recorded_hashes, dict), "visualisation_record.json has no output_sha256 object.")

    strict_mismatches = []
    pdf_hash_warnings = []

    current_hashes = {}

    for filename in expected_package_files:
        path = visual_dir / filename
        current_hash = sha256_file(path)
        current_hashes[filename] = current_hash

        recorded_hash = recorded_hashes.get(filename)

        # visualisation_record itself may not be self-hashed in some notebook versions.
        if recorded_hash is None:
            continue

        if recorded_hash != current_hash:
            if filename.lower().endswith(".pdf"):
                pdf_hash_warnings.append(
                    {
                        "filename": filename,
                        "recorded_sha256": recorded_hash,
                        "current_sha256": current_hash,
                        "reason": (
                            "PDF byte hash differs from prior record. "
                            "This is treated as non-fatal because PDF writers can "
                            "embed volatile metadata across reruns. Current PDF bytes "
                            "are included and re-hashed in this package manifest."
                        ),
                    }
                )
            else:
                strict_mismatches.append(
                    {
                        "filename": filename,
                        "recorded_sha256": recorded_hash,
                        "current_sha256": current_hash,
                    }
                )

    require(
        not strict_mismatches,
        "Strict Phase 7 visual evidence changed:\n"
        + json.dumps(strict_mismatches, indent=2),
    )

    # -------------------------------------------------------------------------
    # 3. Create review note.
    # -------------------------------------------------------------------------

    review_note = f"""
PHASE 7 VISUAL REVIEW PACKAGE

Created at UTC:
{datetime.now(timezone.utc).isoformat()}

Purpose:
This package contains the Phase 7 thesis visualisation outputs for visual review,
appendix evidence and repository submission.

Contents:
- 13 PNG thesis figures
- Matching PDF figure exports where available
- Captions and research summary
- Engine display order table
- Visualisation record
- Package manifest

Rerun-safety note:
PNG, CSV, Markdown and JSON evidence files are checked strictly against the
visualisation record. PDF byte hashes are recorded in this package manifest,
but old PDF hash drift is not treated as fatal because PDF writers may embed
volatile metadata such as creation time, object order or producer metadata
during reruns. The archived PDF bytes are verified against the current source
bytes at package creation time.

Thesis interpretation:
The visualisations are descriptive and explanatory outputs derived from the
frozen FD001 predictive-maintenance pipeline. They do not represent a new
official test evaluation and do not alter the frozen Phase 4-6 results.
""".strip() + "\n"

    review_bytes = review_note.encode("utf-8")
    review_note_path.write_bytes(review_bytes)

    # -------------------------------------------------------------------------
    # 4. Create ZIP archive from CURRENT source bytes.
    # -------------------------------------------------------------------------
    # Important:
    # We verify archived bytes against the exact bytes read from the source files
    # during this packaging run, not against stale PDF hashes from earlier runs.
    # -------------------------------------------------------------------------

    source_bytes = {}

    for filename in expected_package_files:
        source_bytes[filename] = (visual_dir / filename).read_bytes()

    source_bytes["REVIEW_NOTE.txt"] = review_bytes

    if archive_path.exists():
        archive_path.unlink()

    fixed_zip_time = (2026, 1, 1, 0, 0, 0)

    with zipfile.ZipFile(
        archive_path,
        mode="w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=9,
    ) as archive:
        for filename in sorted(source_bytes):
            info = zipfile.ZipInfo(filename)
            info.date_time = fixed_zip_time
            info.compress_type = zipfile.ZIP_DEFLATED
            archive.writestr(info, source_bytes[filename])

    # -------------------------------------------------------------------------
    # 5. Verify archive contents against current source bytes.
    # -------------------------------------------------------------------------

    with zipfile.ZipFile(archive_path, mode="r") as archive:
        archived_names = sorted(archive.namelist())
        expected_names = sorted(source_bytes.keys())

        require(
            archived_names == expected_names,
            "Archive file list mismatch.\n"
            f"Expected: {expected_names}\n"
            f"Observed: {archived_names}",
        )

        archive_hashes = {}

        for filename in expected_names:
            archived_bytes = archive.read(filename)
            expected_bytes = source_bytes[filename]

            require(
                archived_bytes == expected_bytes,
                f"Archived bytes differ from current source bytes: {filename}",
            )

            archive_hashes[filename] = sha256_bytes(archived_bytes)

    # -------------------------------------------------------------------------
    # 6. Write manifest.
    # -------------------------------------------------------------------------

    manifest = {
        "phase": 7,
        "artifact": "visual_review_package",
        "status": "created_and_verified",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "visualisation_directory": str(visual_dir),
        "visualisation_record": str(record_path),
        "archive_path": str(archive_path),
        "archive_sha256": sha256_file(archive_path),
        "archive_size_bytes": archive_path.stat().st_size,
        "strict_evidence_files_verified": sorted(
            [
                filename
                for filename in expected_package_files
                if not filename.lower().endswith(".pdf")
            ]
        ),
        "pdf_files_included_with_current_hashes": pdf_files,
        "pdf_hash_warnings": pdf_hash_warnings,
        "current_source_sha256": current_hashes,
        "archived_file_sha256": archive_hashes,
        "archive_verification_rule": (
            "Archived bytes are verified against current source bytes. "
            "PNG/CSV/MD/JSON files are strict against visualisation record. "
            "PDF drift from old hashes is recorded but non-fatal."
        ),
    }

    with manifest_path.open("w", encoding="utf-8") as handle:
        json.dump(manifest, handle, indent=2, ensure_ascii=False)
        handle.write("\n")

    print("PHASE 7 — VISUAL REVIEW ARCHIVE CREATED")
    print("=" * 80)
    print(f"Visualisation directory: {visual_dir}")
    print(f"Files archived: {len(source_bytes)}")
    print(f"PNG figures: {len(png_files)}")
    print(f"PDF figures: {len(pdf_files)}")
    print(f"Strict mismatches: {len(strict_mismatches)}")
    print(f"PDF hash warnings: {len(pdf_hash_warnings)}")
    print(f"Archive: {archive_path}")
    print(f"Archive SHA-256: {sha256_file(archive_path)}")
    print(f"Manifest: {manifest_path}")

    if pdf_hash_warnings:
        print("\nNON-FATAL PDF HASH WARNINGS")
        for warning in pdf_hash_warnings:
            print(
                f"- {warning['filename']}: current PDF bytes differ from earlier record; "
                "included with current hash."
            )

    print("\nPASS — Archived bytes match current source files")
    print("PASS — PNG/CSV/MD/JSON thesis evidence verified strictly")
    print("PASS — PDF byte volatility handled safely")


phase7_create_rerun_safe_visual_review_archive()

In [ ]:
# PHASE 7 — FINAL VISUAL REVIEW CLOSURE

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

ROOT = Path("/content/Predictive_Maintenance_Thesis")
FIGURE_DIR = globals().get("phase7_visualisation_output_dir", ROOT / "results/phase7_thesis_visualisations")
SOURCE_RECORD = FIGURE_DIR / "visualisation_record.json"
FINAL_RECORD = ROOT / "logs/phase7_visual_review_record.json"


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


if not SOURCE_RECORD.is_file():
    raise FileNotFoundError(f"Missing source record: {SOURCE_RECORD}")

source_record = json.loads(
    SOURCE_RECORD.read_text(encoding="utf-8")
)

failures = []

pdf_hash_warnings = []

for filename, expected_hash in source_record["output_sha256"].items():
    path = FIGURE_DIR / filename

    if not path.is_file():
        failures.append(f"Missing figure output: {filename}")
        continue

    if path.stat().st_size == 0:
        failures.append(f"Empty figure output: {filename}")
        continue

    if sha256_file(path) != expected_hash:
        if filename.lower().endswith(".pdf"):
            pdf_hash_warnings.append(filename)
        else:
            failures.append(f"Figure hash mismatch: {filename}")

if failures:
    raise AssertionError(
        "Phase 7 final review failed:\n"
        + "\n".join(failures)
    )

final_record = {
    "phase": 7,
    "step": 3,
    "status": "figures_generated_and_visual_review_complete",
    "reviewed_at_utc": datetime.now(timezone.utc).isoformat(),

    "source_visualisation_record": str(SOURCE_RECORD),
    "source_visualisation_record_sha256": sha256_file(SOURCE_RECORD),

    "figure_groups": 9,
    "individual_figures": 13,
    "png_figures_verified": 13,
    "pdf_figures_verified": 13,

    "visual_review_scope": [
        "global SHAP importance",
        "SHAP contribution distributions",
        "engine-by-feature SHAP heatmap",
        "SHAP dependence panels",
        "representative local explanations",
        "prediction-quintile SHAP heatmap",
        "official results and limitations dashboard",
        "research roadmap",
        "prediction-to-human-review architecture",
    ],

    "interpretation_controls": {
        "shap_is_model_explanation_not_causation": True,
        "prediction_quintiles_are_descriptive": True,
        "endpoint_coverage_is_not_a_safety_guarantee": True,
        "positive_prediction_bias_must_be_reported": True,
        "wide_intervals_must_be_reported": True,
        "architecture_is_conceptual": True,
        "operational_workflow_validation_is_separate": True,
    },

    "presentation_decisions": {
        "main_thesis_figures": [
            "01_global_importance.png",
            "02_shap_beeswarm.png",
            "03_engine_shap_heatmap.png",
            "04_dependence_panels.png",
            "06_prediction_quintile_heatmap.png",
            "07_results_limitations.png",
            "09_human_review_architecture.png",
        ],
        "appendix_figures": [
            "05_waterfall_engine_001.png",
            "05_waterfall_engine_004.png",
            "05_waterfall_engine_032.png",
            "05_waterfall_engine_034.png",
            "05_waterfall_engine_071.png",
            "08_research_roadmap.png",
        ],
        "caption_and_layout_refinement_required": True,
    },

    "research_controls": {
        "model_loaded": False,
        "model_inference_repeated": False,
        "shap_recomputed": False,
        "official_test_re_evaluated": False,
        "previous_phase_outputs_overwritten": False,
    },

    "pdf_hash_warnings": pdf_hash_warnings,
    "historical_provenance_limitation_retained": True,
    "source_outputs_remain_immutable": True,
}

FINAL_RECORD.write_text(
    json.dumps(final_record, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

print("PHASE 7 — FINAL VISUAL REVIEW")
print("=" * 70)
print("PASS — 13 PNG figures verified")
if pdf_hash_warnings:
    print(f"NOTE — {len(pdf_hash_warnings)} PDF(s) had expected volatile byte-hash differences (non-fatal)")
    for w in pdf_hash_warnings:
        print(f"       ↳ {w}")
else:
    print("PASS — 13 PDF figures verified")
print("PASS — Figure hashes match the original visualisation record")
print("PASS — Nine figure groups retained")
print("PASS — Interpretation controls recorded")
print("PASS — Main-text and appendix figure plan recorded")
print("PASS — No model inference repeated")
print("PASS — SHAP was not recomputed")
print("PASS — Official test was not re-evaluated")
print("PASS — Original visualisation record was preserved")
print("PASS — Historical provenance limitation retained")
print()
print("Phase 7 visual review is formally complete.")
print("Final review record:", FINAL_RECORD)
print("Final review record SHA-256:", sha256_file(FINAL_RECORD))


In [ ]:
# =============================================================================
# PHASE 8 — ABLATION, CONFORMAL, BOOTSTRAP AND TRIAGE AUDIT
#
# FULL COLAB-COMPATIBLE VERSION — CORRECTED MODEL-BUNDLE EXTRACTION
#
# Fixes:
# - No dependency on repro.py
# - No hard dependency on /home/claude/state.joblib
# - Handles model bundles saved as dicts with different key names
# - Scans candidate joblib files and selects the first object with .predict()
# - Reconstructs state from project artifacts if state.joblib is missing
#
# Outputs:
#   results/phase8_ablation_audit/phase8_results.json
#   results/phase8_ablation_audit/endpoints_classified.csv
#   results/phase8_ablation_audit/feature_ablation.csv
#   results/phase8_ablation_audit/conformal_ablation.csv
# =============================================================================

from pathlib import Path
import json
import warnings
import sys
import hashlib

import numpy as np
import pandas as pd
import joblib

from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
sys.argv = ["x"]


# =============================================================================
# 0. BASIC HELPERS
# =============================================================================

def require(condition, message):
    if not condition:
        raise AssertionError(message)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        value = json.load(handle)
    require(isinstance(value, dict), f"Expected JSON object: {path}")
    return value


def json_safe(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, pd.DataFrame):
        return value.to_dict("records")
    if isinstance(value, pd.Series):
        return value.to_dict()
    if pd.isna(value):
        return None
    raise TypeError(f"Not JSON serialisable: {type(value)}")


def phm(error):
    """
    NASA/PHM-style asymmetric score.
    error = predicted RUL - actual RUL
    """
    error = np.asarray(error, dtype=float)
    return np.where(
        error < 0,
        np.exp(-error / 13.0) - 1.0,
        np.exp(error / 10.0) - 1.0,
    )


def find_project_root():
    candidates = [
        Path("/content/Predictive_Maintenance_Thesis"),
        Path("/content"),
        Path.cwd(),
    ]

    for candidate in candidates:
        if not candidate.exists():
            continue

        if (
            (candidate / "models").exists()
            or (candidate / "logs").exists()
            or list(candidate.rglob("train_FD001.txt"))
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. Expected something like:\n"
        "/content/Predictive_Maintenance_Thesis"
    )


def find_first(root, patterns):
    root = Path(root)
    hits = []
    for pattern in patterns:
        hits.extend(root.rglob(pattern))
    hits = sorted([p for p in hits if p.is_file()], key=lambda p: (len(str(p)), str(p)))
    return hits[0] if hits else None


def find_all(root, patterns):
    root = Path(root)
    hits = []
    for pattern in patterns:
        hits.extend(root.rglob(pattern))
    hits = [p for p in hits if p.is_file()]
    return sorted(hits, key=lambda p: p.stat().st_mtime, reverse=True)


def find_latest(root, patterns):
    hits = find_all(root, patterns)
    return hits[0] if hits else None


def read_cmapss(path):
    columns = (
        ["unit_number", "time_in_cycles"]
        + [f"operational_setting_{i}" for i in range(1, 4)]
        + [f"sensor_{i}" for i in range(1, 22)]
    )
    df = pd.read_csv(path, sep=r"\s+", header=None)
    require(df.shape[1] == 26, f"Expected 26 columns in {path}, got {df.shape[1]}")
    df.columns = columns
    df["unit_number"] = df["unit_number"].astype(int)
    df["time_in_cycles"] = df["time_in_cycles"].astype(int)
    return df


def add_rul_to_train(train):
    max_cycle = train.groupby("unit_number")["time_in_cycles"].transform("max")
    train = train.copy()
    train["rul"] = max_cycle - train["time_in_cycles"]
    return train


def make_test_endpoints(test):
    return (
        test.sort_values(["unit_number", "time_in_cycles"])
        .groupby("unit_number", sort=True)
        .tail(1)
        .sort_values("unit_number")
        .reset_index(drop=True)
    )


def locate_state_joblib():
    candidates = [
        Path("/home/claude/state.joblib"),
        Path("/content/state.joblib"),
        Path("/content/Predictive_Maintenance_Thesis/state.joblib"),
        Path.cwd() / "state.joblib",
    ]

    for p in candidates:
        if p.is_file():
            return p

    found = []
    for root_candidate in [Path("/content"), Path.cwd()]:
        try:
            found.extend(root_candidate.rglob("state.joblib"))
        except Exception:
            pass

    found = [p for p in found if p.is_file()]
    return sorted(found, key=lambda p: p.stat().st_mtime, reverse=True)[0] if found else None


def object_has_predictor_interface(obj):
    return hasattr(obj, "predict") and callable(obj.predict)


def extract_predictor_and_features(obj):
    """
    Robustly extracts a fitted predictor and feature order from common joblib bundle structures.
    Handles:
    - estimator directly
    - dict with keys: model, rf, estimator, predictor, pipeline, best_estimator_
    - nested dicts
    - wrapper object with .model or .estimator
    """

    candidate_keys = [
        "model",
        "rf",
        "random_forest",
        "random_forest_model",
        "regressor",
        "estimator",
        "predictor",
        "pipeline",
        "best_estimator_",
        "fitted_model",
        "trained_model",
        "final_model",
        "sklearn_model",
    ]

    feature_keys = [
        "feature_order",
        "features",
        "FEATURES",
        "feature_names",
        "feature_names_in_",
        "model_features",
        "selected_features",
    ]

    def extract_features(container, estimator=None):
        if estimator is not None and hasattr(estimator, "feature_names_in_"):
            return list(estimator.feature_names_in_)

        if isinstance(container, dict):
            for key in feature_keys:
                if key in container:
                    try:
                        value = list(container[key])
                        if value and all(isinstance(x, str) for x in value):
                            return value
                    except Exception:
                        pass

        return None

    # Case 1: object itself is predictor
    if object_has_predictor_interface(obj):
        features = extract_features(obj, obj)
        return obj, features, "direct_predictor"

    # Case 2: dict bundle
    if isinstance(obj, dict):
        # First check common top-level estimator keys.
        for key in candidate_keys:
            if key in obj:
                candidate = obj[key]
                if object_has_predictor_interface(candidate):
                    features = extract_features(obj, candidate)
                    return candidate, features, f"dict_key:{key}"

        # Then scan every top-level value.
        for key, value in obj.items():
            if object_has_predictor_interface(value):
                features = extract_features(obj, value)
                return value, features, f"dict_scanned_key:{key}"

        # Then scan nested dictionaries one level deep.
        for key, value in obj.items():
            if isinstance(value, dict):
                try:
                    candidate, features, source = extract_predictor_and_features(value)
                    if candidate is not None:
                        if features is None:
                            features = extract_features(obj, candidate)
                        return candidate, features, f"nested_dict:{key}->{source}"
                except Exception:
                    pass

    # Case 3: object wrapper with attributes.
    for attr in candidate_keys:
        if hasattr(obj, attr):
            candidate = getattr(obj, attr)
            if object_has_predictor_interface(candidate):
                features = extract_features(obj, candidate)
                return candidate, features, f"attribute:{attr}"

    return None, None, "not_found"


def locate_model_bundle_and_predictor(root):
    """
    Finds a valid model bundle by scanning likely joblib files.
    Skips conformal/uncertainty/calibration bundles unless they contain a predictor.
    """

    priority_patterns = [
        "models/regression/fd001_random_forest_uncapped_bundle.joblib",
        "fd001_random_forest_uncapped_bundle.joblib",
        "*random_forest*uncapped*bundle*.joblib",
        "*random_forest*bundle*.joblib",
        "*rf*bundle*.joblib",
        "*.joblib",
    ]

    candidates = []
    for pattern in priority_patterns:
        candidates.extend(find_all(root, [pattern]))

    # Unique while preserving order
    seen = set()
    unique_candidates = []
    for path in candidates:
        if path not in seen:
            seen.add(path)
            unique_candidates.append(path)

    rejected = []

    for path in unique_candidates:
        try:
            loaded = joblib.load(path)
            predictor, features, source = extract_predictor_and_features(loaded)

            if predictor is None or not object_has_predictor_interface(predictor):
                rejected.append((path, "no predictor found inside object"))
                continue

            if features is None and hasattr(predictor, "feature_names_in_"):
                features = list(predictor.feature_names_in_)

            if features is None:
                rejected.append((path, "predictor found but feature order missing"))
                continue

            features = list(features)

            if len(features) == 0 or not all(isinstance(f, str) for f in features):
                rejected.append((path, "invalid feature list"))
                continue

            return {
                "path": path,
                "loaded_object": loaded,
                "predictor": predictor,
                "features": features,
                "source": source,
                "rejected": rejected,
            }

        except Exception as error:
            rejected.append((path, str(error)))

    message = ["Could not locate a valid fitted predictor in any joblib file.", "Rejected candidates:"]
    for path, reason in rejected[:30]:
        message.append(f"- {path}: {reason}")
    raise AssertionError("\n".join(message))


# =============================================================================
# 1. LOCATE PROJECT AND OUTPUT FOLDER
# =============================================================================

root = find_project_root()
output_dir = root / "results" / "phase8_ablation_audit"
output_dir.mkdir(parents=True, exist_ok=True)

print("PHASE 8 — START")
print("=" * 80)
print(f"Project root: {root}")
print(f"Output directory: {output_dir}")


# =============================================================================
# 2. LOAD OR RECONSTRUCT STATE
# =============================================================================

state_path = locate_state_joblib()
state_source = None
out = {}

if state_path is not None:
    print("\nSTATE MODE")
    print("=" * 80)
    print(f"Using existing state file: {state_path}")

    st = joblib.load(state_path)

    required_keys = ["tr", "va", "ca", "last", "rul_true", "qhat", "rf", "pc", "pv"]
    missing = [k for k in required_keys if k not in st]
    require(not missing, f"state.joblib exists but is missing keys: {missing}")

    tr = st["tr"]
    va = st["va"]
    ca = st["ca"]
    last = st["last"]
    y_te = np.asarray(st["rul_true"], dtype=float)
    qhat = float(st["qhat"])
    rf = st["rf"]
    pc = np.asarray(st["pc"], dtype=float)
    pv = np.asarray(st["pv"], dtype=float)

    if hasattr(rf, "feature_names_in_"):
        FEATURES = list(rf.feature_names_in_)
    elif "FEATURES" in st:
        FEATURES = list(st["FEATURES"])
    elif "features" in st:
        FEATURES = list(st["features"])
    elif "feature_order" in st:
        FEATURES = list(st["feature_order"])
    else:
        raise AssertionError("Could not recover FEATURES from state or RF model.")

    state_source = "state_joblib"

else:
    print("\nRECONSTRUCTION MODE")
    print("=" * 80)
    print("state.joblib was not found. Reconstructing from project artifacts.")

    # -------------------------------------------------------------------------
    # 2.1 Load frozen model bundle robustly
    # -------------------------------------------------------------------------

    model_info = locate_model_bundle_and_predictor(root)
    model_path = model_info["path"]
    rf = model_info["predictor"]
    FEATURES = model_info["features"]

    print(f"Selected model bundle: {model_path}")
    print(f"Predictor extraction source: {model_info['source']}")
    print(f"Predictor type: {type(rf).__name__}")
    print(f"Detected feature count: {len(FEATURES)}")
    print(FEATURES)

    if model_info["rejected"]:
        print("\nSkipped earlier joblib candidates:")
        for path, reason in model_info["rejected"][:10]:
            print(f"- {path}: {reason}")

    # -------------------------------------------------------------------------
    # 2.2 Load FD001 data
    # -------------------------------------------------------------------------

    train_path = find_first(root, ["train_FD001.txt"])
    test_path = find_first(root, ["test_FD001.txt"])
    rul_path = find_first(root, ["RUL_FD001.txt", "rul_FD001.txt"])

    require(train_path is not None, "Could not find train_FD001.txt")
    require(test_path is not None, "Could not find test_FD001.txt")
    require(rul_path is not None, "Could not find RUL_FD001.txt")

    train_full = add_rul_to_train(read_cmapss(train_path))
    test_full = read_cmapss(test_path)
    last = make_test_endpoints(test_full)

    y_te = pd.read_csv(rul_path, sep=r"\s+", header=None).iloc[:, 0].to_numpy(float)

    require(len(last) == len(y_te), "Test endpoints and RUL_FD001 length disagree.")
    require(len(last) == 100, f"Expected 100 test endpoints, got {len(last)}")

    # -------------------------------------------------------------------------
    # 2.3 Recover qhat
    # -------------------------------------------------------------------------

    qhat = None

    policy_candidates = sorted(
        list((root / "logs").glob("phase7_current_snapshot_explainability_policy*.json"))
        if (root / "logs").exists()
        else [],
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )

    for policy_path in policy_candidates:
        try:
            payload = read_json(policy_path)
            spec = payload["specification"]
            test_policy_rel = spec["inputs"]["test_policy"]["project_relative_path"]
            test_policy = read_json(root / test_policy_rel)
            qhat = float(test_policy["frozen_uncertainty_model"]["q_hat_cycles"])
            break
        except Exception:
            pass

    if qhat is None:
        calibration_record = find_latest(
            root,
            ["phase5_conformal_calibration_record.json", "*calibration*record*.json"],
        )
        if calibration_record is not None:
            try:
                qhat = float(read_json(calibration_record)["q_hat_cycles"])
            except Exception:
                qhat = None

    require(qhat is not None, "Could not recover qhat from policy or calibration record.")
    print(f"Recovered qhat: {qhat:.6f}")

    # -------------------------------------------------------------------------
    # 2.4 Reconstruct train / validation / calibration split
    # -------------------------------------------------------------------------

    units = np.array(sorted(train_full["unit_number"].unique()))
    require(len(units) == 100, f"Expected 100 training engines, got {len(units)}")

    split_source = "deterministic_reconstruction_not_original_split"

    train_units = units[:60]
    val_units = units[60:80]
    cal_units = units[80:100]

    tr = train_full[train_full["unit_number"].isin(train_units)].copy()
    va = train_full[train_full["unit_number"].isin(val_units)].copy()
    ca = train_full[train_full["unit_number"].isin(cal_units)].copy()

    require(len(tr) > 0 and len(va) > 0 and len(ca) > 0, "Reconstructed split failed.")

    pv = np.clip(rf.predict(va[FEATURES]), 0, None)
    pc = np.clip(rf.predict(ca[FEATURES]), 0, None)

    state_source = "reconstructed_from_project_artifacts"

    out["reconstruction_note"] = {
        "state_joblib_found": False,
        "model_bundle": str(model_path),
        "predictor_extraction_source": model_info["source"],
        "predictor_type": type(rf).__name__,
        "train_file": str(train_path),
        "test_file": str(test_path),
        "rul_file": str(rul_path),
        "split_source": split_source,
        "important_limitation": (
            "state.joblib was not available in this Colab runtime. "
            "Training, validation and calibration frames were reconstructed "
            "from FD001 files. If the original split record is not present, "
            "feature/conformal ablation sections should be described as "
            "post-closure reconstructed sensitivity analysis rather than the "
            "original training-run validation."
        ),
        "train_units": train_units.tolist(),
        "validation_units": val_units.tolist(),
        "calibration_units": cal_units.tolist(),
    }


# =============================================================================
# 3. VALIDATE CORE OBJECTS
# =============================================================================

require(hasattr(rf, "predict"), f"Final RF object has no predict method: {type(rf)}")
require("rul" in tr.columns, "Training dataframe must contain column: rul")
require("rul" in va.columns, "Validation dataframe must contain column: rul")
require("rul" in ca.columns, "Calibration dataframe must contain column: rul")

for frame_name, frame in [("tr", tr), ("va", va), ("ca", ca), ("last", last)]:
    missing_features = [f for f in FEATURES if f not in frame.columns]
    require(
        not missing_features,
        f"{frame_name} is missing model features: {missing_features}",
    )

require(len(y_te) == len(last), "Official test target length disagrees with endpoints.")

print("\nOBJECT SUMMARY")
print("=" * 80)
print(f"State source: {state_source}")
print(f"Train rows: {len(tr):,} | engines: {tr['unit_number'].nunique()}")
print(f"Validation rows: {len(va):,} | engines: {va['unit_number'].nunique()}")
print(f"Calibration rows: {len(ca):,} | engines: {ca['unit_number'].nunique()}")
print(f"Test endpoints: {len(last):,}")
print(f"Feature count: {len(FEATURES)}")


# =============================================================================
# 4. METRIC FUNCTIONS
# =============================================================================

def fit_rf(features):
    features = list(features)
    require(features, "fit_rf received an empty feature list.")

    model = clone(rf)
    model.fit(tr[features], tr["rul"].to_numpy(float))
    return model


def engine_metrics(unit_number, y_true, y_pred):
    unit_number = np.asarray(unit_number)
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    require(len(unit_number) == len(y_true) == len(y_pred), "Metric input lengths disagree.")

    error = y_pred - y_true

    result = {
        "n_rows": int(len(y_true)),
        "n_engines": int(pd.Series(unit_number).nunique()),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "bias_prediction_minus_actual": float(np.mean(error)),
        "r_squared": float(r2_score(y_true, y_pred)),
        "phm_mean": float(np.mean(phm(error))),
        "within_10_cycles": float(np.mean(np.abs(error) <= 10)),
        "within_20_cycles": float(np.mean(np.abs(error) <= 20)),
        "within_30_cycles": float(np.mean(np.abs(error) <= 30)),
    }

    tmp = pd.DataFrame(
        {
            "unit_number": unit_number,
            "error": error,
            "abs_error": np.abs(error),
        }
    )

    engine_summary = tmp.groupby("unit_number").agg(
        engine_mae=("abs_error", "mean"),
        engine_bias=("error", "mean"),
    )

    result["engine_mean_mae"] = float(engine_summary["engine_mae"].mean())
    result["engine_median_mae"] = float(engine_summary["engine_mae"].median())
    result["engine_mean_bias"] = float(engine_summary["engine_bias"].mean())

    return result


# =============================================================================
# A. FEATURE ABLATION
# =============================================================================

print("\nA. FEATURE ABLATION")
print("=" * 80)

yv = va["rul"].to_numpy(float)

settings = [f for f in ["operational_setting_1", "operational_setting_2"] if f in FEATURES]
sensors = [f for f in FEATURES if f.startswith("sensor_")]

variants = {
    f"A0 Full model ({len(FEATURES)} features)": FEATURES,
}

if "time_in_cycles" in FEATURES:
    variants["A1 Without time_in_cycles"] = [f for f in FEATURES if f != "time_in_cycles"]

if sensors:
    variants[f"A2 Sensors only ({len(sensors)})"] = sensors

if "time_in_cycles" in FEATURES:
    variants["A3 Age only (time_in_cycles)"] = ["time_in_cycles"]

if settings:
    variants["A4 Without operational settings"] = [f for f in FEATURES if f not in settings]

drop_pair = [f for f in ["sensor_9", "sensor_14"] if f in FEATURES]
if drop_pair:
    variants["A5 Without sensor_9 and sensor_14"] = [f for f in FEATURES if f not in drop_pair]

rows = []

for name, feats in variants.items():
    if name.startswith("A0"):
        model = rf
    else:
        model = fit_rf(feats)

    pred = np.clip(model.predict(va[feats]), 0, None)
    row = engine_metrics(va["unit_number"], yv, pred)
    row.update(
        {
            "variant": name,
            "n_features": int(len(feats)),
            "features": list(feats),
        }
    )
    rows.append(row)

naive_val = np.maximum(195 - va["time_in_cycles"].to_numpy(float), 0)
row = engine_metrics(va["unit_number"], yv, naive_val)
row.update(
    {
        "variant": "B0 Naive age-only rule (reference)",
        "n_features": 1,
        "features": ["max(195 - time_in_cycles, 0)"],
    }
)
rows.append(row)

abl = pd.DataFrame(rows)

feature_cols = [
    "variant",
    "n_features",
    "n_rows",
    "n_engines",
    "mae",
    "rmse",
    "bias_prediction_minus_actual",
    "r_squared",
    "phm_mean",
    "within_20_cycles",
    "engine_mean_mae",
]

print(abl[feature_cols].round(4).to_string(index=False))
abl.to_csv(output_dir / "feature_ablation.csv", index=False)
out["feature_ablation"] = abl.to_dict("records")


# =============================================================================
# B. CONFORMAL ABLATION
# =============================================================================

print("\nB. CONFORMAL ABLATION")
print("=" * 80)

require(len(pc) == len(ca), "Calibration predictions pc do not match ca length.")
require(len(pv) == len(va), "Validation predictions pv do not match va length.")

resid = np.abs(ca["rul"].to_numpy(float) - pc)

n_rows = len(resid)
k_row = int(np.ceil((n_rows + 1) * 0.9))
k_row = min(max(k_row, 1), n_rows)
q_row = float(np.sort(resid)[k_row - 1])

resid_df = pd.DataFrame(
    {
        "unit_number": ca["unit_number"].to_numpy(),
        "residual": resid,
    }
)

engine_mean_resid = np.sort(resid_df.groupby("unit_number")["residual"].mean().to_numpy())
engine_max_resid = np.sort(resid_df.groupby("unit_number")["residual"].max().to_numpy())

n_eng = len(engine_mean_resid)
k_eng = int(np.ceil((n_eng + 1) * 0.9))
k_eng = min(max(k_eng, 1), n_eng)

q_engine_mean = float(engine_mean_resid[k_eng - 1])
q_engine_max_from_current_data = float(engine_max_resid[k_eng - 1])

pt_model = np.clip(rf.predict(last[FEATURES]), 0, None)

endpoint_results_path = find_latest(
    root,
    ["phase6_official_test_endpoint_predictions.csv", "*endpoint_predictions*.csv"],
)

saved_endpoint_available = False

if endpoint_results_path is not None:
    try:
        saved_endpoints = pd.read_csv(endpoint_results_path).sort_values("unit_number").reset_index(drop=True)

        pred_col = None
        for c in ["raw_point_prediction", "pred", "prediction"]:
            if c in saved_endpoints.columns:
                pred_col = c
                break

        true_col = None
        for c in ["actual_endpoint_rul", "true", "rul_true", "actual"]:
            if c in saved_endpoints.columns:
                true_col = c
                break

        if pred_col is not None and len(saved_endpoints) == len(pt_model):
            pt = saved_endpoints[pred_col].to_numpy(float)
            saved_endpoint_available = True

            if true_col is not None:
                y_te = saved_endpoints[true_col].to_numpy(float)

            print(f"Using saved endpoint predictions: {endpoint_results_path}")
        else:
            pt = pt_model
    except Exception:
        pt = pt_model
else:
    pt = pt_model

qs = {
    "C0 Engine-maximum residual (frozen)": float(qhat),
    "C1 Row-level split conformal": float(q_row),
    "C2 Engine-mean residual": float(q_engine_mean),
}

conformal_rows = []

for name, q in qs.items():
    lo_v = np.maximum(0, pv - q)
    up_v = pv + q
    cov_v = (yv >= lo_v) & (yv <= up_v)

    eng_cov = (
        pd.DataFrame(
            {
                "unit_number": va["unit_number"].to_numpy(),
                "covered": cov_v,
            }
        )
        .groupby("unit_number")["covered"]
        .all()
        .mean()
    )

    lo_t = np.maximum(0, pt - q)
    up_t = pt + q
    cov_t = (y_te >= lo_t) & (y_te <= up_t)

    conformal_rows.append(
        {
            "method": name,
            "q": float(q),
            "val_row_cov": float(cov_v.mean()),
            "val_engine_sim_cov": float(eng_cov),
            "val_mean_width": float((up_v - lo_v).mean()),
            "test_endpoint_cov": float(cov_t.mean()),
            "test_mean_width": float((up_t - lo_t).mean()),
            "test_clip": float((pt - q <= 0).mean()),
            "test_miss_above": int((y_te > up_t).sum()),
            "test_miss_below": int((y_te < lo_t).sum()),
        }
    )

cab = pd.DataFrame(conformal_rows)
print(cab.round(4).to_string(index=False))

cab.to_csv(output_dir / "conformal_ablation.csv", index=False)

out["conformal_ablation"] = cab.to_dict("records")
out["row_rank"] = [int(k_row), int(n_rows)]
out["engine_rank"] = [int(k_eng), int(n_eng)]
out["engine_max_q_from_current_calibration_data"] = q_engine_max_from_current_data
out["saved_endpoint_predictions_used"] = bool(saved_endpoint_available)


# =============================================================================
# C. BOOTSTRAP 95% CONFIDENCE INTERVALS
# =============================================================================

print("\nC. BOOTSTRAP CONFIDENCE INTERVALS")
print("=" * 80)

rng = np.random.default_rng(2026)
B = 5000
n_test = len(y_te)

require(n_test == len(pt), "Test predictions and targets disagree.")
require(n_test == 100, f"Expected 100 official test endpoints, got {n_test}")

error_rf = pt - y_te

bootstrap = {
    "MAE": [],
    "RMSE": [],
    "Bias": [],
    "R2": [],
}

naive_test_pred = np.maximum(195 - last["time_in_cycles"].to_numpy(float), 0)
error_naive = naive_test_pred - y_te

naive_bootstrap = {
    "MAE": [],
    "RMSE": [],
}

for _ in range(B):
    idx = rng.integers(0, n_test, n_test)

    e = error_rf[idx]
    yy = y_te[idx]

    bootstrap["MAE"].append(float(np.mean(np.abs(e))))
    bootstrap["RMSE"].append(float(np.sqrt(np.mean(e ** 2))))
    bootstrap["Bias"].append(float(np.mean(e)))

    denom = np.sum((yy - yy.mean()) ** 2)
    bootstrap["R2"].append(float(1 - np.sum(e ** 2) / denom) if denom > 0 else np.nan)

    en = error_naive[idx]
    naive_bootstrap["MAE"].append(float(np.mean(np.abs(en))))
    naive_bootstrap["RMSE"].append(float(np.sqrt(np.mean(en ** 2))))

ci = {
    k: [
        float(np.nanpercentile(v, 2.5)),
        float(np.nanpercentile(v, 97.5)),
    ]
    for k, v in bootstrap.items()
}

naive_test = {
    "MAE": float(np.mean(np.abs(error_naive))),
    "RMSE": float(np.sqrt(np.mean(error_naive ** 2))),
    "Bias": float(error_naive.mean()),
    "R2": float(1 - np.sum(error_naive ** 2) / np.sum((y_te - y_te.mean()) ** 2)),
    "phm_mean": float(phm(error_naive).mean()),
    "within20": float((np.abs(error_naive) <= 20).mean()),
    "ci": {
        k: [
            float(np.percentile(v, 2.5)),
            float(np.percentile(v, 97.5)),
        ]
        for k, v in naive_bootstrap.items()
    },
}

diff = np.abs(error_naive) - np.abs(error_rf)

gain_boot = [
    float(np.mean(diff[rng.integers(0, n_test, n_test)]))
    for _ in range(B)
]

out["bootstrap_ci_rf"] = ci
out["naive_test_reference"] = naive_test
out["mae_gain_rf_vs_naive"] = [
    float(diff.mean()),
    float(np.percentile(gain_boot, 2.5)),
    float(np.percentile(gain_boot, 97.5)),
]

print("RF bootstrap 95% CI:")
print(json.dumps(ci, indent=2, default=json_safe))

print("\nNaive reference:")
print(json.dumps(naive_test, indent=2, default=json_safe))

print("\nMAE gain RF vs naive [mean, 2.5%, 97.5%]:")
print(out["mae_gain_rf_vs_naive"])


# =============================================================================
# D. TRIAGE CLASSIFICATION AUDIT
# =============================================================================

print("\nD. TRIAGE CLASSIFICATION AUDIT")
print("=" * 80)

def cls(values):
    values = np.asarray(values, dtype=float)
    return np.select(
        [values <= 30, values <= 60, values <= 130],
        ["CRITICAL", "HIGH", "MEDIUM"],
        default="LOW",
    )


triage_order = ["CRITICAL", "HIGH", "MEDIUM", "LOW"]
severity = {label: idx for idx, label in enumerate(triage_order)}


def audit(y, p, lo, label):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    lo = np.asarray(lo, dtype=float)

    true_class = cls(y)
    pred_class = cls(p)

    cm = pd.crosstab(
        pd.Categorical(true_class, categories=triage_order, ordered=True),
        pd.Categorical(pred_class, categories=triage_order, ordered=True),
        dropna=False,
    )

    true_idx = np.array([severity[x] for x in true_class])
    pred_idx = np.array([severity[x] for x in pred_class])

    critical_count = int((true_class == "CRITICAL").sum())
    critical_high_count = int((true_idx <= 1).sum())

    return {
        "label": label,
        "n": int(len(y)),
        "exact": float((true_idx == pred_idx).mean()),
        "under_triage": float((pred_idx > true_idx).mean()),
        "over_triage": float((pred_idx < true_idx).mean()),
        "crit_recall": float(
            ((true_class == "CRITICAL") & (pred_class == "CRITICAL")).sum()
            / max(critical_count, 1)
        ),
        "crit_or_high_recall": float(
            ((true_idx <= 1) & (pred_idx <= 1)).sum()
            / max(critical_high_count, 1)
        ),
        "uncertainty_flag_rate": float((lo <= 0).mean()),
        "confusion_labels": triage_order,
        "confusion": cm.values.tolist(),
    }


audit_test = audit(
    y_te,
    pt,
    np.maximum(0, pt - qhat),
    "Official test endpoints (n=100)",
)

audit_val = audit(
    yv,
    pv,
    np.maximum(0, pv - qhat),
    f"Validation rows, full trajectories (n={len(yv)})",
)

for item in [audit_test, audit_val]:
    print("\n" + item["label"])
    print({k: v for k, v in item.items() if k not in ["label", "confusion"]})
    print(np.array(item["confusion"]))

out["triage_test"] = audit_test
out["triage_val"] = audit_val


# =============================================================================
# E. ENDPOINT CLASSIFICATION AND FAILURE CASES
# =============================================================================

print("\nE. ENDPOINT CLASSIFICATION AND FAILURE CASES")
print("=" * 80)

if "unit_number" in last.columns:
    endpoint_units = last["unit_number"].to_numpy(int)
else:
    endpoint_units = np.arange(1, len(y_te) + 1)

end = pd.DataFrame(
    {
        "unit_number": endpoint_units,
        "true": y_te,
        "pred": pt,
        "lower_90": np.maximum(0, pt - qhat),
        "upper_90": pt + qhat,
        "time_in_cycles": last["time_in_cycles"].to_numpy(int),
    }
)

end["true_class"] = cls(end["true"])
end["pred_class"] = cls(end["pred"])
end["err"] = end["pred"] - end["true"]
end["abs_err"] = end["err"].abs()
end["covered_90"] = (end["true"] >= end["lower_90"]) & (end["true"] <= end["upper_90"])

true_sev = end["true_class"].map(severity)
pred_sev = end["pred_class"].map(severity)

under = (
    end[true_sev < pred_sev]
    .sort_values("err", ascending=False)
    .reset_index(drop=True)
)

largest_over = (
    end.sort_values("err", ascending=False)
    .head(5)
    .reset_index(drop=True)
)

largest_abs = (
    end.sort_values("abs_err", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

print("\nUnder-triaged cases:")
if len(under) == 0:
    print("No under-triaged cases found.")
else:
    print(under.round(2).to_string(index=False))

print("\nLargest over-estimation cases:")
print(largest_over.round(2).to_string(index=False))

print("\nLargest absolute-error cases:")
print(largest_abs.round(2).to_string(index=False))

out["under_triaged_cases"] = under.round(3).to_dict("records")
out["largest_over"] = largest_over.round(3).to_dict("records")
out["largest_absolute_errors"] = largest_abs.round(3).to_dict("records")


# =============================================================================
# F. SAVE OUTPUTS
# =============================================================================

out["phase8_metadata"] = {
    "state_source": state_source,
    "project_root": str(root),
    "output_dir": str(output_dir),
    "feature_count": int(len(FEATURES)),
    "features": FEATURES,
    "train_rows": int(len(tr)),
    "validation_rows": int(len(va)),
    "calibration_rows": int(len(ca)),
    "test_endpoints": int(len(y_te)),
    "qhat": float(qhat),
    "note": (
        "If state_source is reconstructed_from_project_artifacts, validation "
        "and calibration ablations are post-closure reconstructed sensitivity "
        "analyses unless the original split records are later supplied."
    ),
}

classified_path = output_dir / "endpoints_classified.csv"
results_path = output_dir / "phase8_results.json"

end.to_csv(classified_path, index=False)

with results_path.open("w", encoding="utf-8") as handle:
    json.dump(out, handle, indent=2, default=json_safe)

print("\nPHASE 8 — COMPLETE")
print("=" * 80)
print(f"Saved classified endpoints: {classified_path}")
print(f"Saved Phase 8 JSON results: {results_path}")
print(f"Saved feature ablation: {output_dir / 'feature_ablation.csv'}")
print(f"Saved conformal ablation: {output_dir / 'conformal_ablation.csv'}")
print(f"State source: {state_source}")
print("Dependency on repro.py: Removed")
print("Dependency on /home/claude/state.joblib: Removed")
print("Model bundle extraction: Robust dict scanning enabled")

In [ ]:
# =============================================================================
# PHASE 8 — REVIEW POLICY AUDIT
#
# Fixes:
# - Removes: from repro import FEATURES
# - Does not require state.joblib
# - Uses Phase 8 saved endpoints_classified.csv
# - Reconstructs validation rows and validation predictions if state.joblib is absent
# - Fixes end.unit -> end.unit_number
#
# Output:
#   /content/Predictive_Maintenance_Thesis/results/phase8_ablation_audit/policy_results.json
# =============================================================================

from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import joblib

warnings.filterwarnings("ignore")


# =============================================================================
# 0. HELPERS
# =============================================================================

def require(condition, message):
    if not condition:
        raise AssertionError(message)


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def json_safe(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if pd.isna(value):
        return None
    raise TypeError(f"Not JSON serialisable: {type(value)}")


def find_project_root():
    candidates = [
        Path("/content/Predictive_Maintenance_Thesis"),
        Path("/content"),
        Path.cwd(),
    ]

    for candidate in candidates:
        if not candidate.exists():
            continue

        if (
            (candidate / "models").exists()
            or (candidate / "logs").exists()
            or list(candidate.rglob("train_FD001.txt"))
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate project root. Expected /content/Predictive_Maintenance_Thesis."
    )


def find_latest(root, patterns):
    root = Path(root)
    hits = []
    for pattern in patterns:
        hits.extend(root.rglob(pattern))
    hits = [p for p in hits if p.is_file()]
    if not hits:
        return None
    return sorted(hits, key=lambda p: p.stat().st_mtime, reverse=True)[0]


def read_cmapss(path):
    columns = (
        ["unit_number", "time_in_cycles"]
        + [f"operational_setting_{i}" for i in range(1, 4)]
        + [f"sensor_{i}" for i in range(1, 22)]
    )

    df = pd.read_csv(path, sep=r"\s+", header=None)
    require(df.shape[1] == 26, f"Expected 26 columns in {path}, got {df.shape[1]}")
    df.columns = columns
    df["unit_number"] = df["unit_number"].astype(int)
    df["time_in_cycles"] = df["time_in_cycles"].astype(int)
    return df


def add_rul_to_train(train):
    max_cycle = train.groupby("unit_number")["time_in_cycles"].transform("max")
    train = train.copy()
    train["rul"] = max_cycle - train["time_in_cycles"]
    return train


def object_has_predictor_interface(obj):
    return hasattr(obj, "predict") and callable(obj.predict)


def extract_predictor_and_features(obj):
    candidate_keys = [
        "model",
        "rf",
        "random_forest",
        "random_forest_model",
        "regressor",
        "estimator",
        "predictor",
        "pipeline",
        "best_estimator_",
        "fitted_model",
        "trained_model",
        "final_model",
        "sklearn_model",
    ]

    feature_keys = [
        "feature_order",
        "features",
        "FEATURES",
        "feature_names",
        "feature_names_in_",
        "model_features",
        "selected_features",
    ]

    def extract_features(container, estimator=None):
        if estimator is not None and hasattr(estimator, "feature_names_in_"):
            return list(estimator.feature_names_in_)

        if isinstance(container, dict):
            for key in feature_keys:
                if key in container:
                    try:
                        value = list(container[key])
                        if value and all(isinstance(x, str) for x in value):
                            return value
                    except Exception:
                        pass

        return None

    if object_has_predictor_interface(obj):
        return obj, extract_features(obj, obj), "direct_predictor"

    if isinstance(obj, dict):
        for key in candidate_keys:
            if key in obj:
                candidate = obj[key]
                if object_has_predictor_interface(candidate):
                    return candidate, extract_features(obj, candidate), f"dict_key:{key}"

        for key, value in obj.items():
            if object_has_predictor_interface(value):
                return value, extract_features(obj, value), f"dict_scanned_key:{key}"

        for key, value in obj.items():
            if isinstance(value, dict):
                try:
                    predictor, features, source = extract_predictor_and_features(value)
                    if predictor is not None:
                        if features is None:
                            features = extract_features(obj, predictor)
                        return predictor, features, f"nested_dict:{key}->{source}"
                except Exception:
                    pass

    for attr in candidate_keys:
        if hasattr(obj, attr):
            candidate = getattr(obj, attr)
            if object_has_predictor_interface(candidate):
                return candidate, extract_features(obj, candidate), f"attribute:{attr}"

    return None, None, "not_found"


def locate_model(root):
    candidates = []

    patterns = [
        "models/regression/fd001_random_forest_uncapped_bundle.joblib",
        "fd001_random_forest_uncapped_bundle.joblib",
        "*random_forest*uncapped*bundle*.joblib",
        "*random_forest*bundle*.joblib",
        "*rf*bundle*.joblib",
        "*.joblib",
    ]

    for pattern in patterns:
        candidates.extend(root.rglob(pattern))

    candidates = [p for p in candidates if p.is_file()]

    # Keep unique paths while preserving order.
    seen = set()
    unique = []
    for path in candidates:
        if path not in seen:
            unique.append(path)
            seen.add(path)

    rejected = []

    for path in unique:
        try:
            loaded = joblib.load(path)
            model, features, source = extract_predictor_and_features(loaded)

            if model is None:
                rejected.append((path, "no predictor found"))
                continue

            if features is None and hasattr(model, "feature_names_in_"):
                features = list(model.feature_names_in_)

            if features is None:
                rejected.append((path, "predictor found but feature order missing"))
                continue

            features = list(features)

            if not features or not all(isinstance(x, str) for x in features):
                rejected.append((path, "invalid feature list"))
                continue

            return model, features, path, source

        except Exception as error:
            rejected.append((path, str(error)))

    lines = ["Could not locate a valid fitted model.", "Rejected joblib files:"]
    for path, reason in rejected[:30]:
        lines.append(f"- {path}: {reason}")
    raise AssertionError("\n".join(lines))


def locate_state_joblib(root):
    candidates = [
        Path("state.joblib"),
        Path("/content/state.joblib"),
        root / "state.joblib",
        Path("/home/claude/state.joblib"),
    ]

    for path in candidates:
        if path.is_file():
            return path

    found = list(root.rglob("state.joblib"))
    found = [p for p in found if p.is_file()]
    if found:
        return sorted(found, key=lambda p: p.stat().st_mtime, reverse=True)[0]

    return None


def recover_qhat(root, end=None):
    if end is not None and {"upper_90", "pred"}.issubset(end.columns):
        q_values = end["upper_90"].to_numpy(float) - end["pred"].to_numpy(float)
        q_values = q_values[np.isfinite(q_values)]
        if len(q_values) > 0:
            return float(np.median(q_values))

    phase8_json = root / "results" / "phase8_ablation_audit" / "phase8_results.json"
    if phase8_json.is_file():
        try:
            data = read_json(phase8_json)
            if "phase8_metadata" in data and "qhat" in data["phase8_metadata"]:
                return float(data["phase8_metadata"]["qhat"])
        except Exception:
            pass

    logs_dir = root / "logs"
    if logs_dir.exists():
        policies = sorted(
            logs_dir.glob("phase7_current_snapshot_explainability_policy*.json"),
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )

        for policy_path in policies:
            try:
                payload = read_json(policy_path)
                spec = payload["specification"]
                test_policy_path = root / spec["inputs"]["test_policy"]["project_relative_path"]
                test_policy = read_json(test_policy_path)
                return float(test_policy["frozen_uncertainty_model"]["q_hat_cycles"])
            except Exception:
                pass

    calibration = find_latest(
        root,
        ["phase5_conformal_calibration_record.json", "*calibration*record*.json"],
    )
    if calibration is not None:
        try:
            return float(read_json(calibration)["q_hat_cycles"])
        except Exception:
            pass

    raise AssertionError("Could not recover qhat.")


def cls(values):
    values = np.asarray(values, dtype=float)
    return np.select(
        [values <= 30, values <= 60, values <= 130],
        [0, 1, 2],
        default=3,
    )


def policy_audit(y, p, q, label):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)

    require(len(y) == len(p), f"Length mismatch in {label}")

    lo = np.maximum(0, p - q)
    true_class = cls(y)
    pred_class = cls(p)

    # Mandatory human review rule:
    # 1. predicted CRITICAL/HIGH
    # 2. OR lower uncertainty bound <= 30 cycles
    review = (pred_class <= 1) | (lo <= 30)

    critical_caught = (
        ((true_class == 0) & review).sum()
        / max((true_class == 0).sum(), 1)
    )

    critical_high_caught = (
        ((true_class <= 1) & review).sum()
        / max((true_class <= 1).sum(), 1)
    )

    missed_crit_high = int(((true_class <= 1) & ~review).sum())
    burden = float(review.mean())

    precision_crit_high = (
        ((true_class <= 1) & review).sum()
        / max(review.sum(), 1)
    )

    interval_only = (lo <= 30) & (pred_class > 1)

    result = {
        "label": label,
        "n": int(len(y)),
        "review_burden": burden,
        "crit_caught": float(critical_caught),
        "crit_high_caught": float(critical_high_caught),
        "missed_crit_high": missed_crit_high,
        "precision_crit_high_in_review": float(precision_crit_high),
        "escalated_by_interval_only": float(interval_only.mean()),
        "review_count": int(review.sum()),
        "interval_only_review_count": int(interval_only.sum()),
        "class_counts_pred": np.bincount(pred_class, minlength=4).tolist(),
        "class_counts_true": np.bincount(true_class, minlength=4).tolist(),
    }

    print("\n" + label)
    print(result)

    return result, review, lo, true_class, pred_class


# =============================================================================
# 1. LOAD TEST ENDPOINTS FROM PHASE 8 OUTPUT
# =============================================================================

root = find_project_root()
phase8_dir = root / "results" / "phase8_ablation_audit"

end_path_candidates = [
    phase8_dir / "endpoints_classified.csv",
    Path("endpoints_classified.csv"),
    Path("/content/endpoints_classified.csv"),
]

end_path = None
for candidate in end_path_candidates:
    if candidate.is_file():
        end_path = candidate
        break

require(
    end_path is not None,
    "Could not find endpoints_classified.csv. Run Phase 8 first.",
)

end = pd.read_csv(end_path)

# Normalise expected column names.
rename = {}

if "unit" in end.columns and "unit_number" not in end.columns:
    rename["unit"] = "unit_number"

if "actual" in end.columns and "true" not in end.columns:
    rename["actual"] = "true"

if "rul_true" in end.columns and "true" not in end.columns:
    rename["rul_true"] = "true"

if "prediction" in end.columns and "pred" not in end.columns:
    rename["prediction"] = "pred"

if "raw_point_prediction" in end.columns and "pred" not in end.columns:
    rename["raw_point_prediction"] = "pred"

if rename:
    end = end.rename(columns=rename)

require("unit_number" in end.columns, "Endpoint file needs unit_number column.")
require("true" in end.columns, "Endpoint file needs true column.")
require("pred" in end.columns, "Endpoint file needs pred column.")

q = recover_qhat(root, end)

print("PHASE 8 — HUMAN REVIEW POLICY AUDIT")
print("=" * 80)
print(f"Project root: {root}")
print(f"Endpoint file: {end_path}")
print(f"Recovered qhat: {q:.6f}")


# =============================================================================
# 2. LOAD OR RECONSTRUCT VALIDATION ROWS
# =============================================================================

state_path = locate_state_joblib(root)

if state_path is not None:
    print("\nSTATE MODE")
    print("=" * 80)
    print(f"Using state file: {state_path}")

    st = joblib.load(state_path)
    require("va" in st and "pv" in st, "state.joblib must contain va and pv.")

    va = st["va"]
    yv = va["rul"].to_numpy(float)
    pv = np.asarray(st["pv"], dtype=float)

else:
    print("\nRECONSTRUCTION MODE")
    print("=" * 80)
    print("state.joblib not found. Reconstructing validation rows from project artifacts.")

    model, FEATURES, model_path, model_source = locate_model(root)

    train_path = find_latest(root, ["train_FD001.txt"])
    require(train_path is not None, "Could not find train_FD001.txt")

    train_full = add_rul_to_train(read_cmapss(train_path))

    units = np.array(sorted(train_full["unit_number"].unique()))
    require(len(units) == 100, f"Expected 100 FD001 training engines, got {len(units)}")

    # Same deterministic reconstruction used in Phase 8.
    validation_units = units[60:80]

    va = train_full[train_full["unit_number"].isin(validation_units)].copy()
    require(len(va) > 0, "Validation reconstruction produced zero rows.")

    missing_features = [f for f in FEATURES if f not in va.columns]
    require(not missing_features, f"Validation data missing features: {missing_features}")

    yv = va["rul"].to_numpy(float)
    pv = np.clip(model.predict(va[FEATURES]), 0, None)

    print(f"Model: {model_path}")
    print(f"Model source: {model_source}")
    print(f"Validation rows: {len(va)}")
    print(f"Validation engines: {va['unit_number'].nunique()}")


# =============================================================================
# 3. APPLY REVIEW POLICY
# =============================================================================

test_result, test_review, test_lo, test_true_class, test_pred_class = policy_audit(
    end["true"].to_numpy(float),
    end["pred"].to_numpy(float),
    q,
    "test",
)

val_result, val_review, val_lo, val_true_class, val_pred_class = policy_audit(
    yv,
    pv,
    q,
    "validation_rows",
)

results = [test_result, val_result]

print("\nPREDICTION THRESHOLD EQUIVALENT")
print("=" * 80)
print(
    "lower_bound <= 30 is equivalent to prediction <= qhat + 30 "
    f"when prediction > qhat. Threshold: {q + 30:.6f}"
)


# =============================================================================
# 4. CASE CHECKS
# =============================================================================

print("\nSELECTED TEST ENGINES")
print("=" * 80)

selected_units = [15, 41, 18, 1]
selected = end[end["unit_number"].isin(selected_units)].copy()

if len(selected) == 0:
    print("No selected units found.")
else:
    display_cols = [
        c for c in [
            "unit_number",
            "true",
            "pred",
            "lower_90",
            "upper_90",
            "time_in_cycles",
            "true_class",
            "pred_class",
            "err",
            "abs_err",
            "covered_90",
        ]
        if c in selected.columns
    ]
    print(selected[display_cols].round(3).to_string(index=False))


print("\nVALIDATION MISSED CRITICAL/HIGH ROWS")
print("=" * 80)

missed_mask = (val_true_class <= 1) & ~((val_pred_class <= 1) | (val_lo <= 30))

print(f"val missed rows: {int(missed_mask.sum())}")

d = pd.DataFrame(
    {
        "u": va["unit_number"].to_numpy(),
        "cyc": va["time_in_cycles"].to_numpy(),
        "y": yv,
        "p": pv,
        "lower": val_lo,
        "true_class": val_true_class,
        "pred_class": val_pred_class,
        "review": val_review,
    }
)

print("\nRows where true RUL == 99 and prediction > 150:")
case_99 = d[(d["y"] == 99) & (d["p"] > 150)]
if len(case_99) == 0:
    print("No matching rows.")
else:
    print(case_99.round(3).to_string(index=False))

print("\nValidation engine 10, cycle 1:")
case_engine10 = d[(d["u"] == 10) & (d["cyc"] == 1)]
if len(case_engine10) == 0:
    print("No matching row. This is expected if engine 10 is not in the reconstructed validation split.")
else:
    print(case_engine10.round(3).to_string(index=False))


# =============================================================================
# 5. SAVE
# =============================================================================

output_dir = phase8_dir
output_dir.mkdir(parents=True, exist_ok=True)

policy_path = output_dir / "policy_results.json"
with policy_path.open("w", encoding="utf-8") as handle:
    json.dump(results, handle, indent=2, default=json_safe)

# Also save detailed reviewed endpoint file.
end_policy = end.copy()
end_policy["policy_lower_bound"] = test_lo
end_policy["policy_true_class_code"] = test_true_class
end_policy["policy_pred_class_code"] = test_pred_class
end_policy["mandatory_review"] = test_review

endpoint_policy_path = output_dir / "endpoints_policy_review.csv"
end_policy.to_csv(endpoint_policy_path, index=False)

print("\nPOLICY AUDIT COMPLETE")
print("=" * 80)
print(f"Saved policy results: {policy_path}")
print(f"Saved endpoint policy review: {endpoint_policy_path}")
print("Dependency on repro.py: Removed")
print("Dependency on state.joblib: Optional only")

In [ ]:
# =============================================================================
# INDEPENDENT REPRODUCTION OF FROZEN FD001 PIPELINE — COLAB VERSION
#
# Fix:
# - Removes hard-coded /home/claude/data/
# - Automatically locates train_FD001.txt, test_FD001.txt, RUL_FD001.txt
# - Reproduces original RNG(42) engine split
# - Saves:
#     /content/Predictive_Maintenance_Thesis/state.joblib
#     /content/Predictive_Maintenance_Thesis/results/phase8_ablation_audit/endpoints.csv
#     /content/Predictive_Maintenance_Thesis/results/phase8_ablation_audit/repro_ok.json
# =============================================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestRegressor


# -----------------------------------------------------------------------------
# 0. Locate project and FD001 files
# -----------------------------------------------------------------------------

def require(condition, message):
    if not condition:
        raise AssertionError(message)


def find_project_root():
    candidates = [
        Path("/content/Predictive_Maintenance_Thesis"),
        Path("/content"),
        Path.cwd(),
    ]

    for candidate in candidates:
        if candidate.exists() and list(candidate.rglob("train_FD001.txt")):
            return candidate

    raise FileNotFoundError(
        "Could not locate train_FD001.txt under /content or current directory."
    )


def find_file(root, filename):
    hits = sorted(
        [p for p in Path(root).rglob(filename) if p.is_file()],
        key=lambda p: (len(str(p)), str(p)),
    )
    require(hits, f"Could not find {filename} under {root}")
    return hits[0]


root = find_project_root()

train_path = find_file(root, "train_FD001.txt")
test_path = find_file(root, "test_FD001.txt")
rul_path = find_file(root, "RUL_FD001.txt")

output_dir = root / "results" / "phase8_ablation_audit"
output_dir.mkdir(parents=True, exist_ok=True)

print("REPRODUCTION SETUP")
print("=" * 80)
print(f"Project root: {root}")
print(f"Train file: {train_path}")
print(f"Test file: {test_path}")
print(f"RUL file: {rul_path}")
print(f"Output directory: {output_dir}")


# -----------------------------------------------------------------------------
# 1. Dataset columns and frozen feature list
# -----------------------------------------------------------------------------

COLS = (
    ["unit_number", "time_in_cycles", "operational_setting_1",
     "operational_setting_2", "operational_setting_3"]
    + [f"sensor_{i}" for i in range(1, 22)]
)

FEATURES = [
    "time_in_cycles",
    "operational_setting_1",
    "operational_setting_2",
    "sensor_2",
    "sensor_3",
    "sensor_4",
    "sensor_6",
    "sensor_7",
    "sensor_8",
    "sensor_9",
    "sensor_11",
    "sensor_12",
    "sensor_13",
    "sensor_14",
    "sensor_15",
    "sensor_17",
    "sensor_20",
    "sensor_21",
]


def load_cmapss(path):
    df = pd.read_csv(path, sep=r"\s+", header=None, names=COLS)
    require(df.shape[1] == 26, f"Unexpected column count in {path}: {df.shape}")
    df["unit_number"] = df["unit_number"].astype(int)
    df["time_in_cycles"] = df["time_in_cycles"].astype(int)
    return df


train = load_cmapss(train_path)
test = load_cmapss(test_path)

rul_true = pd.read_csv(
    rul_path,
    sep=r"\s+",
    header=None,
).iloc[:, 0].to_numpy(float)

require(len(rul_true) == 100, f"Expected 100 RUL labels, got {len(rul_true)}")


# -----------------------------------------------------------------------------
# 2. Reproduce original engine split
# -----------------------------------------------------------------------------

ids = np.sort(train["unit_number"].unique())
require(len(ids) == 100, f"Expected 100 training engines, got {len(ids)}")

perm = np.random.default_rng(42).permutation(ids)

TR = np.sort(perm[:60])
VA = np.sort(perm[60:80])
CA = np.sort(perm[80:])

expected_VA = [10, 16, 17, 31, 32, 35, 45, 49, 55, 56,
               59, 61, 63, 68, 70, 73, 75, 83, 89, 99]

expected_CA = [7, 9, 12, 13, 14, 15, 20, 23, 36, 37,
               42, 48, 50, 54, 65, 66, 67, 78, 86, 91]

require(list(VA) == expected_VA, f"Validation split mismatch: {list(VA)}")
require(list(CA) == expected_CA, f"Calibration split mismatch: {list(CA)}")

train["rul"] = (
    train.groupby("unit_number")["time_in_cycles"].transform("max")
    - train["time_in_cycles"]
)

tr = train[train["unit_number"].isin(TR)].reset_index(drop=True)
va = train[train["unit_number"].isin(VA)].reset_index(drop=True)
ca = train[train["unit_number"].isin(CA)].reset_index(drop=True)

require(
    (len(tr), len(va), len(ca)) == (12310, 4217, 4104),
    f"Split row count mismatch: {(len(tr), len(va), len(ca))}",
)

print("\nSPLIT VERIFIED")
print("=" * 80)
print(f"Train engines: {len(TR)} | rows: {len(tr)}")
print(f"Validation engines: {len(VA)} | rows: {len(va)}")
print(f"Calibration engines: {len(CA)} | rows: {len(ca)}")


# -----------------------------------------------------------------------------
# 3. Metrics and model functions
# -----------------------------------------------------------------------------

def phm(e):
    e = np.asarray(e, dtype=float)
    return np.where(
        e >= 0,
        np.exp(e / 10.0) - 1.0,
        np.exp(-e / 13.0) - 1.0,
    )


def engine_metrics(units, y, p):
    units = np.asarray(units)
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    e = p - y

    df = pd.DataFrame(
        {
            "u": units,
            "y": y,
            "p": p,
            "sq": e ** 2,
        }
    )

    engine_rmse = (
        df.groupby("u")["sq"]
        .mean()
        .pow(0.5)
        .to_numpy(float)
    )

    return {
        "mean_engine_rmse": float(engine_rmse.mean()),
        "median_engine_rmse": float(np.median(engine_rmse)),
        "pooled_rmse": float(np.sqrt(np.mean(e ** 2))),
        "pooled_mae": float(np.mean(np.abs(e))),
        "bias": float(e.mean()),
        "phm_mean": float(phm(e).mean()),
    }


def fit_rf(features, target="rul", data=tr, n=500):
    model = RandomForestRegressor(
        n_estimators=n,
        criterion="squared_error",
        max_depth=8,
        min_samples_leaf=10,
        max_features=0.5,
        bootstrap=True,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(data[features], data[target].to_numpy(float))
    return model


# -----------------------------------------------------------------------------
# 4. Reproduce validation model
# -----------------------------------------------------------------------------

rf = fit_rf(FEATURES)

pv = np.clip(rf.predict(va[FEATURES]), 0, None)

mv = engine_metrics(
    va["unit_number"],
    va["rul"].to_numpy(float),
    pv,
)

print("\nVALIDATION reproduced:")
print(json.dumps(mv, indent=2))

require(
    abs(mv["mean_engine_rmse"] - 29.478541) < 1e-5,
    f"RF validation mean_engine_rmse not reproduced: {mv['mean_engine_rmse']}",
)

require(
    abs(mv["phm_mean"] - 83.670209) < 1e-5,
    f"RF validation phm_mean not reproduced: {mv['phm_mean']}",
)


# -----------------------------------------------------------------------------
# 5. Reproduce conformal q_hat
# -----------------------------------------------------------------------------

pc = np.clip(rf.predict(ca[FEATURES]), 0, None)

ca = ca.copy()
ca["res"] = np.abs(ca["rul"].to_numpy(float) - pc)

S = np.sort(
    ca.groupby("unit_number")["res"]
    .max()
    .to_numpy(float)
)

qhat = float(S[18])

print("\nq_hat reproduced:")
print(qhat)

require(
    abs(qhat - 107.896520) < 1e-5,
    f"q_hat not reproduced: {qhat}",
)


# -----------------------------------------------------------------------------
# 6. Reproduce official test endpoints
# -----------------------------------------------------------------------------

last = (
    test.groupby("unit_number", sort=True)
    .tail(1)
    .sort_values("unit_number")
    .reset_index(drop=True)
)

require(len(last) == 100, f"Expected 100 official endpoints, got {len(last)}")

pt = np.clip(rf.predict(last[FEATURES]), 0, None)

e = pt - rul_true

mae = float(np.mean(np.abs(e)))
rmse = float(np.sqrt(np.mean(e ** 2)))
bias = float(e.mean())

print("\nTEST reproduced:")
print(f"MAE  = {mae}")
print(f"RMSE = {rmse}")
print(f"bias = {bias}")

require(
    abs(mae - 18.882866) < 1e-5,
    f"Test MAE not reproduced: {mae}",
)

require(
    abs(rmse - 25.725353) < 1e-5,
    f"Test RMSE not reproduced: {rmse}",
)


# -----------------------------------------------------------------------------
# 7. Save endpoints and state.joblib
# -----------------------------------------------------------------------------

end = pd.DataFrame(
    {
        "unit": last["unit_number"].to_numpy(int),
        "unit_number": last["unit_number"].to_numpy(int),
        "cycle": last["time_in_cycles"].to_numpy(int),
        "time_in_cycles": last["time_in_cycles"].to_numpy(int),
        "true": rul_true,
        "pred": pt,
        "lower": np.maximum(0, pt - qhat),
        "upper": pt + qhat,
        "lower_90": np.maximum(0, pt - qhat),
        "upper_90": pt + qhat,
        "error_prediction_minus_actual": e,
        "absolute_error": np.abs(e),
    }
)

endpoints_path = output_dir / "endpoints.csv"
endpoints_classified_path = output_dir / "endpoints_reproduced.csv"
repro_path = output_dir / "repro_ok.json"
state_path = root / "state.joblib"

end.to_csv(endpoints_path, index=False)
end.to_csv(endpoints_classified_path, index=False)

repro_record = {
    "status": "all_reproduction_checks_passed",
    "train_file": str(train_path),
    "test_file": str(test_path),
    "rul_file": str(rul_path),
    "validation": mv,
    "qhat": float(qhat),
    "test_mae": mae,
    "test_rmse": rmse,
    "test_bias": bias,
    "train_units": TR.tolist(),
    "validation_units": VA.tolist(),
    "calibration_units": CA.tolist(),
    "features": FEATURES,
}

with repro_path.open("w", encoding="utf-8") as handle:
    json.dump(repro_record, handle, indent=2)

joblib.dump(
    {
        "rf": rf,
        "tr": tr,
        "va": va,
        "ca": ca,
        "last": last,
        "rul_true": rul_true,
        "qhat": qhat,
        "S": S,
        "pc": pc,
        "pv": pv,
        "pt": pt,
        "FEATURES": FEATURES,
    },
    state_path,
)

# Optional convenience copy for notebook cells that call joblib.load("state.joblib")
joblib.dump(
    {
        "rf": rf,
        "tr": tr,
        "va": va,
        "ca": ca,
        "last": last,
        "rul_true": rul_true,
        "qhat": qhat,
        "S": S,
        "pc": pc,
        "pv": pv,
        "pt": pt,
        "FEATURES": FEATURES,
    },
    Path("/content/state.joblib"),
)

print("\nALL REPRODUCTION CHECKS PASSED")
print("=" * 80)
print(f"Saved state: {state_path}")
print(f"Saved convenience state: /content/state.joblib")
print(f"Saved endpoints: {endpoints_path}")
print(f"Saved reproduced endpoints: {endpoints_classified_path}")
print(f"Saved reproduction record: {repro_path}")

In [ ]:
# CREATE A SELF-CONTAINED HTML PREVIEW FROM SAVED PHASE 4–7 OUTPUTS

from pathlib import Path
from html import escape
import base64
import json
import mimetypes
from datetime import datetime, timezone

ROOT = Path("/content/Predictive_Maintenance_Thesis")
FIGURE_DIR = ROOT / "results/phase7_thesis_visualisations"
OUTPUT_DIR = ROOT / "results/html_preview"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HTML_PATH = OUTPUT_DIR / "Predictive_Maintenance_Thesis_Phases_4_to_7_Preview.html"


def file_to_data_uri(path):
    mime_type = mimetypes.guess_type(path.name)[0] or "application/octet-stream"
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:{mime_type};base64,{encoded}"


def read_json_if_exists(path):
    if not path.is_file():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None


def figure_block(path, title):
    data_uri = file_to_data_uri(path)
    return f"""
    <section class="figure-card">
        <h3>{escape(title)}</h3>
        <img src="{data_uri}" alt="{escape(title)}">
        <p class="file-name">{escape(path.name)}</p>
    </section>
    """


phase6_summary = ROOT / "results/phase6_official_test_summary_metrics.csv"
phase6_lifecycle = ROOT / "results/phase6_official_test_lifecycle_stage_metrics.csv"
phase7_captions = FIGURE_DIR / "captions_and_research_summary.md"
phase7_record = FIGURE_DIR / "visualisation_record.json"
phase7_review = ROOT / "logs/phase7_visual_review_record.json"

figure_titles = {
    "01_global_importance.png": "Global SHAP feature importance",
    "02_shap_beeswarm.png": "SHAP contribution distributions",
    "03_engine_shap_heatmap.png": "Engine-by-feature SHAP heatmap",
    "04_dependence_panels.png": "SHAP dependence panels",
    "05_waterfall_engine_001.png": "Local explanation — Engine 1",
    "05_waterfall_engine_004.png": "Local explanation — Engine 4",
    "05_waterfall_engine_032.png": "Local explanation — Engine 32",
    "05_waterfall_engine_034.png": "Local explanation — Engine 34",
    "05_waterfall_engine_071.png": "Local explanation — Engine 71",
    "06_prediction_quintile_heatmap.png": "Prediction-quintile SHAP heatmap",
    "07_results_limitations.png": "Official results and limitations",
    "08_research_roadmap.png": "Research progress and roadmap",
    "09_human_review_architecture.png": "Prediction-to-human-review architecture",
}

figure_html = []

for filename, title in figure_titles.items():
    path = FIGURE_DIR / filename
    if path.is_file() and path.stat().st_size > 0:
        figure_html.append(figure_block(path, title))

summary_html = ""
if phase6_summary.is_file():
    summary_html += "<h2>Phase 6 official endpoint summary</h2>"
    summary_html += (
        "<pre>"
        + escape(phase6_summary.read_text(encoding="utf-8"))
        + "</pre>"
    )

if phase6_lifecycle.is_file():
    summary_html += "<h2>Lifecycle-stage metrics</h2>"
    summary_html += (
        "<pre>"
        + escape(phase6_lifecycle.read_text(encoding="utf-8"))
        + "</pre>"
    )

captions_html = ""
if phase7_captions.is_file():
    captions_html = f"""
    <h2>Figure captions and research summary</h2>
    <pre>{escape(phase7_captions.read_text(encoding="utf-8"))}</pre>
    """

records_html = ""

for label, path in [
    ("Phase 7 visualisation record", phase7_record),
    ("Phase 7 final visual review record", phase7_review),
]:
    record = read_json_if_exists(path)
    if record is not None:
        records_html += f"""
        <details>
            <summary>{escape(label)}</summary>
            <pre>{escape(json.dumps(record, indent=2, ensure_ascii=False))}</pre>
        </details>
        """

generated_at = datetime.now(timezone.utc).isoformat()

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Predictive Maintenance Thesis Preview</title>
<style>
body {{
    font-family: Arial, sans-serif;
    max-width: 1400px;
    margin: 0 auto;
    padding: 32px;
    color: #1f2937;
    line-height: 1.55;
}}
h1 {{
    color: #123b63;
    border-bottom: 3px solid #2f6bde;
    padding-bottom: 12px;
}}
h2 {{
    color: #174f7a;
    margin-top: 42px;
}}
h3 {{
    color: #243b53;
    font-size: 18px;
}}
.subtitle {{
    color: #52606d;
    font-size: 17px;
}}
.notice {{
    background: #eef6ff;
    border-left: 5px solid #2f6bde;
    padding: 16px;
    margin: 22px 0;
}}
.grid {{
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(420px, 1fr));
    gap: 24px;
}}
.figure-card {{
    border: 1px solid #d9e2ec;
    border-radius: 8px;
    padding: 16px;
    background: #ffffff;
    box-shadow: 0 2px 8px rgba(0,0,0,0.06);
}}
.figure-card img {{
    width: 100%;
    height: auto;
    display: block;
}}
.file-name {{
    color: #627d98;
    font-size: 12px;
}}
pre {{
    background: #f5f7fa;
    padding: 16px;
    overflow-x: auto;
    border-radius: 6px;
    font-size: 12px;
}}
details {{
    margin: 14px 0;
    border: 1px solid #d9e2ec;
    padding: 12px;
    border-radius: 6px;
}}
footer {{
    margin-top: 48px;
    color: #627d98;
    font-size: 13px;
}}
</style>
</head>
<body>

<h1>Human-in-the-Loop Explainable Predictive Maintenance</h1>

<p class="subtitle">
NASA C-MAPSS FD001 | Phases 4–7 research evidence preview
</p>

<div class="notice">
<strong>Scope:</strong> This preview presents the frozen model-selection evidence,
conformal uncertainty results, official endpoint evaluation and SHAP explainability
figures. It is a descriptive research artefact, not an operational safety guarantee.
</div>

<h2>Research status</h2>
<ul>
<li>Selected model: Random Forest for uncapped RUL.</li>
<li>Official endpoint evaluation: 100 test engines.</li>
<li>SHAP explanations: 100 engines × 18 features.</li>
<li>Conformal confidence level: 90%.</li>
<li>Frozen q-hat: 107.896520 cycles.</li>
<li>Model inference and SHAP computation were not repeated during visual review.</li>
<li>Historical provenance limitation remains documented.</li>
</ul>

{summary_html}

{captions_html}

<h2>Thesis figures</h2>
<div class="grid">
{''.join(figure_html)}
</div>

<h2>Interpretation controls</h2>
<ul>
<li>SHAP values explain fitted-model behaviour; they do not establish physical causation.</li>
<li>Prediction-quintile groups are descriptive, not independently defined operating regimes.</li>
<li>Observed endpoint coverage is not a safety guarantee.</li>
<li>Positive prediction bias and wide intervals must be reported.</li>
<li>The human-review architecture is a conceptual integration target.</li>
<li>Operational workflow validation remains separate from model evaluation.</li>
</ul>

<h2>Evidence records</h2>
{records_html}

<footer>
Generated from saved project outputs on {escape(generated_at)}.
No model, SHAP calculation or official evaluation was rerun for this HTML export.
</footer>

</body>
</html>
"""

HTML_PATH.write_text(html, encoding="utf-8")

print("HTML PREVIEW EXPORT")
print("=" * 70)
print("PASS — Standalone HTML preview created")
print("HTML file:", HTML_PATH)
print("Figure sections included:", len(figure_html))
print("Images embedded inside HTML: Yes")
print("Google Drive used: No")
print("Models rerun: No")
print("SHAP recomputed: No")
print("Official test rerun: No")
print("File size (MB):", round(HTML_PATH.stat().st_size / 1_048_576, 2))

In [ ]:
from pathlib import Path
from google.colab import files

html_path = Path(
    "/content/Predictive_Maintenance_Thesis"
    "/results/html_preview/"
    "Predictive_Maintenance_Thesis_Phases_4_to_7_Preview.html"
)

if not html_path.is_file():
    raise FileNotFoundError(
        f"HTML export not found: {html_path}"
    )

if html_path.stat().st_size == 0:
    raise RuntimeError("The HTML file is empty.")

print("HTML export verified")
print("Size (MB):", round(html_path.stat().st_size / 1_048_576, 2))
print("Starting browser download...")

files.download(str(html_path))

In [ ]:
# =============================================================================
# FULL COLAB NOTEBOOK HTML EXPORT
# Exports current executed Colab notebook from top to bottom as HTML.
# Includes:
# - all code cells
# - markdown cells
# - printed outputs
# - tables
# - plots / PNG figures
# - execution outputs
# =============================================================================

from pathlib import Path
import json
import subprocess
import sys
import time

from google.colab import files

# -----------------------------------------------------------------------------
# 1. Output paths
# -----------------------------------------------------------------------------

export_dir = Path("/content/colab_full_html_export")
export_dir.mkdir(parents=True, exist_ok=True)

ipynb_path = export_dir / "Ranitabh_Thesis_Implementation_FULL_EXECUTED.ipynb"
html_path = export_dir / "Ranitabh_Thesis_Implementation_FULL_EXECUTED.html"

print("FULL COLAB HTML EXPORT")
print("=" * 80)
print(f"Export directory: {export_dir}")

# -----------------------------------------------------------------------------
# 2. Get current notebook from Colab memory
# -----------------------------------------------------------------------------

def get_current_colab_notebook():
    try:
        from google.colab import _message
    except Exception as e:
        raise RuntimeError(f"Could not import google.colab._message: {e}")

    last_response = None
    last_error = None

    for timeout in [30, 60, 120]:
        try:
            print(f"Trying to capture notebook with timeout {timeout}s...")
            response = _message.blocking_request(
                "get_ipynb",
                timeout_sec=timeout,
            )

            last_response = response

            if isinstance(response, dict):
                if "ipynb" in response:
                    nb = response["ipynb"]

                    if isinstance(nb, str):
                        nb = json.loads(nb)

                    if isinstance(nb, dict) and "cells" in nb:
                        print("PASS — Notebook captured from Colab memory.")
                        return nb

                if "cells" in response:
                    print("PASS — Notebook captured directly.")
                    return response

                for key in ["notebook", "data", "content"]:
                    if key in response:
                        nb = response[key]

                        if isinstance(nb, str):
                            try:
                                nb = json.loads(nb)
                            except Exception:
                                pass

                        if isinstance(nb, dict) and "cells" in nb:
                            print(f"PASS — Notebook captured from key: {key}")
                            return nb

        except Exception as e:
            last_error = e
            print(f"Attempt failed: {e}")
            time.sleep(2)

    diagnostic = {
        "last_error": repr(last_error),
        "last_response_type": type(last_response).__name__,
        "last_response_keys": list(last_response.keys()) if isinstance(last_response, dict) else None,
    }

    raise RuntimeError(
        "Colab did not return the current notebook.\n"
        "First try: Runtime → Run all, then run this cell again.\n"
        "If still failing: File → Download → Download .ipynb, then convert manually.\n\n"
        f"Diagnostic:\n{json.dumps(diagnostic, indent=2)}"
    )


notebook = get_current_colab_notebook()

# -----------------------------------------------------------------------------
# 3. Save notebook file
# -----------------------------------------------------------------------------

ipynb_path.write_text(
    json.dumps(notebook, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"PASS — Saved notebook: {ipynb_path}")
print(f"Notebook cells: {len(notebook.get('cells', []))}")

# -----------------------------------------------------------------------------
# 4. Convert notebook to HTML using nbconvert
# -----------------------------------------------------------------------------

print("Converting notebook to HTML...")

# Make sure nbconvert exists
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "nbconvert"],
    check=True,
)

cmd = [
    "jupyter",
    "nbconvert",
    "--to",
    "html",
    "--HTMLExporter.exclude_input_prompt=True",
    "--HTMLExporter.exclude_output_prompt=True",
    "--output",
    str(html_path),
    str(ipynb_path),
]

result = subprocess.run(
    cmd,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

if result.returncode != 0:
    print("STDOUT:")
    print(result.stdout)
    print("STDERR:")
    print(result.stderr)
    raise RuntimeError("HTML conversion failed.")

if not html_path.exists() or html_path.stat().st_size == 0:
    raise RuntimeError("HTML file was not created or is empty.")

# -----------------------------------------------------------------------------
# 5. Download HTML
# -----------------------------------------------------------------------------

print("\nEXPORT COMPLETE")
print("=" * 80)
print("PASS — Full executed notebook exported")
print("PASS — Code cells included")
print("PASS — Markdown cells included")
print("PASS — Outputs included")
print("PASS — Tables and plots included if present in notebook output")
print(f"HTML path: {html_path}")
print(f"HTML size MB: {html_path.stat().st_size / (1024 * 1024):.2f}")

print("\nStarting download...")
files.download(str(html_path))